# Ratatosk 1.9.0.75 — GCPL electrochemical analysis

Galvanostatic cycling data in; cycling performance, dQ/dV curves, fitted redox
features tracked across life, and — this is the part that is new — a statement
about which of those numbers are measurements and which are not.

## How this notebook is arranged

Twelve **definition** cells, then twelve **execution** cells. Run top to bottom.
The definition cells declare functions and change nothing; the execution cells
do the work, in the order the data flows.

```
   DEFINITIONS                                  EXECUTION
   A  read a cycler export                      1  setup
   B  Q, V, I, t -> dQ/dV                       2  what to analyse   <- EDIT
   C  find peaks, choose reference cycle        3  load, open the run
   D  build and run the model                   4  preprocess
   E  does the answer mean anything             5  cycle integrity
   F  track, pair, attribute, classify         6a  cycling tables
   H1 dQ/dV figures                            6b  cycling figures (paper)
   H2 peak/fit/trend figures + adapters        6c  cycling figures (support)
   J1 J2 J3  cycling figures                    7  dQ/dV figures
   K  run folders and the manifest              8  detection
                                                9  fitting
                                               10  quality
                                               11  analysis
                                               12  fit figures
                                               13  export
                                               14  close the run
```

Data flows one way:

```
   A read  ->  B curve  ->  F integrity  ->  C detect  ->  D fit  ->  F analyse  ->  figures
                                 |                          |            |
                                 +-- decides the ------------+            |
                                     reference cycle          E quality --+
                                                              decides what
                                                              is withheld
```

`E quality` is consulted by the others and depends on none of them. Note that
**integrity runs before detection**: the test for a two-phase plateau reads
|dQ/dV| rather than a peak list, so it needs no peaks, and the reference-cycle
chooser can therefore be handed the integrity verdict.

## The four decisions this code makes, and where

Everything else is arithmetic. These four are judgements, and each is made in
one named place so it can be argued with.

**1. Which cycle defines the peak list.** (Cell C, `choose_reference_cycle`)
That cycle's peaks are fitted and tracked in *every* cycle, so a bad reference
cycle is not a bad cycle — it is a bad dataset. On P3 cell C the old hardcoded
default landed on a half-cycle with 19.9% coulombic efficiency, and its peak
list set the fit for all 220 cycles. The rule now: the **lowest** cycle at or
above 2 whose half-cycles both pass the integrity check. Lowest, not best — a
fingerprint should come from early in life.

**2. What the reversed charge was doing.** (Cell F, `classify_reversals`)
Where dQ/dV is negative during charge, the potential is moving backwards under
applied current. Three unrelated things do that and the old single threshold
conflated all three:

| verdict | test | meaning |
|---|---|---|
| `plateau` | full current, and \|dQ/dV\| there is >= 5% of this half-cycle's own maximum | a two-phase equilibrium. Real, intended charge |
| `cv_hold` | current below the constant-current value, within 50 mV of the window edge | protocol, not chemistry |
| `parasitic` | full current, \|dQ/dV\| small, away from the limit | charge with no redox process to account for it |

Every term is measured against the half-cycle **itself** — its own peak
\|dQ/dV\|, its own current, its own capacity. That is why it transfers between
chemistries, and it is exactly what a pooled threshold could not do. Measured
across 1,384 half-cycles of six cells: every anomalous one sits in a cycle
whose coulombic efficiency is below 60%, and above CE 60% nothing anywhere
carries more than 0.4% of its capacity backwards.

**3. How the model is set up.** (Cell D, `spec_for_profile`) The baseline degree
and sigma bounds come from the dataset's own profile class. A sharp two-phase
peak (LTO, LFP) sits on a flat background, so a wide sigma bound would let a
component inflate until it *is* the background: sigma 3-30 mV, linear baseline.
A broad layered oxide sits on a curved background a straight line cannot
follow: sigma 5-200 mV, cubic baseline. Getting this wrong is what a visibly
bad fit looks like — R2 0.27 against 0.54 on the same data.

**4. Whether a capacity share is a measurement.** (Cell E, `closure_interval`;
applied in Cell F, `capacity_attribution`) A broad pseudo-Voigt and a cubic
baseline across a 1.4 V window are nearly the same function. Where that is
true, the split between "peak" and "background" is not determined by the data,
and any single closure value is an artefact of the baseline degree somebody
chose. So the model is refitted at several baseline degrees and the **spread**
is reported. Measured: LTO 0.00-0.01, P3 0.00-0.09, NMC111 0.35-0.40. Above a
spread of 0.15 — inside the gap in those measurements — the per-peak *share* of
capacity is set to NaN and the reason recorded. The fitted areas are kept:
withholding a share is not deleting a number.

## What is measured twice, on purpose

```
   integral fidelity  =  integral(dQ/dV) dV  /  the counter's capacity
                         "does the derivative account for the charge?"

   capacity closure   =  sum(component areas)  /  integral(dQ/dV) dV
                         "do the fitted peaks account for the curve?"
```

They are deliberately orthogonal. Using the counter's capacity as closure's
denominator would hide a preprocessing failure inside a fitting number.

## Provenance

Every run writes to its own timestamped folder with a `run_manifest.json`
recording the input checksums, the package versions, and every switch in force.
With the version number deliberately held across a series of behaviour changes,
those switches are the only thing distinguishing one run from another.


In [ ]:
# CELL 1 — SETUP
# =============================================================================
# The one place the version is written. It names the run folder
# (<timestamp>_v<version>), is recorded in run_manifest.json and in each
# dataset_info.txt, and is printed on every report — so any figure can be
# traced back to the code that made it.
RATATOSK_VERSION = "1.9.0.75"

# All of RATATOSK is in this notebook — there is no package to install and
# nothing to import beyond the scientific stack, so the whole pipeline can be
# read top to bottom. That stack itself is still required:
#
#   conda install -c conda-forge numpy pandas scipy matplotlib seaborn \
#                                lmfit joblib openpyxl python-calamine
#
# python-calamine is optional and reads these Excel exports 5-7x faster than
# openpyxl, with a maximum numeric difference of exactly zero. seaborn is
# needed only for the key-cycle voltage profile. The versions actually
# present are printed below and recorded in the manifest.

import os, sys, re, glob, json, time, shutil, getpass, platform, hashlib
import contextlib, functools
import io as _io
import warnings
from dataclasses import dataclass, field
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import matplotlib.patheffects as path_effects
from matplotlib import gridspec
from matplotlib.colors import Normalize

import builtins as _builtins
from scipy.signal import find_peaks, peak_widths, savgol_filter
from scipy.optimize import linear_sum_assignment
from lmfit.models import PseudoVoigtModel, PolynomialModel

try:
    import seaborn as sns
except ImportError:
    sns = None

# FutureWarnings are suppressed for the whole run. They are the ones that
# change numbers between pandas releases, so this is not free: the fact that
# they were hidden is recorded in the manifest by `check_environment`, beside
# the versions that produced the run. Comment this out when a result looks
# wrong and you want to know why.
warnings.filterwarnings("ignore", category=FutureWarning)

# --------------------------------------------------------------------------
# WHERE THE SCIENTIFIC STACK CHANGED UNDER US
# --------------------------------------------------------------------------
# This is `ratatosk/compat.py`, and it lives in CELL 1 rather than in a
# definition cell of its own because it is part of setting the environment
# up: `check_environment()` is called at the bottom of this cell, and a name
# cannot be called in the cell above the one that defines it.
"""
Where the scientific stack changed under us, handled in one place.

Why this module exists
----------------------
`np.trapz` was REMOVED in numpy 2.0 and `np.trapezoid` does not exist before
it. The codebase had all three responses to that at once: nine direct calls to
`np.trapezoid` (numpy 2 only), a `trapz` alias imported unconditionally from
numpy (crashes on numpy 1), and a careful `getattr` fallback in two places —
one of which sat twenty-five lines AFTER the crashing import, so it could
never run.

The result was an undeclared hard dependency on numpy >= 2. On a numpy 1.26
Anaconda environment, which is still common, Cell 1 died with
`ImportError: cannot import name 'trapezoid'` and nothing in the notebook
explained why.

One name, defined once, used everywhere. Nothing else in the package should
reach for `np.trapz` or `np.trapezoid` directly.

The environment check
---------------------
`check_environment()` reports what is installed and warns where a version is
below what this pipeline was developed and tested against. It WARNS rather
than raising: a floor is a statement about what has been tested, not a
certainty that older will fail, and refusing to run on an untested version
helps nobody at 6pm. What it will not do is let a version problem surface
three cells later as an unrelated error.
"""


import os
import sys

import numpy as np


# `np.trapezoid` on numpy >= 2, `np.trapz` before it. Identical signature and
# identical results; only the name changed.
trapezoid = getattr(np, "trapezoid", None) or getattr(np, "trapz")

# What this pipeline has actually been run against. Not a hard requirement —
# see the module docstring — but the numbers a bug report should quote.
TESTED_AGAINST = {
    "numpy": "1.24",
    "scipy": "1.10",
    "pandas": "2.0",
    "matplotlib": "3.7",
    "lmfit": "1.2",
}
PYTHON_MINIMUM = (3, 9)


def _tuple(version):
    """'2.4.4rc1' -> (2, 4, 4). Anything unparseable sorts as newest."""
    out = []
    for part in str(version).split("."):
        digits = ""
        for ch in part:
            if ch.isdigit():
                digits += ch
            else:
                break
        if not digits:
            break
        out.append(int(digits))
    return tuple(out) if out else (999,)


def installed_versions():
    """`{package: version string}` for everything the pipeline imports."""
    import importlib
    found = {}
    for name in ("numpy", "scipy", "pandas", "matplotlib", "lmfit",
                 "seaborn", "joblib", "openpyxl", "python_calamine"):
        try:
            mod = importlib.import_module(name)
            found[name] = getattr(mod, "__version__", "present")
        except Exception:                              # noqa: BLE001
            found[name] = None
    return found


def numerical_backend():
    """
    The platform and the linear-algebra library the fits were solved with.

    WHY THIS IS PROVENANCE AND NOT TRIVIA. Two runs of the SAME build on the
    SAME file give the same answer to about six figures and not beyond it: the
    least-squares solver reaches a different local minimum depending on the
    order floating-point reductions happen in, which is set by the BLAS and by
    how many threads it uses. Measured on NMC111 — twelve to twenty-four free
    parameters per half-cycle, and a documented non-identifiability between
    overlapping components — one machine and another agreed on every cycling
    number, every reference voltage and every coherence verdict, and disagreed
    on `reliable` for nine of 164 components and on one drift rate by 12 mV
    per cycle. Everything the coherence audit certifies matched; everything
    that moved was already quarantined by it.

    That is the right outcome, but it is only checkable if the run says what
    it ran on. The manifest recorded Python and package versions, which are
    identical across those two machines and therefore explain nothing.
    """
    import platform as _pf
    out = dict(system=_pf.system(), release=_pf.release(),
               machine=_pf.machine(), processor=_pf.processor() or "",
               python_implementation=_pf.python_implementation())
    # The BLAS numpy is linked against. `__config__` has changed shape more
    # than once, so every access is guarded and a failure records that it
    # could not be read rather than dropping the field.
    try:
        import numpy as _np
        _cfg = getattr(_np, "__config__", None)
        _show = getattr(_cfg, "show", None)
        _blas = None
        if _show is not None:
            try:                                  # numpy >= 1.25
                _d = _show(mode="dicts")
                _b = (((_d or {}).get("Build Dependencies") or {})
                      .get("blas") or {})
                _blas = {k: _b.get(k) for k in ("name", "version")
                         if _b.get(k)} or None
            except Exception:                     # noqa: BLE001
                _blas = None
        if _blas is None:                         # older layouts
            for _attr in ("blas_opt_info", "blas_ilp64_opt_info"):
                _i = getattr(_cfg, _attr, None)
                if isinstance(_i, dict) and _i.get("libraries"):
                    _blas = {"name": ",".join(_i["libraries"])}
                    break
        out["blas"] = _blas or "not reported by numpy"
    except Exception:                             # noqa: BLE001
        out["blas"] = "could not be read"
    # Thread counts change the reduction ORDER even on one machine, so they
    # belong beside the library name.
    out["threads"] = {v: os.environ[v] for v in
                      ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS",
                       "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS")
                      if v in os.environ} or "not set (library default)"
    return out


def check_environment(verbose=True):
    """
    Say what is installed, and warn about anything below what was tested.

    Returns a dict the manifest can record verbatim, so a run's provenance
    carries the environment it was produced in rather than leaving it to be
    reconstructed later.
    """
    found = installed_versions()
    old = {n: v for n, v in found.items()
           if v and n in TESTED_AGAINST
           and _tuple(v) < _tuple(TESTED_AGAINST[n])}
    missing = [n for n in ("numpy", "scipy", "pandas", "matplotlib", "lmfit")
               if not found.get(n)]
    py = tuple(sys.version_info[:2])

    state = dict(python=".".join(str(x) for x in sys.version_info[:3]),
                 packages=found, below_tested=old, missing=missing,
                 python_below_minimum=py < PYTHON_MINIMUM,
                 # See `numerical_backend`. Package versions are identical
                 # across two machines that disagree in the sixth figure;
                 # this is the field that says why.
                 numerical_backend=numerical_backend(),
                 # Recorded so a run says whether warnings were being hidden
                 # while it ran. A suppressed FutureWarning is exactly the
                 # kind that changes a number between pandas releases.
                 future_warnings_suppressed=True)

    if verbose:
        print(f"  Python {state['python']}   "
              + "   ".join(f"{n} {found[n]}" for n in
                           ("numpy", "pandas", "scipy", "lmfit")
                           if found.get(n)))
        # SAID ON EVERY RUN, because it is the field that explains why two
        # machines running the same build on the same file agree to six
        # figures and not beyond. See `numerical_backend`.
        _nb = state["numerical_backend"]
        _bl = _nb.get("blas")
        _bl = (f"{_bl.get('name')} {_bl.get('version') or ''}".strip()
               if isinstance(_bl, dict) else str(_bl))
        print(f"  {_nb['system']} {_nb['machine']}   linear algebra: {_bl}")
        if found.get("python_calamine"):
            print("  calamine present — Excel reading is 5-7x faster")
        else:
            print("  calamine NOT present — `pip install python-calamine` "
                  "makes loading 5-7x faster")
        if not found.get("seaborn"):
            print("  seaborn NOT present — the key-cycle voltage profile "
                  "figure will fail; everything else is unaffected")
        if py < PYTHON_MINIMUM:
            print(f"  *** Python {state['python']} is below the "
                  f"{'.'.join(str(x) for x in PYTHON_MINIMUM)} this was "
                  f"written for. ***")
        for n in missing:
            print(f"  *** {n} is not installed — the run will fail. ***")
        for n, v in sorted(old.items()):
            print(f"  *** {n} {v} is older than the {TESTED_AGAINST[n]} this "
                  f"was tested against; results are not guaranteed. ***")
    return state
# --------------------------------------------------------------------------

print(f"Ratatosk {RATATOSK_VERSION}")
ENVIRONMENT = check_environment()



In [ ]:
# DEFINITIONS A0 — HOW THIS NOTEBOOK TALKS TO YOU
# =============================================================================
"""
Console presentation: colour, rules, and aligned fields.

Why this is a module and not a class in each file
-------------------------------------------------
1.8.7 styled exactly one cell — the parameter review — with a `_S` class local
to it, and the other twenty printed flat text. So a run looked designed for
thirty seconds and like a log file for the other ten minutes, and there was
nowhere to change that from. The palette now lives here, every module draws on
it, and the flat notebook emits this cell first so the bare names resolve.

The vocabulary is deliberately small and SEMANTIC. `HEADER`, `SECTION`,
`VALUE`, `WARN` say what a thing is; nothing anywhere else in the codebase
writes an escape code. That is what makes it possible to change the whole
run's appearance — or turn it off — in one place.

Turning it off
--------------
`set_colour(False)`, or the environment variable `NO_COLOR` (the de facto
standard), or any non-terminal destination that is not a notebook. Colour is
an aid to reading, never a carrier of meaning: every line says in words what
its colour says in hue, so a log file with the codes stripped loses nothing.
"""


import math
import os
import sys


RULE_WIDTH = 76


def _enable_windows_vt():
    """Ask a Windows console to interpret ANSI. Harmless everywhere else.

    Anaconda Prompt on Windows 10+ can render escape codes but does not by
    default; Jupyter and Windows Terminal already do. Without this the run
    prints its own escape codes as literal noise, which is worse than no
    colour at all — so a failure here turns colour OFF rather than hoping.
    """
    if os.name != "nt":
        return True
    try:
        import ctypes
        k = ctypes.windll.kernel32
        for handle in (-11, -12):                 # stdout, stderr
            h = k.GetStdHandle(handle)
            mode = ctypes.c_uint32()
            if not k.GetConsoleMode(h, ctypes.byref(mode)):
                continue
            k.SetConsoleMode(h, mode.value | 0x0004)   # VIRTUAL_TERMINAL
        return True
    except Exception:                              # noqa: BLE001
        return False


def supports_colour():
    """Whether escape codes will be rendered rather than printed."""
    if os.environ.get("NO_COLOR") is not None:
        return False
    if os.environ.get("FORCE_COLOR"):
        return True
    # A notebook's stdout is not a tty but does render ANSI.
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            return _enable_windows_vt()
    except Exception:                              # noqa: BLE001
        pass
    if not hasattr(sys.stdout, "isatty") or not sys.stdout.isatty():
        return False
    return _enable_windows_vt()


class Style:
    """The palette. Semantic names only — nothing here is called 'red'."""
    # Sheffield purple banner, from 1.8.7's parameter review.
    HEADER  = '\033[1;48;5;54;97m'
    SECTION = '\033[1;96m'          # bold cyan   — section labels
    VALUE   = '\033[92m'            # green       — confirmed values
    FILE    = '\033[38;5;117m'      # light blue  — values read from the file
    WARN    = '\033[1;91m'          # bold red    — warnings and errors
    CONFIRM = '\033[1;92m'          # bold green  — confirmed / accepted
    OK      = '\033[92m'            # green       — a measurement that passed
    CAUTION = '\033[93m'            # amber       — passed, with a caveat
    BAD     = '\033[91m'            # red         — did not pass
    DIM     = '\033[2m'             # de-emphasis — units, notes, provenance
    BOLD    = '\033[1m'
    RULE    = '\033[38;5;98m'       # muted purple — separators
    RESET   = '\033[0m'


class _NoStyle:
    """Every attribute is an empty string, so styled code needs no branches."""
    def __getattr__(self, _name):
        return ""


_COLOUR = supports_colour()
S = Style() if _COLOUR else _NoStyle()
# 1.8.7's name, kept so the parameter review reads unchanged.
_S = S


def set_colour(on):
    """Force colour on or off for the rest of the run."""
    global S, _S, _COLOUR
    _COLOUR = bool(on)
    S = Style() if _COLOUR else _NoStyle()
    _S = S
    return _COLOUR


# ---------------------------------------------------------------------------
# Layout. Every one of these returns a string; nothing here prints, so a
# caller can put a block in a report as easily as on the console.
# ---------------------------------------------------------------------------

# =============================================================================
# WHAT THE WORKING ION IS CALLED
# =============================================================================
# For a NEGATIVE electrode the step labels are swapped (see
# `io.apply_electrode_convention`) and the figures name the process rather
# than the direction: charge is the working ion going IN, discharge is it
# coming out. Naming it requires knowing which ion it is.
#
# Until 1.9.0.75 the two label helpers read `battery_chemistry` and fell
# through to lithium for anything that was not exactly 'Na-ion' — so an NTO
# anode cycled in a sodium half-cell was labelled "Lithiation" and
# "Delithiation" on every waterfall. `infer_chemistry` answers Unknown for an
# abbreviation ("NTO" is not a formula, and it is genuinely ambiguous between
# sodium titanate and a niobium titanium oxide), which is the right answer,
# and the label then guessed the commonest case in silence.
#
# The pipeline ALREADY KNOWS. `counter_electrode_metal` resolves the working
# ion in a recorded order — stated by the operator, inferred from the
# electrolyte salt, taken from the composition library, or assumed — and the
# interactive path ASKS when it is not established. That answer sits in
# `counter_electrode_metal` with its provenance in
# `counter_electrode_metal_source`, and the labels were reading a weaker
# field two seats along. One quantity, two places, the two disagreeing: the
# same shape of defect this build has now found five times.
WORKING_ION_WORDS = {
    "Li": ("Lithiation", "Delithiation"),
    "Na": ("Sodiation", "Desodiation"),
    "K":  ("Potassiation", "Depotassiation"),
    "Mg": ("Magnesiation", "Demagnesiation"),
    "Ca": ("Calciation", "Decalciation"),
    "Zn": ("Zincation", "Dezincation"),
    "Al": ("Aluminiation", "Dealuminiation"),
}

# Every capacity-column name the labels below can produce, for the readers
# that have to recognise one written by an earlier run. Generated from the
# table rather than typed out, because three separate hard-coded tuples
# (`analyse.cycle_column`, two in `report`) already listed Li and Na only and
# would have silently stopped matching the moment a third ion existed.
CAPACITY_COLUMN_ALIASES = tuple(
    ["Discharge_mAh_g"] + [f"{_d}_mAh_g" for _c, _d in WORKING_ION_WORDS.values()])


def working_ion(params):
    """
    The working ion's element symbol, or "" when it is not established.

    `counter_electrode_metal` FIRST: it is the resolved answer and it carries
    its own provenance, and an "assumed" one is a guess that must not reach a
    figure caption. `battery_chemistry` is the fallback for a params dict
    written before that field existed.
    """
    params = params or {}
    _src = str(params.get("counter_electrode_metal_source") or "").lower()
    _m = str(params.get("counter_electrode_metal") or "").strip().title()
    if _m in WORKING_ION_WORDS and _src and _src != "assumed":
        return _m
    _chem = str(params.get("battery_chemistry") or "").strip().lower()
    if _chem.startswith("na"):
        return "Na"
    if _chem.startswith("li"):
        return "Li"
    # A metal that is only ASSUMED is better than nothing for a mass estimate,
    # where the alternative is no number at all. It is not better than nothing
    # for a word printed on an axis, where the alternative is a neutral word
    # that is true whatever the ion. So it stops here.
    return ""


def half_cycle_labels(params):
    """
    `(charge_label, discharge_label)` for this dataset.

    Neutral "Charge"/"Discharge" for a positive electrode, and ALSO for a
    negative one whose working ion is not established — a caption that says
    "Charge" is merely unspecific, whereas one that says "Lithiation" over a
    sodium cell is wrong.
    """
    if not (params or {}).get("anode_labels_swapped", False):
        return "Charge", "Discharge"
    _ion = working_ion(params)
    if not _ion:
        return "Charge", "Discharge"
    return WORKING_ION_WORDS[_ion]


def rule(title="", width=RULE_WIDTH, char="─"):
    """A horizontal separator, optionally naming the section it opens."""
    if not title:
        return f"{S.RULE}{char * width}{S.RESET}"
    left = char * 2
    right = char * max(2, width - len(title) - 4)
    return f"{S.RULE}{left}{S.RESET} {S.BOLD}{title}{S.RESET} {S.RULE}{right}{S.RESET}"


def heading(text, width=RULE_WIDTH):
    """A banner for the thing being worked on — a dataset, a stage."""
    pad = f" {text} ".ljust(width)
    return f"{S.HEADER}{pad}{S.RESET}"


def section(text):
    """A labelled question or sub-stage inside a block."""
    return f"{S.SECTION}{text}{S.RESET}"


# How wide a three-column `entry` line may run before its note drops to a
# wrapped block underneath. Wider than RULE_WIDTH on purpose: a separator that
# ran this long would look like a fault, but a line of text at this width is
# comfortable in a notebook cell and in any terminal worth using.
ENTRY_LINE_WIDTH = 96


def entry(label, value, note="", label_width=22, value_width=16, indent=2):
    """One aligned `label   value   note` line.

    NOT called `field`: seven modules do `from dataclasses import field`, and
    in the flat notebook — one namespace — whichever ran last would win. The
    build guard now catches that class of collision, but a name this generic
    should not be spent on a formatting helper in the first place.

    The three columns are the whole point: a reader scanning for one number
    should never have to read a sentence to find it.

    THE NOTE COLUMN WRAPS. It used to run on to whatever length it was given,
    which put 130-character lines into a notebook cell that shows about 78 —
    so the end of the sentence was off the side of the screen and read by
    nobody, which is the same defect `bullet` was written to avoid. Overflow
    continues on the next line, aligned under the note, so the value column
    stays scannable.
    """
    import textwrap
    pad = " " * indent
    # A LABEL LONGER THAN ITS COLUMN must not run into the value. `f"{x:<22}"`
    # pads a short label and does nothing at all to a long one, so
    # "discharge across all cycles" and "median 1" printed as
    # "discharge across all cyclesmedian 1". Two spaces is the minimum
    # separation, and the value column simply starts later on that row.
    lab = f"{label:<{label_width}}"
    if len(label) >= label_width:
        lab = label + "  "
    val = f"{value:<{value_width}}" if note else str(value)
    out = f"{pad}{S.DIM}{lab}{S.RESET}{val}"
    if not note:
        return out.rstrip()
    left = indent + label_width + max(len(str(value)), value_width) + 2
    # Against ENTRY_LINE_WIDTH, not RULE_WIDTH. The rule is a deliberate 76
    # because a full-width separator any wider looks like a fault; a line of
    # text may run past it, and forcing three columns into 76 dropped notes
    # of 38 characters onto their own line for no reason.
    room = ENTRY_LINE_WIDTH - left
    if len(str(note)) <= room:
        return (out + f"  {S.DIM}{note}{S.RESET}").rstrip()
    # TOO LONG FOR THE THIRD COLUMN. Wrapping it there would leave a 34-column
    # gutter and break a sentence every four words, so it drops WHOLE onto
    # continuation lines under the label instead — the same block `bullet`
    # produces, and the value column stays scannable above it.
    block = "\n".join(f"{' ' * (indent + label_width)}{S.DIM}{ln}{S.RESET}"
                      for ln in textwrap.wrap(
                          str(note),
                          width=max(20, RULE_WIDTH - indent - label_width)))
    return (out.rstrip() + "\n" + block)


_MARK = {"ok": ("✓", "OK"), "caution": ("!", "CAUTION"), "bad": ("✗", "BAD")}


def verdict(level, text, label="verdict", label_width=22, indent=2):
    """The line a reader looks for first: did this pass, and what does it mean.

    `level` is 'ok', 'caution' or 'bad'. The glyph and the colour agree, and
    the text says it in words as well — so the line survives being pasted
    into an email, a log, or a colourblind reader's terminal.
    """
    import textwrap
    glyph, _ = _MARK.get(level, ("", ""))
    col = {"ok": S.OK, "caution": S.CAUTION, "bad": S.BAD}.get(level, "")
    pad = " " * indent
    lab = f"{label:<{label_width}}"
    # WRAPPED, like `entry`'s note. A verdict is the line a reader looks for
    # first and it was the one line that could run off the side of the cell.
    body = f"{glyph} {text}"
    room = max(20, ENTRY_LINE_WIDTH - indent - label_width)
    lines = textwrap.wrap(body, width=room) or [body]
    out = f"{pad}{S.DIM}{lab}{S.RESET}{col}{lines[0]}{S.RESET}"
    cont = " " * (indent + label_width)
    for extra in lines[1:]:
        out += f"\n{cont}{col}{extra}{S.RESET}"
    return out


def bullet(text, indent=4, label_width=22, width=RULE_WIDTH):
    """Continuation lines under a field or verdict, aligned to its value.

    WRAPPED at the rule width. An unwrapped explanation runs off the side of
    a notebook cell and is then read by nobody, which defeats the point of
    having written it.
    """
    import textwrap
    pad = " " * (indent + label_width - 2)
    lines = textwrap.wrap(str(text), width=max(20, width - len(pad))) or [""]
    return "\n".join(f"{pad}{S.DIM}{ln}{S.RESET}" for ln in lines)

def image_format(params, requested=None):
    """The image format the OPERATOR asked for, in Cell 3b.

    Lives here because BOTH figure modules need it and, defined in each, the
    flat notebook kept only whichever cell ran last — which is what the
    duplicate-definition build guard exists to catch, and did.

    Every plotting function took a `file_format='png'` argument and used it,
    but no notebook cell ever passed one and only three of `cycling.py`'s
    functions fell back to the per-dataset answer. So 42 of a run's figures
    came out PNG whatever was chosen. An explicit argument still wins.
    """
    if requested:
        return str(requested)
    return str((params or {}).get('file_format', 'png') or 'png')


def saved(path):
    """One 'saved' line naming the FILE, not reciting the path.

    Thirty-five of these across the two figure modules printed the absolute
    Windows path in full, once per figure. The run folder is printed once, by
    Cell 3b, and nothing else about the path changes between figures.
    """
    import os as _os
    print(entry("saved", _os.path.basename(str(path))))


def nice_axis_limit(vmax, target_ticks=6):
    """Round an axis maximum up to a round number, with a tick step to match.

    A common axis computed straight from the data lands on 143 mAh/g, and the
    reader is then left deciding whether a curve ending at 136 is meaningfully
    short of the frame or just short of an arbitrary number. Rounding to 150
    with ticks every 25 answers that by inspection: the gridline the curve
    stops near is a number a person would have chosen.

    Steps come from the 1 / 2 / 2.5 / 5 ladder, which is what produces tick
    labels like 25, 50, 75 rather than 23.8, 47.7. Returns
    `(limit, step)`, or `(vmax, None)` if there is nothing sensible to round.

    The limit is always strictly greater than `vmax`, so the largest data
    point never sits on the frame; a maximum that is already an exact
    multiple of the step is given one more step rather than being touched.
    """
    try:
        vmax = float(vmax)
    except (TypeError, ValueError):
        return vmax, None
    if not math.isfinite(vmax) or vmax <= 0:
        return vmax, None
    target_ticks = max(2, int(target_ticks))
    raw = vmax / target_ticks
    mag = 10.0 ** math.floor(math.log10(raw))
    step = 10.0 * mag
    for m in (1.0, 2.0, 2.5, 5.0):
        if raw <= m * mag:
            step = m * mag
            break
    limit = math.ceil(vmax / step) * step
    if limit <= vmax * (1.0 + 1e-9):
        limit += step
    return limit, step


In [ ]:
# DEFINITIONS A — READING A CYCLER EXPORT
# =============================================================================
"""
Reading a cycler export, and knowing what the columns mean.

The one thing this module exists to prevent
-------------------------------------------
For a negative electrode, 1.8.x swaps the Charge/Discharge labels so that
"Discharge" means the useful half-cycle for every dataset. That is right for
presentation and it is a trap for physics: in an LTO dataset the half-cycle
labelled "Charge" is the one where the **voltage falls**.

That trap was sprung. A cycle-integrity measure inferred the expected voltage
direction from the label, and consequently reported all 64 LTO half-cycles as
anomalous at 98-99% coulombic efficiency.

So `Dataset` keeps the swap for labelling, and additionally offers
`step_direction()`, which is **measured from the recorded voltage** and cannot
be fooled. Everything in `signal` and `analyse` uses the measured direction.
The label is for humans; the measurement is for arithmetic.
"""


import hashlib
import os
from dataclasses import dataclass, field, replace

import numpy as np
import pandas as pd


# Neware long names -> the short names used throughout. Anything not listed is
# carried through untouched: Current(A) in particular is needed by the
# constant-voltage detection and must survive unrenamed.
COLUMNS = {
    "Cycle Index": "Cycle",
    "Step Type": "Step",
    "Voltage(V)": "Voltage",
    "Chg. Spec. Cap.(mAh/g)": "Charge_Capacity",
    "DChg. Spec. Cap.(mAh/g)": "Discharge_Capacity",
    "dQm/dV(mAh/V.g)": "dQ/dV",
}
_NUMERIC = ("Cycle", "Voltage", "Charge_Capacity", "Discharge_Capacity",
            "dQ/dV")

_STEP_NAMES = {"CCCV Chg": "Charge", "CCCV CHG": "Charge",
               "CC Chg": "Charge", "CC CHG": "Charge",
               "CC DChg": "Discharge", "CC DCHG": "Discharge",
               "CCCV DChg": "Discharge"}


def file_sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


@dataclass
class Dataset:
    """
    One cell's data, in acquisition order, plus what is known about it.

    `frame` is never re-sorted. Acquisition order is the only order in which
    "what happened next" is answerable, and three of this project's defects
    came from code that sorted by voltage and then asked a time-ordered
    question.
    """
    name: str
    frame: pd.DataFrame
    meta: dict = field(default_factory=dict)
    electrode_type: str = "Positive"
    labels_swapped: bool = False
    source_path: str = ""
    source_sha256: str = ""

    # -- structure ---------------------------------------------------------

    # 1.8.7 Module 1 drops rows with a missing Voltage, Cycle, Step or dQ/dV
    # BEFORE anything else, and every half-cycle it processes comes from that
    # cleaned frame. Reproducing it matters for more than row counts: where
    # several records share a voltage, `sort_values` breaks the tie by input
    # order, so a frame containing extra rows sorts its duplicates differently
    # and the smoothed curve diverges. On the P3 cells 15 of 40 half-cycles
    # differed for exactly this reason until the dropna was matched.
    _REQUIRED = ("Voltage", "Cycle", "Step", "dQ/dV")

    # BOTH OF THESE ARE CACHED, and the cache is the difference between a
    # run you wait two minutes for and one you do not.
    #
    # `half_cycle` used to boolean-mask the WHOLE frame on every call:
    # `f[(f["Cycle"] == cycle) & (f["Step"] == step)]`, 82,337 rows scanned
    # to return 200 of them. Under pandas 3 the `Step == "Charge"` half of
    # that is a string comparison that null-checks the entire column, and it
    # dominates: 10.2 ms per call, with `_isna_string_dtype` alone taking
    # 5.4 s of a 13.4 s `termination_check`.
    #
    # Cells 4 and 5 make 3,444 such calls per dataset — 42 s, or 126 s for a
    # triplicate. ONE `groupby(["Cycle", "Step"])` produces all 440 groups in
    # 0.03 s, which is 168x faster than 440 masks, and every later call is a
    # dict lookup.
    #
    # Safe because `frame` is never re-sorted (see the class docstring) and
    # nothing mutates a half-cycle in place: `orient_dqdv` copies before
    # writing and `strip_cv_hold` only slices. `replace()` builds a NEW
    # Dataset, so a modified frame gets a fresh cache rather than a stale one.
    # The groups preserve the frame's own row order, so the stable-sort
    # tie-breaking that the port fidelity depends on is unchanged.

    @property
    def clean(self) -> pd.DataFrame:
        cached = self.__dict__.get("_clean_cache")
        if cached is None:
            cols = [c for c in self._REQUIRED if c in self.frame.columns]
            cached = self.frame.dropna(subset=cols)
            self.__dict__["_clean_cache"] = cached
        return cached

    @property
    def _half_cycles(self) -> dict:
        """`{(cycle, step): frame}` for the whole dataset, built once."""
        cached = self.__dict__.get("_half_cycle_cache")
        if cached is None:
            f = self.clean
            if not {"Cycle", "Step"} <= set(f.columns):
                cached = {}
            else:
                cached = {(int(c), str(s)): d for (c, s), d
                          in f.groupby(["Cycle", "Step"], sort=True)}
            self.__dict__["_half_cycle_cache"] = cached
        return cached

    def half_cycle_keys(self):
        """(cycle, step) pairs, in cycle order, Charge before Discharge."""
        return list(self._half_cycles)

    def half_cycle(self, cycle, step) -> pd.DataFrame:
        # An absent key returns an EMPTY FRAME WITH THE RIGHT COLUMNS, which
        # is what the boolean mask returned and what every caller's
        # `len(df) < n` and `col in df.columns` tests expect.
        return self._half_cycles.get((int(cycle), str(step)),
                                     self.clean.iloc[:0])

    # -- the part that matters ---------------------------------------------

    def step_direction(self, cycle, step) -> float:
        """
        +1 if the voltage rises through this half-cycle, -1 if it falls.

        MEASURED, never inferred from the label. Falls back to the label only
        when the net travel is too small to read (a pure constant-voltage
        step), and says so by returning the label's implication rather than
        pretending to know.
        """
        v = pd.to_numeric(self.half_cycle(cycle, step)["Voltage"],
                          errors="coerce").values
        v = v[np.isfinite(v)]
        if v.size >= 2:
            net = float(v[-1] - v[0])
            if abs(net) > 1e-4:                     # one voltage LSB
                return 1.0 if net > 0 else -1.0
        # The label fallback must respect the swap. On a negative electrode
        # the half-cycle LABELLED Charge is the one where the voltage falls,
        # so reading the label naively returned the wrong sign — and this
        # branch fires precisely on a pure constant-voltage hold, the one
        # place the measurement cannot correct it.
        implied = 1.0 if str(step).lower().startswith("c") else -1.0
        return -implied if self.labels_swapped else implied

    def capacity(self, cycle, step) -> float:
        """
        Charge passed in this half-cycle, from the cycler's own counter.

        The counter, not the integral of dQ/dV — those two are compared by
        `quality.integral_fidelity`, and using one to define the other would
        make that comparison meaningless.
        """
        col = ("Charge_Capacity" if str(step).lower().startswith("c")
               else "Discharge_Capacity")
        d = self.half_cycle(cycle, step)
        if col not in d.columns:
            return float("nan")
        q = pd.to_numeric(d[col], errors="coerce").values
        q = q[np.isfinite(q)]
        if q.size < 2:
            return float("nan")
        return float(np.nanmax(q) - np.nanmin(q))

    def record_spacing_mV(self, step=None):
        """
        Median and minimum non-zero voltage step between consecutive records.

        Reported because the rebinning decision should be keyed to this and is
        not: 1.8.x keys it to the profile class, so the P3 exports get no
        rebinning at all despite 4% of their consecutive records sitting at an
        identical voltage. Measured here, acted on in 2.0.
        """
        dv, zeros, total = [], 0, 0
        for cyc, st in self.half_cycle_keys():
            if step is not None and st != step:
                continue
            v = pd.to_numeric(self.half_cycle(cyc, st)["Voltage"],
                              errors="coerce").values
            v = v[np.isfinite(v)]
            if v.size < 50:
                continue
            d = np.abs(np.diff(v))
            zeros += int((d == 0).sum())
            total += d.size
            nz = d[d > 0]
            if nz.size:
                dv.append(nz)
        if not dv:
            return dict(median_mV=float("nan"), min_mV=float("nan"),
                        duplicate_fraction=float("nan"))
        allnz = np.concatenate(dv) * 1000.0
        return dict(median_mV=float(np.median(allnz)),
                    min_mV=float(allnz.min()),
                    duplicate_fraction=(zeros / total) if total else float("nan"))


# ---------------------------------------------------------------------------

def _tidy_steps(df):
    out = df.copy()
    if "Step Type" not in out.columns:
        return out
    out["Step Type"] = out["Step Type"].astype(str).str.strip()
    out["Step Type"] = out["Step Type"].replace(_STEP_NAMES)
    out = out[~out["Step Type"].str.contains("Rest", case=False, na=False)]
    return out.reset_index(drop=True)


def _apply_electrode_convention(df, electrode_type):
    """
    For a negative electrode, present the useful half-cycle as "Discharge".

    Swaps the Step labels, the capacity columns and the sign of dQ/dV, exactly
    as 1.8.x Cell 4 does, so that figures and tracked-peak tables stay
    comparable across the versions. `Dataset.step_direction` is unaffected —
    it reads the voltage.
    """
    if str(electrode_type).lower() != "negative":
        return df, False
    d = df.copy()
    if {"Charge_Capacity", "Discharge_Capacity"} <= set(d.columns):
        d = d.rename(columns={"Charge_Capacity": "_swap",
                              "Discharge_Capacity": "Charge_Capacity"})
        d = d.rename(columns={"_swap": "Discharge_Capacity"})
    for a, b in (("Chg. Cap.(Ah)", "DChg. Cap.(Ah)"),
                 ("Chg. Energy(Wh)", "DChg. Energy(Wh)"),
                 ("Chg. Spec. Energy(mWh/g)", "DChg. Spec. Energy(mWh/g)")):
        if a in d.columns and b in d.columns:
            d = d.rename(columns={a: "_swap", b: a}).rename(columns={"_swap": b})
    if "Step" in d.columns:
        d["Step"] = d["Step"].map({"Charge": "Discharge",
                                   "Discharge": "Charge"}).fillna(d["Step"])
    for c in ("dQ/dV", "dQ/dV(mAh/V)"):
        if c in d.columns:
            d[c] = -pd.to_numeric(d[c], errors="coerce")
    return d, True


def apply_electrode_convention(dataset, electrode_type):
    """
    Return a copy of `dataset` with the convention for `electrode_type` applied.

    1.8.7 loaded in Cell 2, asked for the electrode type in Cell 3, and applied
    the convention in Cell 4 — in that order, because the operator has to be
    asked before the answer can be used. `read_neware` can apply it directly
    when the type is already known; this is for the interactive path, where it
    is not.

    Applying it twice is a no-op ONLY for the same answer. Re-applying
    "Negative" to an already-swapped dataset returns it unchanged; re-stamping
    it as "Positive" raises, because that would leave the Step labels,
    capacity columns and dQ/dV sign inverted while the record claimed
    otherwise — silently mirrored data with nothing to notice it by.
    """
    if dataset.labels_swapped:
        if str(electrode_type).lower() == "negative":
            return replace(dataset, electrode_type=str(electrode_type))
        # Re-stamping a swapped dataset as Positive would leave its Step
        # labels, capacity columns and dQ/dV sign inverted while claiming
        # otherwise — silently wrong data with no way to notice.
        raise ValueError(
            f"{dataset.name!r} was loaded as a negative electrode and its "
            f"half-cycle labels are already swapped; it cannot be restamped "
            f"as {electrode_type!r}. Reload the file with the correct "
            f"electrode type.")
    if str(electrode_type).lower() != "negative":
        return replace(dataset, electrode_type=str(electrode_type))
    df, swapped = _apply_electrode_convention(dataset.frame, electrode_type)
    return replace(dataset, frame=df.reset_index(drop=True),
                   electrode_type=str(electrode_type), labels_swapped=swapped)


def truncate_cycles(dataset, max_cycle):
    """
    Cut a dataset at `max_cycle`, returning a new one. None leaves it alone.

    Truncation belongs HERE, to the Dataset, because the Dataset is what
    every later stage reads. 1.8.6 truncated a separate `electrochemical_data`
    dict inside the parameter collection, and 1.9.0's notebook then rebuilt
    that dict from `dataset.frame` two lines later — so the operator's answer
    to "Maximum cycle to analyse" was applied, discarded, and then asserted as
    fact by the console message, `dataset_info.txt` and the run manifest,
    while every capacity, retention and fade number was computed over the
    cycles they had asked to exclude.
    """
    if not max_cycle:
        return dataset
    f = dataset.frame
    if "Cycle" not in f.columns:
        return dataset
    keep = pd.to_numeric(f["Cycle"], errors="coerce") <= float(max_cycle)
    return replace(dataset, frame=f[keep].reset_index(drop=True))


def preferred_engine():
    """calamine if it is installed, openpyxl otherwise. Asked once per file."""
    try:
        import python_calamine  # noqa: F401
        return "calamine"
    except ImportError:
        return "openpyxl"


def open_workbook(path):
    """
    Open one export once, for both sheets that are read out of it.

    Returns `(ExcelFile, engine)`. The caller is responsible for closing it;
    `Dataset` holds no reference to it, so it can be closed as soon as both
    sheets have been read.

    Raises rather than returning None. A workbook that cannot be opened at
    all is a different failure from one that has no 'test' sheet, and the
    two were indistinguishable while both were caught by the same
    `except Exception` inside the metadata reader.
    """
    engine = preferred_engine()
    import warnings as _w
    with _w.catch_warnings():
        _w.filterwarnings("ignore", message=".*no default style.*")
        try:
            return pd.ExcelFile(path, engine=engine), engine
        except Exception as exc:
            if engine == "calamine":
                # calamine is the faster reader, not the more tolerant one.
                # Fall back before giving up on the file.
                try:
                    return pd.ExcelFile(path, engine="openpyxl"), "openpyxl"
                except Exception:
                    pass
            raise OSError(
                f"could not open {os.path.basename(path)} as an Excel "
                f"workbook ({engine}): {exc}") from exc


def read_neware(path, *, name=None, electrode_type="Positive", meta=None,
                engine=None, usecols=True, verbose=True, book=None):
    """
    Read one Neware .xlsx export into a `Dataset`.

    calamine is used when available: measured 5-7x faster than openpyxl on
    these files, with a maximum numeric difference of exactly 0.0.

    `book` is an already-open `pd.ExcelFile` for this same path. A Neware
    export has two sheets we want — 'record' and 'test' — and opening the
    workbook twice parses it twice. Measured on one 34 MB export: 3.02 s to
    read 'test' by path against 1.33 s through an open handle, 36% of the
    total load cost, because the by-path read also silently fell back to
    openpyxl while 'record' was being read by calamine. `path` is still
    required: it is what the SHA-256 and the provenance record refer to.
    """
    name = name or os.path.splitext(os.path.basename(path))[0]

    if engine is None:
        engine = preferred_engine()
    source = book if book is not None else path

    wanted = list(COLUMNS) + [
        "Chg. Cap.(Ah)", "DChg. Cap.(Ah)", "Chg. Energy(Wh)",
        "DChg. Energy(Wh)", "Chg. Spec. Energy(mWh/g)",
        "DChg. Spec. Energy(mWh/g)", "Current(A)", "Spec. Energy(mWh/g)",
        "Power(W)", "Time",
    ]
    picker = (lambda c: c in wanted) if usecols else None

    if verbose:
        print(f"  reading {name} ({engine}) ...", flush=True)
    # These exports carry no default cell style, and openpyxl warns about it
    # once per file — eight files, eight warnings burying the load messages.
    # Scoped, so reading a file does not permanently mute this warning for
    # everything else in the caller's process.
    import warnings as _w
    with _w.catch_warnings():
        _w.filterwarnings("ignore", message=".*no default style.*")
        # An open ExcelFile already knows its engine; passing `engine=` with
        # it is a TypeError on pandas, so it is only named for a path.
        _kw = {} if book is not None else {"engine": engine}
        try:
            raw = pd.read_excel(source, sheet_name="record", header=0,
                                usecols=picker, **_kw)
        except Exception:
            raw = pd.read_excel(source, sheet_name="record", header=0, **_kw)

    df = _tidy_steps(raw).rename(columns=COLUMNS)
    for c in _NUMERIC:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df, swapped = _apply_electrode_convention(df, electrode_type)

    ds = Dataset(name=name, frame=df.reset_index(drop=True),
                 meta=dict(meta or {}), electrode_type=electrode_type,
                 labels_swapped=swapped, source_path=str(path),
                 source_sha256=file_sha256(path))
    if verbose:
        n_cyc = int(df["Cycle"].nunique()) if "Cycle" in df else 0
        print(f"      {len(df):,} records, {n_cyc} cycles"
              + ("  [anode: labels swapped]" if swapped else ""))
    return ds


In [ ]:
# DEFINITIONS A2 — WHAT THE OPERATOR IS ASKED, AND WHAT THE FILE ALREADY KNOWS
# =============================================================================
"""
What the operator is asked, and what the cycler already knows.

This is 1.8.6's Cell 2 and Cell 3, restored. It was dropped when 1.9.0 replaced
them with a hardcoded `DATASETS` list, and that was a mistake: the prompts are
not a convenience layer, they are where a researcher's knowledge of the cell
enters the analysis, and the defaults behind them are read out of the file
rather than invented.

    extract_neware_metadata   read the workbook's 'test' sheet: active mass,
                              voltage limits, C-rate, currents, cycle count
    collect_parameters        ask for the rest, with those values pre-filled,
                              and record which came from the file and which
                              from a person

Why it prompts rather than takes a dict
---------------------------------------
Because the default shown is the answer in most cases, and seeing it is the
point. `_input_with_default` prints "[from file: 12.72]" against a mass read
out of the workbook and "[default: 12.0]" against one that is only a
convention, so the operator knows which numbers they are actually responsible
for. A dict hides that distinction, and a value nobody checked looks exactly
like a value somebody confirmed.

For a headless or scheduled run, pass `answers=` to `collect_parameters` — a
dict keyed by dataset name whose entries pre-answer any prompt. Anything not
answered there falls back to the metadata default without asking. Nothing is
ever silently invented: the manifest records the source of every value.

Estimated quantities are marked as estimates
--------------------------------------------
The counter-electrode mass is estimated from geometry and metal density when
the operator has not weighed it, and the record carries both the mass and the
metal, with `counter_electrode_measured` saying which it is.

NOT implemented, though earlier versions of this docstring said it was:
separator areal density and electrolyte density are neither looked up nor
recorded, and no `*_source` field exists for them. Their lookup tables were
written and never called, so they have been removed rather than left to read
as a feature. The energy and power densities in `cycling.power_and_energy`
use only the active mass, the blend fraction and the electrode area, so no
number currently depends on either quantity.
"""


import builtins as _builtins
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The final summary block prints display names. `_get_display_name` lives in
# `plots` because every figure needs it; importing it here rather than keeping
# a second copy is what stops the two drifting apart. `plots` does not import
# `params`, so there is no cycle.


# =============================================================================
# THE ANALYSIS KNOBS, DEFINED ONCE
# =============================================================================
# Both entry points — the interactive `collect_parameters` and the declarative
# `parameters_from_dict` — build the same record, and for a while they built it
# from two hand-written literal blocks. They diverged twice before anyone
# noticed: `anode_labels_swapped` existed on one path only, and the centre
# tolerance was 50 on one and 70 on the other, so the same dataset fitted
# different peaks depending on which door you came in by. Two independent
# derivations of one record will diverge; the only question is when.
#
# 'auto' means "let the measured profile decide" and is dropped by the
# override readers below. A number is an instruction and is kept.
ANALYSIS_DEFAULTS = {
    "smoothing_window": "auto",
    "polyorder": 3,
    "spike_removal": "auto",
    "spike_window_size": 5,
    "spike_threshold_multiplier": "auto",
    "rebin_width_mV": "auto",
    "second_smooth_window": "auto",
    # THE HISTOGRAM PATH'S OWN KNOBS. Everything above this line belongs to
    # the DERIVATIVE path, and since 1.9.0.14 the default is the histogram —
    # so every preprocessing override an operator could set was silently
    # inert, while the console still echoed "operator overrides: {...}" back
    # at them. These two are what the histogram actually reads.
    # 'auto' = take it from the measured profile, as everything else does.
    "histogram_bin_mV": "auto",
    "histogram_smooth_bins": "auto",
    "peak_prominence_fraction": 0.05,
    # 'auto' = let the profile decide, exactly as the smoothing and rebin
    # fields do. Two peaks 30 mV apart are one feature on a broad layered
    # oxide and two on a sharp two-phase material, so a fixed 30 here
    # silently overrode the profile's own answer. Put a number to force it.
    "peak_min_distance_mV": "auto",
    "peak_min_width_mV": 5,
    "reference_cycle": 2,
    "shoulder_detection": True,
    "edge_exclusion_mV": 50,
    "tracking_tolerance_mV": 80,
}

# Which reader claims which knob. Every key of ANALYSIS_DEFAULTS appears in
# exactly one of these, checked at import — a knob nobody reads is a knob that
# does nothing, and that is how the four deleted fit fields survived.
PREPROCESS_KEYS = ("smoothing_window", "polyorder", "spike_removal",
                   "spike_window_size", "spike_threshold_multiplier",
                   "rebin_width_mV", "second_smooth_window",
                   "histogram_bin_mV", "histogram_smooth_bins")

# Which dQ/dV path each preprocessing knob reaches. A knob set for the path
# that is not running does nothing, and Cell 4 says so rather than echoing it
# back as though it had been applied. The assert below proves every key is
# claimed by a reader; this says WHICH reader, which is the part that was
# actually wrong.
KEYS_BY_DQDV_METHOD = {
    "derivative": ("smoothing_window", "polyorder", "spike_removal",
                   "spike_window_size", "spike_threshold_multiplier",
                   "rebin_width_mV", "second_smooth_window"),
    "histogram": ("histogram_bin_mV", "histogram_smooth_bins"),
}
assert set(sum(KEYS_BY_DQDV_METHOD.values(), ())) == set(PREPROCESS_KEYS), \
    "a preprocessing knob belongs to no dQ/dV path"
DETECTION_KEYS = ("peak_min_distance_mV", "peak_prominence_fraction",
                  "peak_min_width_mV", "edge_exclusion_mV",
                  "shoulder_detection")
ANALYSIS_KEYS = ("reference_cycle", "tracking_tolerance_mV")
assert (set(PREPROCESS_KEYS) | set(DETECTION_KEYS) | set(ANALYSIS_KEYS)
        == set(ANALYSIS_DEFAULTS)), "an analysis knob has no reader"


def _explicit(params, keys):
    """The subset of `keys` the operator actually set. See ANALYSIS_DEFAULTS."""
    out = {}
    for k in keys:
        default = ANALYSIS_DEFAULTS[k]
        v = params.get(k, default)
        if v is None or str(v).lower() == "auto" or v == default:
            continue
        out[k] = v
    return out


def extract_neware_metadata(source):
    """
    Extract metadata from the 'test' sheet of a Neware xlsx file.

    Accepts either a filepath string or an already-opened
    pd.ExcelFile object. Passing an ExcelFile avoids re-parsing the
    workbook when the caller has already opened it for the 'record'
    sheet.

    Handles the Neware merged-cell layout where labels are in columns
    0, 3, 6 and values are at label_col + 2 (with an empty column
    between label and value due to merged cells in Excel).

    Falls back to label_col + 1 if +2 is empty, so also works with
    non-merged layouts from other cycler brands.
    """
    metadata = {
        'active_material_mass_mg': None,
        'voltage_upper_V': None,
        'voltage_lower_V': None,
        'cycle_count': None,
        'c_rate': None,
        'charge_current_A': None,
        'discharge_current_A': None,
        'barcode': None,
        'test_start_date': None,
    }

    # An absent 'test' sheet and an unreadable workbook were both swallowed
    # by one `except Exception`, so a corrupt file looked exactly like a
    # cycler that had not written its metadata: no prompt was pre-filled and
    # nothing said why. They are separated here, and neither is silent.
    if isinstance(source, pd.ExcelFile):
        if 'test' not in source.sheet_names:
            print(f"      no 'test' sheet in this workbook "
                  f"(sheets: {', '.join(map(str, source.sheet_names))}) "
                  f"— nothing to pre-fill from")
            return metadata
    try:
        # These exports carry no default cell style, and openpyxl warns about
        # it once per file. It is not a problem with the data and it buries
        # the loading messages, so it is silenced here rather than left to
        # scroll past eight times.
        import warnings as _w
        with _w.catch_warnings():
            _w.filterwarnings("ignore", message=".*no default style.*")
            df_test = pd.read_excel(source, sheet_name='test', header=None)
    except ValueError:
        # pandas raises ValueError for a sheet that is not in the workbook.
        print("      no 'test' sheet in this workbook — nothing to pre-fill")
        return metadata
    except Exception as exc:                                # noqa: BLE001
        print(f"      *** the 'test' sheet could not be read ({exc}); "
              f"every parameter will have to be typed ***")
        return metadata

    def _get(row, col):
        """Safely get a cell value, returning None for NaN/missing."""
        try:
            val = df_test.iloc[row, col]
            return None if pd.isna(val) else val
        except (IndexError, KeyError):
            return None

    def _parse_numeric(text):
        """Extract leading numeric value from '12.72mg', '4.3V', etc."""
        if text is None:
            return None
        match = re.match(r'^([+-]?\d*\.?\d+)', str(text).strip())
        return float(match.group(1)) if match else None

    n_rows = min(10, len(df_test))
    n_cols = min(15, len(df_test.columns))

    # --- Header section ---
    # Neware layout: label at col X, value at col X+2 (merged cell gap)
    # Fallback: try X+1 for non-Neware formats
    for row in range(n_rows):
        for label_col in range(n_cols - 2):
            label = _get(row, label_col)
            if label is None:
                continue
            ll = str(label).lower().strip()

            # Try +2 first (Neware merged cells), then +1
            value = _get(row, label_col + 2)
            if value is None:
                value = _get(row, label_col + 1)

            if 'active material' in ll:
                mass = _parse_numeric(value)
                if mass is not None and mass > 0:
                    metadata['active_material_mass_mg'] = mass

            elif 'volt' in ll and 'upper' in ll:
                v = _parse_numeric(value)
                if v is not None:
                    metadata['voltage_upper_V'] = v

            elif 'volt' in ll and 'lower' in ll:
                v = _parse_numeric(value)
                if v is not None:
                    metadata['voltage_lower_V'] = v

            elif 'cycle' in ll and 'count' in ll:
                try:
                    metadata['cycle_count'] = int(float(str(value)))
                except (ValueError, TypeError):
                    pass

            elif 'barcode' in ll:
                for offset in [2, 1, 3]:
                    bv = _get(row, label_col + offset)
                    if bv is not None and len(str(bv).strip()) > 3:
                        metadata['barcode'] = str(bv).strip()
                        break

            elif 'start time' in ll:
                for offset in [2, 1]:
                    sv = _get(row, label_col + offset)
                    if sv is not None:
                        metadata['test_start_date'] = str(sv).strip()
                        break

    # --- Step plan section ---
    # Find header row containing "Step Index"
    step_header_row = None
    for row in range(len(df_test)):
        for col in range(min(3, n_cols)):
            val = _get(row, col)
            if (val is not None and
                'step' in str(val).lower() and
                'index' in str(val).lower()):
                step_header_row = row
                break
        if step_header_row is not None:
            break

    if step_header_row is not None:
        # Read column headers from the step plan row
        headers = [str(_get(step_header_row, c) or '')
                   for c in range(n_cols)]

        # Find column indices — exclude "cut-off" variants
        crate_idx = None
        current_idx = None
        step_name_idx = None

        for i, h in enumerate(headers):
            hl = h.lower()
            if ('c-rate' in hl or 'c_rate' in hl) and 'cut' not in hl:
                crate_idx = i
            elif 'current' in hl and 'cut' not in hl:
                current_idx = i
            elif 'step name' in hl or 'step_name' in hl:
                step_name_idx = i

        if step_name_idx is not None:
            for row in range(step_header_row + 1, len(df_test)):
                sn = str(_get(row, step_name_idx) or '').lower()

                if any(t in sn for t in ['chg', 'charge']):
                    if crate_idx is not None:
                        try:
                            cr = float(_get(row, crate_idx))
                            if not np.isnan(cr):
                                metadata['c_rate'] = cr
                        except (ValueError, TypeError):
                            pass
                    if current_idx is not None:
                        try:
                            cu = float(_get(row, current_idx))
                            if not np.isnan(cu):
                                metadata['charge_current_A'] = cu
                        except (ValueError, TypeError):
                            pass

                elif any(t in sn for t in ['dchg', 'discharge']):
                    if crate_idx is not None and metadata['c_rate'] is None:
                        try:
                            cr = float(_get(row, crate_idx))
                            if not np.isnan(cr):
                                metadata['c_rate'] = cr
                        except (ValueError, TypeError):
                            pass
                    if current_idx is not None:
                        try:
                            cu = float(_get(row, current_idx))
                            if not np.isnan(cu):
                                metadata['discharge_current_A'] = cu
                        except (ValueError, TypeError):
                            pass

    return metadata


recommended_palettes = [
    "viridis_r",    # perceptually uniform, colourblind-safe (recommended)
    "Greys",        # greyscale light->dark
    "Greys_r",      # greyscale dark->light
    "plasma",       # perceptually uniform, vibrant
    "inferno",      # perceptually uniform, high contrast
    "magma",        # perceptually uniform, warm
    "cividis",      # perceptually uniform, colourblind-safe
    "Reds",         # sequential red
    "YlOrRd",       # yellow->orange->red
    "OrRd",         # orange->red
    "Blues",        # sequential blue
    "Paired",       # categorical pairs
    "Accent",       # categorical accent colours
    "rocket",       # seaborn sequential (dark->light)
    "mako",         # seaborn sequential (cool)
    "flare",        # seaborn sequential (warm)
    "crest",        # seaborn sequential (teal)
    "vlag",         # seaborn diverging (blue/red)
    "icefire",      # seaborn diverging (cool/warm)
    "PRGn",         # diverging purple/green
    "Spectral",     # diverging rainbow
]


default_palette = "viridis_r"


default_dqdv_charge_colour = '#0072B2'


default_dqdv_discharge_colour = '#D55E00'


default_dqdv_charge_linestyle = '-'


default_dqdv_discharge_linestyle = '-'


default_key_cycles = [1, 5, 10, 20, 30, 40, 50]
# `plots.DEFAULT_KEY_CYCLES` mirrors this (importing params into plots would
# be a cycle), so the two are checked against each other rather than trusted.
_plots_key_cycles = None
try:
    from .plots import DEFAULT_KEY_CYCLES as _plots_key_cycles
except Exception:                                    # pragma: no cover
    pass
if _plots_key_cycles is not None and list(_plots_key_cycles) != default_key_cycles:
    raise RuntimeError(
        "params.default_key_cycles and plots.DEFAULT_KEY_CYCLES disagree: "
        f"{default_key_cycles} vs {list(_plots_key_cycles)}")


default_electrode_diameter_mm = 12.0


_VALID_CMAPS = set(plt.colormaps())


_COUNTER_METAL_DENSITY = {
    'Li': 0.534,
    'Na': 0.971,
}


_DEFAULT_COUNTER_THICKNESS_MM = 0.5   # typical foil thickness


_DEFAULT_COUNTER_DIAMETER_MM = 16.0   # same as separator punch


def _validate_palette(name):
    """
    Return True if name is a valid matplotlib colormap.
    All downstream cells use plt.get_cmap(), so only matplotlib
    colormaps are valid — seaborn-only names will crash at runtime.
    """
    return name in _VALID_CMAPS


def _suggest_cell_id(name, all_names):
    """
    Best guess at which physical cell a dataset came from.

    Looks for a single letter delimited by an underscore OR a hyphen, then
    for the numeric token that varies across the loaded set. Returns '' if it
    cannot tell, which is honest rather than guessing wrongly.

    1.8.6 matched only `_A_`, so a hyphenated name like
    `JQ-P3-NaNiMnO2-35%-2-4.2V-0.1C-A-20062025` fell through to the numeric
    branch and suggested "3" — the varying digit of "35%" — for a cell called
    A. Both delimiters are now accepted, and a lone letter is preferred over
    any digit, because a cell identifier is a letter far more often than not.

    THE LAST DELIMITED LETTER, NOT THE FIRST. `JQ_NNM_C-rate_2-4.2V_A_29042026`
    has two: the C of "C-rate" and the A that names the cell. Taking the first
    labelled cell A as "cell C" — in a run of three cells that is a figure
    legend naming the wrong cell. A cell identifier sits at the end of these
    names, before the date, and everything earlier is protocol.

    An explicit `CellB` or `cell_B` wins outright, because that is somebody
    saying it rather than a convention being inferred. Without it
    `Nik_LTO-DD2_CellB` matched nothing at all and the dataset was labelled
    with its composition alone, so two single-cell runs of two different
    cells produced two identically named folders.
    """
    m = re.search(r'[Cc]ell[\s_-]?([A-Za-z0-9])(?![A-Za-z0-9])', name)
    if m:
        return m.group(1).upper()
    m = re.findall(r'[_-]([A-Za-z])[_-]', name)
    if m:
        return m[-1].upper()
    toks = {n: re.findall(r'\d+', n) for n in all_names}
    mine = toks.get(name, [])
    if mine and len(all_names) > 1:
        n_tok = min((len(t) for t in toks.values()), default=0)
        for pos in range(n_tok):
            if len({t[pos] for t in toks.values() if len(t) > pos}) > 1:
                return mine[pos]
    return ''


def _input_with_default(prompt, default_value, cast_type=str,
                         min_val=None, source=None):
    """Prompt for input with a pre-filled default shown in brackets."""
    # input() does not render ANSI codes — build a plain-text prompt.
    # Styling is applied only to print() calls (errors, confirmations).
    if default_value is not None:
        if source == 'file':
            src_tag = "[from file: "
        elif source == 'library':
            src_tag = "[from library: "
        else:
            src_tag = "[default: "
        full_prompt = f"{prompt} {src_tag}{default_value}]: "
    else:
        full_prompt = f"{prompt}: "
    while True:
        raw = _builtins.input(full_prompt).strip()
        if not raw:
            if default_value is not None:
                return cast_type(default_value) if cast_type != str else default_value
            else:
                print(f"  {_S.WARN}No default available — please enter a value.{_S.RESET}")
                continue
        try:
            value = cast_type(raw)
            if min_val is not None and value < min_val:
                print(f"  {_S.WARN}Value must be >= {min_val}.{_S.RESET}")
                continue
            return value
        except (ValueError, TypeError):
            print(f"  {_S.WARN}Invalid input. Please enter a {cast_type.__name__}.{_S.RESET}")


def _skip_module(module_name,
                 reason="RUN_PEAK_FITTING=False (set in Cell 3)"):
    """
    Print a clear skip message. Does NOT raise — the caller is
    expected to guard the module body with an if/else block so
    execution simply falls through to the next cell.

    Typical usage at the top of a module cell:

        if not RUN_PEAK_FITTING:
            _skip_module("Module 3")
        else:
            # existing module body, indented +4 spaces
    """
    print(f"{_S.WARN}--- {module_name} skipped ({reason}). ---{_S.RESET}")
    print(f"    To run this analysis now, set "
          f"RUN_PEAK_FITTING = True in Cell 3, re-run Cell 3,")
    print(f"    then run this module's cell manually. "
          f"No other cells need to re-run.")


# ---------------------------------------------------------------------------
# Specific capacity library
# ---------------------------------------------------------------------------
# A THEORETICAL CAPACITY IS ARITHMETIC, NOT AN OPINION. It is
# `n * F / M` — electrons per formula unit, Faraday's constant, molar mass —
# and typing it by hand at a prompt, once per cell, is a transcription step
# with nothing checking it. On a triplicate that is three chances to enter a
# different number for the same material, and the capacity divides every
# normalised figure the run produces.
#
# So the library stores `n` and `M` and DERIVES the capacity. The number
# cannot be a typo; it can only be wrong if the stoichiometry is wrong, and
# the stoichiometry is printed at the prompt so a wrong one is visible at the
# moment it is accepted rather than buried in a manifest.
#
# It is a DEFAULT, not a constraint. Every value can be overridden by typing
# one, exactly as before, and a composition that is not in the library gets
# no default at all — which is the current behaviour and the right one. A
# library that guessed would be worse than no library.
#
# F = 96485.33212 C/mol / 3600 s/h = 26801.48 mAh/mol.
FARADAY_mAh_PER_MOL = 96485.33212 / 3.6

# Molar masses computed from IUPAC 2021 conventional atomic weights:
#   Li 6.94  Na 22.98977  Ti 47.867  Mn 54.93804  Ni 58.6934
#   Co 58.93319  O 15.999
#
#   LiCoO2               M =  97.871   n = 1     -> 273.8
#   LiMn2O4              M = 180.812   n = 1     -> 148.2
#   LiNi1/3Mn1/3Co1/3O2  M =  96.460   n = 1     -> 277.9
#   LiNi0.8Mn0.1Co0.1O2  M =  97.280   n = 1     -> 275.5
#   LiCoMnO4             M = 184.807   n = 1     -> 145.0
#   Li4Ti5O12            M = 459.083   n = 3     -> 175.1
#   Na0.67Ni0.33Mn0.67O2 M = 103.578   n = 0.67  -> 173.4
#
# `n` is the FULL theoretical extraction, which for several of these is not
# the number a cell delivers in its working window: LiCoO2 is cycled over
# about half its lithium, and Li4Ti5O12 inserts three lithium into the
# 8a/16c sites without touching the rest. That is the definition the field
# quotes and the one a % -of-theoretical figure is read against, so it is the
# one stored — but it is why the note is printed with it.
#
# NNM IS THE UNDOPED PARENT, AND THAT IS THE POINT OF PRINTING THE FORMULA.
# The library entry is Na0.67Ni0.33Mn0.67O2 -> 173.4, which is the nominal P3
# composition. The cells this project actually cycles are DOPED, and their
# capacity is 168.0 — a property of a specific substituted composition that
# no library can derive and none should try to. So the default is the parent
# arithmetic, the prompt says which formula it came from, and an operator
# working on a doped sample types their own number over it, which the
# manifest then records as an override rather than as a library value.
#
# (Na2/3Ni1/3Mn2/3O2 is a DIFFERENT entry, M = 103.514 and 172.6 mAh/g, and
# is deliberately not aliased to this one. The 0.67 spelling is the one used
# here because it is the one written on the samples.)
SPECIFIC_CAPACITY_LIBRARY = {
    "LiCoO2": dict(
        n=1.0, M=97.871, ion="Li",
        note="1 Li per formula unit (full extraction to CoO2)",
        aliases=("lco", "licoo2", "lithiumcobaltoxide", "lithiumcobaltate")),
    "LiMn2O4": dict(
        n=1.0, M=180.812, ion="Li",
        note="1 Li per formula unit (spinel, 4 V plateau)",
        aliases=("lmo", "limn2o4", "lithiummanganeseoxide",
                 "lithiummanganatespinel")),
    "LiNi1/3Mn1/3Co1/3O2": dict(
        n=1.0, M=96.460, ion="Li",
        note="1 Li per formula unit (full delithiation)",
        aliases=("nmc111", "ncm111", "mnc111", "lini13mn13co13o2",
                 "lini033mn033co033o2", "lini0333mn0333co0333o2")),
    "LiNi0.8Mn0.1Co0.1O2": dict(
        n=1.0, M=97.280, ion="Li",
        note="1 Li per formula unit (full delithiation)",
        aliases=("nmc811", "ncm811", "mnc811", "lini08mn01co01o2")),
    "LiCoMnO4": dict(
        n=1.0, M=184.807, ion="Li",
        note="1 Li per formula unit (5 V spinel)",
        aliases=("licomno4", "lcmo", "limncoo4")),
    "Li4Ti5O12": dict(
        n=3.0, M=459.083, ion="Li",
        note="3 Li per formula unit (8a -> 16c, Ti4+/Ti3+)",
        aliases=("lto", "li4ti5o12", "lithiumtitanate",
                 "lithiumtitaniumoxide")),
    "Na0.67Ni0.33Mn0.67O2": dict(
        n=0.67, M=103.578, ion="Na",
        note="0.67 Na per formula unit, UNDOPED parent P3 composition — "
             "a doped sample is not this number",
        # NOT `Na2/3Ni1/3Mn2/3O2`: that stoichiometry is M = 103.514 and
        # 172.6 mAh/g, and answering it with 173.4 under a formula the
        # operator did not write is exactly the near-match this library
        # refuses everywhere else. It gets no default until it is entered
        # here with its own arithmetic.
        aliases=("nnm", "na067ni033mn067o2")),
}


def _capacity_key(text):
    """Normalise a composition for library lookup.

    Case, spaces, hyphens, brackets and the separators people put in formulae
    are not chemistry. `NMC-111`, `nmc 111` and `NMC111` are one material;
    so are `Li4Ti5O12` and `li4ti5o12`.
    """
    return re.sub(r"[\s\-_,.()\[\]/·]", "", str(text or "")).lower()


def lookup_specific_capacity(composition):
    """
    `(capacity_mAh_g, canonical_formula, note)` for a known material, else
    `(None, None, None)`.

    The capacity is DERIVED from the stored `n` and `M` and rounded to one
    decimal place. Matching is on the normalised composition against the
    canonical formula and its aliases — nothing is inferred from a partial
    match, because "NMC" with no digits is not a stoichiometry and a library
    that guessed 111 would put a number on a cell nobody chose it for.
    """
    key = _capacity_key(composition)
    if not key:
        return (None, None, None)
    for formula, rec in SPECIFIC_CAPACITY_LIBRARY.items():
        if key == _capacity_key(formula) or key in rec["aliases"]:
            cap = round(rec["n"] * FARADAY_mAh_PER_MOL / rec["M"], 1)
            return (cap, formula, rec["note"])
    return (None, None, None)


def infer_chemistry(composition, electrolyte=""):
    """
    Li-ion, Na-ion or Unknown, from the electrolyte salt first.

    The ELECTROLYTE decides it, because that is where the working ion
    actually comes from; the composition is only a fallback. A sodium
    half-cell with a hard-carbon or LTO working electrode has no Na in its
    formula at all, and the declarative path — which tested
    `"Na" in composition` and nothing else — called every one of them
    Li-ion. That picks the counter-electrode metal and its density (Li 0.534
    vs Na 0.97 g/cm3, a 1.8x mass error) and the areal-capacity benchmark
    lines, so the two entry points produced different energy densities from
    the same cell.
    """
    e = str(electrolyte or "").lower()
    c = str(composition or "").lower()
    if "napf6" in e or "naclo4" in e or "natfsi" in e:
        return "Na-ion"
    if "lipf6" in e or "litfsi" in e or "libf4" in e or "liclo4" in e:
        return "Li-ion"
    # The composition fallback matches the ELEMENT SYMBOL, not a prefix: Na
    # or Li followed by anything that is not a lower-case letter, so NaNiMnO2
    # and Li4Ti5O12 match while Nb2O5 and a stray "lithiated" do not. A
    # prefix test missed every formula where the alkali is not written first,
    # and `"Na" in composition` matched things like "NaN".
    comp = str(composition or "")
    if re.search(r"Na(?![a-z])", comp):
        return "Na-ion"
    if re.search(r"Li(?![a-z])", comp):
        return "Li-ion"
    # An abbreviation such as LTO or NMC names no element, so say so rather
    # than guess: the answer sets the counter-electrode metal.
    return "Unknown"


# WHICH METAL IS THE COUNTER ELECTRODE MADE OF?
# `infer_chemistry` deliberately answers `Unknown` for a composition that names
# no element, because an abbreviation is not a formula and the electrolyte is
# what decides the working ion. That is right. What was wrong is what happened
# next: the estimator fell through to `metal = 'Na'` for every Unknown, with
# nothing said.
#
# Every abbreviation this project actually uses is Unknown — LTO, NMC, NNM —
# so a lithium half-cell cycled against lithium metal was reported as
# "~98 mg (Na metal, estimated)" on all four LTO cells today. Li is
# 0.534 g/cm3 and Na is 0.97: a 1.8x mass error, and the counter mass sets the
# areal and gravimetric energy benchmarks.
#
# So the metal is now decided in this order, and the ORDER IS RECORDED:
#   stated      the operator said so
#   chemistry   inferred from the electrolyte salt, or from the formula
#   library     the composition is in SPECIFIC_CAPACITY_LIBRARY, whose entries
#               carry the working ion (LTO -> Li). A weaker claim than the
#               electrolyte, because a lithium-containing material can be
#               cycled in a sodium cell — but far better than a blanket guess
#   assumed     none of the above. The interactive path asks rather than
#               reaching here; the declarative path lands here and the value
#               is labelled `assumed` everywhere it is printed
COUNTER_METAL_DEFAULT = "Na"


def counter_electrode_metal(params_or_composition, *, stated=None,
                            battery_chemistry=None, electrolyte=""):
    """`(metal_symbol, source)` — see COUNTER_METAL_DEFAULT."""
    if stated:
        return str(stated).strip().title()[:2], "stated"
    chem = battery_chemistry or infer_chemistry(
        params_or_composition if isinstance(params_or_composition, str)
        else (params_or_composition or {}).get("composition"), electrolyte)
    if chem == "Li-ion":
        return "Li", "chemistry"
    if chem == "Na-ion":
        return "Na", "chemistry"
    comp = (params_or_composition if isinstance(params_or_composition, str)
            else (params_or_composition or {}).get("composition"))
    key = _capacity_key(comp)
    for formula, rec in SPECIFIC_CAPACITY_LIBRARY.items():
        if key and (key == _capacity_key(formula) or key in rec["aliases"]):
            return rec.get("ion", COUNTER_METAL_DEFAULT), "library"
    return COUNTER_METAL_DEFAULT, "assumed"


def _estimate_counter_electrode_mass_mg(diameter_mm=None,
                                         thickness_mm=None,
                                         battery_chemistry=None,
                                         metal=None):
    """
    Estimate counter electrode mass from geometry and metal density.
    Returns (mass_mg, metal_symbol). `metal` overrides the inference.
    """
    if diameter_mm is None:
        diameter_mm = _DEFAULT_COUNTER_DIAMETER_MM
    if thickness_mm is None:
        thickness_mm = _DEFAULT_COUNTER_THICKNESS_MM

    # Determine counter electrode metal
    if not metal:
        if battery_chemistry == 'Li-ion':
            metal = 'Li'
        elif battery_chemistry == 'Na-ion':
            metal = 'Na'
        else:
            metal = COUNTER_METAL_DEFAULT

    density = _COUNTER_METAL_DENSITY.get(metal, 0.534)
    radius_cm = diameter_mm / 2 / 10
    area_cm2 = np.pi * radius_cm**2
    volume_cm3 = area_cm2 * (thickness_mm / 10)
    mass_mg = volume_cm3 * density * 1000
    return mass_mg, metal


def collect_parameters(electrochemical_data, cycler_metadata=None,
                       answers=None, headless=False, verbose=True):
    """
    Ask for each dataset's parameters, pre-filled from the cycler's own file.

    Returns `(user_parameters, run_peak_fitting)`. Ported from 1.8.6 Cell 3
    without editing the prompt sequence; the only change is that its
    notebook-global state (`_last_composition`, `_POWER_ENERGY_GLOBAL` and the
    rest) are locals here, which is what they always should have been.

    `answers` pre-answers prompts, matched on the start of the prompt text:
    `{"NNM_cellA": {"Theoretical capacity": 168}}`. `headless=True` accepts
    every default without asking. Both exist for scheduled runs and tests; the
    interactive path is the one a researcher should use.
    """
    cycler_metadata = cycler_metadata or {}
    _ANSWERS = dict(answers or {})
    _CURRENT = {"name": None}
    _last_electrode_type = None
    _asked = {}

    # `input` is shadowed UNCONDITIONALLY. Defining it inside `if headless:`
    # makes the name local to this whole function anyway, so on the interactive
    # path every raw `input()` call would raise UnboundLocalError before
    # reaching a person.
    def input(prompt=""):                       # noqa: A001
        text = str(prompt).lower().strip()
        pre = _ANSWERS.get(_CURRENT["name"], {})
        for k, v in pre.items():
            if text.startswith(str(k).lower().strip()):
                if verbose:
                    print(f"{prompt}{v}   [answered]")
                return str(v)
        if not headless:
            return _builtins.input(prompt)
        # Only the confirmation prompt is auto-confirmed. Answering "y" to
        # every (yes/no) question made a headless run silently opt IN to
        # full peak fitting, whose own prompt says "leave blank for no";
        # a blank lets each prompt's own default decide, which is what the
        # docstring promises.
        reply = "y" if "correct?" in text else ""
        _asked[text] = _asked.get(text, 0) + 1
        if _asked[text] > 3:
            raise ValueError(
                f"headless run is stuck on {_CURRENT['name']!r}: {prompt!r} "
                f"rejects a blank answer and has no default. Supply it in "
                f"`answers`.")
        if verbose:
            print(f"{prompt}{reply}   [auto]")
        return reply

    def _ask(prompt, default_value, cast_type=str, min_val=None, source=None):
        """`_input_with_default`, answerable for a headless run."""
        # PREFIX FIRST, THEN CONTAINMENT — longest key wins.
        # The match was `prompt.startswith(key)` alone, so the obvious key for
        # a headless run, `"Composition"`, silently matched NOTHING: the
        # prompt is "Active material composition (e.g., LiCoO2)". While the
        # composition question used a bare `input()` that merely hung; since
        # 1.9.0.62 it goes through here, and a scripted run died on it with
        # "has no default from the cycler file" — which is true and is not
        # the problem. Containment is what a caller means by a key, and
        # taking the LONGEST match keeps "Theoretical capacity" from being
        # captured by a bare "capacity".
        pre = _ANSWERS.get(_CURRENT["name"], {})
        _pl = str(prompt).lower().strip()
        _cands = [(k, v) for k, v in pre.items()
                  if _pl.startswith(str(k).lower().strip())]
        if not _cands:
            _cands = [(k, v) for k, v in pre.items()
                      if str(k).lower().strip() in _pl]
        hit = (max(_cands, key=lambda kv: len(str(kv[0])))[1]
               if _cands else None)
        if hit is not None:
            if verbose:
                print(f"  {prompt}: {hit}   [answered]")
            return cast_type(hit) if cast_type is not str else hit
        if headless:
            if default_value is None:
                raise ValueError(
                    f"headless run needs an answer for {_CURRENT['name']!r}: "
                    f"{prompt!r} has no default from the cycler file.")
            if verbose:
                print(f"  {prompt}: {default_value}   [default accepted]")
            return (cast_type(default_value) if cast_type is not str
                    else default_value)
        return _input_with_default(prompt, default_value, cast_type,
                                   min_val, source)

    user_parameters = {}
    _last_composition = None  # carries forward between datasets
    _last_theoretical_capacity = None  # carries forward for same composition

    # Pipeline-level flag for Cells 8-12 (peak detection, multi-peak
    # fitting, tracking, derived quantities, diagnostics, export). Asked
    # once during the first dataset's inputs, immediately after the
    # power-analysis question. None = not yet asked; True/False = user
    # answered. Cells 8-12 each guard their body with
    # `if not RUN_PEAK_FITTING: _skip_module("Module N")`.
    RUN_PEAK_FITTING = None
    _POWER_ENERGY_GLOBAL = None
    _FILE_FORMAT_GLOBAL = None
    _ELECTROLYTE_COMP_GLOBAL = None
    # The cycle cut-off carries forward, like the electrolyte above it.
    # Datasets analysed together are usually replicates of one experiment, so a
    # cap set on the first is almost always wanted on the rest — and when it is
    # NOT applied consistently the comparative figures quietly compare
    # different cycle ranges, which is how the NNM triplicate came to be
    # analysed to cycle 80 in one build and 75 in the next. None means nothing
    # is being carried, and then the prompt is exactly as it was.
    _MAX_CYCLE_CARRIED = None

    for name, df in electrochemical_data.items():
        _CURRENT["name"] = name
        # 1.8.7 guarded this with `if 'cycler_metadata' in globals()` because
        # Cell 2 might not have run. Here it is a PARAMETER, so the guard is
        # always False and every metadata default silently vanishes — which
        # leaves prompts with no default sitting in a `while True` that
        # rejects a blank answer.
        meta = cycler_metadata.get(name, {}) or {}
        print(f"\n{_S.HEADER}{'='*60}")
        print(f"  Setting parameters for: {name}")
        if any(v is not None for v in meta.values()):
            print(f"  (Values marked [from file] were read from the cycler)")
        print(f"{'='*60}{_S.RESET}")
        confirmed = False
        while not confirmed:
            # ----- MATERIAL -----
            print(f"\n{_S.SECTION}--- Material ---{_S.RESET}")
            # A BLANK COMPOSITION IS NOT A MISSING LABEL, IT CHANGES THE
            # MODEL. `quality.reconcile_mechanisms` groups cells by
            # composition and reconciles a group to its most permissive
            # mechanism, because a material either delivers charge across a
            # composition window or it does not. Its fallback for an unknown
            # composition is the dataset's own name, which is unique, so the
            # cell forms a group of one and keeps whatever its own reference
            # cycle called it.
            #
            # Measured, NNM 1.9.0.61, 2026-09-09: cell A's own call is
            # `multi_transition`, whose model has `band_width_max = 0.0`.
            # Reconciled with cells B and C it is fitted as `mixed` and gets
            # 215 bands; left alone it gets NONE, and R2 falls 0.9930 ->
            # 0.9314 across 160 half-cycles. The only difference between the
            # two runs was an empty answer here.
            #
            # So this prompt no longer accepts one. `_ask` already refuses a
            # blank when there is no default, carries the previous cell's
            # answer when there is one, and is answerable for a headless run.
            composition = str(_ask(
                "Active material composition (e.g., LiCoO2)",
                _last_composition, cast_type=str,
                source='previous cell' if _last_composition else None
            )).strip()
            # ----- CELL IDENTIFIER -----
            # Which physical cell this is. Previously inferred from the
            # filename at display time, only when two datasets shared a
            # composition, and never stored. Load one cell alone and it had no
            # identity at all, which is how one label ends up on two different
            # cells in different figures. Asked when several cells are loaded;
            # otherwise the parsed value is stored silently so a record exists.
            _cid_suggest = _suggest_cell_id(name, list(electrochemical_data))
            if len(electrochemical_data) > 1:
                while True:
                    _cid_in = input(
                        f"Cell identifier (A, B, 1, 2 ...) "
                        f"[{_cid_suggest or 'none'}]: ").strip().upper()
                    cell_id = _cid_in or _cid_suggest
                    if not cell_id:
                        print(f"  {_S.WARN}Enter an identifier. Without one you "
                              f"cannot tell which cell a figure came from."
                              f"{_S.RESET}")
                        continue
                    _clash = [n for n, p in user_parameters.items()
                              if str(p.get('cell_id', '')).upper() == cell_id]
                    if _clash:
                        print(f"  {_S.WARN}'{cell_id}' is already used by "
                              f"{_clash[0]}.{_S.RESET}")
                        print(f"  {_S.WARN}Two datasets labelled as the same "
                              f"physical cell is exactly the ambiguity this "
                              f"field prevents.{_S.RESET}")
                        continue
                    break
            else:
                cell_id = _cid_suggest
            while True:
                blend_input = input(
                    "Electrode blend ratio "
                    "(e.g., 80/10/10, leave blank for 80/10/10): "
                )
                blend = blend_input if blend_input else "80/10/10"
                try:
                    blend_values = [float(p) for p in blend.split('/')]
                    if (len(blend_values) >= 2
                            and all(v >= 0 for v in blend_values)
                            and sum(blend_values) > 0):
                        break
                    else:
                        print(f"  {_S.WARN}Need at least 2 components, all non-negative.{_S.RESET}")
                except ValueError:
                    print(f"  {_S.WARN}Invalid format. Use numbers separated by '/'.{_S.RESET}")
            active_material_mass_mg = _ask(
                "Mass of active material (mg)",
                meta.get('active_material_mass_mg'),
                cast_type=float, min_val=0.001,
                source='file' if meta.get('active_material_mass_mg') else None
            )
            # Carry forward theoretical capacity if composition unchanged.
            # The previous cell's answer wins over the library, because it is
            # what THIS OPERATOR chose for THIS material in THIS run — an
            # override typed once should not have to be typed again.
            _theo_default = (_last_theoretical_capacity
                             if (_last_theoretical_capacity is not None
                                 and _last_composition is not None
                                 and composition == _last_composition)
                             else None)
            _theo_source = 'previous cell' if _theo_default else None
            _lib_formula = None
            if _theo_default is None:
                _lib_cap, _lib_formula, _lib_note = \
                    lookup_specific_capacity(composition)
                if _lib_cap is not None:
                    # PRINT THE STOICHIOMETRY THE NUMBER CAME FROM. A default
                    # is accepted by pressing return, so the assumption behind
                    # it has to be visible at that moment and not in a
                    # manifest read afterwards.
                    print(f"  {_S.VALUE}Library: {_lib_formula} -> "
                          f"{_lib_cap} mAh/g{_S.RESET}  ({_lib_note})")
                    _theo_default = _lib_cap
                    _theo_source = 'library'
            theoretical_capacity_mAh_g = _ask(
                "Theoretical capacity (mAh/g)",
                _theo_default,
                cast_type=float, min_val=0.001,
                source=_theo_source
            )
            # Provenance, not decoration: a number that came from the library
            # and a number somebody chose are different claims, and only one
            # of them is evidence about this electrode.
            if (_theo_source == 'library' and _theo_default is not None
                    and float(theoretical_capacity_mAh_g)
                    != float(_theo_default)):
                _theo_source = 'entered (library offered '\
                               f'{_theo_default})'
            elif _theo_source is None:
                _theo_source = 'entered'
            _theo_provenance = _theo_source
            _theo_library_formula = _lib_formula
            # ----- ELECTRODE GEOMETRY -----
            print(f"\n{_S.SECTION}--- Electrode geometry ---{_S.RESET}")
            electrode_diameter_mm = _ask(
                "Electrode punch diameter (mm)",
                default_electrode_diameter_mm,
                cast_type=float, min_val=0.1,
                source=None
            )
            # v1.8: auto-detect electrode type from Neware protocol.
            # After Cell 2 processing, Rest rows are removed and Step is
            # renamed to 'Charge'/'Discharge'. Anode protocols start with
            # Discharge (drive voltage down to insert Li/Na). Cathode
            # protocols start with Charge (extract Li/Na from as-made
            # cathode). Also carries forward from previous dataset.
            # (1.8.6 guarded this with `'_last_electrode_type' not in
            # globals()`. It is a LOCAL, assigned above and below, so the
            # guard was always true and reset the carry-forward on every
            # dataset. Removed; the initialisation at the top of the
            # function is the one that counts.)

            # Detect polarity from the protocol every time. Carry-forward is
            # only a fallback when detection is inconclusive, otherwise the
            # first dataset in a mixed load sets the default for all of them.
            _detected_electrode = None
            _first_step = None
            try:
                if 'Step' in df.columns and not df.empty:
                    _first_step = str(df['Step'].iloc[0]).strip()
                    if _first_step == 'Discharge':
                        _detected_electrode = "Negative"
                    elif _first_step == 'Charge':
                        _detected_electrode = "Positive"
            except Exception:
                pass

            _suggested_electrode = (_detected_electrode
                                    or _last_electrode_type
                                    or "Positive")

            if _detected_electrode is not None:
                print(f"  Protocol starts with {_first_step} — "
                      f"suggesting {_detected_electrode} electrode")

            _suggest_str = _suggested_electrode
            while True:
                electrode_type_input = input(
                    f"Positive (cathode) or negative (anode) electrode? "
                    f"(p/n, leave blank for {_suggest_str}): "
                ).lower()
                if not electrode_type_input:
                    electrode_type = _suggested_electrode
                    break
                elif electrode_type_input in ["positive", "pos", "p"]:
                    electrode_type = "Positive"
                    break
                elif electrode_type_input in ["negative", "neg", "n"]:
                    electrode_type = "Negative"
                    break
                else:
                    print(f"  {_S.WARN}Enter 'positive'/'p' or 'negative'/'n'.{_S.RESET}")

            _last_electrode_type = electrode_type
            # ----- PIPELINE-LEVEL: POWER/ENERGY (asked once) -----
            if _POWER_ENERGY_GLOBAL is None:
                pe_input = input(
                    "Run power and energy density analysis (Cell 6c)? "
                    "(yes/no, leave blank for yes): "
                ).lower()
                _POWER_ENERGY_GLOBAL = pe_input not in ["no", "n"]
            else:
                _pe_state = (f"{_S.CONFIRM}ON{_S.RESET}" if _POWER_ENERGY_GLOBAL
                             else f"{_S.WARN}SKIP{_S.RESET}")
                print(f"Power/energy (Cell 6c): {_pe_state} "
                      f"{_S.FILE}(set on first dataset){_S.RESET}")
            power_energy_analysis = _POWER_ENERGY_GLOBAL

            # ----- PIPELINE-LEVEL: dQ/dV PEAK FITTING (asked once) -----
            # Cell 7 (dQ/dV curve plotting) runs regardless and is fast.
            # Cells 8-12 (peak detection, multi-peak fitting, tracking,
            # derived quantities, fit diagnostics, single-cycle inspector,
            # export) are slow and only needed when Cell 7 reveals
            # features worth quantifying. Asked once on the first dataset;
            # the answer applies to the whole notebook run. Can be changed
            # by re-running Cell 3.
            if RUN_PEAK_FITTING is None:
                # Blank means YES. dQ/dV peak fitting is what this tool is
                # FOR — everything from Cell 9 to Cell 13 depends on it, and
                # a blank default of "no" meant a full run could complete
                # with no fitted peaks, no tracking, no polarisation and no
                # capacity attribution, having asked once, quietly, twenty
                # prompts earlier. Answer 'no' for a quick cycling-only pass.
                while True:
                    _pk_input = input(
                        "Run full dQ/dV peak fitting (Cells 8-12)? "
                        "Slower, and the point of the tool. "
                        "(yes/no, blank = YES): "
                    ).strip().lower()
                    if _pk_input in ('n', 'no'):
                        RUN_PEAK_FITTING = False
                        break
                    if _pk_input in ('', 'y', 'yes'):
                        RUN_PEAK_FITTING = True
                        break
                    print(f"  {_S.WARN}Enter 'yes' or 'no'.{_S.RESET}")
            else:
                # Second and subsequent datasets: show the existing choice
                # for reassurance, without asking again.
                _pk_state = (f"{_S.CONFIRM}ON{_S.RESET}" if RUN_PEAK_FITTING
                             else f"{_S.WARN}SKIP{_S.RESET}")
                print(f"dQ/dV peak fitting (Cells 8-12): {_pk_state} "
                      f"{_S.FILE}(set on first dataset){_S.RESET}")

            # ----- CYCLING CONDITIONS -----
            print(f"\n{_S.SECTION}--- Cycling conditions ---{_S.RESET}")
            # ASKING WHAT THE C-RATE WAS MAKES NO SENSE WHEN THE FILE RECORDS
            # THE CURRENT. The cycler's header states one rate — its FIRST
            # step — and on a rate-capability run that is a value the
            # operator is being asked to confirm for a protocol it does not
            # describe. `cycling.rate_protocol` reads the recorded current
            # and answers the question from the data, so:
            #
            #   several rates   the prompt is SKIPPED. There is no single
            #                   answer to give, and inviting one produced
            #                   "0.05 C" on every caption of a run that
            #                   ramped C/20 to 5C. The reference block's rate
            #                   is kept for the theoretical-current line and
            #                   the protocol travels on the parameters.
            #   one rate        the MEASURED rate is the default, in place of
            #                   the header value, which describes only the
            #                   first step even when every step matches.
            #   no current      exactly as before.
            #
            # Answered here rather than in Cell 6a because every caption
            # between the two would otherwise still be quoting the header.
            _proto = rate_protocol(df, dict(
                active_material_mass_mg=active_material_mass_mg,
                theoretical_capacity_mAh_g=theoretical_capacity_mAh_g))
            _rate_meas = None
            if _proto.get('available') and _proto.get('blocks'):
                _rate_meas = _proto['blocks'][0].get('c_rate')
            if _proto.get('is_variable'):
                print(f"  {_S.VALUE}This cell was cycled at "
                      f"{_proto['n_blocks']} different rates "
                      f"({_proto['label']}){_S.RESET} — read from the "
                      f"recorded current, so you are not asked for one.")
                for _b in _proto['blocks']:
                    print(f"      {_b['label']:>6}  cycles "
                          f"{_b['first_cycle']}-{_b['last_cycle']}")
                if _proto.get('transition_cycles'):
                    print(f"      rate-transition cycle(s) in no block: "
                          + ", ".join(str(c)
                                      for c in _proto['transition_cycles']))
                if _proto.get('returns_to_start'):
                    print(f"      returns to {_proto['blocks'][0]['label']} "
                          f"at the end — recovery will be reported")
                charge_rate_c = float(snap_crate(_rate_meas)) if _rate_meas else (
                    meta.get('c_rate') or 0.1)
            else:
                charge_rate_c = _ask(
                    "C-rate",
                    # `extract_neware_metadata` always returns the key, set to
                    # None when the file does not state a rate, so `.get`'s
                    # default was unreachable and the prompt had no default at
                    # all — which a headless run cannot answer.
                    (round(float(snap_crate(_rate_meas)), 4)
                     if _rate_meas else (meta.get('c_rate') or 0.1)),
                    cast_type=float, min_val=0.001,
                    source=('measured' if _rate_meas else
                            ('file' if meta.get('c_rate') else None))
                )
            # ----- VOLTAGE WINDOW -----
            # The file's 'Volt. upper' and 'Volt. lower' are the channel safety
            # limits, not the protocol. An LTO cell cycled 1.2-2.5 V reports
            # 0-3.25 V, which is the range the cycler was permitted to use.
            #
            # Derive the tested window from the data instead: the median of the
            # per-cycle minimum and maximum. The first cycle usually starts from
            # open circuit and reads high, and an aborted final cycle reads
            # short; a median across cycles ignores both without needing to
            # identify either. Rest rows were already dropped in Cell 2.
            # Percentiles were tried and are too aggressive, because the cell
            # spends most of its time on the plateau.
            #
            # A SHORT RECORD IS NOT A REASON TO QUOTE THE CHANNEL LIMITS. The
            # median needs three cycles to ignore the first (which starts from
            # open circuit) and the last (which may be cut short), and below
            # that the fallback was the file's safety limits — which on
            # `JQ_NNM_C-rate_2-4.2V_A_29042026`, a two-cycle export of a cell
            # cycled 2-4.2 V, put "1.0--4.6 V" on the run and every caption in
            # it. The filename says otherwise and so does the data.
            #
            # With one or two cycles the OBSERVED RANGE is still a fact: the
            # cell demonstrably reached those voltages, so the protocol window
            # is AT LEAST that wide. It is reported as exactly that — a lower
            # bound, not the protocol — which beats a pair of numbers that are
            # certainly wrong in the other direction.
            _v_meas_lo = _v_meas_hi = None
            _v_source = None
            try:
                if {'Cycle', 'Voltage'} <= set(df.columns) and not df.empty:
                    _pc = df.groupby('Cycle')['Voltage'].agg(['min', 'max'])
                    if len(_pc) >= 3:
                        _v_meas_lo = round(float(_pc['min'].median()), 2)
                        _v_meas_hi = round(float(_pc['max'].median()), 2)
                        _v_source = 'measured from data'
                    elif len(_pc) >= 1:
                        _v_meas_lo = round(float(_pc['min'].min()), 2)
                        _v_meas_hi = round(float(_pc['max'].max()), 2)
                        _v_source = 'observed range (short record)'
            except Exception:
                pass

            _v_file_lo = meta.get('voltage_lower_V')
            _v_file_hi = meta.get('voltage_upper_V')

            if _v_meas_lo is not None:
                voltage_lower_V, voltage_upper_V = _v_meas_lo, _v_meas_hi
                voltage_window_source = _v_source
                if _v_source == 'measured from data':
                    print(f"  {_S.FILE}Voltage window measured from data: "
                          f"{voltage_lower_V}--{voltage_upper_V} V{_S.RESET}")
                else:
                    print(f"  {_S.WARN}Only "
                          f"{len(_pc)} cycle(s) — too few to measure the "
                          f"protocol window from the per-cycle medians. Using "
                          f"the range the cell was OBSERVED to reach, "
                          f"{voltage_lower_V}--{voltage_upper_V} V: the "
                          f"protocol window is at least this wide.{_S.RESET}")
                if (_v_file_lo is not None and _v_file_hi is not None
                        and (abs(float(_v_file_lo) - voltage_lower_V) > 0.05
                             or abs(float(_v_file_hi) - voltage_upper_V) > 0.05)):
                    print(f"  {_S.FILE}(file reports "
                          f"{_v_file_lo}--{_v_file_hi} V, which are the channel "
                          f"limits rather than the protocol){_S.RESET}")
            else:
                voltage_lower_V, voltage_upper_V = _v_file_lo, _v_file_hi
                voltage_window_source = 'channel limits from file'
                if voltage_lower_V is not None:
                    print(f"  {_S.WARN}Voltage window taken from the file's "
                          f"channel limits ({voltage_lower_V}--"
                          f"{voltage_upper_V} V). Too few cycles to measure the "
                          f"tested window, so this may be wider than the "
                          f"protocol.{_S.RESET}")
            while True:
                key_cycles_str = input(
                    f"Key cycles to plot "
                    f"(comma-separated, leave blank for "
                    f"{', '.join(map(str, default_key_cycles))}): "
                )
                if not key_cycles_str:
                    # A copy. Storing the module-level list itself made every
                    # dataset share one object, so an in-place edit anywhere
                    # rewrote the default for the whole session.
                    key_cycles = list(default_key_cycles)
                    break
                try:
                    key_cycles = [int(c.strip())
                                  for c in key_cycles_str.split(",")]
                    break
                except ValueError:
                    print(f"  {_S.WARN}Enter comma-separated integers.{_S.RESET}")
                
            # ----- OPTIONAL CYCLE CUT-OFF (v1.8.5) -----
            # Cap analysis at a chosen cycle for this dataset. Applied to the
            # loaded data table at the end of Cell 3, so every downstream cell
            # sees only cycles 1..max_cycle with no other code changes. Blank
            # keeps all cycles. Use this to trim post-failure 'bad' data from a
            # faulty cell once its reliable range is known from the full run.
            _cyc_hint = (f" (file reports {meta['cycle_count']} cycles)"
                         if meta.get('cycle_count') else "")
            # CARRIED FORWARD, AND OVERRIDABLE — the same shape as the
            # electrolyte question below. Blank keeps what the previous dataset
            # used; a number changes it; 'all' removes the cap for this dataset
            # only. Blank had to keep meaning something, and "keep the cap" is
            # the answer wanted far more often than "drop it", so 'all' is the
            # word that drops it and it is named in the prompt rather than
            # left to be guessed.
            if _MAX_CYCLE_CARRIED is not None:
                print(f"  Current: {_S.VALUE}cycles 1-{_MAX_CYCLE_CARRIED}"
                      f"{_S.RESET} {_S.FILE}(carried from the previous "
                      f"dataset){_S.RESET}")
                _maxcyc_prompt = (
                    f"Maximum cycle to analyse for this dataset{_cyc_hint} "
                    f"— blank keeps {_MAX_CYCLE_CARRIED}, a number changes "
                    f"it, 'all' removes the cap: ")
            else:
                _maxcyc_prompt = (
                    f"Maximum cycle to analyse for this dataset{_cyc_hint} "
                    f"(leave blank for all cycles): ")
            while True:
                _maxcyc_str = input(_maxcyc_prompt).strip()
                if not _maxcyc_str:
                    # None on the first dataset, which is the old behaviour.
                    max_cycle = _MAX_CYCLE_CARRIED
                    break
                if _maxcyc_str.lower() in ("all", "none"):
                    max_cycle = None
                    break
                try:
                    max_cycle = int(_maxcyc_str)
                    if max_cycle >= 1:
                        break
                    print(f"  {_S.WARN}Enter a cycle number >= 1, "
                          f"'all' for no cap, or leave blank.{_S.RESET}")
                except ValueError:
                    print(f"  {_S.WARN}Enter a whole number, 'all' for no "
                          f"cap, or leave blank.{_S.RESET}")
            # Carry whatever this dataset settled on, including 'all': if the
            # cap is dropped here it stays dropped for the next one, and the
            # prompt goes back to its original form.
            _MAX_CYCLE_CARRIED = max_cycle
            # ----- CELL ASSEMBLY -----
            print(f"\n{_S.SECTION}--- Cell assembly (for methods section) ---{_S.RESET}")
            separator_type = input(
                "Separator type (leave blank for GF/6): "
            ) or "SLS Select GF/6 glass microfibre (SLS4522)"
            while True:
                elyte_vol_input = input(
                    "Electrolyte volume in uL (leave blank for 100): "
                )
                if not elyte_vol_input:
                    electrolyte_volume_uL = 100.0
                    break
                try:
                    electrolyte_volume_uL = float(elyte_vol_input)
                    if electrolyte_volume_uL >= 0:
                        break
                    print(f"  {_S.WARN}Volume must be non-negative.{_S.RESET}")
                except ValueError:
                    print(f"  {_S.WARN}Please enter a number.{_S.RESET}")
            print(f"\n{_S.SECTION}Electrolyte composition:{_S.RESET}")
            if _ELECTROLYTE_COMP_GLOBAL is not None:
                print(f"  Current: {_S.VALUE}{_ELECTROLYTE_COMP_GLOBAL}{_S.RESET} "
                      f"{_S.FILE}(set on first dataset){_S.RESET}")
                _elyte_override = input(
                    "Press Enter to keep, or enter 1-4 to change: "
                ).strip()
                if not _elyte_override:
                    electrolyte_composition = _ELECTROLYTE_COMP_GLOBAL
                else:
                    elyte_choice = _elyte_override
                    if elyte_choice == '1':
                        electrolyte_composition = "1M LiPF6 in EC:DMC 1:1 v/v"
                    elif elyte_choice == '2':
                        electrolyte_composition = "1M NaPF6 in EC:DMC 1:1 v/v"
                    elif elyte_choice == '3':
                        electrolyte_composition = "1M NaPF6 in EC:PC 1:1 v/v"
                    elif elyte_choice == '4':
                        electrolyte_composition = input(
                            "Enter electrolyte composition: "
                        ) or "Not specified"
                    else:
                        print(f"  {_S.WARN}Invalid choice, keeping previous.{_S.RESET}")
                        electrolyte_composition = _ELECTROLYTE_COMP_GLOBAL
                    _ELECTROLYTE_COMP_GLOBAL = electrolyte_composition
            else:
                print(f"  1. 1M LiPF6 in EC:DMC 1:1 v/v")
                print(f"  2. 1M NaPF6 in EC:DMC 1:1 v/v")
                print(f"  3. 1M NaPF6 in EC:PC 1:1 v/v")
                print(f"  4. Other")
                print(f"  {_S.FILE}(leave blank to skip){_S.RESET}")
                while True:
                    elyte_choice = input("Enter 1, 2, 3, 4, or blank: ").strip()
                    if not elyte_choice:
                        electrolyte_composition = "Not specified"
                        break
                    elif elyte_choice == '1':
                        electrolyte_composition = "1M LiPF6 in EC:DMC 1:1 v/v"
                        break
                    elif elyte_choice == '2':
                        electrolyte_composition = "1M NaPF6 in EC:DMC 1:1 v/v"
                        break
                    elif elyte_choice == '3':
                        electrolyte_composition = "1M NaPF6 in EC:PC 1:1 v/v"
                        break
                    elif elyte_choice == '4':
                        electrolyte_composition = input(
                            "Enter electrolyte composition: "
                        ) or "Not specified"
                        break
                    else:
                        print(f"  {_S.WARN}Enter 1, 2, 3, 4, or leave blank.{_S.RESET}")
                _ELECTROLYTE_COMP_GLOBAL = electrolyte_composition
            # ----- COUNTER ELECTRODE -----
            # ASK RATHER THAN ASSUME. `infer_chemistry` returns Unknown for an
            # abbreviation, and the estimate then used to fall through to
            # sodium in silence. One question, only when nothing better is
            # available, and the library's working ion is the default.
            # `battery_chemistry` is not assigned until the parameters are
            # built, well below this; `counter_electrode_metal` does the same
            # inference itself from the composition and the electrolyte, both
            # of which are known by here.
            _ce_metal, _ce_metal_src = counter_electrode_metal(
                composition, electrolyte=electrolyte_composition)
            if _ce_metal_src in ("library", "assumed"):
                _why = ("from the composition library — a lithium material can "
                        "still be cycled in a sodium cell, so confirm it"
                        if _ce_metal_src == "library" else
                        "a guess: neither the electrolyte nor the composition "
                        "names a working ion")
                print(f"  {_S.WARN}Counter electrode metal not established "
                      f"({_why}).{_S.RESET}")
                while True:
                    _m = input(f"Counter electrode metal (Li or Na) "
                               f"[{_ce_metal}]: ").strip().title()
                    if not _m:
                        _ce_metal_src = ("library" if _ce_metal_src == "library"
                                         else "assumed")
                        break
                    if _m in _COUNTER_METAL_DENSITY:
                        _ce_metal, _ce_metal_src = _m, "stated"
                        break
                    print(f"  {_S.WARN}Enter Li or Na, or leave blank for "
                          f"{_ce_metal}.{_S.RESET}")
            while True:
                ce_input = input(
                    "Counter electrode mass in mg "
                    "(leave blank for estimate from geometry): "
                )
                if not ce_input:
                    counter_electrode_mass_mg = None
                    counter_electrode_measured = False
                    break
                try:
                    counter_electrode_mass_mg = float(ce_input)
                    if counter_electrode_mass_mg > 0:
                        counter_electrode_measured = True
                        break
                    print(f"  {_S.WARN}Mass must be positive.{_S.RESET}")
                except ValueError:
                    print(f"  {_S.WARN}Please enter a number.{_S.RESET}")
            # ----- OUTPUT PREFERENCES -----
            print(f"\n{_S.SECTION}--- Output preferences ---{_S.RESET}")
            print(f"Recommended colour palettes "
                  f"{_S.FILE}(all are valid matplotlib colormaps){_S.RESET}:")
            for i, p in enumerate(recommended_palettes):
                print(f"  {i+1:>2}. {p}")
            print(f"\n  {_S.WARN}Note: seaborn-only names (bright, deep, dark, hls, husl,")
            print(f"  Greys_d, etc.) are not valid here — they will crash at Cell 5.{_S.RESET}")
            while True:
                palette_choice = input(
                    f"\nEnter number or colormap name "
                    f"(leave blank for '{default_palette}'): "
                ).strip()
                if not palette_choice:
                    colour_palette = default_palette
                    print(f"  {_S.CONFIRM}Colour palette: {colour_palette} ✓{_S.RESET}")
                    break
                try:
                    idx = int(palette_choice) - 1
                    if 0 <= idx < len(recommended_palettes):
                        colour_palette = recommended_palettes[idx]
                        print(f"  {_S.CONFIRM}Colour palette: {colour_palette} ✓{_S.RESET}")
                        break
                    print(f"  {_S.WARN}Enter a number between 1 and "
                          f"{len(recommended_palettes)}.{_S.RESET}")
                    continue
                except ValueError:
                    pass
                if _validate_palette(palette_choice):
                    colour_palette = palette_choice
                    print(f"  {_S.CONFIRM}Colour palette: {colour_palette} ✓{_S.RESET}")
                    break
                else:
                    _suggestions = sorted([
                        c for c in _VALID_CMAPS
                        if palette_choice.lower().replace('_d', '').replace('_r', '')
                        in c.lower()
                    ])[:6]
                    print(f"  {_S.WARN}✗ '{palette_choice}' is not a valid matplotlib colormap.{_S.RESET}")
                    if _suggestions:
                        print(f"    Did you mean: {_S.VALUE}{', '.join(_suggestions)}{_S.RESET}?")
                    else:
                        print(f"    Check spelling, or choose a number from the list above.")
            if _FILE_FORMAT_GLOBAL is None:
                while True:
                    file_format = input(
                        "Plot file format (png or tiff, leave blank for png): "
                    ).lower()
                    if file_format in ["png", "tiff", ""]:
                        file_format = file_format or "png"
                        _FILE_FORMAT_GLOBAL = file_format
                        break
                    print(f"  {_S.WARN}Enter 'png' or 'tiff'.{_S.RESET}")
            else:
                file_format = _FILE_FORMAT_GLOBAL
                print(f"File format: {_S.VALUE}{file_format}{_S.RESET} "
                      f"{_S.FILE}(set on first dataset){_S.RESET}")
            # =====================================================================
            # C. COMPUTED VALUES
            # =====================================================================
            # blend_values already validated during input above
            active_fraction = blend_values[0] / sum(blend_values)
            electrode_radius_cm = electrode_diameter_mm / 2 / 10
            electrode_area_cm2 = np.pi * electrode_radius_cm**2
            active_material_mass_g = active_material_mass_mg / 1000
            total_electrode_coating_mg = active_material_mass_mg / active_fraction
            active_loading_mg_cm2 = active_material_mass_mg / electrode_area_cm2
            total_loading_mg_cm2 = total_electrode_coating_mg / electrode_area_cm2
            theoretical_capacity_mAh = (active_material_mass_g *
                                        theoretical_capacity_mAh_g)
            theoretical_charge_current_mA = (charge_rate_c *
                                             theoretical_capacity_mAh)
            theoretical_charge_rate_mAh_g = (theoretical_charge_current_mA /
                                             active_material_mass_g)

            # Infer battery chemistry from electrolyte and composition (v1.7.2)
            # Used by Cell 6c for areal capacity benchmarks and Cell 14 NAMCC
            # for counter electrode metal selection.
            battery_chemistry = infer_chemistry(composition,
                                                electrolyte_composition)
            # =====================================================================
            # BUILD PARAMETERS DICTIONARY
            # =====================================================================
            # The metal, whether or not the counter electrode was weighed.
            # "97.6 mg of sodium" is a different claim from "97.6 mg", and
            # the declarative path already recorded it.
            _ce_est_mg, _ce_metal = _estimate_counter_electrode_mass_mg(
                battery_chemistry=battery_chemistry, metal=_ce_metal)

            sample_parameters = {
                "composition": composition,
                "cell_id": cell_id,
                "blend": blend,
                "active_material_mass_mg": active_material_mass_mg,
                "theoretical_capacity_mAh_g": theoretical_capacity_mAh_g,
                "theoretical_capacity_source": _theo_provenance,
                "theoretical_capacity_library_formula": _theo_library_formula,
                "electrode_type": electrode_type,
                # Read by cycling.py and plots.py to relabel the half-cycles.
                # Its ABSENCE here — the declarative path has always set it —
                # meant an operator who answered "negative" still got figures
                # and CSV columns labelled Charge/Discharge.
                "anode_labels_swapped": (str(electrode_type).lower()
                                         == "negative"),
                "electrode_diameter_mm": electrode_diameter_mm,
                "electrode_area_cm2": electrode_area_cm2,
                "active_fraction": active_fraction,
                "total_electrode_coating_mg": total_electrode_coating_mg,
                "active_loading_mg_cm2": active_loading_mg_cm2,
                "total_loading_mg_cm2": total_loading_mg_cm2,
                "power_energy_analysis": power_energy_analysis,
                "charge_rate_c": charge_rate_c,
                "key_cycles": key_cycles,
                "max_cycle": max_cycle,
                "theoretical_capacity_mAh": theoretical_capacity_mAh,
                "theoretical_charge_current_mA": theoretical_charge_current_mA,
                "theoretical_charge_rate_mAh_g": theoretical_charge_rate_mAh_g,
                "voltage_upper_V": voltage_upper_V,
                "voltage_lower_V": voltage_lower_V,
                "voltage_window_source": voltage_window_source,
                "voltage_limit_upper_V_from_file": _v_file_hi,
                "voltage_limit_lower_V_from_file": _v_file_lo,
                "separator_type": separator_type,
                "electrolyte_volume_uL": electrolyte_volume_uL,
                "electrolyte_composition": electrolyte_composition,
                "counter_electrode_mass_mg": (counter_electrode_mass_mg
                                              if counter_electrode_measured
                                              else _ce_est_mg),
                "counter_electrode_metal": _ce_metal,
                "counter_electrode_metal_source": _ce_metal_src,
                "counter_electrode_measured": counter_electrode_measured,
                "battery_chemistry": battery_chemistry,
                "barcode": meta.get('barcode'),
                "test_start_date": meta.get('test_start_date'),
                "charge_current_A_from_file": meta.get('charge_current_A'),
                "discharge_current_A_from_file": meta.get('discharge_current_A'),
                "colour_palette": colour_palette,
                "file_format": file_format,
                # The analysis knobs, from the one table. See
                # ANALYSIS_DEFAULTS for why this is not a literal block.
                **ANALYSIS_DEFAULTS,
            }
            # =====================================================================
            # REVIEW
            # =====================================================================
            mass_src = f" {_S.FILE}[from file]{_S.RESET}" if meta.get('active_material_mass_mg') else ""
            # SAY WHERE THE NUMBER CAME FROM, not where a number was
            # available. Marking a measured rate "[from file]" credits the
            # header for an answer the current gave.
            if _proto.get('is_variable'):
                rate_src = (f" {_S.FILE}[reference block of "
                            f"{_proto['n_blocks']}; this run used "
                            f"{_proto['label']}]{_S.RESET}")
            elif _rate_meas:
                rate_src = f" {_S.FILE}[measured from the current]{_S.RESET}"
            elif meta.get('c_rate'):
                rate_src = f" {_S.FILE}[from file]{_S.RESET}"
            else:
                rate_src = ""
            print(f"\n{_S.HEADER}{'='*60}")
            print(f"  Review parameters for: {name}")
            print(f"{'='*60}{_S.RESET}")
            print(f"\n{_S.SECTION}  Material:{_S.RESET}")
            print(f"    Composition:          {_S.VALUE}{composition}{_S.RESET}")
            print(f"    Cell identifier:      {_S.VALUE}"
                  f"{cell_id or '(not recorded)'}{_S.RESET}")
            print(f"    Battery chemistry:    {_S.VALUE}{battery_chemistry}{_S.RESET}")
            # WHAT THE FIGURES WILL CALL THE HALF-CYCLES, said here where the
            # answer can still be changed. A negative electrode names the
            # process rather than the direction, and until 1.9.0.75 an
            # unestablished ion was silently printed as lithium — an NTO
            # anode in a sodium half-cell was captioned "Delithiation" on
            # every waterfall. Neutral labels are no longer a silent
            # outcome: if they are what this dataset will get, it says so and
            # says what to do about it.
            if str(electrode_type).lower().startswith("neg"):
                _ci, _co = half_cycle_labels(
                    dict(anode_labels_swapped=True,
                         counter_electrode_metal=_ce_metal,
                         counter_electrode_metal_source=_ce_metal_src,
                         battery_chemistry=battery_chemistry))
                if _ci == "Charge":
                    print(f"    Half-cycle labels:    {_S.WARN}Charge / "
                          f"Discharge{_S.RESET}")
                    print(f"      {_S.WARN}The working ion is not "
                          f"established, so the figures will not name the "
                          f"process. Give the electrolyte, or a formula "
                          f"rather than an abbreviation, or state the "
                          f"counter electrode metal.{_S.RESET}")
                else:
                    print(f"    Half-cycle labels:    {_S.VALUE}{_ci} / "
                          f"{_co}{_S.RESET} (working ion "
                          f"{_ce_metal}, {_ce_metal_src})")
            print(f"    Theoretical capacity: {_S.VALUE}{theoretical_capacity_mAh_g:.1f} mAh/g{_S.RESET}")
            print(f"    Electrode type:       {_S.VALUE}{electrode_type}{_S.RESET}")
            print(f"\n{_S.SECTION}  Electrode:{_S.RESET}")
            print(f"    Active mass:          {_S.VALUE}{active_material_mass_mg:.2f} mg{_S.RESET}{mass_src}")
            print(f"    Blend:                {_S.VALUE}{blend} ({active_fraction*100:.1f}% active){_S.RESET}")
            print(f"    Punch diameter:       {_S.VALUE}{electrode_diameter_mm:.1f} mm "
                  f"(area = {electrode_area_cm2:.3f} cm2){_S.RESET}")
            print(f"    Active loading:       {_S.VALUE}{active_loading_mg_cm2:.2f} mg/cm2{_S.RESET}")
            print(f"    Total loading:        {_S.VALUE}{total_loading_mg_cm2:.2f} mg/cm2{_S.RESET}")
            print(f"\n{_S.SECTION}  Cycling:{_S.RESET}")
            print(f"    C-rate:               {_S.VALUE}{charge_rate_c:.2f} C{_S.RESET}{rate_src}")
            print(f"    Theoretical current:  {_S.VALUE}{theoretical_charge_current_mA:.3f} mA{_S.RESET}")
            # The window BEING STORED, not the file's. The file's figures
            # are channel safety limits — an LTO cell cycled 1.2-2.5 V
            # reports 0-3.25 V — and the code above deliberately prefers the
            # measured window. Showing the file's under "are these correct?"
            # asked the operator to confirm a number that was not being used.
            if voltage_lower_V is not None and voltage_upper_V is not None:
                print(f"    Voltage window:       {_S.VALUE}"
                      f"{voltage_lower_V:.3f}--{voltage_upper_V:.3f} V"
                      f"{_S.RESET} {_S.FILE}[{voltage_window_source}]"
                      f"{_S.RESET}")
            print(f"    Key cycles:           {_S.VALUE}{key_cycles}{_S.RESET}")
            if max_cycle is not None:
                print(f"    Cycle cut-off:        {_S.VALUE}cycles 1-{max_cycle} "
                      f"(later cycles removed){_S.RESET}")
            else:
                print(f"    Cycle cut-off:        {_S.VALUE}none "
                      f"(all cycles){_S.RESET}")
            if meta.get('charge_current_A') is not None:
                file_current_mA = meta['charge_current_A'] * 1000
                calc_current_mA = theoretical_charge_current_mA
                pct_diff = (abs(file_current_mA - calc_current_mA) /
                            max(file_current_mA, 1e-9) * 100)
                if pct_diff > 5:
                    print(f"    {_S.WARN}WARNING: Current mismatch: calculated "
                          f"{calc_current_mA:.4f} mA vs file "
                          f"{file_current_mA:.4f} mA "
                          f"({pct_diff:.0f}% difference)")
                    print(f"      Check active mass and theoretical capacity.{_S.RESET}")
                else:
                    print(f"    {_S.CONFIRM}Current matches file: "
                          f"{file_current_mA:.4f} mA ✓{_S.RESET}")
            print(f"\n{_S.SECTION}  Cell assembly:{_S.RESET}")
            print(f"    Separator:            {_S.VALUE}{separator_type}{_S.RESET}")
            if electrolyte_volume_uL is not None:
                print(f"    Electrolyte:          {_S.VALUE}{electrolyte_composition} "
                      f"({electrolyte_volume_uL:.0f} uL){_S.RESET}")
            else:
                print(f"    Electrolyte:          {_S.VALUE}{electrolyte_composition}{_S.RESET}")
            if counter_electrode_measured:
                print(f"    Counter electrode:    {_S.VALUE}"
                      f"{counter_electrode_mass_mg:.1f} mg "
                      f"(weighed){_S.RESET}")
            else:
                # THE SAME METAL THE PARAMETERS RECORD. This line used to
                # recompute the estimate without it, so a cell whose counter
                # was resolved to lithium was still reviewed as "~98 mg (Na
                # metal)" — the sodium figure — while the stored parameter
                # said Li. Two call sites, one of them not told the answer.
                _est_ce, _est_metal = _estimate_counter_electrode_mass_mg(
                    battery_chemistry=battery_chemistry, metal=_ce_metal)
                _src_word = {"stated": "you said so",
                             "chemistry": "from the electrolyte / formula",
                             "library": "from the composition library",
                             "assumed": "ASSUMED"}.get(_ce_metal_src, "")
                print(f"    Counter electrode:    {_S.VALUE}"
                      f"~{_est_ce:.0f} mg ({_est_metal} metal, "
                      f"{_DEFAULT_COUNTER_DIAMETER_MM:.0f} mm x "
                      f"{_DEFAULT_COUNTER_THICKNESS_MM:.1f} mm, "
                      f"estimated){_S.RESET}"
                      + (f" {_S.FILE}[{_src_word}]{_S.RESET}"
                         if _src_word else ""))
            print(f"\n{_S.SECTION}  Analysis:{_S.RESET}")
            pe_str = f"{_S.CONFIRM}ON{_S.RESET}" if power_energy_analysis else f"{_S.WARN}SKIP{_S.RESET}"
            pk_str = f"{_S.CONFIRM}ON{_S.RESET}" if RUN_PEAK_FITTING else f"{_S.WARN}SKIP{_S.RESET}"
            print(f"    Power/energy (Cell 6c):           {pe_str}")
            print(f"    dQ/dV curves (Cell 7):            {_S.CONFIRM}ON{_S.RESET}")
            print(f"    dQ/dV peak fitting (Cells 8-12):  {pk_str}")
            # dQ/dV preprocessing summary
            _sw = sample_parameters['smoothing_window']
            _sr = sample_parameters['spike_removal']
            _st = sample_parameters['spike_threshold_multiplier']
            _auto_count = sum(1 for v in [_sw, _sr, _st] if v == 'auto')
            if _auto_count == 3:
                print(f"    dQ/dV preprocessing:               "
                      f"{_S.CONFIRM}AUTO{_S.RESET} "
                      f"(adapts to material type)")
            elif _auto_count > 0:
                print(f"    dQ/dV preprocessing:               "
                      f"{_S.VALUE}MIXED{_S.RESET} "
                      f"({_auto_count} auto, {3-_auto_count} manual)")
            else:
                print(f"    dQ/dV preprocessing:               "
                      f"{_S.VALUE}MANUAL{_S.RESET} "
                      f"(window={_sw}, spike={_sr}, "
                      f"threshold={_st})")
            print(f"\n{_S.SECTION}  Output:{_S.RESET}")
            print(f"    Palette:              {_S.VALUE}{colour_palette}{_S.RESET}")
            print(f"    File format:          {_S.VALUE}{file_format}{_S.RESET}")
            print(f"\n{_S.HEADER}{'='*60}{_S.RESET}")
            while True:
                confirmation = input(
                    "\nAre these parameters correct? (yes/no): "
                ).lower()
                if confirmation in ["yes", "y"]:
                    confirmed = True
                    _last_composition = composition
                    _last_theoretical_capacity = theoretical_capacity_mAh_g
                    print(f"{_S.CONFIRM}  ✓ Parameters confirmed.{_S.RESET}")
                    break
                elif confirmation in ["no", "n"]:
                    print(f"\n{_S.WARN}  Re-entering parameters...{_S.RESET}\n")
                    break
                else:
                    print(f"  {_S.WARN}Enter 'yes' or 'no'.{_S.RESET}")
        user_parameters[name] = sample_parameters

    # =============================================================================
    # FINAL SUMMARY
    # =============================================================================
    print(f"\n{_S.HEADER}{'='*60}")
    print(f"  Parameters set for {len(user_parameters)} dataset(s)")
    print(f"{'='*60}{_S.RESET}")
    for name, params in user_parameters.items():
        display = _get_display_name(name, params, user_parameters)
        mass    = params['active_material_mass_mg']
        loading = params['active_loading_mg_cm2']
        rate    = params['charge_rate_c']
        pe_flag = (f"{_S.CONFIRM}ON{_S.RESET}"
                   if params.get('power_energy_analysis', True)
                   else f"{_S.WARN}SKIP{_S.RESET}")
        print(f"  {_S.VALUE}{display}{_S.RESET}")
        print(f"    {mass:.2f} mg active | "
              f"{loading:.2f} mg/cm2 | "
              f"{rate:.2f} C | "
              f"P/E: {pe_flag}")

    print(f"\n{_S.SECTION}  Pipeline-level:{_S.RESET}")
    pk_flag = (f"{_S.CONFIRM}ON{_S.RESET}" if RUN_PEAK_FITTING
               else f"{_S.WARN}SKIP{_S.RESET}")
    print(f"    dQ/dV peak fitting (Cells 8-12):  {pk_flag}")
    if not RUN_PEAK_FITTING:
        print(f"    {_S.FILE}(Cell 7 still runs. To enable fitting later,")
        print(f"     set RUN_PEAK_FITTING = True above and re-run Cell 3.){_S.RESET}")

    print(f"{_S.HEADER}{'='*60}{_S.RESET}")
    print(f"\n{_S.SECTION}  Helper functions available for all downstream cells:{_S.RESET}")
    print(f"    _get_display_name(name, params, user_parameters)")
    print(f"    _force_integer_cycles(ax)")
    print(f"    _skip_module(module_name)  {_S.FILE}— used by Modules 3-8 guards{_S.RESET}")

    # =============================================================================
    # CYCLE TRUNCATION  (v1.8.5)
    # =============================================================================
    # If a maximum analysis cycle was set for any dataset above, trim the loaded
    # data table now, once, here, so every downstream cell sees only the
    # retained cycles. Only Cell 2 reads the raw files; every other cell works
    # off electrochemical_data, so trimming it here propagates everywhere.
    # Destructive: to restore full data, re-run the data-loading cell (Cell 2).
    # =============================================================================
    # The cut-off is RECORDED here and applied by the caller to the Dataset
    # itself (`io.truncate_cycles`). This function used to truncate a copy of
    # `electrochemical_data`, which the notebook then rebuilt from the
    # untruncated Dataset two lines later — so the answer was collected,
    # applied, discarded, and reported as done.
    _all_cuts = {n: user_parameters.get(n, {}).get('max_cycle')
                 for n in electrochemical_data}
    _cuts = {n: c for n, c in _all_cuts.items() if c}
    if _cuts:
        print(f"\n{_S.HEADER}{'='*60}")
        print(f"  Cycle cut-offs recorded")
        print(f"{'='*60}{_S.RESET}")
        for n, c in _cuts.items():
            print(f"  {_S.VALUE}"
                  f"{_get_display_name(n, user_parameters[n], user_parameters)}"
                  f"{_S.RESET}: cycles above {c} will be excluded")
        # DO THEY AGREE? Datasets analysed together are compared with each
        # other — the comparative capacity, retention and fade figures put
        # them on one axis. Different cut-offs there mean those figures span
        # different cycle ranges, which is a real difference in what is being
        # compared and is invisible once the figure is drawn. Said out loud
        # here rather than left to be noticed.
        if len(_all_cuts) > 1 and len(set(_all_cuts.values())) > 1:
            _uncapped = [n for n, c in _all_cuts.items() if not c]
            print(f"\n  {_S.WARN}The cut-offs are not the same for every "
                  f"dataset.{_S.RESET}")
            if _uncapped:
                print(f"  {_S.FILE}No cap on: "
                      f"{', '.join(_get_display_name(n, user_parameters[n], user_parameters) for n in _uncapped)}"
                      f"{_S.RESET}")
            print(f"  Cells 6b and 6c draw these on shared axes, so those "
                  f"figures will compare different cycle ranges. Deliberate "
                  f"for a cell that died early; otherwise re-run this cell.")
        print(f"{_S.HEADER}{'='*60}{_S.RESET}")
    return user_parameters, RUN_PEAK_FITTING


# =============================================================================
# The declarative path
# =============================================================================
# `collect_parameters` is the interactive cell restored, and it is the one a
# researcher should use: seeing "[from file: 4.72]" against a mass is how you
# know which numbers you are responsible for. But a scheduled run, a test, and
# an application front end all need the same record built without a keyboard,
# and faking keystrokes into a 630-line prompt sequence is not a way to get it.
# So the declarative path is its own function, producing the same 53-field
# record, with the same derivations, from a dict.

# What a dict-mode run must state for itself. `composition` and
# `theoretical_capacity_mAh_g` are properties of the material and the cycler
# file cannot know them. `electrode_type` decides the SIGN of dQ/dV and which
# half-cycle is called "Discharge"; defaulting it to "Positive" meant a run on
# an anode silently produced mirror-image figures that looked entirely
# plausible, so it is asked for rather than assumed.
_REQUIRED = ("composition", "theoretical_capacity_mAh_g", "electrode_type")

_ELECTRODE_TYPES = ("Positive", "Negative")


def parameters_from_dict(specs, cycler_metadata=None, verbose=True):
    """
    Build the same parameter record as `collect_parameters`, without prompting.

    `specs` is `{dataset_name: {field: value}}`. Anything omitted falls back to
    the cycler metadata, then to the documented default. Only `composition` and
    `theoretical_capacity_mAh_g` have no defensible default and must be given —
    a theoretical capacity is a property of the material, not of the file.

    Every derived quantity is computed exactly as the interactive path computes
    it: electrode area from the diameter, active fraction from the blend,
    loadings from the mass and area, and the theoretical current from the
    capacity and the C-rate.
    """
    cycler_metadata = cycler_metadata or {}
    out = {}
    for name, spec in specs.items():
        spec = dict(spec or {})
        meta = dict(cycler_metadata.get(name) or {})
        # THE LIBRARY IS A DEFAULT HERE TOO. A declarative run that names a
        # material the library knows should not have to restate its
        # theoretical capacity, and should not be able to restate it WRONG
        # while the interactive path gets it right. `composition` is still
        # required — it is what the lookup is made from.
        _lib_formula = None
        if spec.get("theoretical_capacity_mAh_g") in (None, ""):
            _lib_cap, _lib_formula, _lib_note = \
                lookup_specific_capacity(spec.get("composition"))
            if _lib_cap is not None:
                spec["theoretical_capacity_mAh_g"] = _lib_cap
                spec.setdefault("theoretical_capacity_source", "library")
                if verbose:
                    print(f"  {name}: theoretical capacity {_lib_cap} mAh/g "
                          f"from the library ({_lib_formula}, {_lib_note})")
        missing = [k for k in _REQUIRED if spec.get(k) in (None, "")]
        if missing:
            raise ValueError(f"{name}: {', '.join(missing)} must be given — "
                             f"the cycler file cannot supply "
                             f"{'them' if len(missing) > 1 else 'it'}.")
        # A theoretical capacity of NaN is not a missing value the guards can
        # see: `x > 1.2 * nan` is False, so the over-theoretical check simply
        # stopped firing and said nothing. Rejected here instead.
        _tc = spec.get("theoretical_capacity_mAh_g")
        try:
            _tc = float(_tc)
        except (TypeError, ValueError):
            raise ValueError(f"{name}: theoretical_capacity_mAh_g must be a "
                             f"number, not {spec.get('theoretical_capacity_mAh_g')!r}")
        if not np.isfinite(_tc) or _tc <= 0:
            raise ValueError(f"{name}: theoretical_capacity_mAh_g is {_tc} — "
                             f"it must be a positive number, because the "
                             f"over-theoretical capacity check compares "
                             f"against it and a NaN silently disables it.")
        spec["theoretical_capacity_mAh_g"] = _tc
        if str(spec["electrode_type"]).strip().title() not in _ELECTRODE_TYPES:
            raise ValueError(
                f"{name}: electrode_type is {spec['electrode_type']!r}; it "
                f"must be one of {', '.join(_ELECTRODE_TYPES)}.")
        spec["electrode_type"] = str(spec["electrode_type"]).strip().title()

        def g(key, default=None, meta_key=None):
            v = spec.get(key)
            if v not in (None, ""):
                return v
            if meta_key and meta.get(meta_key) not in (None, ""):
                return meta[meta_key]
            return default

        mass = float(g("active_material_mass_mg", 0.0,
                       "active_material_mass_mg") or 0.0)
        diameter = float(g("electrode_diameter_mm", default_electrode_diameter_mm))
        area = np.pi * (diameter / 20.0) ** 2          # mm -> cm, then pi r^2
        blend = str(g("blend", "80/10/10"))
        try:
            parts = [float(x) for x in blend.split("/")]
            frac = parts[0] / sum(parts) if sum(parts) else 0.8
        except (ValueError, ZeroDivisionError):
            frac = 0.8
        coating = mass / frac if frac else np.nan
        theo_g = float(spec["theoretical_capacity_mAh_g"])
        theo_mAh = theo_g * mass / 1000.0
        c_rate = g("charge_rate_c", None, "c_rate")
        try:
            c_rate_f = float(c_rate)
        except (TypeError, ValueError):
            c_rate_f = np.nan

        _chem = g("battery_chemistry",
                  infer_chemistry(spec.get("composition"),
                                  g("electrolyte_composition", "")))
        # Same order as the interactive path, minus the question it cannot
        # ask. `assumed` is recorded and printed as such rather than passing
        # for an inference.
        _ce_metal, _ce_metal_src = counter_electrode_metal(
            spec.get("composition"), stated=g("counter_electrode_metal", None),
            battery_chemistry=_chem, electrolyte=g("electrolyte_composition", ""))
        _counter = _estimate_counter_electrode_mass_mg(
            battery_chemistry=_chem, metal=_ce_metal)
        _pal = str(g("colour_palette", default_palette))
        if not isinstance(_counter, tuple):
            _counter = (_counter, None)

        rec = {
            "composition": spec["composition"],
            "cell_id": g("cell_id", _suggest_cell_id(name, list(specs))),
            "blend": blend,
            "active_material_mass_mg": mass,
            "theoretical_capacity_mAh_g": theo_g,
            "theoretical_capacity_source": g("theoretical_capacity_source",
                                             "given"),
            "theoretical_capacity_library_formula": _lib_formula,
            "electrode_type": g("electrode_type", "Positive"),
            "electrode_diameter_mm": diameter,
            "electrode_area_cm2": area,
            "active_fraction": frac,
            "total_electrode_coating_mg": coating,
            "active_loading_mg_cm2": mass / area if area else np.nan,
            "total_loading_mg_cm2": coating / area if area else np.nan,
            "power_energy_analysis": bool(g("power_energy_analysis", True)),
            # The COERCED rate. Storing the raw answer let a string or None
            # through, which printed as "None C" in the report and the
            # dataset_info file while the derived currents used the float.
            "charge_rate_c": (float(c_rate_f)
                              if np.isfinite(c_rate_f) else None),
            "key_cycles": list(g("key_cycles", default_key_cycles)),
            "max_cycle": g("max_cycle"),
            "theoretical_capacity_mAh": theo_mAh,
            "theoretical_charge_current_mA": theo_mAh * c_rate_f,
            "theoretical_charge_rate_mAh_g": theo_g * c_rate_f,
            "voltage_upper_V": g("voltage_upper_V", None, "voltage_upper_V"),
            "voltage_lower_V": g("voltage_lower_V", None, "voltage_lower_V"),
            "voltage_window_source": ("file" if meta.get("voltage_upper_V")
                                      and "voltage_upper_V" not in spec
                                      else "operator"),
            "voltage_limit_upper_V_from_file": meta.get("voltage_upper_V"),
            "voltage_limit_lower_V_from_file": meta.get("voltage_lower_V"),
            "separator_type": g("separator_type", "glass fibre"),
            "electrolyte_volume_uL": g("electrolyte_volume_uL", 100.0),
            "electrolyte_composition": g("electrolyte_composition", ""),
            # The estimator returns (mass, metal); 1.8.6 unpacked it and kept
            # both, because "97.6 mg of sodium" is a very different claim from
            # "97.6 mg" on its own.
            "counter_electrode_mass_mg": g("counter_electrode_mass_mg",
                                           _counter[0]),
            "counter_electrode_metal": _counter[1],
            "counter_electrode_metal_source": _ce_metal_src,
            "counter_electrode_measured": "counter_electrode_mass_mg" in spec,
            "battery_chemistry": _chem,
            "barcode": meta.get("barcode"),
            "test_start_date": meta.get("test_start_date"),
            "charge_current_A_from_file": meta.get("charge_current_A"),
            "discharge_current_A_from_file": meta.get("discharge_current_A"),
            # `_validate_palette` returns a BOOLEAN — it answers "is this a
            # real matplotlib colormap", not "give me one". Storing its result
            # put True in the palette field and seaborn tried to iterate it.
            "colour_palette": (_pal if _validate_palette(_pal)
                               else default_palette),
            "file_format": g("file_format", "png"),
            # The analysis knobs, from the one table — the SAME defaults the
            # interactive path uses, taken from the same place so they cannot
            # drift apart again. 'auto' hands the choice to the code that
            # reads this dataset's own curve; give a number to override it.
            **{k: g(k, v) for k, v in ANALYSIS_DEFAULTS.items()},
            "anode_labels_swapped": str(g("electrode_type", "Positive")
                                        ).lower() == "negative",
        }
        out[name] = rec
        if verbose:
            src = "file" if meta.get("active_material_mass_mg") else "given"
            print(f"  {name}: {rec['composition']} cell {rec['cell_id']}, "
                  f"{mass:.2f} mg [{src}], {rec['electrode_type'].lower()} "
                  f"electrode, {rec['active_loading_mg_cm2']:.2f} mg/cm2")
    return out


def detection_overrides(params):
    """
    The DETECTION fields a user set explicitly, to merge over the profile's.

    'auto' means "let the profile decide" and is dropped; a value is an
    instruction and is kept. The defaults come from ANALYSIS_DEFAULTS — the
    same table the record was built from — so a value equal to one of them is
    not an operator choice.
    """
    return _explicit(params, DETECTION_KEYS)


def analysis_overrides(params):
    """
    The ANALYSIS knobs a user set explicitly: reference cycle and tracking
    tolerance. Same contract as `detection_overrides`.

    These two are kept, and the four that used to sit beside them
    (`fit_centre_tolerance_mV`, `fit_sigma_min_mV`, `fit_sigma_max_mV`,
    `baseline_degree`) are GONE, because they were of a different kind.
    A sigma bound cannot serve LTO and a layered oxide at once, so it is
    derived from the measured profile class — and the record was asserting
    200 mV while the sharp profile fitted at 30. A record field that
    disagrees with what ran is worse than an absent one: it reads as
    provenance. The manifest records what ran, never what was offered.

    The reference cycle and the tracking tolerance are the opposite case:
    genuine judgements the operator can make better than the code. 80 mV
    varies with peak spacing, and published ICA practice is unambiguous that
    the reference should be a POST-FORMATION cycle, which only the operator
    knows the length of.
    """
    return _explicit(params, ANALYSIS_KEYS)


def preprocess_overrides(params):
    """
    The dQ/dV preprocessing fields a user set explicitly, as a dict to merge
    over `signal.auto_preprocess_params`.

    'auto' means "let the profile decide" and is dropped; anything else is an
    instruction and is kept. This is how the smoothing, spike and rebin
    controls from 1.8.6's Cell 3 reach the preprocessing again.
    """
    # Compared against ANALYSIS_DEFAULTS — the values BOTH record builders
    # write when nobody asked for anything — so an override means the OPERATOR
    # chose it, not that a default happened to be a number rather than 'auto'.
    return _explicit(params, PREPROCESS_KEYS)


# 1.8.7 opened a native file dialog. It is the right interface on the machine
# this actually runs on — a Windows desktop with the data on a mapped drive —
# because the operator can see the folder, sort by date, and multi-select the
# three files of a triplicate without typing a path. Set False to skip the
# dialog and always use the typed list below.
USE_FILE_DIALOG = True


def pick_files_dialog(initialdir=None, title="Select one or more cycler files"):
    """
    1.8.7's Tkinter file picker. Returns a list of paths, or None.

    Returns None — not an empty list — when no dialog could be shown at all
    (no display, no tkinter, a headless kernel), so the caller can tell
    "there is no dialog here" from "the operator pressed Cancel".

    The `-topmost` / `update()` pair is not decoration: without it the dialog
    opens BEHIND the Jupyter window on Windows and the run appears to hang.
    """
    if not USE_FILE_DIALOG:
        return None
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception:
        return None
    root = None
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        root.update()
        paths = filedialog.askopenfilenames(
            title=title,
            initialdir=initialdir if initialdir and os.path.isdir(initialdir)
            else None,
            filetypes=[("Excel files", "*.xlsx *.xls"), ("All files", "*.*")])
    except Exception:
        return None
    finally:
        if root is not None:
            try:
                root.destroy()
            except Exception:
                pass
    return [os.path.abspath(p) for p in paths]


def select_files(candidates, *, initialdir=None, verbose=True, headless=False,
                 answer=None, use_dialog=None):
    """
    Ask which files to load: a file dialog where there is one, a list where not.

    `candidates` is what the folder holds, used for the typed fallback and as
    the answer when neither route can ask. The dialog is not restricted to
    them — the whole point of a file browser is to go somewhere else.
    """
    want_dialog = USE_FILE_DIALOG if use_dialog is None else use_dialog
    if want_dialog and not headless and answer is None:
        picked = pick_files_dialog(initialdir)
        if picked is not None:
            if picked:
                if verbose:
                    print(f"  {len(picked)} file(s) chosen from the dialog:")
                    for p in picked:
                        print(f"    {os.path.basename(p)}")
                return picked
            # An explicit Cancel. Fall through to the typed list rather than
            # loading everything, which is what Cancel plainly does not mean.
            if verbose:
                print("  File dialog cancelled — choose from the folder "
                      "listing instead.")
    return choose_files(candidates, verbose=verbose, headless=headless,
                        answer=answer)


def choose_files(paths, verbose=True, headless=False, answer=None):
    """
    Show what is in the folder and ask which files to load.

    Ratatosk used to take every match, which is wrong the moment a folder holds
    more than one experiment: eight files in one directory can be three
    chemistries, and loading all of them puts an LTO anode and an NMC cathode
    on the same axes.

    Accepts a comma-separated list, ranges (`1-3`), `all`, or a blank line
    meaning all. Groups the listing by the leading token of each filename, so
    a triplicate reads as a triplicate.
    """
    if not paths:
        return []
    groups = {}
    for i, p in enumerate(paths, 1):
        base = os.path.splitext(os.path.basename(p))[0]
        # The chemistry is usually the second underscore- or hyphen-delimited
        # token; falling back to the first keeps unusual names together.
        toks = re.split(r"[_-]", base)
        key = toks[1] if len(toks) > 1 else toks[0]
        groups.setdefault(key, []).append((i, base))

    if verbose:
        print(f"{len(paths)} file(s) found:\n")
        for key, items in groups.items():
            print(f"  {key}")
            for i, base in items:
                print(f"    {i:>2}. {base}")
        print()

    if headless or answer is not None:
        raw = "" if answer is None else str(answer)
    else:
        raw = _builtins.input(
            "Which files? (numbers, ranges like 1-3, or blank for all): "
        ).strip()

    if not raw or raw.lower() in ("all", "a", "*"):
        if verbose:
            print(f"  loading all {len(paths)}")
        return list(paths)

    picked = []
    for part in raw.replace(" ", "").split(","):
        if not part:
            continue
        if "-" in part:
            try:
                lo, hi = (int(x) for x in part.split("-", 1))
            except ValueError:
                continue
            picked.extend(range(lo, hi + 1))
        else:
            try:
                picked.append(int(part))
            except ValueError:
                continue
    out = [paths[i - 1] for i in sorted(set(picked))
           if 1 <= i <= len(paths)]
    if not out:
        if verbose:
            print("  nothing recognised in that answer — loading all")
        return list(paths)
    if verbose:
        print(f"  loading {len(out)} of {len(paths)}:")
        for p in out:
            print(f"    {os.path.splitext(os.path.basename(p))[0]}")
    return out


def describe_defaults(metadata):
    """One line per dataset saying what the file already told us."""
    out = []
    for name, meta in metadata.items():
        bits = []
        if meta.get("active_material_mass_mg"):
            bits.append(f"{meta['active_material_mass_mg']:.2f} mg")
        if meta.get("c_rate"):
            bits.append(f"{meta['c_rate']} C")
        if meta.get("voltage_lower_V") and meta.get("voltage_upper_V"):
            bits.append(f"{meta['voltage_lower_V']}-{meta['voltage_upper_V']} V")
        if meta.get("cycle_count"):
            bits.append(f"{meta['cycle_count']} cycles planned")
        out.append(f"  {name}: " + (", ".join(bits) if bits
                                    else "no 'test' sheet — nothing pre-filled"))
    return "\n".join(out)


# =============================================================================
# CELL 2'S ANSWERS, CHECKED BEFORE ANYTHING ACTS ON THEM
# =============================================================================
# =============================================================================
# WHERE THE DATA IS, WITHOUT HARD-CODING ONE PERSON'S DRIVE
# =============================================================================
# A notebook that names `G:\My Drive\...` runs on exactly one machine. It was
# doing that until 1.9.0.74 and it failed on the author's SECOND computer, let
# alone anybody else's — which for a tool about to be handed to somebody else
# is not an inconvenience, it is a defect.
#
# The search is ordered most-specific-first and every step is printed, so a
# reader can see which folder was chosen and why rather than wondering.
#
# `RATATOSK_DATA_DIR` comes FIRST and unconditionally. That is the handle for
# a scheduled run, a CI job, a container, or anyone who simply wants to say
# where the data is without editing a notebook cell.
DATA_DIR_ENV = "RATATOSK_DATA_DIR"

# Folder names to look for under each root, most specific first.
DATA_DIR_LEAVES = (
    os.path.join("Python Scripting", "Example Battery Data"),
    os.path.join("Python Scripting", "Battery Data"),
    "Example Battery Data",
    "Battery Data",
    "",
)


def _candidate_roots():
    """Places a data folder plausibly lives, most likely first."""
    roots, home = [], os.path.expanduser("~")

    def add(p):
        if p and p not in roots:
            roots.append(p)

    # Cloud-drive mounts, however this machine spells them.
    if os.name == "nt":
        # Only drives that actually exist — probing a missing letter on
        # Windows can block for seconds on a disconnected network mapping.
        for _L in "GHIJKLMNOPQRSTUVWXYZABCDEF":
            _d = f"{_L}:\\"
            try:
                if not os.path.exists(_d):
                    continue
            except OSError:
                continue
            add(os.path.join(_d, "My Drive"))
            add(os.path.join(_d, "Shared drives"))
            add(_d)
    for _n in ("Google Drive", "GoogleDrive", "OneDrive", "Dropbox"):
        add(os.path.join(home, _n, "My Drive"))
        add(os.path.join(home, _n))
    add(os.path.join(home, "My Drive"))
    add(os.path.join(home, "Documents"))
    add(home)
    # Beside the notebook, and wherever the kernel happens to be running.
    try:
        add(os.getcwd())
    except OSError:
        pass
    return roots


def find_data_dir(pattern="*.xlsx", extra=(), verbose=True):
    """
    The folder holding the cycler exports, discovered rather than assumed.

    Order: `$RATATOSK_DATA_DIR`, then anything in `extra`, then every
    combination of `_candidate_roots()` and `DATA_DIR_LEAVES`.

    A FOLDER THAT EXISTS BUT HOLDS NO MATCHING FILES IS NOT THE DATA FOLDER.
    Existence alone would stop the search at `~/Documents` on almost every
    machine, so a candidate has to contain at least one file matching
    `pattern` to win. A folder that exists and is empty is remembered as a
    fallback and reported as such, because "I found it but it is empty" and
    "I could not find it" are different problems with different fixes.

    Returns the path, or "" if nothing was found. Everything tried is left on
    `find_data_dir.searched` so the caller's error message can list it.
    """
    import glob as _glob

    tried, empty = [], []
    env = os.environ.get(DATA_DIR_ENV, "").strip()
    order = ([env] if env else []) + [str(p) for p in (extra or [])]
    # LEAF-MAJOR, NOT ROOT-MAJOR. Root-major tries `G:\My Drive` itself — the
    # bare-root leaf — before it tries `~/Documents/Example Battery Data`, so
    # one stray spreadsheet at the top of a cloud drive beats a folder that
    # is actually named after the data. Sweeping the most specific leaf
    # across EVERY root first means a folder called "Example Battery Data"
    # wins wherever it lives, and the bare roots are only reached when no
    # named folder exists anywhere.
    _roots = _candidate_roots()
    for leaf in DATA_DIR_LEAVES:
        for root in _roots:
            order.append(os.path.join(root, leaf) if leaf else root)

    found = ""
    for cand in order:
        if not cand or cand in tried:
            continue
        tried.append(cand)
        try:
            if not os.path.isdir(cand):
                continue
            if _glob.glob(os.path.join(cand, pattern)):
                found = cand
                break
            empty.append(cand)
        except OSError:
            continue                      # unreadable mount, keep looking

    find_data_dir.searched = tried
    find_data_dir.empty = empty
    if verbose:
        if found:
            _why = ("$" + DATA_DIR_ENV if env and found == env
                    else f"{len(_glob.glob(os.path.join(found, pattern)))} "
                         f"file(s) matching {pattern}")
            print(entry("data folder", "found", _why))
            print(entry("", found[:70]))
        elif empty:
            print(entry("data folder", "none with data",
                        f"{len(empty)} folder(s) exist but hold no {pattern}"))
        else:
            print(entry("data folder", "not found",
                        f"searched {len(tried)} location(s)"))
    return found


def validate_run_settings(*, data_dir, output_base, files, pattern,
                          parameter_mode, parameters, max_cycle,
                          dqdv_computation, dqdv_bin_mV, closure_sample,
                          run_peak_fitting, verbose=True):
    """
    Read Cell 2 back and refuse the run if it says something impossible.

    Every one of these used to fail LATER, and most of them failed quietly.
    `PARAMETER_MODE = "Ask"` fell through the `== "ask"` test into the
    declarative branch and ran the whole pipeline on a NaN theoretical
    capacity; `DQDV_COMPUTATION = "hist"` fell through to the derivative path
    and produced a perfectly presentable set of figures computed the way the
    cell said it did not want; a mistyped `DATA_DIR` surfaced as an empty
    glob three cells later. A settings cell that is wrong should say so
    where it was written, not somewhere downstream.

    Returns the settings as a dict, which the manifest records verbatim.
    """
    modes = ("ask", "dict")
    methods = ("histogram", "derivative")

    if not isinstance(data_dir, str) or not data_dir.strip():
        # AN EMPTY DATA_DIR MEANS THE SEARCH FOUND NOTHING, not that the user
        # forgot to type something — Cell 2 leaves it blank on purpose and
        # `find_data_dir` fills it in. Say what was looked for and give the
        # two ways out, rather than "set it to the folder", which is what
        # they were trying to avoid having to do per machine.
        _tried = list(getattr(find_data_dir, "searched", ()) or ())
        _empty = list(getattr(find_data_dir, "empty", ()) or ())
        _msg = ["DATA_DIR is empty and no data folder was found."]
        if _empty:
            _msg.append(
                "These folders exist but hold no file matching the pattern:")
            _msg += [f"    {p}" for p in _empty[:5]]
        elif _tried:
            _msg.append(f"{len(_tried)} location(s) were searched, including:")
            _msg += [f"    {p}" for p in _tried[:5]]
        _msg += [
            "",
            "Either set DATA_DIR in Cell 2 to the folder holding the cycler",
            f"exports, or set the {DATA_DIR_ENV} environment variable to it",
            "before starting Jupyter — which is the one to use if you move",
            "between machines.",
        ]
        raise ValueError("\n".join(_msg))
    if not os.path.isdir(data_dir):
        raise NotADirectoryError(
            f"DATA_DIR does not exist:\n    {data_dir}\n"
            f"Check the drive letter and that any network or Drive folder "
            f"is mounted. To stop naming one machine's drive, clear DATA_DIR "
            f"in Cell 2 and it will be searched for, or set the "
            f"{DATA_DIR_ENV} environment variable.")

    if not isinstance(output_base, str) or not output_base.strip():
        raise ValueError("OUTPUT_BASE is empty — set it to the folder the "
                         "run folders should be created in.")
    # The folder itself is created by `start_run`; what cannot be created is
    # a missing drive, and that is worth saying before an hour of fitting.
    anchor = os.path.splitdrive(os.path.abspath(output_base))[0]
    if anchor and not os.path.exists(anchor + os.sep):
        raise NotADirectoryError(
            f"OUTPUT_BASE is on {anchor} which is not available:\n"
            f"    {output_base}")

    if isinstance(files, str):
        raise TypeError("FILES must be a list of file names, not a single "
                        f"string. Write FILES = [{files!r}].")
    files = list(files or [])
    for f in files:
        if not isinstance(f, str):
            raise TypeError(f"FILES contains {f!r}, which is not a file name.")
    missing = [f for f in files
               if not os.path.isfile(os.path.join(data_dir, f))]
    if missing:
        raise FileNotFoundError(
            "FILES names files that are not in DATA_DIR:\n"
            + "\n".join(f"    {m}" for m in missing)
            + f"\n  looked in {data_dir}")
    if not files and (not isinstance(pattern, str) or not pattern.strip()):
        raise ValueError("PATTERN is empty and FILES is empty, so nothing "
                         "would be listed. Use \"*.xlsx\".")

    mode = str(parameter_mode).strip().lower()
    if mode not in modes:
        raise ValueError(f"PARAMETER_MODE is {parameter_mode!r}; it must be "
                         f"one of {', '.join(repr(m) for m in modes)}.")
    if mode == "dict" and not parameters:
        raise ValueError(
            "PARAMETER_MODE is \"dict\" but PARAMETERS is empty. Dict mode "
            "takes every parameter from PARAMETERS and there is nothing to "
            "take — fill it in, or set PARAMETER_MODE = \"ask\".")

    method = str(dqdv_computation).strip().lower()
    if method not in methods:
        raise ValueError(f"DQDV_COMPUTATION is {dqdv_computation!r}; it must "
                         f"be one of {', '.join(repr(m) for m in methods)}.")

    if max_cycle is not None:
        if not isinstance(max_cycle, (int, np.integer)) or max_cycle < 1:
            raise ValueError(f"MAX_CYCLE is {max_cycle!r}; it must be None or "
                             f"a whole number of cycles, 1 or more.")
    if dqdv_bin_mV is not None:
        if not isinstance(dqdv_bin_mV, (int, float, np.number)) \
                or not np.isfinite(dqdv_bin_mV) or not 0.05 <= dqdv_bin_mV <= 50:
            raise ValueError(
                f"DQDV_BIN_MV is {dqdv_bin_mV!r}; it must be None (let the "
                f"measured profile choose) or a bin width in mV between "
                f"0.05 and 50.")
    if closure_sample is not None:
        if not isinstance(closure_sample, (int, np.integer)) or closure_sample < 3:
            raise ValueError(
                f"CLOSURE_SAMPLE is {closure_sample!r}; it must be None "
                f"(every half-cycle) or at least 3 — a median and a pass "
                f"fraction over fewer than three fits is not a measurement.")
    if not isinstance(run_peak_fitting, (bool, np.bool_)):
        raise TypeError(f"RUN_PEAK_FITTING is {run_peak_fitting!r}; it must "
                        f"be True or False.")

    settings = dict(
        data_dir=data_dir, output_base=output_base, files=files,
        pattern=pattern, parameter_mode=mode, parameter_count=len(parameters or {}),
        max_cycle=max_cycle, dqdv_computation=method, dqdv_bin_mV=dqdv_bin_mV,
        closure_sample=closure_sample, run_peak_fitting=bool(run_peak_fitting))

    if verbose:
        print(entry("data", os.path.basename(data_dir.rstrip("\\/")) or data_dir,
                    data_dir, value_width=34))
        print(entry("output", os.path.basename(output_base.rstrip("\\/")),
                    output_base, value_width=34))
        print(entry("files",
                    f"{len(files)} named" if files else "chosen at Cell 3a",
                    "" if files else f"pattern {pattern}", value_width=34))
        print(entry("parameters", mode,
                    "prompted, pre-filled from the file" if mode == "ask"
                    else f"{len(parameters or {})} taken from PARAMETERS",
                    value_width=34))
        print(entry("dQ/dV", method,
                    f"{dqdv_bin_mV} mV bins" if dqdv_bin_mV
                    else "bin width from the measured profile",
                    value_width=34))
        print(entry("peak fitting", "yes" if run_peak_fitting else "no",
                    (f"closure on {closure_sample} half-cycles"
                     if closure_sample else "closure on every half-cycle")
                    if run_peak_fitting else "cycling analysis only",
                    value_width=34))
        print(entry("cycles", f"to {max_cycle}" if max_cycle else "all",
                    "" if max_cycle else "no cap set", value_width=34))
    return settings


In [ ]:
# DEFINITIONS B — Q, V, I, T  ->  A DQ/DV CURVE
# =============================================================================
"""
Q, V, I, t  ->  a dQ/dV curve, and whether it can be trusted.

What is ported and what is new
------------------------------
PORTED VERBATIM from 1.8.7 Module 1, extracted programmatically rather than
retyped, because 1.9.0 must reproduce every fitted value bit for bit:

    classify_dqdv_profile   auto_preprocess_params   remove_spikes
    smooth_dqdv             rebin_dqdv               _strip_cv_hold

NEW, and measurement only — nothing here changes a processed curve:

    reversals()             every contiguous run where the potential moves
                            against the applied current: how much charge
                            passed, over what voltage band, at what current
    half_cycle_report()     assembles the above per half-cycle

Why reversals are inventoried rather than judged
------------------------------------------------
dQ/dV < 0 during charge means dV/dt < 0: the potential is moving backwards
while current flows. Three different things do that, and 1.8.x conflated them
under one threshold:

    at a redox peak, at full current, every cycle   a two-phase plateau
    at the voltage limit, current tapering          a constant-voltage hold
    away from any peak, at full current, sporadic   charge going elsewhere

Only the third is a fault. Telling them apart needs the peak list, which lives
downstream — so this module measures, records where and how much, and leaves
the classification to `analyse`. The numbers it records are the ones that
distinguish the three: on the P3 cells a healthy half-cycle passes 0 mAh/g
backwards at full current away from a peak, and a sick one passes 34-103.
"""


from dataclasses import dataclass, field

import numpy as np

import pandas as pd
from scipy.signal import savgol_filter


# =============================================================================
# PORTED FROM 1.8.7 MODULE 1 — do not edit without a bit-identity regression
# =============================================================================

EXCLUDE_CV_HOLD = True
CV_CURRENT_FRACTION = 0.99   # trailing |I| below this x the step median = hold
CV_DV_FRACTION = 0.10        # fallback: trailing |dV| below this x median
CV_MIN_POINTS = 3            # ignore shorter runs; not worth trimming

# Profile classification thresholds, from 1.8.7 Module 1.
_SHARP_IQR_FRACTION = 0.08      # IQR < 8% of range  -> two-phase plateau
_MODERATE_IQR_FRACTION = 0.20   # IQR < 20% of range -> moderate


def classify_dqdv_profile(df, verbose=True):
    """
    Classify the electrochemical profile of a dataset to guide dQ/dV 
    preprocessing.
    
    Examines the voltage profile shape across representative half-cycles 
    to determine whether the material exhibits sharp two-phase behaviour
    (e.g. LTO, LFP), moderate peaks, or broad solid-solution behaviour 
    (e.g. NMC, NNM layered oxides).
    
    The classification uses the voltage interquartile range (IQR) as a 
    fraction of the total voltage window, averaged across the first few 
    stable half-cycles. This approach is robust because:
    
    1. Two-phase materials spend most of their capacity at a nearly 
       constant voltage (the plateau), so the IQR of voltage values 
       within a half-cycle is small — typically <5% of the window.
    
    2. Solid-solution materials have continuously varying voltage, so 
       the IQR spans a substantial fraction of the window — typically 
       30-50%.
    
    3. Unlike dQ/dV statistics, this method doesn't suffer from the 
       problem of the "peak" being the bulk of the data distribution.
       The voltage profile is the ground truth that dQ/dV is derived 
       from.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw electrochemical data with 'Voltage', 'Cycle', and 'Step' 
        columns.
    verbose : bool
        Print classification diagnostics.
    
    Returns
    -------
    profile : dict
        {
            'class': str ('sharp', 'moderate', or 'broad'),
            'iqr_fraction': float (mean IQR/range across half-cycles),
            'plateau_fraction': float (fraction of points within 
                                       +/-20mV of median voltage),
            'description': str (human-readable summary)
        }
    """
    required = ['Voltage', 'Cycle', 'Step']
    if not all(c in df.columns for c in required):
        if verbose:
            print(entry("profile", "broad", "assumed — no Voltage/Cycle/Step column"))
        return {
            'class': 'broad',
            'iqr_fraction': np.nan,
            'plateau_fraction': np.nan,
            'description': 'Missing required columns; using default (broad) profile'
        }
    
    # --- Analyse representative half-cycles ---
    # Use cycles 2-5 (or whatever is available after cycle 1) to avoid
    # first-cycle formation effects. Examine both charge and discharge.
    cycles = sorted(df['Cycle'].dropna().unique())
    if len(cycles) < 2:
        analysis_cycles = cycles[:1]
    else:
        analysis_cycles = [c for c in cycles if c >= 2][:4]
        if not analysis_cycles:
            analysis_cycles = cycles[1:5]  # fallback
    
    iqr_fractions = []
    plateau_fractions = []
    
    for cycle in analysis_cycles:
        df_cycle = df[df['Cycle'] == cycle]
        for step in ['Charge', 'Discharge']:
            df_step = df_cycle[df_cycle['Step'] == step]
            voltages = df_step['Voltage'].dropna()
            
            if len(voltages) < 20:
                continue
            
            v_min, v_max = voltages.min(), voltages.max()
            v_range = v_max - v_min
            
            if v_range < 0.01:  # less than 10 mV range — skip
                continue
            
            # IQR as fraction of range
            q25, q75 = voltages.quantile(0.25), voltages.quantile(0.75)
            iqr = q75 - q25
            iqr_frac = iqr / v_range
            iqr_fractions.append(iqr_frac)
            
            # Plateau fraction: what proportion of data points sit 
            # within +/-20 mV of the median voltage?
            v_median = voltages.median()
            near_median = ((voltages >= v_median - 0.020) & 
                          (voltages <= v_median + 0.020))
            plat_frac = near_median.sum() / len(voltages)
            plateau_fractions.append(plat_frac)
    
    if not iqr_fractions:
        if verbose:
            print(entry("profile", "broad", "assumed — no half-cycle had 20+ records spanning 10 mV"))
        return {
            'class': 'broad',
            'iqr_fraction': np.nan,
            'plateau_fraction': np.nan,
            'description': 'No analysable half-cycles; using default (broad) profile'
        }
    
    mean_iqr_frac = np.mean(iqr_fractions)
    mean_plat_frac = np.mean(plateau_fractions)
    
    # --- Classify ---
    # Primary criterion: IQR fraction
    # Secondary criterion: plateau fraction (catches edge cases)
    if mean_iqr_frac < _SHARP_IQR_FRACTION or mean_plat_frac > 0.6:
        profile_class = 'sharp'
        description = (
            f'Sharp two-phase profile. '
            f'Typical of LTO, LFP, or other flat-plateau materials. '
            f'Spike removal disabled; smoothing window reduced.'
        )
    elif mean_iqr_frac < _MODERATE_IQR_FRACTION or mean_plat_frac > 0.35:
        profile_class = 'moderate'
        description = (
            f'Moderate peak sharpness. '
            f'Mixed or moderately defined phase transitions. '
            f'Spike threshold raised; smoothing window reduced.'
        )
    else:
        profile_class = 'broad'
        description = (
            f'Broad solid-solution profile. '
            f'Typical of layered oxides (NMC, NNM, etc.). '
            f'Standard preprocessing applied.'
        )
    
    out = {
        'class': profile_class,
        'iqr_fraction': mean_iqr_frac,
        'plateau_fraction': mean_plat_frac,
        'description': description
    }
    # `verbose` was accepted, documented, and never read. The profile class
    # is the ONLY decision this function makes and it sets the smoothing
    # window, the spike removal, the rebin width, the histogram bin width and
    # kernel, and downstream the baseline degree and the sigma bounds — so a
    # run that does not say which class it chose does not say how any of its
    # numbers were produced. It was silent for four releases.
    if verbose:
        print(entry("profile", profile_class,
                           f"IQR/range {mean_iqr_frac:.3f}, "
                           f"plateau fraction {mean_plat_frac:.2f}"))
        print(bullet(description))
    return out


def auto_preprocess_params(profile, user_params=None):
    """
    Select preprocessing parameters based on voltage profile classification.
    
    Returns a complete set of preprocessing parameters appropriate for the 
    detected profile type. User-supplied non-'auto' values override the 
    auto-selected defaults, allowing fine-tuning without losing the 
    benefit of auto-detection.
    
    Parameter logic by profile class:
    
    sharp (LTO, LFP):
        - Spike removal OFF: the genuine dQ/dV peaks are so extreme 
          relative to the background that any MAD-based despiking will 
          treat them as outliers. With dQ/dV values of thousands at the 
          plateau and near-zero elsewhere, there is no local context 
          for the rolling window to use.
        - Smoothing window 5: the peaks are very narrow in voltage 
          space, so a wide window attenuates the peak height and 
          distorts the shape. Window 5 with polyorder 3 preserves 
          the peak while removing point-to-point noise.
    
    moderate:
        - Spike threshold raised to 6.0 (from default 3.0): protects 
          sharper peaks while still catching genuine noise spikes.
        - Smoothing window 9: compromise between peak preservation and 
          noise reduction.
    
    broad (NMC, NNM):
        - Original defaults: window 15, spike threshold 3.0.
        - These were tuned for layered oxide cathodes and work well.
    
    Parameters
    ----------
    profile : dict
        Output from classify_dqdv_profile().
    user_params : dict or None
        User-supplied parameters from Cell 3. Any value that is not 'auto'
        overrides the auto-selected value.
    
    Returns
    -------
    params : dict
        Complete preprocessing parameter set with keys:
        smoothing_window, polyorder, spike_removal, spike_window_size,
        spike_threshold_multiplier, auto_detected (bool), profile_class (str)
    """
    if user_params is None:
        user_params = {}
    
    profile_class = profile.get('class', 'broad')
    
    # --- Auto-selected defaults by profile class ---
    if profile_class == 'sharp':
        auto = {
            'smoothing_window': 5,
            'polyorder': 3,
            'spike_removal': False,
            'spike_window_size': 5,
            'spike_threshold_multiplier': 10.0,
            'second_smooth_window': 11,
            'rebin_width_mV': 2.0,
        }
    elif profile_class == 'moderate':
        auto = {
            'smoothing_window': 9,
            'polyorder': 3,
            'spike_removal': True,
            'spike_window_size': 5,
            'spike_threshold_multiplier': 6.0,
            'second_smooth_window': None,
            'rebin_width_mV': None,
        }
    else:  # broad
        auto = {
            'smoothing_window': 15,
            'polyorder': 3,
            'spike_removal': True,
            'spike_window_size': 5,
            'spike_threshold_multiplier': 3.0,
            'second_smooth_window': None,
            'rebin_width_mV': None,
        }
    
    # --- Apply user overrides (non-'auto' values take priority) ---
    final = {}
    for key in auto:
        user_val = user_params.get(key, 'auto')
        if user_val == 'auto':
            final[key] = auto[key]
        else:
            final[key] = user_val
    
    # The histogram path's one parameter, chosen the same way everything else
    # here is: from the measured profile class. Only set if the caller has
    # not already fixed it.
    final.setdefault('histogram_bin_mV',
                     histogram_bin_for_profile(profile_class))
    final.setdefault('histogram_smooth_bins',
                     histogram_smooth_for_profile(profile_class))

    # --- Metadata ---
    final['auto_detected'] = True
    final['profile_class'] = profile_class
    final['profile_description'] = profile.get('description', '')
    final['iqr_fraction'] = profile.get('iqr_fraction', np.nan)
    final['plateau_fraction'] = profile.get('plateau_fraction', np.nan)
    
    return final


# --- Preprocessing functions (defined once, used everywhere) ---


def remove_spikes(series, window_size=5, threshold_multiplier=3.0):
    """
    Detect and replace spikes using rolling median and MAD.
    
    Uses median-based statistics rather than mean/std to prevent the spike 
    from contaminating its own detection window — a known failure mode of 
    mean-based rolling outlier detection.
    
    The MAD (median absolute deviation) is scaled by 1.4826 to give a 
    consistent estimator of the standard deviation for normally-distributed 
    data (Rousseeuw & Croux, 1993).
    
    Parameters
    ----------
    series : pd.Series
        The dQ/dV values to despike.
    window_size : int
        Rolling window size (should be odd for centred window).
    threshold_multiplier : float
        Number of scaled MADs for spike detection threshold.
    
    Returns
    -------
    despiked : pd.Series
        Cleaned series with spikes replaced by rolling median.
    n_spikes : int
        Number of spikes detected and replaced.
    """
    rolling_median = series.rolling(
        window=window_size, center=True, min_periods=1
    ).median()
    
    # MAD scaled to approximate std for normal distributions
    abs_dev = (series - rolling_median).abs()
    rolling_mad = abs_dev.rolling(
        window=window_size, center=True, min_periods=1
    ).median() * 1.4826
    
    # Floor to prevent division issues in constant regions
    mad_floor = series.abs().max() * 1e-6
    rolling_mad = rolling_mad.clip(lower=mad_floor)
    
    is_spike = abs_dev > (threshold_multiplier * rolling_mad)
    n_spikes = int(is_spike.sum())
    
    despiked = series.copy()
    despiked[is_spike] = rolling_median[is_spike]
    
    return despiked, n_spikes


def smooth_dqdv(values, window_size, polyorder=3):
    """
    Apply Savitzky-Golay smoothing to dQ/dV data.
    
    Handles edge cases: None/small window sizes, even windows (auto-corrected 
    to odd), insufficient data points, and polyorder >= window_size.
    
    Parameters
    ----------
    values : np.ndarray
        The dQ/dV values to smooth.
    window_size : int or None
        Savitzky-Golay window length. If None or <= 1, no smoothing applied.
    polyorder : int
        Polynomial order for the filter.
    
    Returns
    -------
    smoothed : np.ndarray
        Smoothed values (or original if smoothing not applicable).
    was_smoothed : bool
        Whether smoothing was actually applied.
    """
    if window_size is None or window_size <= 1:
        return values.copy(), False
    
    # Ensure window_size is odd (Savitzky-Golay requirement)
    if window_size % 2 == 0:
        window_size += 1
    
    if len(values) <= window_size:
        return values.copy(), False
    
    if polyorder >= window_size:
        polyorder = window_size - 1
    
    try:
        return savgol_filter(values, window_size, polyorder), True
    except Exception as e:
        print(f"  Warning: Savitzky-Golay smoothing failed ({e}). "
              f"Returning unsmoothed data.")
        return values.copy(), False


# How a bin's dQ/dV is estimated from the records inside it.
#
# 1.8.7 took the plain MEAN of the per-record dQ/dV values. Each record's
# value is dQi/dVi, so an unweighted mean gives a record spanning 0.1 mV the
# same say as one spanning 3 mV — and on a plateau the 0.1 mV records are
# both the most numerous and the least trustworthy, because their dV is at
# the cycler's resolution limit. The bin's actual charge per volt is
# sum(dQi)/sum(|dVi|), which is the |dV|-WEIGHTED mean of the same values.
#
# Measured on LTO cell A cycle 2 charge, sweeping the bin width 0.5-8 mV:
#
#             integrated area      apex height      notch at the apex
#   mean       73 - 91  (24%)      3422 -> 1752     present at 2 mV
#   weighted   77 - 82  ( 5%)      3909 -> 2739     gone at 2 mV
#
# An area that moves by a quarter when the bin width changes is not a
# measurement of anything, and the notch it leaves at the apex reads as a
# split peak. Set False to restore 1.8.7's mean.
REBIN_WEIGHTED = True


def _record_dv(voltage):
    """The voltage span each record stands for, by the mid-point rule."""
    v = np.asarray(voltage, float)
    if v.size < 2:
        return np.ones_like(v)
    e = np.empty(v.size + 1)
    e[1:-1] = 0.5 * (v[1:] + v[:-1])
    e[0], e[-1] = v[0], v[-1]
    return np.abs(np.diff(e))


# =============================================================================
# THE DERIVATIVE-FREE PATH
# =============================================================================
# Ratatosk's original curve is the CYCLER's own dQ/dV column, cleaned: sorted
# by voltage, rebinned, despiked, smoothed. That column is a numerical
# derivative, and on a flat plateau a numerical derivative is in trouble —
# dV sits in the denominator and goes to zero.
#
# Measured on the LTO triplicate: the cycler's dQ/dV column, integrated over
# its own voltage axis, recovers only 43-68% of the cycler's own delivered
# capacity. The charge is in the file; the derivative loses it.
#
# The alternative needs no derivative at all. If you bin the records by
# voltage and add up the CAPACITY each record carried, the sum in a bin
# divided by the bin width IS dQ/dV over that bin:
#
#         dQ/dV  ~  (sum of dq in the bin) / (bin width)
#
# and by construction the whole curve integrates back to the delivered
# capacity, because every record's charge is counted exactly once. That is
# the point: integral fidelity is 1 by arithmetic rather than by luck.
#
# Flores and Clark (J. Electrochem. Soc. 173, 120520, 2026) give this as a
# histogram of voltage values scaled by total capacity, which is the same
# thing when capacity is sampled at uniform intervals. The dq-weighted form
# implemented here is their recommended correction for the case where it is
# not — a rate step, a sampling change, a constant-voltage hold — and it is
# exact rather than proportional, so it is used unconditionally.
#
# What this method does NOT need, and why that matters:
#
#   * no smoothing, so no smoothing parameters chosen by eye;
#   * no despiking;
#   * no `orient_dqdv`, because a histogram does not care which way the
#     voltage was moving when a record was taken — the non-monotonic plateau
#     that inverts a derivative simply puts its charge in the right bin;
#   * one parameter, the bin width, which means exactly what it says.
#
# Measured on the same LTO half-cycles: integral fidelity 0.994-0.996 at
# every bin width from 0.5 to 5 mV, against 0.62-0.71 for the derivative.

HISTOGRAM_BIN_MV = 1.0
# ...but the right bin width depends on the narrowest feature that has to be
# resolved, exactly as the rebin width already did. A bin should be a small
# fraction of a peak's width: too fine and the histogram is sparse, noisy and
# slow to fit; too coarse and it blurs shoulders together.
#
#   sharp     LTO/LFP, FWHM ~15 mV        1 mV   ~15 bins across the peak
#   moderate                              2 mV
#   broad     layered oxides, FWHM 60+    5 mV   ~12 bins across the peak
#
# Measured: on a broad P3 profile, 1 mV bins over a 2.15 V window are mostly
# empty, and fitting the resulting noise took several times longer than the
# same half-cycle at a sensible width — for no extra information, because the
# features are 30-200 mV wide.
HISTOGRAM_BIN_BY_PROFILE = {"sharp": 1.0, "moderate": 2.0, "broad": 5.0}


def histogram_bin_for_profile(profile, default=HISTOGRAM_BIN_MV):
    """Bin width in mV for a measured profile class. See the table above."""
    cls = (profile.get("class") if isinstance(profile, dict) else profile)
    return float(HISTOGRAM_BIN_BY_PROFILE.get(str(cls), default))
# Flores and Clark report good smoothness and voltage reconstruction with a
# Gaussian kernel of one — a weighted average of each bin with its immediate
# neighbours. 0 disables it. Anything larger starts reintroducing the
# parameter-chosen-by-eye problem this method exists to avoid.
HISTOGRAM_SMOOTH_KERNEL = 1
# ...and how wide it should be depends on the profile, for the same reason the
# bin width does. A histogram's bin-to-bin scatter is COUNTING noise, and peak
# DETECTION runs on that scatter unless it is damped. On a sharp profile the
# peak towers over the noise and no amount of kernel changes what is found; on
# a broad one the features are shallow and 1 mV of scatter looks like a
# shoulder.
#
# Measured on P3 cell A, half-width in bins against peaks detected per
# half-cycle (the derivative path finds 5, range 4-7):
#
#     1 -> 13     3 -> 12     4 -> 7     6 -> 6     8 -> 6
#
# 6 gives parity with the derivative path. Integral fidelity over that whole
# sweep moves from 0.9403 to 0.9379 — a quarter of a percent, and it is an
# edge effect of the zero-padded convolution, not a loss of counted charge.
# BROAD RAISED 6 -> 16 IN 1.9.0.26, measured on the NNM (P3 NaNiMnO2)
# triplicate — 100 cycles, CE 84-89%, exactly the noisy broad curve this
# entry is for. Detection was finding a different number of peaks almost
# every cycle (discharge sd 1.16 peaks, range 3-9), which is what puts a
# reference peak list out of step with the rest of the run.
#
# The test was not "does the curve look smoother" — it was whether a feature
# PERSISTS. A real redox feature appears in nearly every cycle at the same
# voltage; noise appears sporadically. Clustering every detected peak across
# 100 cycles of NNM cell C:
#
#   smoothing   discharge count sd   persistent (>=80% of cycles)
#      6 (old)        1.16            2.026, 3.153, 3.547
#     12             0.85             2.031, 3.221, 3.547
#     16 (new)       0.61             2.036, 3.221, 3.547
#     20             0.55             2.039, 3.220, 3.547
#
# The persistent set does not change. Every real feature survives to 20 bins,
# the scatter halves by 16, and on CHARGE a third feature at 3.996 V crosses
# into persistence at 16 (86% of cycles) having been below it at 6 — so the
# smoothing finds one more real feature than it loses, which is none.
# 16 rather than 20 because the gain flattens and there is no reason to
# smooth further than the evidence supports.
#
# This CANNOT affect LTO. The value is keyed to the profile class, and the
# classifier separates the two without overlap: LTO measures IQR/range
# 0.019-0.030 with a plateau fraction of 0.64-0.68 and classifies "sharp"
# (kernel 1); NNM measures 0.386-0.444 with a plateau fraction of 0.02-0.09
# and classifies "broad". Verified unchanged on the LTO triplicate.
HISTOGRAM_SMOOTH_BY_PROFILE = {"sharp": 1, "moderate": 3, "broad": 16}


def histogram_smooth_for_profile(profile, default=HISTOGRAM_SMOOTH_KERNEL):
    """Smoothing half-width in bins for a measured profile class."""
    cls = (profile.get("class") if isinstance(profile, dict) else profile)
    return int(HISTOGRAM_SMOOTH_BY_PROFILE.get(str(cls), default))
# A bin much narrower than the voltage step between records produces empty
# bins and a sparse, noisy curve. Below this multiple of the median record
# spacing the requested width is widened, and the fact is reported.
HISTOGRAM_MIN_BIN_FACTOR = 1.0

# ...and the MEDIAN is the wrong statistic to widen against, because it
# guarantees that half the curve is under-filled. Records arrive at a fixed
# charge interval (constant current, time-based logging), so their voltage
# spacing is dQ_per_record / (dQ/dV): narrow on a peak, wide on a shelf, by
# the same factor the curve itself varies. Widening to the median leaves the
# low half of the curve with fewer records than bins, the bins there alternate
# between one record and none, and the 16-bin smoothing turns that into a
# ripple at the beat period between the record spacing and the bin width.
#
# Measured, NMC111 cell A cycle 5 charge, the 4.10-4.45 V shelf:
#
#   path / bin            empty bins   mean occupancy   maxima found on shelf
#   derivative (1.8.7)         -             -                    0
#   histogram, 6.3 mV        60/223         0.87                  4
#   histogram, 8 mV          36/175         1.11                  0
#   histogram, 10 mV         24/140         1.39                  0
#   histogram, 20 mV         11/70          2.79                  0
#
# The raw Q(V) across that shelf is a straight line (dQ per record constant to
# 0.45%, dV per record 6.6-7.1 mV), so there is nothing there to resolve. The
# four maxima are the beat, and they are gone the moment the bins can be
# filled. They are also absent from the derivative path, which has no bins to
# beat against — which is how the artefact was first noticed.
#
# Mean occupancy is exactly `n_records * width / span`, so requiring a minimum
# mean occupancy is the same thing as widening to the MEAN record spacing.
# 1.0 is the floor with a name: never return a curve with fewer measurements
# in it than bins. It is not a smoothing parameter and it is not tuned to a
# chemistry -- it is the same statement as `sigma_min` from the bin width and
# `WIDTH_MIN_SAMPLES` from the sampling interval.
# THE BIN WIDTH IS SET FROM AN AVERAGE OVER A CURVE WHOSE RECORD DENSITY
# VARIES A HUNDREDFOLD, AND ON A SHARP PROFILE THE PEAK PAYS FOR THE SHELF.
# Measured in 1.9.0.60; the constant is UNCHANGED, because no value of it
# fixes this and the honest answer is to report the limit.
#
# `w_occ` below asks for one record per bin ON AVERAGE across the window. The
# paragraph under `counts` already explains why that average is not the
# quantity that matters: records arrive at a fixed charge interval, so their
# voltage spacing is dQ_per_record / (dQ/dV) and is widest exactly where the
# curve is lowest. The average is therefore set by the SHELF, and the PEAK is
# binned at the shelf's scale.
#
# LTO cell A cycle 5 charge, 373 records: 220 of them (59%) lie in the 13 mV
# across the peak's half maximum, 0.06 mV apart. The rule asks for 3.49 mV
# bins, 3.26 mV after rounding — fifty times coarser than the records there —
# and 236 of the 373 bins hold nothing at all. The peak is described by five
# bins whose occupancies are 1, 107, 47, 39, 26: the leading flank, which is
# where the asymmetry ratio comes from, is one record.
#
# SWEPT over bin width on the LTO triplicate, 64 half-cycles, same detection
# and same fitter, with this floor disabled so the request is honoured:
#
#     bin mV   FWHM/bins   fitted FWHM   sigma   sigma_r   charge k    R2
#       0.50       9.5         4.56       1.89     1.71       5.5     0.9596
#       1.00       7.0         5.57       2.34     3.35       8.3     0.9841
#       2.00       6.0         7.82       3.77     3.16      12.1     0.9940
#       3.26*      5.0        11.12       4.82     4.86       3.4     0.9972
#       5.00       5.0        14.53       6.17     6.35       1.9     0.9979
#                                                       (* what this rule picks)
#
# THE FITTED FWHM IS 2.9x THE BIN WIDTH AT EVERY WIDTH TESTED, over a factor
# of ten. It is the histogram's width, not the peak's. The asymmetry ratio has
# no stable value either — 1.9 to 12.1 on charge, and within a single width it
# ranges 0.11 to 20.0, which is the bound.
#
# AND R2 RISES MONOTONICALLY AS THE BINS COARSEN, 0.960 to 0.998, which is the
# trap: coarser bins are smoother, so a smooth model fits them better. R2 here
# measures how much the histogram has smoothed the data. It is the third time
# in this project that R2 has moved the opposite way to the answer.
#
# WHY NOT JUST BIN FINER. At 0.5 mV the peak gets 9.5 bins — but the shelf is
# then 99% empty, and empty shelf bins are what the quantisation ripple of
# 1.9.0.56 was made of. There is no width that serves both, because the two
# regions differ by two orders of magnitude in record density. Equal-occupancy
# binning is the obvious non-uniform answer and was measured in 1.9.0.55: it
# was worse, with 143 of 186 LTO bins degenerate.
#
# SO THIS IS A LIMIT OF THE MEASUREMENT, not of the code. The remedy is at the
# cycler — log on dV as well as dt, so records are spread in voltage instead of
# piling up on the plateau — and until then the widths on a sharp profile are
# not measurements and the report says so (`report._lineshape_sentence`).
#
# IT IS CONFINED TO `sharp`. Bins across the fitted FWHM, production build:
# LTO 3.7-3.8 (57-70% flagged `width_undersampled`), NNM 17.0-21.3 (0-1 of
# 3834 flagged), NMC 27.7-31.4 (none). The AREAS are unaffected — integral
# fidelity is 0.994-0.996 at every width from 0.5 to 5 mV — so what is
# withheld is the width and its ratio, not the capacity.
HISTOGRAM_MIN_OCCUPANCY = 1.0

# --- how a record's charge is assigned to a bin ---------------------------
# COUNTING (False) drops the whole of an increment's charge into the bin that
# contains the increment's MIDPOINT voltage. A bin's value is then an integer
# multiple of dq_per_record / width, and where the bins are finer than the
# records that integer is 0 or 1: the curve becomes a telegraph signal and the
# smoothing kernel turns it into a ripple at the beat period between the
# record spacing and the bin width. Measured on NMC111 cell A cycle 5 charge,
# 4.15-4.42 V, at 7.18 mV bins: 108 108 107 107 107 108 108 108 109 111 113
# 116 119 122 -- against a raw Q(V) over that span which is a straight line.
#
# EXACT (True) uses the fact that an increment is not a point. It spans
# [v_i, v_i+1], and the charge that passed inside a bin's edges is the part of
# that span the bin covers. Spreading each increment across the bins it
# actually overlaps is exact charge accounting -- it conserves the total by a
# telescoping sum, it introduces no parameter, and it reduces to the counting
# behaviour whenever an increment lies inside one bin. It is the same thing as
# interpolating Q(V) at the bin edges and differencing, which is what the
# quantity means. The same half-cycle, same bins, exact assignment:
# 110 110 111 111 112 113 113 113 113 113 112 112 112 112.
#
#   NMC111 cell A charge, identical bin width      counting     exact
#     empty bins                                     44           0
#     curvature scatter on the shelf                 0.28%        0.07%
#     maxima detected on the 4.1-4.45 V shelf        2            0
#     components detected in the half-cycle          6            3
#
# Zero-span increments -- a plateau where the recorded voltage does not move,
# which is most of an LTO half-cycle -- carry their whole charge at one
# voltage and go wholly into the bin containing it, as before. That case is
# why the histogram exists and it is not changed here.
HISTOGRAM_EXACT_CHARGE = True


def _charge_in_bins(v0, v1, dq, edges):
    """
    Charge in each bin, spreading each increment over the span it covers.

    `v0`/`v1` are the voltages either side of an increment carrying `dq`; the
    order of the pair does not matter. Returns one total per bin.

    Built as a cumulative: G(e) is the charge that passed below voltage `e`,
    so the charge in a bin is a difference of two values of G and the total
    over all bins telescopes to the charge that entered the calculation.
    """
    lo = np.minimum(v0, v1)
    hi = np.maximum(v0, v1)
    span = hi - lo
    moving = span > 0
    G = np.zeros(edges.size, dtype=float)

    if np.any(moving):
        l = lo[moving]; h = hi[moving]; w = dq[moving]
        # Chunked so a long half-cycle cannot build a huge intermediate.
        chunk = max(1, int(4_000_000 // max(edges.size, 1)))
        for a in range(0, l.size, chunk):
            b = min(a + chunk, l.size)
            frac = np.clip(
                (edges[None, :] - l[a:b, None]) / (h[a:b, None] - l[a:b, None]),
                0.0, 1.0)
            G += (w[a:b, None] * frac).sum(axis=0)

    if np.any(~moving):
        # A stationary increment is all at one voltage: it belongs to the bin
        # containing it, and to every cumulative value above that bin.
        k = np.searchsorted(edges, lo[~moving], side="right")
        k = np.clip(k, 1, edges.size - 1)
        add = np.zeros(edges.size, dtype=float)
        np.add.at(add, k, dq[~moving])
        G += np.cumsum(add)

    return np.diff(G)


def histogram_dqdv(voltage, capacity, *, bin_width_mV=None,
                   window=None, smooth_kernel=None, sign=1.0,
                   phase=0.0):
    """
    dQ/dV without differentiating anything. See the block comment above.

    `voltage` and `capacity` are one half-cycle IN ACQUISITION ORDER — the
    order matters, because each record's charge increment is the difference
    from the record before it. `sign` is applied at the end and is Ratatosk's
    display convention (+1 on the half-cycle labelled Charge, -1 on
    Discharge); the magnitude does not depend on it.

    Returns `(centres, dqdv, info)`. `info` carries the bin width actually
    used, the number of empty bins, and the charge that fell outside `window`
    — all three are how you tell a good binning from a bad one.
    """
    # READ THE MODULE CONSTANTS AT CALL TIME, not as default arguments. A
    # default is evaluated once when the function is defined, so
    # `smooth_kernel=HISTOGRAM_SMOOTH_KERNEL` freezes whatever the constant
    # was at import and silently ignores any later change — which is the same
    # import-time binding that once left every cycling figure at the wrong
    # size. Setting the constant has to work, because that is how a build
    # flag is meant to be used.
    if bin_width_mV is None:
        bin_width_mV = HISTOGRAM_BIN_MV
    if smooth_kernel is None:
        smooth_kernel = HISTOGRAM_SMOOTH_KERNEL

    v = np.asarray(voltage, float)
    q = np.asarray(capacity, float)
    m = np.isfinite(v) & np.isfinite(q)
    v, q = v[m], q[m]
    info = dict(bin_width_mV=float(bin_width_mV), n_bins=0, n_empty=0,
                charge_in_window=np.nan, charge_total=np.nan, widened=False,
                widened_by="", unvisited_mV=0.0)
    if v.size < 5:
        return np.empty(0), np.empty(0), info

    # Each record carries the charge passed since the previous one. The first
    # record carries none — it is the start of the count, not an increment.
    dq = np.abs(np.diff(q))
    v_lo_all, v_hi_all = v[:-1], v[1:]  # the span the charge was passed ACROSS
    v_mid = 0.5 * (v[1:] + v[:-1])      # the charge was passed BETWEEN them
    good = np.isfinite(dq) & np.isfinite(v_mid)
    dq, v_mid = dq[good], v_mid[good]
    v_lo_all, v_hi_all = v_lo_all[good], v_hi_all[good]
    if dq.size < 4 or not np.any(dq > 0):
        return np.empty(0), np.empty(0), info
    info["charge_total"] = float(dq.sum())

    lo, hi = (float(np.min(v_mid)), float(np.max(v_mid))) if window is None \
        else (float(window[0]), float(window[1]))
    inside = (v_mid >= lo) & (v_mid <= hi)
    info["charge_in_window"] = float(dq[inside].sum())
    if not np.any(inside) or hi <= lo:
        return np.empty(0), np.empty(0), info

    # --- BIN ONLY WHAT THE HALF-CYCLE ACTUALLY VISITED -------------------
    # The window is computed once per DATASET, deliberately (see
    # `voltage_window`): trimming each half-cycle to its own extremes removes
    # real data from every one of them. But a dataset window is a union, and
    # an individual half-cycle need not reach both ends of it -- on reversing
    # the current the cell jumps by its overpotential, carrying no charge, so
    # a charge step can begin a long way above the window's foot.
    #
    # Measured, NMC111 cell A cycle 5: the discharge before it ended at 3.000
    # V under load, the charge step opens at 3.2847 V with Q = 0.00 for its
    # first two records, and the window starts at 3.050 V. Thirty-seven bins
    # -- 233 mV -- therefore contained NO RECORDS AT ALL, the curve was flat
    # zero across them, and the cubic baseline, with nothing to constrain it,
    # climbed to 48 mAh/g/V in there and booked 5 mAh/g of capacity in a
    # region the cell never entered. That is not a fitting problem; it is a
    # model defined outside its data.
    #
    # So the GRID is built over the visited span, clipped to the window.
    # Empty bins INSIDE that span are kept -- they are real gaps in a region
    # the cell did traverse, and they carry information. This is not the
    # per-half-cycle trim that `voltage_window` warns about: nothing measured
    # is discarded, and the dataset-wide 50 mV trim still applies.
    _seen_lo = float(np.min(np.minimum(v_lo_all[inside], v_hi_all[inside])))
    _seen_hi = float(np.max(np.maximum(v_lo_all[inside], v_hi_all[inside])))
    _lo2, _hi2 = max(lo, _seen_lo), min(hi, _seen_hi)
    if _hi2 - _lo2 > 0:
        info["unvisited_mV"] = ((_lo2 - lo) + (hi - _hi2)) * 1000.0
        lo, hi = _lo2, _hi2

    # A bin narrower than the records themselves cannot be filled. Two floors,
    # and the wider one wins: the median record spacing (local scale), and the
    # width that gives at least HISTOGRAM_MIN_OCCUPANCY records per bin on
    # average across the window (global count). See both constants above.
    step = float(np.median(np.abs(np.diff(np.sort(v_mid))))) if v_mid.size > 2 \
        else 0.0
    width = float(bin_width_mV) / 1000.0
    n_rec = int(np.count_nonzero(inside))
    w_occ = (float(HISTOGRAM_MIN_OCCUPANCY) * (hi - lo) / n_rec) \
        if n_rec > 0 else 0.0
    floor = max(HISTOGRAM_MIN_BIN_FACTOR * step, w_occ)
    if floor > 0 and width < floor:
        width = floor
        info["widened"] = True
        info["widened_by"] = ("occupancy" if w_occ >= HISTOGRAM_MIN_BIN_FACTOR * step
                              else "record spacing")
        info["bin_width_mV"] = width * 1000.0

    # THE PHASE OFFSET IS APPLIED HERE, AFTER THE VISITED-SPAN CLIP.
    # `phase_stable` asks whether a detected feature survives sliding the bin
    # grid by a fraction of a bin — a wiggle that moves with the grid is the
    # grid, not the material. `_rebin_at_phase` implemented that by shifting
    # the WINDOW, and the clip above then overwrote `lo` with `_seen_lo`, the
    # lowest voltage the half-cycle actually visited, which does not depend on
    # the phase at all. Whenever a half-cycle opens above the window's foot —
    # the normal case for a charge step, and the case the clip was written
    # for — all four phases produced a byte-identical grid, every candidate
    # appeared at exactly the same voltage in each, and PHASE_INVARIANCE
    # rejected nothing. Offsetting the grid anchor itself cannot be undone by
    # a later clip.
    _ph = float(phase or 0.0)
    if _ph:
        lo = lo - (_ph - np.floor(_ph)) * width
    n_bins = max(4, int(np.ceil((hi - lo) / width)))
    edges = lo + width * np.arange(n_bins + 1)
    # `weights=dq` is the whole method: the bin's value is the CHARGE that
    # passed inside it, not the number of records that happened to land there.
    if HISTOGRAM_EXACT_CHARGE:
        totals = _charge_in_bins(v_lo_all[inside], v_hi_all[inside],
                                 dq[inside], edges)
    else:
        totals, _ = np.histogram(v_mid[inside], bins=edges, weights=dq[inside])
    centres = 0.5 * (edges[:-1] + edges[1:])
    y = totals / width                      # mAh/g per volt

    # OCCUPANCY: how many raw records actually landed in each bin.
    #
    # `weights=dq` above is what makes this a charge histogram rather than a
    # counting one, and it is right — but it means the curve carries no record
    # of how much MEASUREMENT is behind each value. That matters, because the
    # bin width is chosen from the MEDIAN record spacing over the half-cycle
    # while the actual spacing varies with dQ/dV across it: records arrive at
    # a fixed charge interval (constant current, time-based logging), so their
    # voltage spacing is dQ_per_record / (dQ/dV) and is WIDEST exactly where
    # the curve is lowest. A bin width that is comfortable on a peak is finer
    # than the records on a shelf, and the bins there alternate between one
    # record and none. Smoothing then spreads charge across the empty ones and
    # returns a curve whose fine structure is the beat between the record
    # spacing and the bin width — a ripple of order 1/occupancy, at a period
    # that moves as the record spacing does, i.e. between cycles.
    #
    # Measured on NMC111 cell A charge, 5 mV requested and widened to the
    # median: occupancy 1.06 rec/bin below 3.9 V but 0.78 above 4.1 V with 26%
    # of bins empty (cycle 5), falling to 0.57 and 42% by cycle 13 as capacity
    # fades and there are fewer records to go round. The raw Q(V) over that
    # same span is a straight line — dQ per record constant to 0.45%, dV per
    # record 6.6-7.1 mV — so there is nothing there to resolve, and the four
    # to five maxima detection finds on that shelf are the beat, not chemistry.
    #
    # So the count is carried alongside the curve and detection is entitled to
    # refuse a region that has fewer measurements than bins. Same family as
    # `sigma_min` from the bin width and `WIDTH_MIN_SAMPLES` from the sampling
    # interval: a statement about what the measurement can support.
    counts, _ = np.histogram(v_mid[inside], bins=edges)
    info["n_bins"] = int(n_bins)
    info["n_empty"] = int(np.sum(totals == 0))
    info["occupancy"] = counts.astype(float)
    info["occupancy_median"] = float(np.median(counts)) if counts.size else np.nan
    # The charge one record carries. On a constant-current step with
    # time-based logging this is a constant (measured 0.45% CV on NMC111), and
    # it is the QUANTUM of the histogram: a bin's value can only be an integer
    # multiple of `dq_per_record / width`. See `quantisation_ripple`.
    _dqi = dq[inside]
    info["dq_per_record"] = float(np.median(_dqi[_dqi > 0])) \
        if np.any(_dqi > 0) else np.nan

    # THE KERNEL IS TRIMMED TO FIT THE CURVE. `np.convolve(..., mode="same")`
    # returns an array of length max(len(signal), len(kernel)) — NOT the
    # length of the signal, which is what "same" reads as. A half-cycle short
    # enough to give fewer bins than the kernel has taps therefore came back
    # with MORE dQ/dV values than voltages, and detection died on
    # `operands could not be broadcast together with shapes (12,) (13,)`
    # rather than skipping a half-cycle that was never a curve. Two of P3
    # cell A's 440 half-cycles do this: five records each, binned at a
    # widened 185 mV into twelve bins, against a thirteen-tap kernel.
    #
    # Trimmed rather than skipped, so the behaviour degrades smoothly: a
    # curve with room for a narrower kernel gets one, and a curve with room
    # for none is left unsmoothed. Either way the array lengths agree, and
    # the binomial kernel still sums to one, so the area is still conserved.
    if smooth_kernel and smooth_kernel > 0 and y.size > 2:
        smooth_kernel = int(min(int(smooth_kernel), (y.size - 1) // 2))
    if smooth_kernel and smooth_kernel > 0 and y.size > 2:
        # A BINOMIAL kernel — the discrete Gaussian — applied to the RESULT,
        # not to the voltage record, so it cannot change which bin a record's
        # charge was counted in. It sums to exactly one, so it conserves the
        # total area exactly: the capacity accounting of this method survives
        # any amount of it. `smooth_kernel` is its half-width in bins.
        #
        # This is not the trial-and-error smoothing the histogram method
        # exists to avoid. That kind is applied to the VOLTAGE before
        # differentiating, where it changes the derivative and the integral
        # together and its parameters are unrecoverable. This is applied
        # afterwards, changes nothing that has been counted, and exists only
        # so that peak DETECTION is not run on counting noise.
        k = np.array([1.0])
        for _ in range(2 * int(smooth_kernel)):
            k = np.convolve(k, [0.5, 0.5])
        y = np.convolve(y, k / k.sum(), mode="same")

    # Belt and braces. Every return above this line is a pair built from the
    # same `edges`, and the one operation that could break that is fixed
    # above — but a curve whose two halves disagree in length is the kind of
    # thing that should never leave this function silently.
    if y.size != centres.size:                                  # pragma: no cover
        raise AssertionError(
            f"histogram_dqdv produced {centres.size} voltages and {y.size} "
            f"dQ/dV values; they are built from one set of bin edges and "
            f"cannot differ.")
    return centres, sign * y, info


def binomial_kernel(half_width):
    """The discrete Gaussian used to smooth a histogram, normalised to 1."""
    k = np.array([1.0])
    for _ in range(2 * int(half_width)):
        k = np.convolve(k, [0.5, 0.5])
    return k / k.sum()


def quantisation_ripple(centres, dqdv, *, dq_per_record, bin_width_V,
                        smooth_kernel, window_bins=None):
    """
    How much structure would this binning invent on a perfectly smooth curve?

    THE NULL MODEL. A histogram dQ/dV counts records into voltage bins, and
    each record carries the same charge, so a bin's value before smoothing can
    only be an integer multiple of `dq_per_record / bin_width`. Where the bins
    hold one record or none that quantum is the SIZE OF THE CURVE — on NMC111
    cell A cycle 5 charge, 0.77 mAh/g into a 7.2 mV bin is 107 mAh/g/V against
    a shelf standing at 118 — and the bins alternate between 0, one quantum
    and two. Smoothing damps that telegraph signal but cannot remove it: what
    comes out is a ripple at the beat period between the record spacing and
    the bin width, which moves between cycles because the record spacing does.

    So: take the delivered curve as the truth, work out where records WOULD
    have landed if it were exactly that smooth (equal charge intervals along
    its own integral), bin and smooth them the same way, and compare. The
    difference is structure this binning manufactures from a smooth input.

    Nothing here is fitted or tuned. The inputs are this half-cycle's own bin
    width, its own smoothing kernel and its own charge per record; the output
    is in the same units as the curve, so detection can ask the only question
    that matters — is this maximum bigger than what the binning invents?

    Returns a per-bin ripple amplitude (local RMS), aligned with `centres`.
    """
    y = np.abs(np.asarray(dqdv, float))
    v = np.asarray(centres, float)
    n = y.size
    if n < 8 or not np.isfinite(dq_per_record) or dq_per_record <= 0 \
            or not np.isfinite(bin_width_V) or bin_width_V <= 0:
        return np.zeros(max(n, 0))

    # Where the records would fall on a curve exactly this smooth.
    q = np.concatenate(([0.0], np.cumsum(0.5 * (y[1:] + y[:-1]) * np.diff(v))))
    q_tot = float(q[-1])
    n_rec = int(round(q_tot / float(dq_per_record)))
    if n_rec < 4:
        return np.zeros(n)
    q_rec = (np.arange(1, n_rec + 1) - 0.5) * float(dq_per_record)
    q_rec = q_rec[q_rec <= q_tot]
    if q_rec.size < 4:
        return np.zeros(n)
    v_rec = np.interp(q_rec, q, v)

    edges = np.empty(n + 1)
    edges[:-1] = v - 0.5 * bin_width_V
    edges[-1] = v[-1] + 0.5 * bin_width_V
    counts, _ = np.histogram(v_rec, bins=edges)
    y_null = counts * float(dq_per_record) / float(bin_width_V)

    # Smooth BOTH the same way, so the comparison isolates the quantisation
    # and not the kernel's own low-pass or its edge droop.
    kk = int(smooth_kernel or 0)
    if kk > 0 and n > 2:
        kk = int(min(kk, (n - 1) // 2))
    if kk > 0 and n > 2:
        k = binomial_kernel(kk)
        y_null = np.convolve(y_null, k, mode="same")
        y_ref = np.convolve(y, k, mode="same")
    else:
        y_ref = y

    resid = y_null - y_ref

    # Local RMS, over a span wide enough to contain a beat period but narrow
    # enough that the amplitude still tracks the curve (it scales as
    # 1/occupancy, so it is largest where the curve is lowest).
    if window_bins is None:
        window_bins = max(9, 2 * (kk if kk > 0 else 4) + 1)
    w = int(window_bins) | 1
    pad = w // 2
    padded = np.pad(resid ** 2, pad, mode="edge")
    csum = np.concatenate(([0.0], np.cumsum(padded)))
    ms = (csum[w:] - csum[:-w]) / w
    return np.sqrt(np.maximum(ms[:n], 0.0))


def dqdv_sign(step):
    """
    Ratatosk's display convention: +1 on Charge, -1 on Discharge.

    Read off the STEP LABEL, not the direction of travel — on an anode with
    swapped labels the half-cycle called Charge is the one whose voltage
    falls, and it still carries a positive dQ/dV. Verified against both the
    LTO (labels swapped) and P3 (not swapped) exports.
    """
    return 1.0 if str(step).lower().startswith("charge") else -1.0


def rebin_dqdv(voltage, dqdv, bin_width_mV=2.0, weighted=None):
    """
    Re-bin dQ/dV onto a uniform voltage grid.

    On flat-plateau materials (LTO, LFP) the cycler records many points at
    near-identical voltages, because the voltage barely changes between
    successive capacity increments. That produces duplicate and
    near-duplicate voltages carrying wildly different dQ/dV, depending on
    whether a sub-resolution dV rounded up or down. Savitzky-Golay cannot
    repair it, because it works on point index and the voltage spacing is
    grossly uneven; binning onto a regular grid is what makes the curve a
    function of voltage again.

    The bin's value is the |dV|-weighted mean of the records in it, which is
    sum(dQ)/sum(|dV|) — the charge that bin actually passed, per volt. See
    REBIN_WEIGHTED for what the unweighted mean did instead.

    Returns `(v_binned, dqdv_binned)`; bins holding no record are dropped.
    """
    if weighted is None:
        weighted = REBIN_WEIGHTED
    bin_width_V = bin_width_mV / 1000.0
    voltage = np.asarray(voltage, float)
    dqdv = np.asarray(dqdv, float)
    v_min, v_max = voltage.min(), voltage.max()

    bin_edges = np.arange(v_min, v_max + bin_width_V, bin_width_V)
    bin_indices = np.digitize(voltage, bin_edges) - 1
    bin_indices = np.clip(bin_indices, 0, len(bin_edges) - 2)
    w = _record_dv(voltage) if weighted else np.ones_like(voltage)

    v_binned = []
    dqdv_binned = []

    for i in range(len(bin_edges) - 1):
        mask = bin_indices == i
        if not mask.any():
            continue
        v_binned.append((bin_edges[i] + bin_edges[i + 1]) / 2.0)
        y, ww = dqdv[mask], w[mask]
        ok = np.isfinite(y) & np.isfinite(ww)
        tot = ww[ok].sum() if ok.any() else 0.0
        if ok.any() and tot > 0:
            dqdv_binned.append(float(np.sum(y[ok] * ww[ok]) / tot))
        else:
            # Every record in the bin sits at one voltage, so there is no
            # span to weight by. The mean is all that is left.
            dqdv_binned.append(float(np.nanmean(y)) if ok.any() else np.nan)

    return np.array(v_binned), np.array(dqdv_binned)


def plateau_structure(voltage, capacity, n=40):
    """
    Read the voltage curve directly: is there really a second plateau?

    A second dQ/dV peak REQUIRES a second plateau in V(Q) — the voltage must
    flatten, steepen, and flatten again. This asks that question of V(Q)
    rather than of dQ/dV, and needs no smoothing, binning or orientation to
    do it, because Q is a monotonic integral where V is a quantised
    measurement. It is therefore free of every processing choice that could
    manufacture the feature being tested.

    `voltage` and `capacity` are the RAW records of one half-cycle.

    Returns a dict with `flattest_V` (where |dV/dQ| is smallest — the peak),
    `second_flattest_V` (the flattest point at least 15 mAh/g away), their
    separation, and `n_minima`, the count of interior local minima of
    |dV/dQ|.

    How to read it, from the LTO triplicate:

      * cell B's charge plateau is flattest at 1.546 +/- 1 mV in cycles
        3-10, with the second-flattest 1-3 mV away. ONE plateau, sampled
        twice.
      * `n_minima` runs 2-8 per half-cycle at different voltages every cycle
        and every cell. A phase transition does not move, so those are the
        cycler's 0.1 mV voltage resolution showing through 1/(dV/dQ), not
        chemistry.

    Two further tests no function can do for you, and both matter more than
    anything above: does the feature recur at the SAME voltage in later
    cycles, and does it appear in the other cells of the triplicate? A
    feature that fails either is not a property of the material.
    """
    v = np.asarray(voltage, float)
    q = np.asarray(capacity, float)
    m = np.isfinite(v) & np.isfinite(q)
    v, q = v[m], q[m]
    if v.size < 20:
        return {"flattest_V": np.nan, "second_flattest_V": np.nan,
                "separation_mV": np.nan, "n_minima": 0}
    o = np.argsort(q, kind="mergesort")
    v, q = v[o], q[o]
    qg = np.linspace(q[5], q[-5], n)
    vg = np.interp(qg, q, v)
    a = np.abs(np.gradient(vg, qg))
    k = int(np.argmin(a[3:-3])) + 3
    far = [i for i in range(3, len(a) - 3) if abs(qg[i] - qg[k]) > 15]
    k2 = far[int(np.argmin(a[far]))] if far else None
    n_min = sum(1 for i in range(2, len(a) - 2)
                if a[i] < a[i - 1] and a[i] < a[i + 1]
                and a[i] < a[i - 2] and a[i] < a[i + 2])
    return {"flattest_V": float(vg[k]),
            "second_flattest_V": float(vg[k2]) if k2 else np.nan,
            "separation_mV": (abs(float(vg[k2]) - float(vg[k])) * 1000.0
                              if k2 else np.nan),
            "n_minima": int(n_min)}


def rebin_sensitivity(voltage, dqdv, widths=(1.0, 2.0, 4.0)):
    """
    Does a feature survive a change of bin width, or is it the binning?

    A peak in the material is a property of the cell and does not care what
    grid it was put on. A notch or a shoulder that appears at one bin width
    and not at another is the grid. Returns a dict with the apex voltage and
    height at each width, their spreads, and `apex_stable` / `area_stable`.

    This is NECESSARY AND NOT SUFFICIENT, and the distinction cost a
    version. It rules out the binning; it says nothing about any other fixed
    stage of the pipeline, so a feature can be perfectly grid-stable and
    still be manufactured. Use `plateau_structure` to settle it.
    """
    out = {"widths": list(widths), "apex_V": [], "apex_height": [], "area": []}
    for w in widths:
        v, y = rebin_dqdv(voltage, dqdv, bin_width_mV=w)
        if v.size < 3:
            out["apex_V"].append(np.nan)
            out["apex_height"].append(np.nan)
            out["area"].append(np.nan)
            continue
        j = int(np.nanargmax(np.abs(y)))
        out["apex_V"].append(float(v[j]))
        out["apex_height"].append(float(y[j]))
        out["area"].append(float(trapezoid(np.abs(y), v)))
    a = np.array(out["apex_V"], float)
    ar = np.array(out["area"], float)
    out["apex_spread_mV"] = float(np.nanmax(a) - np.nanmin(a)) * 1000.0
    out["area_spread_frac"] = float(
        (np.nanmax(ar) - np.nanmin(ar)) / np.nanmedian(ar)) if np.isfinite(
        np.nanmedian(ar)) and np.nanmedian(ar) else np.nan
    out["apex_stable"] = bool(out["apex_spread_mV"] <= 10.0)
    out["area_stable"] = bool(out["area_spread_frac"] <= 0.10)
    return out


# Orient the plateau records that the voltage record sends backwards.
#
# On a two-phase plateau the true dV per record falls below the cycler's
# 0.1 mV voltage resolution, so the recorded voltage wobbles: in LTO cell A
# cycle 1's charge, 113 of 401 records (28%) step BACKWARDS in voltage while
# the cell is charging steadily. The instrument still divides its positive
# dQ by that negative dV, so those records carry a dQ/dV of the wrong sign
# and, because dV is tiny, a magnitude 100-200x the honest records' — median
# |dQ/dV| of 2433 against 13.
#
# Preprocessing then SORTS BY VOLTAGE, which interleaves them with the
# forward records at the same voltages, and Savitzky-Golay averages across
# two populations of opposite sign. That is the up-and-down pattern on the
# first cycle; at the LTO plateau it is worse than cosmetic, because the
# smoothed curve crosses zero and the peak comes out INVERTED (+299 on one
# flank, -1233 at the apex).
#
# The records themselves are sound. Through every one of those 113 the
# current is constant at -0.206 mA and the capacity counter rises
# monotonically: the cell is doing exactly one thing. Only the sign of a
# sub-resolution dV is in doubt, so only the sign is corrected, and only
# where the current has NOT reversed and charge is still accumulating. A
# genuine reversal — current or charge actually going backwards — is left
# exactly as recorded, because that is a finding, and `analyse.reversals`
# reads it from the raw records before any of this runs.
#
# Set False to reproduce 1.8.7, which had no such step.
ORIENT_PLATEAU_RECORDS = True


def orient_dqdv(df_step, direction, *, capacity_col=None,
                current_col="Current(A)"):
    """
    Give every record the sign its own half-cycle's honest records carry.

    Returns `(df, n_oriented)`. Operates in ACQUISITION order, before any
    sort, because it is the sort that mixes the two populations.
    """
    if not ORIENT_PLATEAU_RECORDS or len(df_step) < 3:
        return df_step, 0
    d = df_step
    v = pd.to_numeric(d["Voltage"], errors="coerce").to_numpy()
    y = pd.to_numeric(d["dQ/dV"], errors="coerce").to_numpy()
    if not np.isfinite(y).any():
        return df_step, 0

    dv = np.diff(v, prepend=v[0])
    forward = np.sign(dv) == np.sign(direction)
    # The target sign comes from the records whose voltage moved the way the
    # half-cycle is going: those are the ones whose dQ/dV sign is not in
    # question. Fall back to the signed area if there are too few.
    ref = y[forward & np.isfinite(y)]
    target = (np.sign(np.median(ref)) if ref.size >= 3
              else np.sign(np.nansum(y)))
    if target == 0:
        return df_step, 0

    # Charge still going the same way? Both tests must pass, so a genuine
    # reversal is never quietly rectified into a peak.
    ok = ~forward & np.isfinite(y) & (np.sign(y) != target)
    if current_col in d.columns:
        i = pd.to_numeric(d[current_col], errors="coerce").to_numpy()
        i_ref = np.nanmedian(i)
        if np.isfinite(i_ref) and i_ref != 0:
            ok &= (np.sign(i) == np.sign(i_ref))
    if capacity_col and capacity_col in d.columns:
        q = pd.to_numeric(d[capacity_col], errors="coerce").to_numpy()
        dq = np.diff(q, prepend=q[0])
        ok &= ~(dq < 0)          # NaN-safe: only a real decrease disqualifies

    n = int(ok.sum())
    if not n:
        return df_step, 0
    out = d.copy()
    yy = y.copy()
    yy[ok] = np.abs(yy[ok]) * target
    out["dQ/dV"] = yy
    return out, n


def strip_cv_hold(df_step):
    """Drop the trailing constant-voltage hold. Returns (df, n_removed, how).

    Operates in ACQUISITION order (the frame's own index), because the hold is
    defined by what happened last in time, not by where it sits in voltage.
    """
    if not EXCLUDE_CV_HOLD or len(df_step) < 2 * CV_MIN_POINTS:
        return df_step, 0, 'off'
    d = df_step.sort_index()

    def _trailing(mask):
        k, n = len(mask) - 1, 0
        while k >= 0 and bool(mask[k]):
            n += 1
            k -= 1
        return n

    cur_col = next((c for c in ('Current(A)', 'Current', 'current(A)')
                    if c in d.columns), None)
    if cur_col is not None:
        i = pd.to_numeric(d[cur_col], errors='coerce').abs().values
        if np.isfinite(i).sum() >= CV_MIN_POINTS:
            med = np.nanmedian(i)
            if np.isfinite(med) and med > 0:
                n = _trailing(np.nan_to_num(i, nan=np.inf)
                              < med * CV_CURRENT_FRACTION)
                if n >= CV_MIN_POINTS:
                    return d.iloc[:len(d) - n], n, 'current decay'
                return df_step, 0, 'current decay'

    v = pd.to_numeric(d['Voltage'], errors='coerce').values
    dv = np.abs(np.diff(v))
    pos = dv[dv > 0]
    if pos.size == 0:
        return df_step, 0, 'no criterion'
    med = np.median(pos)
    n = _trailing(np.r_[False, dv < med * CV_DV_FRACTION])
    if n >= CV_MIN_POINTS:
        return d.iloc[:len(d) - n], n, 'voltage stall'
    return df_step, 0, 'voltage stall'


# =============================================================================
# NEW IN 1.9.0 — measurement only
# =============================================================================

@dataclass
class Reversal:
    """One contiguous run where the potential moved the wrong way."""
    i0: int
    i1: int                    # inclusive index of the last reversed step
    charge: float              # mAh/g passed during it
    v_start: float
    v_end: float
    span_mV: float             # widest excursion, |v_end - v_start|
    seconds: float
    at_full_current: bool      # |I| still at the step median -> not a CV taper
    n_records: int


@dataclass
class HalfCycleSignal:
    """Everything one half-cycle contributes, measured and processed."""
    cycle: int
    step: str
    direction: float                       # +1 rising, -1 falling. MEASURED.
    voltage: np.ndarray = field(default_factory=lambda: np.empty(0))
    dqdv_raw: np.ndarray = field(default_factory=lambda: np.empty(0))
    dqdv: np.ndarray = field(default_factory=lambda: np.empty(0))
    capacity: float = float("nan")         # from the cycler's counter
    n_records_raw: int = 0
    cv_points_removed: int = 0
    cv_method: str = "off"
    n_spikes: int = 0
    # Records whose dQ/dV sign was corrected because the plateau sent the
    # recorded voltage backwards while the cell kept charging. See
    # `orient_dqdv`; a large number is a flat plateau, not a fault.
    n_oriented: int = 0
    # Which path produced this curve, and — on the histogram path — the bin
    # width actually used (it is widened where the records are sparser than
    # the requested bin) and how many bins came back empty. Both are how you
    # tell a well-chosen binning from one that is too fine.
    dqdv_method: str = ""
    bin_width_mV: float = float("nan")
    n_empty_bins: int = 0
    # HOW MUCH CHARGE THE ANALYSIS WINDOW LEFT OUT, in mAh/g.
    # `preprocess_half_cycle` has computed this since the histogram rewrite —
    # `charge_total - charge_in_window` — and dropped it on the floor: no
    # field held it, nothing read the key. It is the first thing to look at
    # when integral fidelity is below one, because it separates "the curve
    # cannot represent this charge" (a flat plateau, no dV to divide by) from
    # "this charge is outside the voltage range we chose to analyse", and only
    # the second is something the operator can do anything about.
    charge_outside_window: float = float("nan")
    # Raw records per bin, aligned with `voltage`. Empty on the derivative
    # path, where there are no bins. See the occupancy note in
    # `histogram_dqdv`: this is what says whether a wiggle is a measurement.
    occupancy: np.ndarray = field(default_factory=lambda: np.empty(0))
    # Rebuilds this half-cycle's curve with the bin EDGES moved by a fraction
    # of a bin: same records, same width, same kernel. `None` off the
    # histogram path, where there are no edges to move. See
    # `detect.PHASE_INVARIANCE` for what detection does with it.
    rebin_at_phase: object = None
    smoothed: bool = False
    reversals: list = field(default_factory=list)
    note: str = ""

    @property
    def reversed_charge_at_current(self) -> float:
        """mAh/g passed backwards while the current was still at full CC."""
        return float(sum(r.charge for r in self.reversals
                         if r.at_full_current))

    @property
    def widest_reversal_mV(self) -> float:
        return float(max((r.span_mV for r in self.reversals), default=0.0))


def reversals(df_step, direction, *, capacity_col=None, min_records=1):
    """
    Inventory every contiguous run where the voltage moves against `direction`.

    Works in ACQUISITION order. `direction` must come from
    `Dataset.step_direction` — measured — never from the step label.
    """
    d = df_step.sort_index()
    v = pd.to_numeric(d["Voltage"], errors="coerce").values.astype(float)
    n = v.size
    if n < 3:
        return []

    q = None
    if capacity_col and capacity_col in d.columns:
        q = pd.to_numeric(d[capacity_col], errors="coerce").values.astype(float)

    cur_col = next((c for c in ("Current(A)", "Current", "current(A)")
                    if c in d.columns), None)
    i_abs = (pd.to_numeric(d[cur_col], errors="coerce").abs().values
             if cur_col else None)
    med_i = (float(np.nanmedian(i_abs[np.isfinite(i_abs)]))
             if i_abs is not None and np.isfinite(i_abs).any() else np.nan)

    secs = None
    if "Time" in d.columns:
        t = pd.to_timedelta(d["Time"].astype(str), errors="coerce")
        if t.notna().any():
            secs = t.dt.total_seconds().values

    back = (np.diff(v) * direction) < 0
    out, k = [], 0
    while k < back.size:
        if not back[k]:
            k += 1
            continue
        j = k
        while j + 1 < back.size and back[j + 1]:
            j += 1
        idx = slice(k, j + 1)
        charge = (float(np.nansum(np.abs(np.diff(q[k:j + 2]))))
                  if q is not None else float("nan"))
        dt = (float(secs[j + 1] - secs[k]) if secs is not None
              and np.isfinite(secs[k]) and np.isfinite(secs[j + 1]) else float("nan"))
        full = bool(np.isfinite(med_i) and med_i > 0 and i_abs is not None
                    and np.nanmedian(i_abs[k:j + 2]) > 0.99 * med_i)
        if (j - k + 1) >= min_records:
            out.append(Reversal(
                i0=int(k), i1=int(j), charge=charge,
                v_start=float(v[k]), v_end=float(v[j + 1]),
                span_mV=float(abs(v[j + 1] - v[k]) * 1000.0),
                seconds=dt, at_full_current=full, n_records=int(j - k + 1)))
        k = j + 1
    return out


# --- the cut-off approach, and trimming it -------------------------------
# A galvanostatic half-cycle ENDS ON A VOLTAGE. As the cell approaches that
# cut-off, dV/dQ collapses and dQ/dV therefore climbs — steeply, and for
# reasons that are protocol rather than chemistry. Measured on NNM cell C,
# median curve height near the TERMINATING end relative to that half-cycle's
# own median:
#
#     within  20 mV      charge 5.22x     discharge 2.25x
#     within  50 mV      charge 3.78x     discharge 2.47x
#     within 100 mV      charge 1.95x     discharge 2.16x
#     within 300 mV      charge 1.37x     discharge 1.51x
#
# Left in, that rise is fitted as a peak, and the cubic baseline strains to
# follow what the peaks cannot — which is where the non-convergence came from
# (40% of fits reaching the 20,000-evaluation ceiling). Trimming it is the
# same argument the pipeline already makes about a constant-voltage hold:
# charge passed at the voltage limit is real, and is not a redox feature.
# OFF BY DEFAULT, and the reason is a negative result worth keeping.
#
# The mechanism is real: a half-cycle ends on a VOLTAGE, and as the cell
# reaches it dV/dQ collapses so dQ/dV climbs. Removing that rise measurably
# helps — on 30 cycles of NNM cell C an extra 50-100 mV took the quadratic
# baseline from 66% of fits converging to 87-89%, and dropped the median peak
# count from 5 to 4 by removing a component that was fitting the cut-off.
#
# What does NOT work is measuring the trim automatically. Across all 100
# cycles the rule's answer is BIMODAL: half the half-cycles want nothing and
# most of the rest saturate at the 200 mV cap, with nothing in between, and no
# pattern in cycle number —
#
#     median trim by cycle band   charge   discharge
#         1-20                     180        160
#        21-40                       0        150
#        41-60                       0          0
#        61-80                     190          0
#        81-100                      0          0
#
#     50% of charge and 51% of discharge half-cycles trim zero;
#     the 75th percentile is 200 mV, the cap.
#
# That is not a boundary being measured. It is a threshold flipping on and off
# depending on where the interior median happens to fall, and applying its
# median across a dataset would trim a physically arbitrary amount. A 30-cycle
# sample looked clean and was not: see the audit note.
#
# So the machinery stays, the default does not. Set TERMINAL_TRIM = True and
# TERMINAL_TRIM_FIXED_MV to a value you have LOOKED AT for your protocol —
# §A11 has the evidence and the procedure. Automatic detection of the cut-off
# approach is unfinished work, not a solved problem.
TERMINAL_TRIM = False
# When TERMINAL_TRIM is on, trim this fixed span off each half-cycle's
# TERMINATING end (high for charge, low for discharge). None = use the
# measured rule below, which the note above explains is not reliable.
TERMINAL_TRIM_FIXED_MV = 75.0
# How far above the INTERIOR median a 20 mV band must stand to still be the
# cut-off approach, and the most of the window that may be removed at an end.
#
# CALIBRATED ON THE CURVES THE PIPELINE ACTUALLY BUILDS — which already have
# `voltage_window`'s 50 mV off each end, so the sharpest part of the rise is
# gone before this ever sees it. Measured on those curves the elevation is
# gentle and broad (1.5-2.1x the interior, still 1.5x at 200 mV), not the
# 5x spike the raw record shows. Judging a 20 mV BAND rather than the single
# end point is what makes it measurable at all.
#
# Median trim found, NNM cell C against LTO cell A:
#
#   factor   NNM charge   NNM discharge   LTO charge   LTO discharge
#    1.1      (0, 200)      (200, 0)        (0, 0)        (0, 0)
#    1.2      (0, 200)      (200, 0)        (0, 0)        (0, 0)
#    1.3      (0, 180)      (200, 0)        (0, 0)        (0, 0)
#    1.4      (0, 120)      (150, 0)        (0, 0)        (0, 0)
#    1.5      (0, 100)      (  0, 0)        (0, 0)        (0, 0)
#
# Two things decide 1.4. Below it the rule SATURATES at the 10% cap (200 mV
# of a 2.15 V window) — that is "trim everything allowed", not a measurement.
# Above it the discharge side drops to nothing. 1.4 is the only value that
# returns a bounded answer on both steps.
#
# And at EVERY factor in that range LTO trims nothing, on both steps. The rule
# does not fire on a sharp two-phase profile at all, so this cannot reach the
# calibration dataset — that is a property of the rule, not of a tuned number.
TERMINAL_TRIM_FACTOR = 1.4
TERMINAL_TRIM_MAX_FRACTION = 0.10
TERMINAL_TRIM_BAND_MV = 20.0
# The middle of the curve, used as "the interior" the ends are judged against.
TERMINAL_TRIM_INTERIOR = (0.2, 0.8)


def terminal_trim(voltage, dqdv, *, factor=None, max_fraction=None,
                  band_mV=None):
    """How far in from each end the curve is still the cut-off approach.

    Returns `(low_V, high_V)` — the span to remove at the bottom and top of
    this half-cycle's voltage axis. Steps inward in `band_mV` bands while the
    band's MEDIAN stands above `factor` x the interior median, and stops at
    the first band that does not.

    The rule TARGETS ITSELF, which is the reason to trust it: a half-cycle has
    only one cut-off, so only the terminating end trims. On the NNM triplicate
    it returns (0, 120 mV) on CHARGE and (150 mV, 0) on DISCHARGE — the high
    end for charge, the low end for discharge — and (0, 0) on every LTO
    half-cycle, which has no such rise. Neither was put in by hand.
    """
    factor = TERMINAL_TRIM_FACTOR if factor is None else factor
    max_fraction = (TERMINAL_TRIM_MAX_FRACTION if max_fraction is None
                    else max_fraction)
    band = (TERMINAL_TRIM_BAND_MV if band_mV is None else band_mV) / 1000.0
    v = np.asarray(voltage, float)
    y = np.abs(np.asarray(dqdv, float))
    n = v.size
    if n < 40 or y.size != n or band <= 0:
        return 0.0, 0.0
    a, b = (int(TERMINAL_TRIM_INTERIOR[0] * n),
            int(TERMINAL_TRIM_INTERIOR[1] * n))
    interior = float(np.median(y[a:b])) if b > a else float("nan")
    if not np.isfinite(interior) or interior <= 0:
        return 0.0, 0.0
    span = float(v[-1] - v[0])
    if not np.isfinite(span) or span <= 0:
        return 0.0, 0.0
    cap = max_fraction * span
    thr = factor * interior

    def _walk(from_low):
        d = (v - v[0]) if from_low else (v[-1] - v)
        step, out = band, 0.0
        while step <= cap:
            m = (d >= step - band) & (d < step)
            if m.sum() < 2:
                break
            if float(np.median(y[m])) <= thr:
                break
            out = step
            step += band
        return float(out)

    return _walk(True), _walk(False)


def apply_terminal_trim(sig, low_V, high_V):
    """A copy of `sig` with the cut-off approach removed from its curve.

    `capacity` is NOT changed. It is the cycler's count of what the cell
    delivered, and the whole point of trimming is that some of that charge is
    no longer represented in the curve — which is exactly what
    `quality.integral_fidelity` then reports. Silently reducing the capacity
    to match would hide the thing this is meant to surface.
    """
    import dataclasses
    v = np.asarray(sig.voltage, float)
    if v.size == 0 or (low_V <= 0 and high_V <= 0):
        return sig
    keep = (v >= v.min() + low_V) & (v <= v.max() - high_V)
    if keep.sum() < 10:                    # nothing usable would be left
        return sig
    out = dataclasses.replace(
        sig, voltage=v[keep],
        dqdv=np.asarray(sig.dqdv, float)[keep],
        dqdv_raw=(np.asarray(sig.dqdv_raw, float)[keep]
                  if np.size(sig.dqdv_raw) == v.size else sig.dqdv_raw))
    return out

def voltage_window(dataset, voltage_trim_mV=50,
                   required=("Voltage", "Cycle", "Step", "dQ/dV")):
    """
    The trimmed voltage window, computed ONCE PER DATASET.

    This is not a detail. 1.8.7 takes the min and max over the whole cleaned
    frame and trims 50 mV from each end, then clips every half-cycle to that
    one window. Computing it per half-cycle instead — the obvious-looking
    thing — trims 50 mV off each half-cycle's own extremes and silently
    removes real data from every one of them.

    Verified against the 1.8.7 output: P3 cell A spans 1.9972-4.2517, giving a
    window of 2.0472-4.2017, and the reference processed_dqdv.csv spans
    2.0472-4.2013.
    """
    clean = dataset.clean
    v = pd.to_numeric(clean["Voltage"], errors="coerce")
    lo, hi = float(v.min()), float(v.max())
    trim = voltage_trim_mV / 1000.0
    if lo + trim >= hi - trim:            # 1.8.7 falls back to a 10 mV trim
        trim = 0.010
    return lo + trim, hi - trim


# Which dQ/dV the pipeline analyses. Both paths produce the same thing — a
# voltage axis and a dQ/dV on it — so everything downstream is unchanged.
#
#   "derivative"  the CYCLER's dQ/dV column, sorted, rebinned, despiked and
#                 smoothed. 1.8.x behaviour, and what every earlier Ratatosk
#                 result was computed with.
#   "histogram"   `histogram_dqdv`: charge binned by voltage, no derivative,
#                 no smoothing parameters, no sign correction needed.
#
# The default is chosen on measured voltage reconstruction, not on preference
# — see `quality.voltage_reconstruction` and the appendix.
DQDV_METHOD = "histogram"
DQDV_METHODS = ("derivative", "histogram")


def preprocess_half_cycle(df_step, params, *, direction, window,
                          capacity_col=None, step=None):
    """
    One half-cycle, from raw records to a curve ready for detection.

    Two paths, selected by `params["dqdv_method"]` or `DQDV_METHOD`.

    DERIVATIVE. The order of operations is 1.8.7's exactly — strip the
    constant-voltage hold in acquisition order, THEN sort by voltage, trim,
    rebin, despike, smooth — because changing it would change every fitted
    value.

    HISTOGRAM. Strip the constant-voltage hold, then bin the charge by
    voltage. No sort (a histogram is order-free), no rebin (the bin IS the
    rebin), no despike and no smoothing beyond a fixed three-point kernel,
    and no `orient_dqdv` — a record's charge lands in the bin for the voltage
    it was measured at whichever way the voltage happened to be moving.
    """
    raw_n = len(df_step)
    stripped, n_cv, how = strip_cv_hold(df_step)

    method = str(params.get("dqdv_method") or DQDV_METHOD).lower()
    if method not in DQDV_METHODS:
        raise ValueError(f"dqdv_method must be one of {DQDV_METHODS}, "
                         f"not {method!r}")

    if method == "histogram":
        qcol = capacity_col or ("Discharge_Capacity"
                               if str(step).lower().startswith("dis")
                               else "Charge_Capacity")
        if qcol not in stripped.columns:
            return None, dict(n_cv=n_cv, cv_method=how, raw_n=raw_n,
                              n_oriented=0, dqdv_method=method,
                              note=f"no {qcol} column for the histogram path")
        bw = params.get("histogram_bin_mV") or params.get("rebin_width_mV") \
            or HISTOGRAM_BIN_MV
        v_all = pd.to_numeric(stripped["Voltage"], errors="coerce").to_numpy()
        q_all = pd.to_numeric(stripped[qcol], errors="coerce").to_numpy()
        centres, y, hinfo = histogram_dqdv(
            v_all, q_all, bin_width_mV=float(bw), window=window,
            smooth_kernel=params.get("histogram_smooth_bins"),
            sign=dqdv_sign(step) if step is not None else 1.0)
        if centres.size < 5:
            return None, dict(n_cv=n_cv, cv_method=how, raw_n=raw_n,
                              n_oriented=0, dqdv_method=method,
                              note="too few filled bins for a curve")
        # A closure that rebuilds this curve with the bin EDGES shifted by a
        # fraction of a bin. Everything else is held: the same records, the
        # same width actually used, the same kernel, the same sign. Detection
        # uses it to ask whether a maximum is a property of the cell or of
        # where the edges happened to fall. Offsets are a fraction of the
        # width USED, not the width requested, because the width is widened
        # where the records are sparse and it is the used one that sets the
        # beat.
        _w_used = float(hinfo["bin_width_mV"])
        _kern = params.get("histogram_smooth_bins")
        _sign = dqdv_sign(step) if step is not None else 1.0

        def _rebin_at_phase(frac, _v=v_all, _q=q_all, _w=_w_used,
                            _win=window, _k=_kern, _s=_sign):
            # `phase=` rather than a shifted window: see the note beside the
            # grid anchor in `histogram_dqdv`. Shifting the window was undone
            # by the visited-span clip on exactly the half-cycles this test
            # matters for.
            c2, y2, _i2 = histogram_dqdv(_v, _q, bin_width_mV=_w,
                                         window=_win, smooth_kernel=_k,
                                         sign=_s, phase=float(frac))
            return c2, y2

        return (dict(voltage=centres, dqdv_raw=y, dqdv=y,
                     occupancy=hinfo.get("occupancy"),
                     rebin_at_phase=_rebin_at_phase),
                dict(n_cv=n_cv, cv_method=how, raw_n=raw_n, n_spikes=0,
                     n_oriented=0, smoothed=bool(HISTOGRAM_SMOOTH_KERNEL),
                     dqdv_method=method, note="",
                     bin_width_mV=hinfo["bin_width_mV"],
                     n_empty_bins=hinfo["n_empty"],
                     charge_outside_window=float(
                         hinfo["charge_total"] - hinfo["charge_in_window"])
                     if np.isfinite(hinfo["charge_total"]) else np.nan))

    # --- the derivative path -------------------------------------------
    # BEFORE the sort — see `orient_dqdv`. The sort is what turns a wobbling
    # plateau voltage into an inverted peak. The histogram path above needs
    # none of this, which is most of the argument for it.
    stripped, n_oriented = orient_dqdv(stripped, direction,
                                       capacity_col=capacity_col)

    # kind="mergesort" is a STABLE sort, and that is deliberate.
    #
    # 1.8.7 uses pandas' default quicksort, which is not stable, so where
    # several records share a voltage the order among them is decided by the
    # sort's internals. That is not a small population: P3 cell A cycle 1
    # charge has 483 tied voltages out of 1434, because the cycler records to
    # 0.1 mV and the cell sits on a plateau. Despiking and smoothing then run
    # over that arbitrary order, so 1.8.7's smoothed curve is not a
    # well-defined function of its input — reorder the rows and it changes.
    #
    # A stable sort ties-breaks by acquisition order, which is the order the
    # measurements were actually taken in. The result is reproducible under
    # any reordering of the input. This is the one place 1.9.0 knowingly
    # differs from 1.8.7, it is confined to half-cycles containing tied
    # voltages, and it is measured in tests/test_port_fidelity.py.
    d = stripped.sort_values("Voltage", kind="mergesort")
    lo, hi = window
    d = d[(d["Voltage"] >= lo) & (d["Voltage"] <= hi)]
    if len(d) < 5:
        return None, dict(n_cv=n_cv, cv_method=how, raw_n=raw_n,
                          n_oriented=n_oriented, dqdv_method=method,
                          note="fewer than 5 points after trimming")

    voltage = pd.to_numeric(d["Voltage"], errors="coerce").values.copy()
    dqdv_raw = pd.to_numeric(d["dQ/dV"], errors="coerce").values.copy()

    rebin = params.get("rebin_width_mV")
    if rebin:
        voltage, dqdv_raw = rebin_dqdv(voltage, dqdv_raw, bin_width_mV=rebin)

    work = pd.Series(dqdv_raw)
    n_spikes = 0
    if params.get("spike_removal"):
        work, n_spikes = remove_spikes(
            work, params.get("spike_window_size", 5),
            params.get("spike_threshold_multiplier", 3.0))

    smoothed, was = smooth_dqdv(work.values, params.get("smoothing_window"),
                                params.get("polyorder", 3))
    second = params.get("second_smooth_window")
    if second:
        smoothed, _ = smooth_dqdv(smoothed, second, params.get("polyorder", 3))

    return (dict(voltage=voltage, dqdv_raw=dqdv_raw, dqdv=smoothed),
            dict(n_cv=n_cv, cv_method=how, raw_n=raw_n, n_spikes=n_spikes,
                 n_oriented=n_oriented, smoothed=bool(was),
                 dqdv_method=method, note=""))


def half_cycle_report(dataset, cycle, step, params, *, window=None,
                      voltage_trim_mV=50):
    """
    Assemble a `HalfCycleSignal`: the curve, plus what was measured on it.

    `window` should be computed once with `voltage_window(dataset)` and
    reused for every half-cycle — see that function for why.
    """
    if window is None:
        window = voltage_window(dataset, voltage_trim_mV)
    df = dataset.half_cycle(cycle, step)
    direction = dataset.step_direction(cycle, step)
    cap_col = ("Charge_Capacity" if str(step).lower().startswith("c")
               else "Discharge_Capacity")

    sig = HalfCycleSignal(cycle=int(cycle), step=str(step),
                          direction=float(direction),
                          capacity=dataset.capacity(cycle, step),
                          n_records_raw=len(df))
    sig.reversals = reversals(df, direction, capacity_col=cap_col)

    curve, info = preprocess_half_cycle(df, params, direction=direction,
                                        window=window, capacity_col=cap_col,
                                        step=step)
    sig.cv_points_removed = info["n_cv"]
    sig.cv_method = info["cv_method"]
    sig.n_oriented = int(info.get("n_oriented", 0))
    sig.note = info.get("note", "")
    if curve is not None:
        sig.voltage = curve["voltage"]
        sig.dqdv_raw = curve["dqdv_raw"]
        sig.dqdv = curve["dqdv"]
        sig.n_spikes = info["n_spikes"]
        sig.smoothed = info["smoothed"]
        _occ = curve.get("occupancy")
        if _occ is not None and len(_occ) == len(sig.voltage):
            sig.occupancy = np.asarray(_occ, float)
        sig.rebin_at_phase = curve.get("rebin_at_phase")
    sig.dqdv_method = info.get("dqdv_method", "")
    sig.bin_width_mV = float(info.get("bin_width_mV", float("nan")))
    sig.n_empty_bins = int(info.get("n_empty_bins", 0))
    sig.charge_outside_window = float(
        info.get("charge_outside_window", float("nan")))
    return sig


In [ ]:
# DEFINITIONS C — FINDING FEATURES, AND CHOOSING THE REFERENCE CYCLE
# =============================================================================
"""
Where are the features, and which cycle defines them?

Detection has to happen before fitting and it cannot be undone by it. The peak
list taken from ONE cycle is fitted and tracked in every cycle of the dataset,
so a bad reference cycle is not a bad cycle — it is a bad dataset. On P3 cell C
the 1.8.6 default (cycle 2, hardcoded, checked only for existence) landed on a
half-cycle that put in 580.9 mAh/g and got 115.8 back: coulombic efficiency
19.9%, 1301 records against a median of 290, R2 0.074. Its peak list

    P3 A    3.220*, 3.301, 3.643, 3.700*
    P3 B            3.306, 3.648, 3.703*
    P3 C            3.640, 3.879, 3.901, 4.193      <- the same cell chemistry

then set the fit for all 220 cycles, missing the 3.30 V feature the other two
cells track for a hundred cycles at 2-3 mV per cycle and carrying a 4.193 V
ghost instead. Nothing in the output said so.

What is ported and what is new
------------------------------
PORTED from 1.8.7 Module 3, extracted programmatically rather than retyped:

    _voltage_to_samples   _deduplicate_peaks   _assess_truncation
    detect_peaks_single

with exactly one edit, marked in place: the charge/discharge flip now reads the
MEASURED direction rather than the step label. See the comment on that line.

NEW:

    DetectSpec            the detection parameters as a value object
    Detection             one dataset's peaks, its reference cycle, and why
    detect_all()          orchestration over `signal.HalfCycleSignal`
    choose_reference_cycle()

`choose_reference_cycle` takes the integrity check as an ARGUMENT. 1.8.7
reached into notebook globals for `integrity_band`, which meant the selector
could not be tested without a notebook and — separately — that its sibling
`ref_user_set` test had to guess at intent from a parameters dict that Cell 3
populates with defaults for every dataset. The first version of that cell was
inert on every real run for exactly this reason. Here an explicit reference
cycle is an explicit argument, and there is nothing to guess.

Serial by design
----------------
Detection is find_peaks plus one Savitzky-Golay pass: about a millisecond per
half-cycle, a few seconds for a 220-cycle dataset. That is far below
`fitting.PARALLEL_MIN_SECONDS`, and a worker pool would cost more to start than
the whole module costs to run.
"""


from dataclasses import dataclass, field
import warnings

import numpy as np

import pandas as pd
from scipy.signal import (find_peaks, peak_widths, peak_prominences,
                          savgol_filter)
# NOT `from scipy.signal import PeakPropertyWarning` — SciPy raises it but
# does not re-export it from the package namespace (checked on 1.17), so that
# import silently falls through to a stand-in class that catches nothing and
# the warnings keep printing. It lives in the private module; the try/except
# is for the day that moves, and the message filter below is the backstop
# that works whatever the class is called.
try:
    from scipy.signal._peak_finding_utils import PeakPropertyWarning
except Exception:                        # pragma: no cover - layout changed
    PeakPropertyWarning = RuntimeWarning


# =============================================================================
# DEFAULTS — the 1.8.7 values, unchanged. Changing one changes every peak list.
# =============================================================================

DEFAULT_REFERENCE_CYCLE = 2
REFERENCE_CYCLE_AUTO = True

# --- WHEN IS FORMATION OVER? ---------------------------------------------
# There is no consensus number, and the 2024 Energy & Environmental Science
# review of cell formation says so directly: protocols run from a single
# 1.7 h cycle to multi-day multi-cycle schedules, and "it is not possible to
# draw general conclusions and make direct comparisons between different
# studies ... cycling strategy must be adjusted to material and cell design".
# What the field DOES agree on is the criterion rather than the count:
# coulombic efficiency approaching and then holding near 100% is "a commonly
# used indicator of this stabilisation" and "useful in defining a target
# criterion for the end of formation".
#
# So this is MEASURED per cell rather than assumed. The reference cycle is
# the first one whose CE sits within CE_SETTLED_TOLERANCE_PP of the median CE
# of the cycles after it, and which is followed by CE_SETTLED_RUN - 1 more
# that also do. On the LTO triplicate that is cycle 4, 5 and 4 — where all
# three cells give one primary charge peak. At the old default of 2 it was
# cycle 2 for all three, which is the ONLY cycle in eleven where two of them
# split the plateau into two components, and where cell A's peak is
# classified a shoulder and therefore silently dropped from the replicate
# drift statistic. A triplicate reported n = 2 for that reason alone.
#
# Set REFERENCE_CYCLE_FLOOR to impose a house rule — 6 is a defensible
# conservative choice for "formation is certainly over" — and the measured
# criterion then applies at or above it.
CE_SETTLED_TOLERANCE_PP = 0.1
CE_SETTLED_RUN = 3

# TWO DIFFERENT SETTINGS, and they are easy to confuse.
#
#   REFERENCE_CYCLE_FLOOR     a hard minimum. Never use a reference below
#                             this, EVEN IF CE HAS ALREADY SETTLED. Raising
#                             it overrides a successful measurement — set it
#                             to 6 and a cell that settled at cycle 4 is
#                             still fingerprinted at 6. Left at the 1.8.7
#                             default, where it does nothing.
#
#   REFERENCE_CYCLE_FALLBACK  what to use when the measurement FAILS — when
#                             CE never holds within tolerance for long enough
#                             to say formation finished. Six is a defensible
#                             conservative choice: more conservative than
#                             most published protocols, and the point is that
#                             it only applies when nothing better is known.
#
# The distinction matters because the first silently discards a measurement
# and the second only fills a hole. If in doubt, move the fallback.
REFERENCE_CYCLE_FLOOR = DEFAULT_REFERENCE_CYCLE
REFERENCE_CYCLE_FALLBACK = 6

# FORMATION CANNOT END LATE. `formation_end` had a floor and no ceiling, and
# the P3/NNM triplicate showed why that is not a symmetric omission.
#
# The CE test asks whether CE sits within `tolerance_pp` of the median of the
# LATER cycles for `run` cycles running. Read that at the two ends of a
# record:
#
#   * early — "later" is hundreds of cycles of a fading cell, its median is
#     far from the current value, and the test is hard to pass;
#   * late  — "later" is a handful of adjacent cycles, its median is almost
#     the current value, and the test is nearly free.
#
# So the criterion gets EASIER the further in it looks, which is exactly
# backwards, and on a cell whose CE never really settles the first thing it
# passes is near the end of the file. On NNM cell C it returned **cycle 96 of
# 100**, quoting a settled CE of 80.59% for a cell that had lost 56% of its
# capacity; on cell B it returned 68, one cycle after the cell died at 67.
# Every cycle was then fitted with a peak list taken from a dead or
# exhausted cell, which is why that run inferred 208, 681 and 432 seeds and
# reported drift over a four-cycle baseline as 1.23 +/- 4.18 mV/cycle.
#
# Two guards, because there are two separate faults:
#
# CEILING — formation is a physical process that finishes in the first
# handful of cycles. A "formation end" beyond this is not a late formation,
# it is a false positive, and the honest report is that formation could not
# be measured. 15 is deliberately generous against the literature's "five or
# six" so that a genuinely slow-forming cell is not overruled.
REFERENCE_CYCLE_CEILING = 15
# MINIMUM LATER CYCLES — the median has to be a median of something. Below
# this many cycles after the candidate, the comparison is not evidence and
# the scan stops rather than accepting the easiest window in the record.
CE_SETTLED_MIN_LATER = 5
# A SETTLED CE HAS TO BE A POSSIBLE ONE. NNM cell B, which died at cycle 67,
# was reported by 1.9.0.24 as having finished formation at cycle 80 because
# its CE held steady against a later median of **150.00%** — a cell returning
# half again as much charge as it was given. That is not a settled cell, it
# is a broken measurement holding still, and nothing downstream should treat
# it as the end of formation. A little over 100 is ordinary noise on a good
# cell; well over is not.
CE_SETTLED_MAX_PCT = 105.0


def formation_end(efficiency, *, tolerance_pp=None, run=None, floor=None,
                  ceiling=None, min_later=None, max_ce_pct=None):
    """
    The first cycle at which coulombic efficiency has settled, and why.

    `efficiency` is `{cycle: CE in percent}`. Returns `(cycle, reason)`, or
    `(None, reason)` where the record is too short to establish it — which is
    an answer, not a failure, and the caller falls back to the floor saying so.

    The test is against the MEDIAN OF THE LATER CYCLES, not against a fixed
    target, because a cell that settles at 99.5% has settled just as much as
    one that settles at 99.9% and neither is 100%. Half-cycles the run has
    already judged not to be measurements should be left out by the caller;
    an unfinished final cycle reads as CE 52% and would otherwise drag the
    median it is being compared against.
    """
    tolerance_pp = CE_SETTLED_TOLERANCE_PP if tolerance_pp is None else tolerance_pp
    run = CE_SETTLED_RUN if run is None else run
    floor = REFERENCE_CYCLE_FLOOR if floor is None else floor
    ceiling = REFERENCE_CYCLE_CEILING if ceiling is None else ceiling
    min_later = CE_SETTLED_MIN_LATER if min_later is None else min_later
    max_ce_pct = CE_SETTLED_MAX_PCT if max_ce_pct is None else max_ce_pct

    pairs = sorted((int(c), float(v)) for c, v in (efficiency or {}).items()
                   if v is not None and np.isfinite(v))
    if len(pairs) < run + 1:
        return None, (f"only {len(pairs)} cycles with a coulombic efficiency "
                      f"— too few to say when formation ended")
    cycles = [c for c, _ in pairs]
    ce = np.array([v for _, v in pairs], float)
    for i in range(len(cycles) - run + 1):
        later = ce[i + 1:]
        # The median must be a median OF SOMETHING. Without this the test
        # gets easier the later it looks — see REFERENCE_CYCLE_CEILING — and
        # the first window it accepts is the last one in the file.
        if later.size < max(1, int(min_later)):
            break
        # Formation does not end at cycle 96. Past the ceiling the scan is
        # no longer measuring formation, so it stops and says so rather than
        # returning the first thing that happens to be flat.
        if cycles[i] > ceiling:
            # Nothing at or below the ceiling settled. Say exactly that —
            # this loop has not tested the window at `cycles[i]`, so it must
            # not claim CE settled there.
            return None, (
                f"CE never held within {tolerance_pp:g} points of the later "
                f"median for {run} cycles at or below cycle {ceiling} — "
                f"formation does not end later than that, so anything flat "
                f"beyond it is a quiet stretch of a noisy CE series and not "
                f"a measurement of formation")
        median = float(np.median(later))
        if median > max_ce_pct:
            # Steady, and impossible. Keep looking rather than accepting it.
            continue
        if all(abs(ce[j] - median) <= tolerance_pp
               for j in range(i, i + run)):
            if cycles[i] < floor:
                return floor, (f"CE settled by cycle {cycles[i]}; using the "
                               f"floor of {floor}")
            return cycles[i], (
                f"CE within {tolerance_pp:g} points of the later median "
                f"({median:.2f}%) from here for {run} cycles running")
    return None, (f"CE never held within {tolerance_pp:g} points of the "
                  f"later median for {run} cycles — formation may not have "
                  f"finished inside this record")

DEFAULT_PROMINENCE_FRACTION = 0.05
DEFAULT_MIN_DISTANCE_MV = 30
# Two maxima closer than this are one feature. It is really a statement about
# peak WIDTH, so it is allowed to vary with the profile — 30 mV apart is one
# peak on a broad layered oxide whose FWHM is over 100 mV, and might not be on
# a sharp two-phase material whose FWHM is nearer 20.
#
# Every profile is nevertheless set to 30, and the reason is worth recording,
# because 1.9.0.4 briefly set "sharp" to 15 and that was WRONG.
#
# The case was LTO cell B cycle 2 charge, whose dQ/dV carries two maxima about
# 22 mV apart. They survive a six-fold change of the rebin width, and at 15 mV
# separation the fit improves from R2 0.937 to 0.999. All of that is true and
# none of it is evidence:
#
#   * Grid stability rules out the BINNING as the cause. It does not rule out
#     any other fixed stage of the pipeline, so it is necessary and nowhere
#     near sufficient.
#   * A large R2 gain from adding a free component is what overfitting looks
#     like. It is the thing to be suspicious of, not reassured by.
#
# The test that settles it is the voltage curve itself. A second dQ/dV peak
# REQUIRES a second plateau in V(Q) — the voltage must flatten, steepen and
# flatten again — and V(Q) needs no smoothing to read, because Q is a
# monotonic integral where V is a quantised measurement. Read that way:
#
#   * The "flattest point" of cell B's charge plateau sits at 1.546 +/- 1 mV
#     in cycles 3-10, and the second-flattest is 1-3 mV from it. One plateau,
#     sampled twice.
#   * Across the triplicate, local minima of |dV/dQ| appear 2-8 per half-cycle
#     at DIFFERENT voltages every cycle and every cell. A phase transition
#     does not move.
#
# And the thermodynamics agreed all along: Li4Ti5O12 -> Li7Ti5O12 is a
# first-order two-phase transition, so at fixed T and P the potential is
# invariant and there is exactly ONE peak. That is why LTO is the textbook
# flat-plateau anode.
#
# The apparent doublet is the reciprocal of a gently sloping plateau sampled
# at the cycler's 0.1 mV voltage resolution: dQ/dV = 1/(dV/dQ), and where
# dV/dQ is small its quantisation is amplified into structure.
# HOW CLOSE MAY TWO PEAKS BE. Was 30 mV on every profile, which is a
# statement about a BROAD layered oxide's features and nothing else. Swept on
# the LTO triplicate, 64 half-cycles, every value from 30 mV to 2 mV:
#
#     min_distance   detected/hc   median R2   height   fits below 0.85
#         30 mV          1.30        0.9829     0.94        10 of 64
#         12 mV          1.70        0.9850     0.94         9 of 64
#          8 mV          2.11        0.9865     0.94         8 of 64
#          5 mV          2.48        0.9867     0.95         7 of 64
#          2 mV          2.66        0.9867     0.95         7 of 64
#
# The answer stops moving at 5 mV — below it the extra detections change no
# fitted number — so that is where the floor goes rather than at the smallest
# value that still runs. It is also about one FWHM on the narrowest cell in
# hand (2.6-4.7 mV on LTO cell C) and five histogram bins at the sharp
# profile's 1 mV binning, so it is a measurement floor and not a preference.
# Costs 25 s against 5 s to fit the triplicate.
#
# Moderate and broad stay at 30 mV: no moderate dataset exists to measure on,
# and on broad the 30 mV rule is what stopped a pair 24 mV apart with a
# 30-fold height difference from collapsing into each other (see
# `_deduplicate_peaks`).
# NOT LOWERED, AND THE REASON IS THE PHASE RULE, NOT THE FIT.
#
# Lowering this to 5 mV on `sharp` lets the detector resolve the LTO cycle-1
# "doublet", and the fit then improves a great deal: on cell C cycle 1 charge
# R2 0.9161 -> 0.9951, height 0.77 -> 1.03, max residual 31% -> 7%. Swept over
# 64 half-cycles, median R2 0.9829 -> 0.9867 and the fits shorter than 0.85 of
# the data fall from 10 to 7.
#
# **That evidence is inadmissible and the change is reverted.** Appendix A7 of
# this notebook settles the same question against exactly this argument:
# Li4Ti5O12 -> Li7Ti5O12 is a first-order two-phase transition, so by the
# Gibbs phase rule the two-phase field is invariant at fixed T and P — one
# plateau, therefore ONE dQ/dV peak. V(Q), which is processing-free, shows no
# second plateau. The pair appears in one cycle of one cell and not in the
# replicates. And the literature that most threatens the claim (the 2024
# three-phase alpha/beta/gamma study) puts its third phase at 0.53-0.63 V,
# which a cell cycled 1.2-2.5 V never reaches.
#
# A7's rule is: before adding a component, ask what process it would
# correspond to; if the phase diagram does not offer one, the component is
# describing noise or a baseline, and the fit will happily accommodate it. An
# R2 gain from adding a component is what overfitting looks like. So the
# 5 mV floor stays out, and the height the fit is short by on those
# half-cycles stays in the report as a fault rather than being fitted away.
#
# The `_deduplicate_peaks` bug found while measuring this is real and IS
# fixed: for two primaries the rule was `max(shoulder_separation,
# min_distance)`, so a caller asking for anything below the 20 mV shoulder
# threshold was silently refused. At 30 mV that is a no-op (`max(20, 30) ==
# 30`), verified by re-running both broad chemistries to identical numbers —
# but it means this constant now does what it says, and a chemistry that
# genuinely has features closer than 20 mV can be given a floor that works.
PROFILE_MIN_DISTANCE_MV = {"sharp": 30, "moderate": 30, "broad": 30}
DEFAULT_MIN_WIDTH_MV = 5

# MINIMUM WIDTH, BY PROFILE — and the sharp value is the one that mattered.
#
# `DEFAULT_MIN_WIDTH_MV = 5` applied to every profile, and on LTO the measured
# FWHMs are 2.6-4.7 mV (cell C) and 12.6-22.9 mV (cell A). So on cell C the
# primary pass rejected EVERY real peak, and detection ran entirely on the
# curve-maximum rescue path — which returns exactly one component and can
# therefore never find a doublet. That is why the cycle-1 doublet Nik spotted
# on the waterfall is invisible to the pipeline.
#
# The floor a peak-finder needs is set by the MEASUREMENT: a feature narrower
# than the histogram cannot be resolved, and one at the bin width is a single
# sample. Two bins is the least that can be called a peak, and at the sharp
# profile's 1 mV bin that is 2 mV, which admits every LTO feature measured.
#
# Left at 5 mV for moderate and broad deliberately: no moderate dataset exists
# to measure on, and on broad the 5 mV floor is currently the thing standing
# between the fit and needle components (2 of 87 on NMC111 reach it and take
# the composite to 3.4x the data's height). Lowering it there is untested and
# would loosen the one guard that case has.
PROFILE_MIN_WIDTH_BY_CLASS = {"sharp": 2, "moderate": 5, "broad": 5}

# MINIMUM PEAK WIDTH, BY PROFILE. `min_distance` has been profile-dependent
# since 1.9.0 on the argument that peak separation is a statement about peak
# width; the width threshold itself was left at one global 5 mV, which on a
# sharp two-phase material is WIDER THAN THE FEATURES.
#
# What that cost, measured on LTO cell A cycle 4 delithiation: the real peak
# is 5.2 mV FWHM and 22,505 tall, and `find_peaks` rejected it for being
# narrower than the threshold. The second-derivative pass then picked up a
# shoulder at 1.5825 — a third the height, 7,662 — and that is what the run
# reported, fitted, tracked and exported. Because it was flagged a shoulder
# it also carried a FABRICATED prominence (0.1 x height, see the note in
# `detect_peaks_single`) and was excluded from every replicate statistic:
# the LTO discharge drift was reported as "—" for a triplicate where all
# three cells have a clean, tall, well-resolved delithiation peak.
#
# LOWERING THE THRESHOLD IS THE WRONG FIX and this table is not used. Measured
# on the LTO triplicate over its settled cycles: the discharge peak is only
# found as primary at 2.5 mV or below, and at 2.5 mV the CHARGE side admits
# two or three peaks per cycle where there is one. No single width serves
# both, because width is not what is actually wrong.
#
# What is wrong is that the CURVE'S OWN MAXIMUM was not in the peak list at
# all. See `_ensure_curve_maximum`.
PROFILE_MIN_WIDTH_MV = {"sharp": 2, "moderate": 4, "broad": 5}   # unused
SHOULDER_DETECTION = True
SHOULDER_MIN_SEPARATION_MV = 20
SHOULDER_HEIGHT_FRACTION = 0.10
SHOULDER_D2_PROMINENCE_FRACTION = 0.08
# THE PARENT MUST BE A FEATURE. Replaces SHOULDER_D2_ABSOLUTE_FLOOR in
# 1.9.0.57, one gate out and one gate in — the parameter count is unchanged.
#
# What the old floor was for is still right: local gates alone have a failure
# mode that is the mirror image of the global ones they replaced. On a long,
# almost-flat, slowly-rising tail — NNM below 3.0 V — the parent's local
# height is small, so a fraction of it is a very low bar and every noise
# ripple clears it. The first build of the local pass put four markers in
# that tail (2.48, 2.57, 2.67 V and friends).
#
# What was wrong was the QUANTITY chosen to say so. The old floor asked
# whether the CANDIDATE's curvature was large compared with the whole curve's
# d2 range — and the whole curve's d2 range is set by the tallest, sharpest
# feature, which may be 400 mV away and has nothing to do with the candidate.
# That is the same fault the local height gate was built to cure, left in
# place on the one test that kept a global denominator.
#
# It shows as three symptoms, all measured on the NNM triplicate (40 cycles
# per cell, 247 candidates, labelled by their PARENT: a candidate hanging off
# the 3.2/3.3 V redox peak is REAL, one hanging off the sub-2.6 V
# polarisation tail is RIPPLE):
#
#   1. The two populations do not separate on d2 share. REAL runs from 0.001
#      to 0.209 (p25 0.040); RIPPLE runs to 0.100 (p75 0.027). The threshold
#      0.04 sits inside both. It kept 76% of the real shoulders and admitted
#      8.7% of the ripples.
#
#   2. IT IS NOT EVEN ACROSS CELLS. Real shoulders kept: cell A 73.6%, cell B
#      67.2%, cell C 86.7%. Cell B's 3.15 V discharge shoulder scores
#      0.035-0.040 against a threshold of 0.040 — it is half its parent's
#      height, plainly visible in the raw trace, and rejected in 58 of 67
#      cycles, so cell B fitted one component where A and C fitted two.
#
#   3. IT FADES WITH CYCLE, which is worse than either. On discharge the old
#      gate kept 90.9% of real shoulders in cycles 1-5, then 60.0%, 60.9%,
#      50.0% through cycles 6-40. A gate that loses a feature more often as
#      the cell ages manufactures a decline that is not in the data. This is
#      the mechanism behind the P3 structural claim retracted in 1.9.0.56.
#
# The parent's own significance separates them cleanly, because that is what
# actually differs: a shoulder sits on a redox peak, a ripple sits on a
# polarisation tail. Parent local height as a share of the curve's range:
#
#     population    n     min      p25     median    p75      max
#     REAL        171   0.022    0.421    0.499    0.534    0.723
#     RIPPLE       46   0.011    0.056    0.085    0.120    0.139
#
# RIPPLE never exceeds 0.139; REAL's quartile is 0.421. 0.20 sits above the
# whole ripple population with margin and well below the real one:
#
#     rule                     REAL kept   ripple kept   per cell (A/B/C)
#     d2_frac    >= 0.04         76.0%        8.7%       73.6 / 67.2 / 86.7
#     parent_share >= 0.20       88.3%        0.0%       84.9 / 89.7 / 90.0
#
# and on discharge the new rule keeps 100% in every cycle band, 1-5 through
# 21-40 — the fade is gone. On NMC it admits two more candidates (real
# shoulders at 3.65 V on the 3.77 V charge peak, scored 0.037 and 0.038 by
# the old gate) and drops none. On LTO it changes nothing: LTO produces no
# shoulder candidates at all in 30 half-cycles.
#
# A d2 floor was tried ALONGSIDE this at 0.010, 0.015 and 0.020 and changed
# not one candidate at any of them — parent_share already excludes everything
# the noise floor existed to exclude. A parameter that does not pay for
# itself does not go in (Appendix A6), so the old gate is removed rather than
# demoted.
#
# The threshold is not a knife edge in either direction: 0.15, 0.20, 0.25 and
# 0.30 all admit zero ripples and keep 88.9%, 88.3%, 87.7%, 87.7% of the real
# shoulders.
SHOULDER_PARENT_SHARE_MIN = 0.20
# A peak's prominence must clear DEFAULT_PROMINENCE_FRACTION of the
# half-cycle's FULL RANGE. That rule has a failure mode, inherited from 1.8.x
# and visible on P3 cell A: where one feature dominates, it sets the threshold
# for everything else. On cycle 10 discharge the 3.539 V two-phase peak is
# 2075 mAh/V/g and the floor is therefore 104, so the 3.156 V feature —
# prominence 44.5, and a peak this cell tracks for a hundred cycles — is
# invisible. On the same cycle's CHARGE the range is 383, the floor is 19, and
# all four peaks are found. One half-cycle, not the other, for no reason to do
# with the chemistry.
#
# Worse, it flickers: 1.8.7 found 3.148 V on cycle 9 and 3.151 V on cycle 11
# and missed it on cycle 10. A feature that appears and disappears between
# consecutive cycles is precisely what corrupts tracking and area retention.
#
# So a peak is also kept if it is a distinct feature ON ITS OWN SCALE: its
# prominence is at least LOCAL_PROMINENCE_FRACTION of its own height (it is a
# peak, not a ripple on somebody's flank) AND its height is at least
# MIN_HEIGHT_FRACTION of the largest (it is not noise). Both are ratios, so
# neither can be dominated by an unrelated feature elsewhere in the window.
# A FLOOR REFERENCED TO THE NOISE, NOT TO THE TALLEST FEATURE.
#
# `MIN_HEIGHT_FRACTION` below is a fraction of the curve's maximum, which says
# nothing about whether a bump is above the noise. On the LTO triplicate that
# admitted 19 components that are not features at all: 2-5% of their
# half-cycle's largest, 31-87 mV away from it, and — the decisive test — NOT
# REPRODUCIBLE. Same 64 half-cycles, only the processing changed:
#
#     1.0 mV bin, unsmoothed (the run's own setting)   19 extras
#     0.5 mV bin                                        0 extras
#     2.0 mV bin                                        3 extras
#     1.0 mV bin, smoothed x3                           4 extras
#
# ...while the main peak's mean position moved by 0.2 mV across all four. The
# extras' own potentials scatter over 150 mV across cells and cycles. A process
# sits at a potential; these do not. Appendix A7's caution is that bin-width
# stability is NECESSARY BUT NOT SUFFICIENT — these fail necessity.
#
# WHY NOT KEY IT TO THE MECHANISM. The obvious alternative is a stricter floor
# on a `two_phase` run, since the Gibbs phase rule allows one peak in the
# two-phase field. Rejected, for three reasons. The phase rule constrains the
# FIELD, and `PROFILE_MIN_DISTANCE_MV = 30` already forbids a second component
# within 30 mV of the plateau — every one of these 19 is 31-87 mV away, outside
# what the phase rule speaks about. The mechanism is classified from ONE
# reference half-cycle before anything is fitted, so keying detection to it
# would silently delete the features most worth catching later — a second phase
# emerging on ageing, surface pseudocapacitance, an additive or electrolyte
# process. And it would encode a thermodynamic EXPECTATION as a measurement
# THRESHOLD, which makes the tool assert the phase rule rather than test
# against it. The expectation belongs on the page as a finding; see
# `report._phase_rule_sentence`.
#
# THE RULE IS A CONJUNCTION, because neither half separates on its own.
# Measured over 320 components and 112 half-cycles, all three chemistries:
#
#                          n     min SNR   5th pct    median
#     LTO main peaks      64        38.5      44        63
#     LTO extras          19         1.5       -         6.0  (max 10.0)
#     NMC, all           126        16.5      21.2      39.9
#     NNM, all           111        10.5      15.0     165
#
# A single SNR floor cannot work: LTO's worst artefact is 10.0 and NNM's
# weakest REAL component is 10.5. But those populations are nothing alike on
# the other axis — the artefacts are 2-5% of their half-cycle's largest, the
# low-SNR NNM components 28-41%. Substantial features on a noisy curve, not
# bumps on a clean one. So a candidate is rejected only when it is BOTH below
# the noise floor AND a small share of its own half-cycle:
#
#     k = 12, f = 0.10   ->  19 of 19 artefacts removed, 0 of 301 real
#     k = 12, f = 0.15   ->  19 of 19,                   0 of 301
#     k = 15, f = 0.10   ->  19 of 19,                   0 of 301
#     k = 20, f = 0.10   ->  19 of 19,                   1 of 301
#
# Not a knife edge — anything in k = 12-15, f = 0.10-0.20 gives the same split,
# with margin on both axes (worst artefact 10.0 against 12 and 5.0% against
# 10%; closest real component 10.5 but 27.7% of its curve).
#
# Mechanism-blind, chemistry-blind and scale-free: a statement about what the
# measurement can support, in the same family as `sigma_min` from the histogram
# bin and `WIDTH_MIN_SAMPLES` from the sampling interval.
NOISE_FLOOR = True
NOISE_SNR_MIN = 12.0            # prominence / local noise
NOISE_SHARE_MIN = 0.10          # ...unless it is this much of the largest
NOISE_WINDOW_BINS = 30          # half-width of the local noise estimate


def local_noise(signal, index=None, half=NOISE_WINDOW_BINS):
    """
    High-frequency noise near `index`, from the curve itself.

    `d = y[j] - (y[j-1] + y[j+1]) / 2` has variance 1.5 sigma^2 for
    independent noise and is blind to a smooth peak, so it measures scatter
    without being fooled by curvature.

    Two departures from the textbook estimator, both forced by the data:

    - The scale comes from `mean(|d|)` (times sqrt(pi/2)) rather than the MAD.
      A histogram dQ/dV is heavily quantised — long runs of identical values,
      including exact zeros — so the MAD is frequently exactly 0 and the
      estimate collapses.
    - It is LOCAL. dQ/dV noise scales with the signal rather than being
      constant across the window, so a whole-curve estimate is set by
      whichever region happens to be quietest.

    Returns NaN where there is too little to measure, which the caller must
    read as "no opinion" rather than as "no noise".
    """
    y = np.asarray(signal, float)
    if y.size < 7:
        return np.nan
    if index is None:
        seg = y
    else:
        lo = max(0, int(index) - int(half))
        hi = min(y.size, int(index) + int(half) + 1)
        seg = y[lo:hi]
        if seg.size < 7:
            seg = y
    d = seg[1:-1] - 0.5 * (seg[:-2] + seg[2:])
    if d.size == 0:
        return np.nan
    # E|N(0, s)| = s * sqrt(2/pi), so s = mean|d| * sqrt(pi/2); then divide
    # through by sqrt(1.5) for the Laplacian's own variance inflation.
    return float(np.mean(np.abs(d)) * 1.2533141 / np.sqrt(1.5))


LOCAL_PROMINENCE_FRACTION = 0.25
MIN_HEIGHT_FRACTION = 0.02
LOCAL_PROMINENCE = True          # set False to restore 1.8.x exactly

EDGE_EXCLUSION_MV = 50
# Minimum visible fraction of a truncated peak (estimated from half-width
# asymmetry) to retain it for fitting. 0.60 means at least 60% of the peak's
# expected full width must be visible. Below this the peak is discarded: only
# one flank is present and a fitted centre would be an extrapolation.
TRUNCATION_VISIBLE_FRACTION = 0.60
# ...but do not DISCARD on it. The visible fraction is a geometric estimate,
# made before the fit, of a quantity the fit measures directly — and it is
# measured from the half-maximum crossing on a flank that is partly outside
# the window, which is the noisiest place on the curve to measure anything.
# Discarding on it removes a component from the model, which changes the fit
# of every other component in the half-cycle; flagging it costs nothing and
# the post-fit test (`analyse.TRUNCATION_SIGMAS`) then answers the question
# properly, in the same units as everything else in the parameter table.
# Set True to restore 1.8.7's behaviour.
TRUNCATION_DROP = False

# Bands from `Module 1b` / `analyse` that a reference cycle may carry. A band
# outside this set — 'ANOMALOUS', 'unknown', 'too few records' — disqualifies
# a cycle from anchoring a dataset. 'too few records' is excluded deliberately:
# it does not mean the half-cycle was fine, it means we could not tell.
SOUND_BANDS = ("clean", "suspect")

# --- neighbour-informed seeding ------------------------------------------
# Detection is a threshold test on one half-cycle, so a peak that is genuinely
# present can fall below the prominence floor in a single cycle — noise, a
# slightly shallower feature, a rate step — and reappear in the next. Left
# alone, that cycle contributes a "disappeared" to the peak's history and a
# gap to its area trend, and the gap looks like degradation.
#
# So where a reference peak has no detected counterpart in a half-cycle that
# was otherwise fitted, a seed is placed at the position that peak occupied in
# the NEAREST cycle that did detect it. The fit then either finds something
# there or does not; either answer is informative, and neither is a gap.
#
# A component seeded this way is a DIFFERENT KIND OF NUMBER from one detected
# in the cycle it is reported for, and the parameter table says so:
# `is_inferred` is True and `reliable` is False, so it appears in the table and
# in the tracked history but never in a headline trend. An always-False
# provenance flag would be worse than none, because it reads as an assurance.
# --- a detection that finds this many peaks has not found peaks -----------
# Thirty-five components on a curve with four features is not a hard fitting
# problem, it is a detection failure — and left alone it is an EXPENSIVE one:
# every component adds four correlated parameters, and a half-cycle seeded
# that way took minutes to converge onto an answer that meant nothing. It
# happens when detection runs on an unsmoothed curve, so the guard also says
# what to check.
#
# The cap keeps the strongest peaks by prominence, which is the right ordering
# — a shoulder's placeholder prominence is 10% of its height, so genuine
# shoulders survive alongside their parents — and says how many it dropped.
MAX_PEAKS_PER_HALF_CYCLE = 12
INFER_MISSING_SEEDS = True
SEED_INFERENCE_TOLERANCE_MV = 80.0


# =============================================================================
# PORTED FROM 1.8.7 MODULE 3 — do not edit without a bit-identity regression
# =============================================================================

def _voltage_to_samples(mV, voltage_array):
    """Convert millivolts to approximate sample count."""
    if len(voltage_array) < 2:
        return 1
    avg_step = np.abs(np.diff(voltage_array)).mean()
    return max(1, int(round((mV / 1000.0) / avg_step))) if avg_step > 0 else 1


def _ensure_curve_maximum(peaks_df, voltage, signal, *, min_distance_mV):
    """
    The tallest point of a half-cycle's dQ/dV is a peak. Make sure it is in.

    `find_peaks` gates on WIDTH, and on a sharp two-phase material the real
    feature can fail that gate: LTO's delithiation peak is 22,505 tall and
    about 5 mV wide against a 5 mV minimum, so it was rejected outright. The
    second-derivative pass then picked up an inflection 3 mV away, a third
    the height, and THAT was reported as the dataset's discharge feature —
    fitted, tracked, and (being flagged a shoulder) excluded from every
    replicate statistic.

    1.9.0.18 tried to fix this by promoting any shoulder that was the tallest
    row IN THE PEAK TABLE. That was wrong twice over. A lone shoulder is
    trivially the tallest row in a one-row table, so it was always promoted;
    and it promoted the INFLECTION rather than finding the peak beside it, so
    the reported feature was still the wrong one — now mislabelled primary,
    with a prominence of 0, and admitted to the statistics it had rightly
    been excluded from. scipy said so 32 times per run
    (`PeakPropertyWarning: some peaks have a prominence of 0`) and the run
    carried on regardless.

    The rule here is the one that is true by construction: whatever else a
    half-cycle contains, the global maximum of |dQ/dV| is a feature of it.
    If no detected peak sits within `min_distance_mV` of that maximum, it is
    added, as a primary, with its prominence MEASURED. Nothing is relabelled
    and no threshold is loosened, so the charge side — where lowering the
    width gate admitted two and three peaks per cycle — is untouched.
    """
    if voltage is None or signal is None or len(signal) < 3:
        return peaks_df
    y = np.asarray(signal, float)
    if not np.isfinite(y).any():
        return peaks_df
    i = int(np.nanargmax(y))
    if i == 0 or i == len(y) - 1:
        return peaks_df            # an edge is a truncation, not a peak
    v_max = float(voltage[i])
    if peaks_df is not None and not peaks_df.empty:
        near = (peaks_df["voltage"] - v_max).abs() < min_distance_mV / 1000.0
        if bool(near.any()):
            # Already represented. If the representative is a SHOULDER but
            # the curve's maximum is a true local maximum, the label is the
            # thing that is wrong, so correct it — measured, not assumed.
            j = peaks_df.index[near][0]
            if bool(peaks_df.loc[j, "is_shoulder"]):
                from scipy.signal import peak_prominences
                prom = float(peak_prominences(y, [i])[0][0])
                if prom > 0:
                    peaks_df = peaks_df.copy()
                    peaks_df.loc[j, ["voltage", "height", "prominence",
                                     "is_shoulder", "peak_index"]] = [
                        v_max, float(y[i]), prom, False, i]
                    if "prominence_estimated" in peaks_df.columns:
                        peaks_df.loc[j, "prominence_estimated"] = False
            return peaks_df

    from scipy.signal import peak_prominences
    prom = float(peak_prominences(y, [i])[0][0])
    if prom <= 0:
        # Not a local maximum at all — a monotonic run or a flat top. Adding
        # it would be inventing a peak, which is the failure this whole
        # module is written against.
        return peaks_df
    row = {c: np.nan for c in (peaks_df.columns if peaks_df is not None
                               and not peaks_df.empty else [])}
    row.update(voltage=v_max, height=float(y[i]), height_original=float(y[i]),
               prominence=prom, prominence_estimated=False,
               peak_index=i, is_shoulder=False, is_truncated=False,
               peaks_capped=False, width_V=np.nan,
               left_voltage=np.nan, right_voltage=np.nan)
    if peaks_df is None or peaks_df.empty:
        out = pd.DataFrame([row])
    else:
        out = pd.concat([peaks_df, pd.DataFrame([row])], ignore_index=True)
    out["peak_id"] = range(1, len(out) + 1)
    return out.sort_values("voltage").reset_index(drop=True)


def _deduplicate_peaks(peaks_df, min_separation_mV=20,
                       primary_min_separation_mV=None):
    """
    Remove peaks closer than min_separation_mV after merge.

    When both find_peaks and the second-derivative pass detect the same
    physical feature at nearly identical voltages, this keeps the detection
    with higher prominence and drops the duplicate. Generalises to any
    system — LIB, NIB, or otherwise.

    `primary_min_separation_mV` is the wider rule applied when NEITHER peak
    is a shoulder: `min_distance_mV`, the parameter that defines what counts
    as two distinct peaks in the first place. The two passes each honoured
    it internally, but a peak from the scale-free rescue pass and one from
    the primary pass were only ever compared at the 20 mV shoulder
    threshold. On P3 cell A cycle 10 Discharge that admitted a pair 24 mV
    apart whose heights differ 30-fold, and the fit could not resolve them:
    the smaller component slid onto the larger, collapsed to zero area, and
    took a quarter of the larger peak's area and 0.074 of R2 with it. A
    shoulder legitimately sits close to its parent and keeps the narrow
    threshold.
    """
    if peaks_df.empty or len(peaks_df) < 2:
        return peaks_df

    df = peaks_df.sort_values('voltage').reset_index(drop=True)
    min_sep_V = min_separation_mV / 1000.0 + 1e-9  # float tolerance
    primary_sep_V = ((primary_min_separation_mV / 1000.0 + 1e-9)
                     if primary_min_separation_mV else min_sep_V)

    def _sep(i, j):
        # TWO PRIMARIES ARE SEPARATED BY `min_distance_mV`, FULL STOP.
        #
        # This was `max(min_sep_V, primary_sep_V)`, and the reasoning behind
        # it was sound while `min_distance_mV` was 30 mV on every profile:
        # a primary pair should not be held to the NARROW shoulder rule, so
        # take the wider of the two. The moment a profile asks for a
        # separation below the 20 mV shoulder threshold, the `max` silently
        # refuses it — two genuine primaries could not be closer than 20 mV
        # whatever the caller set.
        #
        # Measured cost, on the clearest data we own: LTO cell C cycle 1
        # charge is a visible doublet at 1.5815 V (15,200) and 1.5875 V
        # (11,183), 6 mV apart, with a valley at 4,985 between them. Both
        # passes find both peaks; `_ensure_curve_maximum` keeps both; and
        # this line then dropped the second at every `min_distance_mV` from
        # 30 mV down to 3 mV. One component was fitted across the pair, at
        # 77% of the data's height and with the asymmetry ratio driven to
        # 17.8 — a lineshape parameter standing in for a peak. That is the
        # doublet that has been invisible to the pipeline since 1.8.7.
        #
        # `min_sep_V` still governs any pair involving a shoulder, which is
        # what it was written for: a shoulder legitimately sits close to its
        # parent, and the 20 mV rule stops the second-derivative pass
        # re-reporting a feature the primary pass already has.
        if 'is_shoulder' in df.columns and not (
                bool(df.loc[i, 'is_shoulder'])
                or bool(df.loc[j, 'is_shoulder'])):
            return primary_sep_V
        return min_sep_V

    keep = [True] * len(df)
    for i in range(len(df) - 1):
        if not keep[i]:
            continue
        for j in range(i + 1, len(df)):
            if not keep[j]:
                continue
            if df.loc[j, 'voltage'] - df.loc[i, 'voltage'] < _sep(i, j):
                # Too close — drop lower prominence
                if df.loc[i, 'prominence'] >= df.loc[j, 'prominence']:
                    keep[j] = False
                else:
                    keep[i] = False
                    break
            elif (df.loc[j, 'voltage'] - df.loc[i, 'voltage']
                  >= max(min_sep_V, primary_sep_V)):
                # Past the WIDEST threshold any later pair could use, so
                # nothing further can be a duplicate of i. Breaking on the
                # pair's own threshold would skip a non-shoulder pair that
                # follows a nearer shoulder.
                break

    result = df[keep].reset_index(drop=True)
    result['peak_id'] = range(1, len(result) + 1)
    return result


def _assess_truncation(centre_v, left_v, right_v, data_v_min, data_v_max,
                        visible_fraction_threshold, edge_tol_V=0.002):
    """
    Assess whether a peak is truncated by the voltage window edge.
    Uses half-width asymmetry: for a symmetric peak, the full width would
    be 2 * (unclipped half-width). The visible fraction is estimated as
    (left_hw + right_hw) / (2 * reference_hw), where reference_hw is the
    half-width on the unclipped side.
    Parameters
    ----------
    centre_v : float
        Peak centre voltage.
    left_v : float
        Left half-maximum voltage (from peak_widths).
    right_v : float
        Right half-maximum voltage (from peak_widths).
    data_v_min, data_v_max : float
        Voltage range of the data array.
    visible_fraction_threshold : float
        Minimum visible fraction to retain the peak (e.g. 0.60).
    edge_tol_V : float
        Tolerance for deciding whether a width base is "at the edge" (V).
    Returns
    -------
    is_truncated : bool
        True if peak is retained but flagged as truncated.
    drop : bool
        True if peak should be discarded (< threshold visible).
    """
    left_hw = centre_v - left_v
    right_hw = right_v - centre_v
    near_upper = right_v >= data_v_max - edge_tol_V
    near_lower = left_v <= data_v_min + edge_tol_V
    if near_upper and not near_lower:
        # Right side clipped — use left_hw as reference
        if left_hw > 0:
            visible = (left_hw + right_hw) / (2.0 * left_hw)
        else:
            visible = 0.0
        if visible >= visible_fraction_threshold:
            return True, False   # truncated but usable
        else:
            return False, True   # discard
    elif near_lower and not near_upper:
        # Left side clipped — use right_hw as reference
        if right_hw > 0:
            visible = (left_hw + right_hw) / (2.0 * right_hw)
        else:
            visible = 0.0
        if visible >= visible_fraction_threshold:
            return True, False
        else:
            return False, True
    elif near_lower and near_upper:
        # Clipped on BOTH flanks: neither half-maximum is inside the window,
        # so the visible fraction cannot be measured at all. Certifying that
        # as "not truncated" was the one case where the most severely
        # truncated peak got the most confident label. Flag it and let the
        # fit proceed — its area is a lower bound.
        return True, False
    else:
        return False, False


def _shoulder_sigma(neg_d2, i):
    """
    A shoulder's width, measured — from the inflection points either side.

    `peak_widths` cannot help here: a shoulder is not a local maximum, so
    scipy never measured a width for it and detection left the value at zero.
    That zero was then printed in a results table beside measured widths, and
    (worse) it meant the fit had no starting width for the majority of
    components on a broad profile, where most reference peaks are shoulders.

    But the second derivative already knows. For a Gaussian of width sigma,
    d2 changes sign exactly at the inflection points, which sit at
    centre +/- sigma. `neg_d2` is positive across the top of a feature and
    goes negative outside it, so walking out from the candidate to the first
    sign change on each side measures 2*sigma directly. It is the same array
    the shoulder was found in, so nothing new is assumed.

    Returns sigma in the units of the sample index -> converted by the caller,
    or NaN if either crossing runs off the end of the curve.
    """
    n = len(neg_d2)
    if i <= 0 or i >= n - 1:
        return np.nan
    L = i
    while L > 0 and neg_d2[L] > 0:
        L -= 1
    R = i
    while R < n - 1 and neg_d2[R] > 0:
        R += 1
    if neg_d2[L] > 0 or neg_d2[R] > 0:
        return np.nan          # never crossed: the lobe runs off the window
    half = (R - L) / 2.0
    return half if half > 0 else np.nan


# THE SECOND-DERIVATIVE WINDOW, IN VOLTS RATHER THAN IN SAMPLES.
#
# `_flank_shoulders` smooths with `savgol_filter(signal, w, 3, deriv=2)` and
# `w` was `min(21, max(7, n // 5))` — a count of SAMPLES, fixed, whatever the
# bin width or the chemistry. At the bin widths this build actually produces
# that window spans:
#
#     LTO   327 bins at 2.41 mV  ->  21 samples =  51 mV, against a  10 mV FWHM
#     NNM   334 bins at 6.33 mV  ->  21 samples = 133 mV, against a  70 mV FWHM
#     NMC   205 bins at 6.82 mV  ->  21 samples = 143 mV, against a 200 mV FWHM
#
# So it is five times the feature on LTO, twice on NNM, and about right on
# NMC — which is the one chemistry it was never a problem for. A cubic fitted
# over five times a peak's width cannot see structure on that peak's flank;
# that is the whole complaint behind item 38.
#
# The scale that matters is the FEATURE's, not the curve's and not the grid's,
# and detection already knows it: `peak_widths` measured every primary before
# this function is called. The window is set from the NARROWEST primary,
# because a window wide enough to blur the sharpest feature present will blur
# everything narrower than it too, and a shoulder is by definition narrower
# than its parent.
#
# `SHOULDER_D2_WINDOW_FRACTION` is the fraction of that width the window
# spans. Half of it is the natural choice: a Savitzky-Golay cubic reproduces a
# curve faithfully over a span where the curve is close to cubic, and half a
# peak width is about where that holds. The result is flat over 0.25-1.0 on
# every chemistry, so the exact value is not doing the work — the clamp is.
#
# WHAT IT RECOVERS. NNM's 3.15 V discharge shoulder, the labelled true
# positive, detected in 30 half-cycles of 30 (cell A) and 28 of 30 (cell C)
# at every floor from 7 to 13 samples, against 18 and 19 under the old rule.
# In the pipeline that feature's coherence goes from 18 pairs and a split
# reference set to 125 pairs and a single one — see the note in
# `Ratatosk_1_9_0_61`. LTO is bit-for-bit unchanged: it produces no shoulder
# candidates at any window.
#
# WHY THE FLOOR IS 11 AND NOT 7. Differentiating over a shorter span passes
# more noise, and the synthetic ground truth is where that shows. Failures on
# `tests/test_synthetic_recovery.py`, and the time it takes:
#
#     floor        failures    total time    NNM shoulder (A / C)
#     old rule         9          75 s          18/30, 19/30
#     13              12         254 s          30/30, 28/30
#     11              12         189 s          30/30, 28/30
#      7              15         721 s          30/30, 28/30
#
# At 7 samples the 1.0%-noise scenario alone goes from 34 s and 3 failures to
# 575 s and 7, fitting nine components where the truth has four: the window is
# short enough to differentiate the noise. 11 keeps the whole real-data gain,
# costs three synthetic failures rather than six, and halves the time penalty.
# The existing noise floor cannot help here — it tests prominence against the
# noise of the SIGNAL, and what degrades is the prominence in the second
# DERIVATIVE. A d2-referenced floor is what would let this go narrower, and
# is the next thing to build rather than a reason to hold the window wide.
#
# The clamps can only ever NARROW the window relative to the old rule.
SHOULDER_D2_WINDOW_FRACTION = 0.5
SHOULDER_D2_WINDOW_MIN = 11
SHOULDER_D2_WINDOW_MAX = 21


def _d2_window(voltage, signal, peak_indices, n):
    """Odd sample count for the shoulder second derivative.

    Scaled to the narrowest primary's measured width, clamped, and odd.
    Falls back to the old sample-count rule when no width can be measured.
    """
    w_old = min(SHOULDER_D2_WINDOW_MAX, max(SHOULDER_D2_WINDOW_MIN, n // 5))
    w = w_old
    try:
        idx = np.asarray(sorted(int(i) for i in peak_indices), dtype=int)
        if idx.size:
            widths = peak_widths(np.asarray(signal, float), idx,
                                 rel_height=0.5)[0]
            widths = widths[np.isfinite(widths) & (widths > 0)]
            if widths.size:
                w = int(round(SHOULDER_D2_WINDOW_FRACTION * float(np.min(widths))))
    except Exception:
        w = w_old
    w = int(min(SHOULDER_D2_WINDOW_MAX, max(SHOULDER_D2_WINDOW_MIN, w)))
    if w % 2 == 0:
        w += 1
    return min(w, SHOULDER_D2_WINDOW_MAX if SHOULDER_D2_WINDOW_MAX % 2 else
               SHOULDER_D2_WINDOW_MAX - 1)


def _flank_shoulders(voltage, signal, peak_indices, *,
                     min_separation_mV=SHOULDER_MIN_SEPARATION_MV,
                     height_fraction=SHOULDER_HEIGHT_FRACTION,
                     d2_prominence_fraction=SHOULDER_D2_PROMINENCE_FRACTION,
                     parent_share_min=SHOULDER_PARENT_SHARE_MIN,
                     v_min_edge=-np.inf, v_max_edge=np.inf):
    """
    Shoulders, found ON THE FLANKS OF PRIMARY PEAKS, judged locally.

    The primaries are found first. For each one, each flank is walked out to
    the trough separating it from its neighbour (or to the window edge), and
    the search for a shoulder happens INSIDE that span, against that span's
    own baseline and that peak's own height.

    This replaces a global pass that had three faults, all of which follow
    from asking the question of the whole curve at once:

      1. It placed shoulders IN VALLEYS. A trough is a maximum of the
         negative second derivative just as a shoulder is, so the trough
         between two peaks was detected in most cycles of the NNM
         triplicate. A trough also sits at the same voltage every cycle, so
         it scored as highly "persistent" and a persistence test could not
         catch it. Here a candidate must lie strictly between a primary and
         its trough, at least `min_separation_mV` from both, so the trough
         is not a candidate at all.

      2. BOTH GATES WERE GLOBAL: the height floor was a fraction of the
         TALLEST peak in the curve and the d2 prominence was scaled to the
         whole curve's d2 range. On a layered oxide the tallest charge peak
         is a median of 7.6x the smallest, so a clear shoulder on a small
         feature was judged against a peak seven times taller. Both gates
         are now measured against the parent peak: its height above its own
         local baseline, and the d2 range within its own flank.

      3. A REAL FEATURE FLICKERED between primary, shoulder and absent
         across cycles, because its prominence sat near two global
         thresholds and crossed them when a DIFFERENT peak grew. A local
         gate moves with its own parent, so a feature's classification no
         longer depends on what the rest of the curve is doing.

    A fourth global gate survived that change until 1.9.0.57 — a noise floor
    scaled to the whole curve's d2 range — and had all three faults over
    again: it rejected cell B's 3.15 V shoulder in 58 of 67 cycles, kept
    different fractions of the same feature in different cells of one
    triplicate, and lost the feature progressively as the cell aged. It is
    replaced here by `parent_share_min`, which asks the question the floor
    was really for — is the PARENT a feature, or a bump on a flat tail — of
    the parent rather than of the candidate. See the constant for the
    measurement.

    Returns a list of `(index, sigma_V)` pairs: the shoulder's position, and
    a WIDTH MEASURED FROM THE SAME CONSTRUCTION that found it (see
    `_shoulder_sigma`), or NaN where that could not be measured.
    """
    out = []
    n = len(signal)
    if n < 20 or len(peak_indices) == 0:
        return out
    pk = np.asarray(sorted(int(i) for i in peak_indices), dtype=int)
    min_sep = max(3, _voltage_to_samples(min_separation_mV, voltage))

    v_step = float(np.mean(np.abs(np.diff(voltage))))
    w = _d2_window(voltage, signal, peak_indices, n)
    try:
        neg_d2 = -savgol_filter(signal, w, 3, deriv=2, delta=v_step)
    except Exception:
        return out
    curve_range = float(np.nanmax(signal) - np.nanmin(signal))

    # The trough between each adjacent pair of primaries, and the window
    # ends outside the outermost pair. These are the flank boundaries.
    bounds = [0]
    for a, b in zip(pk, pk[1:]):
        bounds.append(int(a + np.argmin(signal[a:b + 1])) if b > a else int(a))
    bounds.append(n - 1)

    for i, p in enumerate(pk):
        left_edge, right_edge = bounds[i], bounds[i + 1]
        for lo, hi in ((left_edge, p), (p, right_edge)):
            if hi - lo < 2 * min_sep + 3:
                continue
            span = slice(lo, hi + 1)
            # The parent's height above the baseline of THIS flank — the
            # trough (or window edge) the flank runs down to.
            base = float(min(signal[lo], signal[hi]))
            local_height = float(signal[p]) - base
            if not np.isfinite(local_height) or local_height <= 0:
                continue
            # IS THE PARENT A FEATURE? The local gates below say whether a
            # candidate matters to its parent; this says whether the parent
            # is worth having shoulders on. Without it, a flat tail whose
            # parent is barely a rise admits every ripple — the local gates'
            # own failure mode. Asked of the parent, not of the candidate,
            # and once per flank rather than once per candidate.
            if curve_range > 0 and local_height / curve_range < parent_share_min:
                continue
            seg = neg_d2[span]
            seg_range = float(np.nanmax(seg) - np.nanmin(seg))
            if not np.isfinite(seg_range) or seg_range <= 0:
                continue
            try:
                cand, _props = find_peaks(
                    seg, prominence=d2_prominence_fraction * seg_range,
                    distance=min_sep)
            except Exception:
                continue
            for c in cand:
                idx = int(lo + c)
                # ON THE FLANK, not at either end of it: away from the
                # parent, and away from the trough. This is the check whose
                # absence put markers in valleys.
                if abs(idx - p) < min_sep:
                    continue
                if min(abs(idx - left_edge), abs(idx - right_edge)) < min_sep:
                    continue
                if not (lo < idx < hi):
                    continue
                if voltage[idx] < v_min_edge or voltage[idx] > v_max_edge:
                    continue
                # LOCAL height gate: a fraction of the parent's own rise,
                # measured from this flank's own baseline.
                if (float(signal[idx]) - base) < height_fraction * local_height:
                    continue
                # And it must sit on the descending side: the curve at the
                # candidate is below the parent and above the flank's foot.
                if not (base <= signal[idx] <= signal[p]):
                    continue
                out.append((idx, _shoulder_sigma(neg_d2, idx)))
    _seen, _uniq = set(), []
    for _i, _sg in sorted(out):
        if _i not in _seen:
            _seen.add(_i)
            _uniq.append((_i, _sg))
    return _uniq


def detect_peaks_single(voltage, dqdv, orientation=None,
                        prominence_fraction=DEFAULT_PROMINENCE_FRACTION,
                         min_distance_mV=DEFAULT_MIN_DISTANCE_MV,
                         min_width_mV=DEFAULT_MIN_WIDTH_MV,
                         shoulder_detection=SHOULDER_DETECTION,
                         shoulder_min_separation_mV=SHOULDER_MIN_SEPARATION_MV,
                         shoulder_height_fraction=SHOULDER_HEIGHT_FRACTION,
                         shoulder_d2_prominence_fraction=SHOULDER_D2_PROMINENCE_FRACTION,
                         edge_exclusion_mV=EDGE_EXCLUSION_MV,
                         truncation_visible_fraction=TRUNCATION_VISIBLE_FRACTION,
                         local_prominence=LOCAL_PROMINENCE,
                         local_prominence_fraction=LOCAL_PROMINENCE_FRACTION,
                         min_height_fraction=MIN_HEIGHT_FRACTION):
    """
    Detect peaks in a single half-cycle with shoulder detection, dedup,
    and truncation handling.
    Peaks near the voltage window edge are assessed using half-width
    asymmetry. If ≥ truncation_visible_fraction of the expected peak is
    present, the peak is retained with is_truncated=True. Below this,
    it is discarded. This produces consistent detection across cycles
    for features that shift into or out of the voltage window.
    """
    if len(voltage) < 10:
        return pd.DataFrame()

    # 1.8.7 wrote `-dqdv if step == 'Discharge' else dqdv`, taking the sign
    # of the lobe FROM THE LABEL. That is the same class of assumption that
    # flagged all 64 LTO half-cycles anomalous in Module 1b.
    #
    # It is not, however, the label's DIRECTION that matters here — using
    # `Dataset.step_direction` in its place inverts every LTO half-cycle and
    # moves the 1.53 V peak by 30-40 mV, because for a negative electrode the
    # useful half-cycle is the one where the cell voltage FALLS, and
    # `io._apply_electrode_convention` has already negated dQ/dV to suit.
    # After that convention, label, direction and sign of dQ/dV no longer
    # agree, and only one of the three is a property of the curve.
    #
    # So measure the curve. The signed area under dQ/dV over the half-cycle is
    # the net charge passed: large, unambiguous, and independent of every
    # label and convention upstream. Orienting on it reproduces 1.8.7 exactly
    # wherever 1.8.7's label was right — verified on all 246 fitted P3 cell A
    # half-cycles and all 19 LTO cell A half-cycles — and is right where a
    # label would not have been. `orientation` overrides it for a caller who
    # knows better; nothing in the pipeline passes one.
    if orientation is None:
        m = np.isfinite(voltage) & np.isfinite(dqdv)
        area = (float(trapezoid(dqdv[m], voltage[m])) if m.sum() > 1
                else float(np.nansum(dqdv)))
        orientation = 1.0 if area >= 0 else -1.0
    signal = dqdv.copy() if orientation > 0 else -dqdv
    # NaN-safe. `dqdv` comes from a to_numeric(errors="coerce"), so one NaN
    # made signal_range NaN, `NaN <= 0` False, and every find_peaks
    # prominence comparison silently False — zero peaks from a clean curve.
    signal_range = float(np.nanmax(signal) - np.nanmin(signal))
    if not np.isfinite(signal_range) or signal_range <= 0:
        return pd.DataFrame()

    data_v_min = voltage.min()
    data_v_max = voltage.max()
    v_min_edge = data_v_min + edge_exclusion_mV / 1000.0
    v_max_edge = data_v_max - edge_exclusion_mV / 1000.0
    min_dist_s = _voltage_to_samples(min_distance_mV, voltage)
    min_width_s = _voltage_to_samples(min_width_mV, voltage)

    # === PASS 1: Standard find_peaks ===
    peak_indices, properties = find_peaks(
        signal,
        prominence=prominence_fraction * signal_range,
        distance=min_dist_s,
        width=min_width_s
    )

    # === PASS 1b: scale-free rescue ===
    # Everything find_peaks can see, then kept on two RATIOS rather than on a
    # fraction of a range one feature may dominate. See LOCAL_PROMINENCE_FRACTION.
    if local_prominence:
        # `prominence=0` asks find_peaks for EVERY local maximum, which on a
        # histogram curve includes single-bin spikes whose width at half
        # prominence is zero — and SciPy warns once per call about it. That is
        # a warning about the sweep's own design, not about the data: the
        # rescue pass exists precisely to look at maxima the first pass
        # rejected, and it then keeps them on two ratios rather than on width.
        # Eleven identical `PeakPropertyWarning: some peaks have a width of 0`
        # lines per dataset, each carrying an ipykernel temp path, buried the
        # profile block they were printed in the middle of. Silenced HERE, at
        # the one call that provokes it, rather than globally — and the fact it
        # was reporting is not lost: `width_undersampled` in the parameter
        # table is the same statement, measured, per component.
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore", message="some peaks have a width of 0")
            warnings.filterwarnings(
                "ignore", category=PeakPropertyWarning)
            _all_idx, _all_props = find_peaks(
                signal, prominence=0, distance=min_dist_s, width=min_width_s)
        if len(_all_idx):
            _hmax = float(np.nanmax(signal))
            _keep = []
            for _i, _p in zip(_all_idx, _all_props["prominences"]):
                if _i in peak_indices:
                    continue
                _h = float(signal[_i])
                if _h <= 0 or _hmax <= 0:
                    continue
                if (_p >= local_prominence_fraction * _h
                        and _h >= min_height_fraction * _hmax):
                    _keep.append((_i, _p))
            if _keep:
                _ki = np.array([k[0] for k in _keep])
                _kp = np.array([k[1] for k in _keep])
                _order = np.argsort(np.concatenate([peak_indices, _ki]))
                peak_indices = np.concatenate([peak_indices, _ki])[_order]
                properties = {"prominences": np.concatenate(
                    [properties["prominences"], _kp])[_order]}

    # Edge handling (primary pass): genuine local maxima near the window
    # limits are NOT dropped here. They flow into the truncation assessment
    # below, which keeps a fully or sufficiently visible peak (flagging
    # is_truncated only where a flank is actually clipped) and discards a
    # peak only when less than truncation_visible_fraction of it is visible.
    # This stops the hard edge cut from discarding real near-limit features
    # before the truncation logic can judge them. The shoulder pass below
    # keeps its edge guard, since second-derivative detection is noise-prone
    # at the scan reversal.
    # (To reinstate a small primary guard against scan-reversal noise, drop
    #  peaks here whose voltage is within a few mV of data_v_min/data_v_max.)

    # === PASS 2: Second-derivative shoulder detection ===
    #
    # KNOWN DEFECTIVE, AND LEFT ALONE UNTIL IT CAN BE FIXED PROPERLY. Three
    # faults are established by inspection of the curves (not by summary
    # statistics, which hid all three):
    #
    # 1. SHOULDERS ARE PLACED IN VALLEYS. On the NNM triplicate a marker lands
    #    in the trough BETWEEN two peaks — 3.47 V on charge, 3.37 V on
    #    discharge — in most cycles. A trough is not a shoulder of anything.
    #    Because a trough sits at the same voltage every cycle it also scores
    #    as highly "persistent", so a persistence test cannot catch it.
    #
    # 2. BOTH GATES ARE GLOBAL. The height floor is a fraction of the TALLEST
    #    peak and the d2 prominence is scaled to the whole curve's range. On a
    #    layered oxide the tallest charge peak is a median of 7.6x the
    #    smallest, so a clear shoulder on a small feature is judged against a
    #    peak seven times taller. Making the floor local (a fraction of the
    #    NEAREST primary) was tried and measurably found more candidates —
    #    but most of the extra ones were valleys, so it amplified fault 1 and
    #    was reverted.
    #
    # 3. A REAL FEATURE FLICKERS BETWEEN CLASSES. The ~3.13 V discharge
    #    feature of NNM cell C is a PRIMARY at cycle 30, ABSENT at cycle 40,
    #    and reappears as a shoulder only if the d2 threshold is dropped
    #    eightfold. Its prominence sits near two different global thresholds
    #    and crosses them as neighbouring peaks grow. That is the root cause
    #    of the cycle-to-cycle instability, and it is in the PRIMARY pass as
    #    much as here.
    #
    # A correct pass would find the primaries first, then ask of each one
    # whether its own flanks carry either a bulge or an incompletely resolved
    # second peak, judged against that peak's own prominence and its own local
    # baseline — and would require the candidate to be ON a flank, which is
    # the check whose absence causes fault 1. See the audit note.
    # Shoulders, found on the flanks of the primaries just detected, judged
    # against each primary's own height and its own flank. See
    # `_flank_shoulders` for the three faults of the global pass this
    # replaced.
    shoulder_indices = []
    shoulder_sigma_V = {}
    if shoulder_detection:
        _pairs = _flank_shoulders(
            voltage, signal, peak_indices,
            min_separation_mV=shoulder_min_separation_mV,
            height_fraction=shoulder_height_fraction,
            d2_prominence_fraction=shoulder_d2_prominence_fraction,
            v_min_edge=v_min_edge, v_max_edge=v_max_edge)
        shoulder_indices = [i for i, _ in _pairs]
        # sigma is measured in SAMPLES by `_shoulder_sigma`; convert here,
        # where the voltage axis is in scope.
        _vstep = float(np.mean(np.abs(np.diff(voltage)))) if len(voltage) > 1 \
            else np.nan
        shoulder_sigma_V = {i: (sg * _vstep if np.isfinite(sg) else np.nan)
                            for i, sg in _pairs}
        # A shoulder that coincides with a primary is not a second feature.
        if len(peak_indices):
            shoulder_indices = [i for i in shoulder_indices
                                if np.min(np.abs(i - np.asarray(peak_indices)))
                                >= max(3, _voltage_to_samples(
                                    shoulder_min_separation_mV, voltage))]

    # === MERGE ===
    all_indices = sorted(set(list(peak_indices) + shoulder_indices))
    if len(all_indices) == 0:
        # THE CURVE'S OWN MAXIMUM IS NOT NOTHING. `_ensure_curve_maximum`
        # further down is the safety net for a half-cycle whose tallest
        # feature the primary pass rejected, but this early return jumped
        # over it, so a half-cycle that found nothing returned an empty
        # table instead of its own biggest peak.
        #
        # It was invisible because a second accident covered it. On LTO the
        # discharge peak at 1.5795 V carries the ENTIRE signal range
        # (prominence 23,110 of a 23,110 range) and is 2.6 mV wide — below
        # `min_width_mV = 5`, which the sharp profile sets — so the primary
        # pass has always rejected it. The old GLOBAL shoulder pass then
        # found it on the second derivative, and `_ensure_curve_maximum`
        # promoted it back to a primary. Two bugs cancelling: LTO's single
        # most important feature reached the table only because a shoulder
        # pass that was looking for something else happened to trip over it,
        # and it arrived with a fabricated width of 0.0 mV.
        #
        # Replacing the shoulder pass with one that hangs off the primaries
        # removed the accident and exposed the fault. The net is now reached
        # on this path too, so the behaviour no longer depends on it.
        #
        # THE UNDERLYING FAULT IS STILL THERE and is recorded in the audit
        # note: a minimum-width filter that rejects a peak carrying the whole
        # curve is wrong, and `min_width_mV` wants measuring on the sharp
        # profile rather than adjusting on a hunch.
        _rescued = _ensure_curve_maximum(
            pd.DataFrame(), voltage, signal, min_distance_mV=min_distance_mV)
        if _rescued is None or len(_rescued) == 0:
            return pd.DataFrame()
        return _rescued

    all_indices = np.array(all_indices)

    # Widths
    try:
        # `import warnings` used to sit HERE, inside the function body, which
        # made `warnings` a local name for the whole of `detect_peaks_single`
        # — so the module-level import could not be used anywhere earlier in
        # it. The module imports it now; this line was the only reason it
        # could not.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            wr = peak_widths(signal, all_indices, rel_height=0.5)
        left_v_arr = np.interp(wr[2], np.arange(len(voltage)), voltage)
        right_v_arr = np.interp(wr[3], np.arange(len(voltage)), voltage)
        width_V = right_v_arr - left_v_arr
    except Exception:
        width_V = np.full(len(all_indices), 0.05)
        left_v_arr = voltage[all_indices] - 0.025
        right_v_arr = voltage[all_indices] + 0.025

    # A SHOULDER'S WIDTH, FROM ITS INFLECTION POINTS.
    # `peak_widths` returns 0 for a shoulder because scipy never saw it as a
    # maximum. That zero was printed in the reference-peak table beside
    # measured widths, and it left the fit with no starting width for the
    # majority of components on a broad profile — where most reference peaks
    # ARE shoulders. `_shoulder_sigma` measures it from the second
    # derivative's own sign changes, which sit at centre +/- sigma for a
    # Gaussian; FWHM = 2*sigma for the pseudo-Voigt this pipeline fits.
    if shoulder_sigma_V:
        width_V = np.asarray(width_V, float).copy()
        for _k, _ix in enumerate(all_indices):
            _sg = shoulder_sigma_V.get(int(_ix), np.nan)
            if (np.isfinite(_sg) and _sg > 0
                    and not (np.isfinite(width_V[_k]) and width_V[_k] > 0)):
                width_V[_k] = 2.0 * _sg
                left_v_arr[_k] = voltage[_ix] - _sg
                right_v_arr[_k] = voltage[_ix] + _sg

    # Prominence array
    # A shoulder has no measured prominence — find_peaks never saw it as a
    # maximum. The placeholder below is what ranks it in deduplication, so
    # it is kept, but `prominence_estimated` says which values are real.
    prom_array, prom_estimated = [], []
    for p in all_indices:
        if p in peak_indices:
            prom_array.append(
                properties['prominences'][list(peak_indices).index(p)])
            prom_estimated.append(False)
        else:
            prom_array.append(signal[p] * 0.1)
            prom_estimated.append(True)

    is_shoulder = [p in shoulder_indices for p in all_indices]

    # === TRUNCATION ASSESSMENT ===
    # Every clipped peak is FLAGGED and kept — see TRUNCATION_DROP. A peak
    # below the visible-fraction threshold is flagged truncated too, because
    # "we could not measure how much of it is here" is a stronger reason to
    # doubt its area than "we measured, and most of it is".
    is_truncated_list = []
    drop_list = []
    for k, idx in enumerate(all_indices):
        centre_v = voltage[idx]
        truncated, drop = _assess_truncation(
            centre_v, left_v_arr[k], right_v_arr[k],
            data_v_min, data_v_max,
            truncation_visible_fraction
        )
        is_truncated_list.append(bool(truncated or (drop and
                                                    not TRUNCATION_DROP)))
        drop_list.append(bool(drop and TRUNCATION_DROP))

    # Filter: keep non-dropped peaks
    keep_mask = np.array([not d for d in drop_list])
    if not keep_mask.any():
        return pd.DataFrame()

    all_indices   = all_indices[keep_mask]
    left_v_arr    = left_v_arr[keep_mask]
    right_v_arr   = right_v_arr[keep_mask]
    width_V       = width_V[keep_mask]
    prom_array    = [p for p, k in zip(prom_array, keep_mask) if k]
    prom_estimated = [e for e, k in zip(prom_estimated, keep_mask) if k]
    is_shoulder   = [s for s, k in zip(is_shoulder, keep_mask) if k]
    is_truncated_list = [t for t, k in zip(is_truncated_list, keep_mask) if k]

    peaks_df = pd.DataFrame({
        'peak_id':        range(1, len(all_indices) + 1),
        'voltage':        voltage[all_indices],
        'height':         signal[all_indices],
        'height_original': dqdv[all_indices],
        'prominence':     prom_array,
        # False = measured by find_peaks; True = a shoulder's placeholder,
        # 10% of its height, kept only so shoulders can be ranked.
        'prominence_estimated': prom_estimated,
        'width_V':        width_V,
        'left_voltage':   left_v_arr,
        'right_voltage':  right_v_arr,
        'peak_index':     all_indices,
        'is_shoulder':    is_shoulder,
        'is_truncated':   is_truncated_list,
    })

    peaks_df = peaks_df.sort_values('voltage').reset_index(drop=True)
    peaks_df['peak_id'] = range(1, len(peaks_df) + 1)

    # A SHOULDER THAT IS THE TALLEST FEATURE HAS NO PARENT. Done before
    # deduplication, because the fabricated shoulder prominence is what ranks
    # a peak there, and a promoted peak's remeasured prominence is the number
    # that should decide.
    peaks_df = _ensure_curve_maximum(peaks_df, voltage, signal,
                                     min_distance_mV=min_distance_mV)

    # === DEDUPLICATION ===
    peaks_df = _deduplicate_peaks(peaks_df, shoulder_min_separation_mV,
                                  primary_min_separation_mV=min_distance_mV)

    # === THE NOISE FLOOR ===
    # After deduplication, so a candidate is judged once and in its final
    # form, and never applied to the curve's own maximum — `signal` has one
    # largest value and it is a feature by construction. See NOISE_FLOOR.
    if NOISE_FLOOR and len(peaks_df) > 1:
        _h = peaks_df["height"].to_numpy(float)
        _hmax = float(np.nanmax(np.abs(_h))) if _h.size else 0.0
        _tallest = int(np.argmax(np.abs(_h))) if _h.size else -1
        _keep, _why = [], []
        for _k in range(len(peaks_df)):
            if _k == _tallest or _hmax <= 0:
                _keep.append(True); _why.append(np.nan)
                continue
            _sd = local_noise(signal, int(peaks_df["peak_index"].iloc[_k]))
            _pr = float(peaks_df["prominence"].iloc[_k])
            _snr = (_pr / _sd) if (np.isfinite(_sd) and _sd > 0) else np.inf
            _share = abs(float(_h[_k])) / _hmax
            _keep.append(not (_snr < NOISE_SNR_MIN
                              and _share < NOISE_SHARE_MIN))
            _why.append(_snr)
        if not all(_keep):
            peaks_df = peaks_df[np.array(_keep)].reset_index(drop=True)
            peaks_df["peak_id"] = range(1, len(peaks_df) + 1)

    if len(peaks_df) > MAX_PEAKS_PER_HALF_CYCLE:
        peaks_df = peaks_df.nlargest(MAX_PEAKS_PER_HALF_CYCLE, "prominence")
        peaks_df = peaks_df.sort_values("voltage").reset_index(drop=True)
        peaks_df["peak_id"] = range(1, len(peaks_df) + 1)
        peaks_df["peaks_capped"] = True
    else:
        peaks_df["peaks_capped"] = False

    return peaks_df


# =============================================================================
# NEW IN 1.9.0
# =============================================================================

class DetectSpec:
    """
    The detection parameters as a value object.

    Same rationale as `fitting.FitSpec`: no reference to a dataset, a notebook
    global or a file, so it can be constructed in a test, written into a run
    manifest, and compared between runs. The defaults are 1.8.7's.
    """
    __slots__ = ("prominence_fraction", "min_distance_mV", "min_width_mV",
                 "shoulder_detection", "shoulder_min_separation_mV",
                 "shoulder_height_fraction", "shoulder_d2_prominence_fraction",
                 "edge_exclusion_mV", "truncation_visible_fraction")

    def __init__(self,
                 prominence_fraction=DEFAULT_PROMINENCE_FRACTION,
                 min_distance_mV=DEFAULT_MIN_DISTANCE_MV,
                 min_width_mV=DEFAULT_MIN_WIDTH_MV,
                 shoulder_detection=SHOULDER_DETECTION,
                 shoulder_min_separation_mV=SHOULDER_MIN_SEPARATION_MV,
                 shoulder_height_fraction=SHOULDER_HEIGHT_FRACTION,
                 shoulder_d2_prominence_fraction=SHOULDER_D2_PROMINENCE_FRACTION,
                 edge_exclusion_mV=EDGE_EXCLUSION_MV,
                 truncation_visible_fraction=TRUNCATION_VISIBLE_FRACTION):
        self.prominence_fraction = float(prominence_fraction)
        self.min_distance_mV = float(min_distance_mV)
        self.min_width_mV = float(min_width_mV)
        self.shoulder_detection = bool(shoulder_detection)
        self.shoulder_min_separation_mV = float(shoulder_min_separation_mV)
        self.shoulder_height_fraction = float(shoulder_height_fraction)
        self.shoulder_d2_prominence_fraction = \
            float(shoulder_d2_prominence_fraction)
        self.edge_exclusion_mV = float(edge_exclusion_mV)
        self.truncation_visible_fraction = float(truncation_visible_fraction)

    def replace(self, **kw):
        d = {k: getattr(self, k) for k in self.__slots__}
        d.update(kw)
        return DetectSpec(**d)

    def as_dict(self):
        return {k: getattr(self, k) for k in self.__slots__}

    def __repr__(self):
        return (f"DetectSpec(prom={self.prominence_fraction}, "
                f"dist={self.min_distance_mV:.0f} mV, "
                f"shoulders={self.shoulder_detection}, "
                f"visible>={self.truncation_visible_fraction:.0%})")


@dataclass
class Detection:
    """One dataset's detected peaks, and the reference cycle they anchor to."""
    name: str
    peaks: dict = field(default_factory=dict)          # (cycle, step) -> DataFrame
    reference_cycle: int = DEFAULT_REFERENCE_CYCLE
    reference_reason: str = ""
    # How bad the choice was, decided where the outcome is known rather than
    # guessed from the sentence above. See REF_OK / REF_WARN / REF_BAD.
    reference_severity: str = "ok"
    reference_peaks: dict = field(default_factory=dict)   # step -> DataFrame
    spec: DetectSpec = None
    skipped: dict = field(default_factory=dict)        # (cycle, step) -> why

    def counts(self, step):
        """Peaks detected per cycle for one step, in cycle order."""
        return np.array([len(df) for (c, s), df in sorted(self.peaks.items())
                         if s == step], dtype=int)

    def reference_voltages(self, step):
        df = self.reference_peaks.get(step)
        if df is None or df.empty:
            return np.empty(0)
        return df["voltage"].values

    def summary(self):
        """The console block, as a string rather than a side effect.

        Was 1.8.7's `"=" * 60` banners; now the same three-column idiom the
        rest of the notebook uses, so Cell 8 stops being an unstyled island
        between Cells 6 and 9. The CONSTANTS moved to the end and print once
        per dataset rather than being announced as though they were findings.
        """
        out = [entry("reference cycle", str(self.reference_cycle))]
        out.append(bullet(self.reference_reason))
        for step in ("Charge", "Discharge"):
            rp = self.reference_peaks.get(step)
            if rp is not None and not rp.empty:
                parts = []
                for _, pk in rp.iterrows():
                    tag = ("*" if pk["is_shoulder"] else "") + \
                          ("†" if pk["is_truncated"] else "")
                    parts.append(f'{pk["voltage"]:.3f}{tag}')
                notes = []
                if rp["is_shoulder"].any():
                    notes.append("* shoulder, found on the second derivative")
                if rp["is_truncated"].any():
                    notes.append("† truncated, fitted one-sided")
                out.append(entry(f"{step.lower()} peaks",
                                 f"{', '.join(parts)} V",
                                 "; ".join(notes)))
            c = self.counts(step)
            if c.size:
                extras = []
                nsh = sum(int(df["is_shoulder"].sum())
                          for (cy, s), df in self.peaks.items()
                          if s == step and not df.empty)
                ntr = sum(int(df["is_truncated"].sum())
                          for (cy, s), df in self.peaks.items()
                          if s == step and not df.empty)
                if nsh:
                    extras.append(f"{nsh} shoulders")
                if ntr:
                    extras.append(f"{ntr} truncated")
                out.append(entry(f"{step.lower()}, all cycles",
                                 f"median {np.median(c):.0f}",
                                 f"range {c.min()}-{c.max()}"
                                 + ((", " + ", ".join(extras)) if extras else "")))
                # A PEAK LIST THAT ONLY THIS CYCLE HAS is the failure this
                # whole selector exists to avoid: the reference's components
                # are fitted and tracked in every cycle, so one it alone
                # produces is carried through the entire dataset and reported
                # with an empty drift. Said here, where it can still be acted
                # on by setting `reference_cycle` in Cell 3b.
                rn = len(rp) if rp is not None else 0
                if rn and c.size > 2 and rn != int(np.median(c)):
                    out.append(verdict(
                        "caution",
                        f"the reference cycle has {rn} {step.lower()} peak(s) "
                        f"where the median cycle has {int(np.median(c))}"))
                    out.append(bullet(
                        "Every cycle is fitted with the reference's list, so "
                        "a component only this cycle shows is tracked through "
                        "the whole dataset and reported with no drift."))
        if self.skipped:
            out.append(entry("not detectable",
                             f"{len(self.skipped)} half-cycle(s)"))
        if self.spec is not None:
            out.append(bullet(
                f"edge exclusion ±{self.spec.edge_exclusion_mV:.0f} mV; "
                f"truncation threshold "
                f"{self.spec.truncation_visible_fraction:.0%} visible",
                indent=2))
        return "\n".join(out)


# --- is this maximum a property of the cell, or of the bin edges? ---------
# THE PHASE TEST. A histogram dQ/dV counts records into voltage bins, and on a
# constant-current step with time-based logging the records arrive at a fixed
# CHARGE interval, so their voltage spacing is dQ_per_record / (dQ/dV): narrow
# on a peak, wide on a shelf. Where the bins are finer than that spacing they
# hold one record or none, the pre-smoothing curve is a telegraph signal
# between integer multiples of dQ_per_record / bin_width, and the kernel turns
# it into a ripple at the BEAT PERIOD between the record spacing and the bin
# width. It looks like a series of shallow, evenly-ish spaced maxima on the
# flattest part of the curve, and it passes every test we had:
#
#   - the noise floor, because it measures point-to-point scatter and the
#     ripple is correlated over several bins (measured SNR 16-30 against a
#     floor of 12);
#   - the height-share exemption, because a shelf at half the main peak's
#     height puts every wiggle on it above 10% of the curve;
#   - the recurrence pass, because a beat recurs -- that is what a beat is.
#
# Measured on NMC111 cell A, cycle 5 charge, 4.10-4.45 V: dQ per record 0.770
# mAh/g constant to 0.45%, dV per record 6.6-7.1 mV, Q(V) a straight line --
# a shelf with nothing in it. Binned at 7.2 mV that is a quantum of 107
# mAh/g/V against a shelf standing at 118, and detection found four maxima on
# it. The same records through the 1.8.7 derivative path, which has no bins to
# beat against, give 6.5% modulation and no maxima at all.
#
# The test needs no model and no threshold on any amplitude: MOVE THE BIN
# EDGES. Shifting them by a fraction of a bin changes nothing about the data,
# the width or the smoothing -- a feature that sits at a potential does not
# care, and a beat slides, because its phase is set by where the edges fall.
# On NMC111 cycles 5 and 6 charge, four phase shifts leave exactly three
# maxima standing (3.66, 3.76, 3.93 V) and move every one above 4.05 V by
# 50-120 mV.
#
# Measured cost across the three chemistries (medians per half-cycle):
#
#            candidates before -> after     persistent features lost
#   LTO           1.0  ->  1.0  (sd 0)              none
#   NNM           5.0  ->  4.0                      none
#   NMC           4.0  ->  2.0  (sd 1.89 -> 0.90)   4.08, 4.24, 4.37 V
#
# Every feature that persists across cycles in LTO and NNM survives, including
# the P3 discharge shoulder at 3.13 V that 1.9.0.55 was built to catch. What
# goes is the NMC shelf, and the count stops swinging between cycles.
#
# The TALLEST candidate is exempt, for the same reason it is exempt from the
# noise floor: the curve has one largest value and it is a feature by
# construction. Its own position can wander by more than the tolerance when
# the maximum is broad and flat, which is a statement about the centre of a
# broad feature, not about whether the feature is there.
PHASE_INVARIANCE = True
PHASE_FRACTIONS = (0.2, 0.4, 0.6, 0.8)
# How far a maximum may move between phases and still count as the same one.
# Three terms, largest wins: a floor in mV, a floor in bins, and -- the term
# that matters -- half the candidate's own width, because the argmax of a
# broad feature is not determined to the same precision as that of a narrow
# one and asking it to hold 20 mV is asking the wrong question.
PHASE_TOLERANCE_MV = 20.0
PHASE_TOLERANCE_BINS = 3.0
PHASE_TOLERANCE_WIDTH_FRACTION = 0.5


def phase_stable(sig, peaks_df, spec, *, fractions=None):
    """
    Which candidates survive moving the bin edges? See PHASE_INVARIANCE.

    Returns a boolean array aligned with `peaks_df`. All True where the test
    cannot be run -- off the histogram path, or on a curve too short to
    rebuild -- because "not measured" is not "not there".
    """
    n = len(peaks_df)
    keep = np.ones(n, dtype=bool)
    rebin = getattr(sig, "rebin_at_phase", None)
    if not PHASE_INVARIANCE or n == 0 or rebin is None:
        return keep
    fractions = tuple(fractions or PHASE_FRACTIONS)
    if not fractions:
        return keep

    binw = float(getattr(sig, "bin_width_mV", np.nan))
    if not np.isfinite(binw) or binw <= 0:
        return keep

    seen = []
    for frac in fractions:
        try:
            v2, y2 = rebin(frac)
        except Exception:
            return keep                     # cannot test: do not judge
        if v2 is None or len(v2) < 10:
            return keep
        alt = type("PhaseSignal", (), dict(voltage=np.asarray(v2, float),
                                           dqdv=np.abs(np.asarray(y2, float))))()
        try:
            pk2 = detect_peaks_single(
                alt.voltage, alt.dqdv,
                prominence_fraction=spec.prominence_fraction,
                min_distance_mV=spec.min_distance_mV,
                min_width_mV=spec.min_width_mV,
                shoulder_detection=spec.shoulder_detection,
                shoulder_min_separation_mV=spec.shoulder_min_separation_mV,
                shoulder_height_fraction=spec.shoulder_height_fraction,
                shoulder_d2_prominence_fraction=(
                    spec.shoulder_d2_prominence_fraction),
                edge_exclusion_mV=spec.edge_exclusion_mV,
                truncation_visible_fraction=spec.truncation_visible_fraction)
        except Exception:
            return keep
        seen.append(np.asarray(pk2["voltage"], float)
                    if pk2 is not None and len(pk2) else np.empty(0))

    h = np.abs(peaks_df["height"].to_numpy(float))
    tallest = int(np.argmax(h)) if h.size else -1
    v_pk = peaks_df["voltage"].to_numpy(float)
    w_pk = (peaks_df["width_V"].to_numpy(float) * 1000.0
            if "width_V" in peaks_df else np.zeros(n))
    for j in range(n):
        if j == tallest:
            continue                        # a curve's own maximum is a feature
        tol_mV = max(PHASE_TOLERANCE_MV,
                     PHASE_TOLERANCE_BINS * binw,
                     PHASE_TOLERANCE_WIDTH_FRACTION * float(w_pk[j]))
        tol = tol_mV / 1000.0
        for other in seen:
            if other.size == 0 or np.min(np.abs(other - v_pk[j])) > tol:
                keep[j] = False
                break
    return keep


def detect_half_cycle(sig, spec=None):
    """
    Peaks in one `signal.HalfCycleSignal`.

    Returns an empty DataFrame if the half-cycle carries no usable curve —
    a half-cycle that failed preprocessing is not an absence of peaks, and the
    caller records that separately in `Detection.skipped`.
    """
    spec = spec or DetectSpec()
    if sig.voltage.size < 10:
        return pd.DataFrame()
    peaks = detect_peaks_single(
        sig.voltage, sig.dqdv,
        prominence_fraction=spec.prominence_fraction,
        min_distance_mV=spec.min_distance_mV,
        min_width_mV=spec.min_width_mV,
        shoulder_detection=spec.shoulder_detection,
        shoulder_min_separation_mV=spec.shoulder_min_separation_mV,
        shoulder_height_fraction=spec.shoulder_height_fraction,
        shoulder_d2_prominence_fraction=spec.shoulder_d2_prominence_fraction,
        edge_exclusion_mV=spec.edge_exclusion_mV,
        truncation_visible_fraction=spec.truncation_visible_fraction)

    if peaks is None or not len(peaks):
        return peaks
    keep = phase_stable(sig, peaks, spec)
    n_dropped = int((~keep).sum())
    if n_dropped:
        peaks = peaks[keep].reset_index(drop=True)
        peaks["peak_id"] = range(1, len(peaks) + 1)
    # ON THE FUNCTION, NOT ON `.attrs`. `DataFrame.attrs` is not propagated
    # by `pd.concat`, and both seeding passes rebuild every seeded half-cycle's
    # frame with a concat — so this count was destroyed before anything could
    # read it, and nothing read it anyway. It matters more now than it did:
    # until the grid anchor was fixed, `phase_stable` could not reject anything
    # on a half-cycle that opened above the window's foot, so a run has no
    # history to compare against and the number needs to be visible.
    detect_half_cycle.phase_unstable_dropped += n_dropped
    peaks.attrs["phase_unstable_dropped"] = n_dropped
    return peaks


detect_half_cycle.phase_unstable_dropped = 0


# --- how bad is the reference-cycle choice? --------------------------------
# `choose_reference_cycle` returns a sentence, and for a while the per-cell
# report derived its severity from that sentence with
# `"warn" if "failed" in reason else "ok"`. The mapping was exactly inverted:
# the mild outcomes are the ones that say "failed" ("cycle 5 failed the
# integrity check; 6 is the first at or above it that passes" — a sound cycle
# WAS found), and the severe ones contain no such word, so
# "NO cycle passes the integrity check — using 3 unchecked" and
# "FLAGGED ANOMALOUS by the integrity check" both rendered as a green tick.
# The severity is therefore decided HERE, where the outcome is known, and
# carried as a value. A verdict inferred from prose is not a verdict.
REF_OK = "ok"          # a cycle that passed the integrity check
REF_WARN = "warn"      # a fallback, or a cycle that could not be checked
REF_BAD = "bad"        # nothing passed, or the chosen cycle is ANOMALOUS


def choose_reference_cycle(available_cycles, exists, *, integrity=None,
                           default=DEFAULT_REFERENCE_CYCLE,
                           auto=REFERENCE_CYCLE_AUTO, exists_step=None,
                           efficiency=None):
    """
    Pick the cycle whose peak list will anchor the whole dataset, and say why.

    Parameters
    ----------
    available_cycles : sorted sequence of int
    exists : callable(cycle) -> bool
        Whether the cycle has at least one half-cycle to detect on.
    exists_step : callable(cycle, step) -> bool, optional
        Whether that particular half-cycle exists. Used so a cycle is judged
        only on the half-cycles it has; absent, both are required, which is
        1.8.6 behaviour.
    integrity : callable(cycle, step) -> str, optional
        The integrity band. Absent (or `auto=False`) restores 1.8.6 behaviour
        exactly: the default cycle if present, else the second available one.
        Passed in rather than looked up, so this function is testable.
    default : int
    auto : bool

    Returns
    -------
    (cycle, reason)

    efficiency : {cycle: CE percent}, optional
        Where given, `formation_end` reads it and the search starts from the
        cycle at which CE settled rather than from `default`. This is the
        whole point: integrity asks "is this cycle clean?", and a cycle can
        be perfectly clean and still be the wrong fingerprint because the
        cell was still forming. Absent, behaviour is unchanged.

    The rule is LOWEST SOUND AND SETTLED, not best. A fingerprint should come
    from early in the life of the cell — moving further in means tracking
    against a peak list that has already aged — but not from inside
    formation, where the feature has not stopped moving. On the LTO
    triplicate the charge peak sits at 1.520, 1.524, then 1.548 V and stays
    there: a 24 mV jump between cycle 2 and cycle 3 against a measured drift
    of about 1 mV per cycle.

    Only if nothing at or above the start passes does it look below, and it
    takes the highest such cycle — the closest it can get.
    """
    cycles = list(available_cycles)
    if not cycles:
        return default, "no cycles available", REF_BAD
    if exists_step is None:
        exists_step = lambda c, s: True          # noqa: E731

    start, settled_why, settled_ok = default, None, False
    if auto and efficiency:
        settled, settled_why = formation_end(efficiency)
        if settled is not None:
            start, settled_ok = max(default, int(settled)), True
        else:
            # CE never settled. Fall back to the conservative cycle rather
            # than to the 1.8.7 default of 2, which is the case this whole
            # mechanism exists to avoid: a fingerprint taken from inside
            # formation. If the record is too short to reach the fallback,
            # the search below drops to the highest sound cycle it has and
            # says so.
            start = max(default, int(REFERENCE_CYCLE_FALLBACK))

    def _fallback():
        if default in cycles:
            return default, "default"
        c = cycles[1] if len(cycles) > 1 else cycles[0]
        return c, f"default cycle {default} not present"

    if not auto or integrity is None:
        c, why = _fallback()
        if why == "default":
            why = "default (no integrity check available)"
        # Unchecked is not sound. This branch is reached whenever no
        # integrity callable was supplied, and the sentence says so; the
        # severity has to say so too.
        return c, why, REF_WARN

    def _sound(c):
        # Judge only the half-cycles this cycle ACTUALLY HAS. Requiring both
        # meant a dataset with one step direction, or a truncated final
        # cycle, could never pass: the missing half returns "unknown".
        bands = [integrity(c, s) for s in ("Charge", "Discharge")
                 if exists_step(c, s)]
        return bool(bands) and all(b in SOUND_BANDS for b in bands)

    candidates = [c for c in cycles
                  if c >= start and exists(c) and _sound(c)]
    if candidates:
        c = candidates[0]
        if settled_why and not settled_ok:
            return c, (f"formation could not be measured — {settled_why}; "
                       f"fell back to cycle {REFERENCE_CYCLE_FALLBACK} and "
                       f"{c} is the first at or above it that passes"), REF_WARN
        if settled_why and start > default:
            if c == start:
                return c, (f"formation ended here — {settled_why} — and it "
                           f"passes the integrity check"), REF_OK
            return c, (f"formation ended at cycle {start} ({settled_why}), "
                       f"which failed the integrity check; {c} is the first "
                       f"after it that passes"), REF_OK
        why_settled = f"; {settled_why}" if settled_why else ""
        if c == start:
            return (c,
                    f"default, and it passes the integrity check{why_settled}",
                    REF_OK)
        return c, (f"cycle {start} failed the integrity check; "
                   f"{c} is the first at or above it that passes"), REF_OK

    below = [c for c in cycles if c < start and exists(c) and _sound(c)]
    if below:
        c = below[-1]
        if settled_why and not settled_ok:
            return c, (f"formation could not be measured — {settled_why}; "
                       f"the record does not reach cycle "
                       f"{REFERENCE_CYCLE_FALLBACK}, so {c} is the latest "
                       f"sound cycle available"), REF_WARN
        return c, (f"no cycle at or above {start} passes the integrity "
                   f"check; fell back to {c}"), REF_WARN

    c, why = _fallback()
    return c, (f"NO cycle passes the integrity check — using {c} "
               f"unchecked ({why})"), REF_BAD


def detect_spec_for_profile(profile, **overrides):
    """A `DetectSpec` matching the profile of this dataset's own curve.

    `profile` is `signal.classify_dqdv_profile`'s dict, or its class string.
    Anything passed as a keyword overrides the profile's value, so an
    operator setting `peak_min_distance_mV` in Cell 3b still wins.
    """
    cls = (profile.get("class", "broad") if isinstance(profile, dict)
           else str(profile))
    kw = {"min_distance_mV": PROFILE_MIN_DISTANCE_MV.get(
        cls, DEFAULT_MIN_DISTANCE_MV),
        "min_width_mV": PROFILE_MIN_WIDTH_BY_CLASS.get(
            cls, DEFAULT_MIN_WIDTH_MV)}
    kw.update({k: v for k, v in overrides.items() if v is not None})
    return DetectSpec(**kw)


# =============================================================================
# RECURRENCE — a feature is what comes back at the same potential
# =============================================================================
#
# A shoulder on a flank has, by construction, a large HEIGHT and a small
# PROMINENCE, so every prominence gate in this module is hostile to it. The
# NNM discharge shoulder at 3.15 V is the case that proved it. Measured over
# 40 half-cycles of cell A, with no gates applied at all:
#
#     prominence                     median 1.30   (real peaks: 89 and 157)
#     prominence / curve range       median 0.006  (gate wants 0.05)
#     prominence / its own height    median 0.016  (gate wants 0.25)
#     prominence / local noise       median 2.24   (the .52 floor wants 12)
#     height / curve max             median 0.449
#
# It is a genuine local maximum in 28 of those 40 half-cycles and it fails
# every gate. That is not a threshold set badly; prominence is the wrong
# quantity for this feature. At 2.24x the local noise it is, on any ONE
# half-cycle, indistinguishable from noise — and it is obviously real to the
# eye, because it comes back in the same place every cycle.
#
# So this pass asks the question the eye is actually asking. Collect every
# local maximum from every half-cycle of a step, cluster by voltage, and keep
# the clusters that RECUR. It is the same argument that rejected the LTO flank
# bumps (their potentials scattered over 150 mV) and accepted the main peak
# (stable to 0.2 mV): a redox process happens at a potential.
#
# TWO CLAUSES, because recurrence alone is not enough. On NNM cell C a band of
# noise at 2.6-2.7 V recurs in 50-65% of cycles — MORE often than the shoulder
# does — but it is 15% of the curve's height where the shoulder is 46%:
#
#     V       occupancy   share       verdict
#     2.61      0.50       0.15       rejected
#     2.69      0.65       0.15       rejected  (occupancy alone would keep it)
#     3.15      0.55       0.46       KEPT      <- the shoulder
#
# Occupancy >= 0.5 AND share >= 0.25 returns exactly four features on each NNM
# cell independently — 2.09, 3.15, 3.21, 3.53 — with the shoulder among them at
# a prominence two orders of magnitude below the real peaks. The same pair of
# clauses as the `.52` noise floor, used to ADMIT rather than to reject.
#
# THE DENSITY GATE, and why it is not a profile rule. The method needs the
# curve's local maxima to mean something:
#
#     NMC111     7 local maxima per half-cycle
#     NNM       13
#     LTO      110
#
# On LTO it saturates — 70 "recurrent features" over a 1.2 V window — because
# that curve is built from bins holding 0 or 1 raw record, so its local maxima
# are quantisation rather than structure. A noise pre-filter does not rescue it
# (110 -> 96 at 5 sigma, because the quantisation is locally self-similar) and
# it kills the NNM shoulder (lost at 2 sigma). So the pass declines where the
# density says the curve cannot support it. That is a statement about the
# MEASUREMENT, not about the material or the mechanism — the same discipline as
# the noise floor and the width floors. 7-13 against 110 leaves the gate room.
RECURRENCE_SEEDS = True
RECURRENCE_TOLERANCE_MV = 15.0      # how close two maxima must be to be "the same"
RECURRENCE_MIN_OCCUPANCY = 0.50     # in at least half the half-cycles
RECURRENCE_MIN_SHARE = 0.25         # ...and this much of the curve's own height
RECURRENCE_MAX_DENSITY = 30.0       # maxima per half-cycle above which we decline
# A seed must not merely sit NEAR the recurrent voltage; it must be a
# plausible instance of the feature. With noise there is almost always some
# local maximum within the tolerance, so position alone would place a seed on
# every half-cycle whether the feature is there or not. Half the feature's
# usual share is a wide allowance — the shoulder varies 24-42% of the curve
# across the NNM run — and still excludes a half-cycle where the feature is
# genuinely absent.
RECURRENCE_SEED_SHARE_FRACTION = 0.5
# CHECK THE SEPARATION GUARD WHERE THE SEED LANDS. Built and measured in
# 1.9.0.59, and OFF: the bug is real, the repair is one line, and the repair
# costs more than the bug does.
#
# THE BUG. `add_recurrent_seeds` tests `min_distance_mV` against `fv`, the
# dataset-level feature voltage, and then places the seed at that half-cycle's
# own nearest maximum — which its docstring says explicitly, and which can be
# a whole `tolerance_mV` away. So a seed can clear the guard and then land
# inside the peak the guard exists to protect. NNM cell A cycle 6 discharge
# had detected the shoulder at 3.1224 V; the recurrent feature sits at 3.1534,
# 31.0 mV away, clearing the 30 mV guard by one millivolt; the seed was placed
# at 3.1482, which is 25.8 mV from the shoulder already there. Cycle 6 is the
# reference cycle, so both entered the reference list and the split propagated
# to every cycle.
#
# Measured over 152 recurrent seeds on seven cells, the repair is surgical:
#
#     seeds the guard passes at `fv` but would refuse at the placement:  11
#     of those, landing on a neighbour that is ALREADY a shoulder:       11
#     their separations at the placement:                     20.5-28.4 mV
#     legitimate shoulder seeds in NNM's 3.05-3.30 V region, kept:      120
#     their separations:                     30.2-74.5 mV (median 61.6)
#
# The populations do not overlap and the boundary is the existing
# `min_distance_mV`. No new constant, no new threshold.
#
# WHAT IT DOES, AND WHY IT IS OFF. Cells B and C are bit-for-bit unchanged, as
# are LTO and NMC. On cell A it does exactly what it was built to do:
#
#     discharge references                     6 -> 5, matching B and C
#     the shoulder reference   0 pairs "never in consecutive cycles" + 19
#                              "questionable"  ->  58 pairs "coherent"
#     total coherence pairs                          494 -> 518
#     cycles with a component in the shoulder band   127/128 -> 128/128
#     cycles with TWO components in that band         41 -> 10
#
# and then loses more than that elsewhere:
#
#     cycles resolving the MAIN peak (3.185-3.32 V)  125/128 -> 102/128
#     discharge R2                    better on 37/128, median -0.0026
#
# The 26 lost cycles are 102-128, the end of the run, and they are the same
# cycle in every case: 1.9.0.58 fits 3.1375 and 3.2045 with the higher of the
# two REPORTED AS RELIABLE, and 1.9.0.59 fits one component at 3.1692 sitting
# between them, at R2 0.968 against 0.983. The R2 gap says the data supports
# the pair; detection in those late cycles finds only one maximum, and the
# second reference — a duplicate in cycle 6 — was what held them apart at the
# other end of the run.
#
# So the entry at 3.1482 is a duplicate early and load-bearing late, which is
# the same shape as the P3 structural claim retracted in 1.9.0.56: something
# that looks like an artefact in one regime is doing real work in another.
# The honest repair is not this guard but a reference that carries a feature's
# RANGE rather than one voltage, so a feature that migrates 45 mV over 128
# cycles is one entry that moves instead of two entries that do not. That is a
# design change, not a line, and it is not being improvised on the back of
# this one.
RECURRENCE_SEED_GUARD_AT_PLACEMENT = False


def recurrent_maxima(signals, *, tolerance_mV=RECURRENCE_TOLERANCE_MV,
                     min_occupancy=RECURRENCE_MIN_OCCUPANCY,
                     min_share=RECURRENCE_MIN_SHARE,
                     max_density=RECURRENCE_MAX_DENSITY,
                     edge_exclusion_mV=EDGE_EXCLUSION_MV,
                     enabled=RECURRENCE_SEEDS):
    """
    Voltages at which a local maximum recurs across a dataset.

    `signals` is `{(cycle, step): HalfCycleSignal}`. Returns
    `{step: {"features": [...], "density": float, "declined": str or None}}`,
    where each feature is a dict of `voltage`, `occupancy`, `share` and
    `n_cycles`. Nothing is added to any peak table here — see
    `add_recurrent_seeds`.
    """
    out = {}
    if not enabled:
        return out
    tol = float(tolerance_mV) / 1000.0
    for step in ("Charge", "Discharge"):
        keys = [k for k in signals if k[1] == step]
        rows, n_hc = [], 0
        for k in sorted(keys):
            sig = signals[k]
            v = np.asarray(sig.voltage, float)
            y = np.abs(np.asarray(sig.dqdv, float))
            if v.size < 20:
                continue
            n_hc += 1
            idx, _ = find_peaks(y)
            if not len(idx):
                continue
            hmax = float(np.nanmax(y))
            if not np.isfinite(hmax) or hmax <= 0:
                continue
            # THE WINDOW EDGE, excluded here as it is in
            # `detect_peaks_single`. A curve that rises to its limit puts a
            # maximum at the last sample of every half-cycle, which recurs
            # perfectly and is an artefact of where the scan stopped.
            _lo = float(v[0]) + float(edge_exclusion_mV) / 1000.0
            _hi = float(v[-1]) - float(edge_exclusion_mV) / 1000.0
            for j in idx:
                if not (_lo <= v[j] <= _hi):
                    continue
                rows.append((int(k[0]), float(v[j]), float(y[j]) / hmax))
        if not n_hc or not rows:
            continue
        density = len(rows) / float(n_hc)
        entry = {"features": [], "density": density, "declined": None,
                 "n_half_cycles": n_hc}
        if density > float(max_density):
            entry["declined"] = (
                f"{density:.0f} local maxima per half-cycle — this curve's "
                f"maxima are sampling noise rather than structure, so "
                f"recurrence cannot be measured on it")
            out[step] = entry
            continue
        vs = np.array([r[1] for r in rows], float)
        order = np.argsort(vs)
        vs_sorted = vs[order]
        claimed = np.zeros(vs_sorted.size, bool)
        feats = []
        for i in range(vs_sorted.size):
            if claimed[i]:
                continue
            c = float(vs_sorted[i])
            for _ in range(3):              # settle on the local mode
                m = (vs >= c - tol) & (vs <= c + tol)
                if not m.any():
                    break
                c = float(np.median(vs[m]))
            m = (vs >= c - tol) & (vs <= c + tol)
            grp = [rows[j] for j in np.flatnonzero(m)]
            if not grp:
                continue
            occ = len({g[0] for g in grp}) / float(n_hc)
            share = float(np.median([g[2] for g in grp]))
            claimed |= ((vs_sorted >= c - tol) & (vs_sorted <= c + tol))
            if occ >= float(min_occupancy) and share >= float(min_share):
                feats.append(dict(voltage=c, occupancy=occ, share=share,
                                  n_cycles=len({g[0] for g in grp})))
        # Two clusters can settle on centres closer than the tolerance; keep
        # the better-occupied one so a feature is not seeded twice.
        feats.sort(key=lambda d: -d["occupancy"])
        keep = []
        for f in feats:
            if all(abs(f["voltage"] - g["voltage"]) > tol for g in keep):
                keep.append(f)
        entry["features"] = sorted(keep, key=lambda d: d["voltage"])
        out[step] = entry
    return out


def add_recurrent_seeds(peaks, signals, recurrent, *,
                        tolerance_mV=RECURRENCE_TOLERANCE_MV,
                        min_distance_mV=DEFAULT_MIN_DISTANCE_MV,
                        share_tolerance=RECURRENCE_SEED_SHARE_FRACTION):
    """
    Seed each half-cycle with the recurrent features it did not detect itself.

    The seed is placed at the local maximum nearest the recurrent voltage IN
    THAT HALF-CYCLE, not at the run's median — so it is a measurement of this
    curve, not a value copied from another cycle. Where this half-cycle has no
    local maximum near the feature at all, nothing is added: the feature is
    genuinely absent here and inventing it would be fabricating data.

    THE SEPARATION GUARD IS CHECKED WHERE THE SEED LANDS, added 1.9.0.59.
    `min_distance_mV` was tested against `fv`, the dataset-level feature
    voltage — and the paragraph above says in as many words that the seed is
    NOT placed there. The two can differ by up to `tolerance_mV`, and that is
    enough to let a seed through the guard and then land inside the peak the
    guard was protecting.

    It is the phantom reference at 3.1224 V that has been costing the NNM
    shoulder its coherence verdict since 1.9.0.56. Cell A cycle 6 discharge
    had already detected the shoulder at 3.1224. The recurrent feature sits at
    3.1534, which is 31.0 mV away — clear of the 30 mV guard by one
    millivolt — so a seed was placed at that half-cycle's nearest maximum,
    3.1482, which is 25.8 mV from the shoulder already there. Two components
    on one shoulder. Cycle 6 is the reference cycle, so both entered the
    reference list, and from there the split propagated to every cycle: the
    real feature was left with 19 coherence pairs and a `questionable`
    verdict across 128 discharge cycles, where cell C got 30 pairs and
    `coherent` across 122 and cell B 17 and `coherent` across 67.

    Measured over 152 recurrent seeds on seven cells:

        seeds the guard passes at `fv` and would refuse at the placement: 11
        of those, landing on a neighbour that is ALREADY a shoulder:      11
        their separations at the placement:                    20.5-28.4 mV

        legitimate shoulder seeds in the NNM 3.05-3.30 V region kept:    120
        their separations:                    30.2-74.5 mV (median 61.6)
        of those, seeded beside a PRIMARY rather than a shoulder:    114/120

    The two populations do not overlap and the boundary between them is the
    existing `min_distance_mV`, which was not chosen for this. No new constant
    and no new threshold: the rule was always right, it was applied to the
    wrong voltage.

    A WIDTH-RATIO RULE WAS TRIED FIRST AND IS WRONG, recorded so it is not
    tried again. Comparing the separation to the neighbour's FWHM looks
    decisive in aggregate — NNM's seeds sit at a median 0.49 of the
    neighbour's width against NMC's 8.83 — but any threshold in that gap
    refuses 115 of NNM's 131 seeds. It has to: a shoulder legitimately sits
    at about half its parent's FWHM from the parent's centre, so "close
    relative to the width" is the definition of a shoulder, not the signature
    of a duplicate. It would have deleted the shoulder seeding wholesale.

    `peaks` is modified in place; the number of seeds added is returned.
    """
    if not recurrent:
        return 0
    tol = float(tolerance_mV) / 1000.0
    sep = float(min_distance_mV) / 1000.0
    added = 0
    for step, entry in recurrent.items():
        for feat in entry.get("features", []):
            fv = float(feat["voltage"])
            for (c, st), df in list(peaks.items()):
                if st != step or df is None or df.empty:
                    continue
                if (df["voltage"] - fv).abs().min() <= sep:
                    continue                      # already has it
                if len(df) >= MAX_PEAKS_PER_HALF_CYCLE:
                    continue
                sig = signals.get((c, st))
                if sig is None:
                    continue
                v = np.asarray(sig.voltage, float)
                y = np.abs(np.asarray(sig.dqdv, float))
                if v.size < 20:
                    continue
                idx, _ = find_peaks(y)
                if not len(idx):
                    continue
                near = idx[int(np.argmin(np.abs(v[idx] - fv)))]
                if abs(float(v[near]) - fv) > tol:
                    continue                      # not present on this curve
                # THE GUARD AGAIN, AT THE PLACEMENT — see
                # RECURRENCE_SEED_GUARD_AT_PLACEMENT for the measurement and
                # for why this is off. The check above was made against `fv`;
                # this seed is going to `v[near]`, which can be a whole
                # `tolerance_mV` away.
                if (RECURRENCE_SEED_GUARD_AT_PLACEMENT
                        and (df["voltage"] - float(v[near])).abs().min() <= sep):
                    continue                      # already has it, really
                # ...and it has to LOOK like the feature, not merely sit near
                # where the feature is. With noise there is almost always some
                # maximum within the tolerance; a half-cycle where the nearest
                # one is a fraction of the feature's usual height does not have
                # the feature, and seeding it there would be inventing data.
                _hmax = float(np.nanmax(y))
                _share = (float(y[near]) / _hmax) if _hmax > 0 else 0.0
                if _share < float(share_tolerance) * float(feat["share"]):
                    continue
                prom = float(peak_prominences(y, [int(near)])[0][0])
                row = {col: np.nan for col in df.columns}
                row.update(voltage=float(v[near]), height=float(y[near]),
                           height_original=float(np.asarray(sig.dqdv,
                                                            float)[near]),
                           prominence=prom, prominence_estimated=False,
                           peak_index=int(near),
                           peak_id=int(df["peak_id"].max()) + 1,
                           is_shoulder=False, is_truncated=False,
                           is_recurrent=True)
                out = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
                out = out.sort_values("voltage").reset_index(drop=True)
                out["peak_id"] = range(1, len(out) + 1)
                peaks[(c, step)] = out
                added += 1
    # NaN IS NOT False, AND bool(nan) IS True.
    # `row = {col: np.nan for col in df.columns}` then `row.update(...,
    # is_recurrent=True)` adds a column the existing frame does not have, so
    # `pd.concat` back-fills NaN for every peak that was genuinely detected in
    # that half-cycle. This loop only acted when the column was ABSENT, so
    # those NaNs survived to the parameter table and to
    # `*_fitted_parameters.csv` — and `analyse.parameters_frame` reads them as
    # `bool(d.get("is_recurrent", False))`, which is True for NaN. Every
    # detected peak in a seeded half-cycle was therefore published as having
    # been seeded by recurrence rather than found. `infer_missing_seeds` has
    # handled the identical situation correctly since it was written; this
    # function did not.
    for k, df in peaks.items():
        if df is None or df.empty:
            continue
        if "is_recurrent" not in df.columns:
            df["is_recurrent"] = False
        else:
            df["is_recurrent"] = (df["is_recurrent"]
                                  .fillna(False).astype(bool))
    return added


def infer_missing_seeds(peaks, reference_peaks, *,
                        tolerance_mV=SEED_INFERENCE_TOLERANCE_MV,
                        windows=None,
                        enabled=INFER_MISSING_SEEDS):
    """
    Fill single-cycle detection gaps with seeds from the nearest cycle.

    `peaks` is `{(cycle, step): DataFrame}` and is modified in place; the count
    of seeds added is returned. Half-cycles with NO detected peaks at all are
    left alone: nothing was fitted there, and inventing a whole peak list for a
    half-cycle whose curve failed preprocessing would be fabricating data
    rather than bridging a gap.

    `windows` is `{(cycle, step): (v_min, v_max)}`. A SEED MUST LIE INSIDE THE
    HALF-CYCLE IT IS PLACED IN. Without this the position came from a
    NEIGHBOUR and was never checked against this half-cycle's own voltage
    range, and half-cycles do not all cover the same range — a cell that
    starts a cycle part-charged has a charge curve beginning 850 mV above the
    others. NNM cell B cycle 1 charge spans 2.897-4.200 V and was given a seed
    at 2.207 V from cycle 2. The fitter cannot refuse a seed, so it put a
    component there: sigma on the floor, height 190,936, and an analytic area
    of 2791 mAh/g on a cell that holds 110. `reliable` was False, which is the
    only thing that stood between that number and a results table.
    81 components across the NNM triplicate were centred outside their own
    window, 74 of them from this function, and three of those were `reliable`.
    LTO and NMC had none, so this guard cannot change them.

    Returns the count added; seeds refused for falling outside are counted
    separately in `infer_missing_seeds.refused` for the caller to report.

    See INFER_MISSING_SEEDS for why these rows are marked and demoted rather
    than merged in silently.
    """
    infer_missing_seeds.refused = 0
    if not enabled or not reference_peaks:
        return 0
    tol = float(tolerance_mV) / 1000.0
    added = 0
    windows = windows or {}
    for step in ("Charge", "Discharge"):
        ref = reference_peaks.get(step)
        if ref is None or getattr(ref, "empty", True):
            continue
        cycles = sorted({c for (c, s) in peaks if s == step})
        for _, rp in ref.iterrows():
            rv = float(rp["voltage"])
            # Where the peak WAS, cycle by cycle, so a seed can be placed at
            # its neighbour's position rather than at the reference position
            # it may have drifted 40 mV away from by cycle 200.
            found = {}
            for c in cycles:
                df = peaks.get((c, step))
                if df is None or df.empty:
                    continue
                d = (df["voltage"] - rv).abs()
                j = int(d.idxmin())
                if float(d.loc[j]) > tol:
                    continue
                # A DETECTED PEAK CAN ONLY STAND FOR ONE REFERENCE PEAK.
                #
                # A fixed radius alone cannot be right when two reference
                # peaks are closer together than the radius: the same detected
                # maximum then satisfies both, and the weaker of the pair is
                # recorded as present in a cycle that never found it, so no
                # seed is inferred and it is never fitted.
                #
                # Measured on NNM cell A discharge. The reference list carries
                # 3.151 V (the shoulder) and 3.207 V, 56 mV apart against an
                # 80 mV radius. In cycles 1-5 and 8-10 the only maximum near
                # either is at ~3.204, and it was answering for both — so the
                # shoulder was fitted in 2 of the first 10 cycles and inferred
                # in NONE of them, while `infer_missing_seeds` was firing 222
                # times elsewhere on the same dataset. That absence was then
                # read as the feature dying, and written up as chemistry.
                #
                # So the peak has to be NEARER to this reference peak than to
                # any other. 3.204 belongs to 3.207 (3 mV), not to 3.151
                # (53 mV), and the shoulder is correctly a gap to be filled.
                _rv_all = ref["voltage"].to_numpy(float)
                _dv = float(df.loc[j, "voltage"])
                if np.min(np.abs(_rv_all - _dv)) < abs(_dv - rv) - 1e-12:
                    continue                  # that maximum is someone else's
                found[c] = _dv
            if not found:
                continue                      # never seen; not a gap
            for c in cycles:
                if c in found:
                    continue
                df = peaks.get((c, step))
                if df is None or df.empty:
                    continue                  # nothing fitted here at all
                near = min(found, key=lambda k: (abs(k - c), k))
                # INSIDE THIS HALF-CYCLE'S OWN WINDOW, or not at all. See the
                # docstring: the position comes from a neighbour, and a
                # neighbour's voltage range is not this one's.
                _w = windows.get((c, step))
                if _w is not None:
                    _lo, _hi = float(_w[0]), float(_w[1])
                    if not (_lo <= float(found[near]) <= _hi):
                        infer_missing_seeds.refused += 1
                        continue
                row = {col: np.nan for col in df.columns}
                row.update(voltage=found[near],
                           peak_id=int(df["peak_id"].max()) + 1,
                           is_shoulder=bool(rp.get("is_shoulder", False)),
                           is_truncated=bool(rp.get("is_truncated", False)),
                           prominence_estimated=True,
                           is_inferred=True,
                           inferred_from_cycle=int(near))
                out = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
                out = out.sort_values("voltage").reset_index(drop=True)
                out["peak_id"] = range(1, len(out) + 1)
                peaks[(c, step)] = out
                added += 1
    # Every half-cycle now carries the column, so a reader of the table never
    # has to distinguish "not inferred" from "this build did not record it".
    for k, df in peaks.items():
        if df is None or df.empty:
            continue
        if "is_inferred" not in df.columns:
            df["is_inferred"] = False
            df["inferred_from_cycle"] = np.nan
        else:
            df["is_inferred"] = df["is_inferred"].fillna(False).astype(bool)
        # ...AND `is_recurrent` AGAIN, HERE. `add_recurrent_seeds` runs first
        # and normalises its own column, but this function then appends rows
        # built the same way — every existing column NaN — so an inferred seed
        # row picked up `is_recurrent = NaN`, which reads as True. Whichever
        # seeding pass runs last has to leave the flags clean.
        if "is_recurrent" not in df.columns:
            df["is_recurrent"] = False
        else:
            df["is_recurrent"] = df["is_recurrent"].fillna(False).astype(bool)
        for _b in ("is_shoulder", "is_truncated"):
            if _b in df.columns:
                df[_b] = df[_b].fillna(False).astype(bool)
    return added


def detect_all(signals, *, name="", spec=None, reference_cycle=None,
               integrity=None, verbose=True, efficiency=None):
    """
    Detect across a dataset and choose its reference cycle.

    Parameters
    ----------
    signals : {(cycle, step): signal.HalfCycleSignal}
    reference_cycle : int, optional
        An EXPLICIT choice. Supplying it means the user chose it, so it is
        honoured, checked and warned about — never overridden. There is no
        sentinel value and no defaults dict to disambiguate: 1.8.7 inferred
        intent from `'reference_cycle' in params`, which Cell 3 makes true for
        every dataset, and the selector was therefore inert on every real run.
    integrity : callable(cycle, step) -> str, optional
    efficiency : {cycle: CE percent}, optional
        Passed through to `choose_reference_cycle`, which uses it to find the
        end of formation. See `formation_end`.
    """
    spec = spec or DetectSpec()
    peaks, skipped = {}, {}
    detect_half_cycle.phase_unstable_dropped = 0

    for key in sorted(signals):
        sig = signals[key]
        if sig.voltage.size < 10:
            skipped[key] = sig.note or "no usable curve"
            peaks[key] = pd.DataFrame()
            continue
        peaks[key] = detect_half_cycle(sig, spec)

    # --- RECURRENCE: what comes back at the same potential -------------
    # Runs across the whole dataset, after every half-cycle has been detected
    # on its own, because that is the only level at which the question can be
    # asked. See `recurrent_maxima` for the measurement behind the two
    # thresholds and for why it declines on a quantisation-dominated curve.
    # SAID OUT LOUD. See `detect_half_cycle`: the phase-invariance test drops
    # candidates that move with the bin grid rather than with the material,
    # and how many it dropped is a statement about the binning that belonged
    # in the run all along.
    _nph = int(getattr(detect_half_cycle, "phase_unstable_dropped", 0))
    if verbose and _nph:
        print(f"  phase invariance: {_nph} candidate(s) dropped — they moved "
              f"with the bin grid when it was shifted, so they are the "
              f"binning and not the curve")

    recurrent = recurrent_maxima(signals)
    n_seeded = add_recurrent_seeds(peaks, signals, recurrent,
                                   min_distance_mV=spec.min_distance_mV)
    if verbose and recurrent:
        for _st in ("Charge", "Discharge"):
            _e = recurrent.get(_st)
            if not _e:
                continue
            if _e.get("declined"):
                print(f"  recurrence ({_st.lower()}): not measured — "
                      f"{_e['declined']}")
            elif _e["features"]:
                _fv = ", ".join(f"{d['voltage']:.3f} V "
                                f"({d['occupancy']:.0%} of cycles, "
                                f"{d['share']:.0%} of the curve)"
                                for d in _e["features"])
                print(f"  recurrence ({_st.lower()}): {_fv}")
    if verbose and n_seeded:
        print(f"  {n_seeded} recurrent seed(s) added to half-cycles that did "
              f"not detect them on their own")

    n_capped = sum(1 for d in peaks.values()
                   if not d.empty and bool(d.get("peaks_capped",
                                                 pd.Series([False])).any()))
    if n_capped and verbose:
        print(f"\n  *** {n_capped} half-cycle(s) detected more than "
              f"{MAX_PEAKS_PER_HALF_CYCLE} peaks and were capped to the "
              f"strongest. ***")
        print("      That many features on one dQ/dV curve is a detection "
              "failure, not a rich material — check the smoothing window "
              "and the prominence fraction in Cell 4.")

    cycles = sorted({int(c) for c, _ in signals})

    def _exists(c):
        # "Has something to DETECT ON", not "appears in the dict". Every
        # candidate is drawn from the keys of `signals`, so the membership
        # test was tautologically true and this callable — the one the
        # module docstring introduces to make the selector testable — did
        # nothing. A half-cycle whose curve failed preprocessing has an
        # empty peak table, and a reference cycle with no peaks fixes an
        # empty peak list for the whole dataset.
        return any(not peaks.get((c, s), pd.DataFrame()).empty
                   for s in ("Charge", "Discharge"))

    if reference_cycle is not None:
        ref = int(reference_cycle)
        reason = "set explicitly"
        # An operator-chosen cycle has not been checked unless the integrity
        # callable is present AND returns a sound band. "Set explicitly" on
        # its own is not a pass.
        severity = REF_WARN
        if not cycles:
            # The reference_cycle=None branch handles this cleanly; this one
            # raised IndexError on cycles[0].
            return Detection(name=name, peaks=peaks, skipped=skipped,
                             spec=spec, reference_cycle=ref,
                             reference_reason="no cycles available",
                             reference_severity=REF_BAD,
                             reference_peaks={})
        if ref not in cycles:
            # The NEAREST available cycle, not the second one in the record.
            ref = min(cycles, key=lambda c: (abs(c - ref), c))
            reason = f"requested cycle not present; adjusted to {ref}"
        if integrity is not None:
            # ONLY THE HALF-CYCLES THIS CYCLE HAS, exactly as the auto path
            # does. Asking for both unconditionally meant a charge-only or
            # discharge-only dataset — or a cycle whose second half is missing
            # — always got "unknown" back for the absent one, "unknown" is not
            # in SOUND_BANDS, and so an explicitly chosen reference cycle
            # could NEVER be marked REF_OK: the run page carried a permanent
            # "the integrity check could not assess it" caution that no choice
            # of cycle could clear. `choose_reference_cycle` was given
            # `exists_step` for this reason and this branch was not.
            bands = [integrity(ref, s) for s in ("Charge", "Discharge")
                     if (ref, s) in signals]
            if not bands:
                bands = [integrity(ref, s) for s in ("Charge", "Discharge")]
            # Anything outside SOUND_BANDS is "we could not tell", which the
            # auto path already treats as disqualifying; only ANOMALOUS used
            # to be mentioned, so a cycle whose bands were 'too few records'
            # was accepted with a bare "set explicitly" and no caveat.
            if all(b in SOUND_BANDS for b in bands):
                severity = REF_OK
                reason += "; it passes the integrity check"
            elif "ANOMALOUS" not in bands:
                reason += ("; the integrity check could not assess it ("
                           + ", ".join(str(b) for b in bands) + ")")
            if "ANOMALOUS" in bands:
                severity = REF_BAD
                reason += "; FLAGGED ANOMALOUS by the integrity check"
                if verbose:
                    print(f"\n  *** WARNING: reference cycle {ref} is flagged "
                          f"ANOMALOUS. ***")
                    print("      Its peak list will be fitted and tracked in "
                          "every cycle of this dataset.")
                    print("      Pass reference_cycle=None to let Ratatosk "
                          "choose.")
    else:
        ref, reason, severity = choose_reference_cycle(
            cycles, _exists, integrity=integrity, efficiency=efficiency,
            exists_step=lambda c, st: (c, st) in signals)

    ref_peaks = {s: peaks.get((ref, s), pd.DataFrame())
                 for s in ("Charge", "Discharge")}
    # Each half-cycle's own voltage range, so an inferred seed can be checked
    # against the data it is about to be fitted to rather than against the
    # neighbour it was copied from.
    _windows = {}
    for _k, _sig in (signals or {}).items():
        try:
            _v = np.asarray(_sig.voltage, float)
            if _v.size:
                _windows[_k] = (float(np.nanmin(_v)), float(np.nanmax(_v)))
        except Exception:
            continue
    n_inferred = infer_missing_seeds(peaks, ref_peaks, windows=_windows)
    _refused = int(getattr(infer_missing_seeds, "refused", 0))
    if verbose and _refused:
        print(f"\n  {_refused} inferred seed(s) REFUSED: the neighbouring "
              f"cycle's position lies outside that half-cycle's own voltage "
              f"range.")
        print("    A seed outside the data is not a gap to bridge; the "
              "component it would have produced is not a measurement.")
    if verbose and n_inferred:
        print(f"\n  {n_inferred} seed(s) inferred from neighbouring cycles "
              f"where detection found nothing at a reference peak.")
        print("    They are fitted and tracked, marked is_inferred, and "
              "excluded from headline trends.")

    det = Detection(
        name=name, peaks=peaks, reference_cycle=int(ref),
        reference_reason=reason, reference_severity=severity,
        spec=spec, skipped=skipped, reference_peaks=ref_peaks)

    if verbose:
        print("\n" + det.summary())
    return det


In [ ]:
# DEFINITIONS D — BUILDING AND RUNNING THE MODEL
# =============================================================================
"""
Model construction and the fitting engine.

The one design decision that matters
------------------------------------
A worker returns a plain dictionary, never an lmfit `ModelResult`.

In 1.8.x, parallel fitting was blocked because a `ModelResult` cannot be
returned through stdlib pickle: `PolynomialModel` builds its function as a
closure, so only cloudpickle could carry it, which in turn meant only the
`loky` backend, which in turn meant worrying about which globals were
statically referenced from a notebook cell.

None of that is necessary. The optimiser's output that anyone downstream
actually uses is a few dozen floats. Returning those makes every backend work,
makes the payload small, and makes a fit result something that can be written
to disk and compared — which is what the 1.9.0 regression test needs.

Parallelism
-----------
`fit_many` measures before it commits: it times a few half-cycles serially,
estimates the total, and only spawns workers if the estimate clears
`PARALLEL_MIN_SECONDS`. Process start-up on Windows is expensive and a 40-second
job does not repay it. Every branch says out loud which way it went.

It measures MEMORY as well as time, and that is not a refinement — it is the
fix for a kernel that died twice in Cell 9. The old rule was `workers =
cpu_count - 1`, decided from a time estimate alone, on a machine where every
`spawn` worker re-imports numpy, scipy and lmfit and holds that footprint for
as long as it lives. Measured here: **181 MB per worker** (Linux, CPython
3.11), against a job payload of **5 KiB** — so the cost of a worker is the
cost of its imports, not of the data it is sent, and fifteen of them is two to
three gigabytes before a single fit is run.

Three things follow, and all three are implemented below:

* the pool is sized against FREE MEMORY as well as cores, using a per-worker
  cost measured on the machine actually running (`measure_worker_cost_mb`),
  not a constant;
* `PARALLEL_MAX_WORKERS` caps it regardless, because the twelfth worker buys
  little and costs another import footprint plus another Windows spawn;
* `release_workers()` exists and is called at the end of a fitting stage,
  because joblib keeps its workers alive after `Parallel` returns — measured,
  and NOT undone by exiting a `with Parallel(...)` block.

It is a ceiling, not a leak: one worker's RSS over eighty consecutive NNM
fits oscillated between 168 and 203 MB with no trend.
"""


import os
import time
import numpy as np


from lmfit.models import PseudoVoigtModel, PolynomialModel
from lmfit import Model as _LMModel
from scipy.special import erf


# Below this estimated total, run serially. Measured on Windows/Anaconda: a
# loky pool costs ~5-8 s to start, so anything under a minute loses.
PARALLEL_MIN_SECONDS = 60.0

# --- How many workers a machine can afford --------------------------------
# A worker is a fresh interpreter that re-imports numpy, scipy and lmfit and
# then holds that footprint. Measured: 181 MB (Linux, CPython 3.11) against a
# 5 KiB job payload. `cpu_count - 1` on a 16-thread laptop is therefore ~2.7
# GB spent before the first fit, which is what killed the kernel in Cell 9 on
# a machine already at 80% RAM.

# Hard ceiling regardless of cores or memory. Past about eight workers the
# marginal fit rate is small, and each one costs another import footprint and
# another 5-8 s Windows spawn.
PARALLEL_MAX_WORKERS = 8

# Never plan to spend more than this share of the memory that is FREE RIGHT
# NOW. Half leaves room for the parent to keep loading data, for the operating
# system, and for whatever else the person has open — which on the machine
# that crashed was most of it.
WORKER_MEMORY_HEADROOM = 0.5

# The probe measures what a worker costs to START — its imports plus one
# small fit. A worker doing real broad-profile fitting settles higher, because
# lmfit and scipy allocate as they go: measured on the NNM triplicate, 150 MB
# imported against 181 MB in use, so 1.21. Rounded up, because the direction
# that is wrong here kills a kernel and the direction that is wrong the other
# way costs a worker.
WORKER_WORKING_SET_FACTOR = 1.25

# Used ONLY when the probe cannot run at all. Above the Linux measurement
# because Windows/Anaconda imports cost more, and an assumption that is too
# small is the failure being fixed.
WORKER_RSS_ASSUMED_MB = 250.0

# Sanity bounds on the probe, not a substitute for it. A measurement outside
# these is not a worker cost, it is a broken measurement.
WORKER_RSS_SANITY_MIN_MB = 80.0
WORKER_RSS_SANITY_MAX_MB = 2000.0

# Start one worker and ask it what it costs, once per session, before sizing
# the pool. Set False to skip the probe and use WORKER_RSS_FLOOR_MB instead.
WORKER_PROBE = True

# lmfit re-derives `fwhm` and `height` through asteval on every residual
# evaluation — 47.7% of fit time in 1.8.6, for quantities the optimiser never
# reads. Deleted before the fit, recomputed after (see `_derived`).
DROP_DERIVED = True

_FWHM_FACTOR = 2.0          # PseudoVoigtModel: fwhm = 2 * sigma
# sigma_gaussian = sigma / sqrt(2 ln 2), so the Gaussian and the Lorentzian
# halves of a pseudo-Voigt share one FWHM. See rect_pseudo_voigt.
_SIGMA_G_FACTOR = np.sqrt(2.0 * np.log(2.0))
_TINY = 1.0e-15


# --- NO FREE BASELINE, AND BANDS SEEDED FROM WHAT IS LEFT OVER ------------
# In a dQ/dV there is no instrumental background. Every mAh in the curve is
# charge that passed through the cell, so a "baseline" is not a thing to be
# subtracted -- it is an unnamed component that is then left out of the
# capacity attribution.
#
# It was also carrying most of the answer. Measured over 136 half-cycles from
# six cells, fraction of the cell's charge the model could NAME:
#
#                cubic baseline     no baseline
#     LTO             98.2%           100.3%      (and the cubic held 41.9%
#     NNM             62.4%            95.1%       of the area while the
#     NMC             33.1%            96.3%       components summed to 138%)
#
# On LTO the polynomial was going NEGATIVE to cancel components that had
# overshot by 38% -- a decomposition no fit statistic would ever have shown.
#
# What removing it exposes is that a band cannot be seeded from peak
# detection. A band is a REGION, not a maximum, and NMC111's 4.1-4.45 V shelf
# contains no maximum at all once the binning artefact is gone. With nothing
# to hang a band on, the fitter reaches that charge by stretching the Ni peak
# sideways: median FWHM 169 -> 329 mV.
#
# So bands are seeded from the CHARGE THE PEAKS CANNOT ACCOUNT FOR. Fit the
# peaks; find the contiguous span where the curve exceeds them; if it carries
# a material share of the step, put a band there and refit. No flatness
# criterion, no threshold tuned per chemistry, and it is the same question the
# report has to answer anyway -- which charge can we name?
#
# It lands in the same place every time: NMC111 4.094-4.448 V in 44 of 48
# half-cycles across both cells and twelve cycles, median 12.5% of the step.
# And the refit is the test that it belongs there:
#
#                      components   named    median FWHM   max residual
#     NMC peaks only       3.0       95.2%      328.6 mV       35.2%
#     NMC + band           4.0       99.8%      152.2 mV        9.2%
#
# An extra component that merely soaks up residual leaves the others alone.
# This one HALVES the peak widths, because it takes back work that was never
# theirs. That is the opposite signature to overfitting, and it is the reason
# the band is in the model rather than a fourth peak.
#
# GATED ON `band_width_max`, which only `quality.classify_mechanism` ever
# sets: two-phase and multi-transition mechanisms pass 0.0 and can never
# acquire a band this way. That gate is not decoration. Left ungated on LTO --
# one sharp two-phase peak -- residual seeding pushed the named charge to
# 103.1%, components summing to more than the cell delivered.
BAND_FROM_RESIDUAL = True
BAND_RESIDUAL_MIN_SHARE = 0.02      # of the half-cycle's charge
BAND_RESIDUAL_MAX = 2               # bands added per half-cycle
BAND_RESIDUAL_MIN_SPAN_MV = 100.0   # a band narrower than this is a peak

# --- ONE SET OF BANDS FOR THE DATASET, NOT ONE PER HALF-CYCLE -------------
# Seeding a band from THIS half-cycle's unattributed charge fixes the fit and
# breaks the tracking. The same physical region comes out as a band in one
# cycle and a peak in the next, so the component set changes identity from
# cycle to cycle and nothing downstream can follow it.
#
# Measured on NMC111 cell A charge at 1.9.0.56, twelve half-cycles:
#
#     band at 4.246 V   present in  8/12   spread 74 mV
#     band at 3.723 V   present in  4/12   spread 65 mV
#     band at 3.863 V   present in  2/12   spread 44 mV
#     band at 4.087 V   present in  1/12      —
#
# Four different peak/band patterns across those twelve (bbpp x7, ppp x2,
# bpp x2, bppp x1), five across cell B's seventeen. The coherence audit
# graded every one of cell A's features `not assessable`, and cell B's
# 4.206 V feature came back with a median drift of 106 mV per cycle — a
# tracking failure wearing the clothes of a measurement.
#
# Peaks solved this long ago: detect per half-cycle, then anchor to a
# REFERENCE LIST so a component missing from one cycle is a gap rather than a
# different model (`detect.infer_missing_seeds`). Bands get the same
# treatment. The excess regions are collected across a SAMPLE of half-cycles,
# clustered by voltage, and the recurrent ones become the dataset's band set —
# fitted in every half-cycle of that step whether or not that particular curve
# would have asked for one, because a band that comes and goes is not a
# measurement of anything.
#
# The recurrence is plainly there to find: the 4.19 V discharge band appears
# in 86% and 75% of half-cycles with a 6-9 mV spread.
BAND_REGION_SAMPLE = 12          # half-cycles used to establish the set
BAND_REGION_TOLERANCE_MV = 80.0  # two excess regions this close are the same
BAND_REGION_MIN_OCCUPANCY = 0.34 # ...and it must recur in this share of them
# --- RECORDED NEGATIVE: DO NOT POOL BAND REGIONS ACROSS REPLICATES -------
# `reconcile_mechanisms` pools the MECHANISM across the replicates of a
# material, and pooling the band regions the same way looks like the obvious
# next step. It was built, measured three ways, and it is wrong every time.
#
# The argument does not carry over. A mechanism is a PERMISSION and costs
# nothing where it is not needed; a band region is a COMPONENT, and a
# component has to earn its place in every half-cycle it is fitted to.
#
# NMC111 cell A finds two charge regions (3.712 V at 38%, 4.281 V at 88%);
# cell B's own pass finds none, because its detection already places a maximum
# on that shelf. Handing A's regions to B:
#
#                        cell B    blanket   bar at    promote the
#                        alone      union     0.67     maximum
#     components           81        100        81         69
#     at the ceiling       22         38        31         26
#     median FWHM       217.5 mV   315.3 mV  318.3 mV   268.5 mV
#     reliable           52/81      53/100     41/81      37/69
#     persistent (chg)     3          2          2          1
#
# Every refinement helped and none of them recovered cell B. The at-bound
# count rising is the model saying the imposed component does not belong
# there — the same signal that escalates a mechanism, read the other way.
#
# So the region set is established PER DATASET, from that cell's own
# half-cycles, and stops there. That is where the benefit was all along:
#
#                     sporadic components (<40% of cycles)
#                     per-half-cycle bands -> per-dataset set
#         cell A chg          1 -> 0
#         cell A dis          2 -> 0
#         cell B chg          3 -> 2
#         cell B dis          2 -> 1
#
# The code below is kept and the constant disables it, because the next person
# to have this idea should find the measurement rather than repeat it.
BAND_REGION_SHARED_OCCUPANCY = 1.01   # > 1 disables propagation entirely


def residual_excess_regions(voltage, residual, *, min_span_mV=None):
    """
    Contiguous spans where the curve exceeds the model, largest area first.

    `residual` is curve minus fit, in the curve's own units. Returns dicts of
    `lo`, `hi`, `centre`, `span_mV` and `area`; the area is the charge the
    model has not accounted for in that span.
    """
    v = np.asarray(voltage, float)
    r = np.asarray(residual, float)
    if v.size < 4 or r.size != v.size:
        return []
    min_span_mV = (BAND_RESIDUAL_MIN_SPAN_MV if min_span_mV is None
                   else float(min_span_mV))
    pos = np.isfinite(r) & (r > 0)
    out, i = [], 0
    while i < pos.size:
        if not pos[i]:
            i += 1
            continue
        j = i
        while j + 1 < pos.size and pos[j + 1]:
            j += 1
        span_mV = (v[j] - v[i]) * 1000.0
        if span_mV >= min_span_mV and j > i:
            out.append(dict(lo=float(v[i]), hi=float(v[j]),
                            centre=float(0.5 * (v[i] + v[j])),
                            span_mV=float(span_mV),
                            area=float(trapezoid(r[i:j + 1], v[i:j + 1]))))
        i = j + 1
    return sorted(out, key=lambda d: -d["area"])


# A COMPONENT THAT ENDED AT ITS WIDTH BOUND IN THE PEAKS-ONLY PASS IS A
# REGION, whether or not any residual survived it. New in 1.9.0.60.
#
# `band_regions` fits the sample PEAKS ONLY and asks what the peaks cannot
# account for. That is the right question and it has a blind spot: a peak
# allowed to widen to `sigma_max` can cover a shelf entirely, and then there
# is no residual to find. The pass is blinded by the same bound it exists to
# detect, and the result is self-stabilising — no residual, so no region, so
# no band, so the peak stays at the bound.
#
# Measured on the NMC111 pair, top of charge (>4.15 V), 1.9.0.59:
#
#     cell A   18 components, ALL from detection, 14 fitted as BANDS with a
#              real width (0.31-0.48 V), several with sigma 28-60 mV
#     cell B   17 components, ALL from `add_recurrent_seeds`, ALL peaks, ALL
#              with FWHM exactly 0.400 = the ceiling, ALL `at_sigma_max`,
#              none `reliable`, carrying 37% of the charge-side area
#
# Same material, same run, two models. Cell B's chain is: the recurrent pass
# finds a maximum recurring on the shelf and seeds it AS A PEAK in every
# charge half-cycle; the peak widens to the bound to cover 300 mV of shelf;
# no residual survives; no region is found ("Charge bands none - the peaks
# account for this curve"); the band machinery never engages. Cell A escapes
# only because its own detection does not find that maximum.
#
# TWO EXPLANATIONS WERE TESTED FIRST AND BOTH ARE WRONG, recorded so they are
# not tried again:
#
#   * Cross-replicate propagation - handing cell A's regions to cell B. Built
#     three ways in 1.9.0.56 and every one made the ceiling-pinning WORSE
#     (22 at the ceiling alone, 38/31/26 with propagation). See
#     BAND_REGION_SHARED_OCCUPANCY.
#   * A local occupancy gate on detection, on the theory that the recurring
#     maximum is the quantisation beat on an under-sampled shelf. There is
#     only ONE detected charge peak above 4.10 V across both cells and 35
#     half-cycles, and its local occupancy is 2.14 - well sampled. A gate at
#     occ < 1.0 would reject 16% of NNM's real peaks and nothing on the NMC
#     shelf. Detection is not the fault.
#
# So the region is seeded from the bound itself, per dataset, from that
# cell's own half-cycles - which is where BAND_REGION_SHARED_OCCUPANCY's note
# says the benefit was all along. The span is the component's own bound
# width, which is not a tuned number: it is the widest the model was allowed
# to be, and therefore the least the region can be.
BAND_REGION_FROM_AT_BOUND = True


def band_regions(jobs, *, n=None, tolerance_mV=None, min_occupancy=None,
                 min_share=None, n_jobs=None, verbose=True):
    """
    The dataset's band regions, established once. See BAND_REGION_SAMPLE.

    `jobs` is the production list of (voltage, dqdv, spec, key) with
    `key = (cycle, step)`. An evenly spaced sample is fitted PEAKS ONLY —
    whatever the spec says about bands is set aside, because the question is
    what the peaks alone cannot account for — and the contiguous excess spans
    are clustered by voltage WITHIN EACH STEP, charge and discharge being
    different curves with different regions.

    Returns `{step: [{centre, lo, hi, span_mV, occupancy, area_share}, ...]}`,
    ordered by voltage. A step with no recurrent region returns an empty list,
    which is the right answer for a two-phase cell and is what LTO gets.
    """
    n = int(BAND_REGION_SAMPLE if n is None else n)
    tol = float(BAND_REGION_TOLERANCE_MV if tolerance_mV is None
                else tolerance_mV) / 1000.0
    min_occ = float(BAND_REGION_MIN_OCCUPANCY if min_occupancy is None
                    else min_occupancy)
    min_share = float(BAND_RESIDUAL_MIN_SHARE if min_share is None
                      else min_share)
    out = {}
    if not jobs:
        return out
    # A mechanism that permits no band has no regions to look for, and the
    # sample fits would be spent finding tails it will never use. LTO's
    # residual does produce candidate spans (1.87 V charge, 1.78 V discharge)
    # and every one of them is the flank of its single two-phase peak.
    if not any(float(getattr(sp, "band_ceiling", 0.0) or 0.0) > 0
               for _v, _y, sp, _k in jobs):
        return out

    by_step = {}
    for v, y, sp, k in jobs:
        step = str(k[1]) if isinstance(k, (tuple, list)) and len(k) > 1 else ""
        by_step.setdefault(step, []).append((v, y, sp, k))

    for step, items in by_step.items():
        if len(items) > n:
            idx = np.linspace(0, len(items) - 1, n).round().astype(int)
            sample = [items[i] for i in sorted(set(idx.tolist()))]
        else:
            sample = list(items)
        # Peaks only: bands off, and no per-half-cycle seeding, so the excess
        # measured is what the PEAKS cannot account for.
        batch = [(v, y, sp.replace(band_width_max=0.0, band_ceiling=0.0,
                                   band_from_residual=False,
                                   band_seeds=()), k)
                 for v, y, sp, k in sample]
        res = fit_many(batch, n_jobs=n_jobs, verbose=False, label="band regions")

        found = []
        n_ok = 0
        for (v, y, sp, k), r in zip(sample, res):
            if not r.get("success"):
                continue
            n_ok += 1
            va = np.asarray(v, float)
            ya = np.abs(np.asarray(y, float))
            fit = np.abs(evaluate(r, va)["total"])
            area = float(trapezoid(ya, va))
            if area <= 0:
                continue
            for g in residual_excess_regions(va, ya - fit):
                if g["area"] / area >= min_share:
                    g["key"] = k
                    found.append(g)
            # A PEAK AT ITS WIDTH BOUND IS A BAND BEING DENIED, and this pass
            # is the one place that rule was never applied. See
            # BAND_REGION_FROM_AT_BOUND.
            if BAND_REGION_FROM_AT_BOUND:
                _smax = float(r.get("sigma_max") or 0.0)
                for _c in r.get("components", ()):
                    _sg = abs(float(_c.get("sigma", np.nan)))
                    if not (_smax > 0 and np.isfinite(_sg)
                            and _sg >= _smax * SIGMA_BOUND_PROXIMITY_COMPLEMENT):
                        continue
                    _a = abs(float(_c.get("amplitude_area", 0.0) or 0.0))
                    if area > 0 and _a / area < min_share:
                        continue
                    _ctr = float(_c.get("centre", np.nan))
                    if not np.isfinite(_ctr):
                        continue
                    found.append(dict(lo=_ctr - _smax, hi=_ctr + _smax,
                                      centre=_ctr, span_mV=2000.0 * _smax,
                                      area=_a, from_bound=True, key=k))
        if not n_ok or not found:
            out[step] = []
            continue

        # Cluster by centre. Single linkage at `tol`, which is generous
        # because an excess region's edges move with the fit that produced it;
        # what has to be stable is WHERE it is, not exactly how wide.
        found.sort(key=lambda g: g["centre"])
        groups, cur = [], [found[0]]
        for g in found[1:]:
            if g["centre"] - cur[-1]["centre"] <= tol:
                cur.append(g)
            else:
                groups.append(cur); cur = [g]
        groups.append(cur)

        regions = []
        for grp in groups:
            # HALF-CYCLES, NOT CANDIDATES. One half-cycle can now offer both a
            # residual span and an at-bound component in the same place, and
            # counting candidates made occupancy exceed 1.0 — which is not a
            # fraction of anything. See BAND_REGION_FROM_AT_BOUND.
            _seen = len({g.get("key") for g in grp}) or len(grp)
            occ = _seen / n_ok
            if occ < min_occ:
                continue
            regions.append(dict(
                centre=float(np.median([g["centre"] for g in grp])),
                lo=float(np.median([g["lo"] for g in grp])),
                hi=float(np.median([g["hi"] for g in grp])),
                span_mV=float(np.median([g["span_mV"] for g in grp])),
                occupancy=float(occ), n_seen=int(_seen), n_sampled=int(n_ok),
                from_bound=bool(any(g.get("from_bound") for g in grp))))
        out[step] = sorted(regions, key=lambda d: d["centre"])

    if verbose:
        for step in sorted(out):
            rs = out[step]
            if not rs:
                print(entry(f"{step} bands", "none",
                            "no region recurs in enough half-cycles"))
                continue
            for r in rs:
                print(entry(f"{step} band", f"{r['centre']:.3f} V",
                            f"{r['span_mV']:.0f} mV wide, in "
                            f"{r['n_seen']}/{r['n_sampled']} sampled "
                            f"half-cycles ({r['occupancy']:.0%})"))
    return out


def _describe_band_set(bs):
    """The dataset's band regions, laid out to be read."""
    out = []
    for step in sorted(bs or {}):
        rs = bs[step]
        if not rs:
            out.append(entry(f"{step} bands", "none",
                             "the peaks account for this curve"))
            continue
        for r in rs:
            _n = r.get("n_cells")
            out.append(entry(f"{step} band", f"{r['centre']:.3f} V",
                             f"{r['span_mV']:.0f} mV wide, seen in "
                             f"{r['occupancy']:.0%} of sampled half-cycles"
                             + (f" across {_n} cell(s)" if _n else "")))
    return out


def reconcile_band_sets(band_sets, compositions, *, tolerance_mV=None,
                        verbose=True):
    """
    One band set per MATERIAL, across its replicates. See
    `quality.reconcile_mechanisms`, which makes the same argument about the
    mechanism: a material either delivers charge across a composition window
    or it does not, and a replicate that did not resolve one has missed it
    rather than disproved it.

    It matters because the regions are found from a SAMPLE, and a sample can
    miss a region a longer look would find. Measured on the NMC111 pair: over
    every charge half-cycle both cells return 3.72-3.73 V (37%, 47%) and
    4.27-4.28 V (95%, 100%) — the same two regions — while the sampled
    pipeline pass found both on cell A and neither on cell B.

    `band_sets` maps name -> {step: [region, ...]} from `band_regions`;
    `compositions` maps the same names to a material label. Regions are
    unioned within a material and clustered by centre. Mutates and returns
    `band_sets`.
    """
    tol = float(BAND_REGION_TOLERANCE_MV if tolerance_mV is None
                else tolerance_mV) / 1000.0
    groups = {}
    for name in (band_sets or {}):
        groups.setdefault(str((compositions or {}).get(name) or name),
                          []).append(name)

    changed = []
    for comp, names in groups.items():
        if len(names) < 2:
            continue
        steps = set()
        for n in names:
            steps.update(band_sets[n] or {})
        for step in sorted(steps):
            # Only a WELL-ESTABLISHED region may be imposed on a replicate
            # that did not find it. See BAND_REGION_SHARED_OCCUPANCY.
            pooled = [r for n in names
                      for r in (band_sets[n] or {}).get(step, [])
                      if float(r.get("occupancy", 0.0))
                      >= BAND_REGION_SHARED_OCCUPANCY]
            if not pooled:
                continue
            pooled.sort(key=lambda r: r["centre"])
            grp, cur = [], [pooled[0]]
            for r in pooled[1:]:
                if r["centre"] - cur[-1]["centre"] <= tol:
                    cur.append(r)
                else:
                    grp.append(cur); cur = [r]
            grp.append(cur)
            merged = [dict(centre=float(np.median([r["centre"] for r in g])),
                           lo=float(np.median([r["lo"] for r in g])),
                           hi=float(np.median([r["hi"] for r in g])),
                           span_mV=float(np.median([r["span_mV"] for r in g])),
                           occupancy=float(max(r["occupancy"] for r in g)),
                           n_cells=len(set(
                               n for n in names
                               for r2 in (band_sets[n] or {}).get(step, [])
                               if any(abs(r2["centre"] - r["centre"]) <= tol
                                      for r in g))))
                      for g in grp]
            # A cell keeps everything it found itself, and gains only the
            # well-established regions it missed.
            for n in names:
                own = list((band_sets[n] or {}).get(step, []))
                before = len(own)
                for m in merged:
                    if not any(abs(r["centre"] - m["centre"]) <= tol
                               for r in own):
                        own.append(dict(m, from_replicate=True))
                own.sort(key=lambda d: d["centre"])
                band_sets.setdefault(n, {})[step] = own
                if before != len(own):
                    changed.append((n, step, before, len(own)))

    if verbose and changed:
        print(section("  One material, one band set"))
        for n, step, a, b in changed:
            print(entry(f"{str(n)[:24]} {step}", f"{a} -> {b}",
                        "regions, pooled across replicates"))
    return band_sets


def band_ceiling_volts(spec, voltage):
    """
    The widest band this half-cycle may contain, in volts.

    `spec.band_ceiling` is a FRACTION OF THE HALF-CYCLE'S OWN VOLTAGE SPAN,
    not a width — one number cannot mean the same thing on a 1.3 V window and
    a 2.1 V one. See `quality.MECHANISM_MODEL` for the measurement that
    settled the fraction, and `_band_ceilings` for the per-component array
    this feeds.

    Returns 0.0 where the mechanism permits no band, or where the half-cycle
    carries too little voltage to define a span — in both cases every
    component stays a bare pseudo-Voigt.
    """
    f = float(getattr(spec, "band_ceiling", 0.0) or 0.0)
    if f <= 0.0:
        return 0.0
    v = np.asarray(voltage, float)
    v = v[np.isfinite(v)]
    if v.size < 2:
        return 0.0
    return float(f * (v.max() - v.min()))


def _band_ceilings(spec):
    """Per-component band-width ceiling, as an array of len(spec.centres)."""
    n = len(spec.centres)
    bw = getattr(spec, "band_width_max", 0.0)
    if np.ndim(bw) == 0:
        return np.full(n, float(bw or 0.0))
    bw = np.asarray(bw, float)
    if bw.size == n:
        return bw
    out = np.zeros(n)
    out[:min(n, bw.size)] = bw[:min(n, bw.size)]
    return out


class FitSpec:
    """
    Everything needed to fit one half-cycle, and nothing else.

    Deliberately a value object with no reference to a dataset, a notebook
    global or a file — so it can be pickled to a worker and so a test can
    construct one by hand.
    """
    __slots__ = ("centres", "baseline_degree", "sigma_min", "sigma_max",
                 "centre_tol", "max_nfev", "shoulder_of", "fraction",
                 "sigma_ratio", "sigma0", "amp0", "band_width_max",
                 "asymmetry",
                 "asymmetry_edge_only",
                 # Staged band seeding — see BAND_FROM_RESIDUAL.
                 "band_from_residual", "band_residual_min_share",
                 "band_residual_max", "band_residual_min_span_mV",
                 "band_ceiling", "band_seeds",
                 # None means "use the module's FIT_TOLERANCE". Carried on the
                 # spec rather than read from a global because a fit may run
                 # in a worker process, where a global set in the parent does
                 # not exist. `quality.assess_closure` uses it.
                 "tolerance")

    def __init__(self, centres, baseline_degree=3, sigma_min=0.003,
                 sigma_max=0.200, centre_tol=0.070, max_nfev=20000,
                 shoulder_of=None, fraction=None, sigma_ratio=None,
                 sigma0=None, amp0=None, band_width_max=0.0,
                 asymmetry=None,
                 asymmetry_edge_only=None,
                 band_from_residual=None, band_residual_min_share=None,
                 band_residual_max=None, band_residual_min_span_mV=None,
                 band_ceiling=0.0, band_seeds=None,
                 tolerance=None):
        # baseline_degree defaults to 3, matching 1.8.7's BASELINE_DEGREE. It
        # was 1, and that is not a small difference: on P3 cell A cycle 2
        # charge a linear baseline gives R2 0.27 against 0.54 for a cubic.
        # A broad layered-oxide dQ/dV sits on a curved background, and a
        # straight line cannot follow it. Use `spec_for_profile` rather than
        # this default wherever the profile is known.
        self.centres = np.asarray(centres, float)
        # PER-COMPONENT STARTING VALUES, when the caller has measured some.
        #
        # Without these every component of every half-cycle starts from the
        # same two numbers: sigma = min(0.02, sigma_max/2) and amplitude =
        # 5% of the curve's maximum. On a broad profile that is a 20 mV
        # start for components that fit to a median of 85 mV — four times
        # too narrow, on every peak, in every cycle — and one amplitude for
        # the tallest and the smallest alike. The optimiser then spends
        # thousands of evaluations doing nothing but growing the peaks to
        # roughly the size detection had already measured them to be.
        #
        # Detection measures a height and a width for every peak it finds.
        # Passing them costs nothing and starts the fit near the answer.
        # The widest composition window a component may claim. Zero keeps
        # the bare pseudo-Voigt and the historical behaviour exactly.
        # A CEILING PER COMPONENT, NOT ONE FOR THE HALF-CYCLE.
        #
        # A single global ceiling lets EVERY component become a 0.5 V
        # flat-topped band, and with no polynomial to compete with, the
        # optimiser takes it: measured on NMC111 cell A with the ceiling
        # global, the model named ~100% of the charge but put a component on
        # the width bound in three cycles out of four and moved every centre
        # between cycles -- 3.81/3.99, then 3.78/3.93/4.24, then 3.89/4.30.
        # Naming all the charge with a different set of blobs each cycle is
        # not a decomposition.
        #
        # So a detected PEAK gets a ceiling of zero and stays a peak, and only
        # a component seeded from unattributed charge gets a band -- bounded
        # by the SPAN OF THE REGION IT WAS FOUND IN, which is a measurement,
        # rather than by a constant nobody can defend. Scalars still work and
        # apply to every component, which is what the historical behaviour
        # and every existing test expect.
        if np.ndim(band_width_max) == 0:
            self.band_width_max = float(band_width_max or 0.0)
        else:
            self.band_width_max = np.asarray(band_width_max, float)
        # None means "take the module default", so a spec built before the
        # asymmetry existed behaves as the constant says rather than as False.
        self.asymmetry = (ASYMMETRY if asymmetry is None else bool(asymmetry))
        # WHICH components may be asymmetric. See ASYMMETRY_EDGE_ONLY.
        self.asymmetry_edge_only = (ASYMMETRY_EDGE_ONLY
                                    if asymmetry_edge_only is None
                                    else bool(asymmetry_edge_only))
        self.tolerance = (None if tolerance is None else float(tolerance))
        # The widest band the MECHANISM permits, AS A FRACTION OF THE
        # HALF-CYCLE'S OWN VOLTAGE SPAN. `band_width_max` says what each
        # component currently is, in volts; this says what a component seeded
        # from unattributed charge would be allowed to become, and it is a
        # fraction because a FitSpec is built before the voltage array is in
        # hand. `band_ceiling_volts` resolves it. Zero means bands are not
        # this mechanism's model and none can be seeded.
        self.band_ceiling = float(band_ceiling or 0.0)
        # THE DATASET'S band regions, as (centre_V, span_mV) pairs, from
        # `band_regions`. When present these replace the per-half-cycle
        # discovery entirely and are fitted in EVERY half-cycle of the step,
        # whether or not this particular curve would have asked for one — a
        # band that comes and goes is not a measurement. Empty or None keeps
        # the per-half-cycle behaviour. See BAND_REGION_SAMPLE.
        self.band_seeds = (tuple((float(a), float(b)) for a, b in band_seeds)
                           if band_seeds else ())
        self.band_from_residual = (BAND_FROM_RESIDUAL
                                   if band_from_residual is None
                                   else bool(band_from_residual))
        self.band_residual_min_share = float(
            BAND_RESIDUAL_MIN_SHARE if band_residual_min_share is None
            else band_residual_min_share)
        self.band_residual_max = int(
            BAND_RESIDUAL_MAX if band_residual_max is None
            else band_residual_max)
        self.band_residual_min_span_mV = float(
            BAND_RESIDUAL_MIN_SPAN_MV if band_residual_min_span_mV is None
            else band_residual_min_span_mV)
        self.sigma0 = (None if sigma0 is None
                       else np.asarray(sigma0, float))
        self.amp0 = (None if amp0 is None else np.asarray(amp0, float))
        # Which components are shoulders, and of what. `shoulder_of[i]` is
        # the index of component i's parent, or None if i is a primary peak.
        # See COUPLE_SHOULDER_SIGMA.
        self.shoulder_of = (tuple(shoulder_of) if shoulder_of is not None
                            else (None,) * len(self.centres))
        # The two SHAPE parameters, calibrated once per dataset and then
        # held. None means "refine it here", which is what `calibrate_shape`
        # does on its sample and what a bare FitSpec still does. See
        # REFINE_FRACTION_PER_HALF_CYCLE.
        self.fraction = None if fraction is None else float(fraction)
        self.sigma_ratio = None if sigma_ratio is None else float(sigma_ratio)
        self.baseline_degree = baseline_degree
        self.sigma_min = float(sigma_min)
        self.sigma_max = float(sigma_max)
        self.centre_tol = float(centre_tol)
        self.max_nfev = int(max_nfev)

    def replace(self, **kw):
        d = {k: getattr(self, k) for k in self.__slots__}
        d.update(kw)
        return FitSpec(**d)

    def __repr__(self):
        n_sh = sum(1 for x in self.shoulder_of if x is not None)
        return (f"FitSpec(n={len(self.centres)}, deg={self.baseline_degree}, "
                f"sigma={self.sigma_min}-{self.sigma_max}"
                + (f", {n_sh} shoulder(s) coupled" if n_sh else "")
                + (f", eta={self.fraction:.2f}"
                   if self.fraction is not None else ", eta free")
                + (f", k={self.sigma_ratio:.2f}"
                   if self.sigma_ratio is not None else "") + ")")


# 1.8.7 Module 4 chose these from the dataset's profile class, and the choice
# matters more than any other fitting parameter. A sharp two-phase profile
# (LTO, LFP) has a narrow peak on a nearly flat background: a wide sigma bound
# lets a component inflate until it IS the background, so the bound is tight
# and the baseline is linear. A broad layered-oxide profile has overlapping
# features on a curved background, and needs the opposite.
# --- constraints that make the decomposition identifiable -----------------
#
# A pseudo-Voigt has four free parameters and they are strongly correlated:
# amplitude and sigma trade off almost exactly at fixed area, and the mixing
# fraction trades against sigma in the wings. Add a shoulder 20-40 mV from a
# parent whose FWHM is 60 mV and eight correlated parameters are being asked
# of a region that can support perhaps three. The optimiser always finds a
# lower residual — that is what free parameters do — and the standard errors
# it returns are CONDITIONAL ON THE MODEL BEING RIGHT, so they do not widen
# to warn you. R2 cannot see this; area retention can, and did: LTO cell A
# reported 136% area retention on a cell holding 99% of its capacity.
#
# The remedy is the crystallographic one. Nobody refines an independent
# profile width for every reflection in a Rietveld fit; widths are tied to a
# few physically motivated parameters and released only when the data demand
# it. Constrain, don't fix, and don't free.
#
# 1. A SHOULDER'S WIDTH IS TIED TO ITS PARENT'S by a bounded ratio, with one
#    ratio shared by every shoulder in the half-cycle rather than one each.
#    A shoulder is a distinct redox process on the same electrode: it has no
#    reason to be wider than its parent and every reason to be comparable.
#    1.8.7 tied them rigidly; a bounded ratio is the weaker, better
#    constraint.
COUPLE_SHOULDER_SIGMA = True
SHOULDER_SIGMA_RATIO_MIN = 0.30
SHOULDER_SIGMA_RATIO_MAX = 1.00
SHOULDER_SIGMA_RATIO_INIT = 0.70
#
# 2. ONE MIXING FRACTION PER DATASET, not one per component and not one per
#    half-cycle. It is the least identifiable parameter in the set and the
#    one that buys least: the Gaussian/Lorentzian balance is a property of
#    the MEASUREMENT — the cycler's voltage resolution, the rebin width, the
#    smoothing — not of the redox process, and none of those change from
#    cycle to cycle.
#
#    This is not a stylistic preference. On LTO cell A, one peak per
#    half-cycle, with eta refined per cycle: sigma rose 6.9 -> 8.7 mV and
#    eta wandered 0.65 -> 0.36 -> 0.52 across ten cycles, and the fitted
#    analytic area rose from 61 to 85 — 37% growth — on a cell that had lost
#    1.7% of its capacity. R2 was 0.995-0.998 throughout. There is no
#    shoulder in that fit to blame; the free shape parameter alone did it.
#    Every cycle's fit was excellent and the trend they formed was fiction.
#
#    So eta is MEASURED ONCE on a sample of the dataset's own half-cycles
#    (`calibrate_shape`) and then held fixed for the run. The same discipline
#    applies to the shoulder ratio k. Set these True to refine per half-cycle
#    and reproduce the behaviour above.
SHARED_FRACTION = True
REFINE_FRACTION_PER_HALF_CYCLE = False
REFINE_SIGMA_RATIO_PER_HALF_CYCLE = False
# How many half-cycles the calibration fits. Evenly spaced through the run,
# so a shape that genuinely changes with age shows up as a wide spread rather
# than being read off the first cycle.
CALIBRATION_SAMPLE = 8
# ...but only if the sample AGREES. A 10-90 spread wider than this means the
# shape genuinely varies through the run, and pinning it to the median would
# be asserting a constant the data has just denied. Above it, `calibrate_shape`
# declines and every half-cycle refines its own — with the spread printed, so
# the decision is visible rather than silent.
FRACTION_SPREAD_MAX = 0.25
SIGMA_RATIO_SPREAD_MAX = 0.30
# ...and above THIS, the shape is not merely variable but undetermined. eta is
# bounded [0, 1], so a 10-90 spread approaching 1 means the sampled fits
# landed everywhere the parameter is allowed to go — the curve does not
# constrain it. Reported as a different verdict, because "varies" and "was
# never measured" are different things to tell a reader about an area.
FRACTION_UNDETERMINED = 0.90

# ONE ASYMMETRY FOR THE HALF-CYCLE, NOT ONE PER COMPONENT.
#
# `asym_rect_pseudo_voigt` gives each component an independent width either
# side of its centre, which on its own doubles the shape parameters of every
# component — and NNM fits five or six of them per half-cycle. That is the
# objection to a split lineshape and it is a good one.
#
# It is also unnecessary. The asymmetry measured on LTO is a property of the
# HALF-CYCLE, not of the individual peak: the sign is the same on 82% of
# charge and 100% of discharge half-cycles, broad leading flank and sharp
# trailing flank in both directions. So the ratio sigma_r / sigma is ONE
# shared parameter, tied by `expr` exactly as the mixing fraction `eta` and
# the shoulder ratio `sh_k` are. A half-cycle with six components pays one
# parameter for the asymmetry, not six.
#
# Bounds: the measured HWHM ratios are 0.66 (charge) and 2.49 (discharge), so
# [0.25, 4.0] contains them with room and still forbids a component that is
# a peak on one side and a baseline on the other. Init 1.0 — the symmetric
# shape, which is interior to the box, not on it. See _SEED_BOUND_MARGIN.
# Bounds measured, not guessed. The raw half-widths of the LTO curve either
# side of its own maximum give a high/low ratio of 2.0 on charge and 0.5 on
# discharge; the FITTED ratio, which sees the whole flank rather than one
# slice at half maximum, wants 7-10 and 0.10-0.15. Swept: at [0.25, 4] the
# ratio is pinned on 18 of 20 LTO half-cycles, at [0.10, 10] on 3 of 20, and
# at [0.02, 50] on none, with R2 and the fitted ratio identical between the
# last two. So the box below contains the answer with room, and a ratio that
# reaches it is a fault to report rather than a bound doing the fitting.
ASYMMETRY = False
# How close to a bound counts as on it, as a fraction of the bound.
_BOUND_PROXIMITY = 0.01
ASYM_RATIO_MIN = 0.05
ASYM_RATIO_MAX = 20.0
ASYM_RATIO_INIT = 1.00
# Per-component widths instead of one shared ratio. Costs n parameters rather
# than 1 and is not recommended; kept because the shared ratio is an assertion
# about the half-cycle that a dataset could contradict.
# ONE RATIO PER COMPONENT, NOT ONE PER HALF-CYCLE. Flipped in 1.9.0.58.
#
# The argument above for sharing is sound and its evidence does not reach.
# "The asymmetry is a property of the HALF-CYCLE, not of the individual peak"
# was concluded on LTO — which fits exactly ONE component per half-cycle. A
# dataset with one peak cannot distinguish a property of the half-cycle from
# a property of the peak, so that measurement was silent on the question it
# was taken to settle. It is the same fault as the shoulder gate's threshold
# in 1.9.0.57: the sample was chosen where the distinction cannot appear.
#
# On a chemistry that fits five to seven components the two hypotheses come
# apart immediately, because the components are not alike. NNM discharge
# carries a near-symmetric redox peak at 3.54 V and, 1.4 V below it, a
# terminal feature whose flanks differ by a factor of five. One shared ratio
# has to compromise between them, and compromising is what it did — which is
# why asymmetry measured as "a coin toss" on the broad chemistries and was
# left off.
#
# Measured on 278 half-cycles (NNM x3, NMC x2, LTO x3), same data, same
# detection, four models, scored on BIC so the extra parameters are paid for:
#
#     model                              LTO        NMC        NNM
#     A  current, symmetric            2774.599    718.98    1126.26
#     B  symmetric, sigma_max 0.5      2774.600    863.16    1119.03
#     C  shared asymmetry              2774.599    570.84    1051.29
#     D  per-component asymmetry       2774.596    380.83     782.20
#
#     D beats A on   LTO 39/60 (dBIC -0.0)   NMC 62/68 (-251)   NNM 140/150 (-326)
#     D beats C on   LTO 39/60 (dBIC -0.0)   NMC 59/68 (-158)   NNM 136/150 (-213)
#
# D costs six more free parameters on NNM (nvarys 20 -> 26) and wins anyway,
# on 93% and 91% of half-cycles. LTO is unchanged to three decimal places in
# BIC, as it must be: with one component, shared and free are the same model.
#
# WHAT IT DOES TO THE TERMINAL COMPONENT, which is why this was opened at
# all. NNM components centred within 300 mV of a window edge:
#
#     sigma 195.4 -> 138.5 mV   (the ceiling is 200)
#     fraction of the modelled area falling OUTSIDE the window  0.264 -> 0.129
#
# The 400 mV width was never a statement about the feature. NNM's terminal
# peak is resolved — it has a visible maximum 25-40 mV inside the window and
# both flanks present — but its low flank is ~25 mV and its high flank ~120.
# Fitted symmetrically, the only way to cover the high flank is a sigma large
# enough to spill a third of the component out through the low edge. The
# ceiling was holding that spill down, not describing a peak.
#
# WHICH IS ALSO WHY RAISING THE CEILING IS THE WRONG MOVE, and model B is in
# the table to record it: at sigma_max 0.5 the component widens further, the
# outside-window share rises (NMC 0.015 -> 0.087), and BIC gets WORSE on NMC
# (+114, better on only 19 of 68 half-cycles). The obvious fix was measured
# and is a regression.
ASYMMETRY_SHARED = True
# ASYMMETRY FOR THE COMPONENTS AT THE ENDS OF THE WINDOW, AND ONLY THOSE.
#
# WHAT THE TERMINAL COMPONENT ACTUALLY IS. It was reported as a polarisation
# tail running off the edge of the data. It is not. Read off the raw curve,
# NNM's bottom-of-discharge feature has a resolved maximum 25-40 mV INSIDE
# the window with both flanks present — |dQ/dV| climbs from 34 at the cutoff
# to 50 at 2.085 V and then falls away — and its flanks are wildly unequal:
# about 25 mV on the low side against about 120 mV on the high side. NMC's
# is the opposite case: a monotone ramp into the cutoff with no maximum at
# all, and no detected primary within 300 mV of either edge in 54 half-cycles.
#
# Fitted SYMMETRICALLY, the only way to cover a 120 mV high flank is a sigma
# large enough to spill a third of the component out through the low edge.
# That is where the 400 mV FWHM and the 26-40% extrapolated area came from:
# not from a wide feature, but from a narrow asymmetric one being described
# by a shape that cannot be narrow on one side.
#
# WHY RAISING THE CEILING IS THE WRONG MOVE, measured so it stays refuted: at
# sigma_max 0.5 the component widens further, the outside-window share rises
# (NMC 0.015 -> 0.087), and BIC gets WORSE on NMC (+114, better on only 19 of
# 68 half-cycles). The obvious fix is a regression.
#
# WHY NOT ASYMMETRY EVERYWHERE. Per-component asymmetry on every component
# wins decisively on a SINGLE fit — 278 half-cycles, scored on BIC:
#
#     model                            LTO        NMC        NNM
#     current, symmetric             2774.599    718.98    1126.26
#     symmetric, sigma_max 0.5       2774.600    863.16    1119.03
#     shared asymmetry               2774.599    570.84    1051.29
#     per-component asymmetry        2774.596    380.83     782.20
#
# and is a serious regression in the PIPELINE, which is not the same test:
#
#     NNM cell A   R2 0.9886 -> 0.9011   coherence pairs 504 -> 326
#     NNM cell C   R2 0.9876 -> 0.9724   the 3.15 V shoulder pair resolved in
#                                        119/122 discharge cycles -> 67/122
#     NNM cell B   extrapolated area share 24.4% -> 85.4%
#     NMC          R2 0.9908 -> 0.9991   coherence pairs 53 -> 37, 58 -> 32
#
# The single-fit test is fair to the FITTER and blind to the LOOP around it.
# In the pipeline an asymmetric flank competes with the band machinery for
# the same residual: a flank that absorbs the continuum is a band that never
# gets seeded, and on NMC — which has a real solid-solution continuum — half
# the components ended with a flank at the width ceiling, carrying 51-58% of
# the area, spread across 3.2-4.4 V rather than at the edges. R2 rose and the
# decomposition got worse. That is the cubic baseline's failure exactly, and
# BIC did not catch it because BIC charges per parameter and asymmetry buys a
# very flexible shape for one.
#
# So the freedom goes ONLY where the diagnosis says it belongs: a component
# seeded within `sigma_max` of a window edge — one whose model necessarily
# extends past the measured range — fits its two flanks separately. Every
# other component stays exactly symmetric, tied by `expr`, costing nothing
# and unable to impersonate a continuum. The threshold is not a tuned number:
# it is the width the model is already allowed.
# AND EDGE-ONLY ASYMMETRY WAS MEASURED THROUGH THE PIPELINE TOO, AND LOST.
# It does what the diagnosis says it should — where a terminal component was
# pinned at the ceiling it comes off it, and the extrapolated area falls:
#
#     cell   R2                 terminal FWHM      extrapolated area
#     A      0.9886 -> 0.9787   200 -> 249 mV      6.7% -> 5.8%
#     B      0.9883 -> 0.9904   399 -> 293 mV     24.4% -> 24.3%
#     C      0.9876 -> 0.9896   400 -> 269 mV     11.4% ->  9.2%
#
# and the coherence pairs fall in all three cells (504 -> 477, 311 -> 306,
# 568 -> 553), the component count falls on cell A (6.38 -> 5.88), and cell A
# — the cell whose terminal components were ALREADY mostly inside the window,
# 28% at the ceiling against 69% and 78% — gets worse on every measure. A
# change that helps the cells that were wrong and harms the one that was
# right is not a model improvement; it is a change of where the error sits.
# Two of three cells better is not a result. NMC is a wash on the same test
# (cell A 0.9908 -> 0.9937 and pairs 53 -> 55; cell B unchanged and pairs
# 58 -> 56), which is what the diagnosis predicts: NMC's terminal region has
# no resolved peak for an asymmetric lineshape to describe.
#
# DEFAULT OFF, TURNED ON BY `broad` ALONE. This started as True — the policy —
# and LTO's R2 fell from 0.9966 to 0.9562 on all three cells in the first
# pipeline run, because `sharp` does not pass the field and so inherited it
# from here. LTO's single peak sits 320 mV from the nearest edge against a
# 30 mV sigma_max, so "edge only" made it symmetric and threw away the
# asymmetry that class was measured to need. A default that silently reaches a
# profile which never asked for it is the same fault as a global gate; the
# profile chooses the model, so the profile says so explicitly.
ASYMMETRY_EDGE_ONLY = False

# How far inside its own bounds a seeded parameter is held. See the note at
# the seeding site: a start exactly on a bound can stall the optimiser.
_SEED_BOUND_MARGIN = 0.25

# --- the bound stall (item 39) -------------------------------------------
# `_SEED_BOUND_MARGIN` above solves this problem for the START of a fit. These
# solve it for the END: a parameter that WALKS onto a bound during refinement
# has no outward gradient, so the minimiser declares convergence at a wall.
# See the long note at the restart site in `fit_one` for the measurements.
#
#   PROXIMITY  how close to a bound counts as on it, as a fraction of the
#              parameter's own range. 1% of the box, so it scales with the
#              parameter rather than assuming millivolts.
#   STEP       how far inside to restart. The same 25% as the seed margin —
#              far enough that the first step is not back onto the wall,
#              near enough that a genuinely binding bound is found again.
#   GAIN       how much better the restart must be before its answer is
#              taken, as a fraction of chi-square. A restart that lands on
#              the same minimum must not churn the result, so the bar is a
#              real improvement rather than a rounding one.
BOUND_ESCAPE_RESTART = True
BOUND_ESCAPE_PROXIMITY = 0.01
BOUND_ESCAPE_STEP = 0.25
BOUND_ESCAPE_GAIN = 1e-4

# HOW HARD TO ASK. scipy's `leastsq` defaults to ftol = xtol = 1.5e-8: stop
# when a step changes the sum of squares, or the solution vector, by less than
# one part in 70 million. That is a demand for eight significant figures from a
# dQ/dV curve that carries three, and it is the whole of items 9 and 13 on the
# master list — the optimiser was not failing, it was being asked a question
# the data cannot answer, and it kept stepping until it hit the 20,000
# evaluation ceiling and was then recorded as "not converged".
#
# Measured on 16 half-cycles of NNM cell C and 16 of NMC111 cell A, one-shot:
#
#   tolerance    NNM total nfev / capped / median R2    NMC total / capped / R2
#   1.5e-8            20,000 / 11 / 0.9802                  20,000 / 11 / 0.9764
#   1e-5               1,498 /  3 / 0.9802                  20,000 /  9 / 0.9764
#   1e-4                 274 /  0 / 0.9786                   2,519 /  5 / 0.9760
#
# 1e-4 is the choice: the WORST fit is unchanged on both chemistries
# (0.9682 and 0.9340), nothing runs to the ceiling on NNM, and it costs 0.0016
# of median R2 on NNM — an order of magnitude inside the closure interval that
# same dataset reports. LTO is indifferent: identical R2 to four decimal places
# at every tolerance tested, because a 6-parameter fit on a sharp peak
# converges long before the tolerance is the binding constraint.
FIT_TOLERANCE = 1.0e-4

# RETIRED, and the reason is worth keeping.
#
# The idea was Nik's and it is a good one: in a Rietveld refinement you get the
# positions right with the shape held, then release the shape. Implemented
# here it made things worse, and the measurements say why. Stage 1 was given
# the FULL evaluation budget at the 1.5e-8 tolerance above, so it did not
# "hold the shape for a moment" — it solved the WRONG SHAPE EXACTLY, driving
# the fit into a deep minimum of a model with the seeded widths frozen in.
# Releasing the widths there finds a gradient near zero, and stage 2 stops in
# 24-33 evaluations. The signature is a see-saw visible on both chemistries:
# wherever stage 1 ran long, stage 2 stopped at once.
#
# Tested against one-shot at four stage-1 budgets and two tolerances. Staging
# lost every time:
#
#              NNM median R2 / worst      NMC median R2 / worst
#   one-shot      0.9786 / 0.9682            0.9760 / 0.9340
#   stage1 100    0.9777 / 0.9681            0.9504 / 0.8780
#   stage1 300    0.9778 / 0.9681            0.9464 / 0.8783
#   stage1 1000   0.9759 / 0.9681            0.9506 / 0.8827
#
# On NMC it costs 0.025 of R2 — and it is FASTER while doing it, because
# stalling early is cheap. That is the shape of the trap.
#
# It is a flag, not a deletion: the code path is intact and the manifest
# records the choice, so re-testing it after the seeding or the bounds change
# again costs one edit.
#
# What the staging work DID produce, and what is kept: detection-seeded
# starting values and measured shoulder widths. Those were bundled with it in
# 1.9.0.40's "convergence 13% -> 59%" and they are the half of that result
# which survives.
STAGED_REFINEMENT = False


# Below this, `width` is not a band: the component is reported as a peak and
# the analytic w = 0 branch is used. 1 mV is below the cyclers' own voltage
# resolution, so a narrower band is not a claim the data can support.
_BAND_WIDTH_NEGLIGIBLE = 0.001

# How close to its ceiling a band width counts as pinned. Mirrors
# `analyse.SIGMA_BOUND_PROXIMITY`, and means the same thing: a parameter that
# has run to its limit is not a measurement of anything.
SIGMA_BOUND_PROXIMITY_COMPLEMENT = 0.95


# ---------------------------------------------------------------------------
# One component that can be a PEAK or a BAND, and lets the data decide
# ---------------------------------------------------------------------------
# A solid solution does not deliver its charge at one potential. It delivers it
# across a WINDOW of potentials, as the composition moves — which is why a
# layered-oxide dQ/dV is a broad flat-topped envelope rather than a peak, and
# why fitting it with a sum of pseudo-Voigts goes wrong. Measured on NMC111
# cell A cycle 6 discharge, seven pseudo-Voigts reach R2 0.999 with component
# areas summing to 3.65x the curve's own area: not a decomposition, a massive
# cancellation against a baseline driven negative to accommodate it.
#
# It is the same shape, and the same reason, as a carboxylic acid O-H band in
# the infrared: hydrogen bonding gives a distribution of environments, so the
# band is broad and flat-topped instead of sharp.
#
# The right basis function is therefore a rectangle of width `w` convolved
# with the line shape. Both halves of a pseudo-Voigt convolve analytically:
#
#     rect(w) (x) Gaussian(s)    = [erf(a) - erf(b)] / 2w
#     rect(w) (x) Lorentzian(s)  = [atan(a') - atan(b')] / pi.w
#
# so the mixture is closed-form and costs nothing. THE POINT IS THAT IT NESTS:
# at w = 0 this is EXACTLY the pseudo-Voigt the pipeline already fits. There
# is no model selection, no information criterion and no second fit. One extra
# parameter per component, and `w` answers "is this a peak or a band?" by
# refining to zero or not.
#
# Verified numerically before it was written in: unit area across w, sigma and
# eta (the only departures are Lorentzian tails genuinely leaving the window —
# at sigma = 100 mV over a 2 V window the analytic fraction inside is
# (2/pi).atan(10) = 0.9365, which is what came back to four figures); w -> 0
# reproduces the bare pseudo-Voigt to 3e-7 relative; w = 600 mV with
# sigma = 20 mV gives a 534 mV plateau; and the shape is smooth through the
# w = 0 branch, so the optimiser sees no step.
def rect_pseudo_voigt(x, amplitude=1.0, center=0.0, width=0.0,
                      sigma=0.01, fraction=0.5):
    """Unit-area pseudo-Voigt smeared over a rectangle of width `width`."""
    sigma = max(float(sigma), 1e-9)
    w = float(width)
    xc = np.asarray(x, float) - float(center)
    # THE GAUSSIAN'S OWN WIDTH. A pseudo-Voigt is a Lorentzian and a Gaussian
    # of the SAME full width at half maximum, which for the Gaussian means
    # sigma_g = sigma / sqrt(2 ln 2) — that is what lmfit's PseudoVoigtModel
    # uses and what makes `fwhm = 2 * sigma` true.
    #
    # This function used `sigma` directly in the Gaussian until 1.9.0.51, so
    # its w -> 0 limit was NOT the pseudo-Voigt: 15% different in shape at
    # eta = 0 and 17.7% wider at half maximum, while `_derived` went on
    # reporting `fwhm = 2 * sigma`. Every width and height published from a
    # band-model fit — that is, every NNM and NMC component — was computed
    # for a curve of a different width from the one that was drawn, and
    # `evaluate` compounded it by rebuilding narrow components through
    # lmfit's model and wide ones through this one. Both are now the same
    # shape, verified against PseudoVoigtModel to machine precision at w = 0.
    sg = sigma / _SIGMA_G_FACTOR
    if w <= _BAND_WIDTH_NEGLIGIBLE:
        g = np.exp(-xc ** 2 / (2 * sg ** 2)) / (sg * np.sqrt(2 * np.pi))
        l = sigma / np.pi / (xc ** 2 + sigma ** 2)
    else:
        a = (xc + w / 2.0) / (sg * np.sqrt(2))
        b = (xc - w / 2.0) / (sg * np.sqrt(2))
        g = (erf(a) - erf(b)) / (2.0 * w)
        l = (np.arctan((xc + w / 2.0) / sigma)
             - np.arctan((xc - w / 2.0) / sigma)) / (np.pi * w)
    return float(amplitude) * ((1.0 - float(fraction)) * g
                               + float(fraction) * l)


def _rpv_area1(xc, width, sigma, fraction):
    """`rect_pseudo_voigt` at unit amplitude, centred — the shape alone."""
    sigma = max(float(sigma), 1e-9)
    w = float(width)
    xc = np.asarray(xc, float)
    sg = sigma / _SIGMA_G_FACTOR
    if w <= _BAND_WIDTH_NEGLIGIBLE:
        g = np.exp(-xc ** 2 / (2 * sg ** 2)) / (sg * np.sqrt(2 * np.pi))
        l = sigma / np.pi / (xc ** 2 + sigma ** 2)
    else:
        a = (xc + w / 2.0) / (sg * np.sqrt(2))
        b = (xc - w / 2.0) / (sg * np.sqrt(2))
        g = (erf(a) - erf(b)) / (2.0 * w)
        l = (np.arctan((xc + w / 2.0) / sigma)
             - np.arctan((xc - w / 2.0) / sigma)) / (np.pi * w)
    return (1.0 - float(fraction)) * g + float(fraction) * l


# THE PEAK IS ASYMMETRIC AND UNTIL 1.9.0.51 THE MODEL COULD NOT BE.
#
# Measured on LTO without fitting anything — the half-widths of the raw curve
# either side of its own maximum: charge high/low HWHM ratio 0.66, discharge
# 2.49, the sign consistent on 82% and 100% of half-cycles. In time order that
# is a broad leading flank and a sharp trailing flank in both directions, which
# is what a distribution of particle sizes or a kinetic tail looks like and is
# not something a symmetric lineshape can absorb. It is not the baseline: the
# ratios are identical at baseline degree 1, 2 and 3, the sign is preserved on
# 42 of 42 half-cycles, and the flexible baselines fit WORSE on BIC.
#
# This is the generalisation of `rect_pseudo_voigt`, not a second model: each
# side of the centre gets its own width, the two halves are scaled to the same
# peak height so the join is continuous, and the whole is renormalised so
# `amplitude` is still the area. With `sigma_r == sigma` it reduces to
# `rect_pseudo_voigt` exactly (verified to machine precision across w, sigma
# and eta), so a symmetric fit is unchanged and the band model still composes
# with it.
#
# One consequence worth stating: `sigma` is now the LEFT half-width at half
# maximum and `sigma_r` the right, so the full width is `sigma + sigma_r` and
# not `2 * sigma`. Everything downstream reads `fwhm`, which is computed from
# both.
def asym_rect_pseudo_voigt(x, amplitude=1.0, center=0.0, width=0.0,
                           sigma=0.01, sigma_r=0.01, fraction=0.5):
    """Unit-area rect-smeared pseudo-Voigt with an independent width each side."""
    sl = max(float(sigma), 1e-9)
    sr = max(float(sigma_r), 1e-9)
    if abs(sr - sl) <= _TINY:
        return rect_pseudo_voigt(x, amplitude=amplitude, center=center,
                                 width=width, sigma=sl, fraction=fraction)
    xc = np.asarray(x, float) - float(center)
    hl = float(_rpv_area1(0.0, width, sl, fraction))
    hr = float(_rpv_area1(0.0, width, sr, fraction))
    if not (hl > 0 and hr > 0):
        return np.zeros_like(xc)
    shape = np.where(xc < 0.0,
                     _rpv_area1(xc, width, sl, fraction) / hl,
                     _rpv_area1(xc, width, sr, fraction) / hr)
    # Each side contributes half of its own unit-area shape's area, which at
    # unit HEIGHT is 1/h. The composite therefore has area (1/hl + 1/hr)/2.
    return float(amplitude) * shape / (0.5 * (1.0 / hl + 1.0 / hr))


PROFILE_SPECS = {
# The values are 1.8.7 Module 4's: FIT_CENTRE_TOLERANCE_MV = 70,
# FIT_SIGMA_MIN_MV = 5, FIT_SIGMA_MAX_MV = 200, BASELINE_DEGREE = 3, with the
# sharp-profile overrides that cell applies. Read off the source, not recalled.
    #                centre_tol  sigma_min  sigma_max  baseline_degree
    # band_width_max = 0 means "no band, use the bare pseudo-Voigt", and the
    # fit is then bit-identical to what it has always been. A two-phase
    # reaction happens AT a potential, so a sharp profile has no composition
    # window to smear over and giving it one would only add a free parameter
    # that must refine to zero.
    # SHARP sigma_min DROPPED 3.0 -> 0.5 mV IN 1.9.0.51.
    #
    # FWHM = 2*sigma for this model, so 3 mV forced every fitted peak to be at
    # least 6 mV wide — against LTO peaks measured at 2.6-4.7 mV on cell C.
    # The model could not be as narrow as the material, so it conserved area
    # by being wide and SHORT: measured with `fit_quality`, the fitted peak
    # reached only 0.68 of the data's height on discharge, with a residual of
    # 40% of the amplitude. R2 0.78 was the visible symptom and it read as
    # "a bit noisy"; the fit was missing the peak height by a third.
    #
    # 0.5 mV is not a guess. A peak cannot be narrower than the histogram that
    # sampled it, and the sharp profile bins at 1 mV
    # (`signal.HISTOGRAM_BIN_BY_PROFILE`), so sigma >= bin/2 means FWHM >= one
    # bin — the model may be exactly as narrow as the measurement resolves and
    # no narrower. Swept beforehand: at 0.5 mV median R2 went 0.9664 -> 0.9791
    # and NOTHING ran to the new floor, so there is no collapse risk; the
    # floor stops being a constraint rather than becoming a looser one.
    #
    # Moderate and broad are unchanged: no moderate dataset exists to measure
    # on, and on broad the 5 mV floor is what stands between the fit and
    # needle components.
    # ASYMMETRY IS ON FOR SHARP AND OFF FOR THE OTHER TWO, because that is
    # what three chemistries say. Same seeds, same bounds, one extra shared
    # parameter, judged on BIC so the parameter has to pay for itself:
    #
    #                    R2 sym -> asym   max residual    BIC better
    #   LTO   (sharp)    0.826 -> 0.958     27% -> 18%      20 of 20
    #                    0.909 -> 0.994     24% ->  8%
    #   NNM   (broad)    0.984 -> 0.984     12% ->  8%      14 of 20
    #   NMC   (broad)    0.994 -> 0.998     16% ->  7%       8 of 18
    #                    0.986 -> 0.971     11% -> 13%
    #
    # On LTO it is not close: every half-cycle, both cells, both directions,
    # median dBIC -1424 (charge) and -2546 (discharge). It also beats the
    # honest alternative — one more symmetric COMPONENT, which costs three
    # parameters instead of one — on 51 of 64 half-cycles, and on charge the
    # extra component is worse than not adding it at all (dBIC +19.5). So the
    # asymmetry is a lineshape, not an unresolved peak.
    #
    # On the two broad chemistries it is a coin toss that costs 1.2-4.8x the
    # fitting time, and on NMC discharge it is worse. A parameter that does
    # not pay for itself does not go in. This is the same rule the rest of
    # the module follows: the profile chooses the model.
    # BASELINE REMOVED IN 1.9.0.56 — `baseline_degree=None` on every class.
    # The degrees below (1, 2, 2) were each measured and each right for the
    # question being asked at the time, which was "which polynomial fits the
    # background best". That question had no answer because there is no
    # background: see BAND_FROM_RESIDUAL for the measurement, and for what the
    # polynomial was doing instead. `quality.DEFAULT_DEGREES` no longer has a
    # sweep to make; `unattributed_fraction` is the quantity that replaces it.
    "sharp":    dict(centre_tol=0.030, sigma_min=0.0005, sigma_max=0.030,
                     asymmetry=True,
                     baseline_degree=None, band_width_max=0.0),
    # MODERATE DROPPED 3 -> 2 IN 1.9.0.36. See the note below the table.
    # Moderate: OFF until measured, on the same principle as the baseline
    # degree above it. Whoever brings a moderate dataset should try it.
    "moderate": dict(centre_tol=0.070, sigma_min=0.005, sigma_max=0.100,
                     asymmetry=False,
                     baseline_degree=None, band_width_max=0.0),
    # BROAD DROPPED 3 -> 2 IN 1.9.0.28. The cubic was 1.8.7's and it is the
    # single cause of the fitting not converging.
    #
    # Measured on NNM cell C, 30 cycles, same peaks, same starts, only the
    # baseline degree changed:
    #
    #     degree   converged   median nfev   median R2
    #       1         80%          368         0.9677
    #       2         72%          412         0.9678
    #       3 (old)   40%       20,000         0.9716
    #
    # The cubic buys 0.004 of R2 and costs 54x the evaluations and half the
    # convergence. That is §A6's argument arriving as a measurement: adding
    # free parameters always lowers the residual, and R2 rose while the answer
    # got worse — non-converged fits, an undetermined peak/background split,
    # and 126 cycles with their capacity attribution withheld.
    #
    # WHY A CUBIC SPECIFICALLY. The original reasoning was that a broad peak
    # sits on a curved background a straight line cannot follow, and that is
    # still right — which is why this is 2 and not 1. But a cubic can carry an
    # INFLECTION, and an inflection is exactly what lets a baseline impersonate
    # a peak. That is the degenerate direction the optimiser then crawls along
    # for 20,000 evaluations without finding a minimum, because there is no
    # unique one to find. A quadratic curves without being able to do that.
    #
    # Convergence by peak count makes the same point: at 5 peaks a cubic
    # converged on 8% of half-cycles against 50% for a linear baseline. The
    # more components there are to trade with, the worse the cubic is.
    #
    # `quality.DEFAULT_DEGREES` still probes (1, 2, 3), so the closure interval
    # measures the same sensitivity as before and this change does not hide it.
    # NO CLASS TURNS THE BAND ON. The band belongs to a MECHANISM — a solid
    # solution delivering charge across a composition window — and the shape
    # class cannot see mechanisms: NNM and NMC111 are both "broad", and one
    # of them is a series of real transitions. Giving every broad dataset
    # bands cost NNM its determined split (interval 0.052 -> 0.140 and
    # 0.101 -> 0.179, reliable components more than halved).
    #
    # `quality.classify_mechanism` decides, before the fitting, and the
    # caller passes `band_width_max` and `primaries_only` from its answer.
    # 0.5 V is the ceiling it passes when it does turn them on: a generous
    # composition window, still well inside the analysis range, bounding the
    # parameter without deciding the answer.
    # ASYMMETRY ON IN 1.9.0.58. It was off on a measurement taken while the
    # cubic baseline still existed and while one ratio was shared across
    # every component; both premises are gone. See ASYMMETRY_SHARED for the
    # four-model comparison that reopened it. `moderate` stays off: there is
    # still no moderate dataset, and this is not evidence about one.
    # ASYMMETRY STAYS OFF FOR `broad`, MEASURED AGAIN IN 1.9.0.58 AND STILL OFF.
    # `asymmetry_edge_only` is set here so that turning `asymmetry` on is a
    # one-word change for whoever revisits this — the machinery is built and
    # tested, the verdict is what failed. Three candidate fixes for the
    # terminal component were measured through the whole pipeline and all
    # three are refuted; see ASYMMETRY_EDGE_ONLY for the numbers, including
    # this one, which was the closest and still lost a cell.
    "broad":    dict(centre_tol=0.070, sigma_min=0.005, sigma_max=0.200,
                     asymmetry=False, asymmetry_edge_only=True,
                     baseline_degree=None, band_width_max=0.0),
}
# MODERATE, MEASURED IN 1.9.0.36. There is still no real moderate dataset in
# hand, so this was measured on SYNTHETIC half-cycles from `synthetic.py`
# ("mixed", sigma 28-40 mV over a 2.2 V window, which is squarely inside the
# moderate band), 30 seeds per condition. That is weaker evidence than the
# NNM measurement above and is labelled as such — but the effect is not
# subtle, and the condition that exposes it is the realistic one.
#
# Fitted from the TRUE centres with no noise, all three degrees converge 30/30
# and this test says nothing. That is why the cubic survived so long. Give the
# fit what detection actually hands it — centres up to 20 mV off, one spurious
# extra seed, and real noise — and the degrees separate:
#
#   noise   deg   converged   median nfev   median R2
#    0.01    1       30/30         351        0.9974
#    0.01    2       30/30         387        0.9975
#    0.01    3       24/30         828        0.9974
#    0.05    1       29/30         425        0.9543
#    0.05    2       29/30         611        0.9544
#    0.05    3       13/30      20,000        0.9518
#    0.10    1       29/30         442        0.8589
#    0.10    2       27/30       1,288        0.8586
#    0.10    3       15/30      19,844        0.8589
#
# At 5% noise the cubic converges on 43% of half-cycles against 97% for the
# quadratic, sits at the 20,000-evaluation ceiling at the MEDIAN, and fits
# very slightly WORSE. It buys nothing and costs everything the broad
# measurement said it would — the same degeneracy, the same inflection, the
# same crawl along a direction with no minimum in it.
#
# So moderate inherits the answer after all. It was right not to assume it:
# the assumption would have been correct and the reason for believing it
# would not have been. Anyone who brings a real moderate dataset should still
# run the comparison on it — this is a synthetic result, and the honest thing
# is for the next measurement to be able to overturn it.


def shoulder_parents(centres, is_shoulder):
    """
    Assign each shoulder to its NEAREST PRIMARY peak in voltage.

    Returns a tuple the length of `centres`: the parent's index for a
    shoulder, None for a primary. If nothing was flagged primary the whole
    tuple is None — a half-cycle of shoulders has no parent to tie to, and
    tying them to each other would be arbitrary.
    """
    c = np.asarray(centres, float)
    sh = np.asarray(is_shoulder, bool)
    if sh.size != c.size:
        return (None,) * c.size
    primaries = np.flatnonzero(~sh)
    if primaries.size == 0:
        return (None,) * c.size
    out = []
    for i in range(c.size):
        if not sh[i]:
            out.append(None)
        else:
            out.append(int(primaries[np.argmin(np.abs(c[primaries] - c[i]))]))
    return tuple(out)


def spec_for_profile(peaks, profile, *, fraction=None, sigma_ratio=None,
                     primaries_only=False, **overrides):
    """
    A `FitSpec` matching 1.8.7's profile-dependent settings.

    `peaks` is either the detected-peaks DataFrame (columns `voltage` and,
    optionally, `is_shoulder`) or a bare sequence of centres. Passing the
    DataFrame is preferred: it is the only way `shoulder_of` gets filled in,
    and without that the sigma coupling that makes the decomposition
    identifiable cannot be applied. A bare sequence still works, and simply
    fits every component free — which is what 1.9.0.8 did.

    `profile` is `signal.classify_dqdv_profile`'s dict, or its class string.
    Anything passed as a keyword overrides the profile's value.

    Use this rather than a bare `FitSpec` wherever the profile is known. The
    first 1.9.0 run on a real cell used the bare defaults and fitted a broad
    P3 dQ/dV with a LINEAR baseline, which is what a visibly bad fit looks
    like: R2 0.27 where a cubic gives 0.54.
    """
    shoulder_of = None
    sigma0 = amp0 = None
    # THE BAND MODEL REPLACES COMPONENTS, IT DOES NOT JOIN THEM.
    #
    # A sum of Gaussians can only build a flat top by stacking overlapping
    # components, and the flanks of that stack are what the shoulder pass
    # finds. On a solid-solution envelope those shoulders are therefore
    # ARTEFACTS OF THE WRONG SHAPE, not features. Give a component a flat top
    # of its own and they should not be separate components at all.
    #
    # Measured on NMC111 cell A cycle 6 discharge — 7 detected components, 5
    # of them shoulders, so 2 primaries:
    #
    #     components   model          closure       interval   R2
    #     all 7        pseudo-Voigt   0.17-0.51       0.34     0.946
    #     all 7        bands          0.22-1.00       0.78     0.920
    #     2 primaries  pseudo-Voigt   0.22-0.50       0.28     0.952
    #     2 primaries  bands          0.34-0.47       0.12     0.978
    #
    # Bands ON TOP OF the full list are WORSE than what we had: seven extra
    # free widths and no reduction in anything. Bands INSTEAD OF the stack
    # bring the interval to 0.12 — under the 0.15 at which the split counts
    # as determined — with the best R2 of the four. That is the difference
    # between "NMC is not decomposable" and "NMC needed the right model".
    # PRIMARIES ONLY IS NOT A PROPERTY OF "BROAD". Deciding it from the
    # profile class applied it to NNM, where the shoulders are real
    # transitions, and the closure interval went 0.052 -> 0.140 and
    # 0.101 -> 0.179 while reliable components more than halved. The caller
    # passes it, from `quality.classify_mechanism`, which asks what produced
    # the curve rather than what shape it is.
    # A BAND CEILING FROM THE MECHANISM IS A PERMISSION, NOT AN INSTRUCTION.
    # `quality.classify_mechanism` says whether this curve's model may contain
    # bands. It does not say that every detected maximum is one -- a detected
    # maximum is a peak, and a band belongs to a region found from the charge
    # the peaks cannot account for. So the value is moved onto `band_ceiling`
    # and the components start as peaks. See BAND_FROM_RESIDUAL.
    # `band_ceiling` HOLDS A FRACTION OF THE HALF-CYCLE'S SPAN, resolved to
    # volts by `band_ceiling_volts` at fit time, where the voltage array is
    # in hand. A FitSpec is built before any of that is known — see
    # `quality.MECHANISM_MODEL` for why a flat volts ceiling was wrong.
    _ceiling = float(overrides.pop("band_ceiling_span_fraction", 0.0) or 0.0)
    if _ceiling > 0:
        overrides["band_ceiling"] = _ceiling
        overrides["band_width_max"] = 0.0

    if (primaries_only and hasattr(peaks, "columns")
            and "is_shoulder" in getattr(peaks, "columns", [])):
        _prim = peaks[~peaks["is_shoulder"].fillna(False)]
        if len(_prim) >= 1:
            peaks = _prim.reset_index(drop=True)
    if hasattr(peaks, "columns"):
        centres = np.asarray(peaks["voltage"].values, float)
        if "is_shoulder" in peaks.columns:
            shoulder_of = shoulder_parents(
                centres, peaks["is_shoulder"].fillna(False).values)
        # START THE FIT WHERE DETECTION ALREADY IS.
        #
        # Detection measures a height and a width for every peak it finds and
        # nothing used them. Every component of every half-cycle therefore
        # started from the same two numbers — sigma = min(0.02, sigma_max/2),
        # amplitude = 5% of the curve's maximum — so on a broad profile every
        # peak began 20 mV wide and fitted to a median of 85 mV, and the
        # tallest and the smallest began identical. The optimiser's first few
        # thousand evaluations were spent doing nothing but growing the
        # components to the size detection had already measured.
        #
        # Measured on 14 NMC111 discharge half-cycles, same peak list, only
        # the starting values changed:
        #
        #     generic start   2 of 14 converged   median 20,000 nfev   87 s
        #     seeded          11 of 14 converged  median    170 nfev   24 s
        #
        # A hundredfold fewer evaluations. FWHM = 2*sigma exactly for this
        # model, and the area of a peak is about height x FWHM; neither has
        # to be accurate, only close enough that the optimiser starts inside
        # the right basin instead of walking to it.
        #
        # WARM-STARTING FROM THE PREVIOUS CYCLE WAS TRIED AND IS WORSE: 0 of
        # 14 converged. A fitted shape can be degenerate — a component sitting
        # at the sigma bound, acting as baseline — and carrying it forward
        # propagates that into every later cycle, where re-reading detection
        # each time is self-correcting. At 170 evaluations there is nothing
        # left for a warm start to buy anyway.
        def _num(col):
            # numpy only: `fitting` does not import pandas, and the
            # module-isolation build guard rightly refused the version of
            # this that did.
            try:
                return np.asarray(peaks[col].values, dtype=float)
            except (KeyError, TypeError, ValueError):
                return None
        _w = _num("width_V") if "width_V" in peaks.columns else None
        if _w is not None:
            sigma0 = np.where(np.isfinite(_w) & (_w > 0), _w / 2.0, np.nan)
            # A SHOULDER HAS NO MEASURED WIDTH — it is not a local maximum, so
            # `peak_widths` never measured one and detection leaves it at
            # zero. On a broad layered oxide most reference peaks ARE
            # shoulders (five of seven on NMC111 cell A discharge), so
            # without this the seeding fired on almost nothing and every
            # shoulder fell back to the generic 20 mV that caused the problem.
            # The half-cycle's own measured widths are the best available
            # statement of how wide a feature on THIS curve is, so an
            # unmeasured component starts at their median.
            _ok = np.isfinite(sigma0)
            if _ok.any() and not _ok.all():
                sigma0 = np.where(_ok, sigma0, float(np.median(sigma0[_ok])))
            _h = _num("height") if "height" in peaks.columns else None
            if _h is not None:
                amp0 = np.where(np.isfinite(_h) & np.isfinite(sigma0),
                                np.abs(_h) * 2.0 * sigma0, np.nan)
    else:
        centres = np.asarray(peaks, float)

    cls = profile.get("class", "broad") if isinstance(profile, dict) else profile
    kw = dict(PROFILE_SPECS.get(str(cls), PROFILE_SPECS["broad"]))
    if shoulder_of is not None:
        kw["shoulder_of"] = shoulder_of
    if fraction is not None:
        kw["fraction"] = fraction
    if sigma_ratio is not None:
        kw["sigma_ratio"] = sigma_ratio
    if sigma0 is not None:
        kw.setdefault("sigma0", sigma0)
    if amp0 is not None:
        kw.setdefault("amp0", amp0)
    kw.update(overrides)
    return FitSpec(centres, **kw)


def _resolved_shoulders(spec: FitSpec):
    """
    `{child: root_parent}` for every coupled shoulder in `spec`.

    Chains are collapsed to a primary component (a shoulder of a shoulder is
    tied to the primary at the head of the chain) and self-references,
    out-of-range indices and cycles are dropped rather than raised: a bad
    `shoulder_of` should cost the coupling, not the fit.
    """
    n = len(spec.centres)
    raw = getattr(spec, "shoulder_of", None) or ((None,) * n)
    out = {}
    for i in range(n):
        p = raw[i] if i < len(raw) else None
        seen = {i}
        while p is not None:
            p = int(p)
            if p < 0 or p >= n or p in seen:
                p = None
                break
            seen.add(p)
            nxt = raw[p] if p < len(raw) else None
            if nxt is None:
                break
            p = nxt
        if p is not None:
            out[i] = p
    return out


def calibrate_shape(jobs, *, n=CALIBRATION_SAMPLE, n_jobs=None, verbose=True):
    """
    Measure this dataset's peak SHAPE once, so no cycle has to re-decide it.

    `jobs` is the production list of (voltage, dqdv, spec, key). An evenly
    spaced sample is refitted with the mixing fraction and the shoulder ratio
    FREE, and the median of each is returned:

        {"fraction": eta, "sigma_ratio": k, "fraction_spread": ...,
         "sigma_ratio_spread": ..., "n": ...}

    Pass those back through `spec_for_profile(..., fraction=eta,
    sigma_ratio=k)` and every production fit is one or two parameters lighter,
    with the shape held at what the data said it was rather than at whatever
    each cycle's optimiser preferred.

    The SPREAD is returned as well as the median, and it is the number to
    look at: a wide one means the shape genuinely varies through the run, and
    fixing it is then a modelling decision that should be made deliberately
    rather than a tidy-up.

    THE FIGURE THAT USED TO BE HERE WAS STALE. This said "on the LTO
    triplicate it is 0.1-0.2"; re-measured at 1.9.0.56 it is 0.63-0.72, so
    every LTO run refines eta per half-cycle and says so. Measured with the
    cubic baseline restored it is 0.68-0.79 -- WIDER, not narrower, so this is
    not something removing the baseline caused. eta on a sharp two-phase peak
    is simply not well determined by these data, and the calibration is right
    to decline to assert it.

    What that costs is visible in the capacity accounting. On LTO cell A
    charge the unattributed charge tracks eta at r = -0.935 across ten cycles:
    the higher the Lorentzian fraction, the further the wings reach and the
    more the single component over-covers, from +1.0% of the cell's charge at
    eta = 0.41 to -6.0% at eta = 1.00. Before 1.9.0.56 a polynomial absorbed
    that and nobody could see it.

    Returns None for either quantity the sample could not measure — no
    shoulders in the dataset, or nothing converged.
    """
    if not jobs:
        return dict(fraction=None, sigma_ratio=None, n=0,
                    fraction_spread=float("nan"),
                    sigma_ratio_spread=float("nan"))
    idx = np.linspace(0, len(jobs) - 1, min(int(n), len(jobs)))
    sample = [jobs[i] for i in sorted(set(idx.round().astype(int).tolist()))]
    # Free BOTH, whatever the module defaults say: this is the measurement
    # those defaults are supposed to be based on.
    free = [(v, y, spec.replace(fraction=None, sigma_ratio=None), k)
            for (v, y, spec, k) in sample]
    if verbose:
        print(entry("sample", f"{len(free)} of {len(jobs)}",
                    "half-cycles, refitted with the shape free"))
    res = fit_many(free, n_jobs=n_jobs, verbose=False, label="calibration")

    etas, ks = [], []
    for r in res:
        if not r.get("success"):
            continue
        e = r.get("shared_fraction", np.nan)
        if not np.isfinite(e):
            # One component: no shared eta was created, so read the
            # component's own.
            comps = r.get("components", [])
            e = comps[0]["fraction"] if len(comps) == 1 else np.nan
        if np.isfinite(e):
            etas.append(float(e))
        kk = r.get("sigma_ratio_k", np.nan)
        if np.isfinite(kk):
            ks.append(float(kk))

    def _med(xs):
        return (float(np.median(xs)), float(np.percentile(xs, 90)
                                            - np.percentile(xs, 10))) \
            if xs else (None, float("nan"))

    eta, eta_sp = _med(etas)
    k, k_sp = _med(ks)
    eta_ok = eta is not None and eta_sp <= FRACTION_SPREAD_MAX
    k_ok = k is not None and k_sp <= SIGMA_RATIO_SPREAD_MAX
    if verbose:
        if eta is None:
            print(verdict("caution", "mixing fraction not measurable"))
        else:
            _pin = ("  (pinned at a bound)" if eta <= 0.02 or eta >= 0.98
                    else "")
            print(entry("mixing fraction eta", f"{eta:.3f}",
                        f"10-90 spread {eta_sp:.3f} over {len(etas)} "
                        f"half-cycles{_pin}"))
            if eta_ok:
                print(verdict("ok", "held fixed for every fit in this "
                                    "dataset"))
            elif eta_sp >= FRACTION_UNDETERMINED:
                # A SPREAD OF ~1.0 IS NOT A WIDE MEASUREMENT, it is no
                # measurement: eta is bounded [0, 1], so a 10-90 range that
                # spans the whole interval says the data does not constrain
                # the peak shape at all. That is a different statement from
                # "it varies", and it used to print the same verdict as a
                # spread of 0.375.
                print(verdict("bad", f"spread {eta_sp:.2f} spans the whole "
                                     f"allowed range — the peak shape is not "
                                     f"determined by this data"))
                print(bullet("Every half-cycle refines its own eta, which is "
                             "the only honest option, but the areas that "
                             "result carry a shape parameter the curve never "
                             "pinned down. Treat area TRENDS from this "
                             "dataset as indicative; peak positions are "
                             "unaffected."))
            else:
                print(verdict("caution", f"spread {eta_sp:.2f} > "
                                         f"{FRACTION_SPREAD_MAX} — refined "
                                         f"per half-cycle"))
                print(bullet("The peak shape is not constant through this "
                             "run, so it is not asserted to be. Area trends "
                             "from this dataset carry that free parameter."))
        if k is not None:
            _pin = "  (pinned at a bound)" if k >= 0.98 else ""
            print(entry("shoulder ratio k", f"{k:.3f}",
                        f"10-90 spread {k_sp:.3f} over {len(ks)} "
                        f"half-cycles{_pin}"))
            print(verdict("ok", "held fixed") if k_ok else
                  verdict("caution", f"spread > {SIGMA_RATIO_SPREAD_MAX} — "
                                     f"refined per half-cycle"))
    return dict(fraction=(eta if eta_ok and not REFINE_FRACTION_PER_HALF_CYCLE
                          else None),
                sigma_ratio=(k if k_ok and not
                             REFINE_SIGMA_RATIO_PER_HALF_CYCLE else None),
                fraction_median=eta, sigma_ratio_median=k,
                fraction_spread=eta_sp, sigma_ratio_spread=k_sp,
                n=len(res))


def build_model(voltage, dqdv, spec: FitSpec):
    """
    Compose the pseudo-Voigt sum and its baseline. Pure; no fitting.

    Parameters are created from the COMPOSITE model, not merged from the
    components. That is not a style choice: lmfit validates the supplied
    Parameters against the composite's `param_names`, and a Parameters object
    assembled component-by-component fails that check once the derived
    `fwhm`/`height` entries are removed. Building from the composite and then
    deleting works, because the composite is the thing being validated.
    """
    v = np.asarray(voltage, float)
    y = np.abs(np.asarray(dqdv, float))

    _use_band = np.any(np.asarray(_band_ceilings(spec), float) > 0.0)
    _use_asym = bool(getattr(spec, "asymmetry", False))
    model = None
    for i in range(len(spec.centres)):
        if _use_asym:
            # Subsumes the band: `asym_rect_pseudo_voigt` carries `width` too,
            # and `width` is simply held at zero when bands are off.
            m = _LMModel(asym_rect_pseudo_voigt, prefix=f"p{i}_")
        elif _use_band:
            m = _LMModel(rect_pseudo_voigt, prefix=f"p{i}_")
        else:
            m = PseudoVoigtModel(prefix=f"p{i}_")
        model = m if model is None else model + m

    baseline = None
    if spec.baseline_degree is not None:
        baseline = PolynomialModel(degree=spec.baseline_degree, prefix="bg_")
        model = model + baseline

    params = model.make_params()

    amp0 = max(y.max() * 0.05, _TINY)
    sigma0 = min(0.02, max(spec.sigma_max / 2.0, spec.sigma_min))
    # A COMPONENT MAY MOVE, BUT NOT PAST ITS NEIGHBOUR.
    #
    # Every centre used to get the same +/- centre_tol, which on a broad
    # profile is 70 mV. A shoulder sits as little as 20 mV from its parent
    # (`SHOULDER_MIN_SEPARATION_MV`), so the two boxes overlapped almost
    # completely and the pair could simply trade places — a flat direction
    # in the residual that the optimiser walks up and down until it hits the
    # 20,000-evaluation ceiling. It is the same degeneracy the cubic
    # baseline causes, in the centres rather than the background.
    #
    # It was latent while shoulders were being placed in VALLEYS, far from
    # any parent. Rebuilding the picker in 1.9.0.37 put them where they
    # belong — on their parent's flank — and the fitting time on NMC111
    # cell A went from 251 s to 523 s, with nearly every discharge fit
    # running to the ceiling. A better peak list made a worse fit, because
    # the bound was wrong.
    #
    # So each centre is now free within HALF the distance to its nearest
    # neighbouring seed, capped by centre_tol. Two seeds 20 mV apart get
    # 10 mV each and cannot cross; an isolated peak is unaffected. This is
    # a constraint on identity, not on movement: a peak that needs to travel
    # further than half way to its neighbour has stopped being that peak.
    _c = np.asarray(spec.centres, float)
    _tol = np.full(_c.size, float(spec.centre_tol))
    if _c.size > 1:
        _order = np.argsort(_c)
        _sorted = _c[_order]
        _gaps = np.diff(_sorted)
        _half = np.empty(_c.size)
        _half[0] = _gaps[0] / 2.0
        _half[-1] = _gaps[-1] / 2.0
        if _c.size > 2:
            _half[1:-1] = np.minimum(_gaps[:-1], _gaps[1:]) / 2.0
        _tol[_order] = np.minimum(_tol[_order], _half)
    # A CENTRE CANNOT LEAVE THE DATA. The tolerance above is a constraint on
    # identity, and it is applied around the SEED — so a seed placed near an
    # edge could still be fitted to a centre outside the window, where there
    # is nothing to fit it to. `detect.infer_missing_seeds` now refuses to
    # place a seed outside the half-cycle it belongs to, which removes the
    # source; this bounds the destination as well, so no future seeder can
    # reintroduce it. A seed already outside the window is pulled to the edge
    # rather than dropped, because dropping a component silently changes the
    # model and this is a fitter, not a selector.
    _wlo, _whi = float(v.min()), float(v.max())
    for i, c in enumerate(spec.centres):
        pre = f"p{i}_"
        _c0 = float(np.clip(float(c), _wlo, _whi))
        _cmin = max(_wlo, _c0 - float(_tol[i]))
        _cmax = min(_whi, _c0 + float(_tol[i]))
        if _cmin >= _cmax:                     # degenerate window: keep a box
            _cmin, _cmax = _wlo, _whi
        params[pre + "center"].set(value=float(np.clip(_c0, _cmin, _cmax)),
                                   min=_cmin, max=_cmax)
        _s0 = sigma0
        if spec.sigma0 is not None and i < spec.sigma0.size:
            _v = float(spec.sigma0[i])
            if np.isfinite(_v) and _v > 0:
                # NOT ON THE BOUNDARY. A parameter started exactly at its own
                # limit has a one-sided derivative there, and the optimiser
                # can read that as a minimum and stop: seeding LTO's
                # discharge peak at its measured 2.6 mV width clipped to
                # sigma_min = 3 mV, the fit "converged" in 15 evaluations,
                # and R2 fell from 0.905 to 0.808 with a third of the area
                # missing. A better starting point made a worse fit, purely
                # because of where the bound is. The seed is held a little
                # inside the box; the fit is still free to go to the bound
                # if that is where the answer is, and on LTO it does.
                _lo = spec.sigma_min * (1.0 + _SEED_BOUND_MARGIN)
                _hi = spec.sigma_max * (1.0 - _SEED_BOUND_MARGIN)
                if _lo < _hi:
                    _s0 = float(np.clip(_v, _lo, _hi))
                else:
                    _s0 = float(np.clip(_v, spec.sigma_min, spec.sigma_max))
        _a0 = amp0
        if spec.amp0 is not None and i < spec.amp0.size:
            _v = float(spec.amp0[i])
            if np.isfinite(_v) and _v > 0:
                _a0 = _v
        params[pre + "sigma"].set(value=_s0, min=spec.sigma_min,
                                  max=spec.sigma_max)
        params[pre + "amplitude"].set(value=_a0, min=0)
        params[pre + "fraction"].set(value=0.5, min=0, max=1)
        if _use_asym:
            # Tied below when the ratio is shared; given its own bounds here
            # so the per-component mode has a box to work in, and so the
            # value is right if the tie is never made (a single component
            # with ASYMMETRY_SHARED off).
            params[pre + "sigma_r"].set(value=_s0, min=spec.sigma_min,
                                        max=spec.sigma_max)
        if _use_asym and not _use_band:
            # The asymmetric model carries a band width whether or not the
            # profile wants one. Held at zero, not varied: this is the bare
            # split pseudo-Voigt.
            params[pre + "width"].set(value=0.0, vary=False)
        if _use_band:
            # It has to EARN its width from the residual — but it cannot earn
            # anything if it starts on its own floor. Set to exactly 0.0 and
            # bounded below at 0.0, every component came back a peak with
            # w = 0.000 on every half-cycle: the derivative at a bound is
            # one-sided and the optimiser never moved it. That is the same
            # stall that made the seeded sigma "converge" in 15 evaluations
            # at R2 0.808 on LTO, made twice in one afternoon, which is why
            # the margin below is now applied to any seeded-and-bounded
            # parameter rather than to sigma alone.
            #
            # The start is a nudge, not an assertion: 2% of the ceiling, far
            # narrower than any real composition window, and free to fall
            # back to zero. CONVERGING to a bound is ordinary; STARTING on
            # one is the fault.
            _cap = float(_band_ceilings(spec)[i])
            if _cap <= 0.0:
                # This component is a peak: hold the band width at zero.
                params[pre + "width"].set(value=0.0, vary=False)
            else:
                _w0 = max(2.0 * _BAND_WIDTH_NEGLIGIBLE, 0.02 * _cap)
                params[pre + "width"].set(value=min(_w0, _cap), min=0.0,
                                          max=_cap)

    # --- tie the shoulders, share the mixing fraction --------------------
    # Both are lmfit `expr` constraints, so the tied parameters stop being
    # varied and `nvarys` falls: the reduction in free parameters is real and
    # is what makes the standard errors mean something. See the constants
    # block above for why these two and not others.
    shoulders = _resolved_shoulders(spec) if COUPLE_SHOULDER_SIGMA else {}
    if shoulders:
        # ONE ratio for the whole half-cycle, not one per shoulder. Two
        # shoulders sharing a ratio cost one parameter between them; a ratio
        # each costs exactly what tying them was meant to save.
        _k = spec.sigma_ratio
        params.add("sh_k",
                   value=(SHOULDER_SIGMA_RATIO_INIT if _k is None
                          else float(_k)),
                   min=SHOULDER_SIGMA_RATIO_MIN, max=SHOULDER_SIGMA_RATIO_MAX,
                   vary=(_k is None))
        for child, parent in shoulders.items():
            params[f"p{child}_sigma"].set(expr=f"sh_k * p{parent}_sigma")
            # BOTH FLANKS, ADDED IN 1.9.0.58 with per-component asymmetry.
            # Tying `sigma` alone was complete while the lineshape was
            # symmetric. Once each component carries its own `sigma_r`, a
            # shoulder whose low flank is tied and whose high flank is free
            # has regained an independent width parameter — which is the
            # exact ill-determination the coupling exists to prevent, half
            # restored. A shoulder is a scaled copy of its parent's SHAPE,
            # so the same ratio governs both sides and this costs nothing.
            if f"p{child}_sigma_r" in params and f"p{parent}_sigma_r" in params:
                params[f"p{child}_sigma_r"].set(
                    expr=f"sh_k * p{parent}_sigma_r")

    # --- which components are allowed two flanks -------------------------
    # A component seeded within `sigma_max` of a window edge is one whose
    # model necessarily extends past the measured range; that is the one the
    # split lineshape is for. Everything else is tied exactly symmetric by
    # `expr`, so it costs no parameter and cannot use a runaway flank to
    # impersonate a continuum the band machinery is there to carry. See
    # ASYMMETRY_EDGE_ONLY.
    if _use_asym and getattr(spec, "asymmetry_edge_only", False):
        _reach = float(spec.sigma_max)
        for i, c in enumerate(spec.centres):
            _near_edge = (float(c) - v.min() <= _reach) or \
                         (v.max() - float(c) <= _reach)
            if not _near_edge:
                params[f"p{i}_sigma_r"].set(expr=f"p{i}_sigma")
    # --- one asymmetry ratio for the half-cycle --------------------------
    elif _use_asym and ASYMMETRY_SHARED and len(spec.centres) >= 1:
        params.add("asym_k", value=ASYM_RATIO_INIT,
                   min=ASYM_RATIO_MIN, max=ASYM_RATIO_MAX, vary=True)
        for i in range(len(spec.centres)):
            params[f"p{i}_sigma_r"].set(expr=f"asym_k * p{i}_sigma")

    if spec.fraction is not None:
        # CALIBRATED AND HELD. The shape of the instrument's response is not
        # something each cycle gets to re-decide.
        for i in range(len(spec.centres)):
            params[f"p{i}_fraction"].set(value=float(spec.fraction),
                                         vary=False)
    elif SHARED_FRACTION and len(spec.centres) > 1:
        params.add("eta", value=0.5, min=0, max=1, vary=True)
        for i in range(len(spec.centres)):
            params[f"p{i}_fraction"].set(expr="eta")

    if baseline is not None:
        params.update(baseline.guess(y, x=v))

    if DROP_DERIVED:
        for i in range(len(spec.centres)):
            for suffix in ("fwhm", "height"):
                key = f"p{i}_{suffix}"
                if key in params:
                    del params[key]

    return model, params


def _derived(sigma, amplitude, fraction, sigma_r=None):
    """Recompute what we refused to let asteval compute during the fit.

    `sigma` is the half-width at half maximum on the LOW side and `sigma_r`
    on the high side; passing `sigma_r` None (the symmetric case) reproduces
    the old `fwhm = 2 * sigma` exactly. The height of the split shape is the
    area divided by the mean of the two sides' unit-height areas, which is
    what `asym_rect_pseudo_voigt` normalises by.
    """
    sig = max(sigma, _TINY)
    if sigma_r is None or not np.isfinite(sigma_r) or abs(sigma_r - sigma) <= _TINY:
        fwhm = _FWHM_FACTOR * sigma
        height = (amplitude / max(sig * np.sqrt(2 * np.pi), _TINY)) * (
            1 - fraction) + (amplitude / max(np.pi * sig, _TINY)) * fraction
        return float(fwhm), float(height)
    sgr = max(float(sigma_r), _TINY)
    fwhm = sig + sgr
    hl = float(_rpv_area1(0.0, 0.0, sig, fraction))
    hr = float(_rpv_area1(0.0, 0.0, sgr, fraction))
    if not (hl > 0 and hr > 0):
        return float(fwhm), float("nan")
    height = float(amplitude) / (0.5 * (1.0 / hl + 1.0 / hr))
    return float(fwhm), float(height)


# Two fitted components have COLLAPSED ONTO ONE FEATURE when their centres
# lie within this multiple of the narrower one's sigma and one of them holds
# less than DEGENERATE_AREA_FRACTION of the total fitted area. Detection is
# supposed to prevent it — two peaks closer than `min_distance_mV` are one
# peak — but a shoulder seeded next to its parent can still converge onto it.
# The cost is not cosmetic: on P3 cell A cycle 10 Discharge such a pair took
# a quarter of the surviving peak's area with it.
DEGENERATE_CENTRE_SIGMAS = 0.5
DEGENERATE_AREA_FRACTION = 0.01
DROP_DEGENERATE = True


def _degenerate_component(rows):
    """Index of one collapsed component, or None. See DROP_DEGENERATE."""
    if len(rows) < 2:
        return None
    tot = sum(abs(r["amplitude_area"]) for r in rows)
    if not np.isfinite(tot) or tot <= 0:
        return None
    order = sorted(range(len(rows)), key=lambda i: abs(rows[i]["amplitude_area"]))
    for i in order:
        if abs(rows[i]["amplitude_area"]) / tot >= DEGENERATE_AREA_FRACTION:
            break
        for j in range(len(rows)):
            if j == i:
                continue
            sep = abs(rows[i]["centre"] - rows[j]["centre"])
            ref = min(abs(rows[i]["sigma"]), abs(rows[j]["sigma"]))
            if np.isfinite(ref) and sep <= DEGENERATE_CENTRE_SIGMAS * ref:
                return rows[i]["peak_index"]
    return None


def _drop_shoulder_index(spec: FitSpec, drop):
    """
    `shoulder_of` for `spec` with component `drop` removed.

    Every index above the dropped one shifts down by one; a shoulder whose
    PARENT was dropped is released rather than re-parented, because the peak
    it was a shoulder OF no longer exists and inventing a new parent would be
    inventing physics.
    """
    n = len(spec.centres)
    raw = getattr(spec, "shoulder_of", None) or ((None,) * n)
    out = []
    for i in range(n):
        if i == drop:
            continue
        p = raw[i] if i < len(raw) else None
        if p is None or int(p) == drop:
            out.append(None)
        else:
            p = int(p)
            out.append(p - 1 if p > drop else p)
    return tuple(out)


def fit_half_cycle(voltage, dqdv, spec: FitSpec, *, key=None,
                   _refit_depth=0, _seed_index=None,
                   _bands_added=0) -> dict:
    """
    Fit one half-cycle. Returns plain data — see the module docstring.

    Never raises: a failure is a result with success=False and a reason, so
    that one bad half-cycle cannot take down a run of four hundred.

    If two components converge onto the same feature, the smaller is dropped
    and the half-cycle refitted once — see `_degenerate_component`. The
    result records `dropped_degenerate` so the count is visible rather than
    quietly better, and each component keeps `seed_index`: its position in
    the DETECTED peak list it started from. `peak_index` is renumbered by a
    refit, so joining the detected peaks on it labelled every surviving
    component with the properties of the one below it.
    """
    seed_index = (list(range(len(spec.centres))) if _seed_index is None
                  else list(_seed_index))
    v = np.asarray(voltage, float)
    y = np.abs(np.asarray(dqdv, float))
    out = {"key": key, "success": False, "reason": "", "n_points": int(v.size),
           "components": [], "r_squared": np.nan, "redchi": np.nan,
           "nvarys": 0, "baseline_degree": spec.baseline_degree,
           # The BOUNDS THIS FIT ACTUALLY USED. Downstream has to know them:
           # `analyse._flag_reliable_peaks` asks "is this component pinned at
           # its width bound, acting as baseline?", and it was asking against
           # a module constant of 200 mV while a sharp profile's real bound is
           # 30 mV — so on every LTO and LFP dataset the guard could not fire.
           "sigma_max": float(spec.sigma_max),
           "sigma_min": float(spec.sigma_min),
           # THE SAMPLING INTERVAL OF THIS CURVE, so a width can be judged
           # against what the measurement can resolve rather than against a
           # constant. LTO cell C's discharge peak is 2.6 mV wide sampled at
           # 1 mV: three points across the full width at half maximum, and a
           # width read from three points is not a measurement.
           "sample_mV": float(np.median(np.diff(np.asarray(voltage, float)))
                              * 1000.0) if np.asarray(voltage).size > 2
           else float("nan"),
           "centre_tol": float(spec.centre_tol),
           "dropped_degenerate": 0,
           # How much of the parameter count the constraints bought back.
           # `nvarys` alone cannot be read without them: a five-component fit
           # with two tied shoulders and a shared fraction varies 15
           # parameters, not 20, and the difference is the whole argument for
           # believing the standard errors.
           "n_shoulders_coupled": 0,
           "sigma_ratio_k": np.nan,
           "shared_fraction": np.nan,
           "fraction_fixed": False,
           # The ANALYSED WINDOW. The truncation test downstream asks whether
           # a fitted peak's centre +/- 2 sigma lies inside it, which is a
           # question about this fit and cannot be answered from the detected
           # peak list.
           "v_min": float(np.min(v)) if v.size else np.nan,
           "v_max": float(np.max(v)) if v.size else np.nan,
           "curve_area": float(trapezoid(y, v)) if v.size > 1 else np.nan,
           "baseline_coeffs": {},
           "component_area_sum": 0.0, "baseline_area": np.nan,
           "unattributed_area": np.nan, "unattributed_fraction": np.nan,
           "bands_from_residual": 0, "n_at_sigma_max": 0,
           "seconds": 0.0}

    if v.size < 5 or len(spec.centres) == 0:
        out["reason"] = "too few points or no peaks proposed"
        return out

    t0 = time.perf_counter()
    try:
        model, params = build_model(v, y, spec)
        # STAGED REFINEMENT — POSITIONS FIRST, THEN SHAPE.
        #
        # The same discipline a Rietveld refinement uses, and for the same
        # reason: centre, width and amplitude are strongly correlated, and
        # releasing all three at once on a multi-component curve lets the
        # optimiser trade one against another along a nearly flat direction
        # instead of descending. Get the positions right while the shape is
        # held, and the shape that follows is a short, well-conditioned step.
        #
        # Stage 1 fixes sigma and fraction at their seeded values and refines
        # centres and amplitudes (and the baseline). Stage 2 releases
        # everything, starting from stage 1's answer.
        #
        # This is also what makes the seeding worth having: stage 1 is only
        # meaningful if the widths it holds fixed are close, which is exactly
        # what detection now measures for every component including shoulders.
        # The tolerance the DATA supports, not the library default. See
        # FIT_TOLERANCE: 1.5e-8 is what made a 5-component fit spend 20,000
        # evaluations and then be recorded as a failure.
        _tol = getattr(spec, "tolerance", None)
        if _tol is None:
            _tol = FIT_TOLERANCE
        _kws = (None if not _tol
                else dict(ftol=float(_tol), xtol=float(_tol)))
        if STAGED_REFINEMENT and len(spec.centres) > 1:
            _frozen = []
            for _pn, _p in params.items():
                if (_pn.endswith("sigma") or _pn.endswith("fraction")) \
                        and _p.vary and _p.expr is None:
                    _p.set(vary=False)
                    _frozen.append(_pn)
            if _frozen:
                _stage1 = model.fit(y, params, x=v,
                                    max_nfev=spec.max_nfev, fit_kws=_kws)
                params = _stage1.params.copy()
                for _pn in _frozen:
                    params[_pn].set(vary=True)
        res = model.fit(y, params, x=v, max_nfev=spec.max_nfev, fit_kws=_kws)

        # --- THE BOUND STALL (item 39) ---------------------------------
        # A least-squares minimiser reports convergence when it can no
        # longer reduce the cost. On a parameter sitting AT its bound it
        # cannot move outward at all, so the projected gradient is zero and
        # the fit is declared converged — whether or not the bound is where
        # the minimum is. `_SEED_BOUND_MARGIN` starts every width a quarter
        # of the way inside its box for exactly this reason; nothing has
        # ever protected a fit that WALKS onto a bound during refinement.
        #
        # The measured consequence: fits stopping under 100 evaluations have
        # median R2 0.79 and height ratio 0.57, against 0.98 and 0.98 for
        # fits over 1000. And on NMC111 cell B's top of charge, 13 of 15
        # charge components finish pinned at the sigma ceiling with 18.9% of
        # their area outside the fitting window, carrying about a third of
        # the charge-side area — after 1.9.0.61 gave them bands, which moved
        # them from one bound to another rather than freeing the width.
        #
        # THE TEST IS NOT "IS THE BOUND WRONG", IT IS "DID THE OPTIMISER
        # STOP FOR A REAL REASON". A wider ceiling was measured in .58 and
        # was worse; this does not touch the bounds. It restarts the pinned
        # parameters a defined distance INSIDE their box and minimises
        # again, then keeps whichever answer has the lower cost. If the fit
        # walks back to the bound, the bound is genuinely binding and the
        # result is unchanged and still flagged. If it finds an interior
        # minimum, the previous answer was the optimiser stopping at a wall.
        #
        # Cost: one extra fit, only on half-cycles that finished on a bound.
        if BOUND_ESCAPE_RESTART and res is not None:
            _p2 = res.params.copy()
            _moved = []
            for _pn, _p in _p2.items():
                if not _p.vary or _p.expr is not None:
                    continue
                _lo, _hi = _p.min, _p.max
                if not (np.isfinite(_lo) and np.isfinite(_hi)) or _hi <= _lo:
                    continue
                _span = _hi - _lo
                _eps = BOUND_ESCAPE_PROXIMITY * _span
                if _p.value >= _hi - _eps:
                    _p.set(value=_hi - BOUND_ESCAPE_STEP * _span)
                    _moved.append(_pn)
                elif _p.value <= _lo + _eps:
                    _p.set(value=_lo + BOUND_ESCAPE_STEP * _span)
                    _moved.append(_pn)
            if _moved:
                out["bound_escape_tried"] = len(_moved)
                try:
                    _res2 = model.fit(y, _p2, x=v, max_nfev=spec.max_nfev,
                                      fit_kws=_kws)
                except Exception:                  # noqa: BLE001
                    _res2 = None
                if _res2 is not None:
                    _c1 = float(getattr(res, "chisqr", np.inf) or np.inf)
                    _c2 = float(getattr(_res2, "chisqr", np.inf) or np.inf)
                    # STRICTLY better, by more than rounding. A restart that
                    # lands on the same minimum must not churn the answer.
                    if np.isfinite(_c2) and _c2 < _c1 * (1.0 - BOUND_ESCAPE_GAIN):
                        res = _res2
                        out["bound_escape_used"] = True
    except Exception as exc:                       # noqa: BLE001
        out["reason"] = f"{type(exc).__name__}: {exc}"
        out["seconds"] = time.perf_counter() - t0
        return out

    # DID THE OPTIMISER ACTUALLY CONVERGE? Until 1.9.0.26 `success` was set
    # to True a few lines below for every fit that did not raise, and
    # `res.success` — lmfit's own verdict — was never read. On the NNM
    # triplicate that hid a great deal: 57% of half-cycle fits ran to the
    # 20,000-evaluation ceiling, gave up, and returned whatever parameters
    # they were holding, and every one of them was reported with an R2 and
    # standard errors as though it had converged. A 21-30 parameter
    # pseudo-Voigt-plus-polynomial on a noisy broad curve frequently does not
    # converge; LTO, at 6 parameters, uses a median of 183 evaluations and
    # never comes close. Recorded, not deleted — the parameters are still the
    # best description the optimiser reached — but a reader must be told.
    out["converged"] = bool(getattr(res, "success", True))
    out["nfev"] = int(getattr(res, "nfev", 0) or 0)
    out["max_nfev"] = int(spec.max_nfev)
    out["hit_iteration_cap"] = bool(out["nfev"] >= int(spec.max_nfev))

    comps = res.eval_components(x=v)
    peak_total = sum(val for name, val in comps.items()
                     if not name.startswith("bg_"))
    baseline = comps.get("bg_", np.zeros_like(v))

    rows = []
    for i in range(len(spec.centres)):
        pre = f"p{i}_"
        try:
            centre = float(res.params[pre + "center"].value)
            sigma = float(res.params[pre + "sigma"].value)
            amp = float(res.params[pre + "amplitude"].value)
            frac = float(res.params[pre + "fraction"].value)
        except KeyError:
            continue
        def _err(pname):
            e = res.params[pre + pname].stderr
            return float(e) if e is not None else np.nan
        _sigma_r = sigma
        _sigma_r_err = np.nan
        if (pre + "sigma_r") in res.params:
            _sigma_r = float(res.params[pre + "sigma_r"].value)
            _sigma_r_err = _err("sigma_r")
        fwhm, height = _derived(sigma, amp, frac, _sigma_r)
        # PEAK OR BAND — the component's own answer, not a setting.
        # `width` is the composition window it claims. It starts at zero and
        # is bounded at zero, so it can only be non-zero if the residual paid
        # for it; and it is called a band only when it is BOTH wider than the
        # cycler's voltage resolution AND separated from zero by its own
        # standard error. Anything else is a peak, which is what the w = 0
        # branch of the model reduces to exactly.
        _bw = 0.0
        _bw_err = np.nan
        if (pre + "width") in res.params:
            _bw = float(res.params[pre + "width"].value)
            _bw_err = _err("width")
        _is_band = bool(_bw > _BAND_WIDTH_NEGLIGIBLE
                        and (not np.isfinite(_bw_err) or _bw > 2.0 * _bw_err))
        # A BAND AT ITS WIDTH CEILING IS ACTING AS BASELINE, exactly as a
        # component at its sigma bound is. It is not a composition window
        # any more; it is a flat pedestal the fit is using to lift the
        # curve. Flagged here rather than discovered later, and carried out
        # to the parameter table so `reliable` can act on it.
        _bw_max = float(_band_ceilings(spec)[i])
        _bw_pinned = bool(_bw_max > 0 and _bw >=
                          _bw_max * SIGMA_BOUND_PROXIMITY_COMPLEMENT)
        rows.append(dict(
            peak_index=i,
            seed_index=(seed_index[i] if i < len(seed_index) else i),
            centre=centre, centre_stderr=_err("center"),
            sigma=sigma, sigma_stderr=_err("sigma"),
            # The HIGH-side half-width and the ratio between the two. Equal to
            # `sigma` and 1.0 exactly when the lineshape is symmetric, so the
            # columns are always present and always mean the same thing.
            sigma_r=_sigma_r, sigma_r_stderr=_sigma_r_err,
            asymmetry=(float(_sigma_r / sigma) if sigma > 0 else np.nan),
            amplitude_area=amp, amplitude_stderr=_err("amplitude"),
            fraction=frac, fraction_stderr=_err("fraction"),
            band_width=_bw, band_width_stderr=_bw_err,
            band_width_max_fitted=_bw_max,
            band_at_width_bound=_bw_pinned,
            # ETA AT ITS BOUND, for parity with `at_sigma_max` and
            # `asymmetry_at_bound`. eta = 1 is a pure Lorentzian, whose wings
            # carry area far from the centre; on a two-phase transition, whose
            # tails are set by kinetics and inhomogeneity rather than by
            # lifetime broadening, that is a shape the data has been allowed
            # to reach rather than one anything argues for. Measured on LTO
            # cell A cycle 1: eta = 1.000 and the component over-covers the
            # cell's charge by 6%.
            fraction_at_bound=bool(frac >= 1.0 - _BOUND_PROXIMITY
                                   or frac <= _BOUND_PROXIMITY),
            component_kind=("band" if _is_band else "peak"),
            fwhm=fwhm, height=height,
            # Area actually inside the fitted window, which is what any
            # capacity statement must use — the analytic amplitude includes
            # tails that were never measured.
            area_in_window=float(trapezoid(comps[pre], v)),
        ))

    bkg = {k: float(v.value) for k, v in res.params.items()
           if k.startswith("bg_")}

    # --- did two components land on one feature? -------------------------
    if DROP_DEGENERATE and _refit_depth < len(spec.centres):
        bad = _degenerate_component(rows)
        if bad is not None:
            keep = [c for i, c in enumerate(spec.centres) if i != bad]
            keep_seeds = [sx for i, sx in enumerate(seed_index) if i != bad]
            keep_sh = _drop_shoulder_index(spec, bad)
            if keep:
                out2 = fit_half_cycle(
                    voltage, dqdv, spec.replace(centres=tuple(keep),
                                                shoulder_of=keep_sh),
                    key=key, _refit_depth=_refit_depth + 1,
                    _seed_index=keep_seeds)
                if out2.get("success"):
                    out2["dropped_degenerate"] = (
                        out2.get("dropped_degenerate", 0) + 1)
                    out2["seconds"] = (out2.get("seconds", 0.0)
                                       + time.perf_counter() - t0)
                    return out2

    # --- BANDS SEEDED FROM THE CHARGE THE PEAKS CANNOT ACCOUNT FOR ------
    # See BAND_FROM_RESIDUAL. Gated on `band_width_max`, which only the
    # mechanism router sets, so a two-phase cell can never acquire one.
    # Runs once per half-cycle (`_bands_added` guards the recursion) and is
    # KEPT ONLY IF IT NAMES MORE CHARGE than the peaks alone did -- an extra
    # component that does not is not evidence of anything.
    _bands_added = int(_bands_added)
    if (getattr(spec, "band_from_residual", False)
            and float(getattr(spec, "band_ceiling", 0.0) or 0.0) > 0
            and _bands_added == 0 and v.size > 8):
        _curve_area = float(trapezoid(y, v))
        _named0 = float(trapezoid(peak_total, v))
        if _curve_area > 0:
            if spec.band_seeds:
                # THE DATASET DECIDED. Every half-cycle of this step gets the
                # same regions, so the component set has a stable identity and
                # tracking can follow it. A region this curve would not have
                # produced on its own is still fitted; whether there is
                # anything there is then a measurement rather than a change of
                # model.
                #
                # A REGION AND A MAXIMUM INSIDE IT ARE THE SAME FEATURE. Where
                # detection already placed a component in the region, that
                # component becomes the band — it is not joined by a second
                # one. Adding alongside was measured and is worse: NMC111 cell
                # B, whose detection does find a maximum on the 4.28 V shelf,
                # went from 18 components at the ceiling to 31 and from 57 to
                # 41 reliable when a band was appended next to it. Two
                # components on one feature is the degeneracy this whole pass
                # exists to remove, not a way of removing it.
                _existing = np.asarray(spec.centres, float)
                _caps0 = list(_band_ceilings(spec))
                _ceil_V = band_ceiling_volts(spec, v)
                _promoted = False
                _seed = []
                for c, w in spec.band_seeds:
                    c = float(c)
                    if not (v[0] <= c <= v[-1]):
                        continue
                    _lo, _hi = c - float(w) / 2000.0, c + float(w) / 2000.0
                    _inside = np.nonzero((_existing >= _lo)
                                         & (_existing <= _hi))[0]
                    if _inside.size:
                        # Promote the component nearest the region's centre.
                        _j = int(_inside[np.argmin(
                            np.abs(_existing[_inside] - c))])
                        # THE REGION SIZES THE BAND, THE MECHANISM BOUNDS
                        # IT — the same rule the appended-seed branch below
                        # states and follows, and which this branch never
                        # got. Capping a PROMOTED component at its region's
                        # own span is the policy that was measured and
                        # rejected there ("capping at the span exactly pinned
                        # the fitted width to the bound on 49 of 206 NMC
                        # components"). It survived here because the two
                        # branches were written eleven builds apart.
                        #
                        # An excess region is where the peaks fell SHORT, so
                        # its span UNDERSTATES the feature. On the NNM
                        # triplicate every from-bound region is exactly
                        # 2*sigma_max = 400 mV wide by construction, so this
                        # line set a 400 mV ceiling on 616 components and
                        # 616 of them ended pinned at it: the fitted width
                        # was the bound, not a measurement.
                        _caps0[_j] = max(_caps0[_j], _ceil_V)
                        _promoted = True
                        continue
                    _seed.append(dict(centre=c, span_mV=float(w),
                                      lo=_lo, hi=_hi,
                                      area=float("nan"), from_reference=True))
                if _promoted and not _seed:
                    # Nothing to add, but a component has become a band.
                    out2 = fit_half_cycle(
                        voltage, dqdv,
                        spec.replace(band_width_max=np.asarray(_caps0, float)),
                        key=key, _refit_depth=_refit_depth,
                        _seed_index=_seed_index, _bands_added=1)
                    if out2.get("success"):
                        out2["seconds"] = (out2.get("seconds", 0.0)
                                           + time.perf_counter() - t0)
                        return out2
            else:
                _regs = residual_excess_regions(
                    v, y - (peak_total + baseline),
                    min_span_mV=spec.band_residual_min_span_mV)
                _seed = [g for g in _regs
                         if g["area"] / _curve_area
                         >= spec.band_residual_min_share
                         ][:max(0, int(spec.band_residual_max))]
            if _seed:
                _c = list(np.asarray(spec.centres, float)) + \
                     [g["centre"] for g in _seed]
                _sh = list(getattr(spec, "shoulder_of", None)
                           or ((None,) * len(spec.centres)))
                _sh = _sh[:len(spec.centres)] + [None] * len(_seed)
                _s0 = (list(spec.sigma0) if spec.sigma0 is not None
                       else [np.nan] * len(spec.centres))
                _a0 = (list(spec.amp0) if spec.amp0 is not None
                       else [np.nan] * len(spec.centres))
                for g in _seed:
                    # Start the band at the size of the region it belongs to:
                    # half its span as sigma, and its own unattributed charge
                    # as amplitude where that was measured. A REFERENCE region
                    # has no area of its own in this half-cycle, so it starts
                    # from what the curve is actually carrying there.
                    _s0.append(max(spec.sigma_min, g["span_mV"] / 2000.0))
                    _a = g.get("area", np.nan)
                    if not np.isfinite(_a):
                        _m = (v >= g["lo"]) & (v <= g["hi"])
                        _a = float(trapezoid(y[_m], v[_m])) if _m.sum() > 1 \
                            else 0.0
                    _a0.append(max(float(_a), 0.0))
                _si = list(_seed_index or range(len(spec.centres)))
                _si = _si[:len(spec.centres)] + [
                    int(np.searchsorted(v, g["centre"])) for g in _seed]
                # Detected peaks keep whatever ceiling they had (zero, so
                # they stay peaks); each seeded component is allowed a band
                # no wider than the region it was found in.
                # THE REGION SIZES THE BAND, THE MECHANISM BOUNDS IT.
                # The excess region is where the peaks fell SHORT, so its
                # span understates the band's true extent -- capping at the
                # span exactly pinned the fitted width to the bound on 49 of
                # 206 NMC components. The region is therefore the STARTING
                # SIZE (`sigma0`/`amp0` above) and the mechanism's ceiling is
                # the bound. What keeps this from becoming the 0.5 V
                # free-for-all that made the decomposition incoherent is that
                # only a residual-seeded component may be a band at all;
                # every detected maximum stays a peak with a ceiling of zero.
                _caps = (list(_caps0) if spec.band_seeds
                         else list(_band_ceilings(spec)))
                _ceil = band_ceiling_volts(spec, v)
                for g in _seed:
                    _caps.append(_ceil)
                out2 = fit_half_cycle(
                    voltage, dqdv,
                    spec.replace(centres=tuple(_c), shoulder_of=tuple(_sh),
                                 sigma0=np.asarray(_s0, float),
                                 amp0=np.asarray(_a0, float),
                                 band_width_max=np.asarray(_caps, float)),
                    key=key, _refit_depth=_refit_depth,
                    _seed_index=_si, _bands_added=len(_seed))
                if out2.get("success"):
                    _named2 = float(out2.get("component_area_sum", np.nan))
                    # A region the DATASET established is fitted whether or
                    # not it happens to help this half-cycle — that constancy
                    # is the point. A region discovered here still has to earn
                    # its place by naming more charge.
                    _keep2 = (bool(spec.band_seeds)
                              or (np.isfinite(_named2) and _named2 > _named0))
                    if _keep2:
                        out2["seconds"] = (out2.get("seconds", 0.0)
                                           + time.perf_counter() - t0)
                        out2["band_seed_regions"] = _seed
                        return out2

    def _val(pname):
        pr = res.params.get(pname)
        return float(pr.value) if pr is not None else np.nan

    _eta = _val("eta")
    if not np.isfinite(_eta) and rows:
        # No shared `eta` parameter exists when the fraction was FIXED, or
        # when there is a single component. Either way the value that was
        # used is on the components.
        _eta = float(rows[0]["fraction"])
    _ak = _val("asym_k")
    if not np.isfinite(_ak) and rows:
        _r0 = rows[0]
        _ak = (float(_r0["sigma_r"] / _r0["sigma"])
               if _r0["sigma"] > 0 else np.nan)
    # THE TIE CAN BE BROKEN WITHOUT SAYING SO. `sigma_r` carries
    # `expr = asym_k * sigma` AND `min = sigma_min`, and lmfit applies bounds
    # to a constrained parameter AFTER evaluating its expression. So whenever
    # `asym_k * sigma` falls below the width floor, `sigma_r` is clipped and
    # the effective ratio is no longer `asym_k`. Measured on LTO: 19 of 61
    # components, disagreeing by up to 1.94x — and `asym_k` is the number the
    # page was quoting. The EFFECTIVE ratio is the one the curve was drawn
    # with, so that is what is reported; `asymmetry_k_fitted` keeps the
    # optimiser's value and `asymmetry_k_clipped` says they parted company.
    _ak_eff = _ak
    if rows:
        _eff = [float(r["sigma_r"] / r["sigma"]) for r in rows
                if r["sigma"] > 0 and np.isfinite(r["sigma_r"])]
        if _eff:
            _ak_eff = float(np.median(_eff))
    _ak_clipped = bool(np.isfinite(_ak) and np.isfinite(_ak_eff) and _ak > 0
                       and abs(_ak_eff - _ak) / _ak > 0.01)
    # A SHARED PARAMETER ON ITS BOUND IS NOT A MEASUREMENT — the same fault
    # as a sigma pinned on its floor, one level up, and it would be silent
    # because there is only one of these per half-cycle. Reported, not
    # clamped: the fit is still the best available description, but a reader
    # must not read a bound as an answer.
    # Against the FITTED value: that is the parameter that carries the bounds.
    _ak_pinned = bool(np.isfinite(_ak)
                      and (_ak <= ASYM_RATIO_MIN * (1.0 + _BOUND_PROXIMITY)
                           or _ak >= ASYM_RATIO_MAX * (1.0 - _BOUND_PROXIMITY)))
    out.update(n_shoulders_coupled=len(_resolved_shoulders(spec))
               if COUPLE_SHOULDER_SIGMA else 0,
               sigma_ratio_k=_val("sh_k"), shared_fraction=_eta,
               fraction_fixed=spec.fraction is not None,
               asymmetry_k=_ak_eff,
               asymmetry_k_fitted=_ak,
               asymmetry_k_clipped=_ak_clipped,
               asymmetry_at_bound=_ak_pinned,
               asymmetry_fitted=bool(getattr(spec, "asymmetry", False)))

    ss_res = float(np.sum(res.residual ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    # THE FOUR NUMBERS R2 HIDES. Computed here because this is the only place
    # the composite and the baseline both exist; costs one pass each.
    _q = (fit_quality(v, y, peak_total + baseline, baseline) if FIT_QUALITY
          else dict(height_ratio=np.nan, overshoot=np.nan,
                    max_residual_frac=np.nan, residual_runs_z=np.nan))
    out.update(_q)
    out.update(
        success=True,
        r_squared=(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
        redchi=float(res.redchi), nvarys=int(res.nvarys), components=rows,
        # Kept because a lineshape that adds a parameter has to EARN it, and
        # R2 cannot say whether it did. lmfit computes both from the same
        # residual the fit minimised.
        bic=float(getattr(res, "bic", np.nan)),
        aic=float(getattr(res, "aic", np.nan)),
        component_area_sum=float(trapezoid(peak_total, v)),
        baseline_area=float(trapezoid(np.abs(baseline), v)),
        # THE CHARGE THE MODEL CANNOT NAME. Every mAh in a dQ/dV passed
        # through the cell, so this is a number about the cell and not about
        # the fit: it is reported, never absorbed. Positive means components
        # fall short of the curve; negative means they sum to more than the
        # cell delivered, which is a decomposition fault however good the
        # residual looks.
        unattributed_area=float(trapezoid(y - (peak_total + baseline), v)),
        unattributed_fraction=(
            float(trapezoid(y - (peak_total + baseline), v)
                  / trapezoid(y, v)) if v.size > 1
            and trapezoid(y, v) > 0 else np.nan),
        bands_from_residual=int(_bands_added),
        # HOW MANY COMPONENTS ENDED AT THE WIDTH CEILING. A peak pinned at
        # `sigma_max` is not a peak the data chose the width of; on a curve
        # with a continuum in it, it is a BAND BEING DENIED -- the fitter's
        # only way to reach charge spread over a region when the mechanism
        # has not permitted a component shaped like one. Counted here so the
        # mechanism decision can act on it: see
        # `quality.MECHANISM_AT_BOUND_ESCALATES`.
        n_at_sigma_max=int(sum(
            1 for _c in rows
            if float(_c["sigma"]) >= float(spec.sigma_max)
            * SIGMA_BOUND_PROXIMITY_COMPLEMENT)),
        # The baseline coefficients, so the fitted curve can be rebuilt from
        # this dict alone. `evaluate` needs them; keeping them costs four
        # floats and saves shipping a ModelResult, which is the one thing this
        # module refuses to do.
        baseline_coeffs=bkg,
        seconds=time.perf_counter() - t0,
    )
    return out


# ---------------------------------------------------------------------------
# What R2 does not see
# ---------------------------------------------------------------------------
# R2 is dominated by the BULK of a curve, so a composite can be half again
# taller than the data, spike to 3.4x its height, or miss a peak by a third,
# and still score 0.97. Measured on the runs in hand:
#
#                     R2      height  max resid  overshoot  runs z
#   LTO discharge   0.784      0.68       40%        0%       -18
#   NNM discharge   0.977      0.98       12%        0%       -14
#   NMC discharge   0.968      1.42       41%       31%       -13
#
# Three different faults — a fit too short, a fit too tall and outside the
# data, and a wrong lineshape — and R2 announces none of them. These four do,
# they cost one pass over an array each, and they are what the eye was seeing.
FIT_QUALITY = True


def _runs_z(resid):
    """Runs test on the SIGN of the residual.

    Noise changes sign about half the time; a residual whose SHAPE is wrong
    does not — it sits above the data for a stretch and then below. Large
    negative z means long same-sign runs, which is systematic misfit rather
    than scatter, and it is the one of these four that fires even where the
    height is right (NNM: z = -14 at R2 0.98).
    """
    s = np.sign(np.asarray(resid, float))
    s = s[s != 0]
    if s.size < 10:
        return np.nan
    n1 = int((s > 0).sum())
    n2 = int((s < 0).sum())
    if n1 == 0 or n2 == 0:
        return np.nan
    runs = 1 + int((s[1:] != s[:-1]).sum())
    n = n1 + n2
    mu = 2.0 * n1 * n2 / n + 1.0
    var = (2.0 * n1 * n2 * (2.0 * n1 * n2 - n)) / (n ** 2 * (n - 1))
    return float((runs - mu) / np.sqrt(var)) if var > 0 else np.nan


def fit_quality(voltage, dqdv, total, baseline):
    """
    Four numbers about the fit that R2 cannot carry.

    `total` is the composite (peaks + baseline) and `baseline` the baseline
    alone, both evaluated on `voltage`. Returns a dict; every entry is NaN
    rather than absent when it cannot be computed, so the columns are always
    the same shape.
    """
    v = np.asarray(voltage, float)
    y = np.asarray(dqdv, float)
    t = np.asarray(total, float)
    b = np.asarray(baseline, float)
    out = dict(height_ratio=np.nan, overshoot=np.nan,
               max_residual_frac=np.nan, residual_runs_z=np.nan)
    if y.size < 3 or t.size != y.size:
        return out
    amp = float(y.max() - y.min())
    # HEIGHT, measured from the fitted baseline on both sides. The fit being
    # SHORT is what a width floor looks like (LTO); the fit being TALL is what
    # an undetermined peak/background split looks like, because the baseline
    # is drawn high and the peaks are deepened to reach the data (NMC).
    _hd = float(np.max(np.abs(y - b)))
    _hf = float(np.max(np.abs(t - b)))
    if _hd > 0:
        out["height_ratio"] = _hf / _hd
    if amp > 0:
        # How far the composite goes OUTSIDE the data's own range. A fit has
        # no business being anywhere the curve never went.
        out["overshoot"] = float(max(t.max() - y.max(), y.min() - t.min(),
                                     0.0) / amp)
        out["max_residual_frac"] = float(np.max(np.abs(y - t)) / amp)
    out["residual_runs_z"] = _runs_z(y - t)
    return out


def evaluate(fit_result, voltage):
    """
    Rebuild the fitted curves from a result dict — no refit, no `ModelResult`.

    Returns {"p0_": ndarray, ..., "bg_": ndarray, "total": ndarray}, matching
    what `ModelResult.eval_components` would have returned plus the sum.

    This exists because `fit_half_cycle` deliberately returns plain data: a
    `ModelResult` cannot cross a process boundary through stdlib pickle, and
    keeping one would undo the decision that made parallel fitting possible.
    The component shapes are evaluated through lmfit's own model classes rather
    than reimplemented here, so the curve drawn is the curve that was fitted.
    """
    v = np.asarray(voltage, float)
    out, total = {}, np.zeros_like(v)
    for comp in fit_result.get("components", []):
        pre = f"p{comp['peak_index']}_"
        # THE SHAPE THAT WAS FITTED, NOT A BARE PSEUDO-VOIGT.
        #
        # Until 1.9.0.51 this rebuilt every component as a `PseudoVoigtModel`
        # regardless of what the fit had used, so a component carrying a band
        # width or a split width was redrawn as a symmetric peak of the same
        # area — a much taller, much narrower curve. `plots._FitView` takes
        # `best_fit` AND the residual panel from here, so the peak-fits figure
        # was not showing the fit. Measured on the runs in hand: NMC111 cell A
        # cycle 5 charge fits to a maximum residual of 12% of the data's
        # amplitude and was DRAWN at 354%, with the composite 3.75x the height
        # of the curve; NNM cell A cycle 5 discharge, 2% fitted, 220% drawn.
        # That is what "the fits aren't great" was looking at.
        _w = float(comp.get("band_width", 0.0) or 0.0)
        _sr = comp.get("sigma_r", None)
        _sr = (float(_sr) if _sr is not None and np.isfinite(_sr)
               else float(comp["sigma"]))
        _split = abs(_sr - float(comp["sigma"])) > _TINY
        if _w > 0.0 or _split:
            m = _LMModel(asym_rect_pseudo_voigt, prefix=pre)
            _extra = (("width", _w), ("sigma_r", _sr))
        else:
            m = PseudoVoigtModel(prefix=pre)
            _extra = ()
        # Values are SET on the Parameters object rather than passed to
        # make_params: lmfit silently ignores a prefixed keyword there, which
        # leaves `fraction` at its 0.5 default and draws the wrong curve.
        par = m.make_params()
        for suffix, val in (("center", comp["centre"]), ("sigma", comp["sigma"]),
                            ("amplitude", comp["amplitude_area"]),
                            ("fraction", comp["fraction"])) + _extra:
            if (pre + suffix) in par:
                par[pre + suffix].set(value=float(val))
        y = m.eval(par, x=v)
        out[pre] = y
        total = total + y
    bkg = fit_result.get("baseline_coeffs") or {}
    if bkg:
        deg = max(int(k.split("c")[-1]) for k in bkg)
        m = PolynomialModel(degree=deg, prefix="bg_")
        par = m.make_params()
        for k, val in bkg.items():
            if k in par:
                par[k].set(value=float(val))
        y = m.eval(par, x=v)
        out["bg_"] = y
        total = total + y
    out["total"] = total
    return out


# ---------------------------------------------------------------------------
# Parallel execution
# ---------------------------------------------------------------------------

def _cpu_budget(n_jobs):
    if n_jobs is not None:
        return max(1, int(n_jobs))
    return max(1, (os.cpu_count() or 2) - 1)


# Measured once per session, because it cannot change within one: it is the
# cost of importing this module into a fresh interpreter.
_WORKER_RSS_MB = None


def _worker_rss_probe():
    """Run inside a worker: report what this process costs to exist, in MB.

    Unpickling this function is what makes the worker import the fitting
    machinery, so the number it returns is the real cost of a worker for THIS
    build on THIS machine. It is correct for both shapes the code ships in:
    imported from the package, loky pickles it by reference and the worker
    imports `ratatosk.fitting`; flattened into the notebook, cloudpickle sends
    it by value and the worker imports the same modules it references. Either
    way the probe pays exactly what a real job pays.

    IT RUNS A FIT, and that is not decoration. Importing the module measured
    148 MB here; the same worker after real fitting held 181 MB, because lmfit
    and scipy pull more in on first use and the fit allocates. A probe that
    only imports under-reports the thing it exists to report by a fifth, so it
    fits a small synthetic peak first — milliseconds, once per session — and
    measures afterwards.
    """
    import psutil
    x = np.linspace(0.0, 1.0, 201)
    y = 0.05 * np.exp(-0.5 * ((x - 0.5) / 0.05) ** 2)
    try:
        fit_half_cycle(x, y, FitSpec([0.5], baseline_degree=1,
                                     sigma_min=0.005, sigma_max=0.20,
                                     max_nfev=200), key="probe")
    except Exception:
        # A probe that cannot fit is still a probe: the imports it paid for
        # are the point, and the caller clamps whatever comes back.
        pass
    return psutil.Process().memory_info().rss / (1024.0 ** 2)


def measure_worker_cost_mb(*, force=False):
    """What one worker costs on this machine, in MB. None if unmeasurable.

    Starts a single worker in a PRIVATE executor, asks it, and shuts that down.

    Private matters. 1.9.0.43 used `get_reusable_executor`, which is joblib's
    process-wide SINGLETON: the probe created it, shut it down, and joblib then
    had to build a new one on top of a torn-down instance. Shutting a loky
    executor down and immediately recreating it is the classic way to leave its
    resource tracker or call queue in a state the next pool waits on forever,
    and 1.9.0.44's first long run froze in Cell 10. That is a hypothesis, not a
    proof — it did not reproduce on Linux — but the churn bought nothing, so it
    is gone. A private `ProcessPoolExecutor` cannot touch the singleton at all.

    One extra process spawn per session, ~5-8 s on Windows, against a pass
    that has already been estimated at a minute or more. Cheap, and it is a
    measurement rather than a guess about someone else's laptop.
    """
    global _WORKER_RSS_MB
    if _WORKER_RSS_MB is not None and not force:
        return _WORKER_RSS_MB
    if not WORKER_PROBE:
        return None
    ex = None
    try:
        # NOT `get_reusable_executor` — see the docstring. This is a private
        # instance of the same class, so it uses the same cloudpickle path
        # (which is what lets the flattened notebook send a `__main__`
        # function to a worker) without going near joblib's singleton.
        from joblib.externals.loky import ProcessPoolExecutor
        ex = ProcessPoolExecutor(max_workers=1)
        mb = float(ex.submit(_worker_rss_probe).result(timeout=180))
        ex.shutdown(kill_workers=True)
        ex = None
    except Exception:
        # A FAILED PROBE MUST COST NOTHING BUT THE MEASUREMENT. Its own
        # executor is torn down on the way out; joblib's singleton was never
        # touched, so the fitting that follows is unaffected either way, and
        # the caller falls back to the assumed cost.
        try:
            if ex is not None:
                ex.shutdown(kill_workers=True)
        except Exception:
            pass
        return None
    if not np.isfinite(mb) or mb <= 0:
        return None
    _WORKER_RSS_MB = mb
    return mb


def _available_memory_mb():
    """Free memory right now, in MB, or None if it cannot be read."""
    try:
        import psutil
        return psutil.virtual_memory().available / (1024.0 ** 2)
    except Exception:
        return None


def memory_budget_workers():
    """
    How many workers this machine can afford right now, and why.

    Returns (n_workers, reason). `n_workers` is None when memory could not be
    read at all — which is NOT the same as "as many as you like": the caller
    still applies `PARALLEL_MAX_WORKERS`. Zero means not even one worker fits,
    and the caller should run serially.
    """
    avail = _available_memory_mb()
    if avail is None:
        return None, ("free memory could not be read (psutil unavailable) — "
                      "sizing on cores alone")
    raw = measure_worker_cost_mb()
    # Say which number was actually used. A line reading "(measured)" beside a
    # figure a clamp had already replaced is the reporting fault this project
    # spent a week removing from everywhere else.
    if raw is None:
        cost, how = WORKER_RSS_ASSUMED_MB, "assumed — could not be measured"
    else:
        want = float(raw) * WORKER_WORKING_SET_FACTOR
        cost = min(max(want, WORKER_RSS_SANITY_MIN_MB),
                   WORKER_RSS_SANITY_MAX_MB)
        how = (f"measured {raw:.0f} MB to start, "
               f"×{WORKER_WORKING_SET_FACTOR:g} in use"
               if abs(cost - want) < 0.5
               else f"measured {raw:.0f} MB, outside the sanity bounds")
    n = int((avail * WORKER_MEMORY_HEADROOM) // cost)
    return max(0, n), (f"{avail / 1024.0:.1f} GB free, "
                       f"{WORKER_MEMORY_HEADROOM:.0%} of it usable, "
                       f"{cost:.0f} MB per worker ({how})")


def release_workers(*, verbose=False):
    """
    Shut the worker pool down and give its memory back. True if it did.

    joblib keeps its workers alive after `Parallel` returns, and they hold
    their full import footprint until the kernel exits. Measured: 412 MB still
    resident after a two-worker pass had finished, and **exiting a `with
    Parallel(...)` block does not release it** — only shutting the executor
    down does. Cell 9 and Cell 10 therefore call this when their stage is
    finished, so the rest of the notebook, and the next run in the same
    kernel, do not start against a pool that is no longer doing anything.

    NEVER CREATES A POOL IN ORDER TO CLOSE ONE. `get_reusable_executor()` is
    the obvious call and it is the wrong one: on a singleton that has already
    been shut down it starts a fresh pool, so "release the workers" would
    spawn `cpu_count` of them. The existing object is shut down directly, and
    if there is no object there is nothing to do.
    """
    try:
        from joblib.externals.loky import reusable_executor as _re
        ex = getattr(_re, "_executor", None)
        if ex is None:
            return False
        # ALREADY SHUT DOWN IS NOT "RELEASED". loky leaves the object in place
        # after a shutdown, so a bare `is None` check reported True every time
        # it was called again and claimed to have freed memory it had freed
        # once. Small, and exactly the kind of claim this project keeps
        # finding and removing.
        if getattr(getattr(ex, "_flags", None), "shutdown", False):
            return False
        ex.shutdown(kill_workers=True)
    except Exception:
        return False
    if verbose:
        print(entry("worker pool", "released", "memory returned"))
    return True


def plan_workers(jobs, *, n_jobs=None, calibrate=3, verbose=True):
    """
    Decide serially-or-parallel by measuring, and say which and why.

    Returns (n_workers, calibration_results, estimate_seconds). The calibration
    fits are real results and are reused, never thrown away.
    """
    budget = _cpu_budget(n_jobs)
    if len(jobs) <= calibrate or budget == 1:
        if verbose:
            print(f"  fitting {len(jobs)} half-cycle(s) serially "
                  f"({'too few to parallelise' if budget > 1 else 'one core'})")
        return 1, [], 0.0

    done, elapsed = [], 0.0
    for v, y, spec, key in jobs[:calibrate]:
        t = time.perf_counter()
        done.append(fit_half_cycle(v, y, spec, key=key))
        elapsed += time.perf_counter() - t

    per = elapsed / max(len(done), 1)
    # Weight by peak count: a 6-peak fit is not a 2-peak fit.
    n_cal = np.mean([len(j[2].centres) for j in jobs[:calibrate]]) or 1
    n_all = np.mean([len(j[2].centres) for j in jobs]) or 1
    estimate = per * len(jobs) * (n_all / n_cal)

    if estimate < PARALLEL_MIN_SECONDS:
        if verbose:
            print(f"  fitting {len(jobs)} half-cycle(s) serially — estimated "
                  f"{estimate:.0f} s, below the {PARALLEL_MIN_SECONDS:.0f} s "
                  f"floor where a worker pool repays its start-up")
        return 1, done, estimate

    workers = min(budget, len(jobs) - len(done))

    # --- and now the part that stops the kernel dying ----------------------
    # Everything above is a time argument. A worker also costs its imports —
    # measured at 181 MB, and more on Windows — and the old code spent that
    # `cpu_count - 1` times without ever asking whether the machine had it.
    capped_by = None
    if workers > PARALLEL_MAX_WORKERS:
        workers, capped_by = PARALLEL_MAX_WORKERS, "the ceiling"
    mem_n, why = memory_budget_workers()
    if mem_n is not None and mem_n < workers:
        if mem_n < 1:
            if verbose:
                print(f"  fitting {len(jobs)} half-cycle(s) serially — "
                      f"{why}, which is not enough for one worker. This will "
                      f"be slow; closing something would make it faster.")
            return 1, done, estimate
        workers, capped_by = mem_n, "free memory"

    if verbose:
        print(f"  fitting {len(jobs)} half-cycle(s) on {workers} worker(s) — "
              f"estimated {estimate:.0f} s serial, {estimate/workers:.0f} s "
              f"parallel")
        if capped_by == "free memory":
            print(f"      held to {workers} by memory, not cores: {why}")
        elif capped_by == "the ceiling":
            print(f"      held to the {PARALLEL_MAX_WORKERS}-worker ceiling "
                  f"({budget} cores available); {why}")
        elif mem_n is not None:
            print(f"      {why} — room for {mem_n}")
    return workers, done, estimate


# A long fit is a long silence. On a 200-cycle P3 cell the closure-interval
# pass is 600 fits and can run for ten minutes with nothing on screen, which
# is indistinguishable from a hang. Progress is printed at most this often, so
# a fast run stays quiet and a slow one keeps saying it is alive.
PROGRESS_EVERY_SECONDS = 15.0
PROGRESS_MIN_JOBS = 40


def fit_many(jobs, *, n_jobs=None, verbose=True, label=""):
    """
    Fit a list of (voltage, dqdv, spec, key) tuples.

    Order of the returned list matches `jobs`. `label` names the pass in the
    progress line, because a run makes three different ones — calibration,
    the production fit, and the closure interval — and "fitting 600" with no
    label does not say which.
    """
    if not jobs:
        return []

    workers, done, _ = plan_workers(jobs, n_jobs=n_jobs, verbose=verbose)
    total = len(jobs)
    show = verbose and total >= PROGRESS_MIN_JOBS
    tag = f"{label} " if label else ""
    t0 = time.perf_counter()
    state = {"n": len(done), "t": t0}

    def _tick(n):
        """Print at most every PROGRESS_EVERY_SECONDS, with an ETA."""
        now = time.perf_counter()
        if n < total and now - state["t"] < PROGRESS_EVERY_SECONDS:
            return
        state["t"] = now
        el = now - t0
        eta = (el / n * (total - n)) if n else float("nan")
        print(f"      {tag}{n}/{total} fitted   {el:.0f} s elapsed"
              + (f", ~{eta:.0f} s left" if np.isfinite(eta) and n < total
                 else ""), flush=True)

    if workers == 1:
        rest = []
        for v, y, sp, k in jobs[len(done):]:
            rest.append(fit_half_cycle(v, y, sp, key=k))
            if show:
                _tick(len(done) + len(rest))
        return done + rest

    from joblib import Parallel, delayed
    # `return_as="generator_unordered"` lets results be counted as they land
    # rather than all at once at the end, which is the whole point of a
    # progress line. Older joblib does not have it; there the pass runs as it
    # always did and simply reports when it finishes.
    try:
        gen = Parallel(n_jobs=workers, return_as="generator_unordered")(
            delayed(fit_half_cycle)(v, y, sp, key=k)
            for v, y, sp, k in jobs[len(done):])
        got = []
        for r in gen:
            got.append(r)
            if show:
                _tick(len(done) + len(got))
        # generator_unordered gives no ordering guarantee, so the caller's
        # order is restored from the keys it asked for. `fit_many` promises
        # order and several callers zip its output against `jobs`.
        by_key = {}
        for r in got:
            by_key.setdefault(_hashable(r.get("key")), []).append(r)
        rest = []
        for v, y, sp, k in jobs[len(done):]:
            bucket = by_key.get(_hashable(k))
            rest.append(bucket.pop(0) if bucket else
                        {"key": k, "success": False,
                         "reason": "result lost in the worker pool",
                         "components": []})
        return done + rest
    except TypeError:
        rest = Parallel(n_jobs=workers)(
            delayed(fit_half_cycle)(v, y, sp, key=k)
            for v, y, sp, k in jobs[len(done):])
        return done + list(rest)


def _hashable(k):
    """A dict key for a job key that may itself be a list or an ndarray."""
    try:
        hash(k)
        return k
    except TypeError:
        return repr(k)


In [ ]:
# DEFINITIONS E — DOES THE ANSWER MEAN ANYTHING?
# =============================================================================
"""
Does the answer mean anything?

Two orthogonal measurements, kept separate on purpose
-----------------------------------------------------
    integral fidelity  =  integral(dQ/dV) dV  /  capacity from the counter
                          "does the derivative account for the charge?"

    capacity closure   =  sum(component areas)  /  integral(dQ/dV) dV
                          "do the fitted peaks account for the curve?"

Using the counter capacity as closure's denominator would hide a Module 1
failure inside a Module 4 number. They validate different stages and must be
read separately.

Why closure is reported as an INTERVAL
--------------------------------------
A pseudo-Voigt of sigma ~ 100 mV and a cubic across a 1.4 V window are nearly
the same function. Where that is true, the split between "peak" and
"background" is not determined by the data, and any single closure value is an
artefact of the baseline degree somebody chose.

Measured on real cells, refitting only the baseline degree:

    sharp    0.95  0.95  0.95      (deg 1, 2, 3)   R2 identical to 4 dp
    broad    0.48  0.51  0.57
    broad    0.51  0.21  0.16      R2 0.944 -> 0.983 as the peaks empty out

No threshold is placed on the closure VALUE - a reader who sees 0.16-0.51 does
not need to be told the number is unusable, and every attempt in this project
to threshold a pooled closure value has failed on the next dataset. The one
threshold is on the WIDTH of the interval (PARTITION_DETERMINED_BELOW), which
decides whether a capacity share is reported or withheld; the fitted areas are
written out either way.

The interval also carries physical meaning. A partition that survives a change
of baseline is one a polynomial cannot mimic, which is what a first-order
two-phase feature looks like; a partition that collapses is a solid-solution
envelope. That reading is consistent with operando XRD for the layered and
spinel systems it was measured on — see the project notes.
"""


import numpy as np


DEFAULT_DEGREES = (1, 2, 3)

# The closure interval is a DIFFERENCE between three fits, and differencing
# amplifies where each of them stopped: if every fit stops within e of its true
# minimum, their spread carries up to 2e that is not a property of the data. So
# the closure pass has a reason to want a tighter tolerance than the production
# fit, even though it fits the same curve.
#
# Whether it NEEDS one is an open question and this flag exists to settle it in
# one run. `None` means "use `fitting.FIT_TOLERANCE`" and is the default, so
# nothing changes unless it is set.
#
# What is known:
#   * On the LTO triplicate the interval widened from a median 0.0022-0.0033 to
#     0.0037-0.0058 when FIT_TOLERANCE went from the library default to 1e-4 —
#     real, and one to two orders of magnitude inside the 0.15 threshold.
#   * On the NNM triplicate it widened on cell A (+0.004) and cell C (+0.018,
#     0.083 -> 0.125), and did not move on cell B.
#   * A bench on a 40-half-cycle sample said the opposite, that the interval
#     had NARROWED. It sampled different half-cycles and used a locally
#     detected peak list rather than the propagated reference list, so it was
#     not measuring the same quantity. Closure is a full-run number; do not
#     quote it from a bench.
#
# Set this to 1e-6 and re-run Cell 10 to test whether a tighter closure pass
# buys the interval back, and what it costs in time.
CLOSURE_TOLERANCE = None

# --- the ONE threshold on the closure interval ---------------------------
# `analyse.capacity_attribution` withholds a cycle's capacity share when its
# closure interval is wider than this. It is defined HERE, next to the
# measurement, and imported there — because for a while there were two
# numbers: attribution acted on 0.15 while `describe_partition` printed a
# verdict based on 0.05. A dataset could therefore be told in one line that
# "the partition survives a change of baseline" and have every one of its
# cycles withheld two cells later.
PARTITION_DETERMINED_BELOW = 0.15

# --- WHEN THERE IS NO BASELINE, THIS IS THE QUANTITY ----------------------
# The closure interval asks how much the peak/background split moves when the
# baseline degree changes. From 1.9.0.56 there IS no background: every mAh in
# a dQ/dV passed through the cell, the polynomial has gone, and the split it
# was measuring does not exist. `closure_interval` therefore makes no sweep on
# a spec with `baseline_degree=None` and reports a width of zero, which is a
# true statement -- nothing arbitrary is left for the answer to depend on.
#
# What replaces it is the charge the model cannot NAME. That is a statement
# about the cell rather than about the fit, and it is reported rather than
# absorbed. Measured on the six cells at 1.9.0.56:
#
#     LTO   +0.5%      one component, a two-phase transition
#     NNM   +6.0%      five components, no band permitted by the mechanism
#     NMC   +0.1%      four components, one of them a 0.5 V band
#
# 0.15 is deliberately the same number as PARTITION_DETERMINED_BELOW rather
# than a new one: it is the same judgement in the same units -- above this
# much of the cell's charge going unexplained, the decomposition is not
# describing the cell, whichever way the shortfall arose.
#
# READ NNM AND NMC TOGETHER WITH CARE. A high named fraction is not by itself
# a better decomposition: NMC reaches 99.9% partly because a 0.5 V band can
# absorb a great deal, while NNM's 94.0% is five resolved transitions and an
# honest 6% left over. The report must say this rather than let a reader rank
# them on the number.
UNATTRIBUTED_QUALIFIED_ABOVE = 0.15

# --- A PEAK AT ITS WIDTH BOUND IS A BAND BEING DENIED ---------------------
# `classify_mechanism` reads the shoulder fraction of ONE reference cycle, and
# on a replicate pair that is a thin basis. Measured on the NMC111 pair at
# 1.9.0.56, cell A came back `multi_transition` and cell B `mixed` — the same
# material, the same protocol, two different models:
#
#                              cell A (no bands)   cell B (bands)
#     components at ceiling      49/97  (51%)        17/76  (22%)
#     median FWHM                399.6 mV            197.6 mV
#     bands fitted                0                  16
#     reliable                   41/97               50/76
#     above 4.05 V               20 comps, 17 at      26 comps, 14 at
#                                ceiling, 0 bands     ceiling, 9 bands
#
# Cell A's MEDIAN component sits against a 400 mV ceiling. Denied a component
# shaped like a region, the fitter reaches charge spread over one the only way
# left to it: by stretching a peak until it stops.
#
# Nothing else caught this. Both cells named ~100% of the charge, both had
# unattributed at +0.5%, and neither had anything withheld — a stack of 400 mV
# blobs covers a curve perfectly well. That is the caveat on `closure` arriving
# as a measurement: a high named fraction is not by itself a decomposition.
#
# So the fit gets a say in the mechanism it was fitted with. Where the chosen
# mechanism permits no band and the reference fits nonetheless pin this share
# of their components at the width ceiling, the call is escalated to `mixed`
# and the reason recorded. This is not the fit overriding the classifier on a
# whim: it is one specific, named failure — and the only remedy for it is the
# component the classifier declined to allow.
MECHANISM_AT_BOUND_ESCALATES = True
MECHANISM_AT_BOUND_FRACTION = 0.30

# Closure is component area over curve area. Values a little above 1 are
# ordinary — components overlap, and a baseline that dips negative lifts the
# sum. A value above this is not a wide interval, it is a FAILED FIT: one
# component has run away and taken several times the curve's own area. Those
# are counted and named separately, because a single one of them dominates
# any min/max and makes an otherwise clean dataset look catastrophic.
CLOSURE_RUNAWAY_ABOVE = 1.5

# The other end of the same idea. A closure value only says something about
# the peak/background split if the fit it came from DESCRIBES THE CURVE. On
# LTO cell C, cycle 7 charge, the cubic refit lands on one of two minima: a
# sensible one (closure 0.973, R2 0.904) and a collapsed one where the peak
# amplitude goes to zero, the cubic takes the whole curve, and R2 comes back
# NEGATIVE (-0.023 — worse than a horizontal line). `success` is True for
# both, so both were admitted, and the interval for that half-cycle was
# 0.0155 or 0.9573 depending on which minimum that run happened to find. The
# same build gave both answers across repeat runs, and the one number decided
# whether that cycle's capacity share was published or withheld.
#
# A degree whose R2 is at or below zero has not fitted anything, so it is not
# evidence about the split and is dropped from the interval — the low-side
# counterpart of CLOSURE_RUNAWAY_ABOVE. Zero is a deliberately unambiguous
# bar: it means "worse than predicting the mean", not "fits less well than
# the other degrees". A cubic that merely fits worse is still admitted, and
# still widens the interval, which is exactly what the interval is for.
CLOSURE_DEGREE_MIN_R2 = 0.0

# --- when a reconstruction stops vouching for the curve -------------------
# The cumulative integral of dQ/dV against V should give back the (Q, V)
# curve it came from. Two numbers say how well, and both now have a stated
# threshold instead of a literal buried in a notebook cell.
#
# APE is capacity: the curve's integral against the counter's own figure. The
# histogram path reaches 0.00% by construction, and the derivative path on the
# LTO plateau reached 38%. Above 5% a peak area is a share of whatever
# fraction survived, not a share of the cell.
RECONSTRUCTION_APE_ABOVE = 5.0
# RMSE is the voltage axis, and it has to be read against the window rather
# than in absolute terms: 25 mV is 1.8% of LTO's 1.38 V window and 0.6% of a
# 4 V one. Quoted as a fraction of the window for that reason. Above this the
# curve is placing charge at voltages the cell did not visit, which moves
# peak CENTRES — the one quantity that survives where areas do not.
RECONSTRUCTION_RMSE_WINDOW_FRACTION = 0.05


# ---------------------------------------------------------------------------
# Stage 1: is the derivative itself accounting for the charge?
# ---------------------------------------------------------------------------

def integral_fidelity(voltage, dqdv, measured_capacity):
    """
    integral(|dQ/dV|) dV over the half-cycle, divided by the capacity the
    cycler counted.

    A closed physical constraint, not a heuristic: fabricated area shows up as
    an excess, lost features as a deficit. On healthy half-cycles this sits at
    0.998-1.002. The 4.199 V constant-voltage artefact would have failed it on
    sight.

    Returns nan rather than raising when the capacity is unknown, because a
    missing counter column is a reason to say nothing, not to stop.
    """
    v = np.asarray(voltage, float)
    y = np.abs(np.asarray(dqdv, float))
    # The docstring promises nan rather than an exception when the capacity
    # is unknown, and `None` is exactly how "unknown" arrives — np.isfinite
    # raises on it, so it has to be coerced first.
    try:
        cap = float(measured_capacity)
    except (TypeError, ValueError):
        return float("nan")
    if v.size < 2 or not np.isfinite(cap) or cap <= 0:
        return float("nan")
    measured_capacity = cap
    return float(trapezoid(y, v) / measured_capacity)


# ---------------------------------------------------------------------------
# Stage 2: do the peaks account for the curve?
# ---------------------------------------------------------------------------

def voltage_reconstruction(voltage, dqdv, q_measured, v_measured):
    """
    The objective test of a dQ/dV curve: can it give the voltage curve back?

    A derivative is reversible, so the cumulative integral of dQ/dV plotted
    against V should reproduce the (Q, V) curve it came from. Many
    combinations of smoothing and binning produce a dQ/dV that LOOKS right;
    far fewer reconstruct the voltage. That is the criterion Flores and Clark
    (J. Electrochem. Soc. 173, 120520, 2026) propose for choosing between IC
    computation methods, and it is the one used here to choose Ratatosk's
    default.

    Returns `{"ape_pct", "rmse_V", "q_reconstructed", "n"}`:

      ape_pct   absolute percentage error in the maximum reconstructed
                capacity. This is `integral_fidelity` expressed as an error:
                the capacity half of the test.
      rmse_V    root-mean-square difference between the original voltage
                curve and the reconstructed one, over the capacity interval
                they share. This is the half `integral_fidelity` cannot see —
                a curve can recover the right TOTAL charge and still put it
                at the wrong voltages.

    A perfect reconstruction is ape_pct = 0 and rmse_V = 0.
    """
    v = np.asarray(voltage, float)
    y = np.abs(np.asarray(dqdv, float))
    qm = np.asarray(q_measured, float)
    vm = np.asarray(v_measured, float)
    out = dict(ape_pct=float("nan"), rmse_V=float("nan"),
               q_reconstructed=float("nan"), q_measured=float("nan"), n=0)
    ok = np.isfinite(v) & np.isfinite(y)
    v, y = v[ok], y[ok]
    okm = np.isfinite(qm) & np.isfinite(vm)
    qm, vm = qm[okm], vm[okm]
    if v.size < 3 or qm.size < 3:
        return out

    order = np.argsort(v, kind="mergesort")
    v, y = v[order], y[order]

    # COMPARE OVER THE RANGE THE CURVE DESCRIBES, not the whole half-cycle.
    # Ratatosk trims to a common voltage window, so a curve legitimately says
    # nothing about charge passed outside it. On LTO the half-cycle runs to
    # 1.200 V and the window starts at 1.250 V, putting 0.5% of the charge
    # outside — and because that charge sits on the STEEP part of the voltage
    # curve, comparing against the untrimmed measurement reported an RMSE of
    # 390 mV for a curve that reconstructs to 25 mV over its own range. The
    # test would have been measuring the trim, and would have condemned the
    # better method for it.
    keep = (vm >= v.min() - 1e-9) & (vm <= v.max() + 1e-9)
    if int(np.sum(keep)) >= 3:
        qm, vm = qm[keep], vm[keep]
    # Cumulative trapezoid: the capacity accumulated up to each voltage.
    q_rec = np.concatenate([[0.0],
                            np.cumsum(np.diff(v) * 0.5 * (y[1:] + y[:-1]))])
    q_max = float(q_rec[-1])
    out["q_reconstructed"] = q_max

    q_span = float(np.nanmax(qm) - np.nanmin(qm))
    out["q_measured"] = q_span
    if q_span > 0:
        out["ape_pct"] = abs(q_max - q_span) / q_span * 100.0

    # Compare the two curves on the capacity interval they share, with the
    # measured curve interpolated onto the reconstructed capacity grid — the
    # two are defined on different capacity values, so a point-wise
    # difference is only meaningful after that.
    #
    # DIRECTION MATTERS. The reconstruction integrates from low voltage
    # upward, so on a RISING half-cycle its capacity axis runs the same way
    # as the measured one. On a FALLING half-cycle — every discharge, and
    # every charge on an anode with swapped labels — capacity accumulates as
    # voltage drops, so the two axes run opposite ways and must be flipped
    # before they are compared. Without this the test reports an RMSE of
    # 0.8 V on a curve that reconstructs the capacity to 0.5%, which is how
    # a good method gets mistaken for a broken one.
    qm0 = qm - np.nanmin(qm)
    om = np.argsort(qm0, kind="mergesort")
    qm0, vm_s = qm0[om], vm[om]
    if qm0.size > 2 and vm_s[-1] < vm_s[0]:
        qm0, vm_s = (qm0.max() - qm0)[::-1], vm_s[::-1]
    lo, hi = max(qm0.min(), q_rec.min()), min(qm0.max(), q_rec.max())
    inside = (q_rec >= lo) & (q_rec <= hi)
    if int(np.sum(inside)) >= 3 and hi > lo:
        vk = np.interp(q_rec[inside], qm0, vm_s)
        out["rmse_V"] = float(np.sqrt(np.mean((vk - v[inside]) ** 2)))
        out["n"] = int(np.sum(inside))
    return out


def closure(fit_result) -> float:
    """Component area over curve area, from a result `fitting` already made."""
    curve = fit_result.get("curve_area", float("nan"))
    if not np.isfinite(curve) or curve <= 0:
        return float("nan")
    return float(fit_result.get("component_area_sum", np.nan) / curve)


def degrees_for(spec, degrees=DEFAULT_DEGREES):
    """
    Which baseline degrees to refit at. `(None,)` when the model has no
    baseline, because there is then nothing arbitrary to sweep.
    """
    if getattr(spec, "baseline_degree", None) is None:
        return (None,)
    return tuple(degrees)


def closure_interval(voltage, dqdv, spec: FitSpec, *,
                     degrees=DEFAULT_DEGREES, key=None):
    """
    Refit the same data at several baseline degrees; report the spread.

    Returns a dict with `lo`, `hi`, `width`, `by_degree` and `r_squared_by
    _degree`. No verdict — see the module docstring.
    """
    # No background, nothing to sweep. See UNATTRIBUTED_QUALIFIED_ABOVE.
    degrees = degrees_for(spec, degrees)
    vals, r2s, unatt = {}, {}, {}
    for deg in degrees:
        res = fit_half_cycle(
            voltage, dqdv,
            spec.replace(baseline_degree=deg, tolerance=CLOSURE_TOLERANCE),
            key=key)
        vals[deg] = closure(res) if res["success"] else float("nan")
        r2s[deg] = res["r_squared"] if res["success"] else float("nan")
        unatt[deg] = (float(res.get("unattributed_fraction", np.nan))
                      if res["success"] else float("nan"))

    finite = [x for x in vals.values() if np.isfinite(x)]
    lo = float(min(finite)) if finite else float("nan")
    hi = float(max(finite)) if finite else float("nan")
    _u = [x for x in unatt.values() if np.isfinite(x)]
    return dict(key=key, lo=lo, hi=hi,
                width=(hi - lo) if finite else float("nan"),
                by_degree=vals, r_squared_by_degree=r2s,
                unattributed=(float(np.median(_u)) if _u else float("nan")),
                swept=len(degrees) > 1,
                n_degrees=len(finite))


# How many half-cycles the closure interval is measured on by default.
#
# 1.9.0.9 sampled 10. 1.9.0.11 changed it to ALL of them on the argument that
# the parallel path could afford it — and on a 200-cycle P3 cell that is 600
# fits, THREE TIMES the cost of the production fit it is checking. Cell 10
# took longer than Cell 9 and looked like a hang.
#
# 40 is the compromise, and it is a statistical one rather than a budget: the
# quantity reported is a median and the fraction inside a threshold, and 40
# evenly spaced half-cycles fix both to well inside the width being measured.
# Four times the old sample, a fifth of the cost of measuring everything.
# Set CLOSURE_SAMPLE = None in the notebook to measure every half-cycle.
CLOSURE_SAMPLE_DEFAULT = 40


def sample_half_cycles(keys, n=CLOSURE_SAMPLE_DEFAULT):
    """
    Choose a spread of half-cycles for the interval measurement.

    Stability is a property of the data's shape and is consistent within a
    dataset and step — measured spreads of 0.35/0.40 (NMC111), 0.05/0.09 (P3),
    0.00/0.01 (LTO) on pairs of cycles far apart. So it does not need
    computing on all four hundred half-cycles; an evenly spaced sample of ten
    costs three fits each and answers the same question.
    """
    keys = list(keys)
    if len(keys) <= n:
        return keys
    idx = np.linspace(0, len(keys) - 1, n).round().astype(int)
    return [keys[i] for i in sorted(set(idx.tolist()))]


def assess_closure(jobs, *, degrees=DEFAULT_DEGREES, n_jobs=None,
                   verbose=True):
    """
    Run `closure_interval` over a sample, in parallel where that pays.

    `jobs` is a list of (voltage, dqdv, spec, key). Every (job, degree) pair is
    an independent fit, so they are flattened into one batch and handed to
    `fit_many` — which decides for itself whether a worker pool is worth its
    start-up cost.
    """
    if not jobs:
        return []

    # The batch key carries the job's POSITION, not just its caller-supplied
    # key: two jobs sharing a key (the default is None) collapsed into one
    # group and every one of them got the first job's answer.
    # THE TOLERANCE TRAVELS ON THE SPEC, not in a module global: these fits
    # run in worker processes, and a value set in the parent never reaches
    # them. `CLOSURE_TOLERANCE = None` leaves each fit on `FIT_TOLERANCE`.
    flat = [(v, y, spec.replace(baseline_degree=d,
                                tolerance=CLOSURE_TOLERANCE),
             ((i, key), d))
            for i, (v, y, spec, key) in enumerate(jobs)
            for d in degrees_for(spec, degrees)]
    # No count printed here: the caller has already said how many half-cycles
    # it sampled and what that costs, and saying it twice in different words
    # is how a console becomes a wall of text.
    results = fit_many(flat, n_jobs=n_jobs, verbose=verbose,
                       label="closure")

    grouped = {}
    for res in results:
        (i, key), deg = res["key"]
        grouped.setdefault(i, {})[deg] = res

    out = []
    for i, (_, _, _, key) in enumerate(jobs):
        per = grouped.get(i, {})
        r2s = {d: r["r_squared"] for d, r in per.items()}

        def _usable(d, r):
            if not r["success"]:
                return False
            _r2 = r.get("r_squared", float("nan"))
            return bool(np.isfinite(_r2) and _r2 > CLOSURE_DEGREE_MIN_R2)

        vals = {d: (closure(r) if _usable(d, r) else float("nan"))
                for d, r in per.items()}
        n_collapsed = sum(1 for d, r in per.items()
                          if r["success"] and not _usable(d, r))
        finite = [x for x in vals.values() if np.isfinite(x)]
        lo = float(min(finite)) if finite else float("nan")
        hi = float(max(finite)) if finite else float("nan")
        _u = [float(r.get("unattributed_fraction", np.nan))
              for d, r in per.items() if _usable(d, r)]
        _u = [x for x in _u if np.isfinite(x)]
        _nb = [(int(r.get("n_at_sigma_max", 0)), len(r.get("components", [])))
               for d, r in per.items() if _usable(d, r)]
        _n_at = sum(a for a, _ in _nb)
        _n_tot = sum(b for _, b in _nb)
        out.append(dict(key=key, lo=lo, hi=hi,
                        width=(hi - lo) if finite else float("nan"),
                        by_degree=vals, r_squared_by_degree=r2s,
                        n_at_sigma_max=_n_at, n_components=_n_tot,
                        unattributed=(float(np.median(_u)) if _u
                                      else float("nan")),
                        swept=len(per) > 1,
                        n_degrees=len(finite),
                        n_degrees_collapsed=n_collapsed))
    return out


# ---------------------------------------------------------------------------
# Stage 0: is this curve worth decomposing at all?
# ---------------------------------------------------------------------------
# Asked BEFORE the fitting, on the reference half-cycle only, because the
# answer decides whether the next hour is worth spending.
#
# The pipeline already answers it — as `describe_partition`, in Cell 10, after
# every half-cycle has been fitted three times over. On NMC111 that verdict is
# "NOT DETERMINED — do not quote attribution", the capacity attribution is
# withheld on 65 of 69 cycles, and the coherence audit finds zero peaks that
# move like a redox feature. All true, all correct, and all arrived at after
# about an hour of fitting a model the data cannot support.
#
# It can be known in six fits. A material with no resolved peaks — a
# solid-solution layered oxide, where the dQ/dV is a smooth envelope with
# bends rather than maxima — does not become decomposable because the
# optimiser tries harder.
#
# TWO NUMBERS, one free and one cheap:
#
#   shoulder fraction   what proportion of the detected components are
#                       INFLECTIONS rather than maxima. Free: detection has
#                       already worked it out. Measured on the reference
#                       cycle of three chemistries it tracks the verdict
#                       exactly — LTO 0%, NNM 11-22%, NMC111 34-68%.
#
#   closure interval    on the reference half-cycle only, at the same three
#                       baseline degrees Cell 10 uses. Six fits, not six
#                       hundred. LTO 0.00, NNM 0.05-0.08, NMC111 0.51-0.57.
#
# The verdict is taken from the closure interval, against the threshold that
# already governs whether a capacity share is published
# (`PARTITION_DETERMINED_BELOW`), so this cell and Cell 10 cannot disagree.
# The shoulder fraction is reported beside it as corroboration and as the
# thing a reader can see for themselves on the detected-peaks figure.
#
# `RESOLVED_CLOSURE_FLOOR` is the one new number. A split can be perfectly
# determined and still account for a minority of the curve: NNM's peaks are
# real features sitting on a large genuine background (closure ~0.45, interval
# 0.06), which is a different situation from LTO (0.96) and needs saying
# differently in a paper. 0.75 separates the two cases as measured; it is a
# reporting boundary, not a gate, and nothing is withheld on it.
RESOLVED_CLOSURE_FLOOR = 0.75

RESOLVABILITY_RESOLVED = "resolved"
RESOLVABILITY_QUALIFIED = "qualified"
RESOLVABILITY_UNRESOLVED = "unresolved"


def assess_resolvability(jobs, *, shoulder_fraction=None,
                         n_primaries=None, n_components=None,
                         degrees=DEFAULT_DEGREES, verbose=True):
    """
    Should this dataset be decomposed into peaks at all?

    `jobs` is the reference half-cycle(s) only, in `assess_closure`'s shape:
    a list of `(voltage, dqdv, spec, key)`. One per step is the intended use.

    Returns a dict with `verdict`, the measured `width`, `closure` and
    `shoulder_fraction`, and `reason` — a sentence fit to print.
    """
    if not jobs:
        return dict(verdict=RESOLVABILITY_UNRESOLVED, width=np.nan,
                    closure=np.nan, shoulder_fraction=shoulder_fraction,
                    unattributed=np.nan, swept=False,
                    at_bound_fraction=np.nan, mechanism_escalated_from=None,
                    reason="no reference half-cycle to assess",
                    mechanism=MECHANISM_SOLID_SOLUTION, mechanism_reason="",
                    model=dict(MECHANISM_MODEL[MECHANISM_SOLID_SOLUTION]))

    res = assess_closure(jobs, degrees=degrees, verbose=False)
    widths = np.array([r["width"] for r in res if np.isfinite(r["width"])],
                      float)
    mids = np.array([0.5 * (r["lo"] + r["hi"]) for r in res
                     if np.isfinite(r["lo"]) and np.isfinite(r["hi"])], float)
    width = float(np.max(widths)) if widths.size else np.nan
    closure = float(np.median(mids)) if mids.size else np.nan
    _un = np.array([r.get("unattributed", np.nan) for r in res], float)
    _un = _un[np.isfinite(_un)]
    unattributed = float(np.median(np.abs(_un))) if _un.size else np.nan
    swept = any(bool(r.get("swept")) for r in res)

    if not np.isfinite(width) and not np.isfinite(unattributed):
        verdict = RESOLVABILITY_UNRESOLVED
        reason = ("the reference half-cycle could not be fitted, so there is "
                  "nothing to decompose")
    elif not swept and np.isfinite(unattributed):
        # NO BASELINE: the split is not undetermined, because there is no free
        # background for it to be undetermined against. The question that
        # remains is how much of the cell's charge the model can name.
        # See UNATTRIBUTED_QUALIFIED_ABOVE.
        if unattributed > UNATTRIBUTED_QUALIFIED_ABOVE:
            verdict = RESOLVABILITY_QUALIFIED
            reason = (f"the model names {1 - unattributed:.0%} of the charge "
                      f"in the reference half-cycle and leaves "
                      f"{unattributed:.0%} unattributed. Every mAh in a "
                      f"dQ/dV passed through the cell, so that shortfall is a "
                      f"statement about the decomposition, not a residual: "
                      f"areas are shares of what was named")
        else:
            verdict = RESOLVABILITY_RESOLVED
            reason = (f"with no free background to trade against, the model "
                      f"names {1 - unattributed:.0%} of the charge in the "
                      f"reference half-cycle "
                      f"({unattributed:+.0%} unattributed)")
    elif width > PARTITION_DETERMINED_BELOW:
        verdict = RESOLVABILITY_UNRESOLVED
        reason = (f"on the reference cycle — the best-behaved half-cycle in "
                  f"the dataset — the peak/background split moves by {width:.2f} "
                  f"when only the baseline degree changes, against the "
                  f"{PARTITION_DETERMINED_BELOW:.2f} at which it stops being "
                  f"decided by the data. Fitting every cycle will not make "
                  f"that smaller")
    elif np.isfinite(closure) and closure < RESOLVED_CLOSURE_FLOOR:
        verdict = RESOLVABILITY_QUALIFIED
        reason = (f"the split is determined (interval {width:.2f}) but the "
                  f"peaks account for {closure:.0%} of the curve: real "
                  f"features on a large genuine background. Areas are "
                  f"shares of the peaks, not of the cell")
    else:
        verdict = RESOLVABILITY_RESOLVED
        reason = (f"the split holds across baseline degree (interval "
                  f"{width:.2f}) and the peaks account for {closure:.0%} of "
                  f"the curve")

    _mech, _mech_why = classify_mechanism(
        shoulder_fraction, width,
        n_primaries=n_primaries, n_components=n_components)

    # --- the fit gets a say. See MECHANISM_AT_BOUND_ESCALATES -------------
    _n_at = sum(int(r.get("n_at_sigma_max", 0)) for r in res)
    _n_tot = sum(int(r.get("n_components", 0)) for r in res)
    _at_frac = (_n_at / _n_tot) if _n_tot else float("nan")
    _escalated_from = None
    if (MECHANISM_AT_BOUND_ESCALATES and np.isfinite(_at_frac)
            and _at_frac >= MECHANISM_AT_BOUND_FRACTION
            and not MECHANISM_MODEL[_mech].get(
                "band_ceiling_span_fraction", 0.0)):
        _escalated_from = _mech
        _mech = MECHANISM_MIXED
        _mech_why = (
            f"{_at_frac:.0%} of the reference cycle's components "
            f"({_n_at} of {_n_tot}) ended at the width ceiling under "
            f"{_escalated_from.replace('_', ' ')}, which permits no band. A "
            f"peak pinned at its width bound is not a width the data chose — "
            f"it is the only way a sum of peaks can reach charge spread over "
            f"a region. Escalated to MIXED so the model may contain the "
            f"component that shape needs. Original call: {_mech_why}")
    out = dict(verdict=verdict, width=width, closure=closure,
               unattributed=unattributed, swept=swept,
               at_bound_fraction=_at_frac,
               mechanism_escalated_from=_escalated_from,
               shoulder_fraction=shoulder_fraction, reason=reason,
               mechanism=_mech, mechanism_reason=_mech_why,
               model=dict(MECHANISM_MODEL[_mech]))
    if verbose:
        print(describe_resolvability(out))
    return out


def describe_resolvability(a, indent="  "):
    """The resolvability verdict, laid out to be read."""
    L = [indent + section("Are there peaks here to decompose?")]
    if a.get("shoulder_fraction") is not None and np.isfinite(
            a["shoulder_fraction"]):
        L.append(entry("shoulders", f"{a['shoulder_fraction']:.0%}",
                       "of the detected components are inflections on a "
                       "flank, not maxima"))
    if a.get("swept") and np.isfinite(a.get("width", np.nan)):
        L.append(entry("closure interval", f"{a['width']:.2f}",
                       f"reference half-cycle, baseline degrees "
                       f"{DEFAULT_DEGREES[0]}-{DEFAULT_DEGREES[-1]}"))
    if np.isfinite(a.get("closure", np.nan)):
        L.append(entry("charge named", f"{a['closure']:.2f}",
                       "of the curve accounted for by named components"))
    if np.isfinite(a.get("unattributed", np.nan)):
        L.append(entry("unattributed", f"{a['unattributed']:+.1%}",
                       "of the cell's charge the model cannot name — "
                       "reported, never absorbed"))
    if not a.get("swept"):
        L.append(bullet("There is no free background in this model, so there "
                        "is no peak/background split to be undetermined and "
                        "no baseline degree to sweep. A HIGH NAMED FRACTION "
                        "IS NOT BY ITSELF A BETTER DECOMPOSITION: a wide band "
                        "can absorb a great deal, and five resolved "
                        "transitions with an honest few percent left over is "
                        "the better answer. Read it beside the mechanism."))
    v = a.get("verdict")
    if v == RESOLVABILITY_RESOLVED:
        L.append(verdict("ok", "RESOLVED — decompose"))
    elif v == RESOLVABILITY_QUALIFIED:
        L.append(verdict("caution", "QUALIFIED — decompose, and say what the "
                                    "areas are a share OF"))
    else:
        L.append(verdict("bad", "UNRESOLVED — this curve has no peak/"
                                "background split the data decides"))
    L.append(bullet(a.get("reason", "")))
    if a.get("mechanism"):
        _m = {"two_phase": "TWO-PHASE — one transition at a fixed potential",
              "multi_transition": "SERIES OF TRANSITIONS — each one a peak",
              "solid_solution": "SOLID SOLUTION — a continuum, fitted as bands",
              "mixed": "MIXED — a solid solution with a transition on it: "
                       "bands AND peaks"}.get(a["mechanism"], a["mechanism"])
        L.append(entry("mechanism", a["mechanism"],
                       (f"escalated from {a['mechanism_escalated_from']}"
                        if a.get("mechanism_escalated_from") else "")))
        if np.isfinite(a.get("at_bound_fraction", np.nan)):
            L.append(entry("at the width ceiling",
                           f"{a['at_bound_fraction']:.0%}",
                           "of the reference cycle's components — a peak at "
                           "its width bound is a band being denied"))
        L.append(verdict("ok" if a["mechanism"] in
                         ("two_phase", "multi_transition") else "caution", _m))
        L.append(bullet(a.get("mechanism_reason", "")))
    return "\n".join(L)


# ---------------------------------------------------------------------------
# WHICH MECHANISM? — and therefore which model
# ---------------------------------------------------------------------------
# The profile class (sharp / moderate / broad) describes the SHAPE of a dQ/dV
# curve. It does not say what produced that shape, and two materials that need
# completely different models land in the same class: NNM and NMC111 are both
# "broad", and forcing them through one model damages whichever one loses.
#
# Measured, when the flat-top band model was applied to every broad dataset:
#
#     NNM cell B   closure interval  0.101 -> 0.179   reliable 60% -> 28%
#     NNM cell C   closure interval  0.052 -> 0.140   reliable 71% -> 33%
#     NMC111       closure interval  0.34  -> 0.32/0.18, convergence 13% -> 59%
#
# The same change, opposite effect, because the curves are not the same kind
# of object. So the classification has to be about the MECHANISM:
#
#   TWO_PHASE         one transition at a fixed potential. LTO, LFP. A narrow
#                     peak on a flat background; the potential is pinned by
#                     the coexistence of two phases, so there is no window to
#                     smear over.
#
#   MULTI_TRANSITION  a SERIES of distinct transitions. P3 sodium layered
#                     oxides. Several resolved maxima, each a real feature;
#                     the answer is a series of peaks and the split between
#                     them is decided by the data.
#
#   SOLID_SOLUTION    a continuum of site energies. Charge is delivered across
#                     a WINDOW of potentials as the composition moves, so the
#                     curve is a broad flat-topped envelope and its "shoulders"
#                     are artefacts of trying to build a flat top out of
#                     Gaussians.
#
#   MIXED             a solid solution WITH a phase transition on top of it —
#                     which is what NMC111 is. It needs both: bands for the
#                     envelope and peaks for the transition, in one fit.
#
# THE TWO DISCRIMINATORS ARE ALREADY MEASURED, and they separate the three
# chemistries in hand cleanly:
#
#     dataset   shoulder fraction   reference-cycle closure interval
#     LTO             0%                    0.00 - 0.02
#     NNM            11 - 22%               0.05 - 0.08
#     NMC111         34 - 68%               0.32 - 0.36
#
# A shoulder is a bend on a flank rather than a maximum. A curve made of real
# transitions has few; a solid-solution envelope fitted with peaks has many,
# because the shoulder pass is finding the flanks of the stack. The closure
# interval then says whether the peak/background split those components imply
# is decided by the data.
#
# Both thresholds sit in the gap between the measured populations, not at a
# convenient round number, and both are stated here so the next dataset can
# move them.
MECHANISM_SHOULDER_FRACTION = 0.30      # NNM tops out at 0.22, NMC starts 0.34
MECHANISM_TWO_PHASE_MAX_COMPONENTS = 2  # one transition, per half-cycle

MECHANISM_TWO_PHASE = "two_phase"
MECHANISM_MULTI_TRANSITION = "multi_transition"
MECHANISM_SOLID_SOLUTION = "solid_solution"
MECHANISM_MIXED = "mixed"

# What each mechanism asks the fitter for. `band_ceiling_span_fraction` of 0
# means the bare pseudo-Voigt; `primaries_only` drops the shoulder stack,
# which is only ever right where the shoulders are artefacts of the wrong
# shape.
#
# THE CEILING IS A FRACTION OF THE HALF-CYCLE'S OWN VOLTAGE SPAN, NOT VOLTS.
# Until 1.9.0.72 it was 0.5 V flat, for every material and every window. That
# is 38% of LTO's 1.3 V window, 33% of NMC's 1.5 V and 24% of NNM's 2.06 V —
# one number meaning three different things, and on NNM it was far below what
# the data wanted. Measured on the NNM triplicate (3003 components):
#
#   ceiling            bands pinned at it   reliable   R2      area stderr
#                                                              determined
#   0.5 V (flat)               82%            49.9%   0.9916      13-38%
#   1.2 V                       6%            69.4%   0.9957      23-40%
#   unbounded                   5%            65.2%   0.9951         --
#
# A pinned width is not a width the data chose, and lmfit cannot estimate a
# covariance for a parameter sitting on a bound — which is why two thirds of
# NNM's components had no area error bar at all. Unbounded is WORSE than 1.2
# V, not better: 16% of bands then run past 1.2 V, 103 of them end up WIDER
# THAN THE HALF-CYCLE THEY ARE FITTED IN, and not one of those has a
# determinable area. A band wider than its own window has no flanks inside
# the data; it is a pedestal, and the ceiling is what stops it.
#
# 0.6 is where the measured distribution turns. Binned by width/span in the
# unbounded run: 0.4-0.6 is the healthiest population in the dataset (61%
# reliable, 8% truncated), 0.6-0.8 falls to 31% reliable and 47% truncated,
# and above 1.0 nothing is determinable at all. On NNM 0.6 x 2.06 V = 1.24 V,
# which is the arm that measured best.
BAND_CEILING_SPAN_FRACTION = 0.6

MECHANISM_MODEL = {
    MECHANISM_TWO_PHASE:        dict(band_ceiling_span_fraction=0.0,
                                     primaries_only=False),
    MECHANISM_MULTI_TRANSITION: dict(band_ceiling_span_fraction=0.0,
                                     primaries_only=False),
    MECHANISM_SOLID_SOLUTION:   dict(
        band_ceiling_span_fraction=BAND_CEILING_SPAN_FRACTION,
        primaries_only=True),
    # MIXED KEEPS ITS SHOULDERS. 1.9.0.56 first shipped this as
    # `primaries_only=True`, copied from `solid_solution`, and it cost a real
    # feature the same day.
    #
    # The original argument for dropping shoulders was sound FOR ITS TIME: a
    # sum of Gaussians can only build a flat top by stacking overlapping
    # components, and the shoulder pass then finds the flanks of that stack.
    # On a solid-solution envelope those shoulders really are artefacts of the
    # wrong shape, and a band should REPLACE the stack rather than join it.
    #
    # That argument depended on bands being made from the detected peak list.
    # They are not any more: from 1.9.0.56 a band is seeded from the charge
    # the peaks cannot account for, at a region established across the
    # dataset, and it has nothing to do with which components were flagged as
    # shoulders. The two are decoupled, and a real shoulder on a real peak is
    # not an artefact of anything.
    #
    # Measured, NNM cell A cycle 1 discharge. Detection finds
    # 2.081, 3.147(shoulder), 3.239, 3.551, 4.084:
    #
    #   primaries_only=True   seeds drop 3.147 -> fitted 2.089 2.636 3.207
    #                         3.549 3.869 4.082   (one component where there
    #                         are two, sitting between them)
    #   primaries_only=False  seeds keep 3.147  -> fitted 2.063 2.741 3.150
    #                         3.234 3.549 3.829 4.082   (both resolved)
    #
    # The shoulder is detected in 10 of the first 12 discharge half-cycles and
    # was being stripped from every one of them. `mixed` means bands AND
    # peaks; taking the peaks' shoulders away contradicts its own name.
    MECHANISM_MIXED:            dict(
        band_ceiling_span_fraction=BAND_CEILING_SPAN_FRACTION,
        primaries_only=False),
}


def classify_mechanism(shoulder_fraction, closure_interval, *,
                       n_primaries=None, n_components=None):
    """Which redox mechanism is this curve, and therefore which model?

    Returns `(mechanism, reason)`. Both inputs come from
    `assess_resolvability`, which measures them on the reference cycle before
    any full fitting is committed to.
    """
    sf = float(shoulder_fraction) if shoulder_fraction is not None else np.nan
    ci = float(closure_interval) if closure_interval is not None else np.nan

    # THE SHOULDER FRACTION DECIDES THE MECHANISM; THE CLOSURE INTERVAL
    # DECIDES WHETHER THE ANSWER IS QUOTABLE. Two different questions, and
    # the first version conflated them: NNM cell B has 0% shoulders — every
    # component a resolved maximum — and was called a solid solution because
    # its interval came in at 0.17, a whisker over the threshold. A curve
    # with no bends on any flank cannot be a continuum envelope whatever the
    # interval says; a wide interval there means the BASELINE is undetermined,
    # which is what `verdict` already reports, separately and correctly.
    #
    # So the interval no longer gates the mechanism. It is still the reason a
    # dataset's areas may be withheld, and nothing about that changes.
    _resolved = (not np.isfinite(sf)) or sf < MECHANISM_SHOULDER_FRACTION
    if _resolved:
        _few = (n_components is not None
                and n_components <= MECHANISM_TWO_PHASE_MAX_COMPONENTS)
        if _few:
            return (MECHANISM_TWO_PHASE,
                    f"one resolved feature ({n_components} component(s)) and "
                    f"no shoulders to speak of ({sf:.0%}): a two-phase "
                    f"transition at a fixed potential")
        return (MECHANISM_MULTI_TRANSITION,
                f"{n_components if n_components is not None else 'several'} "
                f"resolved features and only {sf:.0%} of them bends on a "
                f"flank: a series of distinct transitions, each one a peak")

    # Not resolved as a set of peaks. Is there still a transition in there?
    _has_peaks = bool(n_primaries) and n_primaries >= 1
    if _has_peaks:
        return (MECHANISM_MIXED,
                f"{sf:.0%} of the components are bends on a flank and the "
                f"split moves by {ci:.2f} with the baseline — a solid-solution "
                f"envelope — but {n_primaries} resolved maxima remain, so a "
                f"transition sits on top of it. Bands AND peaks")
    return (MECHANISM_SOLID_SOLUTION,
            f"{sf:.0%} of the components are bends on a flank and the split "
            f"moves by {ci:.2f} with the baseline, with no resolved maximum "
            f"left: a continuum of site energies, and no peaks to find in it")


# ---------------------------------------------------------------------------
# Presentation — describes, does not judge
# ---------------------------------------------------------------------------

# --- ONE MATERIAL, ONE MODEL ---------------------------------------------
# A mechanism is classified from ONE reference cycle of ONE cell, and that is
# a thin basis on which to choose a model. On the NMC111 pair at 1.9.0.56 it
# split the replicates — cell A `multi_transition`, cell B `mixed` — and the
# run page had to warn that two cells fitted with different models cannot be
# pooled. That warning is right, and it is also an admission: the pooling is
# the point of running replicates.
#
# A material either delivers charge across a composition window or it does
# not. It is not a property that varies between nominally identical coin
# cells, so a replicate that did not detect the continuum is a DETECTION MISS,
# not evidence of absence — which is why this reconciles to the most permissive
# call in the group rather than to a majority. (A majority is undefined at
# n = 2, which is the common case here.)
#
# The disagreement is not hidden: every cell's own call is kept in
# `mechanism_own`, and the run page says which cells were reconciled and from
# what.
_MECHANISM_RANK = {MECHANISM_TWO_PHASE: 0, MECHANISM_MULTI_TRANSITION: 1,
                   MECHANISM_SOLID_SOLUTION: 2, MECHANISM_MIXED: 3}


def reconcile_mechanisms(resolvability, compositions, *, verbose=True):
    """
    One model per MATERIAL, across its replicates. Mutates and returns
    `resolvability` (name -> assess_resolvability dict).

    `compositions` maps the same names to a material label. Names whose
    composition is unknown or unique are left exactly as they were.
    """
    groups = {}
    for name, a in (resolvability or {}).items():
        if not a or not a.get("mechanism"):
            continue
        comp = str((compositions or {}).get(name) or name)
        groups.setdefault(comp, []).append(name)

    changed = []
    for comp, names in groups.items():
        if len(names) < 2:
            continue
        calls = {n: resolvability[n]["mechanism"] for n in names}
        best = max(calls.values(), key=lambda m: _MECHANISM_RANK.get(m, 0))
        if len(set(calls.values())) == 1:
            continue
        for n, own in calls.items():
            a = resolvability[n]
            a.setdefault("mechanism_own", own)
            if own == best:
                continue
            a["mechanism"] = best
            a["model"] = dict(MECHANISM_MODEL[best])
            # `mechanism_reconciled_from` used to be set here, holding `own` —
            # the same value `mechanism_own` two lines above already holds,
            # under a second name, read by nothing. The run page answers "was
            # this cell reconciled, and from what" from `mechanism_own`
            # (report.py:2734). One quantity, one name.
            a["mechanism_reason"] = (
                f"reconciled to {best.replace('_', ' ')}, the most permissive "
                f"call among the {len(names)} {comp} cells: this cell's own "
                f"reference cycle read {own.replace('_', ' ')}. A material "
                f"either delivers charge across a composition window or it "
                f"does not, so a replicate that did not resolve one is a "
                f"detection miss rather than evidence of absence — and two "
                f"replicates fitted with different models cannot be pooled, "
                f"which is what replicates are for. Original: "
                + str(a.get("mechanism_reason", "")))
            changed.append((n, own, best))

    if verbose and changed:
        print(section("  One material, one model"))
        for n, own, best in changed:
            print(entry(str(n)[:28], f"{own} -> {best}",
                        "reconciled across replicates"))
    return resolvability


def describe_partition(intervals, indent="  "):
    """
    What the closure interval says about this dataset, laid out to be read.

    A titled block with aligned fields and a coloured verdict — see `style`.
    The reader is looking for one of four things (how wide, how many pass, the
    verdict, whether anything went wrong) and should land on it without
    parsing a sentence.

    The verdict is decided by the FRACTION of half-cycles inside
    `PARTITION_DETERMINED_BELOW`, the same threshold `capacity_attribution`
    acts on, so the sentence and the decision cannot disagree.

    Three things this deliberately does not do:

    * It does not read a verdict off the median alone. A median of 0.04 and
      one of 0.06 are the same measurement to within the noise on a few
      hundred half-cycles, and the previous version gave them OPPOSITE
      verdicts on either side of a hard 0.05.
    * It does not quote the min and max. One failed fit sends the maximum to
      12 and makes a dataset whose median is 0.04 look ruined.
    * It does not say the areas "may be read as capacities". Closure asks
      whether the fitted peaks account for the CURVE; whether the curve
      accounts for the CELL is `integral_fidelity`, and on a two-phase
      material the answer is no.
    """
    widths = np.array([iv["width"] for iv in intervals
                       if np.isfinite(iv["width"])], float)
    los = np.array([iv["lo"] for iv in intervals if np.isfinite(iv["lo"])])
    his = np.array([iv["hi"] for iv in intervals if np.isfinite(iv["hi"])])
    n_fail = sum(1 for iv in intervals if not np.isfinite(iv["width"]))

    L = [indent + section("Closure — do the fitted peaks account for the "
                          "curve?")]
    if widths.size == 0:
        L.append(verdict("bad", "not measurable"))
        if n_fail:
            L.append(bullet(f"{n_fail} half-cycle(s) never converged"))
        return "\n".join(L)

    runaway = int(np.sum([iv["hi"] > CLOSURE_RUNAWAY_ABOVE
                          for iv in intervals if np.isfinite(iv["hi"])]))
    lo, hi = float(np.median(los)), float(np.median(his))
    w = float(np.median(widths))
    q1, q3 = (float(np.percentile(widths, 25)),
              float(np.percentile(widths, 75)))
    n_ok = int(np.sum(widths <= PARTITION_DETERMINED_BELOW))
    frac = n_ok / widths.size

    L.append(entry("closure range", f"{lo:.2f} – {hi:.2f}",
                   f"median across baseline degrees "
                   f"{DEFAULT_DEGREES[0]}–{DEFAULT_DEGREES[-1]}"))
    L.append(entry("interval width", f"{w:.2f}",
                   f"middle half {q1:.2f} – {q3:.2f}"))
    L.append(entry("determined", f"{n_ok} of {widths.size}",
                   f"{frac:.0%} inside {PARTITION_DETERMINED_BELOW:.2f}"))
    n_collapsed = int(sum(int(iv.get("n_degrees_collapsed") or 0)
                          for iv in intervals))
    if n_collapsed:
        L.append(entry("degrees dropped", f"{n_collapsed}",
                       "fit collapsed (R2 <= 0) — not evidence about the "
                       "split"))

    if frac >= 0.95:
        L.append(verdict("ok", "DETERMINED — the split holds across "
                               "baseline degree"))
        if n_ok < widths.size:
            # A dataset can sit at exactly 95% and still have a half-cycle
            # withheld. The verdict is where a reader stops, and RUN_SUMMARY
            # then says "1 cycle had their capacity attribution withheld" —
            # which reads as a contradiction unless the tick says it first.
            L.append(bullet(f"{widths.size - n_ok} half-cycle(s) still "
                            f"exceed the threshold and have their capacity "
                            f"share withheld individually."))
        L.append(bullet("Whether these areas are CAPACITIES is a separate "
                        "question; see integral fidelity."))
    elif frac >= 0.5:
        L.append(verdict("caution", "MIXED — holds on most half-cycles, "
                                    "not all"))
        L.append(bullet(f"The {widths.size - n_ok} that fail are withheld "
                        f"individually, not the whole dataset."))
    else:
        L.append(verdict("bad", "NOT DETERMINED — do not quote attribution"))
        L.append(bullet("The split depends on the baseline degree chosen, so "
                        "the areas are an artefact of that choice."))

    if runaway:
        L.append(entry("failed fits", f"{runaway} of {widths.size}",
                       f"closure > {CLOSURE_RUNAWAY_ABOVE} at some degree"))
        L.append(bullet("A component ran away and took several times the "
                        "curve's own area — a failed fit, not a wide "
                        "interval."))
    if n_fail:
        L.append(entry("no fit", f"{n_fail} of {len(intervals)}",
                       "never converged at any baseline degree"))
    return "\n".join(L)


def describe_fidelity(fidelity, indent="  ", floor=0.80,
                      ceiling=1.20):
    """
    The other half of the quality question, laid out the same way.

    `fidelity` is `{(cycle, step): ratio}` from `integral_fidelity`. Closure
    asks whether the peaks account for the CURVE; this asks whether the curve
    accounts for the CELL. They are independent, and a dataset can pass one
    completely while failing the other — the LTO triplicate scores 0.98–1.00
    on closure and 0.58–0.68 here.
    """
    per = {}
    for key, f in (fidelity or {}).items():
        step = key[1] if isinstance(key, tuple) and len(key) > 1 else str(key)
        if np.isfinite(f):
            per.setdefault(str(step), []).append(float(f))
    L = [indent + section("Integral fidelity — does the curve account for "
                          "the cell?")]
    if not per:
        L.append(verdict("caution", "not measurable"))
        return "\n".join(L)
    worst = 1.0
    most = 1.0
    for step in sorted(per):
        med = float(np.median(per[step]))
        worst = min(worst, med)
        most = max(most, med)
        L.append(entry(step.lower(), f"{med:.2f}",
                       f"median over {len(per[step])} half-cycles"))
    if worst < floor:
        L.append(verdict("bad", f"AREAS ARE NOT CAPACITIES — below "
                                f"{floor:.2f}"))
        L.append(bullet(f"Only {100 * worst:.0f}% of the delivered charge "
                        f"appears in the dQ/dV curve, so a peak area is a "
                        f"share of that fraction. On the derivative-free "
                        f"curve this means charge delivered OUTSIDE the "
                        f"analysed voltage window — a constant-voltage hold "
                        f"is the usual reason."))
        L.append(bullet("Peak POSITIONS and drift do not use the integral "
                        "and are unaffected."))
    elif most > ceiling:
        # The other half of the docstring's promise. An excess is charge the
        # cell never passed, and nothing physical produces it.
        L.append(verdict("bad", f"AREAS ARE NOT CAPACITIES — above "
                                f"{ceiling:.2f}"))
        L.append(bullet(f"The dQ/dV curve carries {100 * most:.0f}% of the "
                        f"charge the cycler counted. A curve cannot contain "
                        f"more charge than the cell delivered, so some of "
                        f"this area is an artefact — a voltage traversed "
                        f"twice, or a hold spread across voltages it never "
                        f"visited."))
        L.append(bullet("Peak POSITIONS and drift do not use the integral "
                        "and are unaffected."))
    else:
        L.append(verdict("ok", "the curve accounts for the cell's charge"))
    return "\n".join(L)


def describe_reconstruction(records, *, method, window, bin_widths=None,
                            empty_fraction=None):
    """
    Say whether the curve gives the voltage curve back, and what it cost.

    `records` are `voltage_reconstruction` results for one dataset, `window`
    is the (lo, hi) the curve was computed on, and `bin_widths` the width
    each half-cycle was ACTUALLY binned at — not the one that was asked for.
    Those differ, often: on P3 cell A, 354 of 364 half-cycles were widened
    from the requested 5 mV, to a median of 7.9 and a worst of 198, and
    nothing said so while the manifest recorded "5.0".
    """
    out = []
    if not records:
        return "\n".join([entry("reconstruction", "not measured",
                                 "no half-cycle had both a curve and a "
                                 "capacity column")])
    ape = float(np.nanmedian([r["ape_pct"] for r in records]))
    rmse = float(np.nanmedian([r["rmse_V"] for r in records]))
    span = float(window[1] - window[0]) if window else float("nan")
    frac = rmse / span if span and np.isfinite(span) and span > 0 else np.nan

    if bin_widths is not None and len(bin_widths):
        w = np.asarray(bin_widths, float)
        w = w[np.isfinite(w)]
    else:
        w = np.empty(0)
    if w.size:
        if w.max() - w.min() < 1e-9:
            out.append(entry("bin width", f"{w.min():g} mV",
                             "the same on every half-cycle"))
        else:
            # The widening is not a fault: a bin narrower than the records
            # themselves cannot be filled, so it is opened up where they are
            # sparse. It is only a fault to REPORT the requested width as
            # though it had been used.
            n_wide = int((w > w.min() + 1e-9).sum())
            out.append(entry("bin width",
                             f"{np.median(w):.3g} mV median",
                             f"{w.min():g}-{w.max():.3g} mV; widened on "
                             f"{n_wide} of {w.size} half-cycles where the "
                             f"records were sparser than the bin"))
    if empty_fraction is not None and np.isfinite(empty_fraction):
        out.append(entry("bins with no charge",
                         f"{100 * empty_fraction:.0f}%"))
        if empty_fraction > 0.5:
            out.append(bullet(
                "Expected on a flat plateau: the charge is concentrated in a "
                "few mV and the rest of the window is genuinely empty. The "
                "bins are not lost data — an empty bin carries zero and "
                "contributes nothing to any area."))

    out.append(entry("reconstruction", f"{ape:.2f}% APE",
                     f"voltage RMSE {1000 * rmse:.0f} mV"
                     + (f" = {100 * frac:.1f}% of the window"
                        if np.isfinite(frac) else "")
                     + f", median over {len(records)} half-cycles"))

    bad_ape = ape > RECONSTRUCTION_APE_ABOVE
    bad_rmse = np.isfinite(frac) and frac > RECONSTRUCTION_RMSE_WINDOW_FRACTION
    if bad_ape:
        out.append(verdict("caution", "the curve does not account for the "
                                      "cell's charge"))
        out.append(bullet(
            "A peak area on this dataset is a share of the fraction that "
            "survived, not of the cell."
            + ("" if method == "histogram" else
               " Try DQDV_COMPUTATION = \"histogram\" in Cell 2.")))
    elif bad_rmse:
        out.append(verdict("caution", "the curve places charge at voltages "
                                      "the cell did not visit"))
        out.append(bullet(
            f"Capacity is accounted for to {ape:.2f}%, but the voltage axis "
            f"is out by {100 * frac:.1f}% of the window. Peak centres are "
            f"the quantity this moves."))
    else:
        out.append(verdict("ok", "the curve integrates back to the "
                                 "measured capacity"))
    return "\n".join(out)


In [ ]:
# DEFINITIONS F — TRACKING, PAIRING, ATTRIBUTION, MECHANISM
# =============================================================================
"""
From fitted components to statements about a cell — and to what must be withheld.

This module does four things, and one of them is new.

    parameters_frame     fit dicts -> the long component table, with the
                         reliability flags 1.8.7 applied in Module 4
    track_peaks          join independent per-cycle fits into peak histories
    delta_v              charge/discharge pairing and polarisation
    capacity_attribution each peak's share, WITHHELD where the closure
                         interval says the share is not determined
    coherence_audit      does each peak move like a redox feature?
    integrity            NEW: what the reversed charge in a half-cycle means

What the mechanism classification is for
----------------------------------------
1.8.x had one number, `f_wrong_sign` — the share of |dQ/dV| carried by points
of the wrong sign — and used it as a gate. It conflated three unrelated things:

    at a redox plateau, at full current, every cycle      a two-phase feature
    at the voltage limit, current tapering                a constant-voltage hold
    away from any feature, at full current, sporadic      charge going elsewhere

Only the third is a fault, and a single threshold on the first number cannot
find it: it transferred neither between cells of one triplicate nor between
chemistries. LTO's whole plateau is "wrong sign" and its coulombic efficiency
is 99%.

`signal.reversals` already inventories each contiguous backwards run — how much
charge, over what voltage band, at what current. This module asks where each
run sat on the curve:

    on a plateau     |dQ/dV| there is >= PLATEAU_DQDV_FRACTION of this
                     half-cycle's own maximum. The potential is pinned by a
                     two-phase equilibrium and wanders within the measurement;
                     the charge is real and intended.
    at the limit     current below the constant-current value, within
                     LIMIT_TOL_MV of the window edge. Protocol, not chemistry.
    parasitic        full current, |dQ/dV| small, away from the limit. Charge
                     passed with no redox process to account for it.

Every term is a property of THIS half-cycle measured against ITSELF — its own
peak |dQ/dV|, its own current, its own capacity. That is the whole reason it
transfers, and it is what `f_wrong_sign` and `f_voltage_backwards` could not do.

Measured on all six reference cells (1,215 P3 and 61 LTO half-cycles):

    CE band        n     mean parasitic charge    max parasitic fraction
    0-20%         22          6.0 mAh/g                 11.1%
    20-40%        75          2.0                       11.4%
    40-60%       118          0.5                       13.1%
    60-80%       178          0.0                        0.0%
    80-95%       550          0.0                        0.4%
    95-105%       35          0.0                        0.0%
    >105%        237          0.0                        0.0%

Twenty-one of 1,215 P3 half-cycles carry any parasitic charge at all, and every
one of them sits in a cycle whose coulombic efficiency is below 60%. Nothing
above 60% carries more than 0.4% of its capacity backwards. On LTO — a flat
two-phase plateau, the case that broke every previous rule — two half-cycles of
61 carry any, and both are the truncated final step of the record.

This is not a correlation dressed as a diagnostic. The quantity IS the charge
that went in and did not come back, localised to where on the curve it went.

Note the ordering that falls out of it: the plateau test reads |dQ/dV|, not a
peak list, so integrity does not depend on detection. The pipeline stays a
straight line, and `detect.choose_reference_cycle` can be handed the callable
`integrity_report` returns.
"""


from dataclasses import dataclass, field

import numpy as np
import pandas as pd

# The one definition of "this half-cycle is unfinished", shared with the
# cycling tables so the fit and the headline metrics omit the same thing.

from scipy.optimize import linear_sum_assignment
from scipy.special import erf


# =============================================================================
# DEFAULTS — 1.8.7's values, unchanged, except where marked NEW
# =============================================================================

# --- Module 5 -----------------------------------------------------------
TRACKING_TOLERANCE_MV = 80
# How many cycles at each end a drift measurement medians over. See the
# comment where it is used: two single cycles carry sqrt(2) times the
# per-cycle centre scatter, and on a 50-cycle synthetic series that turned
# 90 mV of true drift into 79 mV with no bias anywhere in the centres.
DRIFT_ENDPOINT_CYCLES = 3
# --- the tracker's hard limit -------------------------------------------
# `_assign_components` matches every cycle against the REFERENCE CYCLE's peak
# list within +/- TRACKING_TOLERANCE_MV. A peak that drifts further than that
# over the life of the cell cannot be matched to itself any more — and it does
# not simply stop being tracked, it starts being matched to whichever
# component is now nearest, which on a crowded curve is a DIFFERENT PEAK.
#
# Measured on the synthetic ground truth. At 2 mV/cycle:
#
#     50 cycles  (~100 mV total)   centre rms  4-14 mV    tracking rate 94-100%
#    100 cycles  (~200 mV total)   centre rms 53-141 mV   tracking rate 78-99%
#                                  drift reported as -53 mV against +172 true
#
# The second row is not degradation, it is a different peak. 200 mV of drift
# over 100 cycles is unremarkable for a degrading cathode, so this is a real
# limit and not a corner case, and it is flagged rather than hidden: a peak
# whose shift ever exceeds the tolerance carries `drift_exceeds_tolerance` and
# the summary says the assignment beyond that point is not trustworthy.
#
# 1.9.0.23 DOES THIS. The expectation is now carried forward from the last
# cycle each peak was matched in, walking outward from the reference cycle in
# both directions, so `TRACKING_TOLERANCE_MV` is a PER-CYCLE step limit and
# the lifetime drift is unbounded. Measured on the same harness, 50 cycles at
# 2 mV/cycle:
#
#                      fixed reference          drifting reference
#     centre rms        4-104 mV                 see the test output
#     tracking rate     92-100%
#
# See `_assign_components`. `TRACK_AGAINST_PREVIOUS_CYCLE = False` restores
# the fixed reference, which is how the two are compared.
TRACK_AGAINST_PREVIOUS_CYCLE = True
FLAG_DRIFT_BEYOND_TOLERANCE = True
GHOST_AREA_THRESHOLD = 1e-6
DISCONTINUITY_DROPOUT_RUN = 3
DISCONTINUITY_STEP_FLOOR_MV = 80.0
DISCONTINUITY_STEP_K = 4.0
DISCONTINUITY_TERMINAL_RUN = 10
DISCONTINUITY_ESTABLISH_MIN = 15
_AREA_COLLAPSE_FRAC = 0.05

# --- Module 4's reliability guards --------------------------------------
# "Pinned at its width bound" — ONE definition, used by both the parameter
# filter and the tracking filter. They previously used 0.05 and 0.10 for the
# same idea, so a component could be reliable in one table and not the other.
SIGMA_BOUND_PROXIMITY = 0.05
SIGMA_BOUND_PROXIMITY_COMPLEMENT = 1.0 - SIGMA_BOUND_PROXIMITY
# The fallback width bound, used only when a fit result does not carry the
# bound it was actually run under (an old result, or a hand-built FitSpec).
# The live value comes from `fitting.PROFILE_SPECS` by way of
# `sigma_max_fitted`; see `_flag_reliable_peaks`.
FIT_SIGMA_MAX_MV = 200
FIT_WIDTH_FLOOR_FACTOR = 0.0     # flag only; 0 = do not enforce
FIT_MIN_AREA_FRACTION = 0.0      # 0 = disabled

# A COMPONENT THAT CARRIES NO CHARGE IS NOT A PEAK — measured against the
# largest component in ITS OWN half-cycle, not against a dataset median.
#
# `FIT_MIN_AREA_FRACTION` above compares each area to the median area over
# every reliable component in the dataset, and that test defeats itself: once
# enough components are empty the median is empty too, and nothing is flagged.
# Measured on the LTO triplicate after the detection floor came down to 5 mV,
# 54 of cell B's 78 components carried less than 1% of the charge in their own
# half-cycle and 27 of those were marked `reliable`. They are the price of a
# detector that can now see a doublet: it also seeds the odd bump in the flat
# region, and the fit correctly gives those components no area — 8.9e-13 of
# it, in one case, beside a real peak of 130.
#
# So the test is per half-cycle and scale-free. 1% is not a tuning knob: a
# component at a thousandth of its neighbour is the optimiser saying the
# feature is not there, and there is no dataset in hand where a real feature
# sits below 1% of the largest in its own half-cycle.
EMPTY_COMPONENT_FRACTION = 0.01

# HOW MANY SAMPLING INTERVALS A WIDTH NEEDS BEFORE IT IS A MEASUREMENT.
# Four is the smallest number that gives a half-maximum crossing on each
# flank with a point between: below it the FWHM is being read off three
# points or fewer, which fixes a width the way two points fix a curve.
# Flagged, never enforced — the AREA of such a component is still the best
# estimate available, and it is only the WIDTH that is unsupported.
WIDTH_MIN_SAMPLES = 4.0

# --- Is the AREA a measurement? -----------------------------------------
# The reporting audit added a gate on `converged`, and it was the wrong
# quantity: convergence is a STATUS — did the optimiser stop for its own
# reasons or run out of budget — and status is not quality. Measured on the
# NNM triplicate it pointed the wrong way outright, demoting 405 of 405 and
# 462 of 462 components whose fits had the HIGHER median R2 (0.984 / 0.987
# against 0.975 / 0.972), because "did not converge" there meant "was still
# descending when the budget ran out". With `fitting.FIT_TOLERANCE` fixed the
# budget is rarely exhausted, and the flag is now merely uninformative rather
# than inverted — on 325 components across the three chemistries, NONE of the
# 35 non-converged had a poorly-determined area while 42 of the 290 converged
# ones did. So the gate is gone and the flag is reported instead.
#
# The quantity that IS about the measurement is the one item 16 exposed on the
# FWHM panel: a parameter whose standard error is comparable to its own value
# is not a measurement of anything. Measured, `amplitude_stderr / |area|`:
#
#   LTO cellC    median 0.023   nothing above 0.20
#   NNM cellC    median 0.073   16% above 0.35
#   NMC111 cellA median 0.498   and 73 of 87 components have NO stderr at all
#
# which is the numerical statement of what the closure interval says about
# each chemistry, arrived at independently of it. A missing standard error is
# the starkest case: the covariance could not be estimated, so the area has no
# uncertainty to quote.
#
# FLAGGED, NOT ENFORCED, deliberately — the same decision `too_narrow` records
# above. Enforcing it would demote roughly a third of NNM and most of NMC, and
# that is a change to what the tool will quote, which is Nik's call to make
# with these numbers in front of him rather than mine to slip in beside an
# optimiser fix. Set ENFORCE_AREA_DETERMINACY = True to apply it.
AREA_STDERR_RATIO_MAX = 0.35
ENFORCE_AREA_DETERMINACY = False

# --- Module 6 -----------------------------------------------------------
PAIR_MAX_SEPARATION_MV = 200
EXCLUDE_FORMATION_FROM_DELTAV = True

# --- Module 7c ----------------------------------------------------------
COHERENCE_DRIFT_IMPLAUSIBLE_MV = 20.0
COHERENCE_MEDIAN_COHERENT_MV = 5.0
COHERENCE_MEDIAN_SUSPECT_MV = 15.0
COHERENCE_MIN_PAIRS = 8
COHERENCE_AREA_RATIO_MAX = 2.0
COHERENCE_BAND_TOL_MV = 70.0

# --- NEW in 1.9.0: mechanism and integrity ------------------------------
# A reversal sitting where |dQ/dV| is at least this fraction of the
# half-cycle's OWN maximum is on a plateau. Measured across three chemistries:
# at 0.05, healthy half-cycles (CE 90-110%) carry at most 0.4% of their
# capacity backwards, and every half-cycle that carries more sits in a cycle
# with CE below 60%.
PLATEAU_DQDV_FRACTION = 0.05
LIMIT_TOL_MV = 50.0              # "at the voltage limit"
# Bands. Judged against the half-cycle's OWN counter capacity, never against a
# population — the pooled-threshold mistake was made twice in 1.8.x and
# transferred neither between cells nor between chemistries.
PARASITIC_ANOMALOUS_FRACTION = 0.02
INTEGRITY_MIN_RECORDS = 20

# A half-cycle cannot pass more charge than the material can hold. Above this
# multiple of the theoretical capacity, the intended reaction does not account
# for what happened, whatever the shape of the curve says — and that is true
# independently of the plateau/parasitic split, which reads the curve.
#
# This test exists because the split alone got it wrong. On P3 cell A it
# reported 598 mAh/g of reversed charge as sitting on a two-phase plateau, and
# called that reassuring. 94% of it is on nine half-cycles delivering 1.3 to
# 4.3 times the theoretical capacity — cycle 1 charge alone puts in 681 mAh/g
# against a theoretical 160. Charge at 4.3x theoretical is something breaking,
# not a phase transition. On the half-cycles that ARE physically possible the
# plateau charge is 38 mAh/g, not 598.
#
# 1.2 rather than 1.0: the first cycle legitimately exceeds the reversible
# capacity through SEI formation and other irreversible processes, and the
# theoretical figure is itself a nominal number for the composition.
OVER_THEORETICAL_FRACTION = 1.2
# Below this a constant-voltage hold has carried nothing worth reporting.
# A cycler records capacity to about a microamp-hour; on an 11 mg electrode
# that is ~1e-4 mAh/g, so a hold under 0.01 mAh/g is the arithmetic of
# rounding rather than a protocol artefact worth a line of console.
CV_HOLD_REPORT_FLOOR = 0.01

# A cell that has stopped delivering charge. Judged against the dataset's OWN
# working capacity, like everything else here, so it transfers between a
# 50 mAh/g cathode and a 130 mAh/g anode. On P3 cell A the record runs to cycle
# 220 while the cell delivers 0.00-0.04 mAh/g from about cycle 130: ninety
# cycles of nothing, each of which the fitter would otherwise fit.
DEAD_CELL_CAPACITY_FRACTION = 0.02
DEAD_CELL_RUN = 3

# The one threshold in 1.9.0's reporting, and it sits inside a measured gap.
# Closure-interval widths: LTO 0.00-0.01, P3 0.00-0.09, NMC111 0.35-0.40,
# synthetic `degenerate` 0.81. Nothing observed lies between 0.09 and 0.35.
# Above this, the peak/background split is not determined by the data, so the
# per-peak SHARE of capacity is not a measurement and is withheld. The areas
# themselves are kept — withheld means "not reported as a fraction", not
# "deleted".
# ONE number, defined beside the measurement it thresholds. Until 1.9.0.13
# attribution withheld above 0.15 while `quality.describe_partition` printed
# its verdict against 0.05, so a dataset could be told "the partition survives
# a change of baseline" and then have every one of its cycles withheld.
ATTRIBUTION_WITHHOLD_ABOVE = PARTITION_DETERMINED_BELOW
# ...and the same rule for the charge the model could not name. Kept as its
# own name because the two grounds are different questions and a reader
# chasing one should not land on the other: the interval asks whether the
# split is decided by the data, this asks whether the components add up to
# the cell. See `quality.UNATTRIBUTED_QUALIFIED_ABOVE` for the measurement.
UNATTRIBUTED_WITHHOLD_ABOVE = UNATTRIBUTED_QUALIFIED_ABOVE

# How much of a component's analytic area has to fall OUTSIDE the fitted
# window before its area is materially a lower bound rather than a
# measurement. `fit_truncated` and `area_is_lower_bound` have been computed
# and carried to the CSV since 1.9.0.5x and read by NOTHING — the third time
# this build has found a correct diagnostic consumed by nobody, after
# `band_at_width_bound`'s attribution half and `band_width_max_fitted`.
#
# Measured on the NNM triplicate, 1.9.0.72: 507 of 3008 components (17%) are
# more than a tenth outside the window, and 214 OF THOSE ARE MARKED
# `reliable` — a median 34% of their area, up to 50%, sitting outside the
# data, with nothing on the page saying so. They are the 2.1 V and 4.1 V
# features at the ends of the voltage window.
#
# A tenth is where the number stops being a rounding matter: below it the
# enclosed-fraction correction is smaller than the area's own scatter
# between cycles, above it the quoted area understates the feature by more
# than any trend being read off it.
AREA_LOWER_BOUND_REPORTABLE = 0.10

# Should a component whose area is a LOWER BOUND be struck from `reliable`,
# or only reported as one? The same open question as
# `ENFORCE_AREA_DETERMINACY`, and answered the same way for the same reason:
# `reliable` is a statement about the FIT, and whether the window caught the
# whole feature is a statement about the MEASUREMENT. Flag, report, do not
# enforce — turn this on if you decide `reliable` should mean "the area is
# quotable as it stands". On NNM 1.9.0.72 it costs 214 of 2066 reliable
# components (68.7% -> 61.6%).
#
# THIS WAS NOT A FREE CHOICE — IT WAS MEASURED. `sigma_max` was swept
# 0.200 -> 0.350 V on the NNM triplicate to test whether it censors widths
# the way the band ceiling did. Result:
#
#   at the sigma bound      10.0% -> 1.8%
#   reliable                68.7% -> 76.3%
#   median R2             0.99566 -> 0.99567     <- IDENTICAL
#   area stderr determined  41.3% -> 49.1%
#
# and, matching the SAME components across the two runs, the 257 that were at
# the 0.200 bound moved their larger flank a median of 26 mV (0.200 ->
# 0.226) while the area sitting outside the window barely moved (39.5% ->
# 38.8%) — and 79% of them became `reliable`.
#
# 203 components flip to reliable. 200 OF THEM ARE STILL MORE THAN A TENTH
# OUTSIDE THE WINDOW, a median 39% and up to 50%, and 99% sit within one
# sigma_max of a window edge (median 55 mV from it). 6.6 of the 6.7 points
# are areas that are still lower bounds.
#
# So sigma_max is NOT the band ceiling wearing a different hat. The band
# ceiling bounded a quantity the data could measure, and freeing it improved
# R2 from 0.9916 to 0.9957. This bounds a quantity the data CANNOT measure —
# a component at the window edge has no data on one flank, so nothing but the
# bound stops it — and freeing it changes R2 in the fifth decimal place while
# moving 200 unquotable areas into the reliable column. Raising it was
# rejected. What the population needed was to be NAMED, which is what
# `area_mostly_outside_window` and the report line now do.
ENFORCE_AREA_COMPLETENESS = False


# =============================================================================
# PORTED FROM 1.8.7 — extracted programmatically, do not edit without a
# regression. `_flag_reliable_peaks` from Cell 3, the rest from Modules 5 and 6.
# =============================================================================

def _flag_reliable_peaks(params_df, sigma_max_mV=None,
                         area_floor=0.01,
                         bound_proximity=SIGMA_BOUND_PROXIMITY):
    """
    Return a boolean Series (aligned to params_df): True for peaks that are
    physically reliable, False for degenerate fits that should not appear in
    a publication parameter table.

    Degenerate = integrated area essentially zero (the fitter retained a peak
    that has effectively vanished) OR width inflated to the upper sigma bound
    (the 'peak' is acting as baseline). These mirror two of Module 7's health
    flags. The LOWER sigma bound is deliberately not used: a sharp, genuine
    peak (e.g. the LTO two-phase feature) sits near it by design, so dropping
    on that criterion would remove real data.
    """
    import pandas as pd
    if params_df is None or len(params_df) == 0:
        return pd.Series(dtype=bool)
    if 'amplitude_area' not in params_df or 'sigma' not in params_df:
        return pd.Series(True, index=params_df.index)
    # PER ROW, from the bound that fit actually used. A module-level 200 mV
    # could never be reached by a sharp profile (bound 30 mV) or a moderate
    # one (100 mV), so on every LTO and LFP dataset this test — the one the
    # docstring says catches a component acting as baseline — was dead.
    if "sigma_max_fitted" in params_df:
        bound = pd.to_numeric(params_df["sigma_max_fitted"], errors="coerce")
    else:
        bound = pd.Series(np.nan, index=params_df.index)
    if sigma_max_mV is not None:
        bound = bound.fillna(float(sigma_max_mV) / 1000.0)
    bound = bound.fillna(FIT_SIGMA_MAX_MV / 1000.0)
    sigma_high = bound * (1 - bound_proximity)
    area = params_df['amplitude_area'].abs()
    sigma = params_df['sigma']
    # A comparison against NaN is False on float dtype, not NaN, so
    # `.fillna(True)` never fired and a component with a NaN area or sigma
    # was marked RELIABLE. Unknown is not the same as sound: it is tested
    # explicitly.
    # BOTH FLANKS, 1.9.0.58. `sigma` alone was the whole width while the
    # lineshape was symmetric. With a split one, a component whose HIGH flank
    # is pinned at the ceiling — which is what a terminal feature does — was
    # passing this test and being reported as reliable. See
    # `fitting.ASYMMETRY_SHARED`.
    sigma_r = params_df["sigma_r"] if "sigma_r" in params_df else sigma
    wide = (sigma >= sigma_high) | (sigma_r >= sigma_high)
    degenerate = ((area < area_floor) | wide
                  | area.isna() | sigma.isna())
    return ~degenerate


def _coherent_net_change(cycles, values, gap_break=3, edge_n=3,
                         min_points=5):
    """
    Model-free magnitude of change over the leading coherent window.

    Makes no linearity assumption: returns the difference between the
    median of the last edge_n points and the first edge_n points within
    the coherent window (the same leading window used elsewhere). Use this
    for descriptors whose evolution is clearly not linear, e.g. ΔV.

    Returns (net_change, v_start, v_end, c_start, c_end, n_used).
    """
    s = pd.DataFrame({'c': np.asarray(cycles, float),
                      'v': np.asarray(values, float)}).dropna()
    s = s.sort_values('c')
    if len(s) < 2:
        return np.nan, np.nan, np.nan, np.nan, np.nan, 0
    c = s['c'].to_numpy(); v = s['v'].to_numpy()
    end = len(c)
    for k in range(1, len(c)):
        # MISSING cycles, as the docstring says: a cycle-number jump of 3
        # is two missing cycles, and comparing the jump itself truncated the
        # coherent window one gap earlier than documented.
        if (c[k] - c[k - 1] - 1) >= gap_break:
            end = k
            break
    c, v = c[:end], v[:end]
    n = len(c)
    if n < min_points:
        return (np.nan, np.nan, np.nan,
                c[0] if n else np.nan, c[-1] if n else np.nan, n)
    m = max(1, min(edge_n, n // 2))
    v_start = float(np.median(v[:m]))
    v_end = float(np.median(v[-m:]))
    return v_end - v_start, v_start, v_end, c[0], c[-1], n


def _discontinuity_events(pk_df):
    """
    Detect discontinuity events for one (step, peak_id).

    pk_df: all rows for this peak (any status), with 'cycle',
    'status', 'centre', and 'reliable' columns.

    An event is EITHER:
      (a) a sustained dropout — a run of >= DISCONTINUITY_DROPOUT_RUN
          consecutive cycles that are not reliably tracked, which is
          preceded AND followed by at least one reliable tracked
          cycle (vanish-and-return mid-life); OR
      (b) a single-cycle centre step between consecutive reliable
          tracked cycles exceeding max(FLOOR, K x median step).

    Returns (n_events, first_event_cycle, max_centre_step_mV, established).
    """
    d = pk_df.sort_values('cycle')
    # WHERE THE RUN STOPPED FITTING IS NOT WHERE THE FEATURE FAILED.
    # `track_peaks` writes a `no_fit` row for every cycle past the fit limit,
    # for every peak, so a peak still perfectly healthy at the limit acquired a
    # terminal run of not-good cycles beginning at limit+1 and was published as
    # "failed mid-life ... this is a feature being lost". NNM cell A said "Cell
    # stopped delivering at cycle 128. Peak fitting stopped there" and, four
    # lines above it, listed two features as failing "from cycle 129"; cell B
    # listed three "from cycle 68" against a fit limit of 67. Those rows record
    # that nothing was ASKED of the model, so they are dropped from the end of
    # the record before any dropout is measured. A `no_fit` in the middle of a
    # record is a real failure to converge and is left exactly where it is.
    if 'status' in d.columns and len(d):
        _st = d['status'].astype(str).to_numpy()
        _end = len(_st)
        while _end > 0 and _st[_end - 1] in ('no_fit', 'not_fitted'):
            _end -= 1
        if _end == 0:
            return 0, np.nan, np.nan, True
        d = d.iloc[:_end]
    cycles = d['cycle'].to_numpy()
    # "good" = reliably tracked this cycle
    if 'reliable' in d.columns:
        good = ((d['status'] == 'tracked') & (d['reliable'])).to_numpy()
    else:
        good = (d['status'] == 'tracked').to_numpy()

    if good.sum() < 3:
        return 0, np.nan, np.nan, True

    event_cycles = []

    # --- (a) sustained dropouts bracketed by good tracking ---
    first_good = np.argmax(good)            # index of first reliable track
    run_len = 0
    run_start_idx = None
    # Scan from the first reliable track to the end. A run of
    # not-good cycles that begins AFTER the peak was once reliably
    # tracked is a sustained dropout — whether or not the peak ever
    # returns cleanly. The most severe failures never return cleanly,
    # so we do NOT require a reliable cycle after the run.
    for i in range(first_good, len(good)):
        if not good[i]:
            if run_len == 0:
                run_start_idx = i
            run_len += 1
        else:
            if run_len >= DISCONTINUITY_DROPOUT_RUN:
                event_cycles.append(int(cycles[run_start_idx]))
            run_len = 0
    # close out a run that extends to the end of life. A terminal
    # run (one reaching the final cycle) is held to a longer
    # threshold: a few lost cycles at the very end is the record
    # ending, not a feature failing. A mid-life dropout that simply
    # never recovers will still be long enough to clear this.
    if run_len >= DISCONTINUITY_TERMINAL_RUN:
        event_cycles.append(int(cycles[run_start_idx]))

    # --- (b) single-cycle centre steps on consecutive good cycles ---
    gd = d[good]
    gcyc = gd['cycle'].to_numpy()
    gcen = gd['centre'].to_numpy()
    steps_mV = np.full(len(gcen), np.nan)
    adjacent = np.diff(gcyc) == 1          # only consecutive cycles
    raw_steps = np.abs(np.diff(gcen)) * 1000.0
    steps_mV[1:] = np.where(adjacent, raw_steps, np.nan)
    _valid = steps_mV[1:][np.isfinite(steps_mV[1:])]
    if _valid.size >= 3:
        med = np.median(_valid)
        # robust spread: median absolute deviation, scaled to ~sigma
        mad = np.median(np.abs(_valid - med)) * 1.4826
        thresh = max(DISCONTINUITY_STEP_FLOOR_MV,
                     med + DISCONTINUITY_STEP_K * mad)
    else:
        thresh = DISCONTINUITY_STEP_FLOOR_MV
    for j in range(1, len(steps_mV)):
        if np.isfinite(steps_mV[j]) and steps_mV[j] > thresh:
            event_cycles.append(int(gcyc[j]))

    # np.nanmax over an all-NaN array warns and returns NaN; a peak tracked
    # only on non-consecutive cycles hits that on every dataset.
    max_step = (float(np.nanmax(steps_mV))
                if np.isfinite(steps_mV).any() else np.nan)
    if not event_cycles:
        return 0, np.nan, max_step, True  # established irrelevant; no event
    first_event = int(min(event_cycles))
    # Reliable cycles accrued before the first event: was the peak ever
    # established, or did it only flicker at the start of life?
    n_good_before = int(np.sum(good & (cycles < first_event)))
    established = n_good_before >= DISCONTINUITY_ESTABLISH_MIN
    return len(event_cycles), first_event, max_step, established


def _auto_pair_peaks(charge_ref_peaks, discharge_ref_peaks,
                      max_sep_mV=PAIR_MAX_SEPARATION_MV,
                      is_anode=False):
    """
    Pair charge and discharge reference peaks by voltage proximity.

    Uses globally optimal one-to-one assignment (Hungarian algorithm),
    minimising the total charge-discharge separation across all pairs at
    once, subject to two constraints: a pairing is allowed only within
    max_sep_mV, and it must give a ΔV of the sign expected for the
    electrode. For a cathode the charge peak sits above the discharge peak
    so ΔV is non-negative; for an anode (is_anode=True) the labels are
    swapped upstream so the delithiation "Discharge" peak sits above the
    lithiation "Charge" peak, so the expected ΔV is negative and the sign
    test is flipped.

    A per-peak nearest-neighbour scheme is fragile when two couples sit
    close: one charge peak takes the other's natural partner, forcing a
    crossed assignment and a wrong-sign ΔV. Global assignment with the
    expected-sign constraint avoids this.

    All peaks participate (primary and shoulder). Returns list of
    (charge_id, discharge_id, charge_V, discharge_V, pair_is_shoulder).
    """
    if charge_ref_peaks.empty or discharge_ref_peaks.empty:
        return []

    from scipy.optimize import linear_sum_assignment

    max_sep_V = max_sep_mV / 1000.0
    neg_tol_V = 0.005   # tolerate tiny negative from reference fit noise

    charge_sorted = charge_ref_peaks.sort_values('voltage').reset_index(drop=True)
    discharge_list = discharge_ref_peaks.reset_index(drop=True)

    n_c, n_d = len(charge_sorted), len(discharge_list)
    BIG = 1.0e6
    cost = np.full((n_c, n_d), BIG)
    for i in range(n_c):
        cv = charge_sorted.loc[i, 'voltage']
        for j in range(n_d):
            dv = discharge_list.loc[j, 'voltage']
            sep = abs(cv - dv)
            # Expected-sign test, electrode-aware. For a cathode, charge
            # sits at or above discharge (ΔV = cv - dv >= 0). For an anode
            # the labels were swapped upstream so "Discharge" is the
            # delithiation peak, which sits ABOVE the lithiation "Charge"
            # peak, so the expected ΔV is negative and the test flips.
            if is_anode:
                sign_ok = cv <= dv + neg_tol_V
            else:
                sign_ok = dv <= cv + neg_tol_V
            if sep <= max_sep_V and sign_ok:
                cost[i, j] = sep

    row_idx, col_idx = linear_sum_assignment(cost)

    pairs = []
    for i, j in zip(row_idx, col_idx):
        if cost[i, j] >= BIG:
            continue   # too far apart, or would give negative ΔV
        cpk = charge_sorted.loc[i]
        dpk = discharge_list.loc[j]
        either_shoulder = (bool(cpk.get('is_shoulder', False)) or
                           bool(dpk.get('is_shoulder', False)))
        pairs.append((
            int(cpk['peak_id']), int(dpk['peak_id']),
            cpk['voltage'], dpk['voltage'],
            either_shoulder
        ))

    return pairs


# WHICH COMPONENTS COUNT AS CAPACITY.
#
# `band_at_width_bound` was half-wired: it demoted a component from `reliable`
# and its area then went on entering `total_area`, `retained_fraction` and
# every `peakN_fraction` regardless. On NMC 14 of 68 tracked components per
# cell were pinned bands being counted in shares they were too unreliable to
# appear in individually, and that is what produced the page's own warning
# about 570% area retention. A band at its width ceiling is not a composition
# window any more; it is a flat pedestal the fit is using to lift the curve,
# and a pedestal is baseline, not charge.
#
# `area_empty` joins it for the same reason from the other end: a component
# carrying a thousandth of its neighbour's charge contributes nothing but
# noise to a sum.
#
# The excluded areas are not deleted — `total_area_all` reports the sum with
# them in, and `n_excluded_from_area` says how many there were, so the
# difference between the two numbers is visible rather than silent.
# WHICH AREA IS THE CAPACITY. Switched in 1.9.0.60.
#
# Every component carries two areas and until now every capacity number used
# the wrong one. `amplitude_area` is the ANALYTIC integral of the fitted
# shape over +/- infinity; `area_in_window` is the numerical integral over the
# voltage range actually measured. They are the same thing for a peak that
# sits well inside the window and they are not the same thing at all for one
# that does not.
#
# The half-cycle level already used the in-window integral —
# `component_area_sum`, `unattributed_area` and `unattributed_fraction` in
# `fitting` are all `trapezoid` over the curve — so the per-component areas
# did not sum to the quantity the closure test checks them against. Measured
# over the current runs, sum(analytic) / sum(in-window) per half-cycle:
# LTO 1.010-1.012, NMC 1.061-1.083, NNM 1.169-1.222.
#
# WHAT CHANGES. Each component's share of its own half-cycle moves by:
#
#     LTO   median 0.00 pp   p95 0.00 pp   max 0.0 pp
#     NMC   median 0.7 pp    p95 4.1 pp    max 6.7 pp
#     NNM   median 1.8 pp    p95 8.0 pp    max 10.8 pp
#
# Nothing at all on LTO, because there the over-count is uniform (every
# component 0.99) and a uniform factor cancels out of a share. What moves is
# exactly what should move: the per-component ratio falls to 0.472 at worst,
# a component reporting more than twice the charge it accounts for inside the
# measured range, and those are the edge components whose analytic tail is an
# extrapolation rather than a measurement.
#
# The analytic value stays in the table. It is the better estimate of a
# process's TOTAL charge for a peak that is fully contained, and the two
# columns side by side are how a reader sees the difference. What it stops
# being is the number the capacity attribution, the tracked areas and the
# coherence audit's area test are computed from.
CAPACITY_AREA_COLUMN = "area_in_window"


def _area(df):
    """The area column the capacity numbers are computed from.

    Falls back to `amplitude_area` where `area_in_window` is absent, so a
    table written by an older build still reads.
    """
    if CAPACITY_AREA_COLUMN in getattr(df, "columns", ()):
        _a = pd.to_numeric(df[CAPACITY_AREA_COLUMN], errors="coerce")
        if _a.notna().any():
            return _a.abs()
    return pd.to_numeric(df["amplitude_area"], errors="coerce").abs()


ATTRIBUTION_EXCLUDES = ("band_at_width_bound", "area_empty")


def _countable_area(cd):
    """Boolean mask: components whose area may be counted as capacity."""
    keep = pd.Series(True, index=cd.index)
    for col in ATTRIBUTION_EXCLUDES:
        if col in cd.columns:
            keep &= ~cd[col].fillna(False).astype(bool)
    return keep


def _attribution_basis(step_tracked, ref_cycle):
    """
    Fixed normalisation basis for capacity attribution: the summed fitted
    area of every peak tracked at the reference cycle.

    Using the reference cycle — the same baseline Module 5 uses for drift
    and retention — keeps attribution consistent with those quantities and
    makes the basis predictable and defensible across datasets rather than
    chosen by a heuristic. By construction the stack is 100% at the
    reference cycle and falls below it as capacity is lost. A feature
    present in the reference cycle that then fades shows its band
    collapsing, which is reported, not excluded.

    Returns (basis_area, basis_cycle).
    """
    cd = step_tracked[step_tracked['cycle'] == ref_cycle]
    if cd.empty:
        # Reference cycle has no tracked peaks for this step (unusual):
        # fall back to the earliest tracked cycle so a basis still exists.
        cycles = sorted(step_tracked['cycle'].unique())
        if not cycles:
            return np.nan, None
        ref_cycle = cycles[0]
        cd = step_tracked[step_tracked['cycle'] == ref_cycle]
    basis_area = float(_area(cd.loc[_countable_area(cd)]).sum())
    return basis_area, ref_cycle


# =============================================================================
# 1. FROM FIT RESULTS TO A COMPONENT TABLE
# =============================================================================

_PARAM_COLS = ("centre", "centre_stderr", "amplitude_area", "amplitude_stderr",
               "sigma", "sigma_stderr", "fraction", "fraction_stderr",
               "fwhm", "height", "area_in_window",
               # The solid-solution band: its width, and whether the fit
               # decided this component is a band at all or a plain peak.
               # `band_at_width_bound` is the band's equivalent of a sigma
               # pinned at its ceiling — a component acting as baseline.
               "band_width", "band_width_stderr", "component_kind",
               # THE CEILING THE FLAG WAS MEASURED AGAINST, not just the flag.
               # `band_at_width_bound` was in the table from 1.9.0.5x and the
               # ceiling it compares to was not, so a reader could see that a
               # component was pinned but not what it was pinned at — and the
               # ceiling is NOT one number. A component promoted inside a
               # dataset region is capped at that region's span; one seeded
               # from the residual gets the mechanism's own value. On the NNM
               # triplicate those are 0.400 V and 0.500 V, and the missing
               # column is why the pile-up at 0.400 went unseen for nine
               # builds. See `_band_ceilings`.
               "band_width_max_fitted",
               "band_at_width_bound",
               # eta pinned at 0 or 1 — the mixing fraction's equivalent of
               # the two flags either side of it. See `fraction_at_bound` in
               # `fitting`: a pure Lorentzian's wings carry area far from the
               # centre, and on LTO that shows up directly as the model
               # over-covering the cell's charge.
               "fraction_at_bound",
               # THE SPLIT LINESHAPE. `sigma` is the low-side half-width at
               # half maximum and `sigma_r` the high side, so `fwhm` is their
               # sum and not `2 * sigma`. Both are present on every fit:
               # a symmetric one reports `sigma_r == sigma` and
               # `asymmetry == 1.0`, so the column always means the same
               # thing and nothing downstream needs to know which model ran.
               "sigma_r", "sigma_r_stderr", "asymmetry")

# Computed in `parameters_frame` rather than read from the fit result, so
# they are listed separately — but they must reach the exported table and the
# tracking table, which is what `is_inferred` failed to do for four builds.
# THIS LIST HAD DUPLICATES AND A BROKEN INDENT — three names appeared twice
# from two separate edits that each appended rather than merging. Harmless in
# effect (the consumers deduplicate) but it is the visible edge of the bug
# that HAS bitten: a flag computed here and never added to a list like this
# one silently never reaches the CSV. `is_inferred` was missing for four
# builds; `fraction_at_bound` and `mechanism_own` each went missing once.
# Sorted and unique so an addition is obvious and a duplicate is not.
_DERIVED_FLAG_COLS = ("area_determinacy",
                      "area_empty",
                      "area_poorly_determined",
                      "area_stderr_missing",
                      "area_stderr_ratio",
                      "at_sigma_floor",
                      "at_sigma_max",
                      "reliability_reason",
                      "sigma_floor_side",
                      # NEW IN 1.9.0.58, and the reason the duplicates above
                      # were noticed: which flank reached the width ceiling.
                      "sigma_max_side",
                      "width_undersampled")

# --- when is a peak truncated? -------------------------------------------
# Detection answers this from GEOMETRY, before fitting: it compares the two
# half-maximum half-widths and calls a peak truncated when the clipped side is
# short. That is an estimate of a quantity the fit measures directly, and it
# is made from the noisiest part of the curve — a half-maximum crossing on a
# flank that is partly outside the window.
#
# The fit answers it properly. A component is truncated when its own
# centre +/- TRUNCATION_SIGMAS * sigma is not wholly inside the window that
# was fitted. Two sigma covers 95% of a Gaussian, so a peak that passes has at
# most a few per cent of its area outside and its integral is a measurement;
# one that fails has an area that is a LOWER BOUND, and the column says so
# rather than the peak being dropped.
#
# The corroborating flag is free and independent: a truncated peak's centre is
# poorly determined because half the information that fixes it is missing, so
# `centre_stderr / sigma` rises. Above 0.1 the centre is uncertain at the
# 10%-of-width level, which for a 60 mV FWHM feature is +/- 3 mV.
TRUNCATION_SIGMAS = 2.0
TRUNCATION_CENTRE_STDERR_RATIO = 0.10


def _anomalous_half_cycles(P, integrity):
    """
    Per-component: was this component's half-cycle banded ANOMALOUS?

    `integrity` is `cell_integrity`'s table — one row per half-cycle, with
    `cycle`, `step` and `band`. Absent (a cycling-only run, or an older
    caller) it returns all-False rather than raising: a flag that cannot be
    evaluated must not silently condemn everything, and the column has to
    exist either way so nothing downstream has to ask whether this build
    recorded it.

    ANOMALOUS ONLY, not `suspect`. The band ladder is deliberate: `suspect`
    means something was seen and could not be explained, `ANOMALOUS` means the
    half-cycle passed more charge than the material can hold or lost charge to
    nothing. The first is a caveat; only the second says the curve is not a
    measurement of the material.
    """
    _false = pd.Series(False, index=P.index)
    if integrity is None or getattr(integrity, "empty", True):
        return _false
    if not {"cycle", "step", "band"} <= set(integrity.columns):
        return _false
    bad = {(int(c), str(s)) for c, s, b in zip(
        integrity["cycle"], integrity["step"], integrity["band"])
        if str(b) == "ANOMALOUS" and pd.notna(c)}
    if not bad:
        return _false
    return pd.Series(
        [(int(c), str(s)) in bad for c, s in zip(P["cycle"], P["step"])],
        index=P.index)


def parameters_frame(fit_results, detection, *, name=None,
                     sigma_max_mV=FIT_SIGMA_MAX_MV,
                     width_floor_factor=FIT_WIDTH_FLOOR_FACTOR,
                     min_area_fraction=FIT_MIN_AREA_FRACTION,
                     integrity=None):
    """
    The long component table, one row per fitted component.

    `fit_results` is what `fitting.fit_many` returns: a list of plain dicts,
    each carrying `key=(cycle, step)`. The seeded centres came from
    `detection.peaks[key]`, in that order, so component `peak_index` i is the
    fit of detected peak i and inherits its `is_shoulder` / `is_truncated` /
    `detected_voltage`.

    The reliability columns are 1.8.7's, in 1.8.7's order: `_flag_reliable_peaks`
    first, then the two RELATIVE floors — width and area, each measured against
    this dataset's own median reliable component rather than an absolute bound,
    because an absolute one cannot serve a sharp LTO profile and a broad
    layered-oxide profile at once. Both are recorded in their own columns so the
    exclusion is auditable rather than silent.
    """
    rows = []
    for fr in fit_results:
        key = fr.get("key")
        if key is None:
            continue
        cycle, step = key
        det_pk = detection.peaks.get((cycle, step))
        for comp in fr.get("components", []):
            i = comp["peak_index"]
            # Join on the SEED index — the component's position in the
            # detected peak list it started from. `peak_index` is renumbered
            # when a degenerate component is dropped and the half-cycle
            # refitted, so joining on it gave every surviving component the
            # detected voltage, shoulder flag and truncation flag of the one
            # below it, with nothing to say so.
            sx = comp.get("seed_index", i)
            d = {}
            if det_pk is not None and sx < len(det_pk):
                d = det_pk.iloc[sx]
            row = dict(
                dataset=name if name is not None else detection.name,
                cycle=int(cycle), step=str(step),
                fit_success=bool(fr.get("success")),
                # `success` is "the fit ran without raising". CONVERGENCE is
                # a different question and it is the one that matters here:
                # `fit_half_cycle` records `res.success` separately, and
                # until 1.9.0.36 nothing but a console tally read it, so a
                # half-cycle that ran to the 20,000-evaluation ceiling
                # reached the parameter table indistinguishable from one
                # that settled. On NMC111 that was 87% of them, and those
                # same components then fed drift, area retention and
                # capacity attribution. Carried here so the reliability
                # test can act on it.
                converged=bool(fr.get("converged", True)),
                hit_iteration_cap=bool(fr.get("hit_iteration_cap", False)),
                # DID A PARAMETER FINISH ON A BOUND, AND DID MOVING IT OFF
                # HELP? A diagnostic computed in the fitter and consumed by
                # nothing is the pattern this project has now caught five
                # times, so it travels to the table with the rest.
                # See `fitting.BOUND_ESCAPE_RESTART`.
                bound_escape_tried=int(fr.get("bound_escape_tried", 0) or 0),
                bound_escape_used=bool(fr.get("bound_escape_used", False)),
                nfev=int(fr.get("nfev", 0) or 0),
                r_squared=fr.get("r_squared", np.nan),
                reduced_chi_sq=fr.get("redchi", np.nan),
                # THE FOUR NUMBERS R2 HIDES. Per half-cycle, repeated on each
                # of its component rows exactly as `r_squared` is, so the
                # parameter table can be read without joining anything.
                height_ratio=fr.get("height_ratio", np.nan),
                overshoot=fr.get("overshoot", np.nan),
                max_residual_frac=fr.get("max_residual_frac", np.nan),
                residual_runs_z=fr.get("residual_runs_z", np.nan),
                # The half-cycle's one shared asymmetry ratio, and whether it
                # settled on a bound. There is only one of these per
                # half-cycle, so nothing else would notice it pinning.
                asymmetry_k=fr.get("asymmetry_k", np.nan),
                asymmetry_k_fitted=fr.get("asymmetry_k_fitted", np.nan),
                asymmetry_k_clipped=bool(fr.get("asymmetry_k_clipped", False)),
                asymmetry_at_bound=bool(fr.get("asymmetry_at_bound", False)),
                # A per-CYCLE detection index, not a cross-cycle identity:
                # it shifts whenever detection finds a different number of
                # peaks. `tracked_peak_id` is the one that means what
                # `reference_peak_id` invited a reader to assume.
                component_index=int(sx) + 1,
                # The width bound this half-cycle was actually fitted under,
                # so the reliability test can ask whether a component is
                # pinned at ITS OWN bound rather than at a module constant.
                sigma_max_fitted=float(fr.get("sigma_max", np.nan)),
                # ...and the LOWER bound, and the curve's own sampling
                # interval. A width sitting on its floor and a width read
                # from three points are both statements about the
                # MEASUREMENT rather than about the peak, and neither was
                # available downstream. See `at_sigma_floor` and
                # `width_undersampled`.
                sigma_min_fitted=float(fr.get("sigma_min", np.nan)),
                sample_mV=float(fr.get("sample_mV", np.nan)),
                dropped_degenerate=int(fr.get("dropped_degenerate", 0)),
                # HOW MUCH DATA THIS FIT HAD, AND HOW MUCH OF IT IT SPENT.
                # `fit_one` has computed `n_points`, `nvarys` and
                # `n_shoulders_coupled` since the histogram rewrite and not one
                # of the three reached a file, a figure or a sentence: the only
                # consumer of the per-fit dictionary is a hard-coded five-column
                # summary used to pick which half-cycles to draw. The comment
                # beside `n_shoulders_coupled` in `fitting.fit_one` says it
                # exactly — "`nvarys` alone cannot be read without them ... the
                # difference is the whole argument for believing the standard
                # errors" — and then the argument was never put. A fit of 60
                # bins with 14 varied parameters and one of 600 with the same 14
                # are not the same measurement, and every standard error, every
                # `*_poorly_determined` flag and every reliability verdict on
                # this page rests on which of the two it was.
                n_points=int(fr.get("n_points", 0) or 0),
                nvarys=int(fr.get("nvarys", 0) or 0),
                n_shoulders_coupled=int(fr.get("n_shoulders_coupled", 0) or 0),
                points_per_parameter=(
                    float(fr.get("n_points", 0) or 0) / float(fr["nvarys"])
                    if fr.get("nvarys") else np.nan),
                # WHERE THE COMPONENTS CAME FROM. A band added by the residual
                # sweep is not a detected feature, and until now the table gave
                # no way to tell one from the other at half-cycle level.
                bands_from_residual=int(fr.get("bands_from_residual", 0) or 0),
                # HOW MUCH OF THE CURVE THE BASELINE TOOK. `baseline_area` and
                # `curve_area` were both computed and neither published, so the
                # one question a reader of an attributed area always asks —
                # how much of this half-cycle did the model call background? —
                # had no answer anywhere in the run. As a fraction, because the
                # absolute is only meaningful beside a curve area nobody sees.
                baseline_area_fraction=(
                    float(fr["baseline_area"]) / float(fr["curve_area"])
                    if (np.isfinite(fr.get("baseline_area", np.nan))
                        and np.isfinite(fr.get("curve_area", np.nan))
                        and fr.get("curve_area")) else np.nan),
                unattributed_fraction_fit=float(
                    fr.get("unattributed_fraction", np.nan)),
                detected_voltage=float(d["voltage"]) if len(d) else np.nan,
                is_shoulder=bool(d["is_shoulder"]) if len(d) else False,
                # SEEDED BY RECURRENCE, not by this half-cycle's own peak
                # picker: a feature the run finds at the same potential cycle
                # after cycle, whose prominence here is below every gate.
                # Provenance matters — see `detect.recurrent_maxima`.
                is_recurrent=bool(d.get("is_recurrent", False)) if len(d)
                else False,
                # The DETECTION-STAGE geometric guess, kept because it is
                # what seeded the fit. `fit_truncated` below is the answer.
                is_truncated=bool(d["is_truncated"]) if len(d) else False,
                window_v_min=float(fr.get("v_min", np.nan)),
                window_v_max=float(fr.get("v_max", np.nan)),
                # A component seeded from a NEIGHBOURING cycle's peak list
                # because this cycle's detection found nothing there. Set by
                # `infer_missing_seeds`; the provenance matters, because a
                # component that was never detected in the cycle it is
                # reported for is a different kind of number from one that
                # was.
                is_inferred=bool(d.get("is_inferred", False)) if len(d)
                else False,
            )
            for c in _PARAM_COLS:
                row[c] = comp.get(c, np.nan)
            rows.append(row)

    P = pd.DataFrame(rows)
    if P.empty:
        return P

    # --- the truncation test, from the fit rather than from geometry ----
    # EACH FLANK MEASURED WITH ITS OWN WIDTH. `sigma` is the LOW-side
    # half-width of a split lineshape and `sigma_r` the high side; both the
    # window test and the outside-area integral used `sigma` on both sides. On
    # an LTO terminal component with sigma 2.3 mV and sigma_r 30 mV that puts
    # the modelled high flank 65 mV inside the window when it is in fact 30 mV
    # outside it: `fit_truncated` False, `area_outside_window_frac` 0.000, and
    # roughly 5% of the component's area published as a measurement rather than
    # as a lower bound. `_flag_reliable_peaks` and the sigma-bound test were
    # both corrected to read both flanks in earlier builds; this block was
    # missed. Where there is no `sigma_r` the shape is symmetric and the two
    # widths are the same number, so single-flank fits are unaffected.
    _sig_l = P["sigma"].abs()
    if "sigma_r" in P.columns:
        _sig_r = pd.to_numeric(P["sigma_r"], errors="coerce").abs()
        _sig_r = _sig_r.where(_sig_r.notna() & (_sig_r > 0), _sig_l)
    else:
        _sig_r = _sig_l
    _lo = P["centre"] - TRUNCATION_SIGMAS * _sig_l
    _hi = P["centre"] + TRUNCATION_SIGMAS * _sig_r
    P["fit_truncated"] = ((_lo < P["window_v_min"]) |
                          (_hi > P["window_v_max"])).fillna(False)
    # How much of the modelled peak fell outside the window, as a fraction of
    # its analytic area. This is the number to quote when an area is a lower
    # bound — "truncated" alone does not say whether it matters.
    #
    # For a split normal the two halves are not equally weighted: the density
    # that is continuous at the centre and integrates to one puts
    # sigma_l/(sigma_l+sigma_r) of the area below the centre. The enclosed
    # fraction between the window edges is therefore
    #
    #     2/(s_l+s_r) * [ s_l*(PHI(z_hi^-) - PHI(z_lo^-))
    #                   + s_r*(PHI(z_hi^+) - PHI(z_lo^+)) ]
    #
    # with ^- clipped at or below zero and ^+ at or above it, which reduces to
    # the symmetric expression when s_l == s_r.
    _z_lo = ((P["window_v_min"] - P["centre"]) / _sig_l).astype(float)
    _z_hi = ((P["window_v_max"] - P["centre"]) / _sig_r).astype(float)
    # The same two edges expressed in the OTHER flank's width, needed where an
    # edge falls on the far side of the centre from the flank it belongs to.
    _z_lo_r = ((P["window_v_min"] - P["centre"]) / _sig_r).astype(float)
    _z_hi_l = ((P["window_v_max"] - P["centre"]) / _sig_l).astype(float)
    _phi = lambda z: 0.5 * (1.0 + erf(z / np.sqrt(2.0)))      # noqa: E731
    _w = (_sig_l + _sig_r).replace(0.0, np.nan)
    _below = _sig_l * (_phi(np.minimum(_z_hi_l, 0.0))
                       - _phi(np.minimum(_z_lo, 0.0)))
    _above = _sig_r * (_phi(np.maximum(_z_hi, 0.0))
                       - _phi(np.maximum(_z_lo_r, 0.0)))
    _inside = (2.0 / _w) * (_below + _above)
    P["area_outside_window_frac"] = (1.0 - _inside).clip(
        lower=0.0, upper=1.0)
    P["area_is_lower_bound"] = P["fit_truncated"]
    P["centre_stderr_ratio"] = (P["centre_stderr"] / P["sigma"].abs())
    P["centre_poorly_determined"] = (
        P["centre_stderr_ratio"] > TRUNCATION_CENTRE_STDERR_RATIO).fillna(False)

    P["reliable"] = _flag_reliable_peaks(P, sigma_max_mV=sigma_max_mV).values
    # A component that was never DETECTED in the cycle it is reported for is
    # kept in the table and tracked, but is not evidence: it is excluded from
    # every headline trend by the one flag those trends already read. See
    # `detect.INFER_MISSING_SEEDS`.
    P["reliable"] = P["reliable"] & ~P["is_inferred"].fillna(False)

    # A COMPONENT OF A HALF-CYCLE THAT PASSED MORE CHARGE THAN THE MATERIAL
    # CAN HOLD IS NOT EVIDENCE EITHER.
    #
    # `cell_integrity` bands such a half-cycle ANOMALOUS and the per-cell page
    # says so loudly. Nothing downstream asked. `not_measurements` — the set
    # the fit skips — is in-progress plus early terminations; the anomalous
    # band was not in it, so those half-cycles were fitted, tracked, and their
    # areas went into the retention trends and the capacity attribution beside
    # clean ones. Measured on Jiaqi's NNM, summed fitted component area per
    # charge half-cycle: cell A clean median 96.0 against 234.9 on its nine
    # anomalous half-cycles (2.45x), cell C 3-4x.
    #
    # AND THE EXISTING GUARDS RATE THEM AS GOOD. `integral_fidelity` asks
    # whether the curve accounts for the charge the cell delivered, and it
    # does — faithfully, because the charge really was delivered: on cell A
    # the anomalous half-cycles score 0.957 against a clean median of 0.928.
    # `reliable` fired on 0.43 of their components against 0.50 on clean ones.
    # No separation, from either. The integral is right; what it is an
    # integral OF is not. Cell C cycle 28 charge carries 1040 records against
    # ~350, 102 voltage reversals against 3, and 143 mAh/g of plateau charge —
    # the cell traversed the same voltages a hundred times, and a differential
    # capacity curve built from that is not a measurement of the material
    # however well it integrates.
    #
    # NOTHING IS DELETED, exactly as for `is_inferred`: the components stay in
    # the table, stay tracked, and stay in every figure. They are marked, and
    # the one flag the headline trends read excludes them.
    P["half_cycle_anomalous"] = _anomalous_half_cycles(P, integrity)
    P["reliable"] = P["reliable"] & ~P["half_cycle_anomalous"]
    # THE CONVERGENCE GATE IS GONE. It was added here by the reporting audit
    # on an argument that was right — do not quote numbers from a fit that did
    # not work — applied to a quantity that could not carry it. `converged` is
    # lmfit's exit status, and on NNM it was exactly `not hit_iteration_cap`:
    # it demoted 405 of 405 and 462 of 462 components whose fits had the
    # HIGHER median R2, because running out of evaluations while still
    # descending is not a failure. See AREA_STDERR_RATIO_MAX above for what
    # replaced it and why that is flagged rather than enforced.
    #
    # `converged` and `hit_iteration_cap` are still measured, still in the
    # parameter table, and still printed by Cell 9. They are facts about the
    # optimiser and they are reported as such; they no longer decide what
    # counts as a measurement.

    # Is the AREA determined? Flagged, not enforced — see the constant.
    _se = pd.to_numeric(P.get("amplitude_stderr"), errors="coerce") \
        if "amplitude_stderr" in P else pd.Series(np.nan, index=P.index)
    _ar = pd.to_numeric(P.get("amplitude_area"), errors="coerce").abs()
    with np.errstate(divide="ignore", invalid="ignore"):
        _ratio = _se / _ar.where(_ar > 0)
    P["area_stderr_ratio"] = _ratio
    # A MISSING standard error is not a small one. lmfit leaves it None when
    # the covariance could not be estimated, which is the plainest statement
    # available that the data did not determine this parameter.
    P["area_stderr_missing"] = _se.isna() & _ar.notna()
    P["area_poorly_determined"] = (
        (_ratio > AREA_STDERR_RATIO_MAX).fillna(False)
        | P["area_stderr_missing"])
    if ENFORCE_AREA_DETERMINACY:
        P["reliable"] = P["reliable"] & ~P["area_poorly_determined"]
    # BOTH PLACES OR NEITHER. `track_peaks` builds its own table and recomputes
    # `reliable` from scratch, so a demotion applied only here reaches the
    # parameter CSV and nothing else — the trap this module's own
    # SIGMA_BOUND_PROXIMITY note warns about, walked into twice already. The
    # matching clause is in `_reliable`.
    if ENFORCE_AREA_COMPLETENESS and "area_outside_window_frac" in P:
        P["reliable"] = P["reliable"] & ~(
            P["area_is_lower_bound"].fillna(False).astype(bool)
            & (pd.to_numeric(P["area_outside_window_frac"], errors="coerce")
               > AREA_LOWER_BOUND_REPORTABLE).fillna(False))
    # A band that has run to its width ceiling is a pedestal, not a
    # composition window — the same failure as a component at its sigma
    # bound, and demoted the same way.
    if "band_at_width_bound" in P:
        P["reliable"] = P["reliable"] & ~(
            P["band_at_width_bound"].fillna(False).astype(bool))
    P["too_narrow"] = False
    P["area_collapsed"] = False
    base = P["reliable"]
    if int(base.sum()) >= 20:
        typ_s = P.loc[base, "sigma"].median()
        # `width_floor_factor` is the MULTIPLE below the median width that
        # counts as narrow, so 4.0 means "narrower than a quarter of the
        # median" — the same test the fallback below always applied. The
        # previous reading inverted it (a plausible-looking 0.25 flagged
        # everything narrower than 4x the median, i.e. nearly everything)
        # AND, unlike the flag-only fallback, struck them from `reliable`.
        # Both branches now flag only; enforcement is a separate decision.
        _wf = width_floor_factor if width_floor_factor > 0 else 4.0
        if np.isfinite(typ_s):
            P["too_narrow"] = base & (P["sigma"] * _wf < typ_s)
        typ_a = P.loc[base, "amplitude_area"].median()
        if np.isfinite(typ_a) and min_area_fraction > 0:
            collapsed = P["amplitude_area"].abs() < typ_a * min_area_fraction
            P["area_collapsed"] = base & collapsed
            P["reliable"] = P["reliable"] & ~collapsed
    # --- two statements about the MEASUREMENT, not about the peak ---------
    #
    # `_flag_reliable_peaks` tests only the UPPER sigma bound, and its
    # docstring gives the reason: a genuine sharp peak sits near the lower
    # one by design, so demoting on it would delete real data. That reasoning
    # is right and it is not a reason to stay SILENT. A component exactly on
    # its width floor has a width of "one histogram bin or less" — the number
    # printed with a standard error beside it is the bound, not a fit — and
    # on LTO 43-60% of components sat there while every one of them was
    # marked reliable.
    #
    # Flagged, not enforced, like `too_narrow` and `band_at_width_bound`.
    # BOTH FLANKS. This tested `sigma` alone, which is right for a symmetric
    # fit and wrong for a split one: on LTO the low-voltage flank sits on the
    # floor in the discharge direction and the HIGH-voltage flank does on the
    # charge direction. 44 of 61 components had a flank on the floor and 26
    # were flagged; the 19 charge cases were silent. Item 32.
    P["at_sigma_floor"] = False
    if "sigma_min_fitted" in P:
        _floor = pd.to_numeric(P["sigma_min_fitted"], errors="coerce")
        _lo = (pd.to_numeric(P["sigma"], errors="coerce")
               <= _floor * (1.0 + SIGMA_BOUND_PROXIMITY)).fillna(False)
        _hi = pd.Series(False, index=P.index)
        if "sigma_r" in P:
            _hi = (pd.to_numeric(P["sigma_r"], errors="coerce")
                   <= _floor * (1.0 + SIGMA_BOUND_PROXIMITY)).fillna(False)
        P["at_sigma_floor"] = _lo | _hi
        # ...and WHICH flank, because they mean different things: the low side
        # on the floor is a leading edge steeper than the sampling, the high
        # side is a trailing edge.
        P["sigma_floor_side"] = np.where(
            _lo & _hi, "both", np.where(_lo, "low", np.where(_hi, "high", "")))
    # ITEM 36. A component at its UPPER width bound is acting as baseline and
    # `_flag_reliable_peaks` already demotes it — but nothing NAMED it, so a
    # reader saw `reliable = False` with no reason. 25 of 143 on NMC111 and
    # 48-86 per cell on NNM.
    #
    # BOTH FLANKS, 1.9.0.58 — the same correction `at_sigma_floor` above
    # received as Item 32, made on the ceiling instead of the floor, and made
    # now because per-component asymmetry is what gives the two flanks
    # different widths. Testing `sigma` alone was complete while the
    # lineshape was symmetric; with a split one it goes silent on exactly the
    # components this flag exists to catch. NNM's terminal component reaches
    # the ceiling on its HIGH flank, which is the flank that was not tested.
    P["at_sigma_max"] = False
    P["sigma_max_side"] = ""
    if "sigma_max_fitted" in P:
        _ceil = pd.to_numeric(P["sigma_max_fitted"], errors="coerce")
        _clo = (pd.to_numeric(P["sigma"], errors="coerce")
                >= _ceil * SIGMA_BOUND_PROXIMITY_COMPLEMENT).fillna(False)
        _chi = pd.Series(False, index=P.index)
        if "sigma_r" in P:
            _chi = (pd.to_numeric(P["sigma_r"], errors="coerce")
                    >= _ceil * SIGMA_BOUND_PROXIMITY_COMPLEMENT).fillna(False)
        P["at_sigma_max"] = _clo | _chi
        # WHICH flank, on the same reasoning as `sigma_floor_side`: the high
        # side at the ceiling is a trailing edge the window did not contain,
        # the low side is a leading one.
        P["sigma_max_side"] = np.where(
            _clo & _chi, "both",
            np.where(_clo, "low", np.where(_chi, "high", "")))
    # And the width that no sampling could have measured. LTO cell C's
    # discharge peak is 2.6 mV wide sampled at 1 mV — three points across the
    # full width at half maximum.
    P["width_undersampled"] = False
    if "sample_mV" in P and "fwhm" in P:
        _samp = pd.to_numeric(P["sample_mV"], errors="coerce") / 1000.0
        P["width_undersampled"] = (
            pd.to_numeric(P["fwhm"], errors="coerce")
            < _samp * WIDTH_MIN_SAMPLES).fillna(False)

    # ...and the per-half-cycle test, which runs whatever the above decided.
    # See EMPTY_COMPONENT_FRACTION.
    P["area_empty"] = False
    if EMPTY_COMPONENT_FRACTION > 0 and {"cycle", "step"} <= set(P.columns):
        _biggest = (P.groupby(["cycle", "step"])["amplitude_area"]
                    .transform(lambda a: a.abs().max()))
        _empty = (P["amplitude_area"].abs()
                  < _biggest * EMPTY_COMPONENT_FRACTION)
        _empty = _empty.fillna(False) & np.isfinite(_biggest) & (_biggest > 0)
        P["area_empty"] = _empty
        P["reliable"] = P["reliable"] & ~_empty

    # --- WHY a component is not reliable, and a separate axis for whether
    # --- its area is determined at all.
    #
    # `reliable` is one boolean carrying several different statements, and it
    # is the most-read field in the table. Measured across the three
    # chemistries, what actually sets it False is:
    #
    #                     NNM (1903 not reliable)   NMC (124)   LTO (1)
    #   band_at_width_bound        59.0%              45.2%        0%
    #   at_sigma_max               55.5%              30.6%      100%
    #   is_inferred                37.9%              46.8%        0%
    #   area_empty                  7.1%              14.5%        0%
    #
    # Those are not one statement. "This component sits on a bound" says the
    # width is a constraint rather than a fit — and, being knife-edge, is not
    # even reproducible across machines. "This component was never detected in
    # this half-cycle" says the centre is another cycle's answer. "The fit put
    # a thousandth of the charge here" says the feature is not there. A reader
    # given one `False` cannot tell which, and they call for different actions.
    #
    # AND THE DETERMINACY OF THE AREA IS A DIFFERENT AXIS AGAIN. It reaches
    # `reliable` only when ENFORCE_AREA_DETERMINACY is on, which it is not —
    # so `reliable = True` today says nothing whatever about whether the area
    # has a quotable uncertainty. On NNM 83% and on NMC 89% of components have
    # NO standard error on their area at all. A reader who takes `reliable`
    # to mean "I can quote this area with its error" is wrong about nearly
    # every component of a layered oxide, and nothing on the page said so.
    #
    # So: `reliable` keeps EXACTLY its present value — this changes what the
    # tool says, not what it does — and two new columns say what it means.
    _reason = pd.Series("ok", index=P.index, dtype=object)
    _B = lambda c: (P[c].fillna(False).astype(bool) if c in P
                    else pd.Series(False, index=P.index))
    # Named here rather than in the ladder so the threshold has one home and
    # the report can quote the same number. Flag only: this does not demote
    # anything, it says why something already demoted is not usable.
    P["area_mostly_outside_window"] = (
        _B("area_is_lower_bound")
        & (pd.to_numeric(P.get("area_outside_window_frac"), errors="coerce")
           > AREA_LOWER_BOUND_REPORTABLE).fillna(False))
    # Most specific first: a component that was never detected here is not a
    # measurement whatever else is true of it.
    for _flag, _word in (
            # FIRST: this one is a statement about the whole half-cycle, and
            # it survives whatever else is true of the component.
            ("half_cycle_anomalous",
             "this half-cycle passed more charge than the material can hold"),
            ("is_inferred", "not detected in this half-cycle"),
            ("area_empty", "the fit put almost no charge here"),
            ("area_collapsed", "area collapsed against the reference cycle"),
            ("band_at_width_bound", "band at its width ceiling"),
            # ABOVE `at_sigma_max`, DELIBERATELY. A component sitting at the
            # window edge runs BOTH flanks to the sigma bound because there
            # is no data on one side to stop it, so it earns both flags — and
            # naming the ceiling sends the reader to the width when the cause
            # is the window. On NNM 1.9.0.72, 214 of the 245 components whose
            # reason read "width at its ceiling" had more than a tenth of
            # their area outside the window; 81% of everything at the sigma
            # bound sits within one sigma_max of an edge, against 27% of the
            # dataset. Raising the bound does not help these — see
            # `quality.MECHANISM_MODEL` for why this is NOT the band-ceiling
            # problem wearing a different hat.
            ("area_mostly_outside_window",
             "most of this component lies outside the measured window"),
            ("at_sigma_max", "width at its ceiling"),
    ):
        _reason = _reason.mask((_reason == "ok") & _B(_flag) & ~P["reliable"],
                               _word)
    _reason = _reason.mask((_reason == "ok") & ~P["reliable"],
                           "width at its ceiling or area not finite")
    _reason = _reason.mask(P["reliable"], "ok")
    P["reliability_reason"] = _reason
    # Orthogonal to `reliable`, and printed beside it rather than folded in.
    P["area_determinacy"] = np.where(
        _B("area_stderr_missing"), "not estimated",
        np.where((P["area_stderr_ratio"] > AREA_STDERR_RATIO_MAX).fillna(False),
                 "imprecise", "determined"))
    return P


# =============================================================================
# 2. TRACKING  (Module 5)
# =============================================================================

@dataclass
class Tracking:
    """Peak histories for one dataset, and what became of each peak."""
    name: str
    tracked_df: pd.DataFrame = field(default_factory=pd.DataFrame)
    summary: dict = field(default_factory=dict)
    reference_peaks: dict = field(default_factory=dict)
    reference_cycle: int = 0
    ghost_peaks: set = field(default_factory=set)
    tolerance_mV: float = TRACKING_TOLERANCE_MV
    # The cell's own discharge-capacity retention, once `flag_area_growth` has
    # been given it. NaN until then, and NaN is not 100%.
    capacity_retention_pct: float = float("nan")
    # Median `quality.integral_fidelity` per step, once `flag_low_fidelity`
    # has been given it. See INTEGRAL_FIDELITY_FLOOR.
    integral_fidelity: dict = field(default_factory=dict)

    @property
    def cell_discontinuity(self):
        """Primary, non-ghost peaks that failed after being established."""
        return sum(1 for s in self.summary.values()
                   if not s["is_shoulder"] and not s["is_ghost"]
                   and s["discontinuity_is_failure"])

    @property
    def transient_losses(self):
        return sum(1 for s in self.summary.values()
                   if not s["is_shoulder"] and not s["is_ghost"]
                   and s["discontinuity_events"]
                   and not s["discontinuity_is_failure"])

    def summary_text(self):
        L = [entry("tracking against", f"cycle {self.reference_cycle}",
                   f"±{self.tolerance_mV:.0f} mV per cycle, position "
                   f"carried forward"
                   if TRACK_AGAINST_PREVIOUS_CYCLE
                   else f"±{self.tolerance_mV:.0f} mV of the reference")]
        if self.ghost_peaks:
            L.append("  ghost peaks (reference area ~ 0, retention not "
                     "reported): "
                     + ", ".join(f"{s} peak {p}"
                                 for s, p in sorted(self.ghost_peaks)))
        _far = [s for s in self.summary.values()
                if s.get("drift_exceeds_tolerance")]
        if _far:
            L.append(verdict("caution", f"{len(_far)} peak(s) jumped further "
                             f"in ONE cycle than the "
                             f"±{self.tolerance_mV:.0f} mV matching window"))
            L.append(bullet("A feature does not move that far between "
                            "consecutive cycles, so on either side of that "
                            "step the history is probably not one feature. "
                            "Total drift is not the test — the tracker "
                            "follows a peak as far as it goes — a single-"
                            "cycle jump is."))
        for _st, _f in sorted(self.integral_fidelity.items()):
            if _f < INTEGRAL_FIDELITY_FLOOR:
                L.append(f"  *** {_st}: the dQ/dV curve accounts for only "
                         f"{100 * _f:.0f}% of the delivered capacity. Peak "
                         f"AREAS on this step are not capacities and their "
                         f"trends are not retention; peak POSITIONS and "
                         f"drift are unaffected. ***")
            elif _f > INTEGRAL_FIDELITY_CEILING:
                L.append(f"  *** {_st}: the dQ/dV curve carries "
                         f"{100 * _f:.0f}% of the delivered capacity — more "
                         f"charge than the cell passed. Peak AREAS on this "
                         f"step include charge that was not measured; peak "
                         f"POSITIONS and drift are unaffected. ***")
        if np.isfinite(self.capacity_retention_pct):
            _bound = self.capacity_retention_pct + AREA_GROWTH_HEADROOM_PCT
            L.append(f"  capacity retention "
                     f"{self.capacity_retention_pct:.0f}% — a peak above "
                     f"{_bound:.0f}% area retention is flagged, not deleted")
        L.append(f"  {'step':<10}{'ref V':>8}{'tracked':>9}{'rate':>7}"
                 f"{'drift':>10}{'retention':>11}  note")
        for k in sorted(self.summary):
            s = self.summary[k]
            note = []
            if s["is_shoulder"]:
                note.append("shoulder")
            if s["is_truncated"]:
                note.append("truncated")
            if s["is_ghost"]:
                note.append("ghost")
            if s.get("drift_exceeds_tolerance"):
                note.append(f"JUMPED {s['max_centre_step_mV']:.0f} mV IN ONE "
                            f"CYCLE — BEYOND THE "
                            f"±{self.tolerance_mV:.0f} mV MATCHER")
            if s.get("area_retention_trustworthy") is False:
                note.append("AREA NOT A CAPACITY")
            if s.get("area_exceeds_capacity"):
                note.append(f"AREA > CAPACITY by "
                            f"{s['area_growth_excess_pct']:.0f} pts")
            if s["discontinuity_events"]:
                note.append(
                    f"{s['discontinuity_events']} discontinuit"
                    f"{'y' if s['discontinuity_events'] == 1 else 'ies'}"
                    + (" (FAILURE, established first)"
                       if s["discontinuity_is_failure"]
                       else " (transient, never established)"))
            drift = ("     —" if not np.isfinite(s["voltage_drift_mV"])
                     else f"{s['voltage_drift_mV']:+6.0f}")
            _pc = s.get("voltage_drift_mV_per_cycle", np.nan)
            if np.isfinite(_pc) and s.get("voltage_drift_first_cycle"):
                note.append(f"{_pc:+.2f} mV/cycle over c"
                            f"{s['voltage_drift_first_cycle']}-"
                            f"{s['voltage_drift_last_cycle']}")
            ret = ("      —" if not np.isfinite(s["area_retention_pct"])
                   else f"{s['area_retention_pct']:6.0f}%")
            L.append(f"  {s['step']:<10}{s['reference_voltage']:>8.3f}"
                     f"{s['tracked']:>9}{s['tracking_rate']:>7.0%}"
                     f"{drift:>8} mV{ret:>11}  {'; '.join(note)}")
        return "\n".join(L)


def _assign_components(params_df, ref_list, step_cycles, step, tolerance_V,
                       reference_cycle=None, chain=None):
    """
    One-to-one assignment of fitted components to reference peaks, per cycle.

    Ported from 1.8.7 Module 5. Each component is claimed once, by optimal
    assignment, with PRIMARIES matched before shoulders — so when a primary and
    its shoulder merge into one component it is the shoulder that is recorded
    as disappeared, and the primary's centre history stays continuous. The
    earlier per-peak nearest-neighbour scheme let two reference ids take the
    same component and orphan a real one.

    WHAT THE WINDOW IS MEASURED FROM (1.9.0.23)
    -------------------------------------------
    Until now every cycle was matched against the REFERENCE CYCLE's fitted
    positions, so `tolerance_V` was a LIFETIME allowance: a peak drifting
    2 mV/cycle used it up in forty cycles and was then matched to whichever
    component was nearest, which on a crowded curve is a different peak. That
    made the tolerance a ceiling on the drift the tracker could measure —
    which is the one quantity ICA is usually read for.

    The expectation now MOVES WITH THE PEAK: each reference id carries a
    running centre, initialised to its reference-cycle position and updated to
    the fitted centre every time it is matched. `tolerance_V` becomes a
    PER-CYCLE step limit — how far a feature may move between consecutive
    cycles and still be the same feature — which is what the number was always
    a sensible size for, and the lifetime drift is then unbounded.

    Two properties this has to keep, and does:

    * A peak that is MISSING for some cycles keeps its last matched position
      as the expectation rather than reverting to the reference. Reverting
      would make a gap into a jump and, on the far side of a long gap, hand
      the peak to a neighbour.
    * Matching walks OUTWARD FROM THE REFERENCE CYCLE in both directions,
      never from cycle 1. The reference is the measured end of formation, and
      the cycles before it are where the curve moves fastest; chaining
      forwards from cycle 1 would arrive at the reference having already
      wandered, and the reference cycle is the one position we are certain of.

    The cost of chaining is that an error made in one cycle is inherited by
    the next. That is bounded here by the per-cycle window and reported: a
    single-cycle step larger than the window is a discontinuity, is counted as
    one, and the summary says so.

    `chain=False` restores the fixed-reference behaviour, which is how the
    two are compared on the synthetic ground truth.
    """
    if chain is None:
        chain = TRACK_AGAINST_PREVIOUS_CYCLE
    assignment = {}

    ordered = list(step_cycles)
    if chain and reference_cycle is not None and ordered:
        # Outward from the reference, each direction starting fresh from the
        # reference-cycle positions.
        passes = [[c for c in ordered if c >= reference_cycle],
                  [c for c in ordered if c < reference_cycle][::-1]]
    else:
        passes = [ordered]

    for _pass in passes:
        # The running expectation. Fixed for the whole pass when chaining is
        # off, which reproduces the pre-1.9.0.23 assignment exactly.
        expect = {rid: rv for rid, rv, _sh in ref_list}
        for cyc in _pass:
            cps = params_df[(params_df["cycle"] == cyc) &
                            (params_df["step"] == step)]
            if cps.empty:
                continue
            centres = cps["centre"].values
            used = set()
            for shoulder_pass in (False, True):
                ids = [(rid, expect[rid])
                       for rid, _rv, sh in ref_list if sh == shoulder_pass]
                free = [ci for ci in range(len(centres)) if ci not in used]
                if not ids or not free:
                    continue
                BIG = 1e6
                cost = np.full((len(ids), len(free)), BIG)
                for i, (_rid, rv) in enumerate(ids):
                    for j, ci in enumerate(free):
                        d = abs(centres[ci] - rv)
                        if d <= tolerance_V:
                            cost[i, j] = d
                ri, cj = linear_sum_assignment(cost)
                for i, j in zip(ri, cj):
                    if cost[i, j] >= BIG:
                        continue
                    assignment[(cyc, ids[i][0])] = free[j]
                    used.add(free[j])
                    if chain:
                        # Carry the position forward. An unmatched id keeps
                        # the expectation it already had.
                        expect[ids[i][0]] = float(centres[free[j]])
    return assignment


# A tracked peak's area is a share of the charge the cell delivered. If the
# cell holds 99% of its capacity, no redox feature inside it can have grown to
# 136% — and LTO cell A reported exactly that at 1.9.0.8, from a shoulder that
# had inflated into its parent. R2 could not see it (0.999), the standard
# errors could not see it (they are conditional on the model), and nothing in
# the pipeline asked the one question thermodynamics answers for free.
#
# So ask it. The bound is capacity retention plus headroom, because the two
# are not the same quantity: capacity is the whole cell, a peak is one process
# in it, and a process can legitimately take a larger SHARE as another fades.
# The headroom is what separates that from a fitting artefact. 15 points is
# generous — a genuine redistribution of that size across ten cycles at C/10
# would itself be a result — and it still catches the 136%.
AREA_GROWTH_HEADROOM_PCT = 15.0

# --- when is a peak AREA a capacity at all? ------------------------------
# A fitted peak area is a share of the charge that appears in the dQ/dV
# CURVE, and that curve is only a measurement of the cell's charge to the
# extent that its integral recovers the cell's capacity. `integral_fidelity`
# is exactly that ratio, it has been measured on every half-cycle since
# 1.9.0, and until now it gated nothing.
#
# It should. On the LTO triplicate the CYCLER'S OWN dQ/dV column integrates
# to 43-68% of the CYCLER'S OWN delivered capacity, and the shortfall swings
# by ten points from cycle to cycle. That is not a fitting problem and no
# choice of model fixes it: on a flat two-phase plateau the true dV between
# consecutive records falls below the instrument's voltage resolution, so
# most of the charge is delivered in a voltage interval the derivative cannot
# resolve. Fit that curve as well as you like and its peak area is the area
# of the fraction that survived, which is why LTO cell A reports 141% "area
# retention" on a cell holding 98% of its capacity.
#
# So: below this floor, area-based TRENDS are marked untrustworthy and say
# why. The areas themselves are kept — they are still the best description of
# the curve that was measured — and the peak POSITIONS are unaffected, which
# matters, because voltage drift is the quantity ICA is usually read for and
# it does not depend on the integral at all.
#
# 0.80 is the floor: published ICA practice treats the differential as an
# accounting of the capacity, and a curve missing a fifth of the charge is
# not that. A cell with a constant-voltage hold sits legitimately below 1.0,
# since CV charge passes at one voltage and cannot appear in a dV integral —
# which is a reason to read the floor as "areas are not capacity here",
# not as "the data are bad".
INTEGRAL_FIDELITY_FLOOR = 0.80

# ...AND A CEILING (1.9.0.23). `quality.integral_fidelity`'s own docstring
# says "fabricated area shows up as an excess, lost features as a deficit",
# and until now only the deficit was tested. The failure it names is the more
# damning of the two: a deficit is charge the curve never saw, an excess is
# charge the cell never delivered, and there is no benign mechanism that
# invents a fifth of a cell's capacity. The 4.199 V constant-voltage artefact
# — the one the docstring holds up as what this test would have caught on
# sight — is an EXCESS, and the gate as written would have waved it through.
#
# Where the number comes from. Finite differencing on a quantised voltage
# record legitimately overshoots, because a small dV in the denominator
# inflates the quotient before the trapezoid puts it back. Measured on the
# synthetic cases at 0.1-5 mV voltage resolution, that overshoot reaches
# 1.047 and no further, so a ceiling anywhere below about 1.10 would fire on
# ordinary data. 1.20 mirrors the floor's depth and sits well clear of the
# mechanism.
INTEGRAL_FIDELITY_CEILING = 1.20

# WHAT THE BAND ACTUALLY GUARDS, WHICH DEPENDS ON HOW dQ/dV WAS COMPUTED.
# The chemistry-dependence argued above is real, but it belongs to the
# FINITE-DIFFERENCE path. Measured on the synthetic cases, differencing a
# voltage record quantised at 1 mV recovers:
#
#     two_phase       (sigma   8 mV)   0.851      5% of records lost to dV=0
#     tracking        (sigma  12 mV)   1.031      none lost
#     mixed           (sigma  28 mV)   1.007
#     solid_solution  (sigma 220 mV)   1.010
#
# — and at 2 mV the two-phase curve falls to 0.500. So fidelity there is set
# by how flat the plateau is against the instrument's voltage resolution, not
# by chemistry as such; chemistry only enters through flatness, and a
# two-phase plateau is the worst case there is. 0.80 is a defensible line for
# that path.
#
# Ratatosk's DEFAULT path is the derivative-free histogram, which bins charge
# by voltage and divides by the bin width, so its integral returns the
# delivered capacity BY CONSTRUCTION: the same sweep gives 1.000 at every
# resolution for every case, and the LTO triplicate measures 0.99-1.00 on
# every dataset. On that path a deficit no longer means "the derivative could
# not resolve the plateau" — it can only mean charge that fell OUTSIDE the
# analysed voltage window, a constant-voltage hold being the usual reason.
# That is still worth catching and the floor still catches it. It is a
# different claim from the one this comment used to make, and the verdict
# text now says which.


def cycle_column(t, wanted):
    """Find a cycling-table column by bare name, then by cell-prefixed name.

    The per-cell table uses the bare name (`Discharge_mAh_g`); the combined
    export prefixes it with the cell id. A COMBINED table carrying more than
    one cell returns None rather than the first match, because taking the
    first would put cell A's capacities in cell C's report.
    """
    if t is None or getattr(t, "empty", True):
        return None
    for w in wanted:
        if w in t.columns:
            return w
    for w in wanted:
        hit = [c for c in t.columns if c.endswith("_" + w)]
        if len(hit) == 1:
            return hit[0]
        if len(hit) > 1:
            return None
    return None


def capacity_retention_pct(cycle_table, reference_cycle=None):
    """
    Delivered discharge capacity at the end as a percentage of the reference.

    Both ends are a median of up to three cycles, matching how `track_peaks`
    measures a peak's area retention — the two numbers are only comparable if
    they are computed the same way. Incomplete cycles are excluded. Returns
    NaN when the table cannot answer, which is not the same as 100%.
    """
    t = cycle_table
    if t is None or getattr(t, "empty", True) or "Cycle" not in t.columns:
        return np.nan
    # FROM THE ION TABLE, not typed out. Three separate tuples listed Li and
    # Na only, so a third working ion would have stopped matching in silence.
    dcol = cycle_column(t, CAPACITY_COLUMN_ALIASES)
    if dcol is None:
        return np.nan
    icol = cycle_column(t, ("Incomplete",))
    good = t[t[icol] != True] if icol else t              # noqa: E712
    good = good[good["Cycle"].notna()].sort_values("Cycle")
    if reference_cycle is not None:
        good = good[good["Cycle"] >= float(reference_cycle)]
    d = pd.to_numeric(good[dcol], errors="coerce").dropna()
    d = d[d > 0]
    if len(d) < 2:
        return np.nan
    a_ref = float(d.head(3).median())
    a_end = float(d.tail(3).median())
    return (a_end / a_ref * 100.0) if a_ref > 0 else np.nan


def flag_low_fidelity(tracking, fidelity, *, floor=INTEGRAL_FIDELITY_FLOOR,
                      ceiling=INTEGRAL_FIDELITY_CEILING):
    """
    Mark a dataset's area trends untrustworthy where the dQ/dV curve does not
    account for the cell's charge. See INTEGRAL_FIDELITY_FLOOR.

    `fidelity` is `{(cycle, step): ratio}` — what `quality.integral_fidelity`
    returns per half-cycle — or a single float. The test is applied PER STEP,
    because a half-cycle with a constant-voltage hold and one without behave
    quite differently and a dataset average would hide it.

    Nothing is deleted. `area_retention_trustworthy` goes False and
    `area_withheld_reason` says why, so a reader of the summary sees the
    number and the reason for doubting it side by side.
    """
    if isinstance(fidelity, (int, float)):
        per_step = {"Charge": float(fidelity), "Discharge": float(fidelity)}
    else:
        acc = {}
        for (_cyc, step), f in (fidelity or {}).items():
            if np.isfinite(f):
                acc.setdefault(str(step), []).append(float(f))
        per_step = {k: float(np.median(v)) for k, v in acc.items() if v}
    tracking.integral_fidelity = dict(per_step)
    marked = []
    for k, sm in tracking.summary.items():
        f = per_step.get(sm.get("step"), np.nan)
        sm["integral_fidelity"] = f
        # BOTH SIDES. A curve carrying more charge than the cell delivered
        # is not a better measurement than one carrying less; it is a worse
        # one, because nothing physical produces it.
        _short = bool(np.isfinite(f) and f < float(floor))
        _over = bool(np.isfinite(f) and f > float(ceiling))
        bad = _short or _over
        sm["area_retention_trustworthy"] = not bad
        if _short:
            sm["area_withheld_reason"] = (
                f"the dQ/dV curve accounts for only {100 * f:.0f}% of this "
                f"step's delivered capacity, so a peak area is not a "
                f"capacity")
        elif _over:
            sm["area_withheld_reason"] = (
                f"the dQ/dV curve carries {100 * f:.0f}% of this step's "
                f"delivered capacity — more charge than the cell passed, so "
                f"some of this area was not measured")
        else:
            sm["area_withheld_reason"] = ""
        if bad:
            marked.append(k)
    return marked


def flag_area_growth(tracking, capacity_retention,
                     *, headroom_pct=AREA_GROWTH_HEADROOM_PCT):
    """
    Mark tracked peaks whose area grew by more than the cell's capacity allows.

    Writes `area_exceeds_capacity` and `area_growth_excess_pct` into every
    entry of `tracking.summary` and records the bound on the `Tracking`, so
    the flag travels with the object rather than being recomputed by whoever
    prints it. Returns the list of offending keys.

    Nothing is deleted and no fit is rejected: this is a WARNING that the
    decomposition in that step is not supported by the cell's own charge
    balance, and the right response is to look at the fit, not to drop it.
    """
    tracking.capacity_retention_pct = float(capacity_retention)
    flagged = []
    if not np.isfinite(capacity_retention):
        for s in tracking.summary.values():
            s["area_exceeds_capacity"] = False
            s["area_growth_excess_pct"] = np.nan
        return flagged
    bound = float(capacity_retention) + float(headroom_pct)
    for k, s in tracking.summary.items():
        a = s.get("area_retention_pct", np.nan)
        excess = (a - bound) if np.isfinite(a) else np.nan
        bad = bool(np.isfinite(excess) and excess > 0 and not s.get("is_ghost"))
        s["area_exceeds_capacity"] = bad
        s["area_growth_excess_pct"] = excess
        if bad:
            flagged.append(k)
    return flagged


def track_peaks(params_df, detection, *, tolerance_mV=TRACKING_TOLERANCE_MV,
                verbose=True):
    """
    Match fitted components to the reference peak list, cycle by cycle.

    Drift and retention are measured against the REFERENCE CYCLE, not cycle 1:
    a peak that behaves differently during formation would otherwise be given
    an artefactual retention. Peaks whose reference-cycle area is already
    ~zero — the fitter kept a collapsed component — are ghosts, and retention
    is not reported for them at all rather than reported as a ratio of two
    numbers that mean nothing.
    """
    tol_V = tolerance_mV / 1000.0
    ref_cycle = detection.reference_cycle
    P = params_df
    # Inferred components are NOT struck from the table here. They were, on
    # the reasoning that an inferred-and-unreliable component is worthless —
    # but `reliable` is now False for every inferred component by definition,
    # so that test discarded all of them and the seeding it was written to
    # guard could never show its work. They are tracked; the `reliable` flag
    # keeps them out of the drift and retention numbers, which is where the
    # exclusion belongs.

    nan_params = {c: np.nan for c in
                  ("centre", "centre_stderr", "amplitude_area",
                   "amplitude_stderr", "sigma", "sigma_stderr", "fraction",
                   "fraction_stderr", "fwhm", "height", "voltage_shift",
                   "sigma_r", "sigma_r_stderr", "asymmetry",
                   # Carried through so the reliability test below can ask
                   # whether a component is pinned at the bound ITS OWN fit
                   # used, not at a module constant.
                   "sigma_max_fitted",
                   # PROVENANCE, carried for the same reason.
                   # `parameters_frame` demotes every inferred component with
                   # `P["reliable"] &= ~P["is_inferred"]`, and the comment
                   # above claims that flag "keeps them out of the drift and
                   # retention numbers". It did not: `_reliable` below
                   # recomputes reliability from scratch on THIS table, and
                   # `is_inferred` was never copied into it. Measured on
                   # NMC111 cell A: 34 inferred components, all correctly
                   # unreliable in the parameter table, 31 of them reaching
                   # this table and 25 coming back reliable — feeding drift,
                   # area retention, the tracking rate and the discontinuity
                   # test, while detection printed "marked is_inferred, and
                   # excluded from headline trends".
                   "is_inferred",
                   # The fit's own verdict on itself. Reported, no longer a
                   # gate — see AREA_STDERR_RATIO_MAX.
                   "converged", "hit_iteration_cap",
                   # ...and the quantity that replaced it, carried for the
                   # same reason `is_inferred` had to be: a flag computed in
                   # `parameters_frame` and not copied here is a flag that
                   # reaches none of the trends.
                   "area_stderr_ratio", "area_stderr_missing",
                   "area_poorly_determined", "area_empty",
                   # WHY the boolean says what it says, and whether the area
                   # has an uncertainty at all. See the note where they are
                   # built: one boolean was carrying four statements.
                   "reliability_reason", "area_determinacy",
                   # SAME REASON AS `is_inferred` DIRECTLY ABOVE, and the same
                   # trap: `_reliable` recomputes reliability from scratch on
                   # THIS table, so a flag set only in `parameters_frame`
                   # reaches the parameter CSV and nothing else.
                   "half_cycle_anomalous",
                   # WHAT THE FIT DECIDED THIS COMPONENT IS. The mechanism
                   # classifier chooses whether bands are available at all;
                   # the fit then decides, per component, whether each one
                   # became a band or stayed a peak. That answer was in the
                   # fitted-parameter CSV and nowhere else — not in this
                   # table, not in the per-peak summary, not in either
                   # report — so a reader could not tell which shape had
                   # been used for the feature they were reading about.
                   "component_kind",
                   # THE AREA THE CAPACITY NUMBERS ARE NOW COMPUTED FROM.
                   # Carried for the same reason `is_inferred` and
                   # `component_kind` had to be: `CAPACITY_AREA_COLUMN`
                   # switched the attribution, the area trend and the
                   # coherence area test to `area_in_window`, and `_area`
                   # falls back to the analytic value when the column is
                   # absent — so leaving it out of this table would have
                   # silently reverted every one of them here while the
                   # parameter table used the new one. That is the fourth
                   # time a field computed upstream has failed to reach a
                   # consumer; the fallback would have made it the quietest.
                   "area_in_window",
                   # THE FLAG LIST IS NOW ACTUALLY USED. `_DERIVED_FLAG_COLS`
                   # was defined with a comment warning that "a flag computed
                   # here and never added to a list like this one silently
                   # never reaches the CSV" — and then read by nothing at
                   # all, while THIS list, the one that decides what reaches
                   # the tracked table, was maintained separately and drifted
                   # from it.
                   #
                   # `band_at_width_bound` is the cost. It is absent here, so
                   # `ATTRIBUTION_EXCLUDES` — which names it — could never
                   # exclude anything (`if col in cd.columns` was always
                   # False), and every capacity share, `total_area`,
                   # `retained_fraction` and `peakN_fraction` still counted
                   # pinned bands. The demotion landed in `parameters_frame`
                   # in 1.9.0.5x; the attribution half of the same fix did
                   # not. It is 59% of NNM's not-reliable components and 45%
                   # of NMC's.
                   *_DERIVED_FLAG_COLS,
                   "band_at_width_bound", "band_width_max_fitted",
                   "component_kind",
                   "area_outside_window_frac", "fit_truncated",
                   # ...and the flag `_reliable` reads when
                   # ENFORCE_AREA_COMPLETENESS is on. Carried for exactly the
                   # reason the comment above gives: a demotion that cannot
                   # see its own flag here is a demotion that does nothing.
                   "area_is_lower_bound", "centre_poorly_determined")}
    rows = []
    for step in ("Charge", "Discharge"):
        ref_peaks = detection.reference_peaks.get(step, pd.DataFrame())
        if ref_peaks is None or ref_peaks.empty:
            continue
        step_cycles = sorted({int(c) for (c, s) in detection.peaks if s == step})
        ref_list = [(int(r["peak_id"]), float(r["voltage"]),
                     bool(r.get("is_shoulder", False)))
                    for _, r in ref_peaks.iterrows()]
        assignment = _assign_components(
            P, ref_list, step_cycles, step, tol_V,
            reference_cycle=detection.reference_cycle)

        # NEVER FITTED IS NOT THE SAME AS DISAPPEARED.
        #
        # The roster is built from the DETECTED reference list while the fit
        # may have been given a SUBSET of it — `primaries_only`, set by the
        # mechanism classifier, drops the shoulder stack. Those features then
        # appeared in START_HERE as tracked peaks and were recorded as
        # `disappeared` in every cycle of the run: on NMC111, 3 of 7 discharge
        # features tracked in zero of 18 cycles. "Disappeared" says the peak
        # was there and went, which is a finding about the CELL; the truth
        # was that the model was never asked to fit it, which is a fact about
        # the RUN. The page was advertising features it had not fitted and
        # blaming the cell for their absence.
        #
        # Decided from the data rather than from a flag, so it is right
        # whatever the reason: a reference peak matched in NO cycle at all
        # was never fitted.
        _never = {rid for (rid, _v, _sh) in ref_list
                  if not any(assignment.get((c, rid)) is not None
                             for c in step_cycles)}

        for _, ref_pk in ref_peaks.iterrows():
            ref_v = float(ref_pk["voltage"])
            ref_id = int(ref_pk["peak_id"])
            base_meta = dict(is_shoulder=bool(ref_pk.get("is_shoulder", False)),
                             is_truncated=bool(ref_pk.get("is_truncated",
                                                          False)))
            for cycle in step_cycles:
                cp = P[(P["cycle"] == cycle) & (P["step"] == step)]
                base = dict(dataset=detection.name, cycle=cycle, step=step,
                            tracked_peak_id=ref_id, reference_voltage=ref_v,
                            r_squared=(cp["r_squared"].iloc[0]
                                       if not cp.empty else np.nan),
                            **base_meta)
                if ref_id in _never:
                    rows.append({**base, "status": "not_fitted", **nan_params})
                    continue
                if cp.empty:
                    rows.append({**base, "status": "no_fit", **nan_params})
                    continue
                idx = assignment.get((cycle, ref_id))
                if idx is None:
                    rows.append({**base, "status": "disappeared", **nan_params})
                    continue
                m = cp.iloc[idx]
                rows.append({**base, "status": "tracked",
                             **{c: m.get(c, np.nan) for c in nan_params
                                if c != "voltage_shift"},
                             "voltage_shift": m["centre"] - ref_v})

    T = pd.DataFrame(rows)
    tk = Tracking(name=detection.name, tracked_df=T,
                  reference_peaks=detection.reference_peaks,
                  reference_cycle=ref_cycle, tolerance_mV=tolerance_mV)
    if T.empty:
        return tk

    # --- reliability of a tracked point ---------------------------------
    # From the fit, per row, with the module constant only as a fallback —
    # see `_flag_reliable_peaks` for why a fixed 200 mV disabled this test on
    # every sharp and moderate profile.
    if "sigma_max_fitted" in T:
        _bound = pd.to_numeric(T["sigma_max_fitted"], errors="coerce")
        sigma_max_by_row = _bound.fillna(FIT_SIGMA_MAX_MV / 1000.0)
    else:
        sigma_max_by_row = pd.Series(FIT_SIGMA_MAX_MV / 1000.0, index=T.index)
    ref_areas, ghosts = {}, set()
    at_ref = T[(T["cycle"] == ref_cycle) & (T["status"] == "tracked")]
    for _, r in at_ref.iterrows():
        k = (r["step"], int(r["tracked_peak_id"]))
        ref_areas[k] = r["amplitude_area"]
        if pd.notna(r["amplitude_area"]) and \
                r["amplitude_area"] < GHOST_AREA_THRESHOLD:
            ghosts.add(k)

    def _reliable(row):
        if row["status"] != "tracked":
            return False
        # An inferred component is a seed carried in from a neighbouring
        # cycle because detection found nothing here. Its centre is another
        # cycle's answer; it is not a measurement of this one.
        #
        # NaN IS NOT FALSE HERE. `bool(np.nan)` is True, so reading these two
        # flags with a bare `bool()` marked every row of a frame that lacks
        # the column — the tracking table fills missing parameters with NaN —
        # as inferred and unreliable. Absent means "not stated", which for a
        # provenance flag is the same as False; only an explicit True demotes.
        _inf = row.get("is_inferred")
        if _inf is not None and not pd.isna(_inf) and bool(_inf):
            return False
        # ...and the half-cycle's own verdict, read the same NaN-safe way.
        # See the note in `parameters_frame`.
        _an = row.get("half_cycle_anomalous")
        if _an is not None and not pd.isna(_an) and bool(_an):
            return False
        # THE CONVERGENCE GATE IS GONE HERE TOO, and it had to go from both
        # places at once: the parameter table and this one were the pair the
        # module's own SIGMA_BOUND_PROXIMITY note warns about, where a
        # component can be reliable in one table and not the other. See
        # AREA_STDERR_RATIO_MAX in the constants for the measurements and for
        # what is flagged in its place.
        if ENFORCE_AREA_DETERMINACY:
            _ad = row.get("area_poorly_determined")
            if _ad is not None and not pd.isna(_ad) and bool(_ad):
                return False
        # The completeness half of the same open question, NaN-safe like the
        # rest. Applied here AND in `parameters_frame`; see the note there.
        if ENFORCE_AREA_COMPLETENESS:
            _lb = row.get("area_is_lower_bound")
            _of = row.get("area_outside_window_frac")
            if (_lb is not None and not pd.isna(_lb) and bool(_lb)
                    and _of is not None and not pd.isna(_of)
                    and float(_of) > AREA_LOWER_BOUND_REPORTABLE):
                return False
        # NOT behind ENFORCE_AREA_DETERMINACY. "The error on this area is too
        # large to quote" is a judgement about a measurement and is the open
        # decision; "this component carries a thousandth of the charge its
        # neighbour does" is the fit saying the feature is not there, and
        # there is nothing to decide. Struck in `parameters_frame` too, and it
        # has to be struck in BOTH — that is the pairing this module's own
        # SIGMA_BOUND_PROXIMITY note warns about.
        _ae = row.get("area_empty")
        if _ae is not None and not pd.isna(_ae) and bool(_ae):
            return False
        k = (row["step"], int(row["tracked_peak_id"]))
        if k in ghosts:
            return False
        # A BAND AT ITS WIDTH CEILING IS A PEDESTAL, and `parameters_frame`
        # has demoted one since 1.9.0.5x. This table could not, because the
        # column never travelled — so the same component was `reliable` here
        # and not there, and this is the table that feeds drift, area
        # retention and the tracking rate.
        _bw = row.get("band_at_width_bound")
        if _bw is not None and not pd.isna(_bw) and bool(_bw):
            return False
        _b = sigma_max_by_row.get(row.name, FIT_SIGMA_MAX_MV / 1000.0)
        # BOTH FLANKS. `parameters_frame` was corrected in 1.9.0.58 — a
        # component whose HIGH flank is pinned at the ceiling is what a
        # terminal feature does — and this copy of the test was not, so a
        # component could be reliable in one table and not the other. That is
        # precisely the pairing this module's own SIGMA_BOUND_PROXIMITY note
        # warns about.
        _sr = row.get("sigma_r")
        _pin = SIGMA_BOUND_PROXIMITY_COMPLEMENT * _b
        if (pd.notna(row["sigma"]) and row["sigma"] >= _pin):
            return False
        if _sr is not None and pd.notna(_sr) and float(_sr) >= _pin:
            return False
        ra = ref_areas.get(k, np.nan)
        if (pd.notna(ra) and ra > 0 and pd.notna(row["amplitude_area"])
                and row["amplitude_area"] < _AREA_COLLAPSE_FRAC * ra):
            return False
        return True

    T["reliable"] = T.apply(_reliable, axis=1)
    tk.ghost_peaks = ghosts

    # --- per-peak summary ------------------------------------------------
    def _kinds(ts):
        if ts is None or "component_kind" not in getattr(ts, "columns", []):
            return []
        return [str(k) for k in ts["component_kind"].dropna().tolist() if k]

    def _modal_kind(ts):
        ks = _kinds(ts)
        if not ks:
            return None
        return max(set(ks), key=ks.count)

    def _kind_is_mixed(ts):
        return len(set(_kinds(ts))) > 1

    summary = {}
    for step in ("Charge", "Discharge"):
        sdf = T[T["step"] == step]
        if sdf.empty:
            continue
        for tid in sorted(sdf["tracked_peak_id"].unique()):
            pk = sdf[sdf["tracked_peak_id"] == tid].copy()
            n_trk = int((pk["status"] == "tracked").sum())
            n_dis = int((pk["status"] == "disappeared").sum())
            n_nf = int((pk["status"] == "no_fit").sum())
            # Detected, rostered, and never given to the model. See the
            # `_never` note in `track_peaks`.
            n_never = int((pk["status"] == "not_fitted").sum())
            n_tot = len(pk)
            key = (step, int(tid))
            is_ghost = key in ghosts
            ts = pk[pk["status"] == "tracked"].sort_values("cycle")
            ref_row = ts[ts["cycle"] == ref_cycle]
            v_drift = a_ret = np.nan
            _vend_cycle = _vstart_cycle = None
            _span = 0
            if len(ts) >= 2 and not ref_row.empty:
                # FROM THE REFERENCE CYCLE ONWARD. `ts` is every tracked
                # cycle, so head(3) was cycles 1-3 — formation, on almost
                # every dataset — and the retention this function documents
                # as measured against the reference cycle was measured
                # against the wrong end of the run. `ref_row` existed for
                # exactly this and was reachable only as a fallback.
                rel = ts[ts["reliable"]] if "reliable" in ts else ts
                rel = rel[rel["cycle"] >= ref_cycle]
                # BOTH ENDS ARE A MEDIAN OF UP TO THREE CYCLES, not two
                # single points. A centre is recovered with a few mV of
                # scatter, so a difference of two single cycles carries
                # sqrt(2) times that scatter — on the synthetic 50-cycle
                # series, 90 mV of true drift came back as 79 mV with the
                # single-point endpoints and no bias in the centres
                # themselves. It was noise in the two chosen cycles, and the
                # same median-of-three the area retention below already uses
                # removes it. `voltage_drift_first_cycle` and
                # `voltage_drift_last_cycle` name the interval it was
                # measured over, which is what a reader needs in order to
                # turn it into mV per cycle.
                _vs = rel["voltage_shift"].dropna()
                if len(_vs) >= 2:
                    _tail = _vs.tail(DRIFT_ENDPOINT_CYCLES)
                    _head = _vs.head(DRIFT_ENDPOINT_CYCLES)
                    _vend = float(_tail.median())
                    _vstart = float(_head.median())
                    # The MIDDLE cycle of each window, because a median of
                    # three sits at the middle one. Naming the outermost
                    # cycles instead overstates the interval by two cycles,
                    # and anyone dividing by it to get mV per cycle inherits
                    # that error.
                    _vend_cycle = int(np.median(
                        rel.loc[_tail.index, "cycle"].values))
                    _vstart_cycle = int(np.median(
                        rel.loc[_head.index, "cycle"].values))
                else:
                    _vend = ts["voltage_shift"].iloc[-1]
                    _vstart = ref_row["voltage_shift"].iloc[0]
                    _vend_cycle = int(ts["cycle"].iloc[-1])
                    _vstart_cycle = int(ref_row["cycle"].iloc[0])
                v_drift = (_vend - _vstart) * 1000
                # Per cycle, which is the quantity that can be compared
                # between cells of different lengths — and the one published
                # peak-tracking work quotes.
                _span = _vend_cycle - _vstart_cycle
                if not is_ghost:
                    _ra = _area(rel)
                    b = _ra.head(3).dropna()
                    e = _ra.tail(3).dropna()
                    a_ref = (b.median() if len(b)
                             else float(_area(ref_row).iloc[0]))
                    a_last = (e.median() if len(e)
                              else float(_area(ts).iloc[-1]))
                    a_ret = (a_last / a_ref * 100
                             if (pd.notna(a_ref) and a_ref > 0) else np.nan)
            # How far this peak ever got from its reference position. Under
            # a DRIFTING reference that is no longer a warning — it is the
            # measurement, and a peak that moves 200 mV over 100 cycles is
            # now tracked rather than handed to a neighbour at 80. What still
            # breaks an identity is a single-cycle STEP wider than the
            # matching window, so that is what the flag now tests.
            _shift_mV = (pk["voltage_shift"].abs().max() * 1000
                         if "voltage_shift" in pk else np.nan)
            n_ev, first_ev, max_step, established = _discontinuity_events(pk)
            _gate_mV = (max_step if TRACK_AGAINST_PREVIOUS_CYCLE
                        else _shift_mV)
            _beyond = bool(FLAG_DRIFT_BEYOND_TOLERANCE
                           and np.isfinite(_gate_mV)
                           and _gate_mV > tolerance_mV)
            summary[f"{step}_peak{tid}"] = dict(
                step=step, peak_id=int(tid), reference_voltage=
                float(pk["reference_voltage"].iloc[0]),
                is_shoulder=bool(pk["is_shoulder"].iloc[0]),
                is_truncated=bool(pk["is_truncated"].iloc[0]),
                is_ghost=is_ghost, tracked=n_trk, disappeared=n_dis,
                no_fit=n_nf, not_fitted=n_never,
                never_fitted=bool(n_never and n_never == n_tot),
                total_cycles=n_tot,
                tracking_rate=(n_trk / n_tot if n_tot else 0.0),
                max_shift_mV=_shift_mV,
                drift_exceeds_tolerance=_beyond,
                voltage_drift_mV=v_drift,
                voltage_drift_first_cycle=_vstart_cycle,
                voltage_drift_last_cycle=_vend_cycle,
                voltage_drift_mV_per_cycle=(v_drift / _span if _span
                                            else np.nan),
                area_retention_pct=a_ret,
                discontinuity_events=n_ev,
                discontinuity_is_failure=bool(n_ev and established),
                first_discontinuity_cycle=first_ev,
                max_centre_step_mV=max_step,
                fit_conditional_continuity=(n_trk / (n_trk + n_dis)
                                            if (n_trk + n_dis) else np.nan),
                # The shape this feature was actually fitted with, over the
                # cycles where it was tracked. Reported as the majority plus a
                # count, not a single label, because a feature that is a band
                # in some cycles and a peak in others is telling you something
                # about the fit's stability that a modal label would hide.
                component_kind=_modal_kind(ts),
                component_kind_mixed=_kind_is_mixed(ts))
    tk.summary = summary
    if verbose:
        print("\n" + tk.summary_text())
    return tk


# =============================================================================
# 3. DERIVED QUANTITIES  (Module 6)
# =============================================================================

@dataclass
class DeltaV:
    """Charge/discharge pairing and the polarisation history of each couple."""
    name: str
    pairs: list = field(default_factory=list)
    df: pd.DataFrame = field(default_factory=pd.DataFrame)
    changes: dict = field(default_factory=dict)   # label -> (net, v0, v1)
    windows: dict = field(default_factory=dict)   # label -> (c0, c1, n)

    def summary_text(self):
        L = [section("  Redox pairs")]
        if not self.pairs:
            L.append("    none found within the pairing window")
        for cid, did, cv, dv, sh in self.pairs:
            # SIGNED here and a MAGNITUDE in the trend below, and both are
            # right: the sign is electrode-dependent (see this function's
            # docstring) while polarisation is |dV|, because on an anode a
            # signed trend reports growing polarisation as an improvement.
            # Printed adjacent under one name "dV", a reader saw -31 then
            # +37 for what looked like one quantity. They are now named
            # apart, and the signed value is labelled with its convention.
            L.append(f"    Charge {cid} ({cv:.3f} V) <-> Discharge {did} "
                     f"({dv:.3f} V)   V(chg)-V(dis) "
                     f"{(cv - dv) * 1000:+.0f} mV at the reference cycle"
                     + ("  [incl. shoulder]" if sh else ""))
        for label, (net, v0, v1) in self.changes.items():
            c0, c1, n = self.windows[label]
            rate = net / n if n else np.nan
            trunc = ""
            if not self.df.empty and "pair_involves_truncated" in self.df:
                sub = self.df[self.df["pair_label"] == label]
                if len(sub) and bool(sub["pair_involves_truncated"].iloc[0]):
                    trunc = "  [truncated peak — centre unreliable]"
            L.append(f"    {label}: polarisation |dV| {v0:.0f} -> "
                     f"{v1:.0f} mV (net {net:+.0f} mV = {rate:+.3f} mV/cyc "
                     f"over n={n}) cyc {c0:.0f}-{c1:.0f}{trunc}")
        return "\n".join(L)


def delta_v(tracking, *, is_anode=False, max_sep_mV=PAIR_MAX_SEPARATION_MV,
            exclude_formation=EXCLUDE_FORMATION_FROM_DELTAV, verbose=True,
            first_cycle=None):
    """
    Charge-discharge peak separation for each redox couple.

    dV = V_charge - V_discharge, and its SIGN is electrode-dependent: for a
    cathode the charge peak sits above the discharge peak, while for a negative
    electrode `io._apply_electrode_convention` has swapped the labels so the
    delithiation "Discharge" peak sits above the lithiation "Charge" peak and
    dV is negative. Polarisation is the magnitude, so a widening |dV| means
    rising internal resistance either way.

    The trend is reported as a NET CHANGE over the leading coherent window, not
    as a fitted slope: polarisation growth is sigmoidal, and a single rate would
    misstate it.
    """
    ref = tracking.reference_peaks
    pairs = _auto_pair_peaks(ref.get("Charge", pd.DataFrame()),
                             ref.get("Discharge", pd.DataFrame()),
                             max_sep_mV, is_anode=is_anode)
    T = tracking.tracked_df
    if T is None or T.empty or "step" not in T:
        return DeltaV(name=tracking.name, pairs=pairs, df=pd.DataFrame(),
                      changes={}, windows={})
    rows = []
    for cid, did, cv, dv, is_sh in pairs:
        cdat = T[(T["step"] == "Charge") & (T["tracked_peak_id"] == cid)
                 & (T["status"] == "tracked")].set_index("cycle")
        ddat = T[(T["step"] == "Discharge") & (T["tracked_peak_id"] == did)
                 & (T["status"] == "tracked")].set_index("cycle")
        # A truncated peak's centre is an extrapolation, so a pair resting on
        # one is flagged rather than silently trusted.
        pair_trunc = (bool(cdat.get("is_truncated", pd.Series(dtype=bool)).any())
                      or bool(ddat.get("is_truncated",
                                       pd.Series(dtype=bool)).any()))
        common = sorted(set(cdat.index) & set(ddat.index))
        if exclude_formation:
            # FROM THE END OF FORMATION, not from cycle 2. This was `c > 1`,
            # which drops the first cycle and nothing else — so the
            # polarisation trend was computed across cycles the pipeline had
            # already measured as unsettled, and printed beside a peak-drift
            # trend that excluded them. On the LTO triplicate that is
            # "cyc 2-10" against "over c5-9" in the same block.
            #
            # `first_cycle` is the reference cycle, which since 1.9.0.18 IS
            # the measured end of formation (detect.formation_end). Falling
            # back to 2 keeps the old behaviour where the caller does not
            # say.
            _start = (int(first_cycle) if first_cycle is not None
                      else 2)
            common = [c for c in common if c >= _start]
        for cycle in common:
            c_cen, d_cen = cdat.loc[cycle, "centre"], ddat.loc[cycle, "centre"]
            c_err = cdat.loc[cycle].get("centre_stderr", np.nan)
            d_err = ddat.loc[cycle].get("centre_stderr", np.nan)
            err = (np.sqrt(c_err ** 2 + d_err ** 2)
                   if pd.notna(c_err) and pd.notna(d_err) else np.nan)
            rows.append(dict(
                dataset=tracking.name, cycle=cycle, charge_peak_id=cid,
                discharge_peak_id=did, charge_ref_voltage=cv,
                discharge_ref_voltage=dv, pair_label=f"{cv:.2f}/{dv:.2f} V",
                charge_centre=c_cen, discharge_centre=d_cen,
                delta_v=c_cen - d_cen, delta_v_mV=(c_cen - d_cen) * 1000,
                delta_v_stderr=err,
                delta_v_stderr_mV=(err * 1000 if pd.notna(err) else np.nan),
                pair_involves_truncated=pair_trunc))

    D = pd.DataFrame(rows)
    changes, windows = {}, {}
    if not D.empty:
        for label in D["pair_label"].unique():
            p = D[D["pair_label"] == label].sort_values("cycle")
            # The MAGNITUDE, as the docstring says. Feeding the signed
            # column meant that on an anode — where dV is negative because
            # the labels are swapped — growing polarisation was reported as
            # a negative net change, which reads as an improvement.
            net, v0, v1, c0, c1, n = _coherent_net_change(
                p["cycle"], p["delta_v_mV"].abs())
            if pd.notna(net):
                changes[label] = (net, v0, v1)
                windows[label] = (c0, c1, n)

    out = DeltaV(name=tracking.name, pairs=pairs, df=D, changes=changes,
                 windows=windows)
    if verbose:
        print("\n" + out.summary_text())
    return out


def capacity_attribution(tracking, *, closure_intervals=None,
                         unattributed=None,
                         withhold_above=ATTRIBUTION_WITHHOLD_ABOVE,
                         fidelity_floor=INTEGRAL_FIDELITY_FLOOR,
                         fidelity_ceiling=INTEGRAL_FIDELITY_CEILING,
                         verbose=True):
    """
    Each peak's area as a fraction of a FIXED reference total — where that
    fraction is a measurement, and NaN where it is not.

    The basis is the summed fitted area of every peak tracked at the reference
    cycle: the same baseline tracking uses for drift and retention. A per-cycle
    denominator shrinks as peaks drop out, which redistributes lost capacity
    onto the survivors; a fixed one lets the stack fall below 100%, which is
    what actually happened.

    NEW IN 1.9.0 — attribution is withheld where the split is not determined.
    `closure_intervals` maps (cycle, step) -> (lo, hi) from
    `quality.closure_interval`. Where hi - lo exceeds `withhold_above`, a
    pseudo-Voigt and the polynomial baseline are near-degenerate over that
    window: refitting with a different baseline degree moves the "peak" share
    of the capacity by more than the number being reported. The AREA is kept —
    it is what the fit found — but the FRACTION is set to NaN and the row
    carries `attribution_withheld` and the interval that caused it. On NMC111
    that suppresses the whole dataset, which is the correct answer; on LTO and
    P3 it suppresses nothing.

    NEW IN 1.9.1 — attribution is ALSO withheld where the curve does not
    account for the cell's charge. Closure asks "do the fitted peaks account
    for the CURVE"; integral fidelity asks "does the curve account for the
    CELL". A dataset can pass the first perfectly and fail the second
    completely, and the LTO triplicate does: closure 0.98-1.00, fidelity
    0.58-0.68. Attributing capacity to peaks on a curve carrying 60% of the
    charge would be attributing a capacity that is not there. The reason is
    recorded per step in `fidelity_withheld`, so the two grounds stay
    distinguishable in the CSV.
    """
    T = tracking.tracked_df
    # `Tracking.tracked_df` defaults to a bare DataFrame, and `track_peaks`
    # returns exactly that when nothing was tracked — reachable whenever the
    # reference cycle has no detected peaks. `coherence_audit` already
    # handled it; these two raised KeyError: 'step'.
    if T is None or T.empty or "step" not in T:
        return {}
    ref_v_lookup = {}
    for step in ("Charge", "Discharge"):
        rp = tracking.reference_peaks.get(step, pd.DataFrame())
        if rp is not None and not rp.empty:
            for _, pk in rp.iterrows():
                ref_v_lookup[(step, int(pk["peak_id"]))] = float(pk["voltage"])

    # `quality.sample_half_cycles` measures the closure interval on about
    # ten half-cycles per dataset, deliberately, because stability is a
    # property of the curve's SHAPE and is consistent within a dataset and
    # step. Reading the map cycle by cycle therefore gave every unmeasured
    # cycle a NaN width and a quiet "not withheld" — the safeguard was inert
    # on ~97% of the run. The sample is propagated the same way it was
    # taken: the widest interval seen for a step stands for that step.
    _step_width = {}
    for (_c, _s), _iv in (closure_intervals or {}).items():
        if _iv is None:
            continue
        _lo, _hi = _iv[0], _iv[1]
        # A MEASURED-BUT-FAILED interval (no baseline degree converged) is
        # not evidence of stability; it is the absence of evidence.
        _w = (float(_hi - _lo) if np.isfinite(_lo) and np.isfinite(_hi)
              else np.inf)
        _step_width[_s] = max(_step_width.get(_s, -np.inf), _w)

    out = {}
    for step in ("Charge", "Discharge"):
        st = T[(T["step"] == step) & (T["status"] == "tracked")]
        if st.empty:
            continue
        _sw = _step_width.get(step, np.nan)
        # Does this step's CURVE account for the cell's charge? Set by
        # `flag_low_fidelity`; absent, this test cannot fire and does not.
        _fid = tracking.integral_fidelity.get(step, np.nan)
        # TWO-SIDED, like every other reading of this number in the package.
        # `flag_low_fidelity`, `Tracking.summary_text` ("AREA NOT A
        # CAPACITY") and `quality.describe_fidelity` ("AREAS ARE NOT
        # CAPACITIES — above 1.20") all test both ends; this gate tested only
        # the floor, so a curve carrying 155% of the charge the cell passed
        # had every per-peak share published with `fidelity_withheld=False`,
        # one page after the reader was told those areas are not capacities.
        # The excess is the more damning of the two: a deficit can be charge
        # the window did not cover, an excess is area that is not there.
        _fid_bad = bool(np.isfinite(_fid)
                        and (_fid < float(fidelity_floor)
                             or _fid > float(fidelity_ceiling)))
        pids = sorted(st["tracked_peak_id"].unique())
        basis_area, basis_cycle = _attribution_basis(st,
                                                     tracking.reference_cycle)
        basis = (basis_area if (pd.notna(basis_area) and basis_area > 0)
                 else np.nan)
        rows = []
        for cycle in sorted(st["cycle"].unique()):
            cd = st[st["cycle"] == cycle]
            iv = (closure_intervals or {}).get((int(cycle), step))
            if iv is not None:
                width = (float(iv[1] - iv[0])
                         if np.isfinite(iv[0]) and np.isfinite(iv[1])
                         else np.inf)
                measured = True
            else:
                width, measured = _sw, False
            # A THIRD GROUND, AND THE ONLY ONE LEFT WHEN THERE IS NO BASELINE.
            # The closure interval measures how much the peak/background split
            # moves with the baseline degree. From 1.9.0.56 there is no free
            # background, so that width is zero by construction and this test
            # can no longer fire — which is correct, the split IS determined.
            # What can still go wrong is that the named components do not add
            # up to the cell's charge, and a per-peak SHARE of a total that is
            # missing a fifth of the capacity is not a measurement either.
            # See `quality.UNATTRIBUTED_QUALIFIED_ABOVE`.
            _un = (unattributed or {}).get((int(cycle), step), np.nan)
            _un_bad = bool(np.isfinite(_un)
                           and abs(float(_un)) > UNATTRIBUTED_WITHHOLD_ABOVE)
            withheld = bool(width == np.inf
                            or (np.isfinite(width) and width > withhold_above)
                            or _un_bad or _fid_bad)
            # np.inf in a CSV is not a width. Report the number where there
            # is one; the withheld flag beside it carries the verdict.
            width_out = width if np.isfinite(width) else np.nan
            _keep = _countable_area(cd)
            total = float(_area(cd.loc[_keep]).sum())
            _total_all = float(_area(cd).sum())
            row = dict(cycle=int(cycle), total_area=total,
                       total_area_all=_total_all,
                       n_excluded_from_area=int((~_keep).sum()),
                       basis_area=basis_area, basis_cycle=basis_cycle,
                       closure_interval_width=width_out,
                       closure_interval_measured=measured,
                       attribution_withheld=withheld,
                       unattributed_fraction=(float(_un)
                                              if np.isfinite(_un) else np.nan),
                       unattributed_withheld=_un_bad,
                       integral_fidelity=_fid,
                       fidelity_withheld=_fid_bad,
                       retained_fraction=(np.nan if withheld or pd.isna(basis)
                                          else total / basis))
            for pid in pids:
                pk = cd[cd["tracked_peak_id"] == pid]
                area = (float(_area(pk).iloc[0]) if not pk.empty
                        else np.nan)
                # A component excluded from the SUM must not be published as
                # a SHARE of it either — that was the half-wiring.
                _counts = bool(_keep.loc[pk.index].all()) if not pk.empty else False
                frac = (np.nan if (withheld or pk.empty or pd.isna(basis)
                                   or not _counts)
                        else area / basis)
                row[f"peak{pid}_area"] = area
                row[f"peak{pid}_fraction"] = frac
                row[f"peak{pid}_ref_V"] = ref_v_lookup.get((step, pid), np.nan)
                row[f"peak{pid}_is_truncated"] = (
                    bool(pk["is_truncated"].any()) if not pk.empty else False)
            rows.append(row)
        out[step] = pd.DataFrame(rows)

    if verbose:
        print("\n" + section("  Capacity attribution "
                             "(basis = summed reference-cycle area)"))
        for step, A in out.items():
            if A.empty:
                continue
            n_w = int(A["attribution_withheld"].sum())
            last = A.iloc[-1]
            print(f"    {step} (basis cycle {int(last['basis_cycle'])}, "
                  f"{len(A)} cycles)")
            if bool(A["fidelity_withheld"].any()):
                print(f"      WITHHELD on every cycle — the dQ/dV curve "
                      f"accounts for only "
                      f"{100 * float(A['integral_fidelity'].iloc[0]):.0f}% of "
                      f"this step's delivered capacity, so there is no "
                      f"capacity here to attribute. The fitted AREAS are "
                      f"kept and so are the peak positions.")
            if n_w:
                w = A.loc[A["attribution_withheld"], "closure_interval_width"]
                print(f"      WITHHELD on {n_w}/{len(A)} cycles — closure "
                      f"interval {w.min():.2f}-{w.max():.2f} wide, above the "
                      f"{withhold_above:.2f} at which the peak/background "
                      f"split stops being determined by the data.")
                print(f"      The fitted areas are still in the table; the "
                      f"SHARE of capacity is not a measurement here.")
            rep = A[~A["attribution_withheld"]]
            if rep.empty:
                continue
            last = rep.iloc[-1]
            print(f"      cycle {int(last['cycle'])}: "
                  f"{last['retained_fraction'] * 100:.0f}% of basis retained")
            for pid in sorted({int(c.split('_')[0][4:]) for c in A.columns
                               if c.startswith("peak")
                               and c.endswith("_fraction")}):
                f = last.get(f"peak{pid}_fraction", np.nan)
                if pd.notna(f) and f > 0:
                    tr = (" [truncated — area is a lower bound]"
                          if last.get(f"peak{pid}_is_truncated", False) else "")
                    print(f"        peak {pid} "
                          f"({last[f'peak{pid}_ref_V']:.3f} V): "
                          f"{f * 100:.1f}% of basis{tr}")
    return out


# =============================================================================
# 4. COHERENCE AUDIT  (Module 7c, as a function)
# =============================================================================

def _assign_to_reference(rel, ref_voltages, tol_V):
    """One component per reference peak per cycle, chosen once.

    Reference peaks less than 2 x COHERENCE_BAND_TOL_MV apart have
    OVERLAPPING bands, and detection allows peaks 30 mV apart against a
    70 mV tolerance. Selecting each band independently let two reference
    peaks be judged from the same fitted component while another component
    was never examined at all. Assigning them jointly, cycle by cycle, on
    minimum total distance makes the mapping one-to-one.

    Returns {(cycle, ref_index): row index into `rel`}.
    """
    out = {}
    if rel.empty or not len(ref_voltages):
        return out
    rv = np.asarray(ref_voltages, float)
    for cyc, g in rel.groupby("cycle", dropna=True):
        cv = pd.to_numeric(g["centre"], errors="coerce").to_numpy()
        ok = np.isfinite(cv)
        if not ok.any():
            continue
        idx = g.index.to_numpy()[ok]
        cv = cv[ok]
        BIG = 1e6
        cost = np.abs(rv[:, None] - cv[None, :])
        cost = np.where(cost <= tol_V, cost, BIG)
        ri, cj = linear_sum_assignment(cost)
        for i, j in zip(ri, cj):
            if cost[i, j] >= BIG:
                continue
            out[(int(cyc), int(i))] = idx[j]
    return out


def coherence_audit(params_df, detection, *, profile="broad", verbose=True,
                   rate_protocol=None):
    """
    Does each peak move like a redox feature, or like a fitting artefact?

    At C/10 a redox centre moves a few mV per cycle and holds its area. On P3
    cell B the three well-behaved peaks drifted a median of 1.0, 1.3 and 4.5 mV
    per cycle; the truncated peak drifted 15.2 mV with half its consecutive
    pairs over 20 mV.

    The verdict is driven by the MEDIAN drift, not by the tail fractions. An
    earlier version used the tails and denied "coherent" to a peak drifting
    1.4 mV per cycle because 14% of its pairs crossed 20 mV — the tails are a
    noisy statistic on a few dozen pairs and the median is not. Tails are still
    reported, as corroboration.

    Returns a DataFrame, one row per reference peak. It deliberately declines
    to judge below COHERENCE_MIN_PAIRS: three consecutive pairs cannot
    establish a trend, and saying so is more useful than a verdict.
    """
    P = params_df
    if P is None or P.empty:
        return pd.DataFrame()
    rel = P[P["reliable"] == True] if "reliable" in P else P   # noqa: E712
    tol = COHERENCE_BAND_TOL_MV / 1000.0

    # A PAIR THAT CROSSES A RATE CHANGE IS NOT A PAIR. This audit compares
    # CONSECUTIVE cycles and asks whether a peak moved more than a redox
    # centre should and whether its area more than doubled. Across a rate
    # boundary both are the correct answer: on
    # `JQ_NNM_C-rate_2-4.2V_C_29042026` cycle 25 -> 26 steps from 2C to 5C,
    # where the area legitimately halves and the centre legitimately moves
    # by a hundred millivolts of overpotential. Counting those as
    # incoherence took the median drift from the 0.6-1.8 mV of the
    # constant-rate NNM triplicate to 5.0 mV, and put `f_area_over_2x` at
    # 0.2-0.5 on nearly every reference — condemning real features for
    # obeying Ohm's law.
    #
    # So cross-block pairs are DROPPED, not counted. The peak is then judged
    # only on cycle pairs measured at the same current, which is the
    # comparison the audit was always meant to make.
    _rate_of = {}
    if rate_protocol and rate_protocol.get("is_variable"):
        for _bi, _b in enumerate(rate_protocol.get("blocks") or []):
            for _c in _b.get("cycles", ()):
                _rate_of[int(_c)] = _bi
        for _c in (rate_protocol.get("transition_cycles") or []):
            _rate_of[int(_c)] = None
    _n_cross = 0

    rows = []
    for step in ("Charge", "Discharge"):
        ref = detection.reference_peaks.get(step)
        if ref is None or ref.empty:
            continue
        _rel_step = rel[rel["step"] == step]
        _assign = _assign_to_reference(
            _rel_step, [float(v) for v in ref["voltage"]], tol)
        for _ri, (_, rr) in enumerate(ref.iterrows()):
            rv = float(rr["voltage"])
            rec = dict(step=step, reference_voltage=rv,
                       is_truncated=bool(rr.get("is_truncated", False)),
                       pairs=0, median_drift_mV=np.nan,
                       f_drift_over_20mV=np.nan, f_area_over_2x=np.nan,
                       verdict="never recovered")
            _rows = {c: i for (c, k), i in _assign.items() if k == _ri}
            if not _rows:
                rows.append(rec)
                continue
            # Whole ROWS, indexed by cycle. `groupby("cycle").first()` takes
            # the first non-null value per column INDEPENDENTLY, so a single
            # "row" could be assembled from a centre in one component and an
            # area in another.
            b = _rel_step.loc[[_rows[c] for c in sorted(_rows)]].copy()
            b.index = sorted(_rows)
            cyc = list(b.index)
            dv, ar = [], []
            for a, c in zip(cyc, cyc[1:]):
                if c - a != 1:
                    continue
                if _rate_of:
                    _ra, _rc = _rate_of.get(int(a)), _rate_of.get(int(c))
                    if _ra is None or _rc is None or _ra != _rc:
                        _n_cross += 1
                        continue
                dv.append((b.loc[c, "centre"] - b.loc[a, "centre"]) * 1000)
                _ba = _area(b)
                a0 = _ba.loc[a]
                ar.append(_ba.loc[c] / a0
                          if (pd.notna(a0) and a0) else np.nan)
            if not dv:
                rec["verdict"] = "never in consecutive cycles"
                rows.append(rec)
                continue
            dv = np.asarray(dv, float)
            ar = np.asarray(ar, float)
            f_v = float((np.abs(dv) > COHERENCE_DRIFT_IMPLAUSIBLE_MV).mean())
            with np.errstate(invalid="ignore"):
                f_a = float(np.nanmean((ar > COHERENCE_AREA_RATIO_MAX)
                                       | (ar < 1 / COHERENCE_AREA_RATIO_MAX)))
            med = float(np.median(np.abs(dv)))
            if len(dv) < COHERENCE_MIN_PAIRS:
                verdict = "not assessable"
            elif med > COHERENCE_MEDIAN_SUSPECT_MV or f_v > 0.5 or f_a > 0.5:
                verdict = "NOT trend-worthy"
            elif med <= COHERENCE_MEDIAN_COHERENT_MV:
                verdict = "coherent"
            else:
                verdict = "questionable"
            rec.update(pairs=len(dv), median_drift_mV=med,
                       f_drift_over_20mV=f_v, f_area_over_2x=f_a,
                       verdict=verdict)
            rows.append(rec)

    C = pd.DataFrame(rows)
    if verbose and _n_cross:
        print(f"    {_n_cross} consecutive-cycle pair(s) crossing a rate "
              f"change excluded — a peak that moves when the current "
              f"changes is not drifting.")
    if verbose and not C.empty:
        print("\n" + section(f"  Fit coherence (profile {profile}, "
                              f"{len(P)} components, {len(rel)} reliable)"))
        print(f"    {'step':<10}{'ref V':>8}{'pairs':>7}{'median drift':>14}"
              f"{'>20 mV':>9}{'area >2x':>10}  verdict")
        for r in C.itertuples():
            med = ("      —" if not np.isfinite(r.median_drift_mV)
                   else f"{r.median_drift_mV:9.1f} mV")
            fv = ("    —" if not np.isfinite(r.f_drift_over_20mV)
                  else f"{r.f_drift_over_20mV:8.0%}")
            fa = ("     —" if not np.isfinite(r.f_area_over_2x)
                  else f"{r.f_area_over_2x:9.0%}")
            note = ("TRUNCATED; " if r.is_truncated else "") + r.verdict
            print(f"    {r.step:<10}{r.reference_voltage:>8.3f}{r.pairs:>7}"
                  f"{med:>14}{fv:>9}{fa:>10}  {note}")
        tally = C["verdict"].value_counts()
        print("    " + "   ".join(f"{int(tally.get(k, 0))} {k}" for k in
                                  ("coherent", "questionable",
                                   "NOT trend-worthy", "not assessable"))
              + f"   ({len(C)} peak(s) examined)")
    return C


# =============================================================================
# 5. MECHANISM AND INTEGRITY  —  NEW IN 1.9.0
# =============================================================================

@dataclass
class Integrity:
    """What the charge that went backwards in one half-cycle was doing."""
    cycle: int
    step: str
    capacity: float
    n_records: int
    plateau_charge: float = 0.0       # mAh/g, on a two-phase plateau
    cv_charge: float = 0.0            # mAh/g, at the limit with current tapering
    parasitic_charge: float = 0.0     # mAh/g, full current, no redox to explain it
    n_reversals: int = 0
    theoretical_mAh_g: float = float("nan")
    capacity_ratio: float = float("nan")   # capacity / theoretical
    # The threshold ACTUALLY USED, so the `over_theoretical` property and the
    # banding cannot disagree. Reading the module constant in the property
    # while the banding read the argument let a half-cycle be simultaneously
    # not-ANOMALOUS and over_theoretical=True.
    over_theoretical_fraction: float = OVER_THEORETICAL_FRACTION
    n_unknown_charge: int = 0
    outside_window_charge: float = 0.0   # mAh/g, reversed outside the window
    band: str = "unknown"
    reason: str = ""

    @property
    def over_theoretical(self):
        return (np.isfinite(self.capacity_ratio)
                and self.capacity_ratio > self.over_theoretical_fraction)

    @property
    def parasitic_fraction(self):
        if not np.isfinite(self.capacity) or self.capacity <= 0:
            return np.nan
        return self.parasitic_charge / self.capacity


def classify_reversals(sig, *, window, plateau_fraction=PLATEAU_DQDV_FRACTION,
                       limit_tol_mV=LIMIT_TOL_MV):
    """
    Say what each reversal in one `signal.HalfCycleSignal` was.

    Returns a list of dicts: the `Reversal` fields plus `mechanism` and the
    |dQ/dV| there as a fraction of the half-cycle's own maximum.

    The plateau test reads the CURVE, not a peak list — which is why integrity
    does not depend on detection, and why it transfers between a chemistry with
    one sharp two-phase peak and one with six broad solid-solution features.
    A peak-proximity test does not: with a 100 mV tolerance and six peaks across
    1.4 V, almost every voltage counts as "at a peak", and with the peaks' own
    half-maximum bands, LTO's plateau — which is many times wider than its
    peak's FWHM — counts as parasitic in half its healthy half-cycles.
    """
    if sig.voltage.size == 0 or not sig.reversals:
        return []
    a = np.abs(np.asarray(sig.dqdv, float))
    amax = float(np.nanmax(a)) if a.size else np.nan
    if not np.isfinite(amax) or amax <= 0:
        return []
    vlo, vhi = window
    tol = limit_tol_mV / 1000.0

    # How far apart consecutive samples of the TRIMMED curve are. The
    # reversals are found on the raw, untrimmed, acquisition-ordered
    # records, so a reversal inside the trimmed edge band — which is exactly
    # where a constant-voltage hold lives — has no corresponding sample, and
    # an unbounded argmin silently returned the window edge, whose |dQ/dV|
    # has nothing to do with it.
    _dv = (np.diff(np.sort(sig.voltage)) if sig.voltage.size > 1
           else np.array([np.inf]))
    _dv = _dv[_dv > 0]
    _reach = (5.0 * float(np.median(_dv)) if _dv.size else np.inf)

    out = []
    for r in sig.reversals:
        mid = 0.5 * (r.v_start + r.v_end)
        j = int(np.argmin(np.abs(sig.voltage - mid)))
        near = abs(float(sig.voltage[j]) - mid) <= _reach
        share = float(a[j] / amax) if near else np.nan
        at_limit = (mid >= vhi - tol) or (mid <= vlo + tol)
        if not r.at_full_current:
            mech = "cv_hold" if at_limit else "current_taper"
        elif at_limit:
            # Tested BEFORE the plateau share. A reversal at the cell's own
            # cut-off is a hold whether or not |dQ/dV| happens to be high
            # there; calling it a plateau misattributed the charge at the
            # one voltage where the protocol explains it.
            mech = "cv_hold"
        elif np.isfinite(share) and share >= plateau_fraction:
            mech = "plateau"
        elif not np.isfinite(share):
            # Outside the analysed window: no evidence either way.
            mech = "outside_window"
        else:
            mech = "parasitic"
        out.append(dict(i0=r.i0, i1=r.i1, charge=r.charge, v_start=r.v_start,
                        v_end=r.v_end, span_mV=r.span_mV, seconds=r.seconds,
                        at_full_current=r.at_full_current,
                        n_records=r.n_records, voltage=mid,
                        dqdv_share=share, mechanism=mech))
    return out


def half_cycle_integrity(sig, *, window,
                         plateau_fraction=PLATEAU_DQDV_FRACTION,
                         limit_tol_mV=LIMIT_TOL_MV,
                         anomalous_fraction=PARASITIC_ANOMALOUS_FRACTION,
                         min_records=INTEGRITY_MIN_RECORDS,
                         theoretical_mAh_g=None,
                         over_theoretical=OVER_THEORETICAL_FRACTION):
    """
    One half-cycle's band, from the charge it could not account for.

    Two independent tests, and either can condemn a half-cycle:

      the CURVE       where the reversed charge sat — plateau, hold, or
                      nothing that accounts for it
      the TOTAL       whether more charge passed than the material can hold

    The second needs no dQ/dV at all and catches what the first cannot: a
    half-cycle can put its charge somewhere that looks like a redox plateau
    and still be passing four times the theoretical capacity.

    clean            no charge passed backwards away from a redox process
    suspect          some did, but under `anomalous_fraction` of this
                     half-cycle's own capacity
    ANOMALOUS        at or above it
    too few records  the half-cycle is too short to read. NOT the same as
                     clean, and `detect.SOUND_BANDS` excludes it for that
                     reason: it means we could not tell.

    Every comparison is against this half-cycle itself. The two thresholds
    1.8.x tried — `f_wrong_sign` and `f_voltage_backwards` — were set on a
    pooled distribution and transferred neither between the three cells of one
    triplicate nor between chemistries. A fraction of a half-cycle's own
    capacity cannot fail that way.
    """
    g = Integrity(cycle=int(sig.cycle), step=str(sig.step),
                  capacity=float(sig.capacity),
                  n_records=int(sig.n_records_raw),
                  over_theoretical_fraction=float(over_theoretical))
    if sig.n_records_raw < min_records or sig.voltage.size == 0:
        g.band = "too few records"
        g.reason = (f"{sig.n_records_raw} records"
                    if sig.n_records_raw < min_records
                    else "no usable curve")
        return g

    events = classify_reversals(sig, window=window,
                                plateau_fraction=plateau_fraction,
                                limit_tol_mV=limit_tol_mV)
    g.n_reversals = len(events)
    n_unknown = 0
    for e in events:
        if not np.isfinite(e["charge"]):
            # `signal.reversals` sets charge=NaN whenever the capacity
            # column is absent. Folding that into a zero made
            # `parasitic_charge` 0 and the verdict "clean" — a positive
            # all-clear derived from no information at all.
            n_unknown += 1
            continue
        q = e["charge"]
        if e["mechanism"] == "plateau":
            g.plateau_charge += q
        elif e["mechanism"] in ("cv_hold", "current_taper"):
            g.cv_charge += q
        elif e["mechanism"] == "outside_window":
            g.outside_window_charge += q
        else:
            g.parasitic_charge += q
    g.n_unknown_charge = n_unknown

    # --- the total, against what the material can hold --------------------
    # `if theoretical_mAh_g` turned 0.0 into NaN, and NaN disables the
    # over-theoretical guard completely — `x > 1.2 * nan` is False for every
    # x. Cell 2 now rejects a non-positive capacity on the declarative path;
    # this closes the same hole on the interactive one, where a typed 0 could
    # still get through. An unusable value is NaN either way, but it is now
    # NaN because it was unusable, not because it was falsy.
    try:
        _theo = float(theoretical_mAh_g)
    except (TypeError, ValueError):
        _theo = np.nan
    g.theoretical_mAh_g = _theo if np.isfinite(_theo) and _theo > 0 else np.nan
    if (np.isfinite(g.theoretical_mAh_g) and g.theoretical_mAh_g > 0
            and np.isfinite(g.capacity)):
        g.capacity_ratio = g.capacity / g.theoretical_mAh_g
        if g.capacity_ratio > over_theoretical:
            g.band = "ANOMALOUS"
            g.reason = (f"{g.capacity:.0f} mAh/g passed — "
                        f"{g.capacity_ratio:.1f}x the theoretical capacity of "
                        f"{g.theoretical_mAh_g:.0f} mAh/g. More charge went in "
                        f"than the material can hold, so the intended reaction "
                        f"does not account for it")
            return g

    f = g.parasitic_fraction
    if n_unknown:
        # Excluded from detect.SOUND_BANDS for the same reason as
        # "too few records": we could not tell.
        g.band = "unknown"
        g.reason = (f"{n_unknown} of {g.n_reversals} reversal(s) carry no "
                    f"charge figure — the capacity counter is missing, so "
                    f"how much charge went backwards cannot be said")
    elif not np.isfinite(f):
        g.band = "unknown"
        g.reason = "no counter capacity to compare against"
    elif g.parasitic_charge <= 0 and np.isfinite(g.capacity_ratio) \
            and g.capacity_ratio > 1.0:
        # OVER THEORETICAL IS NOT CLEAN, even under the escalation threshold.
        # `over_theoretical` is 1.2, a tolerance for the uncertainty in the
        # active mass and in the quoted theoretical capacity — not a licence
        # to pass 1.0-1.2x and be called clean. NNM cell A cycle 9 passed
        # 1.069x with ten voltage reversals and 25.6 mAh/g going backwards
        # onto a plateau, and was reported as "no charge passed backwards
        # away from a redox process": a sentence its own `plateau_charge`
        # column contradicts.
        g.band = "suspect"
        g.reason = (f"{g.capacity:.0f} mAh/g passed — {g.capacity_ratio:.2f}x "
                    f"the theoretical {g.theoretical_mAh_g:.0f} mAh/g. No "
                    f"charge went backwards away from a redox process, but "
                    f"more went in than the material can hold"
                    + (f", and {g.plateau_charge + g.cv_charge:.1f} mAh/g of "
                       f"it passed backwards during "
                       f"{g.n_reversals} voltage reversal(s)"
                       if (g.plateau_charge + g.cv_charge) > 1.0 else "")
                    + ". Below the "
                    f"{over_theoretical:.1f}x at which this is called a fault, "
                    f"so it is flagged rather than escalated")
    elif g.parasitic_charge <= 0:
        g.band = "clean"
        # SAY WHAT WAS FOUND, not only what was not. The old sentence was
        # true and read as "nothing happened here" on half-cycles carrying
        # tens of mAh/g backwards on a plateau. Where a plateau is expected
        # physics that is fine; whether it IS expected depends on the
        # mechanism, which is decided two cells later — so this states the
        # measurement and leaves the reading to the page that knows.
        _back = g.plateau_charge + g.cv_charge
        g.reason = ("no charge passed backwards away from a redox process"
                    if _back <= 1.0 else
                    f"no charge passed backwards away from a redox process; "
                    f"{_back:.1f} mAh/g did go backwards during "
                    f"{g.n_reversals} voltage reversal(s), on a plateau or at "
                    f"the voltage limit")
    elif f >= anomalous_fraction:
        g.band = "ANOMALOUS"
        g.reason = (f"{g.parasitic_charge:.1f} mAh/g ({f:.1%} of this "
                    f"half-cycle's capacity) passed at full current where "
                    f"|dQ/dV| was below {plateau_fraction:.0%} of its own "
                    f"maximum")
    else:
        g.band = "suspect"
        g.reason = (f"{g.parasitic_charge:.2f} mAh/g ({f:.2%}) unaccounted for")
    return g


# --- is this a cell with a fault, or a cell with some bad cycles? ---------
# The run counted anomalous half-cycles and stopped there. "9 anomalous
# half-cycles of 160" reads like a handful of bad readings; what it was on NNM
# cell A is a cell that stalls on charge over and over, from the first cycle to
# the sixty-ninth, accepting up to four times the charge the material can hold
# and giving 87% of it back. Those are the same numbers and opposite readings,
# and the difference between them is RECURRENCE — which nothing aggregated.
#
# Three cases have to be told apart, and the run already holds everything
# needed to do it:
#   formation   flagged cycles all early, none after the cell settled
#   isolated    one or two, scattered, no pattern
#   recurring   enough of them, spread across the life of the cell
RECURRENCE_MIN_EVENTS = 3        # fewer than this is not a pattern
RECURRENCE_MIN_SPAN_FRACTION = 0.5   # of the cycles analysed
RECURRENCE_FORMATION_CYCLES = 5      # "early" if it is all inside this


def cell_integrity_verdict(integrity, *, reference_cycle=None,
                           n_cycles=None):
    """
    One verdict for the CELL, from the per-half-cycle integrity table.

    Returns a dict with `verdict` (one of "sound", "formation", "isolated",
    "recurring"), the numbers behind it, and `sentence` — the thing a person
    should be told. Never raises on an empty or partial table.
    """
    # "UNKNOWN" IS THE DEFAULT, NOT "SOUND". A table this cannot read must not
    # come back looking like a clean cell: the first version returned "sound"
    # for an absent or malformed table, and the page rendered nothing for
    # "sound", so a failure to assess and a cell with nothing wrong produced
    # the same silence. That is the shape of a false all-clear.
    out = dict(verdict="unknown", n_flagged=0, n_total=0, first=None,
               last=None, span=0, gaps=(), worst_ratio=np.nan,
               worst_cycle=None, steps=(),
               sentence="There is no cycle-integrity table for this dataset, "
                        "so whether its cycles are sound has not been "
                        "assessed. That is not the same as nothing being "
                        "wrong.")
    if integrity is None or getattr(integrity, "empty", True):
        return out
    d = integrity.copy()
    if "band" not in d or "cycle" not in d:
        out["sentence"] = ("the cycle-integrity table is missing the columns "
                           "this reads, so it could not be assessed")
        return out
    out["n_total"] = int(len(d))
    # FLAGGED means the run itself said something was wrong here — the bands
    # it escalates on, plus anything over theoretical, which is the physical
    # impossibility and is not always escalated (see `classify_integrity`).
    # TWO THRESHOLDS, DELIBERATELY, AND THE SENTENCE MUST SAY WHICH IT USED.
    # This tripwire is `capacity_ratio > 1.0` — any charge at all beyond what
    # the material can hold — while `Integrity.over_theoretical`, and the
    # report finding built on it, escalate only past
    # `OVER_THEORETICAL_FRACTION` (1.2), a tolerance for the uncertainty in the
    # theoretical capacity and in the electrode mass. Both are wanted, and
    # `test_cell_verdict` pins this one at 1.0. What was wrong was that one
    # page carried both lists under the same words: NNM cell A read "4
    # half-cycle(s) passed more charge than the material can hold ... Cycles:
    # 1, 13, 19, 69" three findings above "5 cycles were flagged on charge —
    # 1, 13, 19, 26, 69", and nothing reconciled them. Cycle 26 was at 1.026x:
    # over one, inside the tolerance. The counts are separated below so the
    # sentence can name the difference instead of hiding it.
    _ratio = (pd.to_numeric(d.get("capacity_ratio"), errors="coerce")
              if "capacity_ratio" in d else pd.Series(np.nan, index=d.index))
    _over = (_ratio > 1.0).fillna(False)
    _escalated = (_ratio > OVER_THEORETICAL_FRACTION).fillna(False)
    out["n_over_theoretical"] = int(_over.sum())
    out["n_over_theoretical_escalated"] = int(_escalated.sum())
    out["n_over_theoretical_within_tolerance"] = int((_over & ~_escalated).sum())
    flag = d["band"].astype(str).eq("ANOMALOUS") | _over.fillna(False)
    f = d[flag]
    out["n_flagged"] = int(len(f))
    if f.empty:
        out["verdict"] = "sound"
        out["sentence"] = (f"No half-cycle of {out['n_total']} was flagged: "
                           f"none passed more charge than the material can "
                           f"hold, and none lost charge away from a redox "
                           f"process.")
        return out
    out["verdict"] = "sound"        # provisional; refined below
    cyc = sorted(int(c) for c in pd.to_numeric(f["cycle"], errors="coerce")
                 .dropna().unique())
    out["first"], out["last"] = cyc[0], cyc[-1]
    out["span"] = cyc[-1] - cyc[0]
    out["gaps"] = tuple(np.diff(cyc).tolist())
    out["steps"] = tuple(sorted(set(f["step"].astype(str))))
    if "capacity_ratio" in f:
        _r = pd.to_numeric(f["capacity_ratio"], errors="coerce")
        if _r.notna().any():
            out["worst_ratio"] = float(_r.max())
            out["worst_cycle"] = int(f.loc[_r.idxmax(), "cycle"])
    _n = int(n_cycles or (pd.to_numeric(d["cycle"], errors="coerce").max() or 0))
    _ref = int(reference_cycle or RECURRENCE_FORMATION_CYCLES)
    _late = [c for c in cyc if c > max(_ref, RECURRENCE_FORMATION_CYCLES)]
    # TWO DECIMALS, BECAUSE THE GATE IS AT THE SECOND ONE. This clause exists
    # only where `worst_ratio > 1.0`, and at one decimal place a cycle flagged
    # at 1.0265x printed as "up to 1.0x the theoretical capacity" — a sentence
    # that says nothing exceeded theoretical, inside a sentence about a
    # half-cycle that did.
    _worst = (f" and up to {out['worst_ratio']:.2f}x the theoretical capacity"
              f" (cycle {out['worst_cycle']})"
              if np.isfinite(out["worst_ratio"]) and out["worst_ratio"] > 1.0
              else "")
    _where = (f" on {' and '.join(s.lower() for s in out['steps'])}"
              if out["steps"] else "")
    # WHY THIS LIST IS LONGER THAN THE ONE ABOVE IT. Said here rather than left
    # for the reader to work out from two sets of cycle numbers.
    _tol_n = int(out.get("n_over_theoretical_within_tolerance") or 0)
    _tol = (f" {_tol_n} of them "
            + ("is" if _tol_n == 1 else "are")
            + f" over the theoretical capacity by less than the "
              f"{OVER_THEORETICAL_FRACTION - 1.0:.0%} allowed for uncertainty "
              f"in that capacity and in the electrode mass, so "
            + ("it is" if _tol_n == 1 else "they are")
            + " flagged here and not counted as passing more charge than the "
              "material can hold."
            if _tol_n else "")

    if not _late:
        out["verdict"] = "formation"
        out["sentence"] = (
            f"{len(cyc)} cycle(s) were flagged{_where}, all of them at or "
            f"before cycle {cyc[-1]}{_worst}.{_tol} Nothing was flagged after "
            f"the cell settled, so this reads as formation rather than a "
            f"fault.")
        return out

    _long = _n and out["span"] >= RECURRENCE_MIN_SPAN_FRACTION * _n
    if len(cyc) >= RECURRENCE_MIN_EVENTS and _long:
        _gap = int(np.median(out["gaps"])) if out["gaps"] else 0
        out["verdict"] = "recurring"
        out["sentence"] = (
            f"{len(cyc)} of {_n} cycles were flagged{_where}, from cycle "
            f"{cyc[0]} to cycle {cyc[-1]} — every "
            f"{_gap} cycles or so, across "
            f"{100 * out['span'] / _n:.0f}% of the life of the cell{_worst}."
            f"{_tol} "
            f"That is not settling-in and it is not a handful of bad "
            f"readings: it is a fault that keeps happening. Capacity and "
            f"retention measured on this cell describe the fault as much as "
            f"they describe the material.")
        return out

    # THREE OR MORE EVENTS IS A PATTERN EVEN IF IT STOPPED. NNM cell B was
    # flagged at cycles 1, 12, 16, 20 and 25 on a record running to 67 —
    # "too few to call a pattern" was plainly the wrong sentence for that. A
    # run of events that ends is its own finding, and often a more interesting
    # one than a fault that simply continues.
    _list = (", ".join(str(c) for c in cyc[:8])
             + (" ..." if len(cyc) > 8 else ""))
    if len(cyc) >= RECURRENCE_MIN_EVENTS:
        _gap = int(np.median(out["gaps"])) if out["gaps"] else 0
        out["verdict"] = "clustered"
        out["sentence"] = (
            f"{len(cyc)} cycles were flagged{_where} — {_list} — every "
            f"{_gap} cycles or so{_worst}, and then nothing"
            + (f" for the remaining {_n - cyc[-1]} cycles" if _n > cyc[-1]
               else "") + "." + _tol + " A run of events that stopped is not "
            "the same as a cell that is fine: something was happening and then "
            "was not. "
            "Worth knowing what changed at cycle "
            f"{cyc[-1]}.")
        return out

    out["verdict"] = "isolated"
    out["sentence"] = (
        f"{len(cyc)} cycle(s) were flagged{_where}, at cycle(s) "
        + _list + _worst
        + "." + _tol
        + " Too few to call a pattern — read them as individual events.")
    return out


def last_useful_cycle(capacities, *, fraction=DEAD_CELL_CAPACITY_FRACTION,
                      run=DEAD_CELL_RUN):
    """
    The last cycle before the cell stopped delivering, and why.

    `capacities` is `{(cycle, step): mAh/g}` — `Dataset.capacity` for every
    half-cycle, or the `capacity` column of an integrity report.

    Returns `(cycle, reason)`. `cycle` is None when the cell never dies, which
    is the common case and must not be confused with dying at cycle 0.

    This exists to stop the FIT, not the plots. Fitting pseudo-Voigt components
    to a flat line consumes most of a run's time and every one of those fits is
    discarded downstream as unreliable — but the heatmaps and the waterfall
    should still show the whole record, because the cell going flat at cycle
    130 is exactly what the reader needs to see.
    """
    disc = {int(c): v for (c, s), v in capacities.items()
            if str(s).lower().startswith("d") and np.isfinite(v)}
    if len(disc) < run + 1:
        return None, "too few cycles to judge"
    cycles = sorted(disc)
    # The 90th percentile, NOT the median. On P3 cell B the cell dies at cycle
    # 67 and the record runs to 220, so 153 of its 220 cycles are dead and the
    # median discharge capacity is 0.04 mAh/g — a floor of 2% of that is zero
    # and the test never fires. The 90th percentile asks what this cell
    # delivered WHEN IT WAS WORKING, which is 78.6 mAh/g, and is unmoved by how
    # much dead record follows.
    ref = float(np.percentile([disc[c] for c in cycles], 90))
    if not np.isfinite(ref) or ref <= 0:
        return None, "no usable reference capacity"
    floor = fraction * ref
    dead = 0
    for i, c in enumerate(cycles):
        if disc[c] < floor:
            dead += 1
            if dead >= run:
                first = cycles[i - dead + 1]
                # A cell that is dead from its FIRST cycle has no earlier
                # cycle to fit, but returning None there means "never dies"
                # to every caller — including the notebook, which tests the
                # value for truthiness — so the 220 flat cycles this
                # function exists to skip were fitted after all. Return the
                # first cycle itself: one cycle is fitted, not all of them,
                # and the reason says what happened.
                prev = (cycles[cycles.index(first) - 1]
                        if first != cycles[0] else cycles[0])
                return prev, (f"discharge capacity fell below {floor:.2f} "
                              f"mAh/g ({fraction:.0%} of this cell's working "
                              f"capacity, {ref:.1f}) for {run} consecutive "
                              f"cycles from cycle {first}")
        else:
            dead = 0
    return None, f"never fell below {floor:.2f} mAh/g for {run} cycles running"


# How far short of the cut-off a half-cycle may end and still count as
# finished. A cycler terminates a half-cycle ON VOLTAGE, so a completed one
# lands on the limit to within the resolution of the ADC: on the LTO
# triplicate every falling half-cycle ends at 1.2000 V exactly and every
# rising one within 0.6 mV of 2.4995. The two unfinished ones stop 341 mV
# and 913 mV short. Three orders of magnitude separate the cases, so the
# threshold is not a tuned parameter.
TERMINATION_TOL_MV = 20.0
# ...or this multiple of the spread the completed half-cycles themselves
# show, whichever is larger, for a cycler with a coarser voltage record.
TERMINATION_TOL_SPREADS = 5.0
# Only the most recent same-direction half-cycles set the reference, so a
# protocol that changes its voltage window part-way through does not average
# the old limit with the new one.
TERMINATION_REFERENCE_N = 10


# --- how long did the half-cycle last? -----------------------------------
# The voltage test says a half-cycle stopped short of its cut-off. It cannot
# say WHY, and the two reasons are not the same kind of event:
#
#   still running     the export was written mid-flight. Benign. The next
#                     export has the finished half-cycle. Its duration is a
#                     plausible FRACTION of a normal one and it is the last
#                     record in the file.
#
#   terminated early  the half-cycle ran for a normal length of time and then
#                     stopped on something other than voltage — a step timer,
#                     a safety limit, an aborted schedule. That is a FINDING
#                     about the experiment and belongs in the integrity table,
#                     not in a footnote.
#
# Duration separates them, and it is free: every export carries a time column.
# Where the cycler records its own step-end reason, that column settles the
# question outright and is preferred to any inference — Neware writes one on
# some schedules, so it is looked for by name and used when found.
TERMINATION_DURATION_FRACTION = 0.90
STEP_END_REASON_COLUMNS = ("Step end reason", "End reason", "Stop reason",
                           "Cut-off reason", "Step Stop Reason")
_TIME_COLUMNS = ("Time", "Cumulative Time", "Test Time", "Total Time")


def _to_seconds(series):
    """A cycler time column as float seconds. NaN where it cannot be read."""
    s = pd.Series(series)
    if s.empty:
        return s.astype(float)
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    td = pd.to_timedelta(s, errors="coerce")
    if td.notna().any():
        return td.dt.total_seconds()
    # "D-HH:MM:SS", which pandas does not parse.
    def _one(x):
        try:
            t = str(x).strip()
            days = 0
            if "-" in t:
                d, t = t.split("-", 1)
                days = int(d)
            parts = [float(q) for q in t.split(":")]
            while len(parts) < 3:
                parts.insert(0, 0.0)
            return (days * 86400.0 + parts[0] * 3600.0
                    + parts[1] * 60.0 + parts[2])
        except Exception:                              # noqa: BLE001
            return np.nan
    return s.map(_one).astype(float)


def _half_cycle_seconds(frame):
    """Elapsed time of one half-cycle, in seconds. NaN if no time column."""
    for col in _TIME_COLUMNS:
        if col not in frame.columns:
            continue
        t = _to_seconds(frame[col]).dropna()
        if t.size < 2:
            continue
        # `Time` restarts at each step, so its maximum IS the duration;
        # a cumulative column needs the span. Both are covered by taking
        # the larger of the two, which for a resetting column is the max
        # and for a cumulative one is the span.
        return float(max(t.max(), t.max() - t.min()))
    return np.nan


def _step_end_reason(frame):
    """The cycler's own termination reason for this step, if it wrote one."""
    for col in STEP_END_REASON_COLUMNS:
        if col in frame.columns:
            v = frame[col].dropna()
            if len(v):
                txt = str(v.iloc[-1]).strip()
                if txt and txt.lower() not in ("nan", "none", "-"):
                    return txt
    return ""


def termination_check(dataset, *, tol_mV=None):
    """
    Which half-cycles ended before reaching their cut-off voltage, and how far short.

    Returns a DataFrame: cycle, step, direction, v_end, v_limit, short_mV,
    reached, is_last.

    A galvanostatic half-cycle ENDS ON VOLTAGE — the cycler drives until the
    cut-off and then turns round — so "did it reach the limit" is the direct
    question, asked of the quantity the instrument actually terminates on.
    Capacity is a poor proxy for it: a cell genuinely fading, or one caught
    four fifths of the way through, is indistinguishable by capacity from a
    finished half-cycle, and on this very triplicate an 80%-of-the-previous
    rule separated a truncated half-cycle from a complete one by five
    percentage points. On voltage the same two cases are 341 mV apart.

    The limit is measured from the record rather than read from the
    parameters, because the file's own `Volt. upper`/`Volt. lower` are
    channel safety limits — an LTO cell cycled 1.2-2.5 V reports 0-3.25 V.
    Half-cycles are grouped by DIRECTION OF TRAVEL, not by step label: every
    half-cycle whose voltage rises ends at the upper cut-off whatever it is
    called, which also means an anode dataset with swapped labels needs no
    special case. A constant-voltage hold still reaches the limit and still
    counts as finished.
    """
    keys = sorted(dataset.half_cycle_keys())
    rows = []
    for k in keys:
        hc = dataset.half_cycle(*k)
        v = pd.to_numeric(hc["Voltage"], errors="coerce").dropna().to_numpy()
        if v.size == 0:
            continue
        rows.append(dict(cycle=int(k[0]), step=str(k[1]),
                         direction=float(dataset.step_direction(*k)),
                         v_end=float(v[-1]), order=keys.index(k),
                         duration_s=_half_cycle_seconds(hc),
                         end_reason=_step_end_reason(hc)))
    T = pd.DataFrame(rows)
    if T.empty:
        return T

    # The LAST half-cycle in ACQUISITION order — the only one that can still
    # be running, because every earlier one finished before the next began.
    fr = dataset.frame
    last_key = (int(fr["Cycle"].iloc[-1]), str(fr["Step"].iloc[-1]))
    T["is_last"] = [(c, st) == last_key for c, st in zip(T.cycle, T.step)]

    T["v_limit"] = np.nan
    T["short_mV"] = np.nan
    T["duration_frac"] = np.nan
    T["reached"] = True
    for d in (1.0, -1.0):
        m = T["direction"] == d
        if not m.any():
            continue
        # Reference from the OTHER half-cycles going the same way, most
        # recent first, so a half-cycle is never compared with itself.
        for idx in T.index[m]:
            peers = T[m & (T.index != idx)].sort_values("order")
            # THE NEAREST PEERS, NOT THE LAST ONES IN THE FILE.
            # `peers.tail(N)` took the N most recent same-direction half-cycles
            # in the whole record for EVERY half-cycle, so an early one was
            # judged against late ones. On a protocol that widens its window
            # part-way — cycles 1-50 to 4.20 V, 51-100 to 4.40 V, which is an
            # ordinary thing to do — every one of the first fifty is 200 mV
            # short of a limit it was never asked to reach, and all fifty are
            # published as "terminated early, not on voltage". The reference
            # has to be local to be a reference at all.
            _o = float(T.at[idx, "order"])
            peers = peers.iloc[
                (peers["order"] - _o).abs().to_numpy().argsort(
                    kind="mergesort")[:TERMINATION_REFERENCE_N]]
            if len(peers) < 2:
                continue
            lim = float(peers["v_end"].median())
            # A ROBUST SPREAD. This was `max - min`, a range, multiplied by
            # five: the widest possible statistic, and one that the very
            # anomalies being detected inflate. A cell with two stalled
            # half-cycles among ten peers — ending at 4.00 and 3.90 V against
            # 4.20 — gave a 300 mV range and a 1500 mV tolerance, so each
            # stall masked the other AND every clean half-cycle in the dataset
            # inherited the same 1500 mV, after which `early_terminations`
            # returns empty and a truncated final half-cycle is fitted as
            # though it had finished. The interquartile range of the peers
            # survives up to a quarter of them being anomalous, which is the
            # case this function exists for.
            _v = pd.to_numeric(peers["v_end"], errors="coerce").dropna()
            spread = (float(_v.quantile(0.75) - _v.quantile(0.25))
                      if len(_v) >= 4 else
                      float(_v.max() - _v.min()) if len(_v) else 0.0)
            tol = max(TERMINATION_TOL_MV if tol_mV is None else float(tol_mV),
                      TERMINATION_TOL_SPREADS * spread * 1000.0)
            short = d * (lim - T.at[idx, "v_end"]) * 1000.0
            T.at[idx, "v_limit"] = lim
            T.at[idx, "short_mV"] = short
            T.at[idx, "reached"] = bool(short <= tol)
            # How long it ran, against its own peers. A half-cycle caught
            # mid-flight is SHORT; one that stopped on something other than
            # voltage ran a normal length and then stopped.
            dpeers = pd.to_numeric(peers["duration_s"],
                                   errors="coerce").dropna()
            dur = T.at[idx, "duration_s"]
            if len(dpeers) >= 2 and np.isfinite(dur) and dpeers.median() > 0:
                T.at[idx, "duration_frac"] = float(dur / dpeers.median())

    T["termination_class"] = "reached the cut-off"
    short_of = ~T["reached"].astype(bool)
    frac = pd.to_numeric(T["duration_frac"], errors="coerce")
    # SHORT and last in the file = the export caught it. Anything else that
    # stopped short of the cut-off ran a normal length of time and stopped on
    # something else, which is a finding about the experiment.
    caught = short_of & T["is_last"] & (
        frac.isna() | (frac < TERMINATION_DURATION_FRACTION))
    T.loc[short_of, "termination_class"] = "terminated early, not on voltage"
    T.loc[caught, "termination_class"] = "still running when exported"
    # The cycler's own answer wins wherever it wrote one.
    if T["end_reason"].astype(str).str.len().gt(0).any():
        known = T["end_reason"].astype(str).str.len() > 0
        T.loc[known & short_of, "termination_class"] = (
            "ended on: " + T.loc[known & short_of, "end_reason"])
    return T.drop(columns=["order"])


def half_cycles_in_progress(dataset, *, tol_mV=None, checked=None):
    """
    The half-cycle an export was written during, if there is one.

    Returns `(keys, reason)` where `keys` is a set of `(cycle, step)` — at
    most one, because only the last half-cycle in the file can still be
    running.

    A cycler export is routinely taken while the cell is CYCLING, so its
    final half-cycle is usually unfinished. That is a snapshot of a running
    experiment, not a measurement: its curve is truncated, so fitting it
    invents a peak set that changes with every re-export of the same cell,
    and its capacity is not the capacity of anything.

    The granularity is the HALF-CYCLE. On the LTO triplicate, cell B's export
    was written during cycle 11's delithiation; the lithiation half of that
    same cycle had already run to 1.2000 V and is a perfectly good
    measurement. Dropping the whole cycle would throw it away.

    Like `last_useful_cycle` this limits the FIT and not the plots — the
    partial curve still belongs in the heatmaps and the waterfall, because
    seeing where the run has got to is the point.
    """
    # `checked` is a frame this dataset's `termination_check` already
    # produced. Cell 5 asks two questions of it — what was caught in flight,
    # and what stopped early without being caught — and used to recompute the
    # whole thing for each: 10.6 s a time on P3 cell A, 21 s per dataset, for
    # one identical answer.
    T = termination_check(dataset, tol_mV=tol_mV) if checked is None else checked
    if T.empty:
        return set(), "no half-cycles to judge"
    last = T[T["is_last"]]
    if last.empty or bool(last["reached"].iloc[0]):
        return set(), ("the last half-cycle in the file reached its cut-off, "
                       "so nothing was caught mid-flight")
    r = last.iloc[0]
    if not np.isfinite(r["short_mV"]):
        return set(), "too few half-cycles to establish the cut-off voltage"
    if not str(r["termination_class"]).startswith("still running"):
        # Short of the cut-off, but it ran a NORMAL LENGTH OF TIME first. The
        # export did not catch this one; it stopped on something else, and
        # calling that "still running" would file a fault as a footnote.
        return set(), (
            f"cycle {int(r['cycle'])} {r['step']} ended "
            f"{r['short_mV']:.0f} mV short of the cut-off after "
            f"{100 * r['duration_frac']:.0f}% of a normal half-cycle's "
            f"duration — {r['termination_class']}. It is treated as a "
            f"finished half-cycle and reported in the integrity table")
    key = (int(r["cycle"]), str(r["step"]))
    _frac = ("" if not np.isfinite(r["duration_frac"])
             else f" after {100 * r['duration_frac']:.0f}% of a normal "
                  f"half-cycle's duration")
    return {key}, (
        f"cycle {key[0]} {key[1]} ended at {r['v_end']:.4f} V, "
        f"{r['short_mV']:.0f} mV short of the {r['v_limit']:.4f} V cut-off "
        f"that every other half-cycle going the same way reaches{_frac}. "
        f"It is the last half-cycle in the file, so the export was written "
        f"while it was still running. Not a measurement yet: omitted from "
        f"the fit, kept in every figure, and complete in the next export")


def early_terminations(dataset, *, tol_mV=None, checked=None):
    """
    Half-cycles that stopped short of the cut-off but were NOT caught in
    flight — a fault, not a snapshot. See TERMINATION_DURATION_FRACTION.

    Returns the subset of `termination_check`'s frame worth reporting, with
    the columns a reader needs to judge it: how far short, how long it ran
    against its peers, and the cycler's own reason where it wrote one.
    """
    T = termination_check(dataset, tol_mV=tol_mV) if checked is None else checked
    if T.empty:
        return T
    bad = T[~T["reached"].astype(bool)
            & ~T["termination_class"].astype(str).str.startswith(
                "still running")]
    cols = ["cycle", "step", "v_end", "v_limit", "short_mV", "duration_frac",
            "termination_class", "is_last"]
    return bad[[c for c in cols if c in bad.columns]].reset_index(drop=True)


def integrity_report(signals, *, window, name="", verbose=True, **kw):
    """
    Band every half-cycle, and return a lookup that `detect` can use.

    Returns (DataFrame, band) where `band(cycle, step) -> str` is exactly the
    `integrity` callable `detect.detect_all` and `detect.choose_reference_cycle`
    take. In 1.8.7 that callable was a notebook global; here it is a return
    value, which is the whole difference between a testable selector and one
    that was inert on every real run for a fortnight without anyone noticing.
    """
    recs = {}
    rows = []
    for key in sorted(signals):
        g = half_cycle_integrity(signals[key], window=window, **kw)
        recs[(int(g.cycle), str(g.step))] = g.band
        rows.append(dict(dataset=name, cycle=g.cycle, step=g.step,
                         capacity=g.capacity,
                         theoretical_mAh_g=g.theoretical_mAh_g,
                         capacity_ratio=g.capacity_ratio,
                         over_theoretical=g.over_theoretical,
                         n_records=g.n_records,
                         n_reversals=g.n_reversals,
                         plateau_charge=g.plateau_charge,
                         cv_charge=g.cv_charge,
                         parasitic_charge=g.parasitic_charge,
                         parasitic_fraction=g.parasitic_fraction,
                         band=g.band, reason=g.reason))
    R = pd.DataFrame(rows)

    def band(cycle, step):
        return recs.get((int(cycle), str(step)), "unknown")

    if verbose and not R.empty:
        counts = R["band"].value_counts()
        bad = R[R["band"] == "ANOMALOUS"].sort_values("parasitic_fraction",
                                                      ascending=False)
        over = R[R["over_theoretical"] == True]          # noqa: E712
        sane = R[R["over_theoretical"] != True]          # noqa: E712
        print(entry("half-cycles", str(len(R)),
                    ", ".join(f"{int(counts.get(k, 0))} {k}" for k in
                              ("clean", "suspect", "ANOMALOUS",
                               "too few records") if counts.get(k, 0))))
        if not bad.empty:
            tot = bad["parasitic_charge"].sum()
            print(verdict("bad", f"{tot:.0f} mAh/g unaccounted for across "
                                 f"{len(bad)} half-cycle(s)"))
            for r in bad.head(5).itertuples():
                print(bullet(f"cycle {r.cycle} {r.step}: "
                             f"{r.parasitic_charge:.0f} mAh/g of "
                             f"{r.capacity:.0f} ({r.parasitic_fraction:.0%})"))
        if len(over):
            print(verdict("bad", f"{len(over)} half-cycle(s) passed more "
                                 f"charge than the material can hold"))
            print(bullet(f"up to {over['capacity_ratio'].max():.1f}x "
                         f"theoretical. A curve cannot vouch for charge the "
                         f"material could not have held, so these are "
                         f"condemned whatever their shape."))
        if (sane["plateau_charge"] > 0).any():
            p = sane.loc[sane["plateau_charge"] > 0, "plateau_charge"]
            print(entry("plateau reversals", f"{len(p)} of {len(sane)}",
                        f"median {p.median():.1f} mAh/g"))
            print(bullet("A flat two-phase feature, not a fault — counted "
                         "over the half-cycles that stayed within the "
                         "material's capacity."))
        # `> 0` let through holds carrying a thousandth of a mAh/g, which
        # then printed as "median 0.0 mAh/g" — an alarm about zero. The floor
        # is what a cycler can resolve, and the number is printed to enough
        # figures to be read.
        if (R["cv_charge"] > CV_HOLD_REPORT_FLOOR).any():
            c = R.loc[R["cv_charge"] > CV_HOLD_REPORT_FLOOR, "cv_charge"]
            print(entry("constant-voltage hold", f"{len(c)} half-cycle(s)",
                        f"median {c.median():.3g} mAh/g — protocol, "
                        f"not chemistry"))
    return R, band


In [ ]:
# DEFINITIONS H1 — FIGURES: SHARED HELPERS AND THE DQ/DV VIEWS
# =============================================================================
"""
The presentation layer, ported unchanged.

The waterfall, the heatmaps, the key-cycle overlays and the split
charge/discharge panels are the best part of Ratatosk and their output does not
change in 1.9.0. So this module does not reimplement them: every plotting
function below was extracted from the 1.8.7 notebook programmatically and is
byte-for-byte the code that drew the figures you already have.

    from Module 2   plot_dqdv_all_cycles           plot_dqdv_waterfall
                    plot_dqdv_key_cycles_combined  plot_dqdv_heatmap
                    plot_dqdv_key_cycles_split     plot_preprocessing_qc
    from Module 3   plot_detected_peaks
    from Module 4   plot_fit_quality
    from Module 5   plot_tracked_trends
    from Module 6   plot_delta_v   plot_capacity_attribution

    shared          _get_display_name  _force_integer_cycles
                    _charge_label      _discharge_label   _apply_pub_style

How the 1.9.0 objects reach 1.8.7's plotting code
-------------------------------------------------
Those functions take the notebook's dictionaries — `processed_dqdv`,
`detected_peaks`, `fit_results`, `tracked_peaks`, `user_parameters` — and the
alternative to keeping them was rewriting fourteen tested plotting functions
against `Dataset`, `HalfCycleSignal`, `Detection` and `Tracking`, which is
precisely how figures stop being identical.

So the adapters go the other way. `as_processed_dqdv`, `as_detected_peaks`,
`as_fit_results` and `as_tracked` build the 1.8.7 shapes from the 1.9.0
objects. They are small, they are the only new code here, and they are tested.

The one thing that could not be adapted directly
------------------------------------------------
`plot_fit_quality` asks its fit for `.best_fit` and `.eval_components(x=...)` —
that is, for a live lmfit `ModelResult`. 1.9.0 deliberately does not keep one:
a `ModelResult` cannot cross a process boundary through stdlib pickle, and
carrying it would undo the single decision that made parallel fitting possible.

Rather than edit the plotting code, `as_fit_results` supplies a `_FitView`,
which answers both questions from `fitting.evaluate` — the component curves
rebuilt analytically from the stored parameters, through lmfit's own model
classes, with no refit. Verified against the data to 3e-7. The plotting
function is untouched and does not know the difference.

The one change to the fitted model, and why it is not cosmetic
-------------------------------------------------------------
1.9.0 originally prefixed its polynomial baseline `bkg_`; 1.8.7 uses `bg_`.
`plot_fit_quality` decides which component to draw as the dashed black baseline
by testing `'bg_' in comp_name`, which `'bkg_'` fails — the baseline would have
been drawn as a filled peak. The prefix is now `bg_` throughout, matching 1.8.7
everywhere including the parameter names.
"""


import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors            # noqa: F401  (used by ported code)
import matplotlib.patheffects as path_effects
import matplotlib.ticker as mticker    # noqa: F401  (used by ported code)
from matplotlib import gridspec
from matplotlib.colors import Normalize


# =============================================================================
# FIGURE GEOMETRY — Cell 3's globals, as module state
# =============================================================================
# 6.73 in is the RSC double-column width. The ported functions read these as
# globals, exactly as they did in the notebook.
figure_width_inches = 6.73
figure_height_inches = 5.0


def set_figure_size(width_inches=None, height_inches=None):
    """Set the figure geometry every plot in this module inherits."""
    global figure_width_inches, figure_height_inches
    if width_inches is not None:
        figure_width_inches = float(width_inches)
    if height_inches is not None:
        figure_height_inches = float(height_inches)


# =============================================================================
# MODULE 2 DEFAULTS — unchanged
# =============================================================================

# Matplotlib resolves font weights against the weights a font actually ships.
# The DejaVu Sans that ships with matplotlib has no 'medium' face, so every
# figure title emitted 'findfont: Failed to find font weight medium, now
# using 400' and then used 400 anyway. 'normal' IS 400, so this asks for what
# it was always going to get, silently. Set 'bold' or a number if you want
# heavier titles and your font has the face.
FIGURE_TITLE_WEIGHT = 'normal'

# --- the multi-peak fit panels -------------------------------------------
# Where the curve is, for the purpose of framing the panel. The floor is a
# fraction of the panel's own maximum |dQ/dV|, so it adapts to the chemistry
# rather than to an absolute current; the pad is a fraction of the span it
# found; and the minimum width stops a 40 mV LTO peak from filling the axis
# edge to edge with no context around it. See the note in `plot_fit_quality`.
# HOW WIDE AN ERROR BAND IS ALLOWED TO BE DRAWN. Purely a display cap: a
# single standard error of half a volt on a millivolt-scale panel makes every
# other point invisible. It is not a claim about the error, and both figures
# that use it now say on the figure how many points were capped. The numbers
# themselves are unclipped in the CSVs.
# THE HEATMAP'S UNVISITED CELLS, and the envelope that explains them.
# The colour is a dark neutral rather than the colormap's own zero: close
# enough to keep the panel looking like a heat map, far enough that a region
# the cell never entered is not the same pixel as a region where dQ/dV was
# measured to be zero. The envelope is Okabe-Ito sky blue — the one cool
# colour in a warm colormap, so it reads at any density and survives
# deuteranopia, protanopia and greyscale.
# Near the zero end of the ramp, one step off it. The fill is background;
# the ENVELOPE LINE is what announces the collapse, and a large field of a
# distinctly different colour competes with it for the eye without adding
# anything the line does not already say.
HEATMAP_UNVISITED_COLOUR = '#12121A'
HEATMAP_ENVELOPE_COLOUR = '#56B4E9'
HEATMAP_ENVELOPE_MIN_BINS = 3.0
# ...AND a share of the panel. Bins alone let a 38 mV wobble on a 1.25 V axis
# draw a line hugging the edge and a caption announcing it, which is clutter
# dressed as a finding. Both gates have to clear: resolvable by the binning,
# AND large enough on the axis to be worth a reader's attention.
HEATMAP_ENVELOPE_MIN_FRACTION = 0.05

DELTA_V_BAND_CLIP_MV = 20.0
TREND_BAND_CLIP_V = 0.020

PEAK_FIT_XLIM_FLOOR = 0.02
PEAK_FIT_XLIM_PAD = 0.08
PEAK_FIT_XLIM_MIN_V = 0.20

# The fallback set of key cycles, when a params dict does not carry one.
# There were FIVE different literals for this — [1,10,20,30,40,50] in three
# places, [1,5,10,20,30,40,50] in one and [1,5,10] in another — so within a
# single run the key-cycle overlay, the split panel, the detected-peaks
# figure and the fit-quality figure each showed a different set of cycles.
# `params.default_key_cycles` is the definition; this mirrors it without
# importing params into plots (which would be a cycle).
DEFAULT_KEY_CYCLES = [1, 5, 10, 20, 30, 40, 50]

# A shaded region around a line is a BAND — that is the word journals use,
# and "±1 s.e." or "±1 standard error" is how its extent is stated. Not a
# strip, not an area (an area is what you integrate). Every band Ratatosk
# draws is ±1 standard error on ONE fitted parameter, taken from the
# covariance the least-squares fit returns, so it is the fit's own statement
# of how well that number is determined by the data — not a confidence
# interval on the measurement, and not propagated between parameters.
WATERFALL_HEIGHT_PER_CYCLE_IN = 0.35
WATERFALL_MIN_HEIGHT_IN = 4.0
WATERFALL_MAX_HEIGHT_IN = 10.0
WATERFALL_OVERLAP = 0.5
WATERFALL_DENSE_NOTE_ABOVE = 40
WATERFALL_MAX_LABELS = 12


# =============================================================================
# PORTED FROM 1.8.7 — extracted programmatically. Do not edit: the 1.9.0
# regression requires these figures to be identical.
# =============================================================================


_BAND_NOTE = ("Shaded bands span \u00b11 standard error on the fitted "
              "parameter, as returned by the least-squares fit; where a band "
              "is wider than the trend it describes, the parameter is not "
              "determined by the data.")


def _caption(text):
    """Print a figure caption in the same house style as the cycling cells."""
    print(section("  Suggested caption"))
    print(bullet(text, indent=2, label_width=2))

EXCLUDE_FORMATION_FROM_YLIM = True
SHORT_HALFCYCLE_THRESHOLD = 0.8
# ONE charge/discharge convention for the whole package. Blue means charge in
# every figure Ratatosk draws; vermillion means discharge. Both figure
# families import these rather than restating them, because until 1.9.1 they
# disagreed — blue was charge in the dQ/dV figures and DISCHARGE in the
# cycling figures, in the same run, which is indefensible whatever the
# convention.
#
# From Okabe-Ito. Blue/vermillion is the most robustly distinguishable pair
# under deuteranopia and protanopia, and the two differ in lightness so they
# survive greyscale printing. There is no universal published convention for
# which is which, so the obligation is internal consistency, and this is it.
# Okabe-Ito, colourblind-safe. The one categorical palette for the whole run;
# `cycling` imports it rather than restating it, exactly as it does the
# charge/discharge colours below.
CATEGORICAL_PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
                       '#D55E00', '#56B4E9', '#F0E442', '#000000']
CATEGORICAL_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', 'h']


def categorical_colours(n):
    """`n` distinguishable colours, taken BY INDEX from the shared palette.

    Four figures used to build their colours as
    `plt.colormaps['tab10'](np.linspace(0, 1, n))`, which samples a
    CATEGORICAL colormap as though it were continuous. That is not a style
    difference, it is a defect: at n = 2 it returns #1f77b4 and #17becf — a
    blue and a cyan, which is precisely the pair a colourblind reader cannot
    separate — and at n = 3 a blue, a brown and a cyan. tab10's designed
    sequence (blue, orange, green, red, purple) is never reached. Two cells or
    three peaks is the common case, so the worst behaviour was the usual one.

    Indexing the Okabe-Ito palette instead gives the colours it was designed
    with, in order, and cycles beyond eight rather than interpolating.
    """
    n = max(int(n), 1)
    return [CATEGORICAL_PALETTE[i % len(CATEGORICAL_PALETTE)]
            for i in range(n)]


def categorical_markers(n):
    """The matching marker shapes, so a series is identified by SHAPE as well
    as colour — the part that survives greyscale printing."""
    n = max(int(n), 1)
    return [CATEGORICAL_MARKERS[i % len(CATEGORICAL_MARKERS)]
            for i in range(n)]


# THE THREE PEAK KINDS, from the same palette as everything else.
# `plot_detected_peaks` encoded primary/shoulder/truncated as red/green/purple
# with darkred/darkgreen/indigo annotation text. The markers differ in shape,
# which carries the distinction for the scatter — but the peak-ID numerals
# beside them are distinguished by COLOUR ALONE, and red against green is
# exactly the pair `CATEGORICAL_PALETTE` exists to keep out of this run.
# NMC111 cell A draws three shoulders against four primaries in one panel.
# Vermillion, bluish-green and reddish-purple are separable under both
# deuteranopia and protanopia and differ in lightness for greyscale.
PEAK_KIND_COLOURS = {'primary': '#D55E00',      # Okabe-Ito vermillion
                     'shoulder': '#009E73',     # Okabe-Ito bluish green
                     'truncated': '#CC79A7'}    # Okabe-Ito reddish purple
PEAK_KIND_EDGES = {'primary': '#8C3D00',
                   'shoulder': '#006146',
                   'truncated': '#8A4A6E'}

COLOUR_CHARGE = '#0072B2'        # Okabe-Ito blue
COLOUR_DISCHARGE = '#D55E00'     # Okabe-Ito vermillion

# The old names, kept so nothing outside breaks.
DEFAULT_CHARGE_COLOUR = COLOUR_CHARGE
DEFAULT_DISCHARGE_COLOUR = COLOUR_DISCHARGE

# The waterfall does not drop cycles to fit the page. The figure height is
# bounded and the vertical pitch reduced until the whole series fits, so
# nothing is hidden from the reader. WATERFALL_OVERLAP sets how much a trace
# may be overlapped by its neighbour: 0.0 is a fully separated stack, 0.9 a
# dense ridgeline.

def _protocol_window(params, fallback=None):
    """The window the cell was CYCLED between, for a caption.

    NOT `voltage_range`, which is the analysis axis: the full extent of the
    data trimmed 50 mV, so that every half-cycle of a dataset shares one axis.
    Those are different quantities and the captions were quoting the second as
    though it were the first — stating "cycled between 1.25 and 2.63 V" for a
    cell cycled 1.2-2.5 V, and giving three different cut-offs for a triplicate
    cycled identically, because one cell started from open circuit at 2.684 V.
    A wrong experimental detail in text written to be pasted into a manuscript.

    Cell 3b measured the real thing (the median of the per-cycle limits) and
    put it in `user_parameters`; this reads it back.
    """
    lo = (params or {}).get("voltage_lower_V")
    hi = (params or {}).get("voltage_upper_V")
    try:
        lo, hi = float(lo), float(hi)
    except (TypeError, ValueError):
        return fallback
    return (lo, hi) if np.isfinite(lo) and np.isfinite(hi) and hi > lo \
        else fallback


def _get_display_name(name, params, all_params):
    """
    Build a display name that distinguishes datasets with the same 
    composition. Uses composition alone if unique across loaded datasets.
    If multiple datasets share a composition (triplicates), appends a 
    cell identifier extracted from the filename (e.g., 'Cell A') or 
    a numeric index.

    v1.8.1 fix: when filenames contain multiple numbers (e.g. 
    Sample_2_1, Sample_2_2 ... Sample_2_9), the function now finds 
    the numeric token that *varies* across the triplicate set rather 
    than always capturing the first or last number. This prevents all 
    cells receiving the same label (e.g. all labelled "Cell 2") when 
    the batch number precedes the cell number.

    Usage in all plotting cells:
        composition = _get_display_name(name, params, user_parameters)
    """
    composition = params.get('composition', name)
    # Prefer the identifier confirmed in Cell 3 over re-parsing the
    # filename. The stored value is what the operator actually agreed to.
    _cid = str(params.get('cell_id', '') or '').strip()
    same_comp = [n for n, p in all_params.items()
                 if p.get('composition') == composition]
    if len(same_comp) > 1:
        if _cid:
            return f"{composition} (Cell {_cid})"
        # --- Pattern 1: single letter between underscores (_A_, _B_) ---
        match = re.search(r'_([A-Za-z])_', name)
        if match:
            return f"{composition} (Cell {match.group(1).upper()})"

        # --- Pattern 2: find the numeric token that distinguishes files ---
        # Extract all digit groups from each filename in the triplicate set
        all_tokens = {n: re.findall(r'\d+', n) for n in same_comp}
        my_tokens = all_tokens.get(name, [])

        if my_tokens:
            # Check each token position: the distinguishing token is the
            # one whose value differs across the triplicate filenames
            n_tokens = min(len(t) for t in all_tokens.values())
            for pos in range(n_tokens):
                values_at_pos = {t[pos] for t in all_tokens.values()
                                 if len(t) > pos}
                if len(values_at_pos) > 1:
                    # This position varies — it's the cell identifier
                    return f"{composition} (Cell {my_tokens[pos]})"

            # All token positions are identical (shouldn't happen with 
            # genuinely different filenames) — fall back to last number
            return f"{composition} (Cell {my_tokens[-1]})"

        # --- Fallback: positional index ---
        idx = same_comp.index(name) + 1
        return f"{composition} (Cell {idx})"
    return composition


def _force_integer_cycles(ax):
    """
    Force x-axis to show only integer tick values starting from 0.
    Prevents matplotlib from displaying decimal cycle numbers (e.g., 
    2.5, 7.5) when the data range is small. Cycle number is always 
    an integer — 2.5 cycles has no physical meaning.

    v1.8: also enforces steps of 5 or 10 for datasets with >20 cycles,
    and ensures the axis starts at 0 with sensible tick placement.

    Usage in all plotting cells:
        _force_integer_cycles(ax)
    """
    ax.set_xlim(left=0)
    x_max = ax.get_xlim()[1]

    if x_max <= 10:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
    elif x_max <= 25:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
    elif x_max <= 55:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
    elif x_max <= 110:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
    elif x_max <= 275:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(25))
    elif x_max <= 550:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(50))
    else:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(100))


def _discharge_label(params):
    """The label for the 'discharge' half-cycle. See `style.working_ion`."""
    return half_cycle_labels(params)[1]


def _charge_label(params):
    """The label for the 'charge' half-cycle. See `style.working_ion`."""
    return half_cycle_labels(params)[0]


def _apply_pub_style(ax, xlabel='Voltage / V',
                      ylabel='dQ/dV / mAh V$^{-1}$ g$^{-1}$'):
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.tick_params(axis='both', labelsize=11, width=0.8, direction='in',
                   top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)


def _detect_short_halfcycles(data, step,
                              threshold=SHORT_HALFCYCLE_THRESHOLD):
    """
    Split the half-cycles of one step into usable and too-short.

    A half-cycle is flagged short if its voltage span is less than
    `threshold` times the median span across all half-cycles of that
    step. This is a plotting-quality filter for dQ/dV, and is unrelated
    to the protocol-level completeness check in Cell 5b, which asks
    whether the cycler finished the cycle at all. Both were named
    alike before v1.8.6; they are different tests and were separated
    so they can no longer be confused.

    Returns (complete, short): two sorted lists of cycle numbers.
    """
    if threshold <= 0.0:
        all_cycles = sorted([c for c, s in data.keys() if s == step])
        return all_cycles, []

    ranges = {}
    for (cyc, s), df in data.items():
        if s == step:
            ranges[cyc] = df['Voltage'].max() - df['Voltage'].min()

    if not ranges:
        return [], []

    median_range = np.median(list(ranges.values()))
    complete = []
    short = []
    for cyc in sorted(ranges.keys()):
        if ranges[cyc] >= threshold * median_range:
            complete.append(cyc)
        else:
            short.append(cyc)

    return complete, short


def _waterfall_label_cycles(cycles, max_labels=WATERFALL_MAX_LABELS):
    """
    Choose a readable subset of cycle numbers to annotate.

    Always includes the first and last cycle plotted. Everything else
    is evenly spaced, so the annotation density stays constant whether
    the run is 20 cycles or 500.
    """
    if not cycles:
        return set()
    stride = max(1, int(np.ceil(len(cycles) / max(1, max_labels))))
    chosen = set(cycles[::stride])
    chosen.add(cycles[0])
    chosen.add(cycles[-1])
    return chosen


def plot_dqdv_all_cycles(processed_dqdv, user_parameters,
                          exclude_formation=EXCLUDE_FORMATION_FROM_YLIM,
                          save_location=None, file_format=None):
    for name, result in processed_dqdv.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        colour_palette = params.get('colour_palette', 'viridis_r')
        data = result['data']

        all_cycles = sorted(set(c for c, s in data.keys()))

        complete_c, short_c = _detect_short_halfcycles(data, 'Charge')
        complete_d, short_d = _detect_short_halfcycles(data, 'Discharge')
        short_all = set(short_c + short_d)
        plot_cycles = [c for c in all_cycles if c not in short_all]

        if short_all:
            print(f"  Note: excluding short cycle(s) "
                  f"{sorted(short_all)} from all-cycle overlay")

        if not plot_cycles:
            print(f"  WARNING: No complete cycles for {name}")
            continue

        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        cmap = plt.get_cmap(colour_palette)
        norm = Normalize(vmin=min(plot_cycles), vmax=max(plot_cycles))

        ylim_vals = []

        for cycle in plot_cycles:
            colour = cmap(norm(cycle))

            if (cycle, 'Charge') in data:
                df_c = data[(cycle, 'Charge')]
                ax.plot(df_c['Voltage'], df_c['dQ/dV_processed'],
                       color=colour, linewidth=0.7, alpha=0.8)
                if not (exclude_formation and cycle == 1):
                    ylim_vals.extend(df_c['dQ/dV_processed'].values)

            if (cycle, 'Discharge') in data:
                df_d = data[(cycle, 'Discharge')]
                ax.plot(df_d['Voltage'], df_d['dQ/dV_processed'],
                       color=colour, linewidth=0.7, alpha=0.8)
                if not (exclude_formation and cycle == 1):
                    ylim_vals.extend(df_d['dQ/dV_processed'].values)

        ax.axhline(y=0, color='grey', linewidth=0.5, linestyle='-')
        _apply_pub_style(ax)
        ax.set_title(f'{composition} — dQ/dV (all cycles)', fontsize=13)

        if ylim_vals:
            ymin, ymax = min(ylim_vals), max(ylim_vals)
            ypad = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - ypad, ymax + ypad)

        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02, aspect=30)
        cbar.set_label('Cycle number', fontsize=12)
        cbar.ax.tick_params(labelsize=10)

        plt.tight_layout()

        # The cycling window for the caption; the analysis
        # axis is not what the cell was cycled between.
        v_min, v_max = _protocol_window(
            params, fallback=result['voltage_range'])
        charge_rate = params.get('charge_rate_c', 'the specified')
        # "ALL N CYCLES" IS ONLY TRUE IF NONE WERE DROPPED. Short half-cycles
        # are excluded from this overlay a hundred lines above, announced on
        # the console and nowhere else — on NNM cell A that is 93 of 220
        # cycles, the whole second half of the cell's life, under a caption
        # written for pasting into a manuscript that says "all 127
        # galvanostatic cycles". A caption is a claim about the figure.
        _excluded = len(all_cycles) - len(plot_cycles)
        _cov = (f"for all {len(plot_cycles)} galvanostatic cycles of "
                if not _excluded else
                f"for {len(plot_cycles)} of the {len(all_cycles)} "
                f"galvanostatic cycles of ")
        caption = (
            f"Figure X. Differential capacity (dQ/dV) vs. voltage "
            + _cov
            + f"{composition}, cycled between {v_min:.2f} and "
            f"{v_max:.2f} V at {charge_rate} C. Positive values "
            f"correspond to {_charge_label(params).lower()} and "
            f"negative values to {_discharge_label(params).lower()}. "
            f"Colour indicates cycle number as shown in the colourbar."
            + (f" {_excluded} cycle(s) whose charge or discharge half-cycle "
               f"did not span the analysed window are not shown; they are in "
               f"the processed-dQ/dV table."
               if _excluded else "")
        )
        # A half-cycle Cell 5 judged was not a measurement is KEPT in the
        # figure deliberately — seeing where the run has got to is the point —
        # but a caption bound for a manuscript has to say so. Cell 6's caption
        # did; this one said "for all 11 galvanostatic cycles" and left the
        # reader to find out that the eleventh was still running.
        _inc = sorted(int(c) for c in (params.get("incomplete_cycles") or ()))
        if _inc:
            caption += (
                f" Cycle{'s' if len(_inc) > 1 else ''} "
                f"{', '.join(str(c) for c in _inc)} "
                f"{'were' if len(_inc) > 1 else 'was'} incomplete at export "
                f"and {'are' if len(_inc) > 1 else 'is'} shown for "
                f"completeness only.")
        _caption(caption)

        if save_location:
            fpath = os.path.join(
                save_location,
                f'{name}_dQdV_all_cycles.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)


def plot_dqdv_key_cycles_combined(processed_dqdv, user_parameters,
                                   exclude_formation=EXCLUDE_FORMATION_FROM_YLIM,
                                   save_location=None, file_format=None):
    for name, result in processed_dqdv.items():
        params = user_parameters.get(name, {})
        key_cycles = params.get('key_cycles', DEFAULT_KEY_CYCLES)
        composition = _get_display_name(name, params, user_parameters)
        colour_palette = params.get('colour_palette', 'viridis_r')
        data = result['data']

        available_cycles = sorted(set(c for c, s in data.keys()))
        key_cycles = [c for c in key_cycles if c in available_cycles]

        if not key_cycles:
            print(f"  WARNING: No key cycles found for {name}")
            continue

        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        cmap = plt.get_cmap(colour_palette)
        norm = Normalize(vmin=min(key_cycles), vmax=max(key_cycles))

        ylim_vals = []

        for cycle in key_cycles:
            colour = cmap(norm(cycle))

            if (cycle, 'Charge') in data:
                df_c = data[(cycle, 'Charge')]
                ax.plot(df_c['Voltage'], df_c['dQ/dV_processed'],
                       color=colour, linewidth=1.2, alpha=0.9,
                       linestyle='-')
                if not (exclude_formation and cycle == 1):
                    ylim_vals.extend(df_c['dQ/dV_processed'].values)

            if (cycle, 'Discharge') in data:
                df_d = data[(cycle, 'Discharge')]
                ax.plot(df_d['Voltage'], df_d['dQ/dV_processed'],
                       color=colour, linewidth=1.2, alpha=0.9,
                       linestyle='-')
                if not (exclude_formation and cycle == 1):
                    ylim_vals.extend(df_d['dQ/dV_processed'].values)

        ax.axhline(y=0, color='grey', linewidth=0.5, linestyle='-')
        _apply_pub_style(ax)
        ax.set_title(f'{composition} — dQ/dV (key cycles)', fontsize=13)

        if ylim_vals:
            ymin, ymax = min(ylim_vals), max(ylim_vals)
            ypad = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - ypad, ymax + ypad)

        if len(key_cycles) > 6:
            sm = cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax, pad=0.02, aspect=30)
            cbar.set_label('Cycle number', fontsize=12)
            cbar.ax.tick_params(labelsize=10)
        else:
            from matplotlib.lines import Line2D
            handles = [
                Line2D([0], [0], color=cmap(norm(c)), linewidth=1.5,
                       label=f'Cycle {int(c)}')
                for c in key_cycles
            ]
            ax.legend(handles=handles, fontsize=9, framealpha=0.7,
                     loc='best')

        plt.tight_layout()

        cycles_str = ', '.join(str(int(c)) for c in key_cycles)
        # The cycling window for the caption; the analysis
        # axis is not what the cell was cycled between.
        v_min, v_max = _protocol_window(
            params, fallback=result['voltage_range'])
        charge_rate = params.get('charge_rate_c', 'the specified')
        caption = (
            f"Figure X. Differential capacity (dQ/dV) vs. voltage "
            f"for selected cycles ({cycles_str}) of {composition}, "
            f"cycled between {v_min:.2f} and {v_max:.2f} V at "
            f"{charge_rate} C. Positive values correspond to "
            f"{_charge_label(params).lower()} and negative values to "
            f"{_discharge_label(params).lower()}."
        )
        # See the same note on `plot_dqdv_all_cycles`: a key-cycle figure can
        # include an incomplete cycle too, if the operator listed it.
        _inc = sorted(int(c) for c in (params.get("incomplete_cycles") or ())
                      if int(c) in set(int(x) for x in key_cycles))
        if _inc:
            caption += (
                f" Cycle{'s' if len(_inc) > 1 else ''} "
                f"{', '.join(str(c) for c in _inc)} "
                f"{'were' if len(_inc) > 1 else 'was'} incomplete at export "
                f"and {'are' if len(_inc) > 1 else 'is'} shown for "
                f"completeness only.")
        _caption(caption)

        if save_location:
            fpath = os.path.join(
                save_location,
                f'{name}_dQdV_key_cycles.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)


def plot_dqdv_key_cycles_split(processed_dqdv, user_parameters,
                                exclude_formation=EXCLUDE_FORMATION_FROM_YLIM,
                                save_location=None, file_format=None):
    for name, result in processed_dqdv.items():
        params = user_parameters.get(name, {})
        key_cycles = params.get('key_cycles', DEFAULT_KEY_CYCLES)
        composition = _get_display_name(name, params, user_parameters)
        data = result['data']

        available_cycles = sorted(set(c for c, s in data.keys()))
        key_cycles = [c for c in key_cycles if c in available_cycles]

        if not key_cycles:
            continue

        # Two side-by-side panels: full journal width, shorter height
        fig, (ax_charge, ax_discharge) = plt.subplots(
            1, 2, figsize=(figure_width_inches, figure_height_inches * 0.75),
            sharey=False)
        fig.suptitle(f'{composition}', fontsize=14, fontweight=FIGURE_TITLE_WEIGHT)

        norm = Normalize(vmin=min(key_cycles), vmax=max(key_cycles))
        cmap_charge = plt.colormaps['Blues']
        cmap_discharge = plt.colormaps['Oranges']

        ylim_data = {'Charge': [], 'Discharge': []}

        for cycle in key_cycles:
            intensity = 0.3 + 0.7 * norm(cycle)

            if (cycle, 'Charge') in data:
                df_c = data[(cycle, 'Charge')]
                ax_charge.plot(
                    df_c['Voltage'], df_c['dQ/dV_processed'],
                    color=cmap_charge(intensity),
                    linewidth=1.2, label=f'Cycle {cycle}')
                if not (exclude_formation and cycle == 1):
                    ylim_data['Charge'].extend(
                        df_c['dQ/dV_processed'].values)

            if (cycle, 'Discharge') in data:
                df_d = data[(cycle, 'Discharge')]
                ax_discharge.plot(
                    df_d['Voltage'], df_d['dQ/dV_processed'],
                    color=cmap_discharge(intensity),
                    linewidth=1.2, label=f'Cycle {cycle}')
                if not (exclude_formation and cycle == 1):
                    ylim_data['Discharge'].extend(
                        df_d['dQ/dV_processed'].values)

        chg_t = _charge_label(params)
        dch_t = _discharge_label(params)
        for ax, title, step in [(ax_charge, chg_t, 'Charge'),
                                 (ax_discharge, dch_t, 'Discharge')]:
            _apply_pub_style(ax)
            ax.set_title(title, fontsize=12)
            ax.legend(fontsize=8, framealpha=0.7, loc='best')
            ax.axhline(y=0, color='grey', linewidth=0.5, linestyle='-')

            if ylim_data[step]:
                ymin = min(ylim_data[step])
                ymax = max(ylim_data[step])
                ypad = (ymax - ymin) * 0.10
                ax.set_ylim(ymin - ypad, ymax + ypad)

        plt.tight_layout()

        if save_location:
            fpath = os.path.join(
                save_location,
                f'{name}_dQdV_key_cycles_split.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)


def plot_dqdv_waterfall(processed_dqdv, user_parameters, step='Charge',
                         every_n=1, offset_scale=None,
                         overlap=WATERFALL_OVERLAP,
                         max_height_in=WATERFALL_MAX_HEIGHT_IN,
                         save_location=None, file_format=None):
    """
    Stacked-offset dQ/dV traces, one per cycle.

    Every complete cycle is plotted. Nothing is thinned automatically:
    a long run produces a denser ridgeline, not a shorter subset, so a
    change confined to a handful of cycles cannot be lost to sampling.

    Density is controlled by geometry rather than by discarding data.
    The figure height is clamped between WATERFALL_MIN_HEIGHT_IN and
    max_height_in, and the vertical pitch between traces is set to
    (1 - overlap) times the median trace amplitude, so raising
    `overlap` packs more cycles into the same page.

    Pass every_n > 1 to sample deliberately; the omitted cycles are
    then listed in the console so the choice is on the record.
    """
    for name, result in processed_dqdv.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        data = result['data']

        complete, short = _detect_short_halfcycles(data, step)
        if short:
            print(f"  Note: excluding short {step.lower()} cycle(s) "
                  f"{short} from waterfall")

        cycles_to_plot = complete[::every_n]
        if every_n > 1:
            omitted = [c for c in complete if c not in set(cycles_to_plot)]
            print(f"  every_n={every_n}: plotting {len(cycles_to_plot)} of "
                  f"{len(complete)} cycles. Omitted: {omitted}")

        if not cycles_to_plot:
            print(f"  WARNING: No {step} data found for {name}.")
            continue

        n_cycles = len(cycles_to_plot)

        # --- Vertical pitch between traces -------------------------
        amplitudes = []
        for cyc in cycles_to_plot:
            if (cyc, step) in data:
                vals = data[(cyc, step)]['dQ/dV_processed']
                amplitudes.append(float(vals.max() - vals.min()))
        median_amp = float(np.median(amplitudes)) if amplitudes else 1.0

        if offset_scale is not None:
            pitch = float(offset_scale)
        else:
            overlap = float(np.clip(overlap, 0.0, 0.95))
            pitch = max(median_amp * (1.0 - overlap), median_amp * 0.05)

        # --- Figure height, bounded --------------------------------
        wanted_h = n_cycles * WATERFALL_HEIGHT_PER_CYCLE_IN
        _lo = min(max(WATERFALL_MIN_HEIGHT_IN, figure_height_inches),
                  max_height_in)
        fig_h = float(np.clip(wanted_h, _lo, max_height_in))
        if wanted_h > max_height_in and n_cycles > WATERFALL_DENSE_NOTE_ABOVE:
            print(f"  {n_cycles} cycles in {fig_h:.1f} in: traces overlap "
                  f"by {overlap:.0%}. All cycles are shown. Raise "
                  f"max_height_in for a taller figure, or use the "
                  f"heatmap for a compact overview.")

        fig, ax = plt.subplots(figsize=(figure_width_inches, fig_h))

        cmap = plt.colormaps['viridis_r']
        norm = Normalize(vmin=min(cycles_to_plot),
                         vmax=max(cycles_to_plot))

        dense = n_cycles > 25
        line_w = 0.6 if dense else 0.9
        effects = ([path_effects.withStroke(linewidth=line_w + 1.1,
                                            foreground='white')]
                   if dense else None)

        label_cycles = _waterfall_label_cycles(cycles_to_plot)
        v_lo, v_hi = np.inf, -np.inf

        for i, cycle in enumerate(cycles_to_plot):
            if (cycle, step) not in data:
                continue
            df = data[(cycle, step)]
            offset = i * pitch
            colour = cmap(norm(cycle))

            ax.plot(df['Voltage'], df['dQ/dV_processed'] + offset,
                    color=colour, linewidth=line_w, alpha=1.0,
                    zorder=2 + i, path_effects=effects,
                    solid_capstyle='round')

            v_lo = min(v_lo, float(df['Voltage'].min()))
            v_hi = max(v_hi, float(df['Voltage'].max()))

            if cycle in label_cycles:
                ax.text(float(df['Voltage'].max()), offset,
                        f' {cycle}', fontsize=8, va='center', ha='left',
                        color=colour, zorder=2 + n_cycles + i,
                        clip_on=False)

        # Room on the right for the cycle labels
        if np.isfinite(v_lo) and np.isfinite(v_hi) and v_hi > v_lo:
            ax.set_xlim(v_lo, v_hi + 0.06 * (v_hi - v_lo))

        _apply_pub_style(
            ax, ylabel=f'dQ/dV (offset) / mAh V$^{{-1}}$ g$^{{-1}}$')
        step_display = (_charge_label(params) if step == 'Charge'
                        else _discharge_label(params))
        ax.set_title(
            f'{composition} — {step_display} dQ/dV waterfall '
            f'({n_cycles} cycles)', fontsize=13)
        ax.set_yticks([])

        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.6, aspect=30, pad=0.08)
        cbar.set_label('Cycle number', fontsize=11)

        # --- HOW FAR DOES THE TALLEST TRACE REACH? ---------------------
        # The pitch is a fraction of the MEDIAN amplitude, which is what keeps
        # a hundred-cycle waterfall readable — at OVERLAP = 0.5 a median trace
        # spans two lanes BY DESIGN. So "taller than the pitch" is true of
        # nearly every trace and says nothing; the first version of this note
        # reported 17 of 17 traces and was worse than silence. What matters is
        # a trace much taller than the median the pitch came from: on LTO cell
        # C, cycle 1 is 1.54x the median and is drawn through the two rows
        # above it, which is how its doublet was read as a feature of cycles
        # 2 and 3. The heatmap does not offset and so cannot bleed — which is
        # why the same feature is absent there.
        #
        # No threshold is chosen. The note always states the worst case, so a
        # reader can calibrate the figure in front of them rather than trust a
        # number picked here; the console speaks up only when it is large.
        _amp = {}
        for cyc in cycles_to_plot:
            if (cyc, step) in data:
                _v = data[(cyc, step)]['dQ/dV_processed']
                _amp[int(cyc)] = float(_v.max() - _v.min())
        if _amp and pitch > 0 and median_amp > 0:
            _tall = max(_amp, key=_amp.get)
            _ratio = _amp[_tall] / median_amp
            _lanes = _amp[_tall] / pitch
            _over = max(0, int(np.ceil(_lanes)) - 1)
            fig.text(0.01, -0.01,
                     f"Traces are offset by {pitch / median_amp:.0%} of the "
                     f"median amplitude, so a median trace spans "
                     f"{median_amp / pitch:.1f} rows. The tallest (cycle "
                     f"{_tall}) is {_ratio:.1f}x the median and is drawn "
                     f"through the {_over} row(s) above it — a feature seen "
                     f"there may belong to a lower cycle. The heatmap does "
                     f"not offset.",
                     fontsize=7, style="italic", va="top", wrap=True)
            if _ratio > 1.5:
                _big = sorted((c for c, a in _amp.items()
                               if a / median_amp > 1.5),
                              key=lambda c: -_amp[c])
                print(f"  {step}: cycle {_tall} is {_ratio:.1f}x the median "
                      f"trace and reaches {_over} row(s) above its own"
                      + (f" ({len(_big)} trace(s) over 1.5x: "
                         + ", ".join(str(c) for c in _big[:8])
                         + (" ..." if len(_big) > 8 else "") + ")"
                         if len(_big) > 1 else "")
                      + ". Use the heatmap to place a feature exactly.")

        plt.tight_layout()

        if save_location:
            fpath = os.path.join(
                save_location,
                f'{name}_dQdV_waterfall_{step}.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)


def plot_dqdv_heatmap(processed_dqdv, user_parameters, step='Charge',
                       v_bins=200, save_location=None, file_format=None,
                       envelope=True, suffix=''):
    """
    dQ/dV against voltage and cycle, as a heat map.

    `envelope` draws the step line at the edge of the range each cycle
    actually traversed, and the note that goes with it. It is the honest
    reading of the panel — the cells outside that line are unmeasured, not
    zero — but it is also a line across a figure, and a figure meant for a
    talk sometimes wants the shape alone. Both are produced by Cell 7b, which
    is why `suffix` exists: the two versions cannot share a filename.
    """
    for name, result in processed_dqdv.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        data = result['data']
        # The cycling window for the caption; the analysis
        # axis is not what the cell was cycled between.
        v_min, v_max = _protocol_window(
            params, fallback=result['voltage_range'])

        complete, short = _detect_short_halfcycles(data, step)
        if short:
            print(f"  Note: excluding short {step.lower()} cycle(s) "
                  f"{short} from heatmap")

        cycles = complete
        if not cycles:
            continue

        # THE GRID COMES FROM THE DATA, NOT FROM THE PROTOCOL WINDOW.
        # `v_min, v_max` above are the window the cell was CYCLED between and
        # are the right thing to put in the caption; they are the wrong thing
        # to bin onto. The analysed axis is not the protocol window — LTO cell
        # A is cycled 1.2-2.5 V and analysed to 2.634 V — so a grid ending at
        # 2.5 V threw away 130 mV of real charge data, about a tenth of the
        # figure. At the other end `np.interp` does not return NaN outside the
        # data range: it CLAMPS, repeating the endpoint value, so the 51 mV
        # below the first measured point was painted as a flat band of colour
        # that no measurement supports. Both faults are silent, and both are
        # present on every dataset in every run on disk.
        _vals = [data[(c, step)]['Voltage'].values
                 for c in cycles if (c, step) in data]
        _vals = [v[np.isfinite(v)] for v in _vals]
        _vals = [v for v in _vals if v.size]
        if _vals:
            v_lo = float(min(v.min() for v in _vals))
            v_hi = float(max(v.max() for v in _vals))
        else:
            v_lo, v_hi = float(v_min), float(v_max)
        if not np.isfinite(v_lo) or not np.isfinite(v_hi) or v_hi <= v_lo:
            v_lo, v_hi = float(v_min), float(v_max)
        v_edges = np.linspace(v_lo, v_hi, v_bins + 1)
        v_centres = 0.5 * (v_edges[:-1] + v_edges[1:])

        heatmap_data = np.full((len(cycles), v_bins), np.nan)

        for i, cycle in enumerate(cycles):
            if (cycle, step) not in data:
                continue
            df = data[(cycle, step)]
            _v = df['Voltage'].values
            _y = df['dQ/dV_processed'].values
            _m = np.isfinite(_v) & np.isfinite(_y)
            if _m.sum() < 2:
                continue
            _v, _y = _v[_m], _y[_m]
            _row = np.interp(v_centres, _v, _y)
            # ...and a cell that did not reach a voltage this cycle gets NaN
            # there, drawn as the axes background, rather than a repeat of its
            # nearest measured value. A short half-cycle should look short.
            _row[(v_centres < _v.min()) | (v_centres > _v.max())] = np.nan
            heatmap_data[i, :] = _row

        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        plot_data = (np.abs(heatmap_data) if step == 'Discharge'
                     else heatmap_data)
        # An all-NaN panel is possible now that unvisited voltages are NaN
        # rather than a repeated endpoint, and `nanpercentile` raises on one.
        if not np.isfinite(plot_data).any():
            print(f"  {name}: no {step.lower()} data in the analysed window, "
                  f"heatmap skipped")
            plt.close(fig)
            continue
        vmax_clim = np.nanpercentile(plot_data, 98)

        # NOT-VISITED IS DARK, AND THE COLLAPSE IS A LINE.
        #
        # Two things were wrong with the original and one with the first fix.
        #
        # ORIGINAL: `np.interp` clamps outside the data range rather than
        # returning NaN, so every voltage a half-cycle never reached was
        # painted with a repeat of its nearest measured value. On this
        # colormap that renders near-black, which looks right and is the
        # reason it survived — but a fabricated zero and a measured zero are
        # the same pixel, and the region carried no information at all.
        #
        # FIRST FIX: NaN there, drawn light grey. Honest, and unusable. The
        # visited span varies enormously between cycles — the MEDIAN cycle
        # covers 56% of the grid on LTO cell A and 33% on NNM cell B — so the
        # panel became a large pale wedge with a thin bright line beside it.
        #
        # WHAT IT DOES NOW. Unvisited stays dark, close enough to the zero end
        # of the ramp that the figure reads as a heat map rather than as a
        # hole; the information the wedge was carrying is drawn as an ENVELOPE
        # instead — a step line at the highest and lowest voltage each cycle
        # actually reached. A line states the collapse more loudly than a void
        # does, because it has a shape you can read off a scale, and it costs
        # no panel area. Nothing is fabricated: the cells behind the envelope
        # are masked, not filled.
        _note_room = 0.0        # reserved only if the envelope note is drawn
        _n_missing = int(np.isnan(plot_data).sum())

        # DECIDED BEFORE THE COLOURS ARE CHOSEN, because the tint on the
        # unvisited cells is only defensible where the line and the note that
        # explain it are also drawn. On LTO cell A's delithiation the two
        # edges move 38 mV — below both gates, so no line and no note — and
        # the panel still carried a faint unexplained shade along one edge,
        # which is a mark on a figure that nothing on the figure accounts for.
        # Where the envelope is not worth drawing, the mask takes the
        # colormap's own zero and the diagnostic panel is byte-identical to
        # the plain one: nothing to explain, so nothing said.
        _bw = float(v_edges[1] - v_edges[0])
        _hi_env = np.array([np.nanmax(v) if v.size else np.nan for v in _vals])
        _lo_env = np.array([np.nanmin(v) if v.size else np.nan for v in _vals])
        _spread = max(np.nanmax(_hi_env) - np.nanmin(_hi_env),
                      np.nanmax(_lo_env) - np.nanmin(_lo_env))
        _panel = float(v_edges[-1] - v_edges[0])
        _worth = bool(envelope and _n_missing and np.isfinite(_spread)
                      and _spread > HEATMAP_ENVELOPE_MIN_BINS * _bw
                      and _panel > 0
                      and _spread / _panel > HEATMAP_ENVELOPE_MIN_FRACTION)

        _cmap = plt.get_cmap('inferno').copy()
        # `envelope=False` IS "as it was before", not "as it is now, minus the
        # line". The point of the second file is a panel with nothing on it to
        # explain — so the unvisited cells take the colormap's own zero colour
        # and the panel reads as one continuous surface, which is what the
        # clamped fill used to produce. The cells are still MASKED rather than
        # filled with a repeat of the nearest measured value: the appearance is
        # the old one, but nothing per-cycle is invented to get it, so the two
        # files differ in what they SAY and never in what was measured.
        _cmap.set_bad(HEATMAP_UNVISITED_COLOUR if _worth else _cmap(0.0))
        im = ax.pcolormesh(
            v_centres, cycles, np.ma.masked_invalid(plot_data),
            cmap=_cmap, shading='auto',
            vmin=0, vmax=vmax_clim)

        # The envelope itself. A cell whose every half-cycle covers the same
        # span has a straight line at each edge and does not need it; `_worth`
        # above is that decision, and it also chose the mask colour.
        if _worth:
            for _e in (_hi_env, _lo_env):
                if np.nanmax(_e) - np.nanmin(_e) <= _bw:
                    continue          # that edge is flat; drawing it is clutter
                ax.step(_e, cycles, where='mid',
                        color=HEATMAP_ENVELOPE_COLOUR, linewidth=1.3,
                        solid_joinstyle='miter', zorder=6)
            # BELOW the axes, and WRAPPED. Anywhere inside them collides with
            # either the peak (left) or the envelope itself (right), and a note
            # that obscures the thing it explains is worse than no note. It has
            # to be wrapped by hand: `bbox_inches='tight'` grows the CANVAS to
            # fit a figure-level text, so one long line widens the saved image
            # and squeezes the axes into the middle of it.
            # BOTH EDGES, DESCRIBED AS BOTH EDGES. Either end of the
            # traversed range can be the one that moves — LTO's charge step
            # stops progressively lower, NMC's starts progressively higher —
            # and a note that says "furthest voltage reached" is simply wrong
            # about the low edge, which is the one drawn on NMC.
            # Balanced across two lines: `bbox_inches='tight'` grows the
            # canvas to fit the widest line, so a long first line widens the
            # saved image and shrinks the axes inside it.
            _env_note = (f'Blue line: the edge of the voltage range the cell '
                         f'traversed on that cycle,\nmoving '
                         f'{1000 * _spread:.0f} mV over this record. Outside '
                         f'it the cell did not go — unmeasured, not zero.')
            fig.text(0.5, 0.105, _env_note, ha='center', va='top',
                     fontsize=8.5, style='italic', color='#444444',
                     linespacing=1.4)
            _note_room = 0.12

        cbar_label = '|dQ/dV|' if step == 'Discharge' else 'dQ/dV'
        cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
        cbar.set_label(
            f'{cbar_label} / mAh V$^{{-1}}$ g$^{{-1}}$', fontsize=11)

        _apply_pub_style(ax, ylabel='Cycle number')
        step_display = (_charge_label(params) if step == 'Charge'
                        else _discharge_label(params))
        ax.set_title(f'{composition} — {step_display} dQ/dV evolution',
                     fontsize=13)

        fig.tight_layout(rect=(0, _note_room, 1, 1))

        if save_location:
            fpath = os.path.join(
                save_location,
                f'{name}_dQdV_heatmap_{step}{suffix}'
                f'.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)


def plot_preprocessing_qc(processed_dqdv, user_parameters, cycle=1, *,
                          save_location=None, file_format=None):
    """Raw against processed, for one cycle.

    `save_location` is new. This was the ONLY plot function in the module
    without it: it drew the figure, called `plt.show()`, and that was the end
    of it. Seven dQ/dV figures reached the run folder and the one that
    justifies the preprocessing did not — so a run folder read a month later
    had no record of what the smoothing and binning actually did.
    """
    for name, result in processed_dqdv.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        data = result['data']

        # Two side-by-side panels: full journal width, shorter height
        fig, (ax_c, ax_d) = plt.subplots(
            1, 2, figsize=(figure_width_inches, figure_height_inches * 0.75))
        fig.suptitle(
            f'{composition} — Cycle {cycle} preprocessing QC',
            fontsize=13)

        chg_t = _charge_label(params)
        dch_t = _discharge_label(params)
        for ax, step, title in [(ax_c, 'Charge', chg_t),
                                 (ax_d, 'Discharge', dch_t)]:
            if (cycle, step) not in data:
                ax.text(0.5, 0.5, f'No {step} data\nfor cycle {cycle}',
                        transform=ax.transAxes, ha='center',
                        va='center', fontsize=12, color='grey')
                ax.set_title(title, fontsize=12)
                continue

            df = data[(cycle, step)]
            ax.plot(df['Voltage'], df['dQ/dV_raw'],
                    color='grey', alpha=0.5, linewidth=0.8,
                    label='Raw')
            ax.plot(df['Voltage'], df['dQ/dV_processed'],
                    color='#0072B2', linewidth=1.3,
                    label='Processed')

            _apply_pub_style(ax)
            ax.set_title(title, fontsize=12)
            ax.legend(fontsize=9, framealpha=0.7)
            ax.axhline(y=0, color='grey', linewidth=0.4, linestyle='-')

        plt.tight_layout()
        if save_location:
            fpath = os.path.join(
                save_location,
                f'{name}_preprocessing_qc.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)
        plt.show()
        plt.close(fig)


In [ ]:
# DEFINITIONS H2 — FIGURES: PEAKS, FITS, TRENDS — AND THE ADAPTERS THAT FEED THEM
# =============================================================================
def plot_detected_peaks(processed_dqdv, detected_peaks, user_parameters,
                         cycles_to_show=None, save_location=None,
                         file_format=None):
    """
    Annotated dQ/dV plots with detected peaks marked.
    Markers:
      ▲/▼  red   = primary peaks
      ■    green  = shoulder peaks (d² detected)
      ◆    purple = truncated peaks (retained but clipped at window edge)
    """
    import matplotlib.pyplot as plt

    for name in processed_dqdv:
        result = processed_dqdv[name]
        det    = detected_peaks[name]
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        ref_cycle   = det['reference_cycle']
        data        = result['data']

        available = sorted(set(c for c, s in data.keys()))
        # Which cycles have BOTH half-cycles present (a complete cycle).
        _steps_by_cycle = {}
        for (c, s) in data.keys():
            _steps_by_cycle.setdefault(c, set()).add(s)
        complete_cycles = sorted(
            c for c, steps in _steps_by_cycle.items()
            if {'Charge', 'Discharge'} <= steps)
        if cycles_to_show is None:
            # Show the user's key cycles (those that exist in this
            # dataset), and always add the reference cycle and the last
            # complete cycle, so the fingerprint and the aged end are
            # both visible regardless of what key_cycles contains. The
            # last cycle is often incomplete (odd half-cycle count), so
            # fall back to the last complete one rather than plotting a
            # half-cycle on its own.
            key_cycles = params.get('key_cycles',
                                    DEFAULT_KEY_CYCLES)
            wanted = [c for c in key_cycles if c in available]
            wanted.append(ref_cycle)
            if complete_cycles:
                wanted.append(complete_cycles[-1])
            elif available:
                wanted.append(available[-1])
            cycles_to_show_actual = sorted(set(wanted))
        else:
            cycles_to_show_actual = [c for c in cycles_to_show
                                      if c in available]

        n_show = len(cycles_to_show_actual)
        fig, axes = plt.subplots(n_show, 2, figsize=(13, 4 * n_show),
                                  squeeze=False)
        fig.suptitle(f'{composition} — detected peaks', fontsize=14,
                     fontweight=FIGURE_TITLE_WEIGHT)

        for row, cycle in enumerate(cycles_to_show_actual):
            for col, step in enumerate(['Charge', 'Discharge']):
                ax = axes[row, col]
                if (cycle, step) not in data:
                    ax.set_visible(False)
                    continue

                df = data[(cycle, step)]
                v, dq = df['Voltage'].values, df['dQ/dV_processed'].values
                is_ref = (cycle == ref_cycle)
                clr = COLOUR_CHARGE if step == 'Charge' else COLOUR_DISCHARGE
                ax.plot(v, dq, color=clr, linewidth=1.5 if is_ref else 1.0,
                        alpha=0.9)

                peaks = det['peaks'].get((cycle, step), pd.DataFrame())
                if not peaks.empty:
                    primary   = peaks[~peaks['is_shoulder'] & ~peaks['is_truncated']]
                    shoulders = peaks[peaks['is_shoulder']]
                    truncated = peaks[peaks['is_truncated']]
                    mk = '^' if step == 'Charge' else 'v'

                    # THE MARKER GOES ON THE CURVE THAT IS DRAWN. It used to
                    # be placed at `height_original`, which is the height on
                    # the RAW dQ/dV, while the line plotted here is
                    # `dQ/dV_processed`. Wherever processing changes the
                    # curve — a sharp formation spike is the clearest case —
                    # the marker floated off the line, and a peak found
                    # exactly at the maximum LOOKED like a miss. Detection was
                    # right and the picture said otherwise, which is the worse
                    # of the two failures.
                    #
                    # Interpolating onto the drawn curve is correct whatever
                    # the columns mean, so this cannot drift again if the
                    # processing changes.
                    def _on_curve(sub):
                        return np.interp(np.asarray(sub['voltage'], float),
                                         v, dq)

                    if not primary.empty:
                        ax.scatter(primary['voltage'], _on_curve(primary),
                                   color=PEAK_KIND_COLOURS['primary'], s=50,
                                   zorder=5, marker=mk,
                                   edgecolors=PEAK_KIND_EDGES['primary'],
                                   linewidths=0.5)
                    if not shoulders.empty:
                        ax.scatter(shoulders['voltage'], _on_curve(shoulders),
                                   color=PEAK_KIND_COLOURS['shoulder'], s=50,
                                   zorder=5, marker='s',
                                   edgecolors=PEAK_KIND_EDGES['shoulder'],
                                   linewidths=0.5)
                    if not truncated.empty:
                        ax.scatter(truncated['voltage'], _on_curve(truncated),
                                   color=PEAK_KIND_COLOURS['truncated'], s=60,
                                   zorder=5, marker='D',
                                   edgecolors=PEAK_KIND_EDGES['truncated'],
                                   linewidths=0.5)
                    for _, pk in peaks.iterrows():
                        if pk['is_truncated']:
                            clr_a = PEAK_KIND_EDGES['truncated']
                        elif pk['is_shoulder']:
                            clr_a = PEAK_KIND_EDGES['shoulder']
                        else:
                            clr_a = PEAK_KIND_EDGES['primary']
                        ax.annotate(
                            f'{int(pk["peak_id"])}',
                            (pk['voltage'],
                             float(np.interp(float(pk['voltage']), v, dq))),
                            textcoords='offset points',
                            xytext=(0, 12 if step == 'Charge' else -14),
                            ha='center', fontsize=9, fontweight='bold',
                            color=clr_a
                        )

                n_p  = len(peaks)
                n_sh = peaks['is_shoulder'].sum()  if not peaks.empty else 0
                n_tr = peaks['is_truncated'].sum() if not peaks.empty else 0
                extras = []
                if n_sh: extras.append(f'{n_sh} shoulder')
                if n_tr: extras.append(f'{n_tr} truncated')
                label = f'{n_p} peaks' + (f' ({", ".join(extras)})' if extras else '')
                suffix = ' (REFERENCE)' if is_ref else ''
                ax.set_title(f'Cycle {cycle} — {step}{suffix}', fontsize=11)
                ax.set_xlabel('Voltage / V', fontsize=11)
                ax.set_ylabel('dQ/dV / mAh V$^{-1}$ g$^{-1}$', fontsize=11)
                ax.tick_params(axis='both', labelsize=10, direction='in',
                              top=True, right=True)
                ax.axhline(y=0, color='grey', linewidth=0.4)
                ax.text(0.02, 0.95, label, transform=ax.transAxes,
                       fontsize=9, va='top', ha='left',
                       bbox=dict(boxstyle='round,pad=0.3',
                                facecolor='white', alpha=0.8))

        plt.tight_layout()
        if save_location:
            fpath = os.path.join(save_location,
                                f'{name}_detected_peaks.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)
        plt.show()
        plt.close(fig)

        # A shoulder is not a local maximum, so `find_peaks` never measured a
        # prominence or a width for it: `detect.py` fills the prominence with
        # a placeholder of 0.1 x height and leaves the width at zero. Both are
        # printed as what they are. Until 1.9.0.33 this table showed the
        # placeholder as "18.014" and the missing width as "0.0", side by side
        # with measured values in the same columns — on a broad layered oxide
        # where most detections are shoulders, almost every number in the
        # table was fabricated and nothing said so.
        print(f"\n  Reference peak summary (Cycle {ref_cycle}):")
        _any_estimated = False
        for step in ['Charge', 'Discharge']:
            rp = det['reference_peaks'][step]
            if rp is not None and not rp.empty:
                print(f"\n  {step}:")
                print(f"  {'Peak':>4}  {'Voltage/V':>10}  {'Height':>10}  "
                      f"{'Prominence':>12}  {'FWHM/mV':>8}  {'Type':>12}")
                for _, pk in rp.iterrows():
                    if pk['is_truncated']:
                        ptype = 'truncated'
                    elif pk['is_shoulder']:
                        ptype = 'shoulder'
                    else:
                        ptype = 'primary'
                    est = bool(pk.get('prominence_estimated', False))
                    _any_estimated = _any_estimated or est
                    prom = (f"~{pk['prominence']:.3f}" if est
                            else f"{pk['prominence']:.3f}")
                    _w = float(pk['width_V']) * 1000.0
                    width = ("--" if (not np.isfinite(_w) or _w <= 0.0)
                             else f"{_w:.1f}")
                    print(f"  {int(pk['peak_id']):>4}  {pk['voltage']:>10.4f}  "
                          f"{pk['height']:>10.3f}  {prom:>12}  "
                          f"{width:>8}  {ptype:>12}")
        if _any_estimated:
            print("\n  ~ prominence NOT measured — a shoulder is not a local "
                  "maximum, so this is a\n    placeholder of 0.1 x height, and "
                  "it is what ranks the peak if the list is\n    capped. "
                  "-- width not measured, for the same reason.")


def plot_fit_quality(processed_dqdv, detected_peaks, fit_results,
                      user_parameters, cycles_to_show=None,
                      save_location=None, file_format=None):
    """
    Show fitted curves overlaid on data with individual components
    and Rietveld-style residual panels.

    Grey = data, coloured line = composite fit, shaded fills = individual
    named components. The dashed black baseline is drawn only where one
    exists: from 1.9.0.56 the model has no free background, and drawing a
    flat zero line labelled "baseline" would tell the reader something
    false about what the model contains.

    Automatically includes the worst converged cycle and one failed
    cycle (if any) for honest quality assessment.
    """
    for name in fit_results:
        data = processed_dqdv[name]['data']
        det = detected_peaks[name]
        fits = fit_results[name]['fits']
        fs = fit_results[name]['fit_summary']
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        key_cycles = params.get('key_cycles', DEFAULT_KEY_CYCLES)
        ref_cycle = det['reference_cycle']

        if cycles_to_show is None:
            available = sorted(set(c for c, s in data.keys()))
            cycles_to_show_actual = sorted(set(
                [c for c in key_cycles if c in available][:5]
            ))
            if (ref_cycle not in cycles_to_show_actual and
                ref_cycle in available):
                cycles_to_show_actual = sorted(set(
                    cycles_to_show_actual + [ref_cycle]
                ))

            # Add worst-fitting converged cycle for honest QC
            converged = fs[fs['success'] == True]
            if not converged.empty:
                worst = converged.loc[converged['r_squared'].idxmin()]
                worst_cycle = int(worst['cycle'])
                if worst_cycle not in cycles_to_show_actual:
                    cycles_to_show_actual = sorted(set(
                        cycles_to_show_actual + [worst_cycle]
                    ))
                    print(f"  Including cycle {worst_cycle} "
                          f"(lowest R² = {worst['r_squared']:.4f}, "
                          f"{worst['step']}) for QC")
                else:
                    # Worst already shown — find worst unseen
                    unseen = converged[
                        ~converged['cycle'].isin(cycles_to_show_actual)
                    ]
                    if not unseen.empty:
                        worst2 = unseen.loc[
                            unseen['r_squared'].idxmin()]
                        w2_cycle = int(worst2['cycle'])
                        cycles_to_show_actual = sorted(set(
                            cycles_to_show_actual + [w2_cycle]
                        ))
                        print(f"  Including cycle {w2_cycle} "
                              f"(lowest unseen R² = "
                              f"{worst2['r_squared']:.4f}, "
                              f"{worst2['step']}) for QC")

            # Also include one failed cycle if any exist
            failed = fs[fs['success'] == False]
            if not failed.empty:
                fail_cycle = int(failed['cycle'].iloc[0])
                fail_step = failed['step'].iloc[0]
                if fail_cycle not in cycles_to_show_actual:
                    cycles_to_show_actual = sorted(set(
                        cycles_to_show_actual + [fail_cycle]
                    ))
                    print(f"  Including cycle {fail_cycle} "
                          f"(FAILED, {fail_step}) for QC")
        else:
            cycles_to_show_actual = cycles_to_show

        n_show = len(cycles_to_show_actual)

        _drew_baseline = False
        fig = plt.figure(figsize=(13, 5.0 * n_show))

        outer_gs = gridspec.GridSpec(
            n_show, 1, figure=fig,
            hspace=0.35, top=0.95, bottom=0.04
        )

        for row, cycle in enumerate(cycles_to_show_actual):
            inner_gs = gridspec.GridSpecFromSubplotSpec(
                2, 2, subplot_spec=outer_gs[row],
                height_ratios=[5, 1], hspace=0.05, wspace=0.3
            )

            for col, step in enumerate(['Charge', 'Discharge']):
                ax = fig.add_subplot(inner_gs[0, col])
                ax_res = fig.add_subplot(inner_gs[1, col], sharex=ax)

                if (cycle, step) not in data:
                    ax.set_visible(False)
                    ax_res.set_visible(False)
                    continue

                df = data[(cycle, step)]
                voltage = df['Voltage'].values
                dqdv = df['dQ/dV_processed'].values

                colour = COLOUR_CHARGE if step == 'Charge' else COLOUR_DISCHARGE
                ax.plot(voltage, dqdv, color='grey', linewidth=1.0,
                        alpha=0.6, label='Data')

                fit = fits.get((cycle, step))
                if (fit is not None and
                    fit.get('lmfit_result') is not None):
                    result = fit['lmfit_result']

                    best_fit = result.best_fit
                    if step == 'Discharge':
                        best_fit = -best_fit

                    ax.plot(voltage, best_fit, color=colour,
                            linewidth=1.5, label='Composite fit')

                    # Individual components
                    comps = result.eval_components(x=voltage)
                    for comp_name, comp_vals in comps.items():
                        if step == 'Discharge':
                            comp_vals = -comp_vals

                        if 'bg_' in comp_name:
                            # Only if there IS one. With no free background
                            # the component is identically zero, and a flat
                            # line at zero labelled "Baseline" tells the
                            # reader the model has something it has not.
                            if float(np.nanmax(np.abs(comp_vals))) <= 0.0:
                                continue
                            _drew_baseline = True
                            ax.plot(voltage, comp_vals, 'k--',
                                   linewidth=0.8, alpha=0.5,
                                   label='Baseline')
                        else:
                            ax.fill_between(voltage, comp_vals,
                                           alpha=0.15, color=colour)
                            ax.plot(voltage, comp_vals, color=colour,
                                   linewidth=0.6, alpha=0.5,
                                   linestyle='--')

                    is_ref = (cycle == ref_cycle)
                    suffix = ' (REF)' if is_ref else ''
                    ax.text(
                        0.02, 0.95,
                        f'R² = {fit["r_squared"]:.4f}{suffix}',
                        transform=ax.transAxes, fontsize=9,
                        va='top', ha='left',
                        bbox=dict(boxstyle='round,pad=0.3',
                                 facecolor='white', alpha=0.8)
                    )

                    # Difference plot
                    residuals = fit['residuals']
                    if residuals is not None:
                        if step == 'Discharge':
                            residuals = -residuals
                        ax_res.plot(voltage, residuals,
                                   color=colour, linewidth=0.6,
                                   alpha=0.7)
                        ax_res.axhline(y=0, color='grey',
                                      linewidth=0.4)
                        ax_res.fill_between(voltage, residuals,
                                           alpha=0.1, color=colour)

                elif fit is not None and not fit.get('success', False):
                    ax.text(
                        0.5, 0.5, 'FIT FAILED',
                        transform=ax.transAxes, fontsize=14,
                        ha='center', va='center',
                        color='red', fontweight='bold', alpha=0.5
                    )

                # THE PANEL SHOWS THE FEATURE, NOT THE WHOLE WINDOW.
                # No x-limit was ever set here, so both panels spanned the
                # full analysis range. On a sharp profile that is unreadable:
                # LTO's single peak is about 40 mV wide inside a 1.3 V window
                # — roughly 3% of the axis — and the discharge panel renders
                # as a vertical line. A fit was misread by 4x because of it.
                #
                # The limits are taken from where the CURVE actually is: the
                # span over which |dQ/dV| stays above PEAK_FIT_XLIM_FLOOR of
                # its own maximum, padded, with a minimum width so a very
                # narrow peak still gets context around it rather than
                # filling the panel edge to edge. `sharex` carries it to the
                # residual panel. Nothing is cropped that carries signal, and
                # a curve that genuinely fills the window is unchanged.
                try:
                    _amp = np.abs(np.asarray(dqdv, float))
                    _mx = float(np.nanmax(_amp)) if _amp.size else 0.0
                    if _mx > 0:
                        _keep = np.asarray(voltage, float)[
                            _amp >= PEAK_FIT_XLIM_FLOOR * _mx]
                        if _keep.size:
                            _lo, _hi = float(_keep.min()), float(_keep.max())
                            _pad = max(PEAK_FIT_XLIM_PAD * (_hi - _lo),
                                       0.5 * (PEAK_FIT_XLIM_MIN_V - (_hi - _lo)),
                                       0.0)
                            _v0 = float(np.nanmin(voltage))
                            _v1 = float(np.nanmax(voltage))
                            _lo, _hi = max(_v0, _lo - _pad), min(_v1, _hi + _pad)
                            if _hi > _lo:
                                ax.set_xlim(_lo, _hi)
                except Exception:
                    pass

                # Fit panel formatting
                ax.set_title(f'Cycle {cycle} — {step}', fontsize=11)
                ax.set_ylabel('dQ/dV / mAh V$^{-1}$ g$^{-1}$',
                             fontsize=11)
                ax.tick_params(axis='both', labelsize=10, direction='in',
                              top=True, right=True)
                ax.axhline(y=0, color='grey', linewidth=0.4)
                ax.legend(fontsize=8, loc='best')
                plt.setp(ax.get_xticklabels(), visible=False)

                # Residual panel formatting
                ax_res.set_xlabel('Voltage / V', fontsize=11)
                ax_res.set_ylabel('Resid.', fontsize=9)
                ax_res.tick_params(axis='both', labelsize=9,
                                  direction='in', top=True, right=True)
                for sp in ax_res.spines.values():
                    sp.set_linewidth(0.6)

        fig.suptitle(f'{composition} — multi-peak fits', fontsize=14,
                     fontweight=FIGURE_TITLE_WEIGHT)

        if save_location:
            fpath = os.path.join(
                save_location, f'{name}_peak_fits.{image_format(params, file_format)}'
            )
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        _caption(
            f"Figure X. Multi-peak pseudo-Voigt fits to the differential "
            f"capacity of {composition}, for "
            f"{len(cycles_to_show_actual)} representative cycles. Grey points "
            f"are the processed data, the solid line the summed model, "
            f"filled curves the individual named components"
            + (" and the dashed line the polynomial baseline"
               if _drew_baseline else
               "; the model carries no free background, so every part of "
               "the curve shown is charge attributed to a named component "
               "or reported as unattributed")
            + f". Each panel is annotated with its "
            f"coefficient of determination; the lower trace of each pair is "
            f"the residual, on the same voltage axis. Fitted parameters for "
            f"every cycle are tabulated in "
            f"{name}_fitted_parameters.csv.")

        plt.show()
        plt.close(fig)


# --- is a peak AREA a capacity on this dataset? ----------------------------
# `analyse.flag_low_fidelity` answers that per peak and writes the answer into
# `tracking.summary` as `area_retention_trustworthy` / `area_withheld_reason`.
# Until 1.9.0.36 no figure read it, so on a run whose START_HERE says in bold
# "Peak areas here are not capacities" the peak-trend figure was captioned
# "peak area, which is proportional to the capacity stored by that process" —
# and that caption is the line that gets pasted into a manuscript.
def _areas_are_capacities(summary, step=None):
    """False if any tracked peak (of this step) had its area caveated."""
    if not summary:
        return True, ""
    _reason = ""
    _ok = True
    for _k, _s in summary.items():
        if step is not None and _s.get("step") != step:
            continue
        if _s.get("area_retention_trustworthy") is False:
            _ok = False
            _reason = _reason or str(_s.get("area_withheld_reason") or "")
    return _ok, _reason


def plot_tracked_trends(tracked, user_parameters,
                         save_location=None, file_format=None):
    """
    Three-panel trend plots: centre voltage, peak area, FWHM vs cycle.

    Each tracked peak gets its own colour. Shoulder peaks use square
    markers; ghost peaks are omitted entirely from trend plots as their
    area values are meaningless.
    """
    for name, tr in tracked.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        tdf = tr['tracked_df']
        ghost_peaks = tr.get('ghost_peaks', set())
        _summary = tr.get('summary') or {}

        for step in ['Charge', 'Discharge']:
            step_df = tdf[
                (tdf['step'] == step) & (tdf['status'] == 'tracked')
            ]
            if step_df.empty:
                continue

            # Exclude ghost peaks from trend plots
            plot_ids = sorted([
                pid for pid in step_df['tracked_peak_id'].unique()
                if (step, int(pid)) not in ghost_peaks
            ])
            if not plot_ids:
                continue
            colours = categorical_colours(len(plot_ids))

            fig, (ax1, ax2, ax3) = plt.subplots(
                3, 1, figsize=(10, 10), sharex=True)
            fig.suptitle(f'{composition} — {step} peak evolution',
                        fontsize=14, fontweight=FIGURE_TITLE_WEIGHT)
            _n_band_clipped = 0
            _split_seen = False

            for i, pid in enumerate(plot_ids):
                pk = step_df[
                    step_df['tracked_peak_id'] == pid
                ].sort_values('cycle')

                ref_v = pk['reference_voltage'].iloc[0]
                is_sh = pk['is_shoulder'].iloc[0]
                is_tr = pk.get('is_truncated', pd.Series([False])).iloc[0]
                label = f'Peak {pid} ({ref_v:.3f} V)'
                if is_sh:
                    label += ' [sh]'
                if is_tr:
                    label += ' [trunc]'
                clr    = colours[i]
                marker = 's' if is_sh else 'o'

                if 'reliable' in pk.columns:
                    pk_good = pk[pk['reliable']]
                    pk_bad  = pk[~pk['reliable']]
                else:
                    pk_good = pk
                    pk_bad  = pk.iloc[0:0]

                for ax, col, ylabel in [
                    (ax1, 'centre',         'Centre voltage / V'),
                    (ax2, 'amplitude_area', 'Peak area (∝ capacity)'),
                    (ax3, 'fwhm',           'FWHM / mV'),
                ]:
                    scale = 1000 if col == 'fwhm' else 1
                    ax.plot(pk['cycle'], pk[col] * scale,
                            color=clr, linewidth=0.5, alpha=0.3)
                    ax.plot(pk_good['cycle'], pk_good[col] * scale,
                            color=clr, marker=marker, markersize=4,
                            linewidth=0, label=label)
                    if not pk_bad.empty:
                        ax.plot(pk_bad['cycle'], pk_bad[col] * scale,
                                color=clr, marker=marker, markersize=4,
                                linewidth=0, fillstyle='none', alpha=0.35)
                    # A +/-1 sigma envelope on all three panels. Until
                    # 1.9.0.4 only the centre carried one, although the fit
                    # returns a standard error for every parameter and
                    # `track_peaks` carries them all into `tracked_df`: the
                    # area panel simply was not reading `amplitude_stderr`,
                    # and FWHM is 2 x sigma exactly (`fitting._FWHM_FACTOR`),
                    # so its error is 2 x sigma_stderr. A trend drawn without
                    # them invites the reader to believe a wobble that the
                    # fit itself calls noise.
                    _err_col, _err_scale, _clip = {
                        'centre':         ('centre_stderr',    1.0,
                                           TREND_BAND_CLIP_V),
                        'amplitude_area': ('amplitude_stderr', 1.0, None),
                        'fwhm':           ('sigma_stderr',     2.0,
                                           TREND_BAND_CLIP_V),
                    }[col]
                    if (_err_col in pk_good.columns and not pk_good.empty
                            and pk_good[_err_col].notna().any()
                            and len(pk_good) >= 2):
                        se = pk_good[_err_col].fillna(0) * _err_scale
                        if col == 'fwhm' and 'sigma_r_stderr' in pk_good.columns \
                                and pk_good['sigma_r_stderr'].notna().any():
                            _split_seen = True
                        if col == 'fwhm' and 'sigma_r_stderr' in pk_good.columns:
                            # FWHM IS NO LONGER 2 x SIGMA. With a split
                            # lineshape it is sigma + sigma_r, so its error is
                            # the two errors ADDED — not in quadrature: the
                            # widths are tied by one shared ratio
                            # (sigma_r = k * sigma), so they move together and
                            # a quadrature sum would understate a correlated
                            # pair. On LTO, where k is about 9, the old
                            # 2 x sigma_stderr drew a band five times too
                            # narrow.
                            _sr = pk_good['sigma_r_stderr']
                            if _sr.notna().any():
                                se = (pk_good['sigma_stderr'].fillna(0)
                                      + _sr.fillna(0))
                        if _clip is not None:
                            # Clipped for display: one unconverged cycle
                            # otherwise draws a triangle across the panel.
                            # COUNTED, because `_BAND_NOTE` in the caption
                            # tells the reader to look for a band wider than
                            # the trend it describes — and after a 20 mV cap
                            # no such band can be drawn on these two panels,
                            # so the instruction was unfollowable. NMC cell B
                            # has a `centre_stderr` of 14.7 V on a row flagged
                            # reliable; P3 cell C one of 153 V. Both rendered
                            # as a tidy +/-20 mV.
                            _n_band_clipped += int((se > _clip).sum())
                            se = se.clip(upper=_clip)
                        else:
                            # No natural ceiling on an area, so clip to the
                            # peak's own median area instead of a constant.
                            _med = float(pk_good['amplitude_area'].abs()
                                         .median())
                            if np.isfinite(_med) and _med > 0:
                                se = se.clip(upper=_med)
                        _lo = (pk_good[col] - se) * scale
                        if col in ('fwhm', 'amplitude_area'):
                            # A NEGATIVE WIDTH IS NOT A LOWER BOUND. On LTO
                            # cell C cycle 10 `sigma_stderr` (3.603e-3)
                            # exceeded `sigma` (3.000e-3) — a component
                            # sitting on its floor, whose error is therefore
                            # not a measurement either — and the band was
                            # drawn through zero into negative FWHM. Clipped
                            # here so the figure cannot state an impossibility;
                            # the flag that says WHY belongs in the table.
                            _lo = _lo.clip(lower=0.0)
                        ax.fill_between(
                            pk_good['cycle'], _lo,
                            (pk_good[col] + se) * scale,
                            alpha=0.15, color=clr)

            _cap_ok, _cap_why = _areas_are_capacities(_summary, step)
            for ax, ylabel in [
                (ax1, 'Centre voltage / V'),
                (ax2, 'Peak area (∝ capacity)' if _cap_ok
                 else 'Peak area / mAh V⁻¹ g⁻¹ (NOT a capacity)'),
                (ax3, 'FWHM / mV'),
            ]:
                ax.set_ylabel(ylabel, fontsize=12)
                ax.tick_params(axis='both', labelsize=10, direction='in',
                              top=True, right=True)
                ax.legend(fontsize=8, loc='best', framealpha=0.7)
                for sp in ax.spines.values():
                    sp.set_linewidth(0.8)
            ax3.set_xlabel('Cycle number', fontsize=12)
            _force_integer_cycles(ax3)

            plt.tight_layout()
            if save_location:
                fpath = os.path.join(save_location,
                    f'{name}_peak_trends_{step}.{image_format(params, file_format)}')
                fig.savefig(fpath, dpi=300, bbox_inches='tight')
                saved(fpath)
            _caption(
                f"Figure X. Evolution of the fitted {step.lower()} peaks of "
                f"{composition} with cycle number: (a) peak centre, (b) peak "
                + ("area, which is proportional to the capacity stored by "
                   "that process, " if _cap_ok else
                   "area — which on this dataset is NOT proportional to the "
                   "capacity stored by that process"
                   + (f" ({_cap_why})" if _cap_why else "")
                   + ", so the panel shows how the fitted area changes and "
                     "not how much charge the process carried, ")
                + f"and (c) full width at half maximum. Filled "
                f"markers are cycles in which the fit met the reliability "
                f"criteria; open markers are cycles in which it did not, and "
                f"are shown for completeness but excluded from the trends. "
                + f"{_BAND_NOTE} "
                + ("The width band is the standard errors on sigma and "
                   "sigma_r ADDED, since this model's FWHM is their sum and "
                   "the two are tied by one ratio, so a quadrature sum would "
                   "understate a correlated pair."
                   if _split_seen else
                   "The width band is twice the standard error on sigma, "
                   "since FWHM = 2\u03c3 exactly for the symmetric form of "
                   "this model.")
                + (f" The centre and width bands are capped at \u00b1"
                   f"{1000 * TREND_BAND_CLIP_V:.0f} mV for legibility; "
                   f"{_n_band_clipped} point(s) have a larger standard error "
                   f"than that and are drawn narrower than they are. The "
                   f"unclipped values are in the tracked-peaks table."
                   if _n_band_clipped else ""))
            plt.show()
            plt.close(fig)


def _build_ref_voltage_lookup(tracked_peaks, name):
    """
    Build a fixed peak_id → reference_voltage lookup for labelling.
    Uses the reference_peaks source (which always has ref_V) rather
    than per-cycle tracked data (which may not include all peaks in
    every cycle, causing nan labels).
    """
    lookup = {}
    ref_peaks = tracked_peaks[name]['reference_peaks']
    for step in ['Charge', 'Discharge']:
        rp = ref_peaks.get(step, pd.DataFrame())
        if rp is not None and not rp.empty:
            for _, pk in rp.iterrows():
                key = (step, int(pk['peak_id']))
                lookup[key] = pk['voltage']
    return lookup


def plot_delta_v(delta_v_results, user_parameters,
                  save_location=None, file_format=None):
    """Plot ΔV evolution with error bands and linear trend."""
    for name, dvr in delta_v_results.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        dv_df = dvr['delta_v_df']
        dv_changes = dvr.get('delta_v_changes', {})

        if dv_df.empty:
            continue

        pair_labels = sorted(dv_df['pair_label'].unique())
        colours = categorical_colours(len(pair_labels))

        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        _n_clipped = 0
        for i, label in enumerate(pair_labels):
            pd_pair = dv_df[dv_df['pair_label'] == label].sort_values('cycle')
            rate_str = (f' (net {dv_changes[label][0]:+.0f} mV)'
                       if label in dv_changes else '')

            ax.plot(pd_pair['cycle'], pd_pair['delta_v_mV'],
                   color=colours[i], marker='o', markersize=5,
                   linewidth=1.2, label=f'{label}{rate_str}')

            if pd_pair['delta_v_stderr_mV'].notna().any():
                se = pd_pair['delta_v_stderr_mV'].fillna(0)
                # CLIPPED FOR DISPLAY, AND THE FIGURE SAYS SO. The cap keeps
                # one absurd standard error from flattening the whole panel —
                # P3 cell C carries a `delta_v_stderr_mV` of 498245 — but the
                # caption below asserts the band IS the fit's own standard
                # error, so on cells A and B (239 mV and 196 mV) a reader was
                # shown a tight +/-20 mV band under a sentence promising the
                # real one. Clipping is the right choice for the drawing;
                # silence about it is not.
                _n_clipped += int((se > DELTA_V_BAND_CLIP_MV).sum())
                se = se.clip(upper=DELTA_V_BAND_CLIP_MV)
                ax.fill_between(pd_pair['cycle'],
                    pd_pair['delta_v_mV'] - se,
                    pd_pair['delta_v_mV'] + se,
                    alpha=0.15, color=colours[i])

        ax.set_xlabel('Cycle number', fontsize=13)
        _force_integer_cycles(ax)
        ax.set_ylabel('ΔV (charge − discharge) / mV', fontsize=13)
        ax.set_title(f'{composition} — polarisation evolution (ΔV)',
                    fontsize=14)
        if _n_clipped:
            ax.text(0.99, 0.02,
                    f'shaded band capped at ±{DELTA_V_BAND_CLIP_MV:.0f} mV; '
                    f'{_n_clipped} point(s) have a larger standard error',
                    transform=ax.transAxes, ha='right', va='bottom',
                    fontsize=8, style='italic', alpha=0.8)
        ax.tick_params(axis='both', labelsize=11, direction='in',
                      top=True, right=True)
        ax.legend(fontsize=9, framealpha=0.7)
        for sp in ax.spines.values():
            sp.set_linewidth(0.8)

        plt.tight_layout()
        if save_location:
            fpath = os.path.join(save_location,
                                f'{name}_delta_v.{image_format(params, file_format)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)
        _caption(
            f"Figure X. Polarisation of {composition}, expressed as the "
            f"separation \u0394V between the charge and discharge peak "
            f"centres of each redox couple, against cycle number. The "
            f"legend gives each couple's net change over the cycles in which "
            f"it was tracked continuously. A widening |\u0394V| indicates "
            f"rising internal "
            f"resistance. {_BAND_NOTE} Here the band is the two centre "
            f"errors added in quadrature, since \u0394V is their difference."
            + (f" The shaded band is capped at \u00b1"
               f"{DELTA_V_BAND_CLIP_MV:.0f} mV for legibility; "
               f"{_n_clipped} point(s) have a larger standard error than "
               f"that and are drawn narrower than they are. The unclipped "
               f"values are in the delta-V table."
               if _n_clipped else ""))
        plt.show()
        plt.close(fig)


def plot_capacity_attribution(attribution, tracked_peaks,
                               user_parameters,
                               save_location=None, file_format=None):
    """
    Two-panel capacity attribution: absolute areas + stacked fractions.
    Uses fixed reference voltage lookup for consistent labels.
    """
    for name, step_dfs in attribution.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        ref_v_lookup = _build_ref_voltage_lookup(tracked_peaks, name)

        for step, adf in step_dfs.items():
            if adf.empty:
                continue

            frac_cols = sorted([c for c in adf.columns
                                if c.startswith('peak')
                                and c.endswith('_fraction')])
            if not frac_cols:
                continue

            # Build labels from reference lookup
            peak_info = []
            for col in frac_cols:
                pid_str = col.replace('_fraction', '')
                pid = int(pid_str.replace('peak', ''))
                ref_v = ref_v_lookup.get((step, pid), np.nan)
                ref_str = f'{ref_v:.3f}' if pd.notna(ref_v) else '?'
                peak_info.append((pid, pid_str, ref_str, col))

            colours = categorical_colours(len(peak_info))

            # Two vertically stacked panels: journal width, taller
            fig, (ax1, ax2) = plt.subplots(
                2, 1,
                figsize=(figure_width_inches, figure_height_inches * 1.5),
                sharex=True)
            fig.suptitle(
                f'{composition} — {step} capacity attribution',
                fontsize=14, fontweight=FIGURE_TITLE_WEIGHT)

            # Panel 1: Absolute areas
            for i, (pid, pid_str, ref_str, frac_col) in enumerate(peak_info):
                area_col = f'{pid_str}_area'
                if area_col in adf.columns:
                    valid = adf[adf[area_col].notna()]
                    if not valid.empty:
                        ax1.plot(valid['cycle'], valid[area_col],
                                color=colours[i], marker='o', markersize=4,
                                linewidth=1,
                                label=f'Peak {pid} ({ref_str} V)')

            # THE SAME GATE THE TREND FIGURE APPLIES. `plot_tracked_trends`
            # asks `_areas_are_capacities` before it writes this label, and
            # this function — which is titled "capacity attribution" and is the
            # more likely of the two to be quoted as one — did not, although
            # it is handed the same `tracked_peaks` and the verdict is on it.
            # NNM cell B has all ten peaks marked
            # `area_retention_trustworthy=False` with the reason "the dQ/dV
            # curve accounts for only 5% of this step's delivered capacity, so
            # a peak area is not a capacity", and its two figures in one folder
            # read "NOT a capacity" and "(∝ capacity)".
            _cap_ok, _cap_why = _areas_are_capacities(
                (tracked_peaks.get(name) or {}).get('summary') or {}, step)
            ax1.set_ylabel('Peak area (∝ capacity)' if _cap_ok
                           else 'Peak area / mAh V⁻¹ g⁻¹ (NOT a capacity)',
                           fontsize=12)
            ax1.legend(fontsize=8, loc='best', framealpha=0.7)

            # Panel 2: Stacked fractions
            # WITHHELD IS NOT ZERO. `capacity_attribution` writes NaN for a
            # cycle whose peak/background split is not determined by the
            # data — deliberately, "the SHARE is blank because it is not a
            # measurement". `fillna(0)` turned each of those into a bar of
            # height zero, drawn identically to a real collapse, under a
            # caption saying withheld cycles are not plotted. On a dataset
            # where withholding fires on every cycle the figure was a flat
            # row of zeros captioned "the stack is free to fall below 100%
            # as processes are lost".
            _plotted = adf.copy()
            _kept = np.zeros(len(_plotted), dtype=bool)
            for _, _, _, _fc in peak_info:
                _kept |= _plotted[_fc].notna().values
            _plotted = _plotted[_kept]
            _n_withheld = int(len(adf) - len(_plotted))
            # ...AND WITHHELD PER PEAK IS NOT ZERO EITHER. The row filter above
            # keeps a cycle when ANY peak has a fraction, and the `fillna(0)`
            # below then flattens every OTHER peak's withheld cell in that row
            # to a bar of height zero — a real collapse to the eye. On NNM cell
            # A's discharge, peak 1 is NaN on 122 of 128 cycles and peak 5 on
            # 120, and because no row was entirely NaN the withheld count came
            # out as 0 and the explanatory box never appeared: two processes
            # read as contributing exactly nothing from cycle 4 onward. The
            # geometry of a stacked bar cannot show an absent segment, so the
            # count is said instead — in the legend, per peak, where the
            # reader is already looking to identify the colour.
            _cells = int(len(_plotted) * len(peak_info))
            _cells_withheld = 0
            bottoms = np.zeros(len(_plotted))
            for i, (pid, pid_str, ref_str, frac_col) in enumerate(peak_info):
                _miss = int(_plotted[frac_col].isna().sum())
                _cells_withheld += _miss
                fracs = _plotted[frac_col].fillna(0).values * 100
                ax2.bar(_plotted['cycle'], fracs, bottom=bottoms,
                       color=colours[i], alpha=0.8, width=0.8,
                       label=(f'Peak {pid} ({ref_str} V)'
                              + (f' — withheld on {_miss}/{len(_plotted)}'
                                 if _miss else '')))
                bottoms += fracs
            if _n_withheld or _cells_withheld:
                _msg = []
                if _n_withheld:
                    _msg.append(f"{_n_withheld} of {len(adf)} cycles withheld")
                if _cells_withheld:
                    _msg.append(f"{_cells_withheld} of {_cells} peak-cycle "
                                f"shares withheld and drawn as zero")
                ax2.text(0.5, 0.5,
                         ("\n".join(_msg) + " —\nthe peak/background split is "
                          "not\ndetermined by the data"
                          if not len(_plotted) else "; ".join(_msg)),
                         transform=ax2.transAxes, ha='center',
                         va='center' if not len(_plotted) else 'top',
                         fontsize=10 if not len(_plotted) else 8,
                         color='#666666',
                         bbox=dict(boxstyle='round,pad=0.4',
                                   facecolor='white', alpha=0.85,
                                   edgecolor='#999999'))

            ax2.set_ylabel('Capacity fraction vs reference total / %',
                           fontsize=12)
            ax2.set_xlabel('Cycle number', fontsize=12)
            ax2.axhline(100, color='0.4', linewidth=0.8, linestyle='--')
            _stack_top = float(np.nanmax(bottoms)) if len(bottoms) else 100.0
            ax2.set_ylim(0, max(110, _stack_top * 1.05))
            ax2.legend(fontsize=8, loc='upper right', framealpha=0.7)

            for ax in [ax1, ax2]:
                ax.tick_params(axis='both', labelsize=10, direction='in',
                              top=True, right=True)
                for sp in ax.spines.values():
                    sp.set_linewidth(0.8)

            plt.tight_layout()
            if save_location:
                fpath = os.path.join(save_location,
                    f'{name}_capacity_attribution_{step}.{image_format(params, file_format)}')
                fig.savefig(fpath, dpi=300, bbox_inches='tight')
                saved(fpath)
            _caption(
                f"Figure X. Capacity attribution for the {step.lower()} of "
                f"{composition}: (a) the fitted area of each tracked peak "
                f"against cycle number, and (b) the same areas as a "
                f"percentage of the summed peak area at the reference cycle, "
                f"stacked. A fixed reference denominator is used throughout, "
                f"so the stack is free to fall below 100% as processes are "
                f"lost — a per-cycle denominator would redistribute lost "
                f"capacity onto the survivors. Cycles for which the "
                f"attribution is not determined by the data are withheld "
                f"rather than plotted; see the closure interval recorded in "
                f"{name}_fitted_parameters.csv."
                + (f" A withheld share for an individual peak in a cycle that "
                   f"is otherwise plotted cannot be left out of a stacked bar, "
                   f"so it is drawn at zero height; {_cells_withheld} of "
                   f"{_cells} peak-cycle shares here are withheld rather than "
                   f"measured, and the legend gives the count per peak."
                   if _cells_withheld else "")
                + ("" if _cap_ok else
                   f" The areas in panel (a) are NOT capacities on this "
                   f"dataset" + (f" — {_cap_why}" if _cap_why else "") + "."))
            plt.show()
            plt.close(fig)


# =============================================================================
# ADAPTERS — 1.9.0 objects into the shapes the ported code expects
# =============================================================================
# This is the only new code in this module. Everything above is 1.8.7's.

def as_processed_dqdv(dataset, signals, params=None, *, window=None,
                      profile=None):
    """
    `{name: {'data': {(cycle, step): DataFrame}, 'voltage_range': (lo, hi),
              'parameters_used': {...}}}` — Module 1's shape.

    The frames carry exactly the three columns the plotting code reads, in
    voltage order, which is the order `signal.half_cycle_report` already
    returns them in.
    """
    data = {}
    for (cycle, step), sig in signals.items():
        if sig.voltage.size == 0:
            continue
        data[(int(cycle), str(step))] = pd.DataFrame({
            "Voltage": sig.voltage,
            "dQ/dV_raw": sig.dqdv_raw,
            "dQ/dV_processed": sig.dqdv,
        })
    if window is None:
        allv = [d["Voltage"] for d in data.values()]
        window = ((float(min(v.min() for v in allv)),
                   float(max(v.max() for v in allv))) if allv
                  else (np.nan, np.nan))
    used = dict(params or {})
    if profile is not None:
        used.setdefault("profile_class", profile.get("class")
                        if isinstance(profile, dict) else profile)
    return {dataset.name: {"data": data, "voltage_range": tuple(window),
                           "parameters_used": used}}


def as_detected_peaks(detection):
    """`{name: {'peaks', 'reference_cycle', 'reference_peaks', ...}}` —
    Module 3's shape."""
    return {detection.name: {
        "peaks": detection.peaks,
        "reference_cycle": detection.reference_cycle,
        "reference_cycle_reason": detection.reference_reason,
        "reference_peaks": detection.reference_peaks,
        "parameters_used": (detection.spec.as_dict() if detection.spec
                            else {}),
    }}


class _FitView:
    """
    What `plot_fit_quality` thinks is an lmfit `ModelResult`.

    It answers `.best_fit` and `.eval_components(x=...)` from
    `fitting.evaluate`, which rebuilds the component curves analytically from
    the stored parameters through lmfit's own model classes. No refit, and no
    `ModelResult` anywhere near a process boundary — see the module docstring.
    """
    __slots__ = ("_result", "_voltage", "_comps", "best_fit")

    def __init__(self, result, voltage):
        self._result = result
        self._voltage = np.asarray(voltage, float)
        self._comps = evaluate(result, self._voltage)
        self.best_fit = self._comps["total"]

    def eval_components(self, x=None):
        if x is None:
            comps = self._comps
        else:
            x = np.asarray(x, float)
            comps = (self._comps
                     if (x.shape == self._voltage.shape
                         and np.array_equal(x, self._voltage))
                     else evaluate(self._result, x))
        return {k: v for k, v in comps.items() if k != "total"}


def as_fit_results(fit_results, signals, params_df=None, name=""):
    """
    `{name: {'fits': {(cycle, step): {...}}, 'parameters_df', 'fit_summary'}}`
    — Module 4's shape, with `_FitView` standing in for the ModelResult.
    """
    fits, summary = {}, []
    for fr in fit_results:
        key = fr.get("key")
        if key is None:
            continue
        cycle, step = int(key[0]), str(key[1])
        sig = signals.get((cycle, step))
        view = (_FitView(fr, sig.voltage)
                if (fr.get("success") and sig is not None
                    and sig.voltage.size) else None)
        # The `sig is not None` guard above is undone by reading sig.dqdv
        # here, and the value is discarded three lines down when view is
        # None anyway — so the branch could only ever raise.
        resid = (None if view is None
                 else np.abs(sig.dqdv) - view.best_fit)
        fits[(cycle, step)] = {
            "success": bool(fr.get("success")),
            "parameters": fr.get("components", []),
            "lmfit_result": view,
            "r_squared": fr.get("r_squared", np.nan),
            "reduced_chi_sq": fr.get("redchi", np.nan),
            # lmfit's ModelResult.residual is data - model; matched here so
            # the difference panel keeps its sign.
            "residuals": (resid if view is not None else None),
        }
        summary.append(dict(dataset=name, cycle=cycle, step=step,
                            # `success`, not `fit_success`: plot_fit_quality
                            # selects its worst converged and its failed
                            # half-cycles from this column by that name.
                            success=bool(fr.get("success")),
                            r_squared=fr.get("r_squared", np.nan),
                            reduced_chi_sq=fr.get("redchi", np.nan),
                            n_peaks=len(fr.get("components", [])),
                            seconds=fr.get("seconds", np.nan)))
    return {name: {"fits": fits,
                   "parameters_df": (params_df if params_df is not None
                                     else pd.DataFrame()),
                   "fit_summary": pd.DataFrame(summary)}}


def as_tracked(tracking):
    """`{name: {'tracked_df', 'summary', 'reference_peaks', ...}}` —
    Module 5's shape."""
    return {tracking.name: {
        "tracked_df": tracking.tracked_df,
        "summary": tracking.summary,
        "reference_peaks": tracking.reference_peaks,
        "reference_cycle": tracking.reference_cycle,
        "ghost_peaks": tracking.ghost_peaks,
        "cell_discontinuity": tracking.cell_discontinuity,
    }}


def as_user_parameters(dataset, *, composition=None, cell_id=None,
                       key_cycles=None, colour_palette="viridis_r",
                       battery_chemistry="Li-ion", charge_rate_c=None,
                       **extra):
    """
    The subset of Cell 3's parameters the plotting code actually reads:
    `composition`, `cell_id`, `key_cycles`, `colour_palette`,
    `battery_chemistry`, `charge_rate_c` and `anode_labels_swapped`.

    `anode_labels_swapped` is taken from the Dataset rather than asked for —
    it is a fact about how the file was read, not a preference.
    """
    p = dict(
        composition=composition or dataset.meta.get("composition",
                                                    dataset.name),
        cell_id=cell_id or dataset.meta.get("cell_id", ""),
        key_cycles=list(key_cycles) if key_cycles else list(DEFAULT_KEY_CYCLES),
        colour_palette=colour_palette,
        battery_chemistry=battery_chemistry,
        charge_rate_c=(charge_rate_c if charge_rate_c is not None
                       else dataset.meta.get("charge_rate_c",
                                             "the specified")),
        anode_labels_swapped=bool(getattr(dataset, "labels_swapped", False)),
        electrode_type=getattr(dataset, "electrode_type", "Positive"),
    )
    p.update(extra)
    return {dataset.name: p}


In [ ]:
# DEFINITIONS J1 — CYCLING: THE SUMMARY TABLE, VOLTAGE PROFILES AND CYCLE LIFE
# =============================================================================
"""
Capacity, retention, fade, energy, power, voltage, efficiency and rate.

Everything here is 1.8.7's Cells 5 to 13, ported without editing the bodies.
Those cells were top-level scripts driving straight off `electrochemical_data`
and `user_parameters`, so the port was mechanical: each cell's statements were
wrapped in a function whose parameters are exactly the globals it read, its
imports dropped, and its constants hoisted to module scope. Nothing inside a
body was rewritten, because the figures must not move.

The notebook cell each one is called from is 1.9.0's, not 1.8.7's — the
column on the right was the 1.8.7 number and stayed put for three releases
while the console printed "Cell 9b complete" at a reader sitting in Cell 6.

    cycling_summary             6a   the per-cycle table, the headline
                                     metrics, and the incomplete-cycle flags
                                     every later plot reads
    voltage_profiles            6b   every cycle, V against capacity
    voltage_profiles_key        6b   the same overlay, key cycles only
    cycle_life                  6b   capacity and CE against cycle
    comparative_capacity        6b   every dataset on one axis
    capacity_retention          6c   retention against a reference cycle
    fade_rate                   6c   loss per cycle, and cumulative
    power_and_energy            6c   three normalisations, and the Ragone
    average_discharge_voltage   6c   mean discharge voltage against cycle
    energy_efficiency           6c   round-trip
    rate_capability             6c   capacity grouped by C-rate

`cycling_summary` comes first for a reason: it returns `all_cycle_tables`, and
every plot after it reads that to exclude incomplete cycles. Pass it through.
Passing None reproduces the not-yet-run branch exactly, including its notice.

Two constants are NOT hoisted
-----------------------------
`MARKER_SIZE` is 7 in Cells 7-9 and 6 in Cells 9b, 11 and 12; `YAXIS_MIN` is 0
in Cell 9 and 40 in Cell 12; `YAXIS_MAX` and `VISUAL_OUTLIER_THRESHOLD` differ
similarly. They stay local to the functions that define them. Hoisting any of
them would have quietly changed a figure — which is the failure mode this
whole port exists to avoid.

What this module does not do
----------------------------
It does not judge. Capacity, retention and fade are arithmetic on the cycler's
own counters; whether a given cycle's numbers mean anything is
`analyse.integrity`'s question, and the two are deliberately kept apart.
"""


import contextlib
import functools
import io as _io
import os
import re

import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors            # noqa: F401  (used by ported code)
import matplotlib.ticker as mticker            # noqa: F401  (used by ported code)

try:                                            # Cell 6 only
    import seaborn as sns
except ImportError:                             # pragma: no cover
    sns = None

# Imported as a MODULE, not as values. `from .plots import
# figure_width_inches` binds the number at import time, so
# `plots.set_figure_size()` moved the dQ/dV figures and left every cycling
# figure at the old size — and the flattened notebook, having one namespace,
# did the opposite. Two builds, two different sets of figures.
# NOT ALIASED. `flatten.strip()` deletes every relative import, so in the flat
# notebook the name has to be the one the defining module used: an aliased
# import leaves the alias defined nowhere, and the build's free-name check
# does not catch it because the use is inside a function body. Same family as
# the late-relative-import bug the build now refuses outright.


# Okabe-Ito, colourblind-safe. IMPORTED, not restated — `plots` is the one
# home for the run's categorical palette, the same arrangement as the
# charge/discharge colours further down. Two copies of a colour convention is
# how the charge colour came to mean two different things in one run.
MULTI_DATASET_PALETTE = CATEGORICAL_PALETTE
MULTI_DATASET_MARKERS = CATEGORICAL_MARKERS


# =============================================================================
# PORTED FROM 1.8.7 CELLS 5-13 — do not edit; the figures are the regression
# =============================================================================

LABEL_MODE = 'auto'
COMMON_XAXIS_SCALE = True
XAXIS_PADDING_FRACTION = 0.08
AUTO_OVERLAP_THRESHOLD = 0.5
RETENTION_REFERENCE_CYCLE = 2
# THE FADE RATE IS DOCUMENTED AS EXCLUDING FORMATION AND WAS EXCLUDING CYCLE 1.
#
# `FADE_RATE_START_CYCLE = 2` is a guess about where formation ends, printed
# beside a line that says "excluding formation" — while the same run measures
# formation ending at cycles 4, 5 and 4 on the LTO triplicate
# (`detect.formation_end`, from the coulombic efficiency the same table just
# computed). The reported fade rate was **17-22% steeper** than it would be
# from the measured end, consistently across all three cells, because the
# cycles between the constant and the measurement are the steepest in the run.
#
# So the start cycle is now MEASURED per dataset and the constant below is
# only the fallback, used when the record is too short to establish formation
# at all. The cycle used and the reason are printed, because a fade rate is
# meaningless without saying what it excluded.
FADE_RATE_FROM_MEASURED_FORMATION = True
FADE_RATE_START_CYCLE = 2
OVERRIDE_KEY_CYCLES = None
# Imported from plots, not restated. These were amber-charge / blue-discharge
# here and blue-charge / vermillion-discharge there, so one run showed blue
# meaning two different things. See plots.COLOUR_CHARGE for the convention.
COLOUR_EFFICIENCY = '#000000'  # black
MARKER_CHARGE = 's'       # square
MARKER_DISCHARGE = 'o'    # circle
MARKER_EFFICIENCY = 'v'   # triangle down
COMMON_YAXIS_CAPACITY = True
# Hold the efficiency panel to a common floor and ceiling (40-105%, widened
# to fit the data) rather than letting it autoscale per cell. Read in
# `energy_efficiency`; for years it was read nowhere.
COMMON_YAXIS_EFFICIENCY = True
CAPACITY_YAXIS_PADDING = 0.08
EFFICIENCY_YMAX = 110  # CE y-axis upper limit (%)
EXCLUDE_INCOMPLETE = True
CE_START_CYCLE = 2
MARKERS = MULTI_DATASET_MARKERS
YAXIS_PADDING = 0.08
VISUAL_OUTLIER_FILTER = True
FADE_REFERENCE_CYCLE = RETENTION_REFERENCE_CYCLE  # from Cell 9; change there to affect both
ROLLING_WINDOW = 100
VOLTAGE_YMIN = None  # e.g., 2.8
VOLTAGE_YMAX = None  # e.g., 3.8
VOLTAGE_YAXIS_PADDING = 0.02  # V above/below data range
DRIFT_REFERENCE_CYCLE = 2
MARKERS_EE = 'D'   # diamond for energy efficiency
MARKERS_CE = 'o'   # circle for coulombic efficiency
COLOUR_EE = '#D55E00'   # vermillion
COLOUR_CE = '#0072B2'   # blue
EE_START_CYCLE = 2
CURRENT_GROUPING_TOLERANCE = 0.05
MIN_CYCLES_PER_RATE = 2
# How many clean cycles the reference and return blocks each need before a
# recovery is reported at all. Two is the fewest that can carry a mean.
RATE_RECOVERY_MIN_REFERENCE = 2

# THE LABEL SNAPS; THE NUMBER DOES NOT. A measured C-rate is
# `current / (mass x theoretical capacity)`, so it inherits whatever
# theoretical capacity the operator entered. Jiaqi set her currents for C/20
# against a capacity 5% away from the 168 mAh/g entered here, and the
# measured rate came out at C/19.0 — which then appeared as "C/19" on every
# figure and read as a mistake.
#
# So the label is snapped to the nearest conventional rate when the measured
# value is within RATE_LABEL_SNAP of it, and the measured value is exported
# beside it in `c_rate_measured`. Nothing downstream computes on the label.
# The conventional set is spaced far enough apart (C/20 to C/10 is a factor
# of two) that a 10% window cannot merge two rates an experimenter meant to
# distinguish.
RATE_LABEL_SNAP = 0.10
_CONVENTIONAL_RATES = (1/100., 1/50., 1/20., 1/10., 1/5., 1/4., 1/3., 1/2.,
                       1.0, 2.0, 3.0, 5.0, 10.0, 20.0)

BAR_COLOUR = '#0072B2'
RECOVERY_COLOUR = '#009E73'


# One definition of "this cycle's capacity is implausibly low for a figure".
# Six functions used to restate it as a local constant, quote that local in
# the console message, and then call the filter WITHOUT passing it — so
# editing the documented knob changed the sentence and not the data.
VISUAL_OUTLIER_FRACTION = 0.30


# =============================================================================
# THE CANONICAL PER-CYCLE QUANTITIES
# =============================================================================
# One definition each, stated here, used everywhere. Before 1.9.1 delivered
# capacity had FIVE implementations with different admission rules, coulombic
# efficiency had three (two returning 0% where the third returned NaN), and
# average discharge voltage had two names and a third hardcoded guess. Nothing
# compared them, so the number in a figure and the number in the CSV could
# differ with nothing in the run to say so.
#
#   delivered capacity   dQ across the half-cycle from the CYCLER'S OWN
#                        counter (max - min). Never the integral of dQ/dV:
#                        those two are compared by `quality.integral_fidelity`
#                        and using one to define the other would make that
#                        comparison meaningless.
#
#   coulombic efficiency Q_discharge / Q_charge for the SAME cycle, as a
#                        percentage. NaN — never 0 — when either half is
#                        absent. CE is read at the fourth decimal place
#                        (~99.96% is needed for 500 cycles), so a spurious 0%
#                        is not a rounding error: it asserts that the cell
#                        passed charge and returned none, which is a different
#                        physical claim from "we could not measure it".
#
#   average discharge V  the ENERGY-WEIGHTED mean, integral(V dQ) / dQ. This
#                        is the standard definition. A previous fallback used
#                        `voltage_upper_V * 0.85`, a hardcoded guess presented
#                        as a measurement; it is gone.
#
#   capacity retention   Q_discharge(n) / Q_discharge(n_ref), n_ref stated in
#                        the column name — see `RETENTION_REFERENCE_CYCLE`.


def _caption_window(params, df=None):
    """The window a CAPTION should quote: what the cell was cycled between.

    `df['Voltage'].min()/.max()` is the extent of the RECORD, which includes
    an open-circuit start and any excursion the cycler made — so the captions
    quoted 1.20-2.57 V for cell B and 1.20-2.68 V for the triplicate together,
    all cycled 1.2-2.5 V. Same defect as `plots._protocol_window` was written
    for, in a second module: a wrong experimental detail in text written to be
    pasted into a manuscript, and one that differed between nominally
    identical cells.

    Cell 3b measured the protocol window (the median of the per-cycle limits)
    and put it in `user_parameters`; this prefers it and falls back to the
    record only when it is absent.
    """
    lo = (params or {}).get('voltage_lower_V')
    hi = (params or {}).get('voltage_upper_V')
    try:
        lo, hi = float(lo), float(hi)
        if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
            return lo, hi
    except (TypeError, ValueError):
        pass
    if df is not None and 'Voltage' in getattr(df, 'columns', ()):
        v = pd.to_numeric(df['Voltage'], errors='coerce')
        return float(v.min()), float(v.max())
    return float('nan'), float('nan')


def _caption_window_all(user_parameters, fallback=None):
    """The window for a caption spanning EVERY dataset.

    Where the cells agree — a triplicate normally does — that one window is
    quoted. Where they genuinely differ, the widest is, because the caption is
    then describing more than one protocol and should say so honestly rather
    than pick one.
    """
    los, his = [], []
    for p in (user_parameters or {}).values():
        lo, hi = _caption_window(p)
        if np.isfinite(lo) and np.isfinite(hi):
            los.append(lo); his.append(hi)
    if los:
        return min(los), max(his)
    return fallback if fallback else (float('nan'), float('nan'))


def _has_tables(all_cycle_tables):
    """Did the caller hand us `cycling_summary`'s output?

    Was `'all_cycle_tables' in dir() and isinstance(...)`, in eleven copies.
    Inside a function `dir()` lists the LOCALS, and `all_cycle_tables` is a
    parameter — so the first clause was always True and only the isinstance
    ever decided anything. It is a 1.8.7 idiom: there this was a notebook
    global and asking whether it existed was a real question.
    """
    return isinstance(all_cycle_tables, dict)


def _run_image_format(user_parameters):
    """The format for a figure that spans every dataset.

    A run-wide figure has no single dataset to ask, so the datasets have to
    agree. Where they do not, PNG — the safe one to open — and the mismatch
    is the operator's to resolve.
    """
    fmts = {image_format(p) for p in (user_parameters or {}).values()}
    return fmts.pop() if len(fmts) == 1 else 'png'


def _get_endpoints(df_charge, df_discharge, unique_cycles, key_cycles):
    """
    Discharge and charge endpoint coordinates for the key cycles.

    ONE implementation. It was defined twice, nested inside two figure
    functions — byte-identical today, which is the dangerous state: the next
    edit lands in one of them and two figures start labelling their endpoints
    by different rules with nothing in the run to say so.
    """
    endpoints = []
    for cycle in key_cycles:
        if cycle not in unique_cycles:
            continue
        dc = df_discharge[df_discharge['Cycle'] == cycle]
        if not dc.empty:
            max_row = dc.loc[dc['Discharge_Capacity'].idxmax()]
            endpoints.append({
                'cycle': cycle, 'step': 'Discharge',
                'capacity': max_row['Discharge_Capacity'],
                'voltage': max_row['Voltage']
            })
        cc = df_charge[df_charge['Cycle'] == cycle]
        if not cc.empty:
            max_row = cc.loc[cc['Charge_Capacity'].idxmax()]
            endpoints.append({
                'cycle': cycle, 'step': 'Charge',
                'capacity': max_row['Charge_Capacity'],
                'voltage': max_row['Voltage']
            })
    return endpoints


def delivered_capacity(frame):
    """
    Per-cycle delivered capacity, both half-cycles, in mAh/g.

    Returns a frame indexed 0..n with columns Cycle, Charge_mAh_g,
    Discharge_mAh_g. Rows carrying neither capacity column are dropped;
    rows carrying one are kept, because some cycler firmware fills only the
    active column.
    """
    need = ("Cycle", "Step", "Charge_Capacity", "Discharge_Capacity")
    if not all(c in frame.columns for c in need):
        return pd.DataFrame(columns=["Cycle", "Charge_mAh_g",
                                     "Discharge_mAh_g"])
    d = frame.dropna(subset=["Cycle", "Step"])
    d = d[d["Charge_Capacity"].notna() | d["Discharge_Capacity"].notna()]
    dch = d[d["Step"] == "Discharge"].groupby("Cycle")["Discharge_Capacity"]
    chg = d[d["Step"] == "Charge"].groupby("Cycle")["Charge_Capacity"]
    out = pd.DataFrame({
        "Discharge_mAh_g": dch.max() - dch.min(),
        "Charge_mAh_g": chg.max() - chg.min(),
    })
    out.index.name = "Cycle"
    out = out.reset_index()
    if len(out):
        out["Cycle"] = pd.to_numeric(out["Cycle"],
                                     errors="coerce").astype("Int64")
        out = out.dropna(subset=["Cycle"])
        out["Cycle"] = out["Cycle"].astype(int)
    return out


def coulombic_efficiency(discharge, charge):
    """Q_discharge / Q_charge as a percentage. NaN, never 0. See above."""
    d = pd.to_numeric(pd.Series(discharge), errors="coerce")
    c = pd.to_numeric(pd.Series(charge), errors="coerce")
    ok = d.notna() & c.notna() & (c > 0) & (d > 0)
    return (d / c * 100.0).where(ok)


def mean_discharge_voltage(frame):
    """
    Energy-weighted mean discharge voltage per cycle: integral(V dQ) / dQ.

    Returns Cycle, Avg_Discharge_Voltage_V. One name, one definition — the
    same quantity was previously exported as `Avg_Voltage_V` by one function
    and `Avg_V_Discharge` by another. Named `mean_...` because
    `average_discharge_voltage` is the FIGURE function further down; two
    definitions of one name in a flat namespace is how the last collision
    got shipped.
    """
    need = ("Cycle", "Step", "Voltage", "Discharge_Capacity")
    if not all(c in frame.columns for c in need):
        return pd.DataFrame(columns=["Cycle", "Avg_Discharge_Voltage_V"])
    rows = []
    d = frame[frame["Step"] == "Discharge"]
    for cyc, g in d.groupby("Cycle"):
        q = pd.to_numeric(g["Discharge_Capacity"], errors="coerce")
        v = pd.to_numeric(g["Voltage"], errors="coerce")
        m = q.notna() & v.notna()
        q, v = q[m].to_numpy(), v[m].to_numpy()
        if q.size < 3:
            continue
        o = np.argsort(q, kind="mergesort")
        q, v = q[o], v[o]
        span = float(q[-1] - q[0])
        if not np.isfinite(span) or span <= 0:
            continue
        rows.append({"Cycle": int(cyc),
                     "Avg_Discharge_Voltage_V": float(trapezoid(v, q) / span)})
    return pd.DataFrame(rows)


def _honours_verbose(fn):
    """Make `verbose=False` actually silence a cycling function.

    All eleven took the argument and none read it — 227 unconditional
    `print()` calls between them — so `cycling_summary(verbose=False)` printed
    its banner anyway and a scheduled run had no way to quieten the log.
    Rather than thread a flag through 227 call sites, the output is captured
    at the boundary, which is exactly the promise the signature was making.
    """
    @functools.wraps(fn)
    def wrapper(*args, **kw):
        if kw.get("verbose", True):
            return fn(*args, **kw)
        buf = _io.StringIO()
        with contextlib.redirect_stdout(buf):
            return fn(*args, **kw)
    return wrapper


def rate_block_labels(cycles, params):
    """
    Per-cycle rate-block index for a variable-rate run, or None.

    `None` for a single-rate run — there is one population and the caller
    should not partition it. On a variable-rate run each cycle gets the index
    of the block it belongs to, and a rate-TRANSITION cycle (in no block) gets
    its own label, so it is never pooled with either neighbour.
    """
    proto = ((params or {}).get("rate_protocol") or {})
    if not proto.get("is_variable"):
        return None
    blocks = proto.get("blocks") or []
    if not blocks:
        return None
    out = []
    for c in cycles:
        try:
            ci = int(c)
        except (TypeError, ValueError):
            out.append(("bad", c))
            continue
        lab = None
        for i, b in enumerate(blocks):
            if int(b["first_cycle"]) <= ci <= int(b["last_cycle"]):
                lab = i
                break
        out.append(lab if lab is not None else ("transition", ci))
    return out


def _flag_visual_outliers(caps, threshold=VISUAL_OUTLIER_FRACTION,
                          groups=None):
    """
    Flag cycles whose delivered capacity is <threshold of the running
    median of previous non-outlier, non-NaN cycles AT THE SAME RATE.

    Purpose: catch protocol anomalies (cell tripped, early cutoff,
    datalogging glitch) that delivered a small but non-zero discharge,
    so Cell 5b's protocol-level Incomplete flag doesn't catch them.

    Why 30% rather than the old 80%: Bug 1 (v1.7.1) showed that a
    running-median-based filter at 80% falsely flags healthy
    degradation as incomplete. 30% only triggers on cliff-drops — a
    cycle delivering less than a third of recent typical capacity is
    almost certainly an anomaly, not gradual fade.

    WHY `groups`. The running median was taken over every previous valid cycle
    in the record, and on a rate ladder that is a comparison across rates. A
    protocol of C/10, C/5, C/2, 1C, 2C, then 10C, then back to C/10 — the
    standard rate-capability test — reaches its 10C block with a running median
    near the low-rate capacity: at 200 mAh/g of accumulated median, the cut is
    60 mAh/g, and a perfectly healthy 10C discharge of 40 mAh/g is deleted as a
    datalogging glitch. Worse, a flagged cycle is not appended to `prev_valid`
    (the `continue` below), so the median never comes down and EVERY cycle of
    the high-rate block goes with it. Retention, fade rate, cycle life, mean
    discharge voltage, energy efficiency and the areal capacity figure are all
    drawn after this filter, so the whole fast half of a rate test could vanish
    from every one of them, announced only as "excluded N visual outlier(s)".

    A cycle is an outlier relative to cycles at ITS OWN rate. `groups` is one
    label per entry — `rate_block_labels` builds it — and the running median is
    kept per label. `groups=None` is the single-rate case and behaves exactly
    as before. A rate-transition cycle gets a label of its own and so never
    reaches the three-cycle minimum, which is the right answer: a cycle
    straddling two rates has no population to be an outlier in.

    Minimum of 3 prior valid cycles required before the filter kicks
    in (avoids flagging early formation cycles).

    Used by Cells 8, 9, 9b, 10, 11, 12 to clean plots.
    Defined once here to avoid divergent copies.
    """
    flags = [False] * len(caps)
    if groups is None:
        prev_valid = []
        for i, cap in enumerate(caps):
            if pd.isna(cap):
                flags[i] = True
                continue
            if len(prev_valid) >= 3:
                if cap < threshold * np.median(prev_valid):
                    flags[i] = True
                    continue
            prev_valid.append(cap)
        return flags

    # GROUPED: LEAVE-ONE-OUT WITHIN THE BLOCK, NOT A RUNNING MEDIAN.
    # A rate block in the standard test is five cycles long, so a running
    # median over PRECEDING cycles only reaches the three-cycle minimum at the
    # fourth — the filter would be blind to a glitch in the first three cycles
    # of every block, which is most of the record. Within one block the rate is
    # constant and the fade across five cycles is a fraction of a percent, so
    # the block's own median is a better reference than a running one and it is
    # available to every member. Each cycle is compared with the median of the
    # OTHER valid cycles in its block, so a glitch cannot raise the bar that
    # judges it. Blocks with fewer than four valid cycles are left alone: there
    # is no population there to be an outlier in, and a rate-transition cycle
    # (its own label) is always in that case.
    idx_by_group = {}
    for i, cap in enumerate(caps):
        if pd.isna(cap):
            flags[i] = True
            continue
        g = groups[i] if i < len(groups) else None
        idx_by_group.setdefault(g, []).append(i)
    for g, idxs in idx_by_group.items():
        if len(idxs) < 4:
            continue
        vals = np.array([float(caps[i]) for i in idxs])
        for j, i in enumerate(idxs):
            ref = np.median(np.delete(vals, j))
            if np.isfinite(ref) and vals[j] < threshold * ref:
                flags[i] = True
    return flags


def _unusable(ct):
    """Cycles no figure should treat as a finished measurement.

    `Incomplete` is protocol-level and deliberately narrow — a cycle whose
    discharge is missing or zero. It cannot see the half-cycle that is
    HALFWAY THROUGH, because half a discharge is a real number, so the
    cycle an export was written during sailed past every filter in this
    module and plotted as a sudden drop at the end of ten curves. Both are
    reasons not to plot a point; they are kept as separate columns so the
    CSV still says which is which.
    """
    if ct is None or not len(ct):
        return pd.Series(dtype=bool)
    bad = pd.Series(False, index=ct.index)
    for col in ('Incomplete', 'Partial_Final'):
        if col in ct.columns:
            bad = bad | ct[col].fillna(False).astype(bool)
    return bad


def _unusable_cycles(ct):
    """The same verdict as a set of cycle numbers, for the figures that
    rebuild their y-values from the raw records and can only be told which
    cycles to drop."""
    bad = _unusable(ct)
    if ct is None or not len(bad) or 'Cycle' not in ct.columns:
        return set()
    return set(pd.to_numeric(ct.loc[bad, 'Cycle'],
                             errors='coerce').dropna().astype(int).tolist())


def _ct_with_flag(ct, value_col):
    """`ct[['Cycle', value_col, 'Incomplete']]` with the partial-final cycle
    folded into the flag, so a caller's existing `~df['Incomplete']` mask
    excludes it too."""
    out = ct[['Cycle', value_col]].copy()
    out['Incomplete'] = _unusable(ct).values
    return out


@_honours_verbose
def cycling_summary(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True,
        in_progress=None):
    """
    Per-cycle capacity, coulombic efficiency and the incomplete-cycle flags every later plot reads.

    Ported from 1.8.7 Cell 5b, wrapped in a function without editing
    the body. `all_cycle_tables` is accepted for signature symmetry with the
    other cycling functions and IGNORED — this function builds the tables and
    returns them; it does not read them.
    """
    print(rule("CYCLING PERFORMANCE"))

    def _detect_incomplete_cycles(discharge_capacities, charge_capacities=None):
        """
        Flag a cycle as incomplete if and only if the cycler did not
        complete its prescribed protocol for that cycle.
 
        Operational definition (v1.7.1):
          - discharge missing or zero AND charge missing or zero
                -> cycler aborted before either half-cycle -> incomplete
          - discharge missing or zero AND charge present
                -> charge completed, discharge did not -> incomplete
          - otherwise -> complete (even if degraded, low-CE, or parasitic)
 
        Health-based filtering (low retention, abnormal CE, parasitic
        current holds) lives in the Triplicate Analysis health pre-filter,
        NOT here. Cell 5b only reports protocol-level status.
 
        Parameters
        ----------
        discharge_capacities : list-like of float
            Discharge capacity (mAh/g) per cycle, in cycle order.
            After Cell 4b normalisation, this is always the "useful"
            half-cycle (delithiation for anodes, discharge for cathodes).
        charge_capacities : list-like of float, optional
            Charge capacity per cycle. If omitted, the function falls
            back to flagging only missing/zero discharges (safe default
            for Cell 5b, which always has charge data in current use).
 
        Returns
        -------
        list of bool
            True = incomplete; False = complete.
        """
        n = len(discharge_capacities)
        if charge_capacities is None:
            charge_capacities = [np.nan] * n
        else:
            charge_capacities = list(charge_capacities)
 
        flags = []
        for d, c in zip(discharge_capacities, charge_capacities):
            d_empty = pd.isna(d) or (pd.notna(d) and d <= 0.0)
            c_empty = pd.isna(c) or (pd.notna(c) and c <= 0.0)
            if d_empty:
                # Either cycler aborted entirely, or charge ran but
                # discharge was not attempted/completed.
                flags.append(True)
            else:
                # Discharge has data — the cycle completed its discharge
                # step. Charge anomalies (parasitic holds, capacity
                # spikes) are not incompleteness.
                flags.append(False)
        return flags

    # Built here and RETURNED; the parameter of the same name exists only
    # because 1.8.7's Cell 5b read a notebook global. It was documented as
    # "passing None reproduces the not-yet-run branch", which it could not
    # do, being overwritten on this line.
    all_cycle_tables = {}

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        charge_rate_c = params.get('charge_rate_c', 'Unknown')
        is_anode = params.get('anode_labels_swapped', False)
        dch_label = _discharge_label(params)
        chg_label = _charge_label(params)
    
        print()
        print(heading(composition))
        if is_anode:
            print(entry("half-cell", "anode",
                        f"{dch_label.lower()} is the useful capacity"))
    
        if not all(col in df.columns for col in ['Cycle', 'Step',
                    'Discharge_Capacity', 'Charge_Capacity']):
            print(verdict("caution", "missing required columns, skipping."))
            continue
    
        # The one definition — see `delivered_capacity`.
        cycle_df = delivered_capacity(df)
    
        # CE = discharge / charge (correct for both electrode types
        # after Cell 4b normalisation)
        cycle_df['CE_%'] = coulombic_efficiency(
            cycle_df['Discharge_mAh_g'], cycle_df['Charge_mAh_g']).values
    
        # --- Detect incomplete cycles (v1.7.1: protocol-level only) ---
        incomplete_flags = _detect_incomplete_cycles(
            cycle_df['Discharge_mAh_g'].tolist(),
            cycle_df['Charge_mAh_g'].tolist()
        )
        cycle_df['Incomplete'] = incomplete_flags

        # --- Half-cycle still running when the export was written (1.9.0) ---
        # A cycler export is routinely taken while the cell is STILL
        # CYCLING, so the file's final half-cycle is usually unfinished.
        # That is a snapshot of a running experiment, not a measurement.
        #
        # `_detect_incomplete_cycles` is protocol-level and deliberately so:
        # it catches a cycle whose discharge is missing or zero. It cannot
        # catch a half-cycle that is HALFWAY THROUGH, because half a
        # discharge is a real number. On the LTO triplicate, cell B's export
        # was written during cycle 11's delithiation: 64.7 mAh/g of the
        # 123.3 it would have delivered. Every headline read that as data —
        # 52% retention, a fade rate of -3.4 mAh/g/cycle, and a CRITICAL
        # health flag predicting failure within ten cycles — for a cell
        # holding 99% of its capacity and still running.
        #
        # WHICH half-cycle is decided by `analyse.half_cycles_in_progress`
        # and handed in as `in_progress`, because the question is
        # about the raw record and not about this table: a galvanostatic
        # half-cycle terminates ON VOLTAGE, so the test is whether it
        # reached the cut-off. Judging it by capacity instead would have
        # condemned 15 perfectly good half-cycles of P3 cell B, which dies
        # at cycle 67 and is then driven to the cut-off, and back, for 153
        # more cycles. A cycle is excluded from the headline metrics here
        # when the half-cycle those metrics are computed FROM is the one
        # still running; the completed half of the same cycle is kept.
        cycle_df['Partial_Final'] = False
        _in_prog = set((in_progress or {}).get(name, ()) or ())
        _pf_note = None
        for _pf_cyc, _pf_step in sorted(_in_prog):
            _row = cycle_df['Cycle'] == _pf_cyc
            if not _row.any():
                continue
            # The headline metrics are built on the discharge half. If the
            # CHARGE half is the unfinished one, the discharge of that cycle
            # has not started, and `Incomplete` already covers it.
            _dch = float(pd.to_numeric(
                cycle_df.loc[_row, 'Discharge_mAh_g'], errors='coerce').iloc[0]
            ) if _row.any() else np.nan
            cycle_df.loc[_row, 'Partial_Final'] = True
            _pf_note = (_pf_cyc, _pf_step, _dch)
            print(verdict("caution", f"cycle {_pf_cyc} {_pf_step} was "
                                     f"still running when this file was "
                                     f"exported"))
            print(bullet("It never reached the cut-off voltage every other "
                         "half-cycle going the same way reaches. Not a "
                         "measurement yet: excluded from the headline metrics "
                         "below, still plotted, and complete in the next "
                         "export."))

        n_incomplete = sum(incomplete_flags)
        incomplete_cycles = cycle_df[cycle_df['Incomplete']]['Cycle'].tolist()
        # THE TWO REASONS, kept apart. `Incomplete` is protocol-level;
        # `Partial_Final` is the export catching a half-cycle mid-flight.
        # `complete_df` below has always excluded both, but every line that
        # TOLD the reader about it counted only the first, so cell B's run
        # excluded cycle 11 silently: full table, unmarked row, a headline
        # that claimed nothing had been dropped, and a caption that said
        # nothing at all.
        _partial_cycles = cycle_df.loc[
            cycle_df['Partial_Final'].fillna(False).astype(bool),
            'Cycle'].tolist()
        _excluded_mask = _unusable(cycle_df)
        n_excluded = int(_excluded_mask.sum())
        excluded_cycles = cycle_df.loc[_excluded_mask, 'Cycle'].tolist()
        # No print here. The verdict above already named the cycle and said
        # it was excluded from the headline metrics; this said it again in
        # weaker words, and the table below marks the row INCOMPL anyway.

        # --- Capacity retention vs reference cycle ---
        ref_row = cycle_df[
            (cycle_df['Cycle'] == RETENTION_REFERENCE_CYCLE) &
            (~cycle_df['Incomplete'])
        ]
        if not ref_row.empty:
            ref_cap = ref_row['Discharge_mAh_g'].iloc[0]
        else:
            complete = cycle_df[~cycle_df['Incomplete']]
            if not complete.empty:
                ref_cap = complete['Discharge_mAh_g'].iloc[0]
                print(f"  Note: reference cycle {RETENTION_REFERENCE_CYCLE} "
                      f"not found, using cycle "
                      f"{int(complete['Cycle'].iloc[0])}")
            else:
                ref_cap = np.nan
    
        if pd.notna(ref_cap) and ref_cap > 0:
            cycle_df['Retention_%'] = (
                cycle_df['Discharge_mAh_g'] / ref_cap * 100
            )
        else:
            cycle_df['Retention_%'] = np.nan

        # --- v1.8: Cell health diagnostics ---
        # Detect progressive parasitic current, voltage anomalies, and
        # definite failure signatures. Flags are stored in cycle_df so
        # every downstream cell can access them.
    
        # 1. Charge excess per cycle (Q_chg - Q_dchg)
        cycle_df['Charge_Excess_mAh_g'] = (
            cycle_df['Charge_mAh_g'] - cycle_df['Discharge_mAh_g'])
    
        # 2. Progressive parasitic current detection
        # Start from cycle 4 to avoid formation-period false positives.
        # Cycles 1-3 typically show CE evolution from SEI/CEI formation
        # that looks like a declining trend but is normal stabilisation.
        _DIAG_START_CYCLE = 4
        _diag_df = cycle_df[
            (~cycle_df['Incomplete']) & (~cycle_df['Partial_Final'])
            & (cycle_df['Cycle'] >= _DIAG_START_CYCLE)
        ].copy()
    
        _cell_flags = []      # list of (severity, message) tuples
    
        if len(_diag_df) >= 4:
            # CE trend: linear fit to CE vs cycle
            _ce_vals = _diag_df['CE_%'].dropna()
            if len(_ce_vals) >= 4:
                _ce_slope, _ce_intercept = np.polyfit(
                    _diag_df.loc[_ce_vals.index, 'Cycle'], _ce_vals, 1)
            
                # Check for monotonic decline over last N cycles
                _last_n = min(5, len(_ce_vals))
                _recent_ce = _ce_vals.tail(_last_n)
                _n_declining = sum(
                    _recent_ce.iloc[i] < _recent_ce.iloc[i-1]
                    for i in range(1, len(_recent_ce)))
                _monotonic = _n_declining >= _last_n - 1  # allow 1 exception
            
                if _ce_slope < -0.5 and _monotonic:
                    _cell_flags.append(('CRITICAL',
                        f'CE declining monotonically at '
                        f'{_ce_slope:.2f}%/cycle — progressive '
                        f'parasitic current'))
                elif _ce_slope < -0.3:
                    _cell_flags.append(('WARNING',
                        f'CE trend negative ({_ce_slope:.2f}%/cycle) — '
                        f'possible parasitic current developing'))
        
            # Charge excess trend: is delta_Q growing?
            _excess = _diag_df['Charge_Excess_mAh_g'].dropna()
            if len(_excess) >= 4:
                _ex_slope, _ = np.polyfit(
                    _diag_df.loc[_excess.index, 'Cycle'], _excess, 1)
                if _ex_slope > 0.3:
                    _cell_flags.append(('CRITICAL',
                        f'Charge excess growing at '
                        f'{_ex_slope:.2f} mAh/g/cycle — charge consumed '
                        f'by side reaction'))
                elif _ex_slope > 0.1:
                    _cell_flags.append(('WARNING',
                        f'Charge excess trend positive '
                        f'({_ex_slope:.2f} mAh/g/cycle) — monitor for '
                        f'parasitic activity'))
    
        # 3. Voltage anomaly on final data points
        # 4. Final cycle with zero or near-zero discharge
        # Skip both if the last cycle is already flagged incomplete —
        # that's a mid-experiment data pull, not a cell failure.
        #
        # `Incomplete` ALONE IS NOT THAT TEST. `Partial_Final` is the flag
        # that marks the mid-experiment data pull, set forty lines above; a
        # half-cycle caught mid-discharge has a small but non-zero delivered
        # capacity, so `Incomplete` is False and check 4 fired. A healthy
        # cell at 99% retention, exported thirty seconds into cycle 11's
        # discharge, was told "charged 123.4 mAh/g but delivered 0.4 mAh/g
        # on discharge — cell died. This cell has failed. Data after the
        # failure point is not meaningful." Both flags gate both checks.
        _last_cycle = int(cycle_df['Cycle'].max())
        _last_row = cycle_df[cycle_df['Cycle'] == _last_cycle]
        _last_incomplete = bool(_last_row['Incomplete'].iloc[0]) or bool(
            _last_row['Partial_Final'].iloc[0]
            if 'Partial_Final' in _last_row else False)
    
        if not _last_incomplete:
            # 3. Check if the last recorded step has voltage going the wrong way
            _last_cycle_data = df[df['Cycle'] == _last_cycle]
            if not _last_cycle_data.empty and 'Step' in _last_cycle_data.columns:
                _last_dchg = _last_cycle_data[
                    _last_cycle_data['Step'] == 'Discharge']
                if not _last_dchg.empty and len(_last_dchg) <= 3:
                    # Discharge step with <=3 data points = aborted
                    _v_start = pd.to_numeric(
                        _last_dchg['Voltage'].iloc[0], errors='coerce')
                    _v_prev_step = None
                    _last_chg = _last_cycle_data[
                        _last_cycle_data['Step'] == 'Charge']
                    if not _last_chg.empty:
                        _v_prev_step = pd.to_numeric(
                            _last_chg['Voltage'].iloc[-1], errors='coerce')
                
                    if (_v_prev_step is not None and pd.notna(_v_start)
                            and pd.notna(_v_prev_step)
                            and _v_start > _v_prev_step + 0.1):
                        _cell_flags.append(('FAILURE',
                            f'Cycle {_last_cycle}: voltage spiked '
                            f'{_v_start:.3f} V on discharge start '
                            f'(> {_v_prev_step:.3f} V end-of-charge) — '
                            f'open circuit / contact failure'))
        
            # 4. Final cycle with zero or near-zero discharge
            _last_row = cycle_df[cycle_df['Cycle'] == _last_cycle]
            if not _last_row.empty:
                _last_dchg_cap = _last_row['Discharge_mAh_g'].iloc[0]
                _last_chg_cap = _last_row['Charge_mAh_g'].iloc[0]
                if (pd.notna(_last_chg_cap) and _last_chg_cap > 0
                        and (pd.isna(_last_dchg_cap) or _last_dchg_cap < 1.0)):
                    _cell_flags.append(('FAILURE',
                        f'Cycle {_last_cycle}: charged '
                        f'{_last_chg_cap:.1f} mAh/g but delivered '
                        f'{"0" if pd.isna(_last_dchg_cap) else f"{_last_dchg_cap:.1f}"} '
                        f'mAh/g on discharge — cell died'))
    
        # Store flags in cycle_df metadata (accessible downstream)
        cycle_df.attrs['cell_health_flags'] = _cell_flags
    
        all_cycle_tables[name] = cycle_df
    
        # --- Print table (v1.8: anode-aware headers) ---
        # Status is 10 wide because the words are now words. The rule under
        # the header is measured from the header rather than hard-coded at
        # 56, which had not matched the columns since the CE field changed.
        _hdr = (f"  {'Cycle':>5}  {chg_label:>10}  {dch_label:>10}  "
                f"{'CE':>7}  {'Retention':>9}  {'Status':>10}")
        print("\n" + _hdr)
        print(f"  {'':>5}  {'mAh/g':>10}  {'mAh/g':>10}  "
              f"{'%':>7}  {'%':>9}  {'':>10}")
        print(f"  {'-' * (len(_hdr) - 2)}")
    
        for _, row in cycle_df.iterrows():
            def _f(val, fmt='.1f'):
                return f'{val:{fmt}}' if pd.notna(val) else '--'
        
            # Until 1.9.0.23 this read `Incomplete` alone. Cells A and C,
            # whose CHARGE half was the unfinished one, printed INCOMPL;
            # cell B, where the export was written during cycle 11's
            # discharge, printed a BLANK — and its row showed 64.7 mAh/g,
            # CE 52.4%, retention 51.8% with nothing to say those were half
            # of a measurement still being taken. The one row in the run
            # that most needed a marker was the only one without one.
            if row['Incomplete']:
                status = 'Incomplete'
            elif bool(row.get('Partial_Final', False)):
                status = 'Running'
            else:
                status = ''
        
            print(f"  {int(row['Cycle']):>5}  "
                  f"{_f(row['Charge_mAh_g']):>10}  "
                  f"{_f(row['Discharge_mAh_g']):>10}  "
                  f"{_f(row['CE_%']):>7}  "
                  f"{_f(row['Retention_%']):>9}  "
                  f"{status:>8}")
    
        # --- Headline metrics (complete cycles only) ---
        # Headline metrics only. The full `cycle_df` — partial cycle and
        # all — is what goes into every figure and the exported CSV.
        complete_df = cycle_df[~cycle_df['Incomplete']
                               & ~cycle_df['Partial_Final']]
    
        print("\n" + section(
            f"  Headline metrics (excluding cycle"
            f"{'s' if n_excluded != 1 else ''} "
            f"{', '.join(str(int(c)) for c in excluded_cycles)})"
            if n_excluded > 0 else "  Headline metrics"))
    
        # First-cycle irreversible loss (v1.8: anode-aware text)
        # After Cell 4b swap:
        #   Cathode: Charge = delithiation/desodiation, Discharge = lithiation/sodiation
        #            irrev = charge - discharge (positive = normal)
        #   Anode:   Charge = lithiation (after swap), Discharge = delithiation
        #            irrev = charge - discharge (positive = normal)
        # In both cases, charge > discharge on cycle 1 means irreversible
        # capacity consumed during first insertion.
        c1 = cycle_df[cycle_df['Cycle'] == 1]
        if not c1.empty and not c1['Incomplete'].iloc[0]:
            q_chg_1 = c1['Charge_mAh_g'].iloc[0]
            q_dchg_1 = c1['Discharge_mAh_g'].iloc[0]
            if pd.notna(q_chg_1) and pd.notna(q_dchg_1) and q_chg_1 > 0:
                irrev_loss = q_chg_1 - q_dchg_1
                irrev_pct = irrev_loss / q_chg_1 * 100
                if irrev_loss >= 0:
                    if is_anode:
                        print(f"    1st cycle irreversible loss: "
                              f"{irrev_loss:.1f} mAh/g ({irrev_pct:.1f}%)")
                        print(f"    ({chg_label}: {q_chg_1:.1f} mAh/g, "
                              f"{dch_label}: {q_dchg_1:.1f} mAh/g)")
                    else:
                        print(f"    1st cycle irreversible loss: "
                              f"{irrev_loss:.1f} mAh/g ({irrev_pct:.1f}%)")
                else:
                    print(f"    1st cycle excess {dch_label.lower()}: "
                          f"{-irrev_loss:.1f} mAh/g ({-irrev_pct:.1f}% "
                          f"more {dch_label.lower()} than "
                          f"{chg_label.lower()})")
                print(f"    1st cycle CE: "
                      f"{q_dchg_1/q_chg_1*100:.1f}%")
    
        # Retention: last complete cycle vs reference
        if pd.notna(ref_cap) and not complete_df.empty:
            last_complete = complete_df.iloc[-1]
            last_cap = last_complete['Discharge_mAh_g']
            last_cycle = int(last_complete['Cycle'])
        
            if pd.notna(last_cap):
                # RETENTION AGAINST BOTH CYCLE 1 AND THE REFERENCE.
                # Cycle 1 includes the first-cycle irreversible loss and is
                # what a reader comparing against a datasheet expects; the
                # reference (2 by default) excludes it and is what a cycling
                # study quotes. They answer different questions and a paper
                # usually needs both, so reporting only one made the other a
                # calculation the reader had to do from the table.
                for _ref in sorted({1, RETENTION_REFERENCE_CYCLE}):
                    _row = complete_df[complete_df['Cycle'] == _ref]
                    if _row.empty:
                        continue
                    _cap = _row.iloc[0]['Discharge_mAh_g']
                    if pd.isna(_cap) or not _cap:
                        continue
                    print(f"    Cycle {_ref} {dch_label.lower()}: "
                          f"{_cap:.1f} mAh/g")
                print(f"    Cycle {last_cycle} "
                      f"{dch_label.lower()}: {last_cap:.1f} mAh/g")
                for _ref in sorted({1, RETENTION_REFERENCE_CYCLE}):
                    _row = complete_df[complete_df['Cycle'] == _ref]
                    if _row.empty:
                        continue
                    _cap = _row.iloc[0]['Discharge_mAh_g']
                    if pd.isna(_cap) or not _cap:
                        continue
                    _note = ("  [includes the first-cycle irreversible loss]"
                             if _ref == 1 else "")
                    print(f"    Retention (cycle {last_cycle} vs {_ref}): "
                          f"{100 * last_cap / _cap:.1f}%{_note}")
    
        # WHERE DOES FORMATION END? Measured from this table's own CE, not
        # assumed. See FADE_RATE_FROM_MEASURED_FORMATION.
        _fade_start = int(FADE_RATE_START_CYCLE)
        _fade_why = (f"the {FADE_RATE_START_CYCLE} in `FADE_RATE_START_CYCLE`, "
                     f"not a measurement")
        if FADE_RATE_FROM_MEASURED_FORMATION and not complete_df.empty:
            _ce_series = {int(r['Cycle']): float(r['CE_%'])
                          for _, r in complete_df.iterrows()
                          if pd.notna(r.get('CE_%'))}
            _fe, _fe_why = formation_end(_ce_series)
            if _fe is not None:
                _fade_start, _fade_why = int(_fe), _fe_why
            else:
                _fade_why = (f"{_fe_why}; falling back to cycle "
                             f"{FADE_RATE_START_CYCLE}")

        # Average CE (complete, excluding formation)
        stable_ce = complete_df[
            complete_df['Cycle'] >= _fade_start
        ]['CE_%'].dropna()
    
        if not stable_ce.empty:
            last_complete_cycle = int(complete_df['Cycle'].max())
            print(f"    Average CE (cycles {_fade_start}"
                  f"--{last_complete_cycle}): "
                  f"{stable_ce.mean():.2f}% "
                  f"(+/-{stable_ce.std():.2f}%)")
    
        # Fade rate (complete, excluding formation)
        stable = complete_df[
            (complete_df['Cycle'] >= _fade_start) &
            (complete_df['Discharge_mAh_g'].notna())
        ]
    
        if len(stable) >= 3:
            slope, _ = np.polyfit(stable['Cycle'],
                                  stable['Discharge_mAh_g'], 1)
            # THE DENOMINATOR MATCHES THE RANGE. `%/cycle` divided by the
            # capacity of `RETENTION_REFERENCE_CYCLE` while the slope was
            # fitted from somewhere else entirely, so the two halves of the
            # sentence described different cycles. It is now the capacity of
            # the first cycle the slope was fitted through.
            _row0 = stable[stable['Cycle'] == stable['Cycle'].min()]
            _base = (float(_row0.iloc[0]['Discharge_mAh_g'])
                     if not _row0.empty else np.nan)
            print(f"    Formation ends at cycle {_fade_start} "
                  f"({_fade_why})")
            if pd.notna(_base) and _base > 0:
                print(f"    Fade rate (cycles {_fade_start}"
                      f"--{int(stable['Cycle'].max())}): "
                      f"{slope:.3f} mAh/g/cycle "
                      f"({slope/_base*100:.2f}%/cycle of the cycle "
                      f"{_fade_start} capacity)")
            else:
                print(f"    Fade rate (cycles {_fade_start}"
                      f"--{int(stable['Cycle'].max())}): "
                      f"{slope:.3f} mAh/g/cycle")
    
        # --- Suggested caption (v1.8: anode-aware) ---
        caption = (
            f"Table X. Specific {chg_label.lower()} and "
            f"{dch_label.lower()} capacities, coulombic efficiency, "
            f"and capacity retention (relative to cycle "
            f"{RETENTION_REFERENCE_CYCLE}) for {composition} cycled "
            f"at {rate_phrase(params)}."
        )
        # Both reasons, named. A caption is the one line that leaves the
        # run and goes into a paper, so "excluded from headline metrics"
        # has to say WHICH cycles and WHY, and a cycle caught mid-flight is
        # not the same claim as a cycle that never discharged.
        _cap_notes = []
        if n_incomplete > 0:
            _cap_notes.append(
                f"cycle{'s' if n_incomplete > 1 else ''} "
                f"{', '.join(str(int(c)) for c in incomplete_cycles)} "
                f"{'were' if n_incomplete > 1 else 'was'} incomplete")
        if _partial_cycles:
            _n_pf = len(_partial_cycles)
            _cap_notes.append(
                f"cycle{'s' if _n_pf > 1 else ''} "
                f"{', '.join(str(int(c)) for c in _partial_cycles)} "
                f"{'were' if _n_pf > 1 else 'was'} still in progress when "
                f"the data were exported")
        if _cap_notes:
            _joined = " and ".join(_cap_notes)
            caption += (" " + _joined[0].upper() + _joined[1:]
                        + ", and excluded from headline metrics.")
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))
    
        # --- v1.8: Print cell health diagnostics ---
        if _cell_flags:
            print(f"\n  {'!'*60}")
            print(f"  CELL HEALTH DIAGNOSTICS")
            print(f"  {'!'*60}")
            for severity, msg in _cell_flags:
                if severity == 'FAILURE':
                    print(f"  ✘ DEFINITE FAILURE: {msg}")
                elif severity == 'CRITICAL':
                    print(f"  ⚠ CRITICAL: {msg}")
                else:
                    print(f"  ● WARNING: {msg}")
            if any(s == 'FAILURE' for s, _ in _cell_flags):
                print(f"\n  This cell has failed. Data after the failure "
                      f"point is not meaningful.")
                print(f"  Check the voltage profile to confirm the "
                      f"failure mode.")
            elif any(s == 'CRITICAL' for s, _ in _cell_flags):
                print(f"\n  This cell shows signs of progressive "
                      f"degradation beyond normal fading.")
                print(f"  If this pattern continues, expect cell "
                      f"failure within the next 5-10 cycles.")
    
            if not any(s == 'FAILURE' for s, _ in _cell_flags):
                print(f"\n  If cycling is still in progress, the final "
                      f"partial cycle may skew")
                print(f"  these trends. Re-run on the completed dataset "
                      f"to confirm.")
            print(f"  {'!'*60}")

    if all_cycle_tables:
        combined = None
        for name, cdf in all_cycle_tables.items():
            params = user_parameters.get(name, {})
            comp = _get_display_name(name, params, user_parameters)
            dch = _discharge_label(params)
            chg = _charge_label(params)
        
            export_df = cdf[['Cycle']].copy()
            export_df[f'{comp}_{chg}_mAh_g'] = cdf['Charge_mAh_g']
            export_df[f'{comp}_{dch}_mAh_g'] = cdf['Discharge_mAh_g']
            export_df[f'{comp}_CE_%'] = cdf['CE_%']
            # Self-describing: a retention figure without its reference is
            # not a measurement. RETENTION_REFERENCE_CYCLE is in the name.
            export_df[f'{comp}_Retention_vs_C{RETENTION_REFERENCE_CYCLE}_%'] = \
                cdf['Retention_%']
            export_df[f'{comp}_Incomplete'] = cdf['Incomplete']
            if 'Partial_Final' in cdf.columns:
                # Two different reasons a cycle is not a measurement, kept
                # apart so a reader of the CSV can tell "the cycler aborted"
                # from "the export was taken during it".
                export_df[f'{comp}_Still_Running'] = cdf['Partial_Final']
        
            if combined is None:
                combined = export_df
            else:
                combined = pd.merge(combined, export_df, on='Cycle',
                                   how='outer')
    
        if combined is not None:
            combined = combined.sort_values('Cycle').reset_index(drop=True)
        
            if save_location:
                fpath = os.path.join(save_location,
                                    'cycling_performance_summary.csv')
                combined.to_csv(fpath, index=False)
                saved(fpath)

    print(rule())

    # 1.8.7 Cell 5b built `all_cycle_tables` as a NOTEBOOK GLOBAL and every
    # later cell read it from there. Wrapped in a function the name became
    # local, and returning it was the one line the port had to add: without
    # it seven downstream figures lost their incomplete-cycle filtering and
    # the START_HERE report had no capacity numbers at all.
    return all_cycle_tables


@_honours_verbose
def voltage_profiles(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Voltage vs specific capacity, every cycle.

    Ported from 1.8.7 Cell 5, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """

    def _check_overlap(endpoints, cap_tol_frac=0.03, volt_tol=0.05):
        """
        Check what fraction of endpoints would overlap.
    
        Returns fraction (0–1) of endpoints that are within tolerance 
        of at least one other endpoint.
        """
        if len(endpoints) < 2:
            return 0.0
    
        caps = [e['capacity'] for e in endpoints]
        cap_range = max(caps) - min(caps) if len(caps) > 1 else max(caps)
        cap_tol = max(cap_range * cap_tol_frac, 2.0)
    
        n_overlapping = 0
        for i, ep in enumerate(endpoints):
            for j, other in enumerate(endpoints):
                if i == j:
                    continue
                if (abs(ep['capacity'] - other['capacity']) < cap_tol and
                    abs(ep['voltage'] - other['voltage']) < volt_tol):
                    n_overlapping += 1
                    break  # only count each endpoint once
    
        return n_overlapping / len(endpoints)

    def _add_annotations(ax, endpoints, unique_cycles, palette):
        """Add cycle number annotations at endpoints (no vertical offset)."""
        if not endpoints:
            return
    
        # Group endpoints by proximity to determine horizontal stagger
        caps = [e['capacity'] for e in endpoints]
        cap_range = max(caps) - min(caps) if len(caps) > 1 else max(caps)
        cap_tol = max(cap_range * 0.03, 2.0)
        volt_tol = 0.05
    
        placed = []
    
        # Process in reverse cycle order (later cycles get priority placement)
        sorted_endpoints = sorted(endpoints, key=lambda e: e['cycle'],
                                   reverse=True)
    
        for ep in sorted_endpoints:
            cycle_idx = list(unique_cycles).index(ep['cycle'])
            colour = palette[cycle_idx]
        
            # Determine offset
            x_offset = 10
            y_offset = 0
        
            # Stagger if this endpoint is close to an already-placed one
            for px, py in placed:
                if (abs(ep['capacity'] - px) < cap_tol and
                    abs(ep['voltage'] - py) < volt_tol):
                    # Shift further right
                    x_offset += 12
                    break
        
            # Discharge endpoints: label to the right
            # Charge endpoints: label to the left
            if ep['step'] == 'Charge':
                x_offset = -x_offset
        
            # Adjust vertical alignment based on position
            va = 'center'
        
            ax.annotate(
                str(int(ep['cycle'])),
                xy=(ep['capacity'], ep['voltage']),
                xytext=(x_offset, y_offset),
                textcoords='offset points',
                ha='center', va=va,
                fontsize=8, fontweight='bold',
                color=colour, alpha=0.9
            )
            placed.append((ep['capacity'], ep['voltage']))

    def _add_colourbar(fig, ax, unique_cycles, colour_palette):
        """Add a colourbar mapping cycle number to colour."""
        cmap = plt.get_cmap(colour_palette)
        norm = mcolors.Normalize(vmin=min(unique_cycles),
                                  vmax=max(unique_cycles))
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
    
        cbar = fig.colorbar(sm, ax=ax, pad=0.02, aspect=30)
        cbar.set_label('Cycle number', fontsize=13)
        cbar.ax.tick_params(labelsize=11)

    # The two nested copies of `_discharge_label`/`_charge_label` that used
    # to sit here are gone. This module imports both from `.plots` at the
    # top and these SHADOWED that import for the whole function, so a fix
    # applied to the real ones reached every figure except the waterfalls —
    # which is exactly where 'Lithiation' over a sodium cell was seen. The
    # build's import-shadowing check looks for a module-level definition
    # hidden by an import, not for a function-local one hiding it.

    global_max_capacity = 0

    if COMMON_XAXIS_SCALE:
        for name, df in electrochemical_data.items():
            for col in ['Charge_Capacity', 'Discharge_Capacity']:
                if col in df.columns:
                    col_max = pd.to_numeric(df[col], errors='coerce').max()
                    if pd.notna(col_max):
                        global_max_capacity = max(global_max_capacity, col_max)
    
        # ROUND NUMBER, not the data's own maximum. The measured maximum
        # plus 8% padding landed the shared axis on 143 mAh/g, and a reader
        # comparing three cells across a frame that ends at 143 has to work
        # out whether the curve stopping at 136 means anything. It ends at
        # 150 now, with a gridline every 25.
        global_max_capacity, global_xaxis_step = nice_axis_limit(
            global_max_capacity)
    
        if global_max_capacity > 0:
            print(entry("common x-axis", f"{global_max_capacity:.0f} mAh/g",
                        f"ticks every {global_xaxis_step:g}"
                        if global_xaxis_step else ""))

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        print(rule(composition))
        charge_rate_c = params.get('charge_rate_c', 'Unknown')
        colour_palette = params.get('colour_palette', 'viridis_r')
        key_cycles = params.get('key_cycles', [])
        is_anode = params.get('anode_labels_swapped', False)
        dch_label = _discharge_label(params)
        chg_label = _charge_label(params)
    
        df_work = df.copy()
        # Robust dropna (v1.8): require Voltage, Cycle, Step, and at
        # least one capacity column. Some cycler firmware versions fill
        # only the active capacity column per row.
        df_cleaned = df_work.dropna(subset=['Voltage', 'Cycle', 'Step'])
        df_cleaned = df_cleaned[
            df_cleaned['Charge_Capacity'].notna() |
            df_cleaned['Discharge_Capacity'].notna()
        ].copy()
    
        if df_cleaned.empty:
            print(verdict("caution", f"no valid data for {name}"))
            continue
    
        df_discharge = df_cleaned[df_cleaned['Step'] == 'Discharge']
        df_charge = df_cleaned[df_cleaned['Step'] == 'Charge']
    
        unique_cycles = sorted(df_cleaned['Cycle'].unique())
        # `voltage_profiles` accepts `all_cycle_tables` and, until 1.9.0.4,
        # never read it — so the half-cycle the export was taken during was
        # drawn as a profile that stops in mid-air, and counted in the
        # caption. Dropped here rather than dimmed: a voltage profile that
        # ends nowhere reads as a cell that failed.
        _skip = _unusable_cycles(all_cycle_tables.get(name)) \
            if isinstance(all_cycle_tables, dict) else set()
        if _skip:
            _before = len(unique_cycles)
            unique_cycles = [c for c in unique_cycles
                             if int(c) not in _skip]
            if len(unique_cycles) < _before:
                print(f"  Cycle(s) {', '.join(str(c) for c in sorted(_skip))} "
                      f"omitted: not a finished measurement")
        num_cycles = len(unique_cycles)
    
        palette = sns.color_palette(colour_palette, n_colors=num_cycles)
    
        fig, ax = plt.subplots()
    
        # --- Plot traces ---
        # After Cell 4b normalisation, Discharge = useful half-cycle for
        # all electrode types. Solid = discharge, dashed = charge.
        for i, cycle in enumerate(unique_cycles):
            colour = palette[i]
        
            dc = df_discharge[df_discharge['Cycle'] == cycle]
            if not dc.empty:
                dc_sorted = dc.sort_values('Discharge_Capacity')
                ax.plot(dc_sorted['Discharge_Capacity'],
                       dc_sorted['Voltage'],
                       color=colour, linestyle='-', linewidth=1.0)
        
            cc = df_charge[df_charge['Cycle'] == cycle]
            if not cc.empty:
                cc_sorted = cc.sort_values('Charge_Capacity')
                ax.plot(cc_sorted['Charge_Capacity'],
                       cc_sorted['Voltage'],
                       color=colour, linestyle='--', linewidth=1.0)
    
        # --- Labelling mode selection ---
        endpoints = _get_endpoints(df_charge, df_discharge,
                                    unique_cycles, key_cycles)
    
        if LABEL_MODE == 'auto':
            overlap_frac = _check_overlap(endpoints)
            use_colourbar = overlap_frac > AUTO_OVERLAP_THRESHOLD
            mode_used = 'colourbar' if use_colourbar else 'annotations'
            print(f"  Label mode: auto → {mode_used} "
                  f"({overlap_frac*100:.0f}% overlap, "
                  f"threshold {AUTO_OVERLAP_THRESHOLD*100:.0f}%)")
        elif LABEL_MODE == 'colourbar':
            use_colourbar = True
            mode_used = 'colourbar'
        else:
            use_colourbar = False
            mode_used = 'annotations'
    
        if use_colourbar:
            _add_colourbar(fig, ax, unique_cycles, colour_palette)
        else:
            _add_annotations(ax, endpoints, unique_cycles, palette)
    
        # --- Formatting ---
        ax.set_xlabel('Specific Capacity / mAh g$^{-1}$', fontsize=16)
        ax.set_ylabel('Voltage / V', fontsize=16)
        ax.tick_params(axis='both', labelcolor='black', labelsize=14,
                       width=1, direction='in', top=True, right=True)
    
        if COMMON_XAXIS_SCALE and global_max_capacity > 0:
            ax.set_xlim(left=0, right=global_max_capacity)
            # The limit is a round number, so the ticks should be the round
            # numbers under it. Matplotlib's own locator, given 0-150, picks
            # 0/20/40/.../140 and leaves the last gridline 10 short of the
            # frame; the step that produced the limit does not.
            if global_xaxis_step:
                ax.xaxis.set_major_locator(
                    mticker.MultipleLocator(global_xaxis_step))
        else:
            ax.set_xlim(left=0)
            ax.set_xlim(right=ax.get_xlim()[1] * (1 + XAXIS_PADDING_FRACTION))
    
        plt.tight_layout()
    
        # --- Save ---
        if save_location:
            file_format = params.get('file_format', 'png')
            filename = f"{name}_full_voltage_profile.{file_format}"
            filepath = os.path.join(save_location, filename)
            fig.savefig(filepath, dpi=300, bbox_inches='tight')
            saved(filepath)

        # --- Figure caption (v1.8: anode-aware) ---
        # The PROTOCOL window, not the extent of the record. See
        # `_caption_window`: the record includes the open-circuit start.
        v_min, v_max = _caption_window(params, df_work)
        caption = (
            f"Figure X. Voltage profiles for {num_cycles} galvanostatic "
            f"cycles of {composition}, cycled between {v_min:.2f} and "
            f"{v_max:.2f} V at {rate_phrase(params)}. "
            f"{dch_label} data are shown as solid lines and "
            f"{chg_label.lower()} data as dashed lines, with colour "
            f"indicating cycle number."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))
    
        plt.show()
        plt.close(fig)

    print(rule())


@_honours_verbose
def voltage_profiles_key(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    The same overlay, filtered to the key cycles.

    Ported from 1.8.7 Cell 6, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """

    def _check_overlap(endpoints, cap_tol_frac=0.03, volt_tol=0.05):
        """Check what fraction of endpoints would overlap."""
        if len(endpoints) < 2:
            return 0.0
    
        caps = [e['capacity'] for e in endpoints]
        cap_range = max(caps) - min(caps) if len(caps) > 1 else max(caps)
        cap_tol = max(cap_range * cap_tol_frac, 2.0)
    
        n_overlapping = 0
        for i, ep in enumerate(endpoints):
            for j, other in enumerate(endpoints):
                if i == j:
                    continue
                if (abs(ep['capacity'] - other['capacity']) < cap_tol and
                    abs(ep['voltage'] - other['voltage']) < volt_tol):
                    n_overlapping += 1
                    break
    
        return n_overlapping / len(endpoints)

    def _add_annotations(ax, endpoints, unique_cycles, palette):
        """Add cycle number annotations at endpoints with staggering."""
        if not endpoints:
            return
    
        caps = [e['capacity'] for e in endpoints]
        cap_range = max(caps) - min(caps) if len(caps) > 1 else max(caps)
        cap_tol = max(cap_range * 0.03, 2.0)
        volt_tol = 0.05
    
        placed = []
        sorted_endpoints = sorted(endpoints, key=lambda e: e['cycle'],
                                   reverse=True)
    
        for ep in sorted_endpoints:
            cycle_idx = list(unique_cycles).index(ep['cycle'])
            colour = palette[cycle_idx]
            is_discharge = ep['step'] == 'Discharge'
        
            n_nearby = sum(
                1 for pc, pv in placed
                if (abs(ep['capacity'] - pc) < cap_tol and
                    abs(ep['voltage'] - pv) < volt_tol)
            )
        
            x_offset = 0
            if n_nearby > 0:
                direction = 1 if n_nearby % 2 == 0 else -1
                stagger = (n_nearby + 1) // 2
                x_offset = direction * stagger * 14
        
            y_offset = -10 if is_discharge else 10
            va = 'top' if is_discharge else 'bottom'
        
            ax.annotate(
                str(int(ep['cycle'])),
                xy=(ep['capacity'], ep['voltage']),
                xytext=(x_offset, y_offset),
                textcoords='offset points',
                ha='center', va=va,
                fontsize=8, fontweight='bold',
                color=colour, alpha=0.9
            )
            placed.append((ep['capacity'], ep['voltage']))

    def _add_colourbar(fig, ax, unique_cycles, colour_palette):
        """Add a colourbar mapping cycle number to colour."""
        cmap = plt.get_cmap(colour_palette)
        norm = mcolors.Normalize(vmin=min(unique_cycles),
                                  vmax=max(unique_cycles))
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
    
        cbar = fig.colorbar(sm, ax=ax, pad=0.02, aspect=30)
        cbar.set_label('Cycle number', fontsize=13)
        cbar.ax.tick_params(labelsize=11)

    global_max_capacity = 0

    if COMMON_XAXIS_SCALE:
        for name, df in electrochemical_data.items():
            params = user_parameters.get(name, {})
            key_cycles = (OVERRIDE_KEY_CYCLES if OVERRIDE_KEY_CYCLES
                         else params.get('key_cycles', []))
        
            df_work = df.copy()
            for col in ['Charge_Capacity', 'Discharge_Capacity']:
                if col in df_work.columns:
                    df_work[col] = pd.to_numeric(df_work[col], errors='coerce')
        
            if key_cycles:
                df_filtered = df_work[df_work['Cycle'].isin(key_cycles)]
            else:
                df_filtered = df_work
        
            for col in ['Charge_Capacity', 'Discharge_Capacity']:
                if col in df_filtered.columns:
                    col_max = df_filtered[col].max()
                    if pd.notna(col_max):
                        global_max_capacity = max(global_max_capacity, col_max)
    
        global_max_capacity, global_xaxis_step = nice_axis_limit(
            global_max_capacity)
    
        if global_max_capacity > 0:
            print(entry("common x-axis", f"{global_max_capacity:.0f} mAh/g",
                        f"key cycles; ticks every {global_xaxis_step:g}"
                        if global_xaxis_step else "key cycles"))

    _have_cycle_tables = _has_tables(all_cycle_tables)

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        print(rule(composition))
        charge_rate_c = params.get('charge_rate_c', 'Unknown')
        colour_palette = params.get('colour_palette', 'viridis_r')
        key_cycles = (OVERRIDE_KEY_CYCLES if OVERRIDE_KEY_CYCLES
                     else params.get('key_cycles', []))
        is_anode = params.get('anode_labels_swapped', False)
        dch_label = _discharge_label(params)
        chg_label = _charge_label(params)
    
        # Work on a copy
        df_work = df.copy()
    
        # Robust dropna (v1.8): require Voltage, Cycle, Step, and at
        # least one capacity column.
        df_cleaned = df_work.dropna(subset=['Voltage', 'Cycle', 'Step'])
        df_cleaned = df_cleaned[
            df_cleaned['Charge_Capacity'].notna() |
            df_cleaned['Discharge_Capacity'].notna()
        ].copy()
    
        if df_cleaned.empty:
            print(verdict("caution", f"no valid data for {name}"))
            continue
    
        # --- Detect and exclude incomplete cycles ---
        all_cycles_sorted = sorted(df_cleaned['Cycle'].unique())
    
        if _have_cycle_tables and name in all_cycle_tables:
            # Prefer the table's protocol-level flags (Cell 6a)
            ct = all_cycle_tables[name]
            incomplete_cycles = set(
                _unusable_cycles(ct))
        else:
            # Fallback: protocol-level only (missing/zero discharge).
            # v1.8: removed the 80%-of-median heuristic which caused
            # cascade failures for anode data where first-cycle capacity
            # is anomalously high.
            incomplete_cycles = set()
            for cyc in all_cycles_sorted:
                dc = df_cleaned[(df_cleaned['Cycle'] == cyc) &
                                (df_cleaned['Step'] == 'Discharge')]
                if dc.empty:
                    incomplete_cycles.add(cyc)
                else:
                    cap = (dc['Discharge_Capacity'].max() -
                           dc['Discharge_Capacity'].min())
                    if pd.isna(cap) or cap <= 0:
                        incomplete_cycles.add(cyc)
    
        # Filter key cycles: must exist in data AND not be incomplete
        available_key_cycles = [
            c for c in key_cycles
            if c in all_cycles_sorted and c not in incomplete_cycles
        ]
    
        if incomplete_cycles & set(key_cycles):
            removed = incomplete_cycles & set(key_cycles)
            print(entry("key cycles excluded",
                        ", ".join(str(int(c)) for c in sorted(removed)),
                        "incomplete"))
    
        if not available_key_cycles:
            print(verdict("caution", f"no valid key cycles found for {name}"))
            # Fall back to all available non-incomplete cycles
            available_key_cycles = [
                c for c in all_cycles_sorted
                if c not in incomplete_cycles
            ]
            if not available_key_cycles:
                continue
            print(f"  Falling back to all complete cycles: "
                  f"{', '.join(str(int(c)) for c in available_key_cycles)}")
    
        # Filter data
        df_key = df_cleaned[df_cleaned['Cycle'].isin(available_key_cycles)]
        df_discharge = df_key[df_key['Step'] == 'Discharge']
        df_charge = df_key[df_key['Step'] == 'Charge']
    
        unique_cycles = sorted(df_key['Cycle'].unique())
        num_cycles = len(unique_cycles)
    
        palette = sns.color_palette(colour_palette, n_colors=num_cycles)
    
        fig, ax = plt.subplots()
    
        # --- Plot traces ---
        # After Cell 4b normalisation, solid = discharge = useful
        # half-cycle for both electrode types.
        for i, cycle in enumerate(unique_cycles):
            colour = palette[i]
        
            dc = df_discharge[df_discharge['Cycle'] == cycle]
            if not dc.empty:
                dc_sorted = dc.sort_values('Discharge_Capacity')
                ax.plot(dc_sorted['Discharge_Capacity'],
                       dc_sorted['Voltage'],
                       color=colour, linestyle='-', linewidth=1.2)
        
            cc = df_charge[df_charge['Cycle'] == cycle]
            if not cc.empty:
                cc_sorted = cc.sort_values('Charge_Capacity')
                ax.plot(cc_sorted['Charge_Capacity'],
                       cc_sorted['Voltage'],
                       color=colour, linestyle='--', linewidth=1.2)
    
        # --- Labelling ---
        endpoints = _get_endpoints(df_charge, df_discharge,
                                    unique_cycles, available_key_cycles)
    
        if LABEL_MODE == 'auto':
            overlap_frac = _check_overlap(endpoints)
            use_colourbar = overlap_frac > AUTO_OVERLAP_THRESHOLD
            mode_used = 'colourbar' if use_colourbar else 'annotations'
            print(f"  Label mode: auto -> {mode_used} "
                  f"({overlap_frac*100:.0f}% overlap)")
        elif LABEL_MODE == 'colourbar':
            use_colourbar = True
        else:
            use_colourbar = False
    
        if use_colourbar:
            _add_colourbar(fig, ax, unique_cycles, colour_palette)
        else:
            _add_annotations(ax, endpoints, unique_cycles, palette)
    
        # --- Formatting ---
        ax.set_xlabel('Specific Capacity / mAh g$^{-1}$', fontsize=14)
        ax.set_ylabel('Voltage / V', fontsize=14)
        ax.tick_params(axis='both', labelcolor='black', labelsize=12,
                       width=1, direction='in', top=True, right=True)
    
        if COMMON_XAXIS_SCALE and global_max_capacity > 0:
            ax.set_xlim(left=0, right=global_max_capacity)
            # The limit is a round number, so the ticks should be the round
            # numbers under it. Matplotlib's own locator, given 0-150, picks
            # 0/20/40/.../140 and leaves the last gridline 10 short of the
            # frame; the step that produced the limit does not.
            if global_xaxis_step:
                ax.xaxis.set_major_locator(
                    mticker.MultipleLocator(global_xaxis_step))
        else:
            ax.set_xlim(left=0)
            ax.set_xlim(right=ax.get_xlim()[1] * (1 + XAXIS_PADDING_FRACTION))
    
        plt.tight_layout()
    
        # --- Save ---
        if save_location:
            file_format = params.get('file_format', 'png')
            filename = f"{name}_key_cycles_voltage_profile.{file_format}"
            filepath = os.path.join(save_location, filename)
            fig.savefig(filepath, dpi=300, bbox_inches='tight')
            saved(filepath)

        # --- Figure caption (v1.8: anode-aware) ---
        # The PROTOCOL window, not the extent of the record. See
        # `_caption_window`: the record includes the open-circuit start.
        v_min, v_max = _caption_window(params, df_work)
        cycles_str = ', '.join(str(int(c)) for c in unique_cycles)
        caption = (
            f"Figure X. Voltage profiles for selected galvanostatic "
            f"cycles ({cycles_str}) of {composition}, cycled between "
            f"{v_min:.2f} and {v_max:.2f} V at a rate of "
            f"{rate_phrase(params)}. {dch_label} data are shown as solid "
            f"lines and {chg_label.lower()} data as dashed lines."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

        plt.show()
        plt.close(fig)
    
        # --- Print which cycles were plotted ---
        print(f"  Plotted cycles: "
              f"{', '.join(str(int(c)) for c in unique_cycles)}")

    print(rule())


@_honours_verbose
def cycle_life(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Capacity and coulombic efficiency against cycle number.

    Ported from 1.8.7 Cell 7, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    MARKER_SIZE = 7

    def _calculate_cycle_capacities(df):
        """The canonical per-cycle capacity, under this function's old
        column names. One definition — see `delivered_capacity`."""
        out = delivered_capacity(df)
        return out.rename(columns={"Discharge_mAh_g": "Discharge_Capacity",
                                   "Charge_mAh_g": "Charge_Capacity"})

    def _calculate_efficiency(row):
        """CE for one row, under this function's old column names. One
        definition — see `coulombic_efficiency`."""
        return float(coulombic_efficiency(
            [row['Discharge_Capacity']], [row['Charge_Capacity']]).iloc[0])

    _have_cycle_tables = _has_tables(all_cycle_tables)

    if EXCLUDE_INCOMPLETE and not _have_cycle_tables:
        print("  Note: Cell 5b not run — using protocol-level incomplete "
              "detection (missing/zero discharge).")

    global_max_capacity = 0

    if COMMON_YAXIS_CAPACITY:
        for name, df in electrochemical_data.items():
            cap_df = _calculate_cycle_capacities(df)
            if not cap_df.empty:
                for col in ['Charge_Capacity', 'Discharge_Capacity']:
                    col_max = cap_df[col].max()
                    if pd.notna(col_max):
                        global_max_capacity = max(global_max_capacity, col_max)
    
        global_max_capacity, global_yaxis_step = nice_axis_limit(
            global_max_capacity)
    
        if global_max_capacity > 0:
            print(entry("common y-axis", f"{global_max_capacity:.0f} mAh/g",
                        f"capacity; ticks every {global_yaxis_step:g}"
                        if global_yaxis_step else "capacity"))

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        print(rule(composition))
        charge_rate_c = params.get('charge_rate_c', 'Unknown')
        is_anode = params.get('anode_labels_swapped', False)
        dch_label = _discharge_label(params)
        chg_label = _charge_label(params)
    
        # Calculate capacities
        cap_df = _calculate_cycle_capacities(df)
    
        if cap_df.empty:
            print(verdict("caution", f"no valid capacity data for {name}"))
            continue
    
        # --- Detect incomplete cycles ---
        if EXCLUDE_INCOMPLETE:
            if _have_cycle_tables and name in all_cycle_tables:
                # Prefer the table's protocol-level flags (Cell 6a)
                ct = all_cycle_tables[name]
                incomplete_set = set(
                    _unusable_cycles(ct))
                cap_df['Incomplete'] = cap_df['Cycle'].isin(incomplete_set)
            else:
                # Fallback: protocol-level only (missing/zero discharge).
                # v1.8: removed 80%-of-median heuristic.
                flags = []
                for _, row in cap_df.iterrows():
                    d = row['Discharge_Capacity']
                    flags.append(pd.isna(d) or d <= 0)
                cap_df['Incomplete'] = flags
        
            n_incomplete = cap_df['Incomplete'].sum()
            if n_incomplete > 0:
                removed = cap_df[cap_df['Incomplete']]['Cycle'].tolist()
                print(entry("cycles excluded",
                            ", ".join(str(int(c)) for c in removed),
                            "incomplete"))
        
            plot_df = cap_df[~cap_df['Incomplete']].copy()
        else:
            plot_df = cap_df.copy()
    
        if plot_df.empty:
            print(verdict("caution", f"no complete cycles for {name}"))
            continue
    
        # Calculate CE (v1.8: no electrode_type needed, swap handles it)
        plot_df['CE'] = plot_df.apply(_calculate_efficiency, axis=1)
    
        # CE: skip formation cycle
        ce_df = plot_df[
            (plot_df['Cycle'] >= CE_START_CYCLE) & 
            (plot_df['CE'].notna())
        ]
    
        # --- Plot ---
        fig, ax1 = plt.subplots()
        ax2 = ax1.twinx()
    
        # Capacity (left axis) — legend labels adapt for anodes
        ax1.plot(plot_df['Cycle'], plot_df['Charge_Capacity'],
                color=COLOUR_CHARGE, marker=MARKER_CHARGE,
                markersize=MARKER_SIZE, linestyle='',
                label=chg_label, zorder=3)
    
        ax1.plot(plot_df['Cycle'], plot_df['Discharge_Capacity'],
                color=COLOUR_DISCHARGE, marker=MARKER_DISCHARGE,
                markersize=MARKER_SIZE, linestyle='',
                label=dch_label, zorder=3)
    
        # CE (right axis)
        if not ce_df.empty:
            ax2.plot(ce_df['Cycle'], ce_df['CE'],
                    color=COLOUR_EFFICIENCY, marker=MARKER_EFFICIENCY,
                    markersize=MARKER_SIZE - 1, linestyle='',
                    label='CE', zorder=2, alpha=0.7)
    
        # --- Axis formatting ---
        ax1.set_xlabel('Cycle number', fontsize=14)
        _force_integer_cycles(ax1)
        ax1.set_ylabel('Specific capacity / mAh g$^{-1}$', fontsize=14)
        ax2.set_ylabel('Coulombic efficiency / %', fontsize=14)
    
        # Y-axis limits: capacity
        if COMMON_YAXIS_CAPACITY and global_max_capacity > 0:
            ax1.set_ylim(bottom=0, top=global_max_capacity)
            if global_yaxis_step:
                ax1.yaxis.set_major_locator(
                    mticker.MultipleLocator(global_yaxis_step))
        else:
            ax1.set_ylim(bottom=0)
            ax1.set_ylim(top=ax1.get_ylim()[1] * (1 + CAPACITY_YAXIS_PADDING))
    
        # Y-axis limits: efficiency
        ax2.set_ylim(bottom=0, top=EFFICIENCY_YMAX)
    
        # Tick formatting
        ax1.tick_params(axis='both', labelsize=12, width=1, direction='in',
                        top=True)
        ax2.tick_params(axis='y', labelsize=12, width=1, direction='in')

        # Combined legend from both axes
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2,
                   fontsize=10, framealpha=0.7, loc='best')
    
        # Remove grids (clean look for dual-axis)
        ax1.grid(False)
        ax2.grid(False)
    
        # Spines
        for ax in [ax1, ax2]:
            for sp in ax.spines.values():
                sp.set_linewidth(0.8)
    
        plt.tight_layout()
    
        # --- Save ---
        if save_location:
            file_format = params.get('file_format', 'png')
            filename = f"{name}_cycle_life.{file_format}"
            filepath = os.path.join(save_location, filename)
            fig.savefig(filepath, dpi=300, bbox_inches='tight')
            saved(filepath)

        # --- Figure caption (v1.8: anode-aware) ---
        # Protocol window; see `_caption_window`.
        v_min, v_max = _caption_window(params, df)
        n_complete = len(plot_df)
    
        caption = (
            f"Figure X. Specific {chg_label.lower()} and "
            f"{dch_label.lower()} capacities (left axis) and "
            f"coulombic efficiency (right axis) for {composition} "
            f"over {n_complete} galvanostatic cycles between "
            f"{v_min:.2f} and {v_max:.2f} V at a rate of "
            f"{rate_phrase(params)}. {chg_label} capacities are shown as "
            f"amber squares, {dch_label.lower()} capacities as blue "
            f"circles, and coulombic efficiency as black triangles."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))
    
        plt.show()
        plt.close(fig)
    
        # --- Print summary ---
        if not ce_df.empty:
            mean_ce = ce_df['CE'].mean()
            std_ce = ce_df['CE'].std()
            print(f"  Average CE (cycles {CE_START_CYCLE}–"
                  f"{int(plot_df['Cycle'].max())}): "
                  f"{mean_ce:.2f}% (+/-{std_ce:.2f}%)")

    print(rule())


In [ ]:
# DEFINITIONS J2 — CYCLING: COMPARISON, RETENTION AND FADE
# =============================================================================
@_honours_verbose
def comparative_capacity(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Discharge capacity for every dataset on one axis.

    Ported from 1.8.7 Cell 8, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    MARKER_SIZE = 7

    # The filter default, read from the one definition rather than

    # restated: six local copies were quoted in the console message

    # while `_flag_visual_outliers` used its own default, so editing

    # one changed the sentence and not the data.

    VISUAL_OUTLIER_THRESHOLD = VISUAL_OUTLIER_FRACTION
    def _comparative_capacity_label():
        """Return the best y-axis label for a multi-dataset capacity plot."""
        labels = set()
        for name in electrochemical_data:
            params = user_parameters.get(name, {})
            labels.add(_discharge_label(params))
        if len(labels) == 1:
            return f"{labels.pop()} capacity / mAh g$^{{-1}}$"
        return 'Specific capacity / mAh g$^{-1}$'

    def _comparative_capacity_word():
        """Return the capacity noun for captions (no units)."""
        labels = set()
        for name in electrochemical_data:
            params = user_parameters.get(name, {})
            labels.add(_discharge_label(params).lower())
        if len(labels) == 1:
            return f"{labels.pop()} capacities"
        return 'specific capacities'

    print(rule("COMPARATIVE DISCHARGE CAPACITY"))

    palette = MULTI_DATASET_PALETTE

    fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

    caption_compositions = []

    caption_rates = set()

    _have_cycle_tables = _has_tables(all_cycle_tables)

    if not _have_cycle_tables:
        print("  ⚠ all_cycle_tables not found — Cell 5b has not been run. "
              "Run Cell 5b first for the cleanest output.")

    for i, (name, df) in enumerate(electrochemical_data.items()):
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        charge_rate_c = params.get('charge_rate_c', None)
        dch_label = _discharge_label(params)

        colour = palette[i % len(palette)]
        marker = MARKERS[i % len(MARKERS)]

        # Primary path: Cell 5b cycle table
        if _have_cycle_tables and name in all_cycle_tables:
            ct = all_cycle_tables[name]
            plot_data = (_ct_with_flag(ct, 'Discharge_mAh_g')
                         .rename(columns={'Discharge_mAh_g':
                                          'Discharge_Capacity'})
                         .dropna(subset=['Cycle', 'Discharge_Capacity'])
                         .copy())
        else:
            # Fallback: derive via groupby (no Python loop)
            discharge = df[df['Step'] == 'Discharge'].dropna(
                subset=['Cycle', 'Discharge_Capacity'])
            if discharge.empty:
                print(f"  {composition}: no {dch_label.lower()} data "
                      f"found, skipping.")
                continue
            grp = discharge.groupby('Cycle')['Discharge_Capacity']
            plot_data = pd.DataFrame({
                'Cycle': grp.max().index.astype(int),
                'Discharge_Capacity': (grp.max() - grp.min()).values,
                'Incomplete': False,
            })

        if plot_data.empty:
            print(f"  {composition}: no {dch_label.lower()} data "
                  f"found, skipping.")
            continue

        plot_data = plot_data.sort_values('Cycle').reset_index(drop=True)

        # Exclude incomplete cycles (protocol-level flag from Cell 5b)
        if EXCLUDE_INCOMPLETE and 'Incomplete' in plot_data.columns:
            n_incomplete = int(plot_data['Incomplete'].sum())
            if n_incomplete > 0:
                removed = plot_data.loc[plot_data['Incomplete'],
                                        'Cycle'].tolist()
                print(f"  {composition}: excluded {n_incomplete} "
                      f"protocol-incomplete cycle(s): "
                      f"{', '.join(str(int(c)) for c in removed)}")
            plot_data = plot_data[~plot_data['Incomplete']].reset_index(drop=True)

        # Visual-outlier filter
        if VISUAL_OUTLIER_FILTER and not plot_data.empty:
            outlier_flags = _flag_visual_outliers(
                plot_data['Discharge_Capacity'].tolist(), threshold=VISUAL_OUTLIER_THRESHOLD,
                groups=rate_block_labels(plot_data['Cycle'], params))
            n_outliers = sum(outlier_flags)
            if n_outliers > 0:
                plot_data['_visual_outlier'] = outlier_flags
                removed = plot_data.loc[plot_data['_visual_outlier'],
                                        'Cycle'].tolist()
                print(f"  {composition}: excluded {n_outliers} visual "
                      f"outlier(s) (<{VISUAL_OUTLIER_THRESHOLD*100:.0f}% "
                      f"of running median): "
                      f"{', '.join(str(int(c)) for c in removed)}")
                plot_data = plot_data[~plot_data['_visual_outlier']].drop(
                    columns='_visual_outlier').reset_index(drop=True)

        if plot_data.empty:
            print(f"  {composition}: no complete cycles, skipping.")
            continue

        # Plot
        ax.plot(plot_data['Cycle'], plot_data['Discharge_Capacity'],
               color=colour, marker=marker, markersize=MARKER_SIZE,
               linestyle='', label=composition, zorder=3)

        caption_compositions.append(composition)
        if charge_rate_c is not None:
            caption_rates.add(str(charge_rate_c))

        print(f"  {composition}: {len(plot_data)} cycles plotted "
              f"({plot_data['Discharge_Capacity'].iloc[0]:.1f} -> "
              f"{plot_data['Discharge_Capacity'].iloc[-1]:.1f} mAh/g)")

    ax.set_xlabel('Cycle number', fontsize=14)

    _force_integer_cycles(ax)

    ax.set_ylabel(_comparative_capacity_label(), fontsize=14)

    ax.tick_params(axis='both', labelsize=12, width=1, direction='in',
                   top=True, right=True)

    for sp in ax.spines.values():
        sp.set_linewidth(0.8)

    ax.set_xlim(left=0)

    ax.set_ylim(bottom=0, top=ax.get_ylim()[1] * (1 + YAXIS_PADDING))

    ax.legend(fontsize=10, framealpha=0.7)

    plt.tight_layout()

    if caption_compositions:
        comp_str = ', '.join(caption_compositions[:-1])
        if len(caption_compositions) > 1:
            comp_str += f' and {caption_compositions[-1]}'
        else:
            comp_str = caption_compositions[0]

        rate_str = (' and '.join(sorted(caption_rates)) + ' C'
                   if caption_rates else 'the specified C-rate')

        # EVERY dataset's protocol window, not the union of their
        # records — which read 1.20-2.68 V for a triplicate cycled
        # 1.2-2.5 V, because one cell started from open circuit.
        _vlo, _vhi = _caption_window_all(user_parameters)
        v_range = (f"{_vlo:.2f} and {_vhi:.2f} V"
                   if np.isfinite(_vlo) and np.isfinite(_vhi)
                   else "the specified voltage limits")

        cap_word = _comparative_capacity_word()
        caption = (
            f"Figure X. Comparative {cap_word} for "
            f"{comp_str}, cycled at {rate_str} between {v_range}."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

    if save_location:
        filepath = os.path.join(save_location,
                               f'comparative_discharge_capacity.{_run_image_format(user_parameters)}')
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        saved(filepath)

    plt.show()

    plt.close(fig)

    if len(electrochemical_data) > 1:
        loadings = {}
        for name in electrochemical_data:
            params = user_parameters.get(name, {})
            loading = params.get('active_loading_mg_cm2')
            if loading is not None:
                comp = _get_display_name(name, params, user_parameters)
                loadings[comp] = loading

        if len(loadings) >= 2:
            lo_vals = list(loadings.values())
            lo_min, lo_max = min(lo_vals), max(lo_vals)
            if lo_min > 0:
                spread_pct = (lo_max - lo_min) / lo_min * 100
                if spread_pct > 30:
                    # Adaptive label for the warning too
                    cap_word = _comparative_capacity_word()
                    print(f"\n  ⚠ LOADING MISMATCH WARNING")
                    print(f"  Active material loadings vary by "
                          f"{spread_pct:.0f}% across datasets:")
                    for comp, val in loadings.items():
                        print(f"    {comp:<30} {val:.2f} mg/cm²")
                    print(f"  Specific capacity comparisons may "
                          f"be misleading at different")
                    print(f"  loadings — electrochemical properties often "
                          f"do not scale with mass loading")
                    print(f"  (Cao et al., Nat. Nanotechnol. 14, 200-207, "
                          f"2019).")
                    print(f"  Consider comparing areal capacity "
                          f"(mAh/cm², Cell 10) instead.")

    print(rule())


@_honours_verbose
def capacity_retention(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Retention against a reference cycle.

    Ported from 1.8.7 Cell 9, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    MARKER_SIZE = 7

    YAXIS_MIN = 0

    YAXIS_MAX = 105  # auto-scales higher if any retention >105%

    # The filter default, read from the one definition rather than

    # restated: six local copies were quoted in the console message

    # while `_flag_visual_outliers` used its own default, so editing

    # one changed the sentence and not the data.

    VISUAL_OUTLIER_THRESHOLD = VISUAL_OUTLIER_FRACTION
    def _retention_ylabel():
        """Return the y-axis label for the retention plot."""
        labels = set()
        for name in electrochemical_data:
            params = user_parameters.get(name, {})
            labels.add(_discharge_label(params))
        if len(labels) == 1:
            lbl = labels.pop()
            return f"{lbl} capacity retention / %"
        return 'Capacity retention / %'

    def _retention_caption_word():
        """Return the capacity retention noun for captions."""
        labels = set()
        for name in electrochemical_data:
            params = user_parameters.get(name, {})
            labels.add(_discharge_label(params).lower())
        if len(labels) == 1:
            return f"{labels.pop()} capacity retention"
        return 'capacity retention'

    _cap_word = _retention_caption_word()

    print(rule(_cap_word.upper()))

    palette = MULTI_DATASET_PALETTE

    fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

    caption_compositions = []

    max_retention_seen = 0

    _have_cycle_tables = _has_tables(all_cycle_tables)

    if not _have_cycle_tables:
        print("  ⚠ all_cycle_tables not found — Cell 5b has not been run. "
              "Run Cell 5b first for the cleanest output.")

    for i, (name, df) in enumerate(electrochemical_data.items()):
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        dch_label = _discharge_label(params)

        colour = palette[i % len(palette)]
        marker = MARKERS[i % len(MARKERS)]

        # Primary path
        if _have_cycle_tables and name in all_cycle_tables:
            ct = all_cycle_tables[name]
            cap_df = (_ct_with_flag(ct, 'Discharge_mAh_g')
                      .rename(columns={'Discharge_mAh_g':
                                       'Discharge_Capacity'})
                      .dropna(subset=['Cycle', 'Discharge_Capacity'])
                      .copy())
        else:
            # Fallback: derive via groupby
            discharge = df[df['Step'] == 'Discharge'].dropna(
                subset=['Cycle', 'Discharge_Capacity'])
            if discharge.empty:
                print(f"  {composition}: no {dch_label.lower()} data, "
                      f"skipping.")
                continue
            grp = discharge.groupby('Cycle')['Discharge_Capacity']
            cap_df = pd.DataFrame({
                'Cycle': grp.max().index.astype(int),
                'Discharge_Capacity': (grp.max() - grp.min()).values,
                'Incomplete': False,
            })

        if cap_df.empty:
            print(f"  {composition}: no {dch_label.lower()} data, "
                  f"skipping.")
            continue

        cap_df = cap_df.sort_values('Cycle').reset_index(drop=True)

        # Exclude incomplete cycles (protocol-level)
        if EXCLUDE_INCOMPLETE and 'Incomplete' in cap_df.columns:
            n_incomplete = int(cap_df['Incomplete'].sum())
            if n_incomplete > 0:
                removed = cap_df.loc[cap_df['Incomplete'],
                                     'Cycle'].tolist()
                print(f"  {composition}: excluded {n_incomplete} "
                      f"protocol-incomplete cycle(s): "
                      f"{', '.join(str(int(c)) for c in removed)}")
            plot_data = cap_df[~cap_df['Incomplete']].copy().reset_index(drop=True)
        else:
            plot_data = cap_df.copy()

        # Visual-outlier filter
        if VISUAL_OUTLIER_FILTER and not plot_data.empty:
            outlier_flags = _flag_visual_outliers(
                plot_data['Discharge_Capacity'].tolist(), threshold=VISUAL_OUTLIER_THRESHOLD,
                groups=rate_block_labels(plot_data['Cycle'], params))
            n_outliers = sum(outlier_flags)
            if n_outliers > 0:
                plot_data['_visual_outlier'] = outlier_flags
                removed = plot_data.loc[plot_data['_visual_outlier'],
                                        'Cycle'].tolist()
                print(f"  {composition}: excluded {n_outliers} visual "
                      f"outlier(s) (<{VISUAL_OUTLIER_THRESHOLD*100:.0f}% "
                      f"of running median): "
                      f"{', '.join(str(int(c)) for c in removed)}")
                plot_data = plot_data[~plot_data['_visual_outlier']].drop(
                    columns='_visual_outlier').reset_index(drop=True)

        if plot_data.empty:
            print(f"  {composition}: no complete cycles, skipping.")
            continue

        # Reference capacity
        ref_row = plot_data[plot_data['Cycle'] == RETENTION_REFERENCE_CYCLE]
        if not ref_row.empty:
            ref_cap = ref_row['Discharge_Capacity'].iloc[0]
        else:
            ref_cap = plot_data['Discharge_Capacity'].iloc[0]
            ref_cycle_used = int(plot_data['Cycle'].iloc[0])
            print(f"  {composition}: reference cycle "
                  f"{RETENTION_REFERENCE_CYCLE} not found, "
                  f"using cycle {ref_cycle_used}")

        if pd.isna(ref_cap) or ref_cap <= 0:
            print(f"  {composition}: invalid reference capacity, skipping.")
            continue

        # Calculate retention
        plot_data['Retention'] = (
            plot_data['Discharge_Capacity'] / ref_cap * 100
        )

        max_retention_seen = max(max_retention_seen,
                                 plot_data['Retention'].max())

        # Plot
        ax.plot(plot_data['Cycle'], plot_data['Retention'],
               color=colour, marker=marker, markersize=MARKER_SIZE,
               linestyle='', label=composition, zorder=3)

        caption_compositions.append(composition)

        last = plot_data.iloc[-1]
        print(f"  {composition}: {len(plot_data)} cycles, "
              f"retention at cycle {int(last['Cycle'])} = "
              f"{last['Retention']:.1f}%")

    ax.set_xlabel('Cycle number', fontsize=14)

    _force_integer_cycles(ax)

    ax.set_ylabel(_retention_ylabel(), fontsize=14)

    ax.tick_params(axis='both', labelsize=12, width=1, direction='in',
                   top=True, right=True)

    for sp in ax.spines.values():
        sp.set_linewidth(0.8)

    ax.set_xlim(left=0)

    y_top = (max(YAXIS_MAX, max_retention_seen * 1.03)
             if max_retention_seen > YAXIS_MAX else YAXIS_MAX)

    ax.set_ylim(bottom=YAXIS_MIN, top=y_top)

    ax.axhline(y=100, color='grey', linestyle=':', linewidth=0.8, alpha=0.5)

    ax.legend(fontsize=10, framealpha=0.7)

    plt.tight_layout()

    if caption_compositions:
        comp_str = ', '.join(caption_compositions[:-1])
        if len(caption_compositions) > 1:
            comp_str += f' and {caption_compositions[-1]}'
        else:
            comp_str = caption_compositions[0]

        caption = (
            f"Figure X. {_cap_word.capitalize()} "
            f"(relative to cycle {RETENTION_REFERENCE_CYCLE}) for "
            f"{comp_str}. A horizontal dotted line at 100% is shown "
            f"for reference."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

    if save_location:
        filepath = os.path.join(save_location,
                               f'capacity_retention_vs_cycle.{_run_image_format(user_parameters)}')
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        saved(filepath)

    plt.show()

    plt.close(fig)

    print(rule())


@_honours_verbose
def fade_rate(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Rate of capacity loss, per cycle and cumulative.

    Ported from 1.8.7 Cell 9b, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    # `palette` was never assigned in 1.8.7 Cell 9b. In the notebook it
    # resolved to whatever an earlier cell had left in globals — the implicit
    # shared state this package exists to remove — and in a function it is a
    # NameError. Cells 8, 9, 11 and 12 all bind it to MULTI_DATASET_PALETTE,
    # and that is the palette the published figure was drawn with, so this
    # reproduces the figure and closes the latent bug.
    palette = MULTI_DATASET_PALETTE

    MARKER_SIZE = 6

    VISUAL_OUTLIER_THRESHOLD = 0.30  # cap < 30% of running median → outlier

    print(rule("CAPACITY FADE RATE"))

    fade_data = {}

    has_rolling_data = False

    _have_cycle_tables = _has_tables(all_cycle_tables)

    if not _have_cycle_tables:
        print("  ⚠ all_cycle_tables not found — Cell 5b has not been run. "
              "Run Cell 5b first for the cleanest output.")

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        dch_label = _discharge_label(params)

        # Primary path: read from Cell 5b
        if _have_cycle_tables and name in all_cycle_tables:
            ct = all_cycle_tables[name]
            cap_df = (_ct_with_flag(ct, 'Discharge_mAh_g')
                      .dropna(subset=['Cycle', 'Discharge_mAh_g'])
                      .copy())
        else:
            # Fallback: derive via groupby (no Python loop)
            discharge = df[df['Step'] == 'Discharge'].dropna(
                subset=['Cycle', 'Discharge_Capacity'])
            if discharge.empty:
                print(f"  {composition}: no {dch_label.lower()} data, "
                      f"skipping.")
                continue
            grp = discharge.groupby('Cycle')['Discharge_Capacity']
            cap_df = pd.DataFrame({
                'Cycle': grp.max().index.astype(int),
                'Discharge_mAh_g': (grp.max() - grp.min()).values,
                'Incomplete': False,
            })

        if cap_df.empty:
            print(f"  {composition}: no {dch_label.lower()} data, "
                  f"skipping.")
            continue

        cap_df = cap_df.sort_values('Cycle').reset_index(drop=True)

        # Exclude incomplete cycles (protocol-level flag from Cell 5b)
        if EXCLUDE_INCOMPLETE and 'Incomplete' in cap_df.columns:
            n_incomplete = int(cap_df['Incomplete'].sum())
            if n_incomplete > 0:
                removed = cap_df.loc[cap_df['Incomplete'],
                                     'Cycle'].tolist()
                print(f"  {composition}: excluded {n_incomplete} "
                      f"protocol-incomplete cycle(s): "
                      f"{', '.join(str(int(c)) for c in removed)}")
            cap_df = cap_df[~cap_df['Incomplete']].copy().reset_index(drop=True)

        # Visual-outlier filter (applied after Incomplete)
        if VISUAL_OUTLIER_FILTER and not cap_df.empty:
            outlier_flags = _flag_visual_outliers(
                cap_df['Discharge_mAh_g'].tolist(), threshold=VISUAL_OUTLIER_THRESHOLD,
                groups=rate_block_labels(cap_df['Cycle'], params))
            n_outliers = sum(outlier_flags)
            if n_outliers > 0:
                cap_df['_visual_outlier'] = outlier_flags
                removed = cap_df.loc[cap_df['_visual_outlier'],
                                     'Cycle'].tolist()
                print(f"  {composition}: excluded {n_outliers} visual "
                      f"outlier(s) (<{VISUAL_OUTLIER_THRESHOLD*100:.0f}% "
                      f"of running median): "
                      f"{', '.join(str(int(c)) for c in removed)}")
                cap_df = cap_df[~cap_df['_visual_outlier']].drop(
                    columns='_visual_outlier').reset_index(drop=True)

        if cap_df.empty:
            continue

        # Reference capacity
        ref_row = cap_df[cap_df['Cycle'] == FADE_REFERENCE_CYCLE]
        if not ref_row.empty:
            ref_cap = ref_row['Discharge_mAh_g'].iloc[0]
        else:
            ref_cap = cap_df['Discharge_mAh_g'].iloc[0]
            ref_cycle_used = int(cap_df['Cycle'].iloc[0])
            print(f"  {composition}: reference cycle "
                  f"{FADE_REFERENCE_CYCLE} not found, "
                  f"using cycle {ref_cycle_used}")

        if pd.isna(ref_cap) or ref_cap <= 0:
            print(f"  {composition}: invalid reference capacity, skipping.")
            continue

        # Cumulative capacity loss (%)
        cap_df['Capacity_Loss_%'] = (
            (ref_cap - cap_df['Discharge_mAh_g']) / ref_cap * 100
        )

        # Per-cycle change (%)
        cap_df['Per_Cycle_Change_%'] = (
            cap_df['Discharge_mAh_g'].diff() / ref_cap * 100
        )

        # Rolling-window fade
        if len(cap_df) >= ROLLING_WINDOW:
            has_rolling_data = True
            cap_df['Per_N_Cycle_Loss_%'] = (
                cap_df['Capacity_Loss_%'].rolling(
                    window=ROLLING_WINDOW, min_periods=ROLLING_WINDOW
                ).apply(lambda x: x.iloc[-1] - x.iloc[0], raw=False)
            )
        else:
            cap_df['Per_N_Cycle_Loss_%'] = np.nan

        cap_df['Composition'] = composition
        fade_data[name] = cap_df

        # Summary
        last = cap_df.iloc[-1]
        print(f"  {composition}: {len(cap_df)} cycles, "
              f"total loss = {last['Capacity_Loss_%']:.1f}%")

    if fade_data:
        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        caption_compositions = []

        for i, (name, fdf) in enumerate(fade_data.items()):
            composition = fdf['Composition'].iloc[0]
            colour = palette[i % len(palette)]
            marker = MARKERS[i % len(MARKERS)]

            ax.plot(fdf['Cycle'], fdf['Capacity_Loss_%'],
                   color=colour, marker=marker, markersize=MARKER_SIZE,
                   linestyle='', label=composition, zorder=3)

            caption_compositions.append(composition)

        ax.set_xlabel('Cycle number', fontsize=14)
        _force_integer_cycles(ax)
        ax.set_ylabel('Cumulative capacity loss / %', fontsize=14)
        ax.tick_params(axis='both', labelsize=12, width=1, direction='in',
                       top=True, right=True)
        ax.axhline(y=0, color='grey', linestyle=':', linewidth=0.8, alpha=0.5)
        for sp in ax.spines.values():
            sp.set_linewidth(0.8)
        ax.legend(fontsize=10, framealpha=0.7)
        plt.tight_layout()

        # Caption
        comp_str = ', '.join(caption_compositions[:-1])
        if len(caption_compositions) > 1:
            comp_str += f' and {caption_compositions[-1]}'
        else:
            comp_str = caption_compositions[0]

        caption = (
            f"Figure X. Cumulative percentage capacity loss relative to "
            f"cycle {FADE_REFERENCE_CYCLE} for {comp_str}. "
            f"A value of 0% indicates capacity equal to the reference cycle."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

        if save_location:
            fpath = os.path.join(save_location, f'capacity_fade_percentage.{_run_image_format(user_parameters)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)

        # =================================================================
        # PLOT 2: PER-CYCLE CAPACITY CHANGE
        # =================================================================

        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        for i, (name, fdf) in enumerate(fade_data.items()):
            composition = fdf['Composition'].iloc[0]
            colour = palette[i % len(palette)]
            marker = MARKERS[i % len(MARKERS)]

            # Skip the first point (NaN from diff)
            valid = fdf.dropna(subset=['Per_Cycle_Change_%'])

            ax.plot(valid['Cycle'], valid['Per_Cycle_Change_%'],
                   color=colour, marker=marker, markersize=MARKER_SIZE - 1,
                   linestyle='', label=composition, zorder=3, alpha=0.7)

        ax.set_xlabel('Cycle number', fontsize=14)
        _force_integer_cycles(ax)
        ax.set_ylabel('Capacity change per cycle / %', fontsize=14)
        ax.tick_params(axis='both', labelsize=12, width=1, direction='in',
                       top=True, right=True)
        ax.axhline(y=0, color='grey', linestyle=':', linewidth=0.8, alpha=0.5)
        for sp in ax.spines.values():
            sp.set_linewidth(0.8)
        ax.legend(fontsize=10, framealpha=0.7)
        plt.tight_layout()

        caption = (
            f"Figure X. Per-cycle capacity change (as percentage of "
            f"cycle {FADE_REFERENCE_CYCLE} capacity) for {comp_str}. "
            f"Negative values indicate capacity loss; positive values "
            f"indicate capacity gain. The horizontal dotted line at 0% "
            f"represents no change."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

        if save_location:
            fpath = os.path.join(save_location, f'capacity_change_per_cycle.{_run_image_format(user_parameters)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)

        # =================================================================
        # PLOT 3: PER-ROLLING_WINDOW FADE (conditional)
        # =================================================================

        if has_rolling_data:
            fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

            plotted_any = False
            for i, (name, fdf) in enumerate(fade_data.items()):
                valid = fdf.dropna(subset=['Per_N_Cycle_Loss_%'])
                if valid.empty:
                    comp = fdf['Composition'].iloc[0]
                    print(f"  {comp}: fewer than {ROLLING_WINDOW} cycles, "
                          f"skipping per-{ROLLING_WINDOW}-cycle plot")
                    continue

                composition = fdf['Composition'].iloc[0]
                colour = palette[i % len(palette)]
                marker = MARKERS[i % len(MARKERS)]

                ax.plot(valid['Cycle'], valid['Per_N_Cycle_Loss_%'],
                       color=colour, marker=marker, markersize=MARKER_SIZE,
                       linestyle='', label=composition, zorder=3)
                plotted_any = True

            if plotted_any:
                ax.set_xlabel('Cycle number', fontsize=14)
                _force_integer_cycles(ax)
                ax.set_ylabel(f'Capacity loss per {ROLLING_WINDOW} cycles / %',
                              fontsize=14)
                ax.tick_params(axis='both', labelsize=12, width=1,
                               direction='in', top=True, right=True)
                ax.axhline(y=0, color='grey', linestyle=':',
                           linewidth=0.8, alpha=0.5)
                for sp in ax.spines.values():
                    sp.set_linewidth(0.8)
                ax.legend(fontsize=10, framealpha=0.7)
                plt.tight_layout()

                caption = (
                    f"Figure X. Capacity loss over a rolling window of "
                    f"{ROLLING_WINDOW} cycles for {comp_str}. Only "
                    f"datasets with {ROLLING_WINDOW}+ cycles are shown."
                )
                print(section("  Suggested caption"))
                print(bullet(caption, indent=2, label_width=2))

                if save_location:
                    fpath = os.path.join(save_location,
                                        f'capacity_loss_per_{ROLLING_WINDOW}_cycles.{_run_image_format(user_parameters)}')
                    fig.savefig(fpath, dpi=300, bbox_inches='tight')
                    saved(fpath)

                plt.show()
            else:
                plt.close(fig)
        else:
            print(f"\n  No datasets have {ROLLING_WINDOW}+ cycles — "
                  f"per-{ROLLING_WINDOW}-cycle plot skipped.")
    else:
        print("\nNo valid fade data to plot.")

    print(rule())


In [ ]:
# DEFINITIONS J3 — CYCLING: ENERGY, POWER, AVERAGE VOLTAGE, EFFICIENCY AND RATE
# =============================================================================
@_honours_verbose
def power_and_energy(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Power and energy density, three normalisations.

    Ported from 1.8.7 Cell 10, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    _run_pe = any(
        user_parameters.get(name, {}).get('power_energy_analysis', True)
        for name in electrochemical_data
    )

    if not _run_pe:
        print("Power and energy density analysis skipped "
              "(disabled in Cell 3 parameters for all datasets).")
    else:
        # =====================================================================
        # USER-ADJUSTABLE DEFAULTS
        # =====================================================================
        DEFAULT_ELECTRODE_DIAMETER_MM = 12.0
        PLOT_PER_ACTIVE_MASS = True
        PLOT_PER_ELECTRODE_MASS = True
        PLOT_PER_AREA = True
        PLOT_RAGONE = True
        RAGONE_BASIS = 'active'  # 'active', 'electrode', or 'area'

        # Areal capacity plot (v1.7.2) — discharge capacity normalised to
        # electrode area. Exposes whether thick electrodes actually deliver
        # proportionally more capacity. Benchmark lines at commercial
        # cathode loadings (2-4 mAh/cm2). Ref: Cao et al., Nat. Nanotechnol.
        # 14, 200-207 (2019).
        PLOT_AREAL_CAPACITY = True

        # Areal capacity benchmarks — set per chemistry.
        # Li-ion NMC commercial: 3-5 mAh/cm2 at 15-25 mg/cm2
        #   (Xie et al., Electrochim. Acta 2023; QuantumScape 2023)
        # Na-ion layered oxide target: 2-3 mAh/cm2 at 20-25 mg/cm2
        #   (Liu et al., Nat. Energy 2025; Zheng et al., ScienceDirect 2023)
        _AREAL_BENCHMARKS = {
            'Li-ion': ([3.0, 5.0], 'Li-ion commercial'),
            'Na-ion': ([2.0, 3.0], 'Na-ion target'),
        }
        _AREAL_BENCHMARK_DEFAULT = ([2.0, 4.0], 'commercial target')

        # Exclude protocol-incomplete cycles (from Cell 5b)
        EXCLUDE_INCOMPLETE = True

        # Visual outlier filter — see Cell 9b for full rationale
        VISUAL_OUTLIER_FILTER = True
        # The filter default, read from the one definition rather than
        # restated: six local copies were quoted in the console message
        # while `_flag_visual_outliers` used its own default, so editing
        # one changed the sentence and not the data.
        VISUAL_OUTLIER_THRESHOLD = VISUAL_OUTLIER_FRACTION
        _have_cycle_tables = _has_tables(all_cycle_tables)
        if EXCLUDE_INCOMPLETE and not _have_cycle_tables:
            print("  ⚠ all_cycle_tables not found — Cell 5b has not been "
                  "run. Incomplete-cycle filtering unavailable.")

        # =====================================================================
        # HELPER FUNCTIONS
        # =====================================================================
        def _parse_active_fraction(blend_str):
            try:
                parts = [float(p) for p in blend_str.split('/')]
                return parts[0] / sum(parts) if sum(parts) > 0 else 0.8
            except (ValueError, AttributeError):
                return 0.8

        def _get_electrode_area(params):
            if 'electrode_area_cm2' in params:
                return params['electrode_area_cm2']
            diameter_mm = params.get('electrode_diameter_mm',
                                     DEFAULT_ELECTRODE_DIAMETER_MM)
            radius_cm = diameter_mm / 2 / 10
            return np.pi * radius_cm**2

        def _apply_pub_style(ax):
            ax.tick_params(axis='both', labelsize=11, width=0.8, direction='in',
                           top=True, right=True)
            for sp in ax.spines.values():
                sp.set_linewidth(0.8)

        def _set_lim_with_padding(ax, axis='y', bottom=0, padding=0.08):
            if axis == 'y':
                current_top = ax.get_ylim()[1]
                ax.set_ylim(bottom=bottom, top=current_top * (1 + padding))
            elif axis == 'x':
                current_right = ax.get_xlim()[1]
                ax.set_xlim(left=bottom, right=current_right * (1 + padding))

        def _extract_power_energy(df, params):
            """
            Per-cycle discharge energy and max power, vectorised via groupby.

            Takes the maximum of the absolute value of Spec. Energy(mWh/g)
            and Power(W) per cycle (the absolute handles sign conventions
            across different cyclers).
            """
            active_mass_mg = params.get('active_material_mass_mg')
            active_mass_g = (active_mass_mg / 1000.0
                            if active_mass_mg is not None else None)
            blend = params.get('blend', '80/10/10')
            active_fraction = params.get('active_fraction',
                                          _parse_active_fraction(blend))
            electrode_area = _get_electrode_area(params)
            total_coating_g = (active_mass_g / active_fraction
                              if active_mass_g is not None else None)

            df_discharge = df[df['Step'] == 'Discharge']
            if df_discharge.empty:
                return pd.DataFrame()

            has_spec_energy = 'Spec. Energy(mWh/g)' in df_discharge.columns
            has_power = 'Power(W)' in df_discharge.columns

            # Cell 2 has already coerced Cycle. Coerce the power/energy
            # columns here since they're not in Cell 2's coercion list.
            work = df_discharge[['Cycle']].copy()
            if has_spec_energy:
                work['_spec_energy_abs'] = pd.to_numeric(
                    df_discharge['Spec. Energy(mWh/g)'],
                    errors='coerce').abs()
            if has_power:
                work['_power_abs'] = pd.to_numeric(
                    df_discharge['Power(W)'], errors='coerce').abs()

            required = ['Cycle'] + [
                c for c in ('_spec_energy_abs', '_power_abs')
                if c in work.columns
            ]
            valid = work.dropna(subset=required)
            if valid.empty:
                return pd.DataFrame()

            # Single vectorised aggregation
            agg_dict = {}
            if has_spec_energy:
                agg_dict['_spec_energy_abs'] = 'max'
            if has_power:
                agg_dict['_power_abs'] = 'max'
            agg = valid.groupby('Cycle').agg(agg_dict).reset_index()
            agg['Cycle'] = agg['Cycle'].astype(int)

            # Derive the normalisation variants (vectorised)
            if has_spec_energy:
                agg['energy_Wh_kg_active'] = agg['_spec_energy_abs']
                if total_coating_g is not None:
                    agg['energy_Wh_kg_electrode'] = (
                        agg['_spec_energy_abs'] * active_fraction)
                if active_mass_g is not None:
                    agg['energy_mWh_cm2'] = (
                        agg['_spec_energy_abs'] * active_mass_g
                        / electrode_area)

            if has_power:
                agg['power_W'] = agg['_power_abs']
                if active_mass_g is not None:
                    agg['power_mW_g_active'] = (
                        agg['_power_abs'] / active_mass_g * 1000)
                if total_coating_g is not None:
                    agg['power_mW_g_electrode'] = (
                        agg['_power_abs'] / total_coating_g * 1000)
                agg['power_mW_cm2'] = (
                    agg['_power_abs'] / electrode_area * 1000)

            return agg.drop(columns=[c for c in ('_spec_energy_abs',
                                                 '_power_abs')
                                     if c in agg.columns])

        # =====================================================================
        # MAIN ANALYSIS
        # =====================================================================
        print(rule("POWER AND ENERGY DENSITY ANALYSIS"))

        all_pe_data = {}

        for name, df in electrochemical_data.items():
            params = user_parameters.get(name, {})
            if not params.get('power_energy_analysis', True):
                print(f"  {name}: power/energy analysis disabled, skipping.")
                continue

            composition = _get_display_name(name, params, user_parameters)
            pe_df = _extract_power_energy(df, params)

            if pe_df.empty:
                dch = _discharge_label(user_parameters.get(name, {}))
                print(f"  {name}: no {dch.lower()} power/energy data "
                      f"found, skipping.")
                continue

            # Exclude protocol-incomplete cycles (Cell 5b flag)
            if (EXCLUDE_INCOMPLETE and _have_cycle_tables
                and name in all_cycle_tables):
                ct = all_cycle_tables[name]
                incomplete_cycles = set(
                    _unusable_cycles(ct))
                if incomplete_cycles:
                    mask = pe_df['Cycle'].isin(incomplete_cycles)
                    removed = pe_df.loc[mask, 'Cycle'].tolist()
                    if removed:
                        print(f"  {composition}: excluded "
                              f"{len(removed)} protocol-incomplete "
                              f"cycle(s): "
                              f"{', '.join(str(c) for c in removed)}")
                    pe_df = pe_df[~mask].reset_index(drop=True)

            # Visual-outlier filter (using energy as the capacity proxy;
            # falls back to power if energy isn't available)
            if VISUAL_OUTLIER_FILTER and not pe_df.empty:
                proxy_col = None
                for candidate in ('energy_Wh_kg_active',
                                  'energy_mWh_cm2', 'power_W'):
                    if candidate in pe_df.columns:
                        proxy_col = candidate
                        break
                if proxy_col is not None:
                    outlier_flags = _flag_visual_outliers(
                        pe_df[proxy_col].tolist(), threshold=VISUAL_OUTLIER_THRESHOLD,
                        groups=rate_block_labels(pe_df['Cycle'], params))
                    n_outliers = sum(outlier_flags)
                    if n_outliers > 0:
                        removed = [int(c) for c, f in
                                   zip(pe_df['Cycle'], outlier_flags) if f]
                        print(f"  {composition}: excluded {n_outliers} "
                              f"visual outlier(s) "
                              f"(<{VISUAL_OUTLIER_THRESHOLD*100:.0f}% of "
                              f"running median on {proxy_col}): "
                              f"{', '.join(str(c) for c in removed)}")
                        pe_df = pe_df[
                            ~pd.Series(outlier_flags,
                                       index=pe_df.index)
                        ].reset_index(drop=True)

            if pe_df.empty:
                print(f"  {composition}: no valid cycles after filtering, "
                      "skipping.")
                continue

            all_pe_data[name] = pe_df

            active_mass = params.get('active_material_mass_mg', '?')
            blend = params.get('blend', '80/10/10')
            active_frac = params.get('active_fraction',
                                      _parse_active_fraction(blend))
            area = _get_electrode_area(params)
            loading = (float(active_mass) / area
                      if isinstance(active_mass, (int, float)) else None)

            print(heading(composition))
            print(f"    Active mass: {active_mass} mg | "
                  f"Blend: {blend} ({active_frac*100:.0f}% active)")
            print(f"    Electrode area: {area:.3f} cm2"
                  f"{f' | Loading: {loading:.2f} mg/cm2' if loading else ''}")
            print(f"    Cycles with data: {len(pe_df)}")
            if 'energy_Wh_kg_active' in pe_df.columns:
                print(f"    Energy range: "
                      f"{pe_df['energy_Wh_kg_active'].min():.1f}--"
                      f"{pe_df['energy_Wh_kg_active'].max():.1f} "
                      f"Wh/kg (active)")

        # =====================================================================
        # PLOTTING
        # =====================================================================
        if not all_pe_data:
            print("\nNo power/energy data available for any dataset.")
        else:
            # BY DATASET, so a cell keeps the same colour AND the same
            # marker shape it has in every other figure of the run. This was
            # `tab10` sampled continuously, which gave both a different
            # palette from the plots above it and, at two cells, a blue and a
            # cyan — the one pair a colourblind reader cannot separate.
            colours = categorical_colours(len(all_pe_data))
            markers = categorical_markers(len(all_pe_data))

            # v1.8: determine electrode type word for captions
            _etypes = set(
                user_parameters.get(n, {}).get('electrode_type', 'Positive')
                for n in all_pe_data)
            if _etypes == {'Negative'}:
                _mat_word = 'anode'
            elif _etypes == {'Positive'}:
                _mat_word = 'cathode'
            else:
                _mat_word = 'electrode'

            # --- Energy density vs cycle ---
            if PLOT_PER_ACTIVE_MASS:
                fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))
                for i, (name, pe_df) in enumerate(all_pe_data.items()):
                    comp = _get_display_name(
                        name, user_parameters.get(name, {}), user_parameters)
                    if 'energy_Wh_kg_active' in pe_df.columns:
                        ax.plot(pe_df['Cycle'],
                               pe_df['energy_Wh_kg_active'],
                               color=colours[i], marker=markers[i], markersize=4,
                               linewidth=1.2, label=comp)
                ax.set_xlabel('Cycle number', fontsize=13)
                _force_integer_cycles(ax)
                ax.set_ylabel(
                    'Specific energy / '
                    'Wh kg$^{-1}_{\\mathrm{active}}$',
                    fontsize=13)
                ax.set_title(
                    'Specific energy vs. cycle (active material basis)',
                    fontsize=13)
                _apply_pub_style(ax)
                ax.legend(fontsize=9, framealpha=0.7)
                _set_lim_with_padding(ax, 'y', bottom=0)
                plt.tight_layout()
                if save_location:
                    fpath = os.path.join(save_location,
                        f'energy_per_active_mass_vs_cycle.{_run_image_format(user_parameters)}')
                    fig.savefig(fpath, dpi=300, bbox_inches='tight')
                    saved(fpath)
                plt.show()
                plt.close(fig)

            if PLOT_PER_ELECTRODE_MASS:
                fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))
                for i, (name, pe_df) in enumerate(all_pe_data.items()):
                    comp = _get_display_name(
                        name, user_parameters.get(name, {}), user_parameters)
                    if 'energy_Wh_kg_electrode' in pe_df.columns:
                        ax.plot(pe_df['Cycle'],
                               pe_df['energy_Wh_kg_electrode'],
                               color=colours[i], marker=markers[i], markersize=4,
                               linewidth=1.2, label=comp)
                ax.set_xlabel('Cycle number', fontsize=13)
                _force_integer_cycles(ax)
                ax.set_ylabel(
                    'Specific energy / '
                    'Wh kg$^{-1}_{\\mathrm{electrode}}$',
                    fontsize=13)
                ax.set_title(
                    'Specific energy vs. cycle (electrode coating basis)',
                    fontsize=13)
                _apply_pub_style(ax)
                ax.legend(fontsize=9, framealpha=0.7)
                _set_lim_with_padding(ax, 'y', bottom=0)
                plt.tight_layout()
                if save_location:
                    fpath = os.path.join(save_location,
                        f'energy_per_electrode_mass_vs_cycle.{_run_image_format(user_parameters)}')
                    fig.savefig(fpath, dpi=300, bbox_inches='tight')
                    saved(fpath)
                plt.show()
                plt.close(fig)

            if PLOT_PER_AREA:
                fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))
                for i, (name, pe_df) in enumerate(all_pe_data.items()):
                    comp = _get_display_name(
                        name, user_parameters.get(name, {}), user_parameters)
                    if 'energy_mWh_cm2' in pe_df.columns:
                        ax.plot(pe_df['Cycle'], pe_df['energy_mWh_cm2'],
                               color=colours[i], marker=markers[i], markersize=4,
                               linewidth=1.2, label=comp)
                ax.set_xlabel('Cycle number', fontsize=13)
                _force_integer_cycles(ax)
                ax.set_ylabel('Areal energy density / mWh cm$^{-2}$',
                              fontsize=13)
                ax.set_title('Areal energy density vs. cycle', fontsize=13)
                _apply_pub_style(ax)
                ax.legend(fontsize=9, framealpha=0.7)
                _set_lim_with_padding(ax, 'y', bottom=0)
                plt.tight_layout()
                if save_location:
                    fpath = os.path.join(save_location,
                        f'energy_per_area_vs_cycle.{_run_image_format(user_parameters)}')
                    fig.savefig(fpath, dpi=300, bbox_inches='tight')
                    saved(fpath)
                plt.show()
                plt.close(fig)

            # --- Power density vs cycle ---
            power_col_map = {
                'active': ('power_mW_g_active',
                           'mW g$^{-1}_{\\mathrm{active}}$',
                           'power_per_active_mass'),
                'electrode': ('power_mW_g_electrode',
                             'mW g$^{-1}_{\\mathrm{electrode}}$',
                             'power_per_electrode_mass'),
                'area': ('power_mW_cm2', 'mW cm$^{-2}$',
                         'power_per_area')
            }

            for basis, plot_flag in [('active', PLOT_PER_ACTIVE_MASS),
                                      ('electrode', PLOT_PER_ELECTRODE_MASS),
                                      ('area', PLOT_PER_AREA)]:
                if not plot_flag:
                    continue

                col, ylabel, fname = power_col_map[basis]

                fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))
                for i, (name, pe_df) in enumerate(all_pe_data.items()):
                    comp = _get_display_name(
                        name, user_parameters.get(name, {}), user_parameters)
                    if col in pe_df.columns:
                        ax.plot(pe_df['Cycle'], pe_df[col],
                               color=colours[i], marker=markers[i], markersize=4,
                               linewidth=1.2, label=comp)

                ax.set_xlabel('Cycle number', fontsize=13)
                _force_integer_cycles(ax)
                ax.set_ylabel(f'Maximum power / {ylabel}', fontsize=13)
                basis_label = {'active': 'active material',
                              'electrode': 'electrode coating',
                              'area': 'electrode area'}[basis]
                ax.set_title(
                    f'Maximum power vs. cycle ({basis_label} basis)',
                    fontsize=13)
                _apply_pub_style(ax)
                ax.legend(fontsize=9, framealpha=0.7)
                _set_lim_with_padding(ax, 'y', bottom=0)
                plt.tight_layout()
                if save_location:
                    fpath = os.path.join(save_location,
                        f'{fname}_vs_cycle.{_run_image_format(user_parameters)}')
                    fig.savefig(fpath, dpi=300, bbox_inches='tight')
                    saved(fpath)
                plt.show()
                plt.close(fig)

            # --- Ragone plot ---
            if PLOT_RAGONE:
                energy_cols = {
                    'active': 'energy_Wh_kg_active',
                    'electrode': 'energy_Wh_kg_electrode',
                    'area': 'energy_mWh_cm2'
                }
                power_cols = {
                    'active': 'power_mW_g_active',
                    'electrode': 'power_mW_g_electrode',
                    'area': 'power_mW_cm2'
                }
                energy_labels = {
                    'active': 'Specific energy / '
                              'Wh kg$^{-1}_{\\mathrm{active}}$',
                    'electrode': 'Specific energy / '
                                 'Wh kg$^{-1}_{\\mathrm{electrode}}$',
                    'area': 'Areal energy / mWh cm$^{-2}$'
                }
                power_labels = {
                    'active': 'Power / '
                              'mW g$^{-1}_{\\mathrm{active}}$',
                    'electrode': 'Power / '
                                 'mW g$^{-1}_{\\mathrm{electrode}}$',
                    'area': 'Power / mW cm$^{-2}$'
                }

                e_col = energy_cols[RAGONE_BASIS]
                p_col = power_cols[RAGONE_BASIS]

                fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))
                for i, (name, pe_df) in enumerate(all_pe_data.items()):
                    comp = _get_display_name(
                        name, user_parameters.get(name, {}), user_parameters)
                    key_cycles = user_parameters.get(name, {}).get(
                        'key_cycles', [])

                    if (e_col not in pe_df.columns or
                        p_col not in pe_df.columns):
                        continue

                    valid = pe_df.dropna(subset=[e_col, p_col])
                    if valid.empty:
                        continue

                    ax.scatter(valid[e_col], valid[p_col],
                              color=colours[i], s=40, alpha=0.6,
                              label=comp)

                    sorted_v = valid.sort_values('Cycle')
                    ax.plot(sorted_v[e_col], sorted_v[p_col],
                           color=colours[i], linewidth=0.8, alpha=0.4)

                    for cycle in key_cycles:
                        cd = valid[valid['Cycle'] == cycle]
                        if not cd.empty:
                            ax.annotate(
                                str(int(cycle)),
                                (cd[e_col].iloc[0], cd[p_col].iloc[0]),
                                textcoords='offset points', xytext=(6, 6),
                                fontsize=8, fontweight='bold',
                                color=colours[i], alpha=0.8
                            )

                ax.set_xlabel(energy_labels[RAGONE_BASIS], fontsize=13)
                ax.set_ylabel(power_labels[RAGONE_BASIS], fontsize=13)
                basis_note = {'active': 'active material',
                             'electrode': 'electrode coating',
                             'area': 'electrode area'}[RAGONE_BASIS]
                ax.set_title(
                    f'Ragone plot -- half-cell ({basis_note} basis)',
                    fontsize=13)
                _apply_pub_style(ax)
                ax.legend(fontsize=9, framealpha=0.7)
                _set_lim_with_padding(ax, 'x', bottom=0)
                _set_lim_with_padding(ax, 'y', bottom=0)
                plt.tight_layout()
                if save_location:
                    fpath = os.path.join(
                        save_location,
                        f'ragone_{RAGONE_BASIS}_basis.{_run_image_format(user_parameters)}')
                    fig.savefig(fpath, dpi=300, bbox_inches='tight')
                    saved(fpath)
                plt.show()
                plt.close(fig)

            # --- Areal capacity vs cycle (v1.7.2) ---
            if PLOT_AREAL_CAPACITY:
                # Calculate areal discharge capacity for each dataset
                all_areal_cap = {}
                for name, pe_df in all_pe_data.items():
                    params = user_parameters.get(name, {})
                    active_mass_mg = params.get('active_material_mass_mg')
                    area_cm2 = params.get('electrode_area_cm2')
                    if active_mass_mg is None or area_cm2 is None:
                        continue
                    active_mass_g = active_mass_mg / 1000.0

                    # Get discharge capacity per cycle from cycle tables
                    if (_have_cycle_tables and name in all_cycle_tables):
                        ct = all_cycle_tables[name]
                        ct_complete = ct[~_unusable(ct)]
                        areal_df = ct_complete[['Cycle', 'Discharge_mAh_g']].copy()
                        areal_df = areal_df.dropna(subset=['Discharge_mAh_g'])
                        # Apply visual outlier filter (same as Cells 8/9/10)
                        if VISUAL_OUTLIER_FILTER and not areal_df.empty:
                            _outlier_flags = _flag_visual_outliers(
                                areal_df['Discharge_mAh_g'].tolist(), threshold=VISUAL_OUTLIER_THRESHOLD,
                                groups=rate_block_labels(areal_df['Cycle'], params))
                            areal_df = areal_df[
                                ~pd.Series(_outlier_flags,
                                           index=areal_df.index)
                            ].reset_index(drop=True)
                        # Convert: mAh/g * g / cm2 = mAh/cm2
                        areal_df['Areal_mAh_cm2'] = (
                            areal_df['Discharge_mAh_g'] * active_mass_g
                            / area_cm2)
                        all_areal_cap[name] = areal_df
                    else:
                        # NO FALLBACK. This branch used to back-calculate
                        # capacity from specific energy divided by
                        # `voltage_upper_V * 0.85` — a hardcoded guess at the
                        # mean discharge voltage, plotted on an axis labelled
                        # mAh/cm2 with nothing to say it was not measured. On
                        # a 4.3 V layered oxide the guess is 3.66 V against a
                        # real mean near 3.8 V, a 4% error in every point.
                        # Areal capacity comes from the canonical cycle table
                        # or it is not plotted.
                        print(f"  {name}: no cycle table, so areal capacity "
                              f"is not plotted (it will not be estimated "
                              f"from energy and an assumed voltage).")

                if all_areal_cap:
                    fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))
                    for i, (name, areal_df) in enumerate(
                            all_areal_cap.items()):
                        comp = _get_display_name(
                            name, user_parameters.get(name, {}),
                            user_parameters)
                        loading = user_parameters.get(name, {}).get(
                            'active_loading_mg_cm2')
                        loading_str = (f" [{loading:.1f} mg/cm²]"
                                      if loading else "")
                        ax.plot(areal_df['Cycle'],
                               areal_df['Areal_mAh_cm2'],
                               color=colours[i], marker=markers[i], markersize=4,
                               linewidth=1.2,
                               label=f"{comp}{loading_str}")

                    # Chemistry-aware benchmark lines (v1.7.2)
                    _chemistries = set(
                        user_parameters.get(n, {}).get('battery_chemistry', 'Unknown')
                        for n in all_areal_cap)
                    _chem = (_chemistries.pop() if len(_chemistries) == 1
                             else 'Unknown')
                    _bench_vals, _bench_label = _AREAL_BENCHMARKS.get(
                        _chem, _AREAL_BENCHMARK_DEFAULT)
                    _bench_annotations = []
                    for bench in _bench_vals:
                        ax.axhline(y=bench, color='grey', linestyle='--',
                                  linewidth=0.8, alpha=0.6)
                        _bench_annotations.append(bench)

                    ax.set_xlabel('Cycle number', fontsize=13)
                    _force_integer_cycles(ax)
                    ax.set_ylabel(
                        'Areal discharge capacity / mAh cm$^{-2}$',
                        fontsize=13)
                    ax.set_title(
                        'Areal discharge capacity vs. cycle',
                        fontsize=13)
                    _apply_pub_style(ax)
                    ax.legend(fontsize=9, framealpha=0.7)
                    ax.set_ylim(bottom=0)
                    _set_lim_with_padding(ax, 'y', bottom=0)
                    # Add benchmark annotations after axis limits are final
                    for bench in _bench_annotations:
                        ax.annotate(
                            f'{bench:.0f} mAh/cm² ({_bench_label})',
                            xy=(ax.get_xlim()[1] * 0.02, bench),
                            xytext=(5, 3), textcoords='offset points',
                            fontsize=8, color='grey', alpha=0.8,
                            fontstyle='italic')
                    plt.tight_layout()

                    # Caption
                    loading_range = [
                        user_parameters.get(n, {}).get(
                            'active_loading_mg_cm2')
                        for n in all_areal_cap
                    ]
                    loading_range = [l for l in loading_range
                                    if l is not None]
                    if loading_range:
                        lo_str = (f"Active material loadings: "
                                 f"{min(loading_range):.1f}--"
                                 f"{max(loading_range):.1f} mg/cm².")
                    else:
                        lo_str = ""
                    caption = (
                        f"Figure X. Areal discharge capacity for all "
                        f"datasets, normalised to electrode geometric "
                        f"area. Dashed grey lines indicate the "
                        f"{_bench_label} range "
                        f"({_bench_vals[0]:.0f}--"
                        f"{_bench_vals[-1]:.0f} "
                        f"mAh/cm²). {lo_str} "
                        f"Half-cell ({_mat_word}) data."
                    )
                    print(section("  Suggested caption"))
                    print(bullet(caption, indent=2, label_width=2))

                    if save_location:
                        fpath = os.path.join(save_location,
                            f'areal_discharge_capacity_vs_cycle.{_run_image_format(user_parameters)}')
                        fig.savefig(fpath, dpi=300, bbox_inches='tight')
                        saved(fpath)
                    plt.show()
                    plt.close(fig)

                    # Print areal capacity summary
                    print(f"\n  Areal capacity summary (cycle 2):")
                    print(f"  {'Composition':<30} {'Loading':>10} "
                          f"{'Q_areal':>10}")
                    print(f"  {'':30} {'mg/cm²':>10} "
                          f"{'mAh/cm²':>10}")
                    print(f"  {'-'*52}")
                    for name, areal_df in all_areal_cap.items():
                        comp = _get_display_name(
                            name, user_parameters.get(name, {}),
                            user_parameters)
                        loading = user_parameters.get(name, {}).get(
                            'active_loading_mg_cm2')
                        c2 = areal_df[areal_df['Cycle'] == 2]
                        if not c2.empty:
                            q_ar = c2['Areal_mAh_cm2'].iloc[0]
                            l_str = (f"{loading:.2f}"
                                    if loading else '--')
                            print(f"  {comp:<30} {l_str:>10} "
                                  f"{q_ar:>10.3f}")
                    print(f"  {'-'*52}")
                else:
                    print("  Areal capacity: insufficient data "
                          "(need active mass and electrode area).")

            # --- Summary table ---
            print()
            print(f"  Power, Energy & Areal Capacity Summary (final cycle)")
            print(f"  {'Composition':<28} {'E/Wh kg':>10} "
                  f"{'E/Wh kg':>10} {'E/mWh cm2':>11} "
                  f"{'P/mW g':>10} {'P/mW cm2':>10} {'Q_areal':>10} {'Cyc':>5}")
            print(f"  {'':28} {'(active)':>10} {'(electr.)':>10} "
                  f"{'(areal)':>11} {'(active)':>10} {'(areal)':>10} "
                  f"{'mAh/cm2':>10} {'':>5}")
            print(f"  {'-'*95}")
            for name, pe_df in all_pe_data.items():
                comp = _get_display_name(
                    name, user_parameters.get(name, {}), user_parameters)
                last = pe_df.iloc[-1]
                def _fmt(val, fmt='.1f'):
                    return f"{val:{fmt}}" if pd.notna(val) else '--'
                e_act = _fmt(last.get('energy_Wh_kg_active'))
                e_elec = _fmt(last.get('energy_Wh_kg_electrode'))
                e_area = _fmt(last.get('energy_mWh_cm2'), '.3f')
                p_act = _fmt(last.get('power_mW_g_active'))
                p_area = _fmt(last.get('power_mW_cm2'), '.3f')
                # Areal capacity from final cycle
                q_areal = '--'
                if (PLOT_AREAL_CAPACITY and 'all_areal_cap' in dir()
                    and name in all_areal_cap):
                    ac_df = all_areal_cap[name]
                    last_ac = ac_df.iloc[-1] if not ac_df.empty else None
                    if last_ac is not None:
                        q_areal = f"{last_ac['Areal_mAh_cm2']:.3f}"
                # Final cycle number
                last_cyc = str(int(last['Cycle'])) if 'Cycle' in last else '--'
                print(f"  {comp:<28} {e_act:>10} {e_elec:>10} "
                      f"{e_area:>11} {p_act:>10} {p_area:>10} "
                      f"{q_areal:>10} {last_cyc:>5}")
            print(f"  {'-'*95}")
            print(f"  Note: Half-cell data (vs metal counter electrode).")
            print(f"  Energy and power reflect {_mat_word} material only.")

            # --- Export ---
            for name, pe_df in all_pe_data.items():
                if save_location:
                    fpath = os.path.join(
                        save_location,
                        f'{name}_power_energy_data.csv')
                    pe_df.to_csv(fpath, index=False)
                    saved(fpath)

        print(rule())


@_honours_verbose
def average_discharge_voltage(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Mean discharge voltage against cycle number.

    Ported from 1.8.7 Cell 11, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    MARKER_SIZE = 6

    # The filter default, read from the one definition rather than

    # restated: six local copies were quoted in the console message

    # while `_flag_visual_outliers` used its own default, so editing

    # one changed the sentence and not the data.

    VISUAL_OUTLIER_THRESHOLD = VISUAL_OUTLIER_FRACTION
    def _calculate_avg_voltage_per_cycle(df):
        """
        Energy-weighted average discharge voltage per cycle:
            V_avg = integral(V * dQ) / integral(dQ)

        Integration is `compat.trapezoid`, which is numpy's under whichever
        name that version of numpy uses. NOT scipy, whatever this docstring
        said until 1.9.0.15.
        """
        # Cell 2 has already coerced the numeric columns.
        discharge = df[df['Step'] == 'Discharge'].dropna(
            subset=['Cycle', 'Voltage', 'Discharge_Capacity'])

        if discharge.empty:
            return pd.DataFrame(columns=['Cycle', 'Avg_Discharge_Voltage_V',
                                          'Discharge_mAh_g'])

        # ONE DEFINITION. `mean_discharge_voltage` at the top of this module
        # was written to be the single definition of this quantity — its
        # docstring says so, and names the two divergent copies that preceded
        # it — and then nothing ever called it, so this function kept a third.
        # The two agreed, which is exactly why it survived: a duplicate that
        # matches is invisible until someone edits one of them. This body is
        # now the module-level function plus the delivered capacity, which is
        # the only thing the figure needs that the canonical one does not
        # return.
        base = mean_discharge_voltage(
            discharge.assign(Step='Discharge'))
        if base.empty:
            return pd.DataFrame(columns=['Cycle', 'Avg_Discharge_Voltage_V',
                                         'Discharge_mAh_g'])
        deliv = {}
        for cycle, cd in discharge.groupby('Cycle'):
            q = pd.to_numeric(cd['Discharge_Capacity'],
                              errors='coerce').dropna()
            if q.size < 3:
                continue
            span = float(q.max() - q.min())
            if span > 0:
                deliv[int(cycle)] = span
        base = base[base['Cycle'].isin(deliv)].copy()
        base['Discharge_mAh_g'] = base['Cycle'].map(deliv)
        return base.reset_index(drop=True)

    _avg_v_labels = set(
        _discharge_label(user_parameters.get(n, {}))
        for n in electrochemical_data)

    _avg_v_word = (_avg_v_labels.pop().lower() if len(_avg_v_labels) == 1
                   else 'discharge')

    print(rule(f"AVERAGE {_avg_v_word.upper()} VOLTAGE"))

    palette = MULTI_DATASET_PALETTE

    _have_cycle_tables = _has_tables(all_cycle_tables)

    if EXCLUDE_INCOMPLETE and not _have_cycle_tables:
        print("  ⚠ all_cycle_tables not found — Cell 5b has not been "
              "run. Protocol-incomplete filtering unavailable; "
              "visual-outlier filter still active.\n")

    all_avg_voltage = {}

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)

        avg_v_df = _calculate_avg_voltage_per_cycle(df)

        if avg_v_df.empty:
            print(f"  {composition}: no valid {_avg_v_word} data, skipping.")
            continue

        # Exclude protocol-incomplete cycles (Cell 5b flag)
        if (EXCLUDE_INCOMPLETE and _have_cycle_tables
            and name in all_cycle_tables):
            ct = all_cycle_tables[name]
            incomplete_cycles = set(
                _unusable_cycles(ct))
            if incomplete_cycles:
                mask = avg_v_df['Cycle'].isin(incomplete_cycles)
                removed = avg_v_df.loc[mask, 'Cycle'].tolist()
                if removed:
                    print(f"  {composition}: excluded {len(removed)} "
                          f"protocol-incomplete cycle(s): "
                          f"{', '.join(str(int(c)) for c in removed)}")
                avg_v_df = avg_v_df[~mask].reset_index(drop=True)

        # Visual-outlier filter
        if VISUAL_OUTLIER_FILTER and not avg_v_df.empty:
            outlier_flags = _flag_visual_outliers(
                avg_v_df['Discharge_mAh_g'].tolist(), threshold=VISUAL_OUTLIER_THRESHOLD,
                groups=rate_block_labels(avg_v_df['Cycle'], params))
            n_outliers = sum(outlier_flags)
            if n_outliers > 0:
                removed = [int(c) for c, f in
                           zip(avg_v_df['Cycle'], outlier_flags) if f]
                print(f"  {composition}: excluded {n_outliers} visual "
                      f"outlier(s) (<{VISUAL_OUTLIER_THRESHOLD*100:.0f}% "
                      f"of running median on Discharge_mAh_g): "
                      f"{', '.join(str(c) for c in removed)}")
                avg_v_df = avg_v_df[
                    ~pd.Series(outlier_flags, index=avg_v_df.index)
                ].reset_index(drop=True)

        if avg_v_df.empty:
            continue

        all_avg_voltage[name] = avg_v_df

        # Voltage drift summary
        ref_row = avg_v_df[avg_v_df['Cycle'] == DRIFT_REFERENCE_CYCLE]
        if not ref_row.empty:
            ref_v = ref_row['Avg_Discharge_Voltage_V'].iloc[0]
            last_v = avg_v_df['Avg_Discharge_Voltage_V'].iloc[-1]
            last_cycle = int(avg_v_df['Cycle'].iloc[-1])
            drift_mV = (last_v - ref_v) * 1000
            n_cycles = last_cycle - DRIFT_REFERENCE_CYCLE
            drift_rate = drift_mV / n_cycles if n_cycles > 0 else 0

            print(f"  {composition}: V_avg = {ref_v:.4f} V (cycle "
                  f"{DRIFT_REFERENCE_CYCLE}) -> {last_v:.4f} V (cycle "
                  f"{last_cycle})")
            print(f"    Total drift: {drift_mV:+.1f} mV over {n_cycles} "
                  f"cycles ({drift_rate:+.2f} mV/cycle)")
        else:
            print(f"  {composition}: {len(avg_v_df)} cycles analysed, "
                  f"V_avg range {avg_v_df['Avg_Discharge_Voltage_V'].min():.4f}"
                  f"--{avg_v_df['Avg_Discharge_Voltage_V'].max():.4f} V")

    if all_avg_voltage:
        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        caption_compositions = []

        for i, (name, avdf) in enumerate(all_avg_voltage.items()):
            composition = _get_display_name(name, user_parameters.get(name, {}), user_parameters)
            colour = palette[i % len(palette)]
            marker = MARKERS[i % len(MARKERS)]

            ax.plot(avdf['Cycle'], avdf['Avg_Discharge_Voltage_V'],
                   color=colour, marker=marker, markersize=MARKER_SIZE,
                   linestyle='-', linewidth=0.8, label=composition, zorder=3)

            caption_compositions.append(composition)

        ax.set_xlabel('Cycle number', fontsize=14)
        _force_integer_cycles(ax)
        ax.set_ylabel(f'Average {_avg_v_word} voltage / V', fontsize=14)
        ax.tick_params(axis='both', labelsize=12, width=1, direction='in',
                       top=True, right=True)
        for sp in ax.spines.values():
            sp.set_linewidth(0.8)

        # Y-axis scaling
        if VOLTAGE_YMIN is not None and VOLTAGE_YMAX is not None:
            ax.set_ylim(bottom=VOLTAGE_YMIN, top=VOLTAGE_YMAX)
        else:
            y_lo, y_hi = ax.get_ylim()
            ax.set_ylim(bottom=y_lo - VOLTAGE_YAXIS_PADDING,
                        top=y_hi + VOLTAGE_YAXIS_PADDING)

        ax.set_xlim(left=0)
        ax.legend(fontsize=10, framealpha=0.7)
        plt.tight_layout()

        # Caption
        comp_str = ', '.join(caption_compositions[:-1])
        if len(caption_compositions) > 1:
            comp_str += f' and {caption_compositions[-1]}'
        else:
            comp_str = caption_compositions[0]

        charge_rate = user_parameters.get(
            list(all_avg_voltage.keys())[0], {}
        ).get('charge_rate_c', 'the specified')

        caption = (
            f"Figure X. Energy-weighted average {_avg_v_word} voltage "
            f"as a function of cycle number for {comp_str}, cycled at "
            f"{charge_rate} C. A declining average voltage indicates "
            f"increasing polarisation or structural degradation, even "
            f"if {_avg_v_word} capacity is maintained."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

        # Save
        if save_location:
            filepath = os.path.join(save_location,
                                   f'average_discharge_voltage_vs_cycle.{_run_image_format(user_parameters)}')
            fig.savefig(filepath, dpi=300, bbox_inches='tight')
            saved(filepath)

        plt.show()
        plt.close(fig)

        # Export
        if save_location:
            for name, avdf in all_avg_voltage.items():
                fpath = os.path.join(save_location,
                                    f'{name}_avg_{_avg_v_word}_voltage.csv')
                avdf.to_csv(fpath, index=False)
                saved(fpath)
    else:
        print("\nNo valid data for average voltage analysis.")

    print(rule())


@_honours_verbose
def energy_efficiency(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Round-trip energy efficiency against cycle number.

    Ported from 1.8.7 Cell 12, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    MARKER_SIZE = 6

    YAXIS_MIN = 40

    YAXIS_MAX = 105

    # The filter default, read from the one definition rather than

    # restated: six local copies were quoted in the console message

    # while `_flag_visual_outliers` used its own default, so editing

    # one changed the sentence and not the data.

    VISUAL_OUTLIER_THRESHOLD = VISUAL_OUTLIER_FRACTION
    def _calculate_energy_per_cycle(df):
        """
        Per-cycle charge and discharge ENERGY, by trapezoidal integration of
        V dQ, plus the efficiencies derived from it.

        The CAPACITIES AND CE ARE NOT COMPUTED HERE. This function used to
        derive its own, with its own admission rule (three points, no
        integrity filter), so `energy_efficiency`'s CE could and did differ
        from `cycle_life`'s for the same cycle of the same cell with nothing
        in the run to say so. They now come from `delivered_capacity` and
        `coulombic_efficiency` — the one definition — and the mean voltages
        are energy / that capacity, which is the standard energy-weighted
        definition and now shares a denominator with everything else.

        Integration is `compat.trapezoid` — numpy's, under whichever name
        that version of numpy uses. NOT scipy, whatever this docstring said
        until 1.9.0.15.
        """
        # Cell 2 has already coerced the numeric columns; we only need to
        # drop rows with missing Cycle/Voltage/Step for the integration.
        df_work = df.dropna(subset=['Cycle', 'Voltage', 'Step'])

        rows = []
        for cycle in sorted(df_work['Cycle'].unique()):
            cd = df_work[df_work['Cycle'] == cycle]
            row = {'Cycle': int(cycle)}

            # --- Discharge energy ---
            dc = cd[
                cd['Step'] == 'Discharge'
            ].dropna(subset=['Discharge_Capacity', 'Voltage'])

            if len(dc) >= 3:
                dc_sorted = dc.sort_values('Discharge_Capacity')
                Q_d = dc_sorted['Discharge_Capacity'].values
                V_d = dc_sorted['Voltage'].values

                row['Discharge_Energy'] = abs(trapezoid(V_d, Q_d))
            else:
                row['Discharge_Energy'] = np.nan

            # --- Charge energy ---
            cc = cd[
                cd['Step'] == 'Charge'
            ].dropna(subset=['Charge_Capacity', 'Voltage'])

            if len(cc) >= 3:
                cc_sorted = cc.sort_values('Charge_Capacity')
                Q_c = cc_sorted['Charge_Capacity'].values
                V_c = cc_sorted['Voltage'].values

                row['Charge_Energy'] = abs(trapezoid(V_c, Q_c))
            else:
                row['Charge_Energy'] = np.nan

            if (pd.notna(row['Charge_Energy']) and
                pd.notna(row['Discharge_Energy']) and
                row['Charge_Energy'] > 0):
                row['Energy_Efficiency_%'] = (
                    row['Discharge_Energy'] / row['Charge_Energy'] * 100
                )
            else:
                row['Energy_Efficiency_%'] = np.nan

            rows.append(row)

        out = pd.DataFrame(rows)
        if out.empty:
            return out

        # --- capacity and CE from the ONE definition ---------------------
        cap = delivered_capacity(df)
        out = out.merge(cap, on='Cycle', how='left')
        for c in ('Charge_mAh_g', 'Discharge_mAh_g'):
            if c not in out.columns:
                out[c] = np.nan
        out['CE_%'] = coulombic_efficiency(out['Discharge_mAh_g'],
                                           out['Charge_mAh_g']).values

        # --- mean voltages: energy / that same capacity ------------------
        # Voltage efficiency decomposes the energy loss: EE ~ CE * VE / 100.
        # CE tells you about material loss, VE about polarisation.
        # Ref: Cao et al., Nat. Nanotechnol. 14, 200-207 (2019)
        qc = pd.to_numeric(out['Charge_mAh_g'], errors='coerce')
        qd = pd.to_numeric(out['Discharge_mAh_g'], errors='coerce')
        out['Avg_V_Charge'] = (out['Charge_Energy'] / qc).where(qc > 0)
        out['Avg_Discharge_Voltage_V'] = (
            out['Discharge_Energy'] / qd).where(qd > 0)
        out['Voltage_Efficiency_%'] = (
            out['Avg_Discharge_Voltage_V'] / out['Avg_V_Charge'] * 100
        ).where(out['Avg_V_Charge'] > 0)

        out['CE_EE_Gap_%'] = out['CE_%'] - out['Energy_Efficiency_%']
        return out

    print(rule("ENERGY EFFICIENCY ANALYSIS"))

    palette_multi = MULTI_DATASET_PALETTE

    markers_multi = MULTI_DATASET_MARKERS

    _have_cycle_tables = _has_tables(all_cycle_tables)

    if EXCLUDE_INCOMPLETE and not _have_cycle_tables:
        print("  ⚠ all_cycle_tables not found — Cell 5b has not been "
              "run. Protocol-incomplete filtering unavailable; "
              "visual-outlier filter still active.")

    all_ee_data = {}

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        # 1.8.7 Cell 12 read `is_anode` here (in its >100% EE notice) but
        # only ASSIGNED it further down the same loop. On the first dataset
        # it therefore fell through to whatever Cell 11 had left in globals,
        # and on every later one it carried the PREVIOUS dataset's flag.
        # Binding it at the top of the loop is what the later line does, one
        # iteration too late.
        is_anode = params.get('anode_labels_swapped', False)

        ee_df = _calculate_energy_per_cycle(df)

        if ee_df.empty:
            print(f"  {composition}: no valid data, skipping.")
            continue

        # Exclude protocol-incomplete cycles (Cell 5b flag)
        if (EXCLUDE_INCOMPLETE and _have_cycle_tables
            and name in all_cycle_tables):
            ct = all_cycle_tables[name]
            incomplete_cycles = set(
                _unusable_cycles(ct))
            if incomplete_cycles:
                mask = ee_df['Cycle'].isin(incomplete_cycles)
                removed = ee_df.loc[mask, 'Cycle'].tolist()
                if removed:
                    print(f"  {composition}: excluded {len(removed)} "
                          f"protocol-incomplete cycle(s): "
                          f"{', '.join(str(int(c)) for c in removed)}")
                ee_df = ee_df[~mask].reset_index(drop=True)

        # Visual-outlier filter
        if VISUAL_OUTLIER_FILTER and not ee_df.empty:
            outlier_flags = _flag_visual_outliers(
                ee_df['Discharge_mAh_g'].tolist(), threshold=VISUAL_OUTLIER_THRESHOLD,
                groups=rate_block_labels(ee_df['Cycle'], params))
            n_outliers = sum(outlier_flags)
            if n_outliers > 0:
                removed = [int(c) for c, f in
                           zip(ee_df['Cycle'], outlier_flags) if f]
                print(f"  {composition}: excluded {n_outliers} visual "
                      f"outlier(s) (<{VISUAL_OUTLIER_THRESHOLD*100:.0f}% "
                      f"of running median on Discharge_mAh_g): "
                      f"{', '.join(str(c) for c in removed)}")
                ee_df = ee_df[
                    ~pd.Series(outlier_flags, index=ee_df.index)
                ].reset_index(drop=True)

        if ee_df.empty:
            continue

        # Flag any EE > 100% (physically suspicious)
        over_100 = ee_df[ee_df['Energy_Efficiency_%'] > 100]
        if not over_100.empty:
            n_over = len(over_100)
            n_total = len(ee_df)
            if n_over == n_total:
                print(f"  {composition}: EE >100% in all {n_total} cycles"
                      f" (expected for anode half-cells)"
                      if is_anode else
                      f"  {composition}: EE >100% in all {n_total} cycles"
                      f" (check for measurement artefact)")
            elif n_over > 5:
                note = ("expected for anode half-cells" if is_anode
                        else "likely formation effect")
                print(f"  {composition}: EE >100% in {n_over}/{n_total} "
                      f"cycles ({note})")
            else:
                print(f"  {composition}: EE >100% in cycle(s) "
                      f"{', '.join(str(int(c)) for c in over_100['Cycle'])} "
                      f"(likely formation effect)")

        all_ee_data[name] = ee_df

        # Summary (excluding formation)
        stable = ee_df[ee_df['Cycle'] >= EE_START_CYCLE]
        if not stable.empty:
            mean_ee = stable['Energy_Efficiency_%'].mean()
            std_ee = stable['Energy_Efficiency_%'].std()
            mean_ce = stable['CE_%'].mean()
            mean_gap = stable['CE_EE_Gap_%'].mean()

            print()
            print(heading(composition))
            print(f"    Energy efficiency (cycles {EE_START_CYCLE}+): "
                  f"{mean_ee:.2f}% (+/-{std_ee:.2f}%)")
            if is_anode and mean_ee > 100:
                print(f"    Note: EE >100% is expected for anode half-cells — "
                      f"delithiation (discharge) voltage is higher than "
                      f"lithiation (charge) voltage, so E_out > E_in.")
                print(f"    This is not an error; the energy balance is closed "
                      f"by the Li/Na metal counter electrode.")
            print(f"    Coulombic efficiency (cycles {EE_START_CYCLE}+): "
                  f"{mean_ce:.2f}%")
            is_anode = params.get('anode_labels_swapped', False)
            if 'Voltage_Efficiency_%' in stable.columns:
                mean_ve = stable['Voltage_Efficiency_%'].mean()
                if is_anode:
                    abs_hysteresis = abs(100 - mean_ve)
                    print(f"    Voltage ratio (cycles {EE_START_CYCLE}+): "
                          f"{mean_ve:.2f}%")
                    print(f"    Voltage hysteresis:   "
                          f"|1 - V_ratio| = {abs_hysteresis:.2f}%")
                    print(f"    Note: For anodes, V_ratio > 100% is expected "
                          f"(the useful half-cycle occurs at higher voltage).")
                    print(f"    The deviation from 100% reflects the same "
                          f"polarisation loss as in cathodes.")
                else:
                    print(f"    Voltage efficiency (cycles {EE_START_CYCLE}+): "
                          f"{mean_ve:.2f}%")
                    print(f"    Loss decomposition:  "
                          f"CE loss = {100 - mean_ce:.2f}% (material)  |  "
                          f"VE loss = {100 - mean_ve:.2f}% (polarisation)")
            abs_gap = abs(mean_gap)
            gap_sign = '' if mean_gap >= 0 else ' (note: negative for anodes)'
            print(f"    |CE - EE| gap: {abs_gap:.2f}% "
                  f"(voltage hysteresis contribution){gap_sign}")

    if all_ee_data:
        for name, ee_df in all_ee_data.items():
            params = user_parameters.get(name, {})
            composition = _get_display_name(name, params, user_parameters)
            charge_rate_c = params.get('charge_rate_c', 'Unknown')

            # Filter to stable cycling (skip formation)
            plot_df = ee_df[ee_df['Cycle'] >= EE_START_CYCLE]

            if plot_df.empty:
                continue

            fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

            # CE
            ax.plot(plot_df['Cycle'], plot_df['CE_%'],
                   color=COLOUR_CE, marker=MARKERS_CE,
                   markersize=MARKER_SIZE, linestyle='-',
                   linewidth=0.8, label='Coulombic efficiency', zorder=3)

            # Energy efficiency
            ax.plot(plot_df['Cycle'], plot_df['Energy_Efficiency_%'],
                   color=COLOUR_EE, marker=MARKERS_EE,
                   markersize=MARKER_SIZE, linestyle='-',
                   linewidth=0.8, label='Energy efficiency', zorder=3)

            # Voltage efficiency / ratio (v1.8: adaptive label for anodes)
            is_anode = params.get('anode_labels_swapped', False)
            if 'Voltage_Efficiency_%' in plot_df.columns:
                _ve_legend = ('Voltage ratio' if is_anode
                              else 'Voltage efficiency')
                ax.plot(plot_df['Cycle'], plot_df['Voltage_Efficiency_%'],
                       color='#009E73', marker='s',
                       markersize=MARKER_SIZE - 1, linestyle='-',
                       linewidth=0.8, label=_ve_legend,
                       zorder=3, alpha=0.8)

            # Shade the gap
            ax.fill_between(
                plot_df['Cycle'],
                plot_df['Energy_Efficiency_%'],
                plot_df['CE_%'],
                alpha=0.10, color='grey',
                label='Voltage hysteresis loss'
            )

            # 100% reference line
            ax.axhline(y=100, color='grey', linestyle=':',
                        linewidth=0.8, alpha=0.5)

            ax.set_xlabel('Cycle number', fontsize=14)
            _force_integer_cycles(ax)
            ax.set_ylabel('Efficiency / %', fontsize=14)
            ax.tick_params(axis='both', labelsize=12, width=1,
                           direction='in', top=True, right=True)
            for sp in ax.spines.values():
                sp.set_linewidth(0.8)

            # Y-axis (v1.8: extend for anode VE > 100%)
            y_lo = min(YAXIS_MIN,
                       plot_df['Energy_Efficiency_%'].min() - 2)
            y_hi = max(YAXIS_MAX,
                       plot_df['CE_%'].max() + 2)
            if ('Voltage_Efficiency_%' in plot_df.columns
                    and plot_df['Voltage_Efficiency_%'].notna().any()):
                y_hi = max(y_hi,
                           plot_df['Voltage_Efficiency_%'].max() + 5)
            # A KNOB THAT DID NOTHING. `COMMON_YAXIS_EFFICIENCY` sat beside
            # `COMMON_YAXIS_CAPACITY`, which works, and was read nowhere — so
            # a user turning it off got the fixed floor and ceiling anyway.
            # Off now means the axis follows the data, which is what its
            # neighbour's name promises.
            if COMMON_YAXIS_EFFICIENCY:
                ax.set_ylim(bottom=y_lo, top=y_hi)
            ax.set_xlim(left=0)

            ax.legend(fontsize=10, framealpha=0.7)
            plt.tight_layout()

            # Caption (v1.8: absolute gap for anodes)
            mean_gap = plot_df['CE_EE_Gap_%'].mean()
            abs_gap = abs(mean_gap)
            chg_word = _charge_label(params).lower()
            caption = (
                f"Figure X. Coulombic efficiency (blue circles) and "
                f"energy efficiency (vermillion diamonds) for "
                f"{composition} cycled at {rate_phrase(params)}. "
                f"The shaded region represents the energy lost to "
                f"voltage hysteresis, averaging {abs_gap:.1f}% of "
                f"the total {chg_word} energy. Formation cycle excluded."
            )
            print(section("  Suggested caption"))
            print(bullet(caption, indent=2, label_width=2))

            # Save
            if save_location:
                filepath = os.path.join(
                    save_location,
                    f'{name}_energy_efficiency.{image_format(params)}')
                fig.savefig(filepath, dpi=300, bbox_inches='tight')
                saved(filepath)

            plt.show()
            plt.close(fig)


        # --- Comparative plot (if multiple datasets) ---
        if len(all_ee_data) > 1:
            fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

            caption_compositions = []

            for i, (name, ee_df) in enumerate(all_ee_data.items()):
                composition = _get_display_name(
                    name, user_parameters.get(name, {}), user_parameters)
                colour = palette_multi[i % len(palette_multi)]
                marker = markers_multi[i % len(markers_multi)]

                stable = ee_df[ee_df['Cycle'] >= EE_START_CYCLE]
                if stable.empty:
                    continue

                ax.plot(stable['Cycle'], stable['Energy_Efficiency_%'],
                       color=colour, marker=marker,
                       markersize=MARKER_SIZE, linestyle='-',
                       linewidth=0.8, label=composition, zorder=3)

                caption_compositions.append(composition)

            ax.axhline(y=100, color='grey', linestyle=':',
                        linewidth=0.8, alpha=0.5)

            ax.set_xlabel('Cycle number', fontsize=14)
            _force_integer_cycles(ax)
            ax.set_ylabel('Energy efficiency / %', fontsize=14)
            ax.tick_params(axis='both', labelsize=12, width=1,
                           direction='in', top=True, right=True)
            for sp in ax.spines.values():
                sp.set_linewidth(0.8)
            ax.set_xlim(left=0)
            # Auto-extend y-axis for anode data where EE > 100%
            _all_ee_vals = []
            for _ee_df in all_ee_data.values():
                _s = _ee_df[_ee_df['Cycle'] >= EE_START_CYCLE]
                if not _s.empty:
                    _all_ee_vals.extend(_s['Energy_Efficiency_%'].dropna().tolist())
            if _all_ee_vals:
                y_hi = max(YAXIS_MAX, max(_all_ee_vals) + 5)
                y_lo = min(YAXIS_MIN, min(_all_ee_vals) - 2)
            else:
                y_hi = YAXIS_MAX
                y_lo = YAXIS_MIN
            ax.set_ylim(bottom=y_lo, top=y_hi)
            ax.legend(fontsize=10, framealpha=0.7)
            plt.tight_layout()

            comp_str = ', '.join(caption_compositions[:-1])
            if len(caption_compositions) > 1:
                comp_str += f' and {caption_compositions[-1]}'
            else:
                comp_str = caption_compositions[0]

            # Check if any dataset is an anode
            _any_anode = any(
                user_parameters.get(n, {}).get('anode_labels_swapped', False)
                for n in all_ee_data)
            _anode_note = (" Note: anode half-cell EE exceeds 100% because "
                           "the useful half-cycle occurs at higher voltage "
                           "than the insertion half-cycle."
                           if _any_anode else "")
            caption = (
                f"Figure X. Comparative energy efficiency for "
                f"{comp_str}. Formation cycle excluded.{_anode_note}"
            )
            print(section("  Suggested caption"))
            print(bullet(caption, indent=2, label_width=2))

            if save_location:
                filepath = os.path.join(save_location,
                                       f'comparative_energy_efficiency.{_run_image_format(user_parameters)}')
                fig.savefig(filepath, dpi=300, bbox_inches='tight')
                saved(filepath)

            plt.show()
            plt.close(fig)


        # --- Export ---
        if save_location:
            for name, ee_df in all_ee_data.items():
                fpath = os.path.join(save_location,
                                    f'{name}_energy_efficiency.csv')
                ee_df.to_csv(fpath, index=False)
                saved(fpath)

    else:
        print("\nNo valid data for energy efficiency analysis.")

    print(rule())


# ===========================================================================
# THE RATE PROTOCOL, DETECTED ONCE
# ===========================================================================
# A variable-rate test is not an exotic case. It is the field-standard way to
# measure rate capability: a reference block, a ramp, and a return to the
# reference rate. Until 1.9.0.64 Ratatosk read ONE C-rate from the cycler
# header — which records only the first step — and then printed it on every
# figure caption, in START_HERE and in the run page. On
# `JQ_NNM_C-rate_2-4.2V_B_29042026` that made every caption say "0.05 C" for
# a run that spans C/20 to 5C: false for 28 of its 33 cycles.
#
# The information was never missing. `rate_capability` has read a Current
# column out of the raw dataframe since 1.8.7 and grouped the cycles into
# blocks correctly. It was simply the only consumer, and it runs in Cell 6c,
# long after the captions are written.
#
# So the detection is lifted to module level and answered ONCE per dataset.
# `rate_capability` now calls these rather than keeping its own copies —
# there is no second definition to drift.
def discharge_current_per_cycle(df):
    """
    Mean absolute discharge current per cycle, in amps, from whatever the
    cycler called its current column. Empty frame when there is none.
    """
    current_col = None
    current_scale = 1.0
    for col in df.columns:
        cl = str(col).lower()
        if 'current' in cl and 'cut' not in cl:
            current_col = col
            if 'ma' in cl:
                current_scale = 0.001
            break
    if current_col is None or 'Cycle' not in df.columns:
        return pd.DataFrame(columns=['Cycle', 'Avg_Current_A'])
    work = df[['Cycle', 'Step', current_col]].copy()
    work[current_col] = pd.to_numeric(work[current_col], errors='coerce')
    dis = work[work['Step'] == 'Discharge'].dropna(subset=['Cycle',
                                                           current_col])
    if dis.empty:
        return pd.DataFrame(columns=['Cycle', 'Avg_Current_A'])
    out = (dis.groupby('Cycle')[current_col]
              .apply(lambda x: x.abs().mean() * current_scale).reset_index())
    out.columns = ['Cycle', 'Avg_Current_A']
    out['Cycle'] = out['Cycle'].astype(int)
    return out


def group_cycles_by_rate(current_per_cycle, tolerance=None):
    """
    Contiguous runs of cycles at the same discharge current.

    A cycle joins the current run when its mean current is within
    `tolerance` of the run's mean so far; otherwise it starts a new run.
    """
    tol = float(CURRENT_GROUPING_TOLERANCE if tolerance is None else tolerance)
    if current_per_cycle is None or current_per_cycle.empty:
        return []
    groups = []
    cyc = [current_per_cycle.iloc[0]['Cycle']]
    cur = [current_per_cycle.iloc[0]['Avg_Current_A']]
    for i in range(1, len(current_per_cycle)):
        row = current_per_cycle.iloc[i]
        prev = np.mean(cur)
        if (prev > 0 and abs(row['Avg_Current_A'] - prev) / prev <= tol):
            cyc.append(row['Cycle']); cur.append(row['Avg_Current_A'])
        else:
            groups.append(dict(cycles=cyc, mean_current_A=float(np.mean(cur)),
                               n_cycles=len(cyc)))
            cyc = [row['Cycle']]; cur = [row['Avg_Current_A']]
    groups.append(dict(cycles=cyc, mean_current_A=float(np.mean(cur)),
                       n_cycles=len(cyc)))
    return groups


def current_to_crate(current_A, active_mass_mg, theoretical_cap):
    """C-rate from current, active mass and theoretical capacity."""
    if (active_mass_mg is None or theoretical_cap is None
            or active_mass_mg <= 0 or theoretical_cap <= 0):
        return None
    theo_A = (active_mass_mg / 1000.0) * theoretical_cap / 1000.0
    return (current_A / theo_A) if theo_A > 0 else None


def snap_crate(crate):
    """
    The conventional rate a measured one is within RATE_LABEL_SNAP of, else
    the measured value unchanged. `None` passes through.

    Used by `format_crate` for the label and by the parameter step for the
    C-rate default, so that a measured 0.1001 C is offered as 0.1 rather
    than printed to four decimal places on every figure caption.
    """
    if crate is None or not np.isfinite(crate) or crate <= 0:
        return crate
    c = float(crate)
    near = min(_CONVENTIONAL_RATES, key=lambda r: abs(np.log(r / c)))
    return near if abs(np.log(near / c)) <= np.log(1.0 + RATE_LABEL_SNAP) else c


def format_crate(crate, snap=True):
    """`0.1 -> C/10`, `0.5 -> C/2`, `1.0 -> 1C`. See RATE_LABEL_SNAP."""
    if crate is None or not np.isfinite(crate) or crate <= 0:
        return '?C'
    c = snap_crate(float(crate)) if snap else float(crate)
    if c < 1.0:
        denom = 1.0 / c
        return f'C/{denom:.0f}' if abs(denom - round(denom)) < 0.05 \
            else f'{c:.2f}C'
    return f'{c:.0f}C' if abs(c - round(c)) < 0.05 else f'{c:.2f}C'


def rate_phrase(params):
    """
    The rate, as a caption says it: `0.05 C` for a single-rate run,
    `rates from C/20 to 5C` for a ramp. Five captions used to interpolate
    one number and append " C", which on a rate-capability run stated a rate
    that was false for most of the cycles in the figure.
    """
    rp = (params or {}).get('rate_protocol') or {}
    if rp.get('available') and rp.get('is_variable') and rp.get('label'):
        return f"rates from {rp['label']}"
    if rp.get('available') and rp.get('label'):
        return rp['label']
    c = (params or {}).get('charge_rate_c')
    return f"{c} C" if c is not None else "an unstated rate"


def rate_protocol(df, params, *, tolerance=None):
    """
    What rates this dataset was cycled at, from the data rather than the
    header.

    Returns a dict with:
      `available`   False when the file carries no current column at all
      `blocks`      [{cycles, first_cycle, last_cycle, n_cycles,
                      mean_current_A, c_rate, label}] in cycle order
      `n_blocks`    blocks holding at least MIN_CYCLES_PER_RATE cycles
      `is_variable` more than one such block
      `transition_cycles`  cycles in blocks too short to be a rate block —
                    at a rate change the cycler's last cycle of a block often
                    carries a mixed mean current and lands in a block of its
                    own. They are real cycles and they are excluded from
                    every block mean, so they are NAMED here rather than
                    silently dropped.
      `label`       'C/20' for a single rate, 'C/20 to 5C' for a ramp
      `returns_to_start`  the last block is at the first block's rate
    """
    out = dict(available=False, blocks=[], n_blocks=0, is_variable=False,
               transition_cycles=[], label=None, returns_to_start=False)
    if df is None or not hasattr(df, "columns"):
        return out
    cpc = discharge_current_per_cycle(df)
    if cpc.empty:
        return out
    mass = (params or {}).get('active_material_mass_mg')
    theo = (params or {}).get('theoretical_capacity_mAh_g')
    groups = group_cycles_by_rate(cpc, tolerance)
    if not groups:
        return out
    out['available'] = True
    blocks, short = [], []
    for g in groups:
        cr = current_to_crate(g['mean_current_A'], mass, theo)
        rec = dict(cycles=[int(c) for c in g['cycles']],
                   first_cycle=int(min(g['cycles'])),
                   last_cycle=int(max(g['cycles'])),
                   n_cycles=int(g['n_cycles']),
                   mean_current_A=float(g['mean_current_A']),
                   c_rate=cr, label=format_crate(cr))
        if g['n_cycles'] >= MIN_CYCLES_PER_RATE:
            blocks.append(rec)
        else:
            short.extend(rec['cycles'])
    out['blocks'] = blocks
    out['n_blocks'] = len(blocks)
    out['transition_cycles'] = sorted(short)
    out['is_variable'] = len(blocks) > 1
    if blocks:
        rates = [b['c_rate'] for b in blocks if b['c_rate'] is not None]
        if not out['is_variable'] or not rates:
            out['label'] = blocks[0]['label']
        else:
            out['label'] = (f"{format_crate(min(rates))} to "
                            f"{format_crate(max(rates))}")
        f, l = blocks[0]['c_rate'], blocks[-1]['c_rate']
        out['returns_to_start'] = (
            f is not None and l is not None and f > 0
            and abs(l - f) / f <= 2 * float(
                CURRENT_GROUPING_TOLERANCE if tolerance is None else tolerance))
    return out


def describe_rate(params, protocol=None):
    """
    The rate to PRINT for this dataset: the measured protocol when one was
    detected, the entered value otherwise. Never a single number for a run
    that used several.
    """
    if protocol and protocol.get('available') and protocol.get('label'):
        if protocol.get('is_variable'):
            return (f"{protocol['label']} "
                    f"({protocol['n_blocks']} rate blocks)")
        return protocol['label']
    c = (params or {}).get('charge_rate_c')
    return (f"{c} C" if c is not None else None)


# --- RATE RECOVERY: what the return to the reference rate actually says ----
# The field-standard rate schedule is a reference block, a ramp, and a return
# to the reference rate, and the number everyone quotes is
# `mean(return) / mean(reference)`. On
# `JQ_NNM_C-rate_2-4.2V_{B,C}_29042026` that is 80.4% and 80.2% — and almost
# all of it is ordinary cycling fade over the 28 cycles in between, not
# damage from the high-rate excursion.
#
# Separating the two needs a fade baseline, and the obvious one does not
# work. Projecting a line from the first reference block forward was tested
# against the CONSTANT-RATE NNM triplicate, where nothing happens between
# cycle 5 and cycle 31: a linear fit to cycles 2-5 predicts NEGATIVE capacity
# at cycle 31 against an actual 75-86 mAh/g, and a log-linear fit predicts
# about a third of it. Neither prediction interval contains the truth. Those
# cycles are formation, not fade — cell B's CE over cycles 1-6 runs 88.4,
# 76.4, 44.5, 94.0, 94.0, 96.7, and does not settle until AFTER the reference
# block has ended. On the same control, a window of cycles 6-28 lands within
# 0.3% on all three cells.
#
# So the slope is harvested from the ramp itself, where every cycle is
# post-formation. Rate sets the LEVEL and cycling sets the SLOPE:
#
#     log C(cycle) = a_block + b . cycle,     one shared b
#
# fitted on post-formation, non-transition cycles. On the two rate-tested
# cells that gives b = -0.505 %/cycle (se 0.061) and -0.517 %/cycle
# (se 0.104), residual sd 0.21% and 0.36% — the replicates agree to
# 0.012 %/cycle.
#
# THE ANCHOR IS THE PART THAT CANNOT BE FIXED HERE. Projecting that slope
# from the pre-ramp block to the return needs a level for the reference rate,
# and the only reference-rate cycles before the ramp are the formation ones.
# Every anchor drawn from them reads high, so every damage estimate drawn
# from them is biased high. Three anchors give 92.6/94.5/98.5% on cell B and
# 92.7/92.4/94.6% on cell C — a spread the same size as the effect. That is
# why this reports a RANGE and calls it an upper bound on damage, and says
# so when the reference block lies inside formation.
#
# The fix is experimental (one reference-rate cycle between blocks, or a
# reference block that outlives formation) and the protocol is field
# standard, so the tool reports the bound honestly rather than inventing a
# point estimate.
def annotate_rate_protocol(electrochemical_data, user_parameters, *,
                           verbose=True):
    """
    Detect each dataset's rate protocol and store it on its parameters.

    Called once, before any figure is drawn, so that every caption, every
    retention figure and the report itself are working from the rate the cell
    was ACTUALLY cycled at rather than the single value the cycler header
    records for its first step. Mutates and returns `user_parameters`.
    """
    any_var = False
    for name, df in (electrochemical_data or {}).items():
        params = (user_parameters or {}).get(name)
        if params is None:
            continue
        # The parameter step already answered this for the interactive path,
        # and answered it BEFORE the C-rate prompt so it could skip it. Do not
        # recompute: the declarative path has no such step, and this is where
        # it gets one.
        proto = params.get("rate_protocol") or rate_protocol(df, params)
        params["rate_protocol"] = proto
        params["rate_is_variable"] = bool(proto.get("is_variable"))
        params["rate_label"] = proto.get("label")
        if not proto.get("available") or not verbose:
            continue
        if proto.get("is_variable"):
            any_var = True
            print(f"  {name}: {proto['n_blocks']} rate blocks, "
                  f"{proto['label']}"
                  + (" , returning to the starting rate"
                     if proto.get("returns_to_start") else ""))
            print("      " + "  ".join(
                f"{b['label']}:{b['first_cycle']}-{b['last_cycle']}"
                for b in proto["blocks"]))
            if proto.get("transition_cycles"):
                print(f"      rate-transition cycle(s) not in any block: "
                      + ", ".join(str(c)
                                  for c in proto["transition_cycles"]))
        elif proto.get("label"):
            print(f"  {name}: single rate, {proto['label']}")
    if any_var and verbose:
        # THE ONE SENTENCE THAT HAS TO BE SAID OUT LOUD. Capacity against
        # cycle number in a rate test is not a fade curve; most of its shape
        # is the rate schedule. Everything downstream that says "retention"
        # now scopes itself to one rate block, and this says why.
        print("  Capacity varies with the RATE SCHEDULE in this run, not "
              "only with age — retention across rate blocks is not "
              "retention.")
    return user_parameters


def rate_recovery(cap_df, protocol, *, formation_end_cycle=None,
                  reference_cycle_min=None):
    """
    Fade rate during a rate ramp, and what the return to the reference rate
    says once that fade is accounted for.

    `cap_df` needs `Cycle` and `Discharge_mAh_g`; `protocol` is
    `rate_protocol`'s dict. Returns None when there is no ramp with a
    return to the starting rate.

    Keys: `fade_pct_per_cycle`, `fade_se`, `residual_sd_pct`, `n_fit`,
    `q_recovery_pct` (the raw, field-standard number), `anchors`
    (list of {name, cycle, capacity, predicted, ratio_pct}),
    `damage_bound_pct` (min, max over anchors), `anchor_in_formation`,
    `formation_end`, `note`.
    """
    if not protocol or not protocol.get('is_variable') \
            or not protocol.get('returns_to_start'):
        return None
    blocks = protocol.get('blocks') or []
    if len(blocks) < 3:
        return None
    first, last = blocks[0], blocks[-1]
    d = cap_df[['Cycle', 'Discharge_mAh_g']].dropna().copy()
    d['Cycle'] = d['Cycle'].astype(int)
    d = d[d['Discharge_mAh_g'] > 0]
    if d.empty:
        return None
    trans = set(protocol.get('transition_cycles') or [])

    # Post-formation, and never the transition cycles: the fit is what the
    # whole result rests on, so it gets only cycles that are measurements of
    # a settled cell at a known rate.
    f_end = (int(formation_end_cycle) if formation_end_cycle
             else int(first['last_cycle']))
    ramp = [b for b in blocks[1:-1]]
    if not ramp:
        return None
    lo = max(f_end + 1, ramp[0]['first_cycle'])
    hi = ramp[-1]['last_cycle']
    fit = d[(d['Cycle'] >= lo) & (d['Cycle'] <= hi)
            & (~d['Cycle'].isin(trans))].copy()
    labels = {}
    for b in ramp:
        for c in b['cycles']:
            labels[c] = b['label']
    fit['block'] = fit['Cycle'].map(labels)
    fit = fit.dropna(subset=['block'])
    keys = sorted(fit['block'].unique())
    if len(fit) < len(keys) + 3 or len(keys) < 2:
        return None

    y = np.log(fit['Discharge_mAh_g'].to_numpy(float))
    X = np.zeros((len(fit), len(keys) + 1))
    for i, k in enumerate(keys):
        X[:, i] = (fit['block'] == k).to_numpy(float)
    X[:, -1] = fit['Cycle'].to_numpy(float)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    b_share = float(coef[-1])
    resid = y - X @ coef
    dof = max(len(fit) - (len(keys) + 1), 1)
    s = float(np.sqrt((resid ** 2).sum() / dof))
    try:
        se_b = float(s * np.sqrt(np.linalg.inv(X.T @ X)[-1, -1]))
    except np.linalg.LinAlgError:
        se_b = float('nan')

    post = d[(d['Cycle'] >= last['first_cycle'])
             & (d['Cycle'] <= last['last_cycle'])
             & (~d['Cycle'].isin(trans))]
    pre = d[(d['Cycle'] >= first['first_cycle'])
            & (d['Cycle'] <= first['last_cycle'])
            & (~d['Cycle'].isin(trans))]
    _cmin = int(RATE_RECOVERY_MIN_REFERENCE if reference_cycle_min is None
                else reference_cycle_min)
    if len(post) < _cmin or len(pre) < _cmin:
        return None
    obs = float(post['Discharge_mAh_g'].mean())
    xt = float(post['Cycle'].mean())

    # THREE ANCHORS, NOT ONE. They differ by as much as the effect, and the
    # difference is not noise — it is how much formation is still in the
    # reference block. Reporting one of them would be choosing an answer.
    anchors = []
    _pre_body = pre[pre['Cycle'] > int(pre['Cycle'].min())] \
        if len(pre) > 1 else pre
    cands = [("block mean", float(_pre_body['Cycle'].mean()),
              float(_pre_body['Discharge_mAh_g'].mean()))]
    for _c in sorted(pre['Cycle'])[-2:]:
        cands.append((f"cycle {int(_c)}", float(_c),
                      float(pre.loc[pre['Cycle'] == _c,
                                    'Discharge_mAh_g'].iloc[0])))
    seen = set()
    for nm, x0, c0 in cands:
        if c0 <= 0 or round(x0, 3) in seen:
            continue
        seen.add(round(x0, 3))
        pred = c0 * np.exp(b_share * (xt - x0))
        anchors.append(dict(name=nm, cycle=x0, capacity=c0,
                            predicted=float(pred),
                            ratio_pct=float(100.0 * obs / pred)))
    if not anchors:
        return None
    ratios = [a['ratio_pct'] for a in anchors]

    # AN UNKNOWN FORMATION END IS NOT A CLEAN ONE. `detect.formation_end`
    # tests coulombic efficiency against the median of the later cycles, and
    # on a rate test that median is not a fixed target: CE moves WITH THE
    # RATE. On `JQ_NNM_C-rate_2-4.2V_B_29042026` the ramp sits near 97% and
    # the return block near 94%, so no window ever holds within tolerance
    # and the answer comes back None — from a reference block of five cycles
    # whose CE runs 88.4, 76.4, 44.5, 94.0, 94.0 and has plainly not settled.
    #
    # So the three states are kept apart: formation demonstrably ends after
    # the reference block (contaminated), demonstrably before it (clean), or
    # could not be established (treated as contaminated, because claiming
    # the clean case without evidence is how a bound becomes an estimate).
    if formation_end_cycle is None:
        in_formation = True
        note = (
            "Formation could not be located from coulombic efficiency: on a "
            "variable-rate run CE moves with the RATE as well as with age, "
            "so the settling test has no fixed target to hold against. The "
            f"reference block is cycles {int(first['first_cycle'])}-"
            f"{int(first['last_cycle'])}, which on this protocol is where "
            "formation happens, so every anchor for the reference level may "
            "still contain formation capacity and read high. The range below "
            "is an UPPER BOUND on the loss caused by the rate excursion, not "
            "an estimate of it.")
    elif int(first['last_cycle']) <= int(formation_end_cycle):
        in_formation = True
        note = ("The reference block ends at cycle "
            f"{int(first['last_cycle'])} and formation is not complete until "
            f"cycle {int(formation_end_cycle)}, so every anchor for the "
            "reference "
            "level still contains formation capacity and reads high. The "
            "range below is an UPPER BOUND on the loss caused by the rate "
            "excursion, not an estimate of it.")
    else:
        in_formation = False
        note = (f"Formation ends at cycle {int(formation_end_cycle)}, before "
                f"the reference block closes at cycle "
                f"{int(first['last_cycle'])}, so the anchors below are "
                f"settled reference-rate cycles and the range is an "
                f"estimate rather than a bound.")
    return dict(fade_pct_per_cycle=100.0 * (np.exp(b_share) - 1.0),
                fade_se_pct=100.0 * se_b, residual_sd_pct=100.0 * s,
                n_fit=int(len(fit)), blocks_fitted=keys,
                q_recovery_pct=float(100.0 * obs
                                     / pre['Discharge_mAh_g'].mean()),
                observed=obs, observed_cycle=xt, anchors=anchors,
                damage_bound_pct=(float(min(ratios)), float(max(ratios))),
                anchor_in_formation=in_formation,
                formation_end=formation_end_cycle,
                note=note)


@_honours_verbose
def rate_capability(electrochemical_data, user_parameters, *, save_location=None,
        all_cycle_tables=None, file_format='png', verbose=True):
    """
    Capacity grouped by C-rate.

    Ported from 1.8.7 Cell 13, wrapped in a function without editing
    the body. `all_cycle_tables` is `cycling_summary`'s output where the
    original read a notebook global; passing None reproduces the
    not-yet-run branch exactly.
    """
    # ONE DEFINITION. These were the originals; they now live at module
    # level so the parameter step and the report can ask the same question
    # and get the same answer. See `rate_protocol`.
    _get_discharge_current_per_cycle = discharge_current_per_cycle
    _group_by_rate = group_cycles_by_rate
    _current_to_crate = current_to_crate
    _format_crate = format_crate

    print(rule("RATE CAPABILITY ANALYSIS"))

    _any_multirate = False

    _have_cycle_tables = _has_tables(all_cycle_tables)

    if EXCLUDE_INCOMPLETE and not _have_cycle_tables:
        print("  ⚠ all_cycle_tables not found — Cell 5b has not been "
              "run. Protocol-incomplete filtering unavailable.\n")

    for name, df in electrochemical_data.items():
        params = user_parameters.get(name, {})
        composition = _get_display_name(name, params, user_parameters)
        active_mass_mg = params.get('active_material_mass_mg')
        theoretical_cap = params.get('theoretical_capacity_mAh_g')

        # Get discharge current per cycle
        current_df = _get_discharge_current_per_cycle(df)

        if current_df.empty:
            print(f"  {composition}: no current data found, skipping.")
            continue

        # Group by rate
        rate_groups = _group_by_rate(current_df)

        # Count distinct rate levels
        unique_currents = [g['mean_current_A'] for g in rate_groups]

        # Cluster unique currents to find distinct rates
        distinct_rates = []
        for curr in unique_currents:
            matched = False
            for dr in distinct_rates:
                if abs(curr - dr) / max(dr, 1e-9) <= CURRENT_GROUPING_TOLERANCE:
                    matched = True
                    break
            if not matched:
                distinct_rates.append(curr)

        n_distinct = len(distinct_rates)

        if n_distinct < 2:
            print(f"  {composition}: single rate detected "
                  f"({_format_crate(_current_to_crate(distinct_rates[0], active_mass_mg, theoretical_cap)) if distinct_rates else '?'}), "
                  f"rate capability analysis skipped.")
            continue

        _any_multirate = True
        print(f"  {composition}: {n_distinct} distinct rates detected "
              f"across {len(rate_groups)} rate blocks")

        # Get capacity data from Cell 5b's cache, falling back to raw derivation
        if _have_cycle_tables and name in all_cycle_tables:
            ct = all_cycle_tables[name]
            cap_df = (_ct_with_flag(ct, 'Discharge_mAh_g')
                      .dropna(subset=['Cycle', 'Discharge_mAh_g'])
                      .copy())
        else:
            # Fallback: derive via groupby
            discharge = df[df['Step'] == 'Discharge'].dropna(
                subset=['Cycle', 'Discharge_Capacity'])
            if discharge.empty:
                print(f"  {composition}: no capacity data, skipping.")
                continue
            grp = discharge.groupby('Cycle')['Discharge_Capacity']
            cap_df = pd.DataFrame({
                'Cycle': grp.max().index.astype(int),
                'Discharge_mAh_g': (grp.max() - grp.min()).values,
                'Incomplete': False,
            })

        if cap_df.empty:
            print(f"  {composition}: no capacity data, skipping.")
            continue

        # Exclude protocol-incomplete cycles (from Cell 5b).
        # The heuristic-based filter that used to live here has been
        # deliberately removed: high-rate cycles in rate-capability
        # experiments genuinely deliver much less capacity than the
        # reference rate, and the heuristic falsely flagged them.
        if EXCLUDE_INCOMPLETE and 'Incomplete' in cap_df.columns:
            n_incomplete = int(cap_df['Incomplete'].sum())
            if n_incomplete > 0:
                removed = cap_df.loc[cap_df['Incomplete'],
                                     'Cycle'].tolist()
                print(f"    Excluded {n_incomplete} "
                      f"protocol-incomplete cycle(s): "
                      f"{', '.join(str(int(c)) for c in removed)}")
            cap_df = cap_df[~cap_df['Incomplete']].copy()

        # Merge current and capacity data
        merged = pd.merge(cap_df, current_df, on='Cycle', how='inner')

        if merged.empty:
            continue

        # Assign C-rate labels to each cycle
        merged['C_rate'] = merged['Avg_Current_A'].apply(
            lambda x: _current_to_crate(x, active_mass_mg, theoretical_cap)
        )
        merged['C_rate_label'] = merged['C_rate'].apply(_format_crate)

        # Assign rate group index
        group_assignments = {}
        for gi, group in enumerate(rate_groups):
            for cyc in group['cycles']:
                group_assignments[cyc] = gi
        merged['Rate_Group'] = merged['Cycle'].map(group_assignments)

        # Build summary per rate group
        rate_summary = []
        for gi, group in enumerate(rate_groups):
            group_data = merged[merged['Rate_Group'] == gi]
            if group_data.empty or len(group_data) < MIN_CYCLES_PER_RATE:
                continue

            crate = _current_to_crate(group['mean_current_A'],
                                       active_mass_mg, theoretical_cap)
            crate_label = _format_crate(crate)

            rate_summary.append({
                'group_index': gi,
                'c_rate': crate,
                'c_rate_label': crate_label,
                'mean_current_A': group['mean_current_A'],
                'mean_capacity': group_data['Discharge_mAh_g'].mean(),
                'std_capacity': group_data['Discharge_mAh_g'].std(),
                'n_cycles': len(group_data),
                'first_cycle': int(group_data['Cycle'].min()),
                'last_cycle': int(group_data['Cycle'].max())
            })

        if len(rate_summary) < 2:
            print(f"  {composition}: insufficient rate groups "
                  f"(need >= 2 with >= {MIN_CYCLES_PER_RATE} cycles each)")
            continue

        rate_df = pd.DataFrame(rate_summary)

        # Reference capacity (first rate group, usually lowest rate)
        ref_cap = rate_df.iloc[0]['mean_capacity']
        # NOT capacity retention: capacity at this RATE against the
        # reference rate group. Exporting both under one header was how a
        # reader ended up comparing two different quantities.
        rate_df['Rate_Capability_%'] = rate_df['mean_capacity'] / ref_cap * 100

        # Recovery: if final group is at the same rate as the first
        first_rate = rate_df.iloc[0]['c_rate']
        last_group = rate_df.iloc[-1]
        recovery = None
        if (first_rate is not None and last_group['c_rate'] is not None and
            abs(last_group['c_rate'] - first_rate) / max(first_rate, 1e-9)
            <= CURRENT_GROUPING_TOLERANCE * 2):
            recovery = last_group['mean_capacity'] / ref_cap * 100

        # --- Print summary ---
        print("\n" + heading(composition))
        print(f"  {'Rate':<10} {'Capacity':>10} {'Retention':>10} "
              f"{'Cycles':>8} {'Range':>12}")
        print(f"  {'':10} {'mAh/g':>10} {'%':>10} {'':>8} {'':>12}")
        print(f"  {'-'*52}")

        for _, row in rate_df.iterrows():
            cycle_range = (f"{row['first_cycle']}-{row['last_cycle']}"
                          if row['first_cycle'] != row['last_cycle']
                          else str(row['first_cycle']))
            print(f"  {row['c_rate_label']:<10} "
                  f"{row['mean_capacity']:>10.1f} "
                  f"{row['Rate_Capability_%']:>10.1f} "
                  f"{row['n_cycles']:>8} "
                  f"{cycle_range:>12}")

        if recovery is not None:
            print(f"\n  Recovery to {rate_df.iloc[0]['c_rate_label']}: "
                  f"{recovery:.1f}%")

        # --- what that recovery is made of -------------------------------
        # THE RECOVERY PERCENTAGE WAS PRINTED AND NEVER EXPORTED. It reached
        # no CSV, no run page and no START_HERE — the fifth diagnostic in
        # this pipeline computed correctly and consumed by nothing, after
        # `area_in_window`, `area_is_lower_bound`, `sig.occupancy` and
        # `height_worst`. It is now carried on `rate_df` and written out
        # with the rest, together with what it decomposes into.
        _proto = rate_protocol(df, params)
        _f_end = None
        if 'CE_%' in cap_df.columns:
            _ce = {int(r['Cycle']): float(r['CE_%'])
                   for _, r in cap_df.iterrows() if pd.notna(r.get('CE_%'))}
            _fe, _ = formation_end(_ce)
            if _fe is not None:
                _f_end = int(_fe)
        _rec = rate_recovery(cap_df, _proto, formation_end_cycle=_f_end)
        if _proto.get('transition_cycles'):
            # NAMED, NOT DROPPED. At a rate change the cycler's last cycle of
            # a block carries a mixed mean current, lands in a block of its
            # own and fails MIN_CYCLES_PER_RATE. Excluding it is right; doing
            # it silently is not — on the NNM rate test that is 7 of 33
            # cycles absent from every block mean with nothing saying so.
            print(f"    {len(_proto['transition_cycles'])} rate-transition "
                  f"cycle(s) excluded from the block means: "
                  + ", ".join(str(c)
                              for c in _proto['transition_cycles']))
        if _rec is not None:
            print(f"\n  Fade during the ramp: "
                  f"{_rec['fade_pct_per_cycle']:+.3f} %/cycle "
                  f"(se {_rec['fade_se_pct']:.3f}, residual sd "
                  f"{_rec['residual_sd_pct']:.2f}%, n={_rec['n_fit']})")
            _lo, _hi = _rec['damage_bound_pct']
            print(f"  Recovery against that fade: {_lo:.1f}-{_hi:.1f}% "
                  f"across {len(_rec['anchors'])} anchor(s) for the "
                  f"reference level")
            for _a in _rec['anchors']:
                print(f"      {_a['name']:<12} {_a['capacity']:6.1f} mAh/g "
                      f"at cycle {_a['cycle']:.1f} -> predicts "
                      f"{_a['predicted']:6.1f}, observed "
                      f"{_rec['observed']:6.1f}  ({_a['ratio_pct']:.1f}%)")
            print(f"  {_rec['note']}")
            rate_df['q_recovery_pct'] = _rec['q_recovery_pct']
            rate_df['fade_pct_per_cycle'] = _rec['fade_pct_per_cycle']
            rate_df['fade_se_pct'] = _rec['fade_se_pct']
            rate_df['fade_residual_sd_pct'] = _rec['residual_sd_pct']
            rate_df['damage_bound_low_pct'] = _rec['damage_bound_pct'][0]
            rate_df['damage_bound_high_pct'] = _rec['damage_bound_pct'][1]
            rate_df['damage_bound_is_upper_bound'] = \
                _rec['anchor_in_formation']
            rate_df['formation_end_cycle'] = _rec['formation_end']
        elif recovery is not None:
            rate_df['q_recovery_pct'] = recovery
        if _proto.get('transition_cycles'):
            rate_df['transition_cycles_excluded'] = ";".join(
                str(c) for c in _proto['transition_cycles'])


        # =================================================================
        # PLOT 1: Capacity vs C-rate bar chart
        # =================================================================

        # Deduplicate: average across groups at the same rate
        unique_rates = []
        for _, row in rate_df.iterrows():
            matched = False
            for ur in unique_rates:
                if (row['c_rate'] is not None and ur['c_rate'] is not None and
                    abs(row['c_rate'] - ur['c_rate']) / max(ur['c_rate'], 1e-9)
                    <= CURRENT_GROUPING_TOLERANCE * 2):
                    # Average with existing
                    ur['capacities'].append(row['mean_capacity'])
                    matched = True
                    break
            if not matched:
                unique_rates.append({
                    'c_rate': row['c_rate'],
                    'c_rate_label': row['c_rate_label'],
                    'capacities': [row['mean_capacity']]
                })

        bar_labels = [ur['c_rate_label'] for ur in unique_rates]
        bar_heights = [np.mean(ur['capacities']) for ur in unique_rates]
        bar_errors = [np.std(ur['capacities']) if len(ur['capacities']) > 1
                      else 0 for ur in unique_rates]

        # Colour: highlight recovery rate in green
        bar_colours = []
        for j, ur in enumerate(unique_rates):
            if (j == len(unique_rates) - 1 and recovery is not None and
                j > 0):
                bar_colours.append(RECOVERY_COLOUR)
            else:
                bar_colours.append(BAR_COLOUR)

        fig, ax = plt.subplots(figsize=(figure_width_inches, figure_height_inches))

        x_pos = np.arange(len(bar_labels))
        bars = ax.bar(x_pos, bar_heights, yerr=bar_errors,
                      color=bar_colours, edgecolor='black', linewidth=0.5,
                      capsize=4, width=0.6, zorder=3)

        # Add capacity labels on bars
        for bar, height in zip(bars, bar_heights):
            ax.text(bar.get_x() + bar.get_width() / 2, height + 1,
                   f'{height:.0f}', ha='center', va='bottom', fontsize=10)

        ax.set_xticks(x_pos)
        ax.set_xticklabels(bar_labels, fontsize=12)
        ax.set_xlabel('C-rate', fontsize=14)
        dch_label = _discharge_label(params)
        ax.set_ylabel(f'{dch_label} capacity / mAh g$^{{-1}}$', fontsize=14)
        ax.set_ylim(bottom=0, top=max(bar_heights) * 1.15)
        ax.tick_params(axis='both', labelsize=12, width=1, direction='in',
                       top=True, right=True)
        for sp in ax.spines.values():
            sp.set_linewidth(0.8)
        plt.tight_layout()

        # Caption
        rates_str = ', '.join(bar_labels[:-1])
        if len(bar_labels) > 1:
            rates_str += f' and {bar_labels[-1]}'
        else:
            rates_str = bar_labels[0]

        recovery_str = ''
        if recovery is not None:
            recovery_str = (f' The final bar (green) shows capacity '
                           f'recovery upon returning to '
                           f'{rate_df.iloc[0]["c_rate_label"]} '
                           f'({recovery:.1f}% of initial).')

        dch_word = _discharge_label(params).lower()
        caption = (
            f"Figure X. Rate capability of {composition} at "
            f"{rates_str}. Values shown are mean {dch_word} capacities "
            f"with error bars indicating standard deviation across "
            f"cycles at each rate.{recovery_str}"
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

        if save_location:
            fpath = os.path.join(save_location,
                                f'{name}_rate_capability.{image_format(params)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()
        plt.close(fig)


        # =================================================================
        # PLOT 2: Capacity vs cycle with rate annotations
        # =================================================================

        fig, ax = plt.subplots(figsize=(10, 6))

        ax.plot(merged['Cycle'], merged['Discharge_mAh_g'],
               color=BAR_COLOUR, marker='o', markersize=5,
               linestyle='-', linewidth=0.8, zorder=3)

        # Shade rate blocks and label them
        prev_right = None
        for _, row in rate_df.iterrows():
            left = row['first_cycle'] - 0.5
            right = row['last_cycle'] + 0.5

            ax.axvspan(left, right, alpha=0.06, color='grey')

            mid_cycle = (row['first_cycle'] + row['last_cycle']) / 2
            y_pos = ax.get_ylim()[1] * 0.95 if prev_right is None else ax.get_ylim()[1] * 0.95

            ax.text(mid_cycle, merged['Discharge_mAh_g'].max() * 1.05,
                   row['c_rate_label'],
                   ha='center', va='bottom', fontsize=10,
                   fontweight='bold', color='#333333')

            prev_right = right

        ax.set_xlabel('Cycle number', fontsize=14)
        _force_integer_cycles(ax)
        ax.set_ylabel(f'{dch_label} capacity / mAh g$^{{-1}}$', fontsize=14)
        ax.set_ylim(bottom=0,
                    top=merged['Discharge_mAh_g'].max() * 1.15)
        ax.set_xlim(left=0)
        ax.tick_params(axis='both', labelsize=12, width=1, direction='in',
                       top=True, right=True)
        for sp in ax.spines.values():
            sp.set_linewidth(0.8)
        plt.tight_layout()

        caption = (
            f"Figure X. {dch_label} capacity vs. cycle number for "
            f"{composition} during rate capability testing. Shaded "
            f"regions indicate cycling at different C-rates as labelled."
        )
        print(section("  Suggested caption"))
        print(bullet(caption, indent=2, label_width=2))

        if save_location:
            fpath = os.path.join(save_location,
                                f'{name}_rate_capability_cycling.{image_format(params)}')
            fig.savefig(fpath, dpi=300, bbox_inches='tight')
            saved(fpath)

        plt.show()


        # --- Export ---
        if save_location:
            # `c_rate_measured` beside the snapped label, because the label
            # is for reading and the number is for computing. See
            # RATE_LABEL_SNAP.
            rate_df['c_rate_measured'] = rate_df['c_rate']
            _extra = [c for c in ('c_rate_measured', 'mean_current_A',
                                  'q_recovery_pct', 'fade_pct_per_cycle',
                                  'fade_se_pct', 'fade_residual_sd_pct',
                                  'damage_bound_low_pct',
                                  'damage_bound_high_pct',
                                  'damage_bound_is_upper_bound',
                                  'formation_end_cycle',
                                  'transition_cycles_excluded')
                      if c in rate_df.columns]
            export_df = rate_df[['c_rate_label', 'mean_capacity',
                                  'std_capacity', 'Rate_Capability_%',
                                  'n_cycles', 'first_cycle', 'last_cycle']
                                 + _extra]
            fpath = os.path.join(save_location,
                                f'{name}_rate_capability_summary.csv')
            export_df.to_csv(fpath, index=False)
            saved(fpath)

    if not _any_multirate:
        print("All datasets are single-rate — "
              "rate capability analysis skipped.")
        print("(Load multi-rate data to use this analysis, e.g., "
              "C/10 -> C/5 -> C/2 -> 1C -> C/10 recovery)")

    print(rule())


In [ ]:
# DEFINITIONS K — RUN FOLDERS, THE MANIFEST, PROVENANCE
# =============================================================================
"""
Where a run's outputs go, and what it records about itself.

Ported from 1.8.7 Cell 3b: every run gets its own timestamped folder under
`runs/`, a `run_manifest.json` recording who ran what against which files with
which package versions, and — at the end — the outputs sorted into numbered
subfolders and the manifest closed with a checksum of everything written.

    start_run(base, version, input_paths)  -> (run_dir, manifest)
    write_manifest(run_dir, manifest)
    finalise_run(run_dir, manifest)         organise, checksum, close
    freeze_run(run_dir)                     copy to published/, never overwritten

The one change: `pipeline_flags`
--------------------------------
1.8.7 captured the module-level switches that change what a run produces by
reading `globals()` for a hardcoded list of names. In a package there is no
single namespace holding them all, so `start_run` and `finalise_run` take a
`flags` dict instead, and `default_flags()` assembles it by asking each module
for its own constants. That makes the manifest complete rather than dependent
on which cells happened to have run — which was 1.8.7's own complaint about it
in the docstring the port preserves.

Why it matters: with the version deliberately held across a series of
behaviour changes, these flags are the only thing distinguishing one run from
another.
"""


import getpass
import hashlib
import json
import os
import platform
import re
import shutil
import sys

from datetime import datetime, timezone


FREEZE_ROOT_NAME = 'published'   # frozen runs live here, never overwritten
HASH_CHUNK = 1 << 20
_MISSING = object()      # "this value is not a recordable flag"

# Where each output lands. Rules are ordered and the first match wins:
# 'capacity_attribution' must be caught as a descriptor before the generic
# 'capacity' rule claims it for cycling.
FOLDER_RULES = [
    ('7_summary',           ['analysis_summary', 'results_at_a_glance',
                             'ratatosk_summary', 'summary.pdf',
                             'cycling_performance_summary']),
    ('6_descriptors',       ['tracked_peaks', 'delta_v', 'peak_trends',
                             'capacity_attribution', 'trend_shapes']),
    # 'fit_coherence' and 'cycle_integrity' are new in 1.9.0; without a rule
    # they land in '9_other'.
    ('5_peak_fitting',      ['detected_peaks', 'peak_fits',
                             'fit_diagnostics', 'fitted_parameters',
                             'fit_coherence']),
    # 'preprocessing_qc' is the raw-against-processed panel for one cycle —
    # a dQ/dV figure whose filename does not contain the string, so it landed
    # in '9_other', on its own, with nothing in the run linking to it.
    ('4_dqdv',              ['dqdv', 'preprocessing_qc']),
    ('2_voltage_profiles',  ['voltage_profile']),
    ('3_energy_power',      ['ragone', 'power_per', 'energy_per',
                             'power_energy']),
    # The ion-named voltage tokens are GENERATED from `WORKING_ION_WORDS`,
    # not typed: one output filename is built from the anode-aware half-cycle
    # label, so an anode dataset writes _avg_desodiation_voltage.csv where a
    # cathode writes _avg_discharge_voltage.csv. It is the only filename in
    # Ratatosk that varies with electrode polarity, and a hand-written list
    # of Li and Na tokens would quietly stop filing the file the day a third
    # ion existed — it would land in the run root with no error.
    ('1_cycling',           ['cycle_integrity',
                             'capacity', 'cycle_life', 'retention', 'fade',
                             'efficiency', 'coulombic',
                             'discharge_voltage']
                            + [f'{_w.lower()}_voltage'
                               for _p in WORKING_ION_WORDS.values()
                               for _w in _p]),
]
COMPARATIVE_DIR = '_comparative'   # outputs not tied to one cell
COPY_COMPARATIVE_TO_CELLS = True
COMPARATIVE_COPY_PREFIX = 'comparative_'


def subfolders():
    """The output folders a run creates, in reading order."""
    return [f for f, _ in sorted(FOLDER_RULES)] + [COMPARATIVE_DIR]


def default_flags():
    """
    Every module-level switch that changes what a run produces.

    Every ALL-CAPS module-level constant, whatever cell defined it. 1.8.7 read
    a HARDCODED LIST of names out of globals() and noted in its own docstring
    that a flag defined in Module 4 could not be seen from Cell 3b. Taking
    everything means the manifest cannot fall behind the code.

    Works in BOTH builds. The package has modules to ask; the inline notebook
    is one flat namespace with no modules at all, so the relative import
    below raised `ImportError` there and — since `start_run` does not catch
    it — the flags were never recorded in the notebook anyone actually runs.
    In that build the constants are plain globals and are read directly.
    """
    def _scalar(v):
        if v is None or isinstance(v, (bool, int, float, str)):
            return v
        if isinstance(v, (list, tuple)) and all(
                isinstance(x, (bool, int, float, str)) for x in v):
            return list(v)
        if isinstance(v, dict) and all(
                isinstance(x, (bool, int, float, str)) for x in v.values()):
            # A per-profile table such as PROFILE_MIN_DISTANCE_MV decides how
            # many peaks a run finds; it is as much a switch as a scalar is.
            return dict(v)
        return _MISSING

    out = {}
    try:
        from . import (signal, detect, fitting, quality, analyse, plots,
                       cycling, io, params, report)  # inline-ok: the notebook
        # has no package, raises ImportError here, and the `except` below
        # reads the same constants from the flat namespace instead. Deliberate
        # and handled — see the docstring.
        mods = [signal, detect, fitting, quality, analyse, plots,
                cycling, io, params, report]
    except ImportError:
        mods = []

    if mods:
        for mod in mods:
            short = mod.__name__.rsplit('.', 1)[-1]
            for k in dir(mod):
                if not re.fullmatch(r"[A-Z][A-Z0-9_]*", k):
                    continue
                v = _scalar(getattr(mod, k))
                if v is not _MISSING:
                    out[f"{short}.{k}"] = v
        return out

    # Flattened notebook: one namespace, no modules. The names are the same;
    # only the module prefix is unavailable, so they are recorded bare.
    g = dict(globals())
    for k, raw in list(g.items()):
        if not re.fullmatch(r"[A-Z][A-Z0-9_]*", k):
            continue
        v = _scalar(raw)
        if v is not _MISSING:
            out[k] = v
    return out

def _sha256(path, chunk=HASH_CHUNK):
    """Checksum a file without loading it into memory."""
    h = hashlib.sha256()
    try:
        with open(path, 'rb') as f:
            for block in iter(lambda: f.read(chunk), b''):
                h.update(block)
        return h.hexdigest()
    except Exception as exc:
        return f'unavailable: {exc}'


def _jsonable(obj):
    """
    Coerce anything the parameter dictionaries might hold into JSON.

    numpy scalars, numpy arrays, pandas types and Path objects all appear in
    user_parameters depending on how a value was entered, and a manifest that
    fails to write is worse than no manifest at all.
    """
    try:
        import numpy as _np
        if isinstance(obj, (_np.integer,)):
            return int(obj)
        if isinstance(obj, (_np.floating,)):
            return float(obj)
        if isinstance(obj, (_np.bool_,)):
            return bool(obj)
        if isinstance(obj, _np.ndarray):
            return obj.tolist()
    except Exception:
        pass
    if isinstance(obj, (set, tuple)):
        return list(obj)
    return str(obj)


def _package_versions():
    """
    Versions of the libraries whose behaviour a result depends on.

    Called twice, like _pipeline_flags, and for a related reason: on
    2026-08-29 a run recorded lmfit as 'not loaded' because lmfit was not
    installed when Cell 3b ran — it was pip-installed after Module 4 raised,
    and Module 4 was re-run. The manifest therefore did not record the version
    of the one library that does the fitting. finalise_run re-reads this.

    'not installed' and 'import failed' are now distinguished, because
    'not loaded' read as "we did not check" when it meant "it was not there".
    """
    import importlib, importlib.util
    out = {}
    for mod in ('numpy', 'scipy', 'pandas', 'matplotlib', 'lmfit',
                'seaborn', 'python_calamine'):
        try:
            if importlib.util.find_spec(mod) is None:
                out[mod] = 'not installed'
                continue
            out[mod] = getattr(importlib.import_module(mod),
                               '__version__', 'unknown version')
        except Exception as _e:
            out[mod] = f'import failed: {type(_e).__name__}'
    return out


def start_run(base_location, version, input_paths=None, note='',
              flags=None):
    """
    Create a run folder and write the opening manifest.

    Returns (run_dir, manifest). The caller rebinds save_location to run_dir.
    """
    stamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
    run_id = f'{stamp}_v{version}'
    # normpath so the manifest does not record a path with mixed
    # separators. Cell 2 builds save_location from a forward-slash string
    # via expanduser, which on Windows yields C:\Users\me/Documents/...
    run_dir = os.path.normpath(
        os.path.join(base_location, 'runs', run_id))
    os.makedirs(run_dir, exist_ok=True)

    inputs = []
    for p in (input_paths or []):
        try:
            st = os.stat(p)
            inputs.append({
                'path': os.path.abspath(p),
                'name': os.path.basename(p),
                'bytes': st.st_size,
                'modified': datetime.fromtimestamp(
                    st.st_mtime, timezone.utc).isoformat(),
                'sha256': _sha256(p),
            })
        except Exception as exc:
            inputs.append({'path': str(p), 'error': str(exc)})

    manifest = {
        'run_id': run_id,
        'ratatosk_version': version,
        'started_utc': datetime.now(timezone.utc).isoformat(),
        'note': note,
        'operator': getpass.getuser(),
        'machine': platform.node(),
        'platform': platform.platform(),
        'python': sys.version.split()[0],
        'packages': _package_versions(),
        # Passed in rather than scraped from globals() — see the module
        # docstring. `default_flags()` builds the complete set.
        'pipeline_flags': (flags if flags is not None
                           else default_flags()),
        'inputs': inputs,
        'datasets': {},
        'outputs': [],
        'status': 'running',
    }
    return run_dir, manifest


def write_manifest(run_dir, manifest):
    path = os.path.join(run_dir, 'run_manifest.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, default=_jsonable)
    return path


_WIN_RESERVED = {'CON', 'PRN', 'AUX', 'NUL'} | {
    f'{p}{i}' for p in ('COM', 'LPT') for i in range(1, 10)}


def _safe_dirname(name):
    out = str(name)
    for ch in '<>:"/\\|?*':
        out = out.replace(ch, '_')
    out = out.strip().rstrip('. ') or 'dataset'
    # Windows cannot create a directory named CON, PRN, AUX, NUL, COM1-9 or
    # LPT1-9, with or without an extension, and rejects a trailing space.
    if out.split('.')[0].upper() in _WIN_RESERVED:
        out += '_'
    return out


def _dataset_folder(name, params, used=None):
    """
    Reader-legible folder name: composition and cell, nothing more.

    Kept short deliberately. Windows caps a full path at 260 characters and
    Ratatosk's own filenames already run to 70, so the descriptive detail
    goes in dataset_info.txt inside the folder rather than in its name.
    """
    # Truncate FIRST, then sanitise: slicing after the strip could leave a
    # trailing space or dot, which Windows will not accept in a path segment.
    comp = _safe_dirname(str(params.get('composition', '') or name)[:34])
    # _safe_dirname substitutes 'dataset' for an empty string, so guard the
    # empty case rather than feeding it through and getting '_celldatase'.
    raw_cid = str(params.get('cell_id', '') or '').strip()
    cid = _safe_dirname(raw_cid[:6]) if raw_cid else ''
    base = f"{comp}_cell{cid}" if cid else comp
    if used is not None:
        stem, n = base, 2
        while base in used:
            base = f"{stem}_{n}"
            n += 1
        used.add(base)
    return base


def write_dataset_info(folder, name, params, manifest=None, decided=None):
    """A plain-text record of the cell, written beside its results.

    `params` is what the operator ASKED for. `decided` is what the pipeline
    MEASURED and then used — the profile class it read off the curve, the
    reference cycle it chose and why, the peak shape it calibrated, the
    integral fidelity it found. Those are the numbers that determined the
    result, and until 1.9.0.11 this file recorded only the request. It is the
    same rule as the manifest's: record what ran, never what was offered.

    `decided` is a plain dict of label -> value, built by the notebook from
    the objects that did the work; anything missing is simply omitted, so an
    older caller still produces a valid record.
    """
    src = ''
    for inp in (manifest or {}).get('inputs', []):
        if inp.get('name', '').startswith(name):
            src = f"{inp.get('name')}  (sha256 {inp.get('sha256', '')[:16]})"
            break
    rows = [
        ('Composition', params.get('composition')),
        ('Cell identifier', params.get('cell_id') or '(not recorded)'),
        ('Dataset key', name),
        ('Source file', src or '(not captured)'),
        # The version, beside the source file, so this folder says what made
        # it without anyone having to open the manifest.
        ('Ratatosk version',
         (manifest or {}).get('ratatosk_version') or '(not captured)'),
        ('Run', (manifest or {}).get('run_id') or '(not captured)'),
        ('Electrode', params.get('electrode_type')),
        ('Active mass (mg)', params.get('active_material_mass_mg')),
        ('Areal loading (mg/cm2)', params.get('active_loading_mg_cm2')),
        ('Blend', params.get('blend')),
        ('Rate (C)', params.get('charge_rate_c')),
        ('Voltage window (V)', f"{params.get('voltage_lower_V')} - "
                               f"{params.get('voltage_upper_V')}"),
        ('Electrolyte', params.get('electrolyte_composition')),
        ('Electrolyte volume (uL)', params.get('electrolyte_volume_uL')),
        ('Separator', params.get('separator_type')),
        ('Theoretical capacity (mAh/g)',
         params.get('theoretical_capacity_mAh_g')),
        ('Test start date', params.get('test_start_date')),
        # 'all cycles', not an empty field. A blank against a named row
        # reads as something the pipeline failed to record, when it means
        # the opposite: no cap was asked for and none was applied.
        ('Max cycle analysed', params.get('max_cycle') or 'all cycles'),
    ]
    decided = dict(decided or {})
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, 'dataset_info.txt'), 'w',
              encoding='utf-8') as f:
        f.write("Ratatosk dataset record\n")
        f.write("=" * 58 + "\n")
        for k, v in rows:
            if isinstance(v, float):
                v = f"{v:.4g}"
            f.write(f"{k:<30} {'' if v is None else v}\n")
        f.write("=" * 58 + "\n")
        if decided:
            f.write("\nWhat the pipeline measured and then used\n")
            f.write("-" * 58 + "\n")
            for k, v in decided.items():
                if isinstance(v, float):
                    v = f"{v:.4g}"
                f.write(f"{k:<30} {'' if v is None else v}\n")
            f.write("-" * 58 + "\n")
        f.write("Full parameters and checksums: ../run_manifest.json\n")


def _classify(filename):
    low = filename.lower()
    for folder, keys in FOLDER_RULES:
        if any(k in low for k in keys):
            return folder
    return '9_other'


def organise_run(run_dir=None, dataset_names=None, user_parameters=None,
                 verbose=True, manifest=None, decided=None):
    """
    Sort a flat run folder into per-dataset, per-stage subfolders.

    Idempotent: only files sitting at the top level are moved, so running
    it twice is harmless and running it mid-analysis then again at the end
    simply picks up whatever arrived in between.
    """
    if not run_dir or not os.path.isdir(run_dir):
        print("  No run folder to organise.")
        return {}
    # 1.8.7 read `user_parameters` out of the notebook's globals. In a package
    # there is no such namespace, so it is a parameter — and without it every
    # output lands in `_comparative/` because no filename matches a dataset.
    all_params = dict(user_parameters or {})
    names = list(dataset_names or all_params.keys())
    names.sort(key=len, reverse=True)   # longest prefix wins
    used = set()
    folder_for = {nm: _dataset_folder(nm, all_params.get(nm, {}), used)
                  for nm in sorted(names)}

    moved, longest = {}, 0
    for fn in sorted(os.listdir(run_dir)):
        src = os.path.join(run_dir, fn)
        if not os.path.isfile(src) or fn == 'run_manifest.json':
            continue
        owner, stem = COMPARATIVE_DIR, fn
        for nm in names:
            if fn.startswith(nm):
                owner, stem = folder_for[nm], fn[len(nm):].lstrip('_')
                break
        # One cell loaded means the run-wide figures are that cell's own.
        if owner == COMPARATIVE_DIR and len(names) == 1:
            owner = folder_for[names[0]]
        dest_dir = os.path.join(run_dir, owner, _classify(stem))
        os.makedirs(dest_dir, exist_ok=True)
        dest = os.path.join(dest_dir, fn)
        if os.path.abspath(src) == os.path.abspath(dest):
            continue
        if os.path.exists(dest):
            os.remove(dest)
        shutil.move(src, dest)
        longest = max(longest, len(os.path.abspath(dest)))
        moved.setdefault(os.path.join(owner, _classify(stem)), []).append(fn)

    # Mirror the run-wide figures into each cell folder.
    comp_root = os.path.join(run_dir, COMPARATIVE_DIR)
    if COPY_COMPARATIVE_TO_CELLS and len(names) > 1 \
            and os.path.isdir(comp_root):
        n_copied = 0
        for stage in sorted(os.listdir(comp_root)):
            stage_dir = os.path.join(comp_root, stage)
            if not os.path.isdir(stage_dir):
                continue
            for fn in sorted(os.listdir(stage_dir)):
                src_f = os.path.join(stage_dir, fn)
                if not os.path.isfile(src_f):
                    continue
                for nm in names:
                    tgt = os.path.join(run_dir, folder_for[nm], stage)
                    os.makedirs(tgt, exist_ok=True)
                    # Some run-wide figures are already called
                    # comparative_*.png, and blindly prefixing gave
                    # comparative_comparative_discharge_capacity.png.
                    stem = (fn if fn.startswith(COMPARATIVE_COPY_PREFIX)
                            else COMPARATIVE_COPY_PREFIX + fn)
                    out = os.path.join(tgt, stem)
                    if not os.path.exists(out):
                        shutil.copy2(src_f, out)
                        n_copied += 1
                        # These are the LONGEST paths a run writes — a
                        # nested cell folder plus a 12-character prefix —
                        # and they were the ones the 260-character warning
                        # never measured.
                        longest = max(longest, len(os.path.abspath(out)))
        if verbose and n_copied:
            print(f"      copied {n_copied} run-wide figure(s) into the "
                  f"{len(names)} cell folder(s), prefixed "
                  f"'{COMPARATIVE_COPY_PREFIX}'")

    for nm in names:
        d = os.path.join(run_dir, folder_for[nm])
        if os.path.isdir(d):
            # The manifest is passed in. `globals().get('RUN_MANIFEST')`
            # was a leftover of the notebook-globals pattern this module
            # exists to remove: run.py's own globals never hold it, so every
            # dataset_info.txt said "Source file: (not captured)".
            write_dataset_info(d, nm, all_params.get(nm, {}), manifest,
                               decided=(decided or {}).get(nm))

    if verbose and moved:
        print(f"\n  Sorted {sum(len(v) for v in moved.values())} file(s):")
        for folder in sorted(moved):
            print(f"      {folder:<44} {len(moved[folder])} file(s)")
    if longest > 240 and verbose:
        print(f"      WARNING: longest output path is {longest} "
              f"characters. Windows fails above 260; shorten the "
              f"composition or move the output folder nearer the "
              f"drive root.")
    return moved


def finalise_run(run_dir=None, manifest=None, verbose=True, organise=True,
                 flags=None, user_parameters=None, decided=None,
                 detection=None):
    """
    Close the manifest: inventory every file written, with checksums.

    Run this after the last analysis cell. Safe to run more than once; it
    simply refreshes the inventory. An empty file is recorded as such rather
    than merely present, because a zero-byte CSV passes an existence check
    while containing nothing.
    """
    run_dir = run_dir or globals().get('RUN_DIR')
    manifest = manifest if manifest is not None else globals().get('RUN_MANIFEST')
    if not run_dir or manifest is None:
        print("  No active run to finalise.")
        return None

    if organise:
        organise_run(run_dir, user_parameters=user_parameters,
                     verbose=verbose, manifest=manifest, decided=decided)

    files = []
    for root, _dirs, fns in os.walk(run_dir):
        for fn in sorted(fns):
            if fn == 'run_manifest.json':
                continue
            fp = os.path.join(root, fn)
            size = os.path.getsize(fp)
            files.append({
                'name': fn,
                'path': os.path.relpath(fp, run_dir).replace(os.sep, '/'),
                'bytes': size, 'empty': size == 0, 'sha256': _sha256(fp)})

    manifest['outputs'] = files
    manifest['finished_utc'] = datetime.now(timezone.utc).isoformat()
    manifest['status'] = 'complete'

    # Re-read the flags at the end. In 1.8.7 this mattered because start_run
    # ran before Modules 1, 1b, 3, 3b and 4 existed and could not see their
    # constants. In 1.9.0 `default_flags()` reads them from the modules, so
    # both readings are complete — but a value the notebook CHANGED mid-run
    # still differs, and the one that was in force at the end is the one that
    # produced the outputs. Anything changed since start_run is kept alongside
    # under the name it had.
    _late = (flags if flags is not None else default_flags())
    _early = manifest.get('pipeline_flags', {}) or {}
    for k, v in _early.items():
        if k in _late and _late[k] != v:
            _late[f'{k}__at_start'] = v
    manifest['pipeline_flags'] = _late

    # Same again for package versions, and for the same reason: a library
    # installed or upgraded mid-run is invisible to the probe at start_run.
    _pkg_late = _package_versions()
    _pkg_early = manifest.get('packages', {}) or {}
    for k, v in _pkg_early.items():
        if k in _pkg_late and _pkg_late[k] != v:
            _pkg_late[f'{k}__at_start'] = v
    manifest['packages'] = _pkg_late

    if 'datasets' in manifest and not manifest['datasets'] and user_parameters:
        # From the ARGUMENT. Reading it from globals() recorded
        # "datasets": {} in every run_manifest.json ever written.
        manifest['datasets'] = dict(user_parameters)
    # The reference cycle is a per-dataset choice with dataset-wide reach, so
    # record which one was used and why. Read-only; nothing is mutated.
    try:
        # FROM THE ARGUMENT — the same correction the comment eleven lines
        # above records for `datasets`, written again immediately below it.
        # `detected_peaks` is never a module global here, so `_det` was
        # always {} and `reference_cycle_used` reached no manifest ever
        # written. The reference cycle decides what every tracked feature is
        # measured against; it belongs in the record of the run.
        _det = detection or {}
        for _n, _d in _det.items():
            _entry = manifest.setdefault('datasets', {}).setdefault(_n, {})
            if not isinstance(_entry, dict):
                continue
            _rc = (_d.get('reference_cycle') if isinstance(_d, dict)
                   else getattr(_d, 'reference_cycle', None))
            _rr = (_d.get('reference_cycle_reason') if isinstance(_d, dict)
                   else getattr(_d, 'reference_cycle_reason', None))
            if _rc is not None:
                _entry['reference_cycle_used'] = _rc
            if _rr:
                _entry['reference_cycle_reason'] = _rr
    except Exception:
        pass
    path = write_manifest(run_dir, manifest)

    if verbose:
        empties = [f['name'] for f in files if f['empty']]
        print(f"\n  Run finalised: {len(files)} output file(s) recorded")
        print(f"  Manifest: {path}")
        if empties:
            print(f"  WARNING: {len(empties)} file(s) written empty: "
                  f"{', '.join(empties[:5])}"
                  f"{' ...' if len(empties) > 5 else ''}")
    return path


def freeze_run(run_dir=None, label='', base_location=None):
    """
    Copy a run into published/ so it can never be overwritten or re-run over.

    Use this the moment a figure goes into a thesis or a paper. The frozen
    copy keeps its manifest, so the provenance travels with it.
    """
    run_dir = run_dir or globals().get('RUN_DIR')
    base = base_location or globals().get('BASE_SAVE_LOCATION')
    if not run_dir or not base:
        print("  Nothing to freeze.")
        return None
    tag = f"{os.path.basename(run_dir)}{('_' + label) if label else ''}"
    dest = os.path.join(base, FREEZE_ROOT_NAME, tag)
    if os.path.exists(dest):
        print(f"  Already frozen: {dest}")
        return dest
    shutil.copytree(run_dir, dest)
    with open(os.path.join(dest, 'README.txt'), 'w', encoding='utf-8') as f:
        f.write(f"Frozen Ratatosk run\n"
                f"Label: {label or '(none)'}\n"
                f"Source run: {run_dir}\n"
                f"Frozen: {datetime.now().isoformat(timespec='seconds')}\n\n"
                f"Do not overwrite or re-run into this folder. Provenance is "
                f"in run_manifest.json.\n")
    print(f"  Frozen to: {dest}")
    return dest


def dataset_folders(all_params, dataset_names=None):
    """
    The folder `organise_run` gives EVERY dataset, as `{name: folder}`.

    Ask for all of them at once, because the answer for one depends on the
    others: two cells sharing a composition and a cell id collide, and
    `_dataset_folder` resolves that by appending _2, _3 — but only when it is
    told what is already taken. The old single-dataset wrapper never passed
    that set, so `organise_run` created X and X_2 while the notebook wrote
    both START_HERE reports into X, and the second silently replaced the
    first beside the other cell's figures.
    """
    # SORTED, BECAUSE `organise_run` SORTS.
    # `_dataset_folder` appends _2 to whichever colliding name it sees SECOND,
    # so the suffix depends entirely on iteration order. `organise_run` builds
    # its map over `sorted(names)`; this one built it over the caller's list
    # order, and the notebook passes `list(datasets)` — load order. Two cells
    # sharing a composition with no cell_id therefore got OPPOSITE folder
    # names from the two functions, and each cell's report was written beside
    # the other cell's figures. That is the exact failure this function's
    # docstring says it exists to prevent, reintroduced by the one line that
    # differs between the two.
    names = sorted(dataset_names if dataset_names is not None else all_params)
    used = set()
    return {n: _dataset_folder(n, (all_params or {}).get(n, {}) or {}, used)
            for n in names}


def dataset_folder(name, params=None, *, used=None):
    """One dataset's folder. Prefer `dataset_folders` — see why there."""
    return _dataset_folder(name, params or {}, used)


In [ ]:
# DEFINITIONS L — THE WAY IN: START_HERE AND THE RUN SUMMARY
# =============================================================================
"""
The first thing a newcomer should open.

A finished run is forty-six files across seven folders. That is the right way
to STORE the output and the wrong way to hand it to somebody: "here are 45
well-organised files, good luck" is not a briefing. This module writes two more
files at the top of the run folder, above the numbered directories:

    START_HERE.html   self-contained, images embedded, safe to email
    START_HERE.md     the same content as text, to paste into a report

Both say the same four things, in this order:

    1. What this cell is, and what it did
    2. What went wrong, if anything, and on which cycles
    3. What Ratatosk declined to tell you, and why
    4. Four figures, in order, with what to look for in each

Everything else stays exactly where it is. This is a way in, not a replacement.

Why the third section exists
----------------------------
The withheld numbers are the ones a newcomer is most likely to misread, because
absence looks like an oversight rather than a decision. A capacity share left
blank because the closure interval was 0.46 wide is a finding; found as a gap
in a spreadsheet three weeks later it is a bug report.
"""


import base64
import os
import html
import re

import numpy as np
import pandas as pd


# Which four, and what to say about each. Ordered as a reader should meet them:
# does the cell work, is it dying, what is the mechanism, is the mechanism
# moving. Each entry is (folder, filename suffix, title, what to look for).
FIGURE_GUIDE = [
    ("1_cycling", "_cycle_life.png", "Does the cell work?",
     "Capacity and coulombic efficiency against cycle number. CE should sit "
     "at or just under 100% from a few cycles in. A first cycle well below "
     "that is formation; a later dip is charge going somewhere it should not, "
     "and section 2 above says where."),
    # RUN-WIDE. `capacity_retention` draws ONE figure for the whole run, so
    # there is no per-cell file to find; `organise_run` mirrors it into each
    # cell folder as `comparative_capacity_retention_vs_cycle.png`. `_find`
    # skips `comparative_` files on purpose — a group figure must not stand in
    # where a cell's own was asked for — so this tile resolved to nothing and
    # every START_HERE report in every run has said "(figure not found)"
    # against question 2 of 4.
    ("1_cycling", "capacity_retention_vs_cycle.png", "Is it dying, and how fast?",
     "Discharge capacity as a percentage of the reference cycle. A straight "
     "decline is ordinary ageing. A cliff is an event — cross-reference the "
     "cycle number against section 2.", True),
    ("4_dqdv", "_dQdV_waterfall_Discharge.png", "What is the mechanism?",
     "Every cycle's differential capacity, stacked. Each peak is a redox "
     "process. A peak that shifts is polarising; one that broadens is losing "
     "kinetics; one that shrinks is losing the material behind it. This is "
     "the figure that says WHY the capacity curve does what it does."),
    ("6_descriptors", "_peak_trends_Discharge.png", "Is the mechanism moving?",
     "Each tracked peak's centre and area against cycle number. A redox peak "
     "at C/10 should drift a few mV per cycle. Tens of mV per cycle is a "
     "fitting artefact, not chemistry — the coherence table in "
     "5_peak_fitting says which peaks passed that test."),
]

_CSS = """
:root{--ink:#1a1a1a;--soft:#5a5a5a;--rule:#dcdcdc;--bg:#ffffff;
      --ok:#0a7d3c;--warn:#b06000;--bad:#a4262c;--tint:#f5f7f9;--accent:#0072B2}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);
     font:16px/1.6 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif}
.wrap{max-width:940px;margin:0 auto;padding:48px 28px 96px}
h1{font-size:30px;line-height:1.2;margin:0 0 4px;letter-spacing:-.02em}
.sub{color:var(--soft);font-size:15px;margin:0 0 32px}
h2{font-size:13px;text-transform:uppercase;letter-spacing:.09em;
   color:var(--soft);font-weight:700;margin:44px 0 14px;
   padding-bottom:8px;border-bottom:1px solid var(--rule)}
h3{font-size:18px;margin:0 0 6px;letter-spacing:-.01em}
p{margin:0 0 12px}
.lede{font-size:18px;line-height:1.55}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));
      gap:14px;margin:18px 0 8px}
.stat{background:var(--tint);border-radius:10px;padding:14px 16px}
.stat .k{font-size:12px;color:var(--soft);text-transform:uppercase;
         letter-spacing:.06em}
.stat .v{font-size:24px;font-weight:650;letter-spacing:-.02em;margin-top:2px}
.stat .n{font-size:12px;color:var(--soft);margin-top:2px}
.flag{border-left:3px solid var(--rule);padding:12px 16px;margin:12px 0;
      background:var(--tint);border-radius:0 8px 8px 0}
.flag.ok{border-color:var(--ok)} .flag.warn{border-color:var(--warn)}
.flag.bad{border-color:var(--bad)}
.flag b{display:block;margin-bottom:3px}
.fig{margin:30px 0 38px}
.fig img{width:100%;height:auto;border:1px solid var(--rule);border-radius:8px;
         background:#fff}
.fig .cap{color:var(--soft);font-size:14.5px;margin-top:10px}
.n{color:var(--soft);font-size:14px}
table{border-collapse:collapse;width:100%;font-size:14.5px;margin:8px 0 4px}
th,td{text-align:left;padding:7px 10px;border-bottom:1px solid var(--rule)}
th{font-size:12px;text-transform:uppercase;letter-spacing:.06em;color:var(--soft)}
td.num{text-align:right;font-variant-numeric:tabular-nums}
code{background:var(--tint);padding:1px 5px;border-radius:4px;font-size:13.5px}
footer{margin-top:56px;padding-top:16px;border-top:1px solid var(--rule);
       color:var(--soft);font-size:13px}
@media (prefers-color-scheme:dark){
  :root{--ink:#e8e8e8;--soft:#a0a0a0;--rule:#333;--bg:#141414;--tint:#1e1e1e;
        --ok:#4ec97e;--warn:#e0a04a;--bad:#e07a80}
  .fig img{background:#fff}
}
"""


# THE FINDINGS ARE WRITTEN IN MARKDOWN. Both renderers share them, and the
# HTML one only escaped — so the page the docstring calls "safe to email"
# printed literal `**mixed**` and backticks, a dozen times per cell. Applied
# AFTER `_esc`, so the input is already HTML-safe and this only re-introduces
# the two tags the findings actually use.
def _md_inline(t):
    t = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", str(t), flags=re.S)
    return re.sub(r"`([^`]+?)`", r"<code>\1</code>", t)


_MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
         ".svg": "image/svg+xml", ".gif": "image/gif", ".webp": "image/webp"}


def _embed(path):
    """Inline a figure as a data URI, with the MIME type its extension says."""
    mime = _MIME.get(os.path.splitext(path)[1].lower())
    if mime is None:
        # TIFF and PDF exports cannot be shown in a browser. Say so in the
        # report rather than emitting a data URI a browser will not render.
        return None
    try:
        with open(path, "rb") as fh:
            return (f"data:{mime};base64,"
                    + base64.b64encode(fh.read()).decode("ascii"))
    except OSError:
        return None


def _esc(x):
    """Every field interpolated into the HTML goes through this.

    Compositions legitimately contain characters that are markup: `Na0.67
    [Ni0.25Mn0.75]O2` is fine, `<Ni>` and `A&B` are not, and an apostrophe
    inside an alt attribute closes it.
    """
    return html.escape("" if x is None else str(x), quote=True)


def _md_cell(x):
    """A value safe to drop into a markdown TABLE cell."""
    return ("" if x is None else str(x)).replace("|", "\\|").replace("\n", " ")


def _rel(path, start):
    """A relative path a markdown viewer will follow.

    `os.path.relpath` returns backslashes on Windows, and a backslash in a
    markdown link is an escape character, so every embedded figure link was
    dead on the platform this is actually run on.
    """
    return os.path.relpath(path, start).replace(os.sep, "/")


def _trailing_partial(t, dcol, in_progress):
    """The cycle whose export caught a half-cycle mid-flight, from `in_progress`.

    `analyse.half_cycles_in_progress` is the authority and the only one:
    it asks whether the half-cycle reached the cut-off VOLTAGE the cycler
    terminates on. There is no capacity fallback, because capacity cannot
    answer that question — on P3 cell B an 80%-of-the-previous rule fires
    15 times on a cell that is simply dying, and misses a half-cycle caught
    four fifths of the way through.

    Returns (cycle, step, previous_cycle, capacity_so_far) or None.
    """
    if not in_progress:
        return None
    cyc, step = sorted(in_progress)[0]
    prev = val = None
    if dcol is not None and "Cycle" in t:
        d = t.dropna(subset=["Cycle"])
        earlier = d[d["Cycle"] < cyc].dropna(subset=[dcol])
        if len(earlier):
            prev = int(earlier["Cycle"].iloc[-1])
        here = pd.to_numeric(d.loc[d["Cycle"] == cyc, dcol], errors="coerce")
        if len(here) and np.isfinite(here.iloc[0]):
            val = float(here.iloc[0])
    return cyc, step, prev, val


def _find(folder, suffix, name, run_wide=False):
    """The figure for this dataset, whether or not its filename is prefixed.

    `run_wide` says this guide entry names a figure that is drawn ONCE for
    the whole run rather than per cell — capacity retention is the only one.
    Without it the comparative-skip below refuses the only candidate there
    is, and the tile renders "(figure not found)".
    """
    if not os.path.isdir(folder):
        return None
    exact = os.path.join(folder, f"{name}{suffix}")
    if os.path.isfile(exact):
        return exact
    plain = os.path.join(folder, suffix)
    if os.path.isfile(plain):
        return plain
    if run_wide:
        mirrored = os.path.join(folder, f"comparative_{suffix}")
        if os.path.isfile(mirrored):
            return mirrored
    for f in sorted(os.listdir(folder)):
        # A run-wide figure is mirrored into every cell folder with a
        # 'comparative_' prefix. Matching one here would show the group
        # figure where the cell's own was asked for — unless the guide has
        # said this entry IS the group figure.
        if f.startswith("comparative_") and not run_wide:
            continue
        if f.endswith(suffix.lstrip("_")) or f.endswith(suffix):
            return os.path.join(folder, f)
    return None


def _facts(name, params, integrity, cycle_table, fit_limit, detection,
           tracking, attribution, closure, profile, in_progress=None,
           coherence=None, resolvability=None, parameters=None):
    """Everything both renderers need, computed once."""
    f = {"name": name, "params": params or {}}
    # THE MECHANISM VERDICT. `quality.assess_resolvability` decides, before a
    # single production fit is run, what produced this curve and therefore
    # which model to fit it with — and until 1.9.0.47 it said so only in the
    # console. It is the single most important thing this build does
    # differently from 1.8.7 and neither document mentioned it, which is the
    # exact defect class the reporting audit was about: a verdict computed and
    # not carried to where a person reads.
    f["resolvability"] = dict(resolvability or {})
    # THE FIT-QUALITY NUMBERS, one row per half-cycle. R2 is dominated by the
    # bulk of a curve; these are the three ways a fit goes wrong that it
    # cannot carry. See `fitting.fit_quality`.
    f["fit_quality"] = {}
    if parameters is not None and not getattr(parameters, "empty", True):
        _pp = parameters
        _cols = ["height_ratio", "overshoot", "max_residual_frac",
                 "residual_runs_z", "r_squared"]
        if all(c in getattr(_pp, "columns", []) for c in _cols):
            _hc = _pp.groupby(["step", "cycle"])[_cols].first().reset_index()
            for _st, _g in _hc.groupby("step"):
                f["fit_quality"][str(_st)] = {
                    c: float(_g[c].median()) for c in _cols}
                f["fit_quality"][str(_st)]["n"] = int(len(_g))
                f["fit_quality"][str(_st)]["height_worst"] = float(
                    _g["height_ratio"].min())
                f["fit_quality"][str(_st)]["height_tallest"] = float(
                    _g["height_ratio"].max())
                # ...and WHERE, because "one half-cycle is at 0.57" is only
                # actionable with a cycle number beside it.
                _hh = _g.dropna(subset=["height_ratio"])
                if not _hh.empty:
                    f["fit_quality"][str(_st)]["height_worst_cycle"] = int(
                        _hh.loc[_hh["height_ratio"].idxmin(), "cycle"])
                    f["fit_quality"][str(_st)]["height_tallest_cycle"] = int(
                        _hh.loc[_hh["height_ratio"].idxmax(), "cycle"])
                    f["fit_quality"][str(_st)]["n_short"] = int(
                        (_hh["height_ratio"] < 0.85).sum())
                    f["fit_quality"][str(_st)]["n_tall"] = int(
                        (_hh["height_ratio"] > 1.15).sum())
                _rr = _g.dropna(subset=["r_squared"])
                if not _rr.empty:
                    f["fit_quality"][str(_st)]["r2_worst"] = float(
                        _rr["r_squared"].min())
                    f["fit_quality"][str(_st)]["r2_worst_cycle"] = int(
                        _rr.loc[_rr["r_squared"].idxmin(), "cycle"])
    # THE LINESHAPE. A sharp two-phase peak is asymmetric and until 1.9.0.51
    # the model could not be; where the split shape was fitted, the ratio it
    # settled on is a measurement and belongs on the page — including when it
    # settled on a bound, which is not one.
    f["lineshape"] = {}
    if parameters is not None and not getattr(parameters, "empty", True):
        _pp = parameters
        if all(c in getattr(_pp, "columns", [])
               for c in ("asymmetry_k", "asymmetry_at_bound")):
            _hc = (_pp.groupby(["step", "cycle"])[["asymmetry_k",
                                                   "asymmetry_at_bound"]]
                   .first().reset_index())
            for _st, _g in _hc.groupby("step"):
                _k = _g["asymmetry_k"].dropna()
                if _k.empty:
                    continue
                # ITEM 35. A median ratio of "19.3x" was quoted over a range
                # of 0.05-3.12 — which crosses 1.0, so one half-cycle leans
                # the other way and the median states a direction the data
                # does not hold. Count how many agree with the median's sign.
                _same = int(((_k > 1.0) == (_k.median() > 1.0)).sum())
                f["lineshape"][str(_st)] = dict(
                    n=int(len(_g)), k=float(_k.median()),
                    k_lo=float(_k.min()), k_hi=float(_k.max()),
                    same_sign=_same, n_k=int(len(_k)),
                    at_bound=int(_g["asymmetry_at_bound"].sum()),
                    clipped=int(_g["asymmetry_k_clipped"].sum())
                    if "asymmetry_k_clipped" in _g else 0,
                    split=bool((_k - 1.0).abs().median() > 0.02))
            # ...and how often the NARROW side of the LARGEST component in a
            # half-cycle sits on the width floor. A split lineshape can put
            # one flank below what the histogram samples, and on LTO
            # discharge it does so on the component carrying all of the area.
        # HOW MANY BINS THE WIDTH IS MEASURED ACROSS. A ratio of two widths
        # is only as good as the widths, and on a sharp profile they are read
        # off three or four bins. See `signal.HISTOGRAM_MIN_OCCUPANCY` for the
        # sweep: on LTO the fitted FWHM tracks the bin width almost linearly
        # over a factor of ten, so it is the histogram's width, not the peak's.
        if {"fwhm", "sample_mV"} <= set(_pp.columns):
            _b = (pd.to_numeric(_pp["fwhm"], errors="coerce") * 1000.0
                  / pd.to_numeric(_pp["sample_mV"], errors="coerce"))
            for _st, _gb in _pp.assign(_bins=_b).groupby("step"):
                if str(_st) not in f["lineshape"]:
                    continue
                _v = _gb["_bins"].replace([np.inf, -np.inf], np.nan).dropna()
                if _v.empty:
                    continue
                f["lineshape"][str(_st)].update(
                    fwhm_bins=float(_v.median()),
                    n_comp=int(len(_gb)),
                    undersampled=int(_gb["width_undersampled"].fillna(False).sum())
                    if "width_undersampled" in _gb else 0)
        if {"at_sigma_floor", "amplitude_area"} <= set(_pp.columns):
                _dom = (_pp.assign(_a=_pp["amplitude_area"].abs())
                        .sort_values("_a")
                        .groupby(["step", "cycle"]).last().reset_index())
                for _st, _g2 in _dom.groupby("step"):
                    if str(_st) in f["lineshape"]:
                        f["lineshape"][str(_st)]["dominant_at_floor"] = int(
                            _g2["at_sigma_floor"].fillna(False).sum())
    # HOW MANY COMPONENTS DID EACH HALF-CYCLE GET, AND HOW MANY DID THE
    # REFERENCE CYCLE GET? Components per half-cycle swing 1-5 on NMC111 with
    # no pattern, and the reference cycle got the THINNEST fit in the dataset
    # (2) with a roster of 7 hung off it — every tracked feature, every drift
    # rate and the whole attribution basis derived from the one half-cycle
    # the model described least well. The reference cycle is chosen on the
    # cell's INTEGRITY, which is the right criterion and should not be traded
    # for a fatter fit; but a reader is entitled to know when the basis is
    # thin, and nothing said so.
    # SET IT BEFORE IT IS READ. `f["reference_cycle"]` was assigned ~270
    # lines below this block, so `_ref` here was ALWAYS None and the whole
    # warn branch — the one the paragraph above exists to explain, saying the
    # basis is the thinnest fit in the dataset — could never fire on any
    # dataset. The heading was permanently the benign one.
    f["reference_cycle"] = getattr(detection, "reference_cycle", None)
    f["component_census"] = {}
    if parameters is not None and not getattr(parameters, "empty", True):
        _pp = parameters
        if {"cycle", "step"} <= set(getattr(_pp, "columns", [])):
            _n = (_pp.groupby(["step", "cycle"]).size()
                  .rename("n").reset_index())
            _ref = f.get("reference_cycle")
            for _st, _g in _n.groupby("step"):
                _med = float(_g["n"].median())
                _refrow = (_g[_g["cycle"] == _ref] if _ref is not None
                           else _g.iloc[0:0])
                f["component_census"][str(_st)] = dict(
                    n_half_cycles=int(len(_g)),
                    lo=int(_g["n"].min()), hi=int(_g["n"].max()),
                    median=_med,
                    reference_cycle=(int(_ref) if _ref is not None else None),
                    reference_n=(int(_refrow.iloc[0]["n"])
                                 if not _refrow.empty else None))
    # ITEM 37. A HALF-CYCLE WITH NO PRIMARY PEAK AT ALL. `primaries_only`
    # drops shoulders, but guards with "keep them if there are no primaries" —
    # so when the picker finds no resolved maximum the whole half-cycle is
    # described by features it called shoulders. 19 of 60 on NMC111, every one
    # a discharge, from cycle 4 onward as the curve flattens, and the two
    # worst fits in that run were among them. Nothing said so.
    f["shoulder_only"] = {}
    if parameters is not None and not getattr(parameters, "empty", True):
        _pp = parameters
        if {"cycle", "step", "is_shoulder"} <= set(getattr(_pp, "columns", [])):
            _so = (_pp.groupby(["step", "cycle"])["is_shoulder"]
                   .all().reset_index(name="all_shoulder"))
            for _st, _g in _so.groupby("step"):
                _bad = _g[_g["all_shoulder"]]
                if len(_bad):
                    f["shoulder_only"][str(_st)] = dict(
                        n=int(len(_bad)), total=int(len(_g)),
                        first=int(_bad["cycle"].min()),
                        last=int(_bad["cycle"].max()))
    # HOW MUCH DATA EACH FIT HAD. Reduced to a handful of numbers here rather
    # than carrying the whole parameter table into the facts dict, which both
    # renderers copy.
    f["identifiability"] = {}
    if parameters is not None and not getattr(parameters, "empty", True):
        _pp = parameters
        _need = ("n_points", "nvarys", "points_per_parameter")
        _key = [c for c in ("cycle", "step") if c in getattr(_pp, "columns", [])]
        if _key and all(c in _pp.columns for c in _need):
            _H = _pp.drop_duplicates(subset=_key)
            _ppp = pd.to_numeric(_H["points_per_parameter"],
                                 errors="coerce").dropna()
            if not _ppp.empty:
                _npt = pd.to_numeric(_H["n_points"], errors="coerce").dropna()
                _nvy = pd.to_numeric(_H["nvarys"], errors="coerce").dropna()
                _cpl = pd.to_numeric(_H.get("n_shoulders_coupled"),
                                     errors="coerce").dropna()
                f["identifiability"] = dict(
                    n_half_cycles=int(len(_H)),
                    median=float(_ppp.median()),
                    worst=float(_ppp.min()),
                    n_thin=int((_ppp < IDENTIFIABILITY_POINTS_PER_PARAM).sum()),
                    median_points=(float(_npt.median()) if len(_npt)
                                   else float("nan")),
                    median_varied=(float(_nvy.median()) if len(_nvy)
                                   else float("nan")),
                    coupled=(int(_cpl.sum()) if len(_cpl) else 0))

    # WHAT `reliable` MEANS ON THIS CELL, broken out. One boolean was
    # carrying four different statements — see the note in
    # `analyse.parameters_frame` — and the page never printed any of them.
    # How many components the anomalous-half-cycle gate actually struck, so
    # the over-theoretical finding can say whether it did anything here.
    f["n_anomalous_components"] = int(
        parameters["half_cycle_anomalous"].fillna(False).astype(bool).sum()
        if (parameters is not None and not getattr(parameters, "empty", True)
            and "half_cycle_anomalous" in getattr(parameters, "columns", []))
        else 0)

    f["reliability"] = {}
    if parameters is not None and not getattr(parameters, "empty", True):
        _pp = parameters
        if "reliable" in getattr(_pp, "columns", []):
            _rel = _pp["reliable"].fillna(False).astype(bool)
            _r = dict(n=int(len(_pp)), ok=int(_rel.sum()))
            if "reliability_reason" in _pp.columns:
                _r["reasons"] = {
                    str(k): int(v) for k, v in
                    _pp.loc[~_rel, "reliability_reason"]
                    .value_counts().items()}
            if "area_determinacy" in _pp.columns:
                _r["determinacy"] = {
                    str(k): int(v) for k, v in
                    _pp["area_determinacy"].value_counts().items()}
            # AREAS THAT ARE LOWER BOUNDS. `area_is_lower_bound` has been in
            # the parameter table for several builds and read by nothing.
            # The count that matters is not how many components are
            # truncated — it is how many are truncated AND marked reliable,
            # because those are the ones whose areas a reader will quote.
            _out = pd.to_numeric(_pp.get("area_outside_window_frac"),
                                 errors="coerce")
            if _out is not None and _out.notna().any():
                _lb = (_pp["area_is_lower_bound"].fillna(False).astype(bool)
                       if "area_is_lower_bound" in _pp.columns
                       else pd.Series(False, index=_pp.index))
                _mat = _lb & (_out > AREA_LOWER_BOUND_REPORTABLE).fillna(False)
                _r["lower_bound"] = int(_mat.sum())
                _r["lower_bound_reliable"] = int((_mat & _rel).sum())
                _r["lower_bound_worst"] = (float(_out[_mat].max())
                                           if int(_mat.sum()) else 0.0)
                _r["lower_bound_median"] = (float(_out[_mat & _rel].median())
                                            if int((_mat & _rel).sum())
                                            else 0.0)
            f["reliability"] = _r

    # THE CELL-LEVEL VERDICT. The page counted anomalous half-cycles and
    # stopped; what it never said is whether they RECUR. See
    # `analyse.cell_integrity_verdict`.
    # IMPORTED AT MODULE SCOPE, not here. A relative import inside a function
    # body survives verbatim into the flattened notebook, where there is no
    # package for it to resolve against — and the `try/except` that used to
    # wrap it turned that ImportError into silence. The verdict vanished from
    # every notebook run and nothing said so. The flattener rewrites top-level
    # imports; `build_inline` now refuses a relative import anywhere else.
    f["cell_integrity"] = cell_integrity_verdict(
        getattr(integrity, "table", integrity))
    p = f["params"]
    # `or name`, not `get(..., name)`: a composition recorded as "" — which
    # the run summary already handles explicitly — gave a title of "— cell A".
    f["title"] = (f"{p.get('composition') or name}"
                  + (f" — cell {p['cell_id']}" if p.get("cell_id") else ""))

    # --- capacity and retention -----------------------------------------
    # Reported at the last cycle the cell was still WORKING, not the last row
    # of the table. A record that runs 90 cycles past the cell's death ends at
    # 0 mAh/g and 0% retention, and reporting that as "the final capacity" is
    # true of the file and false about the cell.
    disc = ret = ce = None
    f["n_cycles"] = f["last_good_cycle"] = None
    f["partial_final"] = None
    f["in_progress"] = []
    if cycle_table is not None and not cycle_table.empty:
        t = cycle_table.copy()
        t["Cycle"] = pd.to_numeric(t["Cycle"], errors="coerce")
        # `cycling_summary` returns a PER-CELL table whose columns are bare
        # ('Discharge_mAh_g'), while its combined export prefixes them with
        # the cell label ('LTO (Cell A)_Delithiation_mAh_g'). Matching only
        # the suffixed spelling found neither: "Discharge_mAh_g" does not
        # end with "_Discharge_mAh_g", so every capacity number was silently
        # dropped from START_HERE. On an anode the useful half-cycle is
        # labelled Delithiation or Desodiation, so that is looked for too.
        dcol = cycle_column(t, CAPACITY_COLUMN_ALIASES)
        # The per-cell table still uses the bare name; the combined export
        # carries the reference cycle in the column, e.g. Retention_vs_C2_%.
        rcol = cycle_column(t, ("Retention_%", "Retention_vs_C2_%"))
        # ...and read the reference OUT of that column name, so the page can
        # say "99% of cycle 2" instead of "of the reference cycle" three
        # lines under a column headed "% of first".
        _m = re.search(r"_vs_C(\d+)_", str(rcol or ""))
        f["retention_reference"] = int(_m.group(1)) if _m else None
        ccol = cycle_column(t, ("CE_%",))
        icol = cycle_column(t, ("Incomplete",))
        _cmax = t["Cycle"].max()
        f["n_cycles"] = int(_cmax) if pd.notna(_cmax) else None
        good = t[t[icol] != True] if icol else t          # noqa: E712
        # A half-cycle the export caught mid-flight is not a measurement.
        f["in_progress"] = sorted(in_progress or ())
        f["partial_final"] = _trailing_partial(good, dcol, f["in_progress"])
        if f["partial_final"]:
            good = good[good["Cycle"] < f["partial_final"][0]]
        if dcol:
            d = pd.to_numeric(good[dcol], errors="coerce")
            # "Still working" uses the SAME rule as `analyse.last_useful_cycle`
            # — 2% of the 90th-percentile discharge capacity — applied to the
            # table, rather than relying on a limit passed in. A `d > 0` test
            # is not enough: the dead cycles here deliver 0.01 mAh/g, which is
            # greater than zero and is not a working cell.
            good = good[good["Cycle"].notna()]
            d = pd.to_numeric(good[dcol], errors="coerce")
            live = good[d.notna()]
            if len(live):
                dv = pd.to_numeric(live[dcol], errors="coerce")
                floor = 0.02 * float(np.percentile(dv.dropna(), 90))
                live = live[dv > floor]
            if fit_limit and len(live):
                live = live[live["Cycle"] <= fit_limit]
            if len(live):
                _lastc = int(live["Cycle"].iloc[-1])
                # A DEATH is a sustained collapse, not the end of the file.
                # Testing `_lastc < n_cycles` called the last cycle of every
                # record a death whenever that cycle was incomplete or
                # partial — which it usually is. The same run length as
                # `analyse.DEAD_CELL_RUN`: three consecutive dead cycles.
                _tail = t[(t["Cycle"] > _lastc) & t["Cycle"].notna()]
                _n_tail = len(_tail)
                _dead_tail = 0
                if _n_tail and dcol is not None:
                    _tv = pd.to_numeric(_tail[dcol], errors="coerce")
                    _dead_tail = int((_tv.isna() | (_tv <= floor)).sum())
                f["death_cycle"] = _lastc if _dead_tail >= 3 else None
                dv = pd.to_numeric(live[dcol], errors="coerce")
                disc = (float(dv.iloc[0]), float(dv.iloc[-1]))
                f["last_good_cycle"] = _lastc
                if rcol:
                    rv = pd.to_numeric(live[rcol], errors="coerce").dropna()
                    ret = float(rv.iloc[-1]) if len(rv) else None
                if ccol:
                    cv = pd.to_numeric(live[ccol], errors="coerce").dropna()
                    ce = float(cv[cv.between(1, 150)].median()) if len(cv) else None
    # Discharge capacity at the key cycles — the numbers that go in a paper.
    # First, the reference cycle, and whichever key cycles the cell reached.
    f["key_capacities"] = []
    if cycle_table is not None and not cycle_table.empty and disc:
        t2 = cycle_table.copy()
        t2["Cycle"] = pd.to_numeric(t2["Cycle"], errors="coerce")
        # Same lookup as above; the second copy of the old suffix-only
        # match survived the first fix and raised KeyError: None.
        dcol2 = cycle_column(t2, CAPACITY_COLUMN_ALIASES)
        if dcol2 is None:
            t2 = t2.iloc[0:0]
        want = [1] + list((params or {}).get("key_cycles") or [])
        if f.get("last_good_cycle"):
            want.append(f["last_good_cycle"])
        seen, base = set(), disc[0]
        for cyc in sorted({int(c) for c in want}):
            row = t2[t2["Cycle"] == cyc]
            if row.empty or cyc in seen:
                continue
            v = pd.to_numeric(row[dcol2], errors="coerce").iloc[0]
            if not np.isfinite(v) or v <= 0:
                continue
            if f.get("last_good_cycle") and cyc > f["last_good_cycle"]:
                continue
            seen.add(cyc)
            f["key_capacities"].append(
                (cyc, float(v), 100.0 * v / base if base else None))

    # IS THIS DATASET A MEASUREMENT AT ALL?
    # `JQ_NNM_C-rate_2-4.2V_A_29042026` is a 485 kB export against 3.4 MB for
    # its two siblings: two cycles, median CE 41.3%, two cycles passing more
    # charge than the material can hold. It received a full analysis, a full
    # report and a place in the replicate mean, and nothing anywhere said the
    # file was short or the efficiency impossible. It was caught by eye.
    #
    # The two tests below are DEFINITIONAL rather than tuned:
    #
    #   too few cycles   retention is measured against
    #                    `cycling.RETENTION_REFERENCE_CYCLE`; with fewer
    #                    cycles than that reference plus one there is nothing
    #                    to measure it against, so the figure cannot exist.
    #   CE below 50%     more charge is lost every cycle than comes back. That
    #                    is not a working cell, whatever else the file says.
    #
    # A dataset failing either still gets its own page — it is data, and the
    # page is where you look to find out what went wrong. What it does not get
    # is a vote in the replicate mean.
    _unusable = []
    if f["n_cycles"] is not None and f["n_cycles"] <= UNUSABLE_MIN_CYCLES:
        _unusable.append(
            f"only {f['n_cycles']} complete cycle(s) — capacity retention is "
            f"measured against cycle {UNUSABLE_MIN_CYCLES}, so there is no "
            f"reference for it here")
    if ce is not None and ce < UNUSABLE_MIN_MEDIAN_CE:
        _unusable.append(
            f"median coulombic efficiency {ce:.1f}% — more charge is lost each "
            f"cycle than returns, which is not a working cell")
    f["unusable"] = _unusable

    f.setdefault("death_cycle", None)
    f["first_last_discharge"] = disc
    # WITHHELD AT SOURCE, so nothing downstream can quote it. A variable-rate
    # run's `Q(n)/Q(ref)` is dominated by the rate schedule, and the run page
    # and the replicate table both pool this field across cells. Keeping the
    # arithmetic under its own key means the number is not lost, only barred
    # from being called retention.
    f["rate_is_variable"] = bool(((params or {}).get("rate_protocol")
                                  or {}).get("is_variable"))
    f["retention_uncorrected"] = ret
    f["retention"] = None if f["rate_is_variable"] else ret
    f["median_ce"] = ce
    f["fit_limit"] = fit_limit

    # --- integrity --------------------------------------------------------
    f["bands"], f["anomalous"] = {}, pd.DataFrame()
    if integrity is not None and not integrity.empty:
        f["bands"] = integrity["band"].value_counts().to_dict()
        # ANOMALOUS for the PARASITIC reason only. A half-cycle condemned for
        # passing more charge than the material can hold is reported under
        # that heading instead; listing it twice reads as two faults.
        _an = integrity[integrity["band"] == "ANOMALOUS"]
        if "over_theoretical" in _an:
            _an = _an[_an["over_theoretical"] != True]       # noqa: E712
        f["anomalous"] = _an.sort_values("parasitic_fraction", ascending=False)
        f["parasitic_total"] = float(integrity["parasitic_charge"].sum())
        f["plateau_total"] = float(integrity["plateau_charge"].sum())
        if "over_theoretical" in integrity:
            ot = integrity["over_theoretical"] == True        # noqa: E712
            f["over_theoretical"] = integrity[ot].sort_values(
                "capacity_ratio", ascending=False)
            _sane_pl = integrity.loc[~ot, "plateau_charge"]
            f["plateau_sane"] = float(_sane_pl.sum())
        else:
            f["over_theoretical"] = None
            f["plateau_sane"] = f["plateau_total"]
            _sane_pl = integrity["plateau_charge"]
        # The scale of ONE half-cycle, so the total can be read against a
        # capacity without being mistaken for one. See the plateau finding.
        _sane_pl = pd.to_numeric(_sane_pl, errors="coerce")
        _nz = _sane_pl[_sane_pl.notna() & (_sane_pl > 0)]
        f["plateau_n_half_cycles"] = int(len(_nz))
        f["plateau_worst_mAh_g"] = (float(_nz.max()) if len(_nz)
                                    else float("nan"))

    # --- what was withheld ------------------------------------------------
    withheld = 0
    total = 0
    widths = []
    for step, A in (attribution or {}).items():
        if A is None or A.empty:
            continue
        withheld += int(A["attribution_withheld"].sum())
        total += len(A)
        w = pd.to_numeric(A["closure_interval_width"], errors="coerce").dropna()
        widths.extend(w.tolist())
    f["withheld"] = (withheld, total)
    f["closure_widths"] = (min(widths), max(widths)) if widths else None
    # The charge the model could not name, and how often that alone was the
    # reason a share was withheld. From 1.9.0.56 there is no free background,
    # so the closure interval is zero by construction and THIS is the number
    # a reader should be looking at.
    _un, _un_with = [], 0
    for step, A in (attribution or {}).items():
        if A is None or A.empty or "unattributed_fraction" not in A:
            continue
        _u = pd.to_numeric(A["unattributed_fraction"],
                           errors="coerce").dropna()
        _un.extend(_u.tolist())
        if "unattributed_withheld" in A:
            _un_with += int(A["unattributed_withheld"].sum())
    f["unattributed"] = (float(np.median(_un)) if _un else None)
    f["unattributed_range"] = ((min(_un), max(_un)) if _un else None)
    f["unattributed_withheld"] = _un_with
    # True when nothing was swept: every measured width is zero (one degree
    # only), or none could be measured at all.
    _w = [x for x in widths if np.isfinite(x)]
    f["no_baseline"] = bool(attribution) and (not _w or max(_w) == 0.0)

    # --- peaks ------------------------------------------------------------
    f["reference_reason"] = getattr(detection, "reference_reason", "")
    f["reference_severity"] = getattr(detection, "reference_severity", "ok")
    f["peaks"] = {s: list(np.round(getattr(detection, "reference_voltages",
                                           lambda _: [])(s), 3))
                  for s in ("Charge", "Discharge")} if detection else {}
    f["tracked"] = []
    if tracking is not None and tracking.summary:
        for k in sorted(tracking.summary):
            s = tracking.summary[k]
            f["tracked"].append(s)
    # THE NUMBER THE AREA-GROWTH FLAG WAS ACTUALLY MADE AGAINST.
    # `analyse.flag_area_growth` compares each peak's area retention with
    # `capacity_retention_pct(cycle_table, reference_cycle)` — median-of-three
    # at each end, measured from the REFERENCE cycle — and stores it on the
    # Tracking. The finding that reports the flag was printing `f["retention"]`
    # instead, which is the last working cycle's `Retention_vs_C2_%`: a
    # different basis, a different reference, and on a cell that died mid-record
    # a different number by tens of points. NNM cell A published "22% area
    # retention ... against the cell's 34%" from a guard whose bound implied
    # ~0%. One flag, one basis, printed from where the flag put it.
    f["area_growth_basis_pct"] = float(
        getattr(tracking, "capacity_retention_pct", np.nan)
        if tracking is not None else np.nan)
    # THE COHERENCE VERDICT, carried to the run summary.
    # `coherence_audit` decides, per reference peak, whether that peak moves
    # like a redox feature or like a fitting artefact. Until 1.9.0.34 the
    # verdict lived only in the console: RUN_SUMMARY's drift table published a
    # rate for every primary peak, captioned as "the quantity that survives
    # where the areas do not", while the audit two cells earlier had called
    # the same peak NOT trend-worthy. On NMC111 that meant a table of drift
    # rates on a run where NOT ONE peak in either cell was coherent.
    f["coherence"] = []
    if coherence is not None and getattr(coherence, "empty", True) is False:
        for _r in coherence.itertuples():
            f["coherence"].append(dict(
                step=str(_r.step),
                reference_voltage=float(_r.reference_voltage),
                verdict=str(_r.verdict),
                pairs=int(_r.pairs)))
    # Does the dQ/dV curve account for the cell's charge? Set by
    # `analyse.flag_low_fidelity`. This belongs in START_HERE and not only in
    # the console: it decides whether the peak AREAS on this page mean
    # anything, and START_HERE is where somebody reads them.
    f["integral_fidelity"] = dict(getattr(tracking, "integral_fidelity", {})
                                  or {})
    f["profile"] = (profile or {}).get("class") if isinstance(profile, dict) \
        else profile
    return f


# What each mechanism means, and what it implies about the model — in the
# words a reader needs rather than the enum name. `quality.MECHANISM_*`.
_MECHANISM_TEXT = {
    "two_phase": (
        "one transition at a fixed potential",
        "The potential is pinned by two phases coexisting, so there is no "
        "composition window to smear over and every component was fitted as "
        "a narrow peak on a straight baseline."),
    "multi_transition": (
        "a series of distinct transitions",
        "Each feature is a real transition in its own right, so the curve "
        "was fitted as a series of peaks — no flat-topped bands, which would "
        "give a resolved feature a width it does not have."),
    "solid_solution": (
        "a continuum of site energies",
        "Charge is delivered across a window of composition rather than at "
        "one potential, so the components were fitted as flat-topped bands. "
        "The shoulders a peak model finds on an envelope like this are "
        "artefacts of the wrong shape, so only the primary features were "
        "seeded."),
    "mixed": (
        "a solid solution with a transition on top of it",
        "Both shapes were used in the same fit: bands for the envelope, "
        "peaks for the transition riding on it, and the data decided which "
        "each component became."),
}


def _mechanism_sentence(f):
    """The verdict that chose the model, in the document that reports it.

    Until 1.9.0.47 this existed only in the console. A reader of START_HERE
    could not tell whether their data had been fitted with pseudo-Voigts,
    with bands, or with both — which is the one thing this build decides
    differently for every dataset.
    """
    r = f.get("resolvability") or {}
    mech = r.get("mechanism")
    if not mech:
        return []
    what, implies = _MECHANISM_TEXT.get(
        str(mech), (str(mech).replace("_", " "), ""))
    why = (r.get("mechanism_reason") or "").strip().rstrip(".")
    sf = r.get("shoulder_fraction")
    bits = [f"The run classified this dataset as **{str(mech).replace('_', ' ')}"
            f"** — {what}."]
    if why:
        bits.append(why[0].upper() + why[1:] + ".")
    elif sf is not None and np.isfinite(sf):
        bits.append(f"{sf:.0%} of the detected components are shoulders "
                    f"rather than resolved maxima.")
    if implies:
        bits.append(implies)
    bits.append("The classification is made on the reference half-cycle "
                "before anything is fitted, and it is what chose the model "
                "below; `component_kind` in the fitted-parameter table says "
                "what each component actually became.")
    return [("ok", "How this curve was modelled", " ".join(bits))]


_KIND_WORD = {"band": "flat-topped band", "peak": "peak"}


def _kind_by_ref(f):
    """(step, reference voltage) -> the shape that feature was fitted with."""
    out = {}
    for t in (f.get("tracked") or []):
        k = t.get("component_kind")
        if not k:
            continue
        out[(t.get("step"), round(float(t.get("reference_voltage", np.nan)), 3))] = (
            _KIND_WORD.get(str(k), str(k))
            + (" (band in some cycles, peak in others)"
               if t.get("component_kind_mixed") else ""))
    return out


_QUALITY_NOTE = (
    "R² is dominated by the bulk of a curve, so a composite can be half again "
    "taller than the data, or miss a peak height by a third, and still score "
    "0.97. These say what it cannot.")


def _quality_sentence(f):
    """How well does the model actually draw this curve?

    Three faults, none of which R² carries: a fit too SHORT (a width floor —
    the model cannot be as narrow as the peak, so it conserves area by being
    wide and low), a fit too TALL and outside the data (the peak/background
    split is not determined, so the baseline is drawn high and the peaks are
    deepened to reach the curve), and a residual with STRUCTURE rather than
    scatter (the lineshape itself is wrong).
    """
    q = f.get("fit_quality") or {}
    if not q:
        return []
    bits, worst = [], "ok"
    for step in ("Charge", "Discharge"):
        s = q.get(step)
        if not s:
            continue
        h, o = s.get("height_ratio", np.nan), s.get("overshoot", np.nan)
        mr, z = s.get("max_residual_frac", np.nan), s.get("residual_runs_z", np.nan)
        _p = [f"**{step}** (median of {s.get('n', 0)} half-cycles): the fitted "
              f"peak is {h:.2f}x the height of the data"]
        if np.isfinite(h) and h < 0.85:
            _p.append("— the model is drawing the peak too SHORT, which is "
                      "what a width floor looks like: it cannot be as narrow "
                      "as the feature, so it conserves area by being wide and "
                      "low")
            worst = "bad"
        elif np.isfinite(h) and h > 1.15:
            _p.append("— the model is drawing the peak TALLER than the curve, "
                      "which is what an undetermined peak/background split "
                      "looks like: the baseline sits high and the peaks are "
                      "deepened to reach the data")
            worst = "bad"
        if np.isfinite(o) and o > 0.05:
            _p.append(f", and the composite goes {o:.0%} of the amplitude "
                      f"outside the range the data ever reached")
            worst = "bad"
        if np.isfinite(mr):
            _p.append(f". The largest residual is {mr:.0%} of the amplitude")
            if mr > 0.20 and worst == "ok":
                worst = "warn"
        if np.isfinite(z) and z < -4:
            # ITEM 34. This used to read "the LINESHAPE is wrong, not just the
            # noise level", and on LTO that is not supported: turning the
            # smoothing off makes z WORSE, binning finer makes it worse still
            # (-46.5), and z does not correlate with how sparsely the peak is
            # sampled. A runs test needs a RANDOM component to test against —
            # a residual carrying a systematic term at half the noise
            # amplitude scores only -2 to -5 even at n = 5000. Reaching -25 to
            # -34 means the residual is almost entirely deterministic, which
            # is what a curve built from bins holding 0 or 1 raw record is.
            # So it says the residual has structure; it does not say what put
            # it there.
            _p.append(f", and the residual runs the same sign far longer than "
                      f"noise would (z = {z:.0f}) — there is structure left in "
                      f"it. That can be the lineshape, the baseline, or a "
                      f"curve with too little independent noise to test "
                      f"against; z alone does not say which")
            if worst == "ok":
                worst = "warn"
        # ITEM 31. THE MEDIAN HID THE FAULT. Every number above is a median,
        # and on NMC111 that reported "0.98x the height of the data" while one
        # half-cycle was at 0.57x and two on the replicate at 0.40x and 0.35x.
        # `height_worst` was computed for exactly this and read by nothing —
        # the same defect class as items 1, 5 and 22, in the paragraph whose
        # own closing line warns that a fit can "miss a peak height by a third
        # and still score 0.97".
        _hw, _hwc = s.get("height_worst", np.nan), s.get("height_worst_cycle")
        _ht, _htc = s.get("height_tallest", np.nan), s.get("height_tallest_cycle")
        _ns, _nt = s.get("n_short", 0), s.get("n_tall", 0)
        _r2w, _r2c = s.get("r2_worst", np.nan), s.get("r2_worst_cycle")
        _ex = []
        if np.isfinite(_hw) and _hw < 0.85 and _hwc is not None:
            _ex.append(f"the WORST is cycle {_hwc} at {_hw:.2f}x"
                       + (f", and {_ns} of {s.get('n', 0)} half-cycles are "
                          f"below 0.85x" if _ns > 1 else ""))
        if np.isfinite(_ht) and _ht > 1.15 and _htc is not None:
            _ex.append(f"the TALLEST is cycle {_htc} at {_ht:.2f}x"
                       + (f", and {_nt} of {s.get('n', 0)} are above 1.15x"
                          if _nt > 1 else ""))
        if _ex:
            _p.append(". A median hides a single bad half-cycle, so: "
                      + "; ".join(_ex))
            worst = "bad"
        elif np.isfinite(_r2w) and _r2w < 0.90 and _r2c is not None:
            _p.append(f". The median is not the whole story — the worst "
                      f"half-cycle is cycle {_r2c} at R2 {_r2w:.3f}")
            if worst == "ok":
                worst = "warn"
        bits.append(" ".join(_p).replace(" ,", ",").replace(" .", ".") + ".")
    if not bits:
        return []
    head = ("The model does not draw this curve well" if worst == "bad"
            else "The fit has structure left in it" if worst == "warn"
            else "The model draws this curve well")
    return [(worst, head, " ".join(bits) + " " + _QUALITY_NOTE)]


def _voltage_envelope_sentence(f):
    """
    How far the cell actually got, cycle by cycle.

    The dQ/dV heatmap draws this as a step line at the edge of the measured
    region; nothing stated it. It is a different quantity from capacity: a
    cell can deliver the same charge every cycle while the voltage window it
    traverses to do so collapses, and that is polarisation rather than loss of
    active material. LTO cell A moves 622 mV over ten cycles at 99 % capacity
    retention.
    """
    env = (f.get("params") or {}).get("voltage_envelope") or {}
    if not env:
        return []
    bits, worst = [], "ok"
    for step in ("Charge", "Discharge"):
        e = env.get(step)
        if not e:
            continue
        _sp = max(float(e.get("hi_spread_mV", 0.0)),
                  float(e.get("lo_spread_mV", 0.0)))
        # `drawn` IS THE GATE. It is decided once, beside the envelope
        # measurement, using the plotting module's own thresholds, so this
        # sentence describes exactly the steps on which the heat map drew a
        # line. See the note there for what went wrong with two thresholds.
        if not e.get("drawn") or not np.isfinite(_sp):
            continue
        _hi_moves = float(e.get("hi_spread_mV", 0.0)) >= float(
            e.get("lo_spread_mV", 0.0))
        _a = e["hi_first"] if _hi_moves else e["lo_first"]
        _b = e["hi_last"] if _hi_moves else e["lo_last"]
        _edge = "furthest" if _hi_moves else "nearest"
        bits.append(
            f"**{step}**: the {_edge} voltage this step reached went from "
            f"{_a:.3f} V to {_b:.3f} V over {int(e.get('n', 0))} half-cycles, "
            f"a span of {_sp:.0f} mV")
        if _sp >= ENVELOPE_SEVERE_MV:
            worst = "warn"
    if not bits:
        return []
    return [(worst,
             "The voltage window the cell traverses is not constant",
             "; ".join(bits)
             + ". This is not a capacity statement — a cell can deliver the "
               "same charge every cycle while the window it needs to do so "
               "moves, and that is polarisation, or a cut-off being reached "
               "sooner, rather than active material lost. Read it beside the "
               "retention above: the two moving together is fade, the window "
               "moving alone is resistance. The blue step line on the dQ/dV "
               "heatmap is this quantity, and the region beyond it is "
               "unmeasured rather than zero.")]


def _identifiability_sentence(f):
    """
    How much data each fit had, against how many parameters it varied.

    Every standard error on the page, every `*_poorly_determined` flag and the
    `reliable` boolean itself are statements about a least-squares fit, and a
    least-squares fit of 60 points with 14 varied parameters says something
    very different from one of 600 with the same 14. `fitting.fit_one` has
    recorded both numbers since the histogram rewrite; nothing read them.
    """
    q = f.get("identifiability") or {}
    if not q:
        return []
    _med, _min = q["median"], q["worst"]
    _thin, _n = q["n_thin"], q["n_half_cycles"]
    _coup = q.get("coupled", 0)
    _mp, _mv = q.get("median_points"), q.get("median_varied")
    _lead = (f"Median {_med:.0f} points per varied parameter across "
             f"{_n} half-cycles"
             + (f" ({_mp:.0f} bins, {_mv:.0f} parameters)"
                if _mp is not None and np.isfinite(_mp)
                and _mv is not None and np.isfinite(_mv) else "")
             + f"; the thinnest fit had {_min:.1f}."
             + (f" {_coup} shoulder parameter(s) were tied to a parent rather "
                f"than varied freely, which is why the parameter count is "
                f"lower than the component count implies."
                if _coup else ""))
    if _thin:
        return [("warn", "Some fits had little data per parameter",
                 _lead + f" {_thin} of {_n} half-cycles fell below "
                 f"{IDENTIFIABILITY_POINTS_PER_PARAM:.0f} points per "
                 f"parameter. Below that the standard errors, and every flag "
                 f"derived from them — `centre_poorly_determined`, "
                 f"`area_poorly_determined`, `reliable` — describe a fit with "
                 f"barely more information than it has unknowns. "
                 f"`points_per_parameter` is in the fitted-parameters table "
                 f"per half-cycle.")]
    return [("ok", "The fits had enough data for their parameters",
             _lead + " Nothing here is close to the point where a "
             "least-squares standard error stops meaning anything. "
             "`points_per_parameter` is in the fitted-parameters table.")]


def _reliability_sentence(f):
    """
    What `reliable` says on this cell, and what it does NOT say.

    The boolean was the most-read field in the parameter table and the page
    printed neither its count nor its reasons. Worse, a reader who takes it
    to mean "I can quote this area with its uncertainty" is wrong about
    nearly every component of a layered oxide: on NNM 83% and on NMC 89% of
    components have no standard error on their area at all, and that never
    reaches `reliable` because `ENFORCE_AREA_DETERMINACY` is off. The two
    are different axes and are now printed as two.
    """
    r = f.get("reliability") or {}
    n, ok = r.get("n"), r.get("ok")
    if not n:
        return []
    bits = [f"{ok} of {n} fitted components are marked reliable."]
    reasons = r.get("reasons") or {}
    if reasons:
        _ordered = sorted(reasons.items(), key=lambda kv: -kv[1])
        # ONLY GLOSS THE REASONS THAT ACTUALLY FIRED. Explaining what an
        # inferred component means on a cell that has none is noise, and the
        # LTO page carried three sentences to cover a single pinned width.
        _gloss = {
            "band at its width ceiling":
                "a component at a bound has a width that is a constraint "
                "rather than a fit — and being knife-edge, one that need not "
                "reproduce on another machine",
            "width at its ceiling":
                "a component at a bound has a width that is a constraint "
                "rather than a fit — and being knife-edge, one that need not "
                "reproduce on another machine",
            "width at its ceiling or area not finite":
                "a component at a bound has a width that is a constraint "
                "rather than a fit",
            "not detected in this half-cycle":
                "a component not detected in its own half-cycle carries a "
                "centre borrowed from a neighbouring one",
            "this half-cycle passed more charge than the material can hold":
                "a half-cycle that passed more charge than the material can "
                "hold did not produce a differential capacity curve of the "
                "material, however faithfully that curve integrates — its "
                "components are fitted, tracked and drawn, and left out of "
                "the trends",
            "the fit put almost no charge here":
                "a component carrying almost no charge is the fit saying the "
                "feature is not there",
            "most of this component lies outside the measured window":
                "a component sitting at an end of the voltage window has "
                "part of its profile outside the data, and runs both flanks "
                "to the width bound because nothing on that side stops "
                "them — the window is the reason, not the width",
            "area collapsed against the reference cycle":
                "a component that has collapsed against the reference cycle "
                "is no longer the feature it was tracked as",
        }
        _seen, _say = set(), []
        for _k, _ in _ordered:
            _g = _gloss.get(_k)
            if _g and _g not in _seen:
                _seen.add(_g); _say.append(_g)
        _lead = ("The rest are not, for reasons that are not interchangeable: "
                 if len(_ordered) > 1 else "The rest are not: ")
        _tail = ""
        if _say:
            _tail = (" — " + "; ".join(_say) + ". "
                     + ("None of these is the same as a bad measurement, and "
                        "none is the same as the others."
                        if len(_say) > 2 else
                        "Neither is the same as a bad measurement, and "
                        "neither is the same as the other."
                        if len(_say) == 2 else
                        "That is not the same as a bad measurement."))
        bits.append(_lead + "; ".join(f"{v} {k}" for k, v in _ordered) + _tail)
    det = r.get("determinacy") or {}
    if det:
        _no = int(det.get("not estimated", 0))
        _im = int(det.get("imprecise", 0))
        _de = int(det.get("determined", 0))
        _s = (f"Separately, and NOT part of that flag: {_de} of {n} "
              f"components have a determined area uncertainty, {_im} have one "
              f"too large to quote, and {_no} have none at all"
              + (" — the covariance could not be estimated, so the area has "
                 "no error bar to put beside it." if _no else
                 ", so every area here can be quoted with one."))
        if _no > 0.5 * n:
            _s += (" That is most of this cell. `reliable` says nothing "
                   "about it, so an area from here should be quoted without "
                   "an uncertainty rather than with an implied one.")
        bits.append(_s)
    # AN AREA THAT IS A LOWER BOUND, said out loud. A component centred near
    # an end of the voltage window has part of its profile outside the data;
    # the analytic area covers the whole profile, so the number is what the
    # feature would carry IF the window reached far enough, and the measured
    # part is less. `reliable` is deliberately silent on this — it is the
    # same separation as the determinacy line above — so the page has to say
    # it instead.
    _lb_rel = int(r.get("lower_bound_reliable", 0) or 0)
    _lb = int(r.get("lower_bound", 0) or 0)
    if _lb:
        _md = float(r.get("lower_bound_median", 0.0) or 0.0)
        _wo = float(r.get("lower_bound_worst", 0.0) or 0.0)
        _s2 = (f"Also separately: {_lb} of {n} components have more than "
               f"{AREA_LOWER_BOUND_REPORTABLE:.0%} of their fitted profile "
               f"outside the measured voltage window, so their areas are "
               f"LOWER BOUNDS rather than measurements")
        if _lb_rel:
            _s2 += (f" — and {_lb_rel} of those are marked reliable, with a "
                    f"median {_md:.0%} outside (worst {_wo:.0%}). `reliable` "
                    f"is a statement about the fit, not about how much of "
                    f"the feature the window caught. Widen the window, or "
                    f"quote these as “at least”.")
        else:
            _s2 += (f", worst {_wo:.0%} outside. None of them is marked "
                    f"reliable, so nothing quotable is affected.")
        bits.append(_s2)
    _lvl = "warn" if (reasons and ok < 0.5 * n) else "ok"
    if _lb_rel:
        _lvl = "warn"
    return [(_lvl, "What \u201creliable\u201d means here", " ".join(bits))]


# How many bins a width must span before the report will call it a width.
# The same number as `analyse.WIDTH_MIN_SAMPLES`, named here so the sentence
# and the flag cannot drift apart.
# Below this, plateau charge is not worth a finding of its own. The tail
# clause of that same finding has always used 1 mAh/g; the gate on the
# finding used 0, so a third of a percent of one half-cycle raised a red mark.
PLATEAU_REPORTABLE_MAH_G = 1.0


def _unattributed_clause(u, rng=None):
    """
    " The model names 99.8% of the charge, leaving +0.2% unattributed."

    At 0 dp this printed "names 100% of the charge, leaving +0% unattributed"
    on all three chemistries — and on LTO, whose median is -0.0012, the
    literal string "-0%". A signed zero is not a measurement of anything, and
    the RANGE was computed for this and rendered only in the other branch.
    """
    if u is None or not np.isfinite(u):
        return ""
    _r = ""
    if rng and all(v is not None and np.isfinite(v) for v in rng):
        _r = f" (over the run, {rng[0]:+.1%} to {rng[1]:+.1%})"
    if abs(u) < 0.005:
        return (f" The model names essentially all of the charge — the median "
                f"unattributed fraction is under 0.5%{_r}.")
    return (f" The model names {1 - u:.1%} of the charge, leaving "
            f"{u:+.1%} unattributed{_r}.")


def _sentence_case(t):
    """
    Upper-case the first letter and LEAVE THE REST ALONE.

    `str.capitalize()` lower-cases everything after the first character, so
    `detect.formation_end`'s "CE never held within 0.1 points of the later
    median" reached every page in this project as "ce never held...".
    """
    t = str(t or "")
    return t[:1].upper() + t[1:]


WIDTH_BINS_MEASURABLE = 4.0

# POINTS PER VARIED PARAMETER, below which a least-squares standard error is
# not worth quoting. Not a significance threshold — there is no such thing here
# — but the point at which the fit has so little information per unknown that
# the covariance matrix it reports is dominated by the model rather than by the
# data. Ten is the conventional rule of thumb for non-linear least squares and
# is used here as a flag, never as a rejection: nothing is dropped on it.
IDENTIFIABILITY_POINTS_PER_PARAM = 10.0

# WHEN THE REACHABLE-VOLTAGE ENVELOPE IS BAD NEWS RATHER THAN A FACT.
# Whether it is worth mentioning at all is not decided here — it is decided
# once, where the envelope is measured, by the same test the heat map uses to
# decide whether to draw the line, so the sentence and the figure cannot
# disagree. This is only the severity: above it, the dQ/dV curves being
# compared cycle to cycle no longer cover the same voltages, which changes
# what a peak area means.
ENVELOPE_SEVERE_MV = 300.0

# WHEN IS A DATASET NOT A MEASUREMENT? Both of these are definitional rather
# than tuned, which is why they are two numbers and not a scoring function.
#
# `UNUSABLE_MIN_CYCLES` is `cycling.RETENTION_REFERENCE_CYCLE`: retention is
# quoted against that cycle, so a record no longer than it has no reference to
# be measured against and the figure cannot exist. Stated here rather than
# imported to keep `report` free of a cycling import; the two are checked
# against each other by `test_reference_cycle_agreement`.
#
# `UNUSABLE_MIN_MEDIAN_CE` is 50%: below it more charge is lost every cycle
# than comes back, which is not a working cell whatever else the file says.
# It is deliberately far below the ~95% at which a cell is merely UNHEALTHY —
# this flag is for "this is not a measurement", not for "this is a poor cell".
UNUSABLE_MIN_CYCLES = 2
UNUSABLE_MIN_MEDIAN_CE = 50.0


def _lineshape_sentence(f):
    """Was the peak fitted as asymmetric, and by how much?

    Reported because it is a measurement the reader would otherwise have to
    take from a CSV, and because a ratio that settled ON its bound is not a
    measurement at all — the same fault as a width pinned on its floor, one
    level up, and invisible without this because there is one ratio per
    half-cycle rather than one per component.
    """
    q = f.get("lineshape") or {}
    if not q:
        return []
    bits, worst = [], "ok"
    for step in ("Charge", "Discharge"):
        s = q.get(step)
        if not s or not s.get("split"):
            continue
        k = s["k"]
        _side = ("high-voltage" if k > 1 else "low-voltage")
        _ratio = k if k > 1 else (1.0 / k if k > 0 else float("nan"))
        # THE RANGE IN THE SAME ORIENTATION AS THE HEADLINE. `_ratio` is
        # flipped to 1/k when the LOW flank is the wide one, so that the
        # sentence can always say "N times the width of the other" with N > 1.
        # `k_lo`/`k_hi` were printed un-flipped, so LTO cell B read "the
        # low-voltage flank is 3.9x the width of the other (ratio 0.09-0.33)" —
        # a headline outside its own quoted range, which reads as scatter in
        # the data rather than as a unit flip. Inverting a range also reverses
        # its ends, so k_hi becomes the lower bound.
        _klo, _khi = float(s["k_lo"]), float(s["k_hi"])
        if not (k > 1):
            _klo, _khi = ((1.0 / _khi if _khi > 0 else float("nan")),
                          (1.0 / _klo if _klo > 0 else float("nan")))
        _ss, _nk = s.get("same_sign"), s.get("n_k", s["n"])
        _p = [f"**{step}** ({s['n']} half-cycles): the {_side} flank is "
              f"{_ratio:.1f}x the width of the other "
              f"(ratio {_klo:.2f}-{_khi:.2f}"
              + (f", and {_ss} of {_nk} lean that way"
                 if _ss is not None and _ss < _nk else "") + ")"]
        if _ss is not None and _nk and _ss < 0.8 * _nk:
            _p.append(f". The sign is NOT consistent — {_nk - _ss} half-cycles "
                      f"lean the other way, so the median above describes a "
                      f"direction this dataset does not hold")
            worst = "warn"
        if s.get("clipped"):
            _p.append(f". On {s['clipped']} of {_nk}, one flank was held at "
                      f"the width floor, so the ratio quoted is the one the "
                      f"curve was drawn with rather than the one the "
                      f"optimiser reached (`asymmetry_k_fitted` in the table)")
        # THE WIDTHS THEMSELVES, BEFORE THEIR RATIO. Said first, because it
        # governs everything after it: a ratio of two numbers that are both
        # the bin width is 1.0 plus noise, whatever the fit reports.
        _fb, _us, _nc = (s.get("fwhm_bins"), int(s.get("undersampled", 0) or 0),
                         int(s.get("n_comp", 0) or 0))
        if _fb is not None and np.isfinite(_fb) and _fb < WIDTH_BINS_MEASURABLE:
            _p.append(f". The width this is a ratio OF is not a measurement: "
                      f"the fitted FWHM spans a median {_fb:.1f} bins"
                      + (f" and {_us} of {_nc} components are flagged "
                         f"`width_undersampled`" if _nc else "")
                      + f", against the {WIDTH_BINS_MEASURABLE:.0f} a width "
                        f"needs. Swept over bin widths from 0.5 to 5 mV on "
                        f"this chemistry the fitted FWHM tracks the BIN, not "
                        f"the peak, so neither the widths nor this ratio "
                        f"should be quoted as a property of the material")
            worst = "warn"
        _daf = int(s.get("dominant_at_floor", 0) or 0)
        if _daf:
            _p.append(f". On {_daf} of {s['n']} half-cycles the NARROW side "
                      f"of the largest component sits exactly on the width "
                      f"floor — the leading edge is steeper than the "
                      f"histogram samples, so that half of the width is the "
                      f"floor and not a measurement, and `fwhm` carries it")
            if worst == "ok":
                worst = "warn"
        if s["at_bound"]:
            _p.append(f". {s['at_bound']} of {s['n']} settled ON the ratio "
                      f"bound, where the number is the limit and not an "
                      f"answer — read those widths as a floor")
            worst = "warn"
        bits.append(" ".join(_p).replace(" .", ".") + ".")
    if not bits:
        return []
    head = ("The peak is asymmetric and was fitted that way"
            if worst == "ok" else
            "The peak is asymmetric, and one flank is not resolved")
    # WHAT THE TWO DIRECTIONS ACTUALLY DID. This closed with "The two
    # directions lean OPPOSITE ways" unconditionally — appended to whatever
    # `bits` held, including a single fitted step, where "the two directions"
    # has no referent at all, and including two steps leaning the SAME way,
    # where the sentence states the opposite of the data above it. It happened
    # to be true of the LTO cells on disk, which is how it survived. The
    # asymmetry itself is the finding; which way the two steps lean is a
    # separate observation and is only worth making when both were measured.
    _ks = [float(q[st]["k"]) for st in ("Charge", "Discharge")
           if q.get(st) and q[st].get("split")
           and np.isfinite(q[st].get("k", np.nan)) and q[st]["k"] > 0]
    if len(_ks) < 2:
        _lean = (" Only one direction was fitted with a split lineshape, so "
                 "there is nothing here to compare it with.")
    elif (max(_ks) > 1.0) and (min(_ks) < 1.0):
        _lean = (" The two directions lean OPPOSITE ways, which is what a "
                 "kinetic tail or a spread of particle sizes gives and is not "
                 "something a symmetric peak can absorb.")
    else:
        _lean = (" Both directions lean the SAME way — towards "
                 + ("high" if min(_ks) > 1.0 else "low") + " voltage. A "
                 "kinetic tail reverses with the sweep, so a bias that does "
                 "not reverse is more likely to be in the model or the "
                 "binning than in the material, and is worth checking against "
                 "the fits before it is read as a property of the electrode.")
    return [(worst, head, " ".join(bits) + _lean
             + " Read which flank is the trailing one from the direction of "
               "the sweep, since a negative-electrode cell has its step "
               "labels the other way round. The width columns are `sigma` "
               "(low voltage) and `sigma_r` (high voltage); `fwhm` is their "
               "sum, not twice either one.")]


def _phase_rule_sentence(f):
    """What the thermodynamics expects, beside what was fitted.

    On a `two_phase` run the Gibbs phase rule is a real constraint and it is
    the argument that settles most "is this a doublet?" questions — see
    Appendix A7. It is stated here rather than enforced in the detector,
    deliberately: the phase rule bounds the two-phase FIELD, not the voltage
    window, and a detector that applied it would be asserting the expectation
    instead of testing against it. The reader gets the expectation at the
    moment they are looking at the fit, and makes the call.
    """
    r = f.get("resolvability") or {}
    if str(r.get("mechanism") or "") != "two_phase":
        return []
    q = f.get("component_census") or {}
    if not q:
        return []
    bits = []
    for step in ("Charge", "Discharge"):
        c = q.get(step)
        if not c:
            continue
        bits.append(f"**{step}**: {c['lo']}-{c['hi']} fitted, median "
                    f"{c['median']:.0f}")
    if not bits:
        return []
    extra = any((q.get(k) or {}).get("hi", 1) > 1 for k in ("Charge",
                                                            "Discharge"))
    return [(
        "warn" if extra else "ok",
        "What the phase rule expects here",
        "This dataset was classified **two phase**, and a two-phase field in "
        "a binary system at fixed temperature and pressure is invariant — the "
        "potential is held while both phases coexist, so the expectation is "
        "**one plateau and therefore one dQ/dV peak per transition**. "
        + "; ".join(bits) + ". "
        + ("Where more than one component was fitted, ask what process the "
           "extra one would correspond to before believing it: an R² gain "
           "from adding a component is what overfitting looks like, and "
           "stability against a change of bin width is necessary but not "
           "sufficient. The processing-free test is V(Q) — Q is a monotonic "
           "integral and V a directly quantised measurement, so a second "
           "plateau shows there if it is real. "
           if extra else "")
        + "Components closer together than the profile's own separation floor "
        "cannot be admitted at all, which is what enforces the constraint "
        "inside the plateau; components further out are outside the field the "
        "phase rule speaks about, and are judged against the noise instead.")]


def _shoulder_only_sentence(f):
    """Half-cycles the model fitted with no primary peak at all."""
    q = f.get("shoulder_only") or {}
    if not q:
        return []
    bits = []
    for step in ("Charge", "Discharge"):
        s = q.get(step)
        if not s:
            continue
        bits.append(f"**{step}**: {s['n']} of {s['total']} half-cycles, "
                    f"cycles {s['first']}-{s['last']}")
    if not bits:
        return []
    return [("bad", "Some half-cycles were fitted with no primary peak",
             "; ".join(bits) + ". The peak picker found no resolved maximum "
             "on these, so the model was given the second-derivative "
             "SHOULDERS instead and the whole half-cycle is described by "
             "features the picker itself called shoulders of something. That "
             "is not a fit to be read as a decomposition — check those "
             "half-cycles against the dQ/dV figure before quoting anything "
             "from them.")]


def _census_sentence(f):
    """Is the reference cycle a thin fit, and do the fits vary in size?"""
    q = f.get("component_census") or {}
    if not q:
        return []
    bits, worst = [], "ok"
    for step in ("Charge", "Discharge"):
        s = q.get(step)
        if not s:
            continue
        _p = [f"**{step}**: {s['lo']}-{s['hi']} components per half-cycle "
              f"over {s['n_half_cycles']}, median {s['median']:.0f}"]
        rn, rc = s.get("reference_n"), s.get("reference_cycle")
        if rn is not None and rc is not None:
            _p.append(f"; the reference cycle ({rc}) got {rn}")
            if rn <= s["lo"] and s["hi"] > s["lo"]:
                _p.append(" — **the thinnest fit in the dataset**, and every "
                          "tracked feature, every drift rate and the whole "
                          "attribution basis is derived from it")
                worst = "warn"
            elif rn < s["median"]:
                _p.append(", below the median for this dataset")
                if worst == "ok":
                    worst = "warn"
        bits.append(" ".join(_p).replace(" ;", ";") + ".")
    if not bits:
        return []
    head = ("The attribution basis is a thin fit" if worst == "warn"
            else "How many components each half-cycle was fitted with")
    tail = (" The reference cycle is chosen on the cell's integrity, not on "
            "how well the model fitted it, and that is the right criterion — "
            "but a basis taken from the thinnest fit in the run is worth "
            "knowing about before the shares below are read as capacities."
            if worst == "warn" else
            " The attribution basis and every tracked feature come from the "
            "reference cycle, so this is how much of the curve that basis "
            "was built from.")
    return [(worst, head, " ".join(bits) + tail)]


def _peak_sentences(f):
    """Findings about the peaks: are their areas capacities, and how fast do
    they move? Separate from `_sentences` only because it needs the tracking,
    which a cycling-only run does not have."""
    out = (_quality_sentence(f) + _voltage_envelope_sentence(f)
           + _identifiability_sentence(f)
           + _reliability_sentence(f)
           + _lineshape_sentence(f)
           + _shoulder_only_sentence(f)
           + _census_sentence(f) + _mechanism_sentence(f)
           + _phase_rule_sentence(f))
    fid = f.get("integral_fidelity") or {}
    low = _fidelity_outside_band(fid)
    if low:
        # The value furthest from 1, whichever side of the band it is on.
        worst = max(low.values(), key=lambda v: abs(v - 1.0))
        _short = worst < 1.0
        # BOTH EXPLANATIONS WHERE BOTH FAULTS ARE PRESENT. A curve can be
        # short on one step and over on the other, and those are opposite
        # faults with opposite causes; picking the explanation from the single
        # worst value attached the under-count reason to an over-count.
        _under = ("Charge delivered outside the analysed voltage window "
                  "cannot appear in the integral — a constant-voltage hold "
                  "is the usual reason, and on a finite-difference curve a "
                  "flat two-phase plateau is another: the voltage change "
                  "between records falls below the instrument's resolution "
                  "and the charge delivered there has no dV to be divided "
                  "by. No choice of fitting model recovers it.")
        _over = ("A curve cannot contain more charge than the cell "
                 "delivered, so the excess is an artefact — a voltage "
                 "traversed twice, or a hold spread across voltages the "
                 "cell never visited. No choice of fitting model removes "
                 "it.")
        _lo_steps = sorted(k for k, v in low.items() if v < 1.0)
        _hi_steps = sorted(k for k, v in low.items() if v > 1.0)
        if _lo_steps and _hi_steps:
            _why = (f"These are opposite faults. On "
                    f"{' and '.join(s.lower() for s in _lo_steps)}: " + _under
                    + f" On {' and '.join(s.lower() for s in _hi_steps)}: "
                    + _over)
        else:
            _why = _under if _lo_steps else _over
        # PER STEP, NOT THE WORST OF THE TWO APPLIED TO BOTH. `worst` is the
        # single value furthest from 1, and the closing clause used it to
        # describe every area on the page: NNM cell B was told "accounts for
        # only 5% of the charge, 56% of the discharge capacity ... Every area
        # and area-retention figure on this page is a share of that 5%", an
        # order of magnitude wrong for half its own figures. Where the two
        # steps differ, name both. The leading verb has the same fault: it is
        # picked from `worst` alone, so a curve carrying 145% of one step and
        # 56% of the other read "accounts for only 145%".
        _lo_all = all(v < 1.0 for v in low.values())
        _hi_all = all(v > 1.0 for v in low.values())
        _verb = ("accounts for only" if _lo_all else
                 "carries" if _hi_all else "accounts for")
        if len(low) > 1 and (max(low.values()) - min(low.values())) >= 0.05:
            _share_clause = (
                " Every area and area-retention figure on this page is a share "
                "of its own step's figure — "
                + ", ".join(f"{100 * v:.0f}% for {k.lower()}"
                            for k, v in sorted(low.items()))
                + " — and the two are not interchangeable. Read them as "
                  "descriptions of the curve, not of the cell. Peak POSITIONS "
                  "and their drift do not depend on the integral and are "
                  "unaffected.")
        else:
            _share_clause = (
                f" Every area and area-retention figure on this page is a "
                f"share of that {100 * worst:.0f}% — read them as "
                f"descriptions of the curve, not of the cell. Peak POSITIONS "
                f"and their drift do not depend on the integral and are "
                f"unaffected.")
        out.append((
            "bad", "Peak areas here are not capacities",
            f"The dQ/dV curve {_verb} "
            + ", ".join(f"{100 * v:.0f}% of the {k.lower()}"
                        for k, v in sorted(low.items()))
            + " capacity this cell delivered. " + _why
            + _share_clause))
    # THE COHERENCE VERDICT REACHES THIS PAGE TOO. `f["coherence"]` was
    # assembled for the run summary and rendered nowhere here, so on a
    # dataset where the audit graded ZERO peaks coherent this section still
    # printed every drift rate as a green finding — while the run-level page
    # one folder up said the same peaks were excluded because none of them
    # moved like a redox feature. Two Ratatosk pages, opposite claims.
    _verd = {(c["step"], round(c["reference_voltage"], 3)): c["verdict"]
             for c in (f.get("coherence") or [])}

    def _grade(t):
        return _verd.get((t.get("step"),
                          round(float(t.get("reference_voltage", np.nan)), 3)))

    moved = [t for t in (f.get("tracked") or [])
             if np.isfinite(t.get("voltage_drift_mV_per_cycle", np.nan))]
    _sound = [t for t in moved
              if _grade(t) in (None, "coherent", "questionable")]
    _rejected = [t for t in moved if t not in _sound]
    if _sound:
        bits = []
        for t in _sound:
            _g = _grade(t)
            bits.append(f"{t['step'].lower()} {t['reference_voltage']:.3f} V "
                        f"{t['voltage_drift_mV_per_cycle']:+.2f} mV/cycle"
                        + (" (questionable)" if _g == "questionable" else ""))
        out.append(("ok" if any(_grade(t) == "coherent" for t in _sound)
                    else "warn",
                    "How fast the features move",
                    "; ".join(bits) + ". Measured between medians of three "
                    "cycles at each end, from the reference cycle onward — a "
                    "difference of two single cycles carries the per-cycle "
                    "centre scatter and can be out by 15 mV."
                    + (f" {len(_rejected)} further peak(s) moved too, and are "
                       f"not quoted: the coherence audit found they do not "
                       f"move like a redox feature."
                       if _rejected else "")))
    elif _rejected:
        out.append(("warn", "No drift rate is quotable",
                    f"All {len(_rejected)} tracked peak(s) with a measurable "
                    f"drift were rejected by the coherence audit — none of "
                    f"them moves like a redox feature. The rates are in "
                    f"the fit-coherence table with their verdicts; they are "
                    f"not reported here because a rate read from a peak the "
                    f"fit could not hold on to is a property of the fit."))
    # DETECTED, ROSTERED, AND NEVER GIVEN TO THE MODEL.
    #
    # The tracked-peak roster is built from the DETECTED reference list while
    # the fit may be given a subset of it (`primaries_only`, chosen by the
    # mechanism classifier). Those features were recorded as `disappeared` in
    # every cycle — 3 of 7 discharge features on NMC111, tracked in zero of
    # 18 cycles — and this page advertised them as tracked peaks that had
    # gone. "Gone" is a finding about the cell; "never fitted" is a fact
    # about the run, and only one of them is true.
    _never = [t for t in (f.get("tracked") or []) if t.get("never_fitted")]
    if _never:
        out.append((
            "warn", "Some detected features were never fitted",
            ", ".join(f"{t['step'].lower()} {t['reference_voltage']:.3f} V"
                      for t in _never)
            + f" — {len(_never)} feature(s) that detection found and the "
              "model was never asked to fit, because the mechanism this run "
              "classified fits primary peaks only. They are absent from the "
              "fit, not from the cell, and they carry no area, no drift and "
              "no retention here. If they matter, the mechanism verdict is "
              "the thing to look at."))
    # NOT ON AREAS THE RUN HAS ALREADY WITHHELD. `flag_area_growth` compares
    # area retention with capacity retention; `flag_untrustworthy_areas` decides
    # separately whether an area retention is a capacity at all, and writes
    # `area_retention_trustworthy=False` with the reason. NNM cell B had all ten
    # peaks withheld — "the dQ/dV curve accounts for only 5% of this step's
    # delivered capacity, so a peak area is not a capacity" — and this finding
    # still printed "129% area retention, excess 114 points" and concluded the
    # decomposition was wrong, four paragraphs below the page's own statement
    # that these are not capacities. A comparison between two numbers is only as
    # good as the weaker of them.
    grown = [t for t in (f.get("tracked") or [])
             if t.get("area_exceeds_capacity")
             and t.get("area_retention_trustworthy") is not False]
    _grown_withheld = [t for t in (f.get("tracked") or [])
                       if t.get("area_exceeds_capacity")
                       and t.get("area_retention_trustworthy") is False]
    if not grown and _grown_withheld:
        _wr = next((str(t.get("area_withheld_reason") or "")
                    for t in _grown_withheld
                    if t.get("area_withheld_reason")), "")
        out.append((
            "ok", "Area growth could not be assessed",
            f"{len(_grown_withheld)} peak(s) have an area retention above what "
            f"the cell's capacity allows, but every one of them is a peak whose "
            f"area this run has already withheld as not being a capacity"
            + (f" — {_wr}" if _wr else "")
            + ". A ratio of two areas that are not capacities cannot be "
              "compared with a capacity retention, so no verdict is offered "
              "here. The excesses are in `*_tracked_peaks_summary.csv` "
              "(`area_growth_excess_pct`) for anyone who wants to look."))
    if grown:
        # NOT "GREW". `area_exceeds_capacity` means the peak's area retention
        # is HIGHER THAN THE CELL'S capacity retention plus headroom — not
        # that the area rose. On NNM cell A, whose capacity retention is 34%,
        # this headed a list of peaks at 49% and 22% area retention with "A
        # peak grew while the cell shrank"; every one of them had shrunk.
        # And the 34% the comparison is against was never printed, which is
        # the only number that makes the list mean anything.
        _cap = f.get("area_growth_basis_pct")
        # AND THE HEADROOM, because the excess is measured from the SUM.
        # "214% area retention, excess 124 points ... against the cell's 75%"
        # invites the reader to check 214 - 75 = 139 and conclude one of the
        # three numbers is wrong. The bound is 75 + 15.
        _against = (f" against the cell's {_cap:.0f}% capacity retention plus "
                    f"{AREA_GROWTH_HEADROOM_PCT:.0f} points of headroom, so "
                    f"the excesses above are measured from {_cap + AREA_GROWTH_HEADROOM_PCT:.0f}%"
                    if _cap is not None and np.isfinite(_cap) else "")
        out.append((
            "warn", "Peak areas held up better than the cell's capacity",
            ", ".join(f"{t['step'].lower()} {t['reference_voltage']:.3f} V "
                      f"({t['area_retention_pct']:.0f}% area retention"
                      + (f", excess {t['area_growth_excess_pct']:.0f} points"
                         if t.get("area_growth_excess_pct") is not None
                         and np.isfinite(t.get("area_growth_excess_pct"))
                         else "") + ")"
                      for t in grown)
            + _against
            + ". A redox feature cannot keep more charge than the cell "
              "delivers, so where this appears the decomposition is wrong — "
              "not the cell. Nothing has been deleted; look at the fit."))
    return out


def _sentences(f):
    """
    The findings, as prose, shared by both renderers.

    Returns a list of (level, heading, text). Level is 'ok', 'warn' or 'bad'
    and is the only styling either renderer applies to a finding.
    """
    out = []
    n = f.get("n_cycles")
    dl = f.get("first_last_discharge")
    ret, ce = f.get("retention"), f.get("median_ce")

    # FIRST, BEFORE ANYTHING ELSE ON THE PAGE. A reader who opens this file
    # must not have to reach the coulombic-efficiency paragraph to discover
    # that the export is not a measurement. See UNUSABLE_MIN_CYCLES.
    if f.get("unusable"):
        out.append(("bad", "This dataset is not a usable measurement",
                    _sentence_case("; ".join(f["unusable"]))
                    + ". Everything below is still computed from what the "
                      "file contains, because that is how you find out what "
                      "went wrong — but none of it is a result, and this cell "
                      "is left out of the replicate mean rather than "
                      "averaged into it. Check the export: a file much "
                      "shorter than its replicates is usually a run that was "
                      "stopped or a save that did not finish."))

    if dl and n:
        first, last = dl
        lg = f.get("last_good_cycle")
        died = f.get("fit_limit") or f.get("death_cycle")
        # "Runs to cycle N" is wrong for an export taken mid-run: the file
        # ends at cycle N because that is when it was written, not because
        # the experiment did.
        s = ((f"The export covers {n} cycles, the last of them still "
              f"running. " if f.get("partial_final") else
              f"The record runs to cycle {n}. ")
             + f"The cell delivered "
             f"{first:.0f} mAh/g on its first complete discharge and "
             f"{last:.0f} mAh/g")
        s += (f" at cycle {lg}, the last one it was still working"
              if died and lg else
              (f" at cycle {lg}" if lg else " on the last"))
        # NAME THE CYCLE. "99% of the reference cycle" sat three lines under
        # a table column headed "% of first", which is a DIFFERENT base — and
        # the reference is read from a column literally called
        # `Retention_vs_C2_%`, so the number was available all along.
        s += (f" — {ret:.0f}% of cycle {f.get('retention_reference') or 2}."
              if ret is not None and not _rate_variable(f) else ".")
        if died and n and died < n:
            s += (f" The {n - died} cycles after that are the cycler still "
                  f"running against a cell that had stopped.")
        out.append(("ok", "What it did", s))
    pf = f.get("partial_final")
    if pf:
        cyc, step, prev, val = pf
        got = (f"It had delivered {val:.0f} mAh/g of cycle {prev}'s "
               f"{dl[1]:.0f} when the file was written. "
               if (val is not None and prev and dl) else "")
        out.append(("ok", "The cell was still cycling when this was exported",
                    f"Cycle {cyc} {step} never reached the cut-off voltage "
                    f"that every other half-cycle going the same way "
                    f"reaches, and it is the last half-cycle in the file — "
                    f"so the export was taken while it was still running. "
                    f"{got}A half-cycle in progress is not a measurement "
                    f"yet, so it is not fitted and is left out of the "
                    f"capacities and the retention above; it is still drawn "
                    f"in every figure, and the completed half of the same "
                    f"cycle is kept. Re-export once it finishes and it "
                    f"counts like any other. Everything above describes "
                    f"cycles 1 to {cyc - 1}."))
    if ce is not None:
        lvl = "ok" if 95 <= ce <= 101 else "warn"
        out.append((lvl, "Coulombic efficiency",
                    f"Median {ce:.1f}%. Below about 95% means charge is going "
                    f"somewhere other than the intended reaction; well above "
                    f"100% usually means the counter electrode, not the "
                    f"material under test."))

    bands = f.get("bands", {})
    anom = f.get("anomalous")
    if anom is not None and len(anom):
        worst = anom.iloc[0]
        cycles = ", ".join(str(c) for c in sorted(
            int(c) for c in anom["cycle"].head(8)))
        out.append(("bad", f"{len(anom)} half-cycle(s) lost charge",
                    f"Charge passed at full current where |dQ/dV| was small "
                    f"and away from the voltage limit — no redox process "
                    f"accounts for it. Worst: cycle {int(worst['cycle'])} "
                    f"{worst['step']}, {worst['parasitic_charge']:.0f} mAh/g "
                    f"of {worst['capacity']:.0f} "
                    f"({worst['parasitic_fraction']:.0%}). Cycles: {cycles}."))
    elif bands and not (f.get("over_theoretical") is not None
                        and len(f["over_theoretical"])):
        # SAID FIRST, AND FOR EITHER BRANCH. The "not every half-cycle is
        # clean" caveat lived inside the `else` only, so a cell whose mechanism
        # was anything but two-phase — NMC111 cell A, integrity table
        # {clean: 36, suspect: 1} — was told "Every half-cycle's reversed
        # charge sat either on a flat region or at the voltage limit" with a
        # suspect row sitting in the table whose `reason` column says exactly
        # what was seen. The guard is about the table, not about the mechanism,
        # so it cannot be conditioned on the mechanism.
        _grey = {k: int(v) for k, v in (bands or {}).items()
                 if str(k) in ("suspect", "unknown") and v}
        if _grey:
            out.append(("warn", "Not every half-cycle is clean",
                        "; ".join(f"{v} {k}" for k, v in _grey.items())
                        + ". None passed more charge than the material "
                          "can hold, so none is anomalous — but they are "
                          "not clean either, and the `reason` column of "
                          "`*_cycle_integrity.csv` says what was seen on "
                          "each."))
        _m2 = str((f.get("resolvability") or {}).get("mechanism") or "")
        if _m2 and _m2 not in ("two_phase", "multi_transition"):
            # Same fault, one paragraph up: "it sat on a plateau" is only an
            # innocent explanation where a plateau is expected.
            out.append(("warn", "Reversed charge, on a plateau this "
                                "mechanism should not have",
                        f"Every half-cycle's reversed charge sat either on a "
                        f"flat region or at the voltage limit with the current "
                        f"tapering. The second is expected. The first is not, "
                        f"on a dataset classified "
                        f"**{_m2.replace('_', ' ')}** — see below."))
        else:
            # The suspect/unknown caveat is now raised once, above, for
            # either mechanism branch. What is left here is the all-clear,
            # and it is only an all-clear when the table is in fact clean.
            if not _grey:
                out.append(("ok", "No unaccounted charge",
                            "Every half-cycle's reversed charge sat either on "
                            "a two-phase plateau or at the voltage limit with "
                            "the current tapering. Both are expected."))
            else:
                out.append(("ok", "The rest is accounted for",
                            "Setting aside the half-cycles named above, every "
                            "half-cycle's reversed charge sat either on a "
                            "two-phase plateau or at the voltage limit with "
                            "the current tapering. Both are expected."))
    if bands.get("too few records"):
        out.append(("warn", f"{bands['too few records']} half-cycle(s) too "
                    f"short to judge",
                    "Usually the tail of the record after the cell stopped "
                    "delivering. Not the same as clean: it means we could not "
                    "tell."))
    # EVERY VERDICT IS RENDERED, including the good ones. While only the bad
    # ones printed, an unreadable integrity table and a healthy cell produced
    # the same page — nothing — which is a silence that reads as reassurance.
    _civ = f.get("cell_integrity") or {}
    if _civ.get("verdict") == "unknown":
        out.append(("warn", "Cycle integrity could not be assessed",
                    _civ.get("sentence", "")))
    elif _civ.get("verdict") == "sound":
        out.append(("ok", "No cycle was flagged",
                    _civ.get("sentence", "")))
    elif _civ.get("verdict") == "recurring":
        out.append(("bad", "This cell has a recurring fault",
                    _civ.get("sentence", "")))
    elif _civ.get("verdict") == "formation":
        out.append(("ok", "The flagged cycles are all early",
                    _civ.get("sentence", "")))
    elif _civ.get("verdict") == "clustered":
        out.append(("warn", "A run of flagged cycles, which then stopped",
                    _civ.get("sentence", "")))
    elif _civ.get("verdict") == "isolated":
        out.append(("warn", "Flagged cycles, with no pattern",
                    _civ.get("sentence", "")))
    over = f.get("over_theoretical")
    if over is not None and len(over):
        w = over.iloc[0]
        # `over` is sorted by ratio, so `head(8)` is the EIGHT WORST and not
        # the first eight — and the sentence opens by counting all of them.
        # On NNM cell A that printed "9 half-cycle(s) passed more charge than
        # the material can hold ... Cycles: 1, 13, 19, 26, 34, 44, 54, 69":
        # nine claimed, eight listed, cycle 6 (the mildest, 1.23x) silently
        # dropped. A count that does not match its own list is the same fault
        # as a caption that does not match its figure.
        _cyc = sorted(int(c) for c in over["cycle"].head(8))
        cy = ", ".join(str(c) for c in _cyc)
        if len(over) > len(_cyc):
            cy += f" (the {len(_cyc)} worst of {len(over)} by ratio)"
        out.append(("bad", f"{len(over)} half-cycle(s) passed more charge than "
                    f"the material can hold",
                    f"Worst: cycle {int(w['cycle'])} {w['step']}, "
                    f"{w['capacity']:.0f} mAh/g — {w['capacity_ratio']:.1f}× "
                    f"the theoretical {w['theoretical_mAh_g']:.0f} mAh/g. Over "
                    f"theoretical is not a phase transition and not a "
                    f"measurement artefact; it is charge going into something "
                    f"other than the material, and it usually means something "
                    f"has broken. Cycles: {cy}."
                    + (" These half-cycles are still fitted, tracked and "
                       "drawn — the figure is where you see what happened — "
                       "but their components carry "
                       "`half_cycle_anomalous` and are left out of the drift "
                       "rates, the area retention and the coherence audit. "
                       "Integral fidelity does NOT catch them: the curve "
                       "accounts for the charge faithfully, because the "
                       "charge really was delivered."
                       if f.get("n_anomalous_components") else "")))
    # THE SAME THRESHOLD ITS OWN TAIL CLAUSE USES. This gate was `> 0`
    # while the "a further N mAh/g" clause twenty lines below is `> 1`, so
    # NMC111 cell A — 0.79 mAh/g of plateau charge out of a 277 mAh/g
    # half-cycle, 0.3% of one half-cycle — earned the loudest mark on the
    # page, with `:.0f` rounding 0.79 up to a printed "1".
    if f.get("plateau_sane", 0) >= PLATEAU_REPORTABLE_MAH_G:
        # IS A PLATEAU EXPECTED PHYSICS HERE, OR IS IT THE FAULT?
        #
        # This paragraph called every flat region "a first-order phase
        # transition holding the potential, not a fault" — a correct reading
        # for LTO and the WRONG one for a solid solution, whose potential must
        # slope with composition because there is no two-phase coexistence to
        # pin it. A flat region in a solid solution's charge curve is
        # therefore evidence OF a fault, not evidence against one.
        #
        # The run already knows which case it is in. `assess_resolvability`
        # classifies the mechanism on the reference cycle before anything is
        # fitted, and this check — one module away — was not consulting it. It
        # printed the two-phase sentence for NNM (a P3 solid solution) and for
        # NMC111 cells the same run had just classified `mixed`.
        _mech = str((f.get("resolvability") or {}).get("mechanism") or "")
        _pinned = _mech in ("two_phase", "multi_transition")
        _tail = (f" A further {f['plateau_total'] - f['plateau_sane']:.0f} "
                 f"mAh/g sat on a plateau in half-cycles that were over "
                 f"theoretical, and is not counted here — the shape of a "
                 f"curve cannot vouch for charge the material could not "
                 f"have held."
                 if f.get("plateau_total", 0) - f.get("plateau_sane", 0) > 1
                 else "")
        # A TOTAL, SAID TO BE A TOTAL. `plateau_sane` is
        # `integrity["plateau_charge"].sum()` over every half-cycle in the
        # record, and it was printed bare in mAh/g on a page whose header
        # gives the theoretical capacity in the same unit: LTO cell A read
        # "222.2 mAh/g passed backwards" against a 175 mAh/g theoretical and a
        # cell delivering 130, which reads as a physical impossibility rather
        # than as 21 half-cycles summed. The per-half-cycle worst case is the
        # number that can be compared with a capacity.
        _pn = int(f.get("plateau_n_half_cycles") or 0)
        _pw = f.get("plateau_worst_mAh_g")
        _lead = (f"{f['plateau_sane']:.1f} mAh/g in total"
                 + (f" across {_pn} half-cycles" if _pn else "")
                 + " passed backwards while sitting on a redox plateau, on "
                   "half-cycles whose total charge is physically possible"
                 + (f" — at most {_pw:.1f} mAh/g in any one of them"
                    if _pw is not None and np.isfinite(_pw) else "")
                 + ".")
        if not _mech:
            out.append(("ok", "Plateau detected",
                        _lead + " On a material whose potential is pinned by "
                        "two phases coexisting that is expected physics; on a "
                        "solid solution it is not. This run did not classify "
                        "the mechanism, so which of the two this is has not "
                        "been established here." + _tail))
        elif _pinned:
            out.append(("ok", "Two-phase plateau detected",
                        _lead + f" This dataset is classified "
                        f"**{_mech.replace('_', ' ')}**, so a region where the "
                        f"potential holds still is the transition itself: a "
                        f"first-order phase change pinning the potential, not "
                        f"a fault." + _tail))
        else:
            out.append(("bad", "A plateau here is not expected physics",
                        _lead + f" This dataset is classified "
                        f"**{_mech.replace('_', ' ')}**. A solid solution "
                        f"delivers its charge across a window of composition, "
                        f"and its potential must move as that composition "
                        f"changes — there is no two-phase coexistence to pin "
                        f"it. A flat region in this cell's curve is therefore "
                        f"evidence OF something, not evidence against it: a "
                        f"voltage stall, a hold, or charge going somewhere "
                        f"other than the intended reaction. Earlier builds "
                        f"reported this as a first-order phase transition; "
                        f"that reading belongs to the other mechanism."
                        + _tail))
    if f.get("fit_limit") or f.get("death_cycle"):
        out.append(("warn", f"Cell stopped delivering at cycle "
                    f"{f.get('fit_limit') or f.get('death_cycle')}",
                    f"Peak fitting stopped there. The plots still show all "
                    f"{n} cycles, because a cell going flat is exactly what "
                    f"the heatmap is for."))

    w, tot = f.get("withheld", (0, 0))
    cw = f.get("closure_widths")
    if tot:
        if w:
            # WHICH GROUND. Attribution is withheld either because the
            # peak/background split is undetermined (closure) or because the
            # curve does not account for the cell's charge (fidelity), and
            # naming the wrong one sends the reader to look at the baseline
            # when the problem is the instrument's voltage resolution.
            _fid = _fidelity_outside_band(f.get("integral_fidelity"))
            if _fid:
                # SHORT, and pointing at the finding that explains it. The
                # full argument is one item below under "Peak areas here are
                # not capacities"; repeating it here made the page read as
                # two separate problems when it is one.
                why = ("There is no capacity here to attribute — see *Peak "
                       "areas here are not capacities* below. The fitted "
                       "areas are still in the CSV.")
                # NOT WHEN THERE IS NO FREE BACKGROUND. `no_baseline` means
                # every measured closure width is zero BY CONSTRUCTION, so
                # this printed "closure interval only 0.00–0.00 wide: the
                # fits are good" — reassurance drawn from a measurement that
                # could not have come out any other way, and in direct
                # contradiction of the branch below, which says the split
                # "cannot be undetermined" because there is nothing to
                # determine.
                if cw and not f.get("no_baseline"):
                    why += (f" Note that the peak/background split itself is "
                            f"well determined, closure interval only "
                            f"{cw[0]:.2f}–{cw[1]:.2f} wide: the fits are "
                            f"good, and that is a different question from "
                            f"the one this fails.")
            elif f.get("unattributed_withheld"):
                _u = f.get("unattributed")
                _ur = f.get("unattributed_range")
                why = ("The named components do not add up to the cell's "
                       "charge. Every mAh in a dQ/dV passed through the "
                       "cell, so a share OF a total that is missing part of "
                       "the capacity is not a measurement — the fitted areas "
                       "are still in the CSV, and the SHARE is blank."
                       + (f" Unattributed {_u:+.0%} of the charge"
                          + (f" (range {_ur[0]:+.0%} to {_ur[1]:+.0%})"
                             if _ur else "") + "."
                          if _u is not None else ""))
            elif f.get("no_baseline"):
                why = ("This model has no free background, so the "
                       "peak/background split cannot be undetermined. These "
                       "cycles were withheld because the check fit did not "
                       "converge on them: there is no measured share to "
                       "report. The fitted areas are still in the CSV.")
            else:
                why = ("Where a broad peak and the polynomial baseline are "
                       "nearly the same function, the split between them is "
                       "not decided by the data: refitting with a different "
                       "baseline moves the peak's share of the capacity by "
                       "more than the number would be reporting. The fitted "
                       "areas are still in the CSV; the SHARE is blank "
                       "because it is not a measurement."
                       + (f" Closure interval {cw[0]:.2f}–{cw[1]:.2f} wide."
                          if cw and not f.get("no_baseline") else ""))
            # HALF-CYCLES. `w` and `tot` are summed over both steps, so a
            # 67-cycle cell was told "134 of 134 cycles" — a count of twice
            # what it ran, in the one unit this page otherwise says
            # carefully everywhere else.
            out.append(("warn", f"Capacity attribution withheld on {w} of "
                        f"{tot} half-cycles", why))
        elif f.get("no_baseline"):
            _u = f.get("unattributed")
            out.append(("ok", "Capacity attribution is reportable",
                        "There is no free background in this model, so there "
                        "is no peak/background split to be undetermined — "
                        "every component is named and its area is a share of "
                        "the cell rather than of a polynomial."
                        + _unattributed_clause(_u, f.get("unattributed_range"))
                        + " A HIGH NAMED FRACTION IS NOT BY ITSELF A BETTER "
                          "DECOMPOSITION: a wide band can absorb a great "
                          "deal, and a few resolved transitions with an "
                          "honest few percent left over is the better "
                          "answer. Read it beside the mechanism."))
        else:
            out.append(("ok", "Capacity attribution is reportable",
                        "Refitting with a different baseline degree barely "
                        "moved the peak/background split"
                        + (f" (closure interval {cw[0]:.2f}–{cw[1]:.2f} wide)"
                           if cw else "") + ", so the per-peak shares mean "
                        "what they say."))

    if f.get("reference_cycle"):
        # The severity is now a VALUE carried from `detect.detect_all`, not a
        # substring search on the sentence. The search read
        # `"warn" if "failed" in reason else "ok"` and was exactly inverted:
        # "NO cycle passes the integrity check — using 3 unchecked" and
        # "FLAGGED ANOMALOUS by the integrity check" contain no "failed" and
        # rendered green, while the benign "cycle 5 failed the integrity
        # check; 6 is the first that passes" rendered as a warning.
        lvl = f.get("reference_severity") or "ok"
        out.append((lvl, f"Peaks are tracked against cycle "
                    f"{f['reference_cycle']}",
                    _sentence_case(f.get("reference_reason", ""))
                    + ". That cycle's peak list "
                    f"is fitted in every other cycle, so it decides what the "
                    f"whole dataset is measured against."))
    bad_peaks = [s for s in f.get("tracked", [])
                 if s.get("discontinuity_is_failure")]
    if bad_peaks:
        out.append(("warn", f"{len(bad_peaks)} tracked peak(s) failed mid-life",
                    ", ".join(f"{s['step']} {s['reference_voltage']:.3f} V "
                              f"from cycle "
                              f"{int(s['first_discontinuity_cycle'])}"
                              for s in bad_peaks)
                    + ". Each was reliably tracked for at least 15 cycles "
                      "before it went, so this is a feature being lost, not a "
                      "feature that never established."))
    return out


def _experiment(f):
    """
    What was done: the cell, not its performance.

    First, because a capacity means nothing until you know the mass it is per,
    the rate it was measured at and the window it was measured in.
    """
    p = f["params"]
    src = " (from file)" if p.get("barcode") else ""
    out = []
    if p.get("active_material_mass_mg"):
        out.append(("Active mass", f"{p['active_material_mass_mg']:.2f} mg",
                    (p.get("blend") or "") + src))
    if p.get("theoretical_capacity_mAh_g"):
        out.append(("Theoretical", f"{p['theoretical_capacity_mAh_g']:.0f}",
                    "mAh/g"))
    # THE RATE THE CELL WAS CYCLED AT, not the one in the cycler header. A
    # rate-capability run reports its range and its block count; a
    # single-rate run reads exactly as it did before. `rate_protocol` is
    # attached to the parameters in Cell 6a. See `cycling.describe_rate`.
    _rp = p.get("rate_protocol")
    if _rp and _rp.get("available") and _rp.get("is_variable"):
        out.append(("Rate", str(_rp.get("label")),
                    f"{_rp.get('n_blocks')} rate blocks — variable-rate run"))
    elif p.get("charge_rate_c"):
        # When the current was recorded, say what it MEASURED as well as what
        # was entered — the two disagreeing is worth seeing, and the entered
        # value comes from a header that only describes the first step.
        _m = (_rp or {}).get("label")
        out.append(("Rate", f"{p['charge_rate_c']} C",
                    f"measured {_m}" if _m else ""))
    if p.get("voltage_lower_V") and p.get("voltage_upper_V"):
        out.append(("Window", f"{p['voltage_lower_V']}–{p['voltage_upper_V']} V",
                    "from the cycler file"
                    if p.get("voltage_window_source") == "file" else
                    ("at least this wide — short record"
                     if str(p.get("voltage_window_source", "")).startswith(
                         "observed range") else "as set")))
    if p.get("active_loading_mg_cm2"):
        out.append(("Loading", f"{p['active_loading_mg_cm2']:.2f}",
                    f"mg/cm² on {p.get('electrode_diameter_mm', 0):.0f} mm"))
    if p.get("counter_electrode_metal"):
        # WHERE THE METAL CAME FROM, not just that the mass was estimated. A
        # counter metal that was ASSUMED sets the density used in every areal
        # and gravimetric energy figure (Li 0.534 against Na 0.97 g/cm3), and
        # reading "estimated" gave no clue that the element itself was a guess.
        _src = {"stated": "as stated",
                "chemistry": "from the electrolyte / formula",
                "library": "from the composition library — confirm it",
                "assumed": "ASSUMED — chemistry could not be inferred"}.get(
            p.get("counter_electrode_metal_source"), "")
        _mass = ("measured" if p.get("counter_electrode_measured")
                 else "mass estimated")
        out.append(("Counter", p["counter_electrode_metal"],
                    f"{_mass}{(', ' + _src) if _src else ''}"))
    if f.get("n_cycles"):
        _died = f.get("fit_limit") or f.get("death_cycle")
        if f.get("partial_final"):
            # The count of COMPLETE cycles, which is what the numbers above
            # this tile are drawn from. The record length goes in the note.
            out.append(("Cycles complete",
                        str(f["partial_final"][0] - 1),
                        f"cycle {f['partial_final'][0]} still running"))
        else:
            out.append(("Cycles run", str(f["n_cycles"]),
                        f"working to {_died}" if _died else ""))
    return out


# RETENTION ACROSS RATE BLOCKS IS NOT RETENTION. On a rate-capability run
# the capacity falls because the current rose and comes back when it falls
# again, and quoting `Q(n)/Q(ref)` across that reads as a cell fading and
# recovering. On `JQ_NNM_C-rate_2-4.2V_B_29042026` it produced 78.5% and a
# key-capacity column of 75%, 64%, 80% — every one of which a reader would
# take for degradation. The figure is not wrong arithmetic; it is the wrong
# question, so it is withheld and the reason is given.
def _rate_variable(f):
    return bool(((f.get("params") or {}).get("rate_protocol")
                 or {}).get("is_variable"))


def _rate_caveat(f):
    rp = (f.get("params") or {}).get("rate_protocol") or {}
    if not rp.get("is_variable"):
        return None
    return (f"This cell was cycled at {rp.get('n_blocks')} different rates "
            f"({rp.get('label')}). Capacity against cycle number here is "
            f"mostly the rate schedule, not age: it falls because the "
            f"current rose and returns when the current falls again. "
            f"Retention and percent-of-first are therefore withheld — they "
            f"would read as fading and recovery. The rate-capability summary "
            f"compares each rate against the reference rate, and the "
            f"recovery to the starting rate is reported there.")


def _stats(f):
    """The results tiles: what the cell delivered."""
    p = f["params"]
    out = []
    # Discharge capacity leads, because that is what a paper reports.
    _var = _rate_variable(f)
    # NOT `[:4]`. The tiles stopped at four, so a run with key cycles
    # 1, 5, 10, 20, 30, 40, 50, 128 showed the first four and the page then
    # said "Retention 34% at cycle 128" with no capacity for cycle 128
    # anywhere on it. The Markdown table always printed them all.
    for cyc, d, pct in f.get("key_capacities", []):
        out.append((f"Cycle {cyc}", f"{d:.0f}",
                    "mAh/g" + (f"  ·  {pct:.0f}%"
                               if pct is not None and not _var else "")))
    if _var:
        out.append(("Retention", "—", "withheld: several rates in this run"))
    elif f.get("retention") is not None and f.get("last_good_cycle"):
        out.append(("Retention", f"{f['retention']:.0f}%",
                    f"at cycle {f['last_good_cycle']}, "
                    f"vs cycle {f.get('retention_reference') or 2}"))
    if f.get("median_ce") is not None:
        out.append(("Median CE", f"{f['median_ce']:.1f}%", ""))
    return out


def build_report(name, *, dataset=None, params=None, integrity=None,
                 detection=None, tracking=None, attribution=None,
                 cycle_table=None, fit_limit=None, profile=None,
                 dataset_dir=None, run_id="", file_format="png",
                 in_progress=None, coherence=None, resolvability=None,
                 parameters=None):
    """
    The two documents, as strings.

    `dataset_dir` is the per-dataset folder inside the run — the one holding
    `1_cycling`, `4_dqdv` and the rest. Figures are looked up there; if it is
    None or the figures are missing, both documents are still written and say
    so rather than failing.
    """
    f = _facts(name, params, integrity, cycle_table, fit_limit, detection,
               tracking, attribution, None, profile, in_progress=in_progress,
               coherence=coherence, resolvability=resolvability,
               parameters=parameters)
    # SEVERITY FIRST, and stably so. `_peak_sentences` came second by
    # construction, so every `bad` finding about the peaks sat below every
    # routine `ok` one about the cycling — on NNM cell B the paragraph that
    # invalidates every area on the page was twelfth, under "Capacity
    # attribution is reportable", and a `warn` twelve lines above it
    # forward-referenced it.
    _SEVERITY = {"bad": 0, "warn": 1, "ok": 2}
    findings = sorted(_sentences(f) + _peak_sentences(f),
                      key=lambda t: _SEVERITY.get(t[0], 3))
    stats = _stats(f)
    experiment = _experiment(f)
    p = f["params"]

    # THE MECHANISM BELONGS IN THE HEADER, next to the profile class it is
    # constantly confused with. `profile` describes the SHAPE of the curve;
    # `mechanism` says what produced it, and it is the mechanism that chose
    # the model. NNM and NMC111 are both "broad" and one of them is a series
    # of real transitions.
    _mech = (f.get("resolvability") or {}).get("mechanism")
    subtitle = " · ".join(x for x in [
        p.get("electrode_type", "") and f"{p['electrode_type'].lower()} electrode",
        p.get("battery_chemistry", ""),
        f"profile {f['profile']}" if f.get("profile") else "",
        f"mechanism {str(_mech).replace('_', ' ')}" if _mech else "",
        run_id] if x)

    # FIGURE_GUIDE names its files with a .png suffix, but a run exported as
    # tiff writes *_cycle_life.tiff and every figure went missing from the
    # report of a run that had produced them.
    _ext = "." + str(file_format or "png").lstrip(".").lower()
    figs = []
    for _entry in FIGURE_GUIDE:
        folder, suffix, title, blurb = _entry[:4]
        run_wide = bool(_entry[4]) if len(_entry) > 4 else False
        suf = suffix[:-4] + _ext if suffix.endswith(".png") else suffix
        path = (_find(os.path.join(dataset_dir, folder), suf, name, run_wide)
                if dataset_dir else None)
        if path is None and dataset_dir and suf != suffix:
            path = _find(os.path.join(dataset_dir, folder), suffix, name,
                         run_wide)
        figs.append((title, blurb, path,
                     _rel(path, dataset_dir) if path else None))

    # ---------------------------------------------------------------- HTML
    h = ["<!doctype html><html lang='en-GB'><head><meta charset='utf-8'>",
         "<meta name='viewport' content='width=device-width,initial-scale=1'>",
         f"<title>{_esc(f['title'])} — Ratatosk</title>"
         f"<style>{_CSS}</style>",
         "</head><body><div class='wrap'>",
         f"<h1>{_esc(f['title'])}</h1>",
         f"<p class='sub'>{_esc(subtitle)}</p>"]

    def _tiles(rows):
        out = ["<div class='grid'>"]
        for k, v, n2 in rows:
            out.append(f"<div class='stat'><div class='k'>{_esc(k)}</div>"
                       f"<div class='v'>{_esc(v)}</div>"
                       f"<div class='n'>{_esc(n2)}</div></div>")
        out.append("</div>")
        return out

    if experiment:
        h.append("<h2>The experiment</h2>")
        h += _tiles(experiment)
    if stats:
        h.append("<h2>Discharge capacity</h2>")
        h += _tiles(stats)
        _cav = _rate_caveat(f)
        if _cav:
            h.append(f"<p class='n'>{_esc(_cav)}</p>")
        else:
            h.append("<p class='n'>Percentages are of the first complete "
                     "discharge.</p>")

    h.append("<h2>What the run found</h2>")
    for lvl, head, text in findings:
        # The Markdown renderer writes "**Head.** text"; the HTML ran the two
        # straight together with no stop and no space.
        h.append(f"<div class='flag {lvl}'><b>{_esc(head)}.</b> "
                 f"{_md_inline(_esc(text))}</div>")

    if f["peaks"].get("Charge") or f["peaks"].get("Discharge"):
        h.append("<h2>Redox features being tracked</h2>")
        _kinds = _kind_by_ref(f)
        h.append("<table><tr><th>Step</th><th>Reference peak / V</th>"
                 "<th>Fitted as</th></tr>")
        for step in ("Charge", "Discharge"):
            for x in (f["peaks"].get(step) or []):
                h.append(f"<tr><td>{step}</td><td>{x:.3f}</td><td>"
                         f"{_esc(_kinds.get((step, round(float(x), 3)), '—'))}"
                         f"</td></tr>")
        h.append("</table>")
        if not _kinds:
            h.append("<p class='n'>Nothing here was fitted — these are the "
                     "detected reference features only.</p>")

    h.append("<h2>Four figures, in this order</h2>")
    for i, (title, blurb, path, rel) in enumerate(figs, 1):
        h.append(f"<div class='fig'><h3>{i}. {_esc(title)}</h3>"
                 f"<p class='cap'>{_esc(blurb)}</p>")
        src = _embed(path) if path else None
        if src:
            h.append(f"<img alt='{_esc(title)}' src='{src}'>")
        elif path:
            h.append(f"<p class='n'>(figure written as "
                     f"<code>{_esc(os.path.splitext(path)[1])}</code>, which "
                     f"a browser cannot show — open it from the folder)</p>")
        else:
            h.append("<p class='n'>(figure not found in this run)</p>")
        if path:
            h.append(f"<p class='n'><code>{_esc(_rel(path, dataset_dir))}"
                     f"</code></p>")
        h.append("</div>")

    h.append("<h2>Where everything else is</h2><table>"
             "<tr><th>Folder</th><th>What is in it</th></tr>")
    for folder, what in _FOLDERS:
        h.append(f"<tr><td><code>{folder}</code></td><td>{what}</td></tr>")
    h.append("</table>")
    h.append(f"<footer>Ratatosk {run_id or ''} — images are embedded, so this "
             f"file can be moved or emailed on its own. Every number here "
             f"comes from the CSVs in the folders above.</footer>")
    h.append("</div></body></html>")

    # ------------------------------------------------------------ Markdown
    m = [f"# {f['title']}", ""]
    if subtitle:
        m += [f"*{subtitle}*", ""]
    if experiment:
        # THREE COLUMNS. The note is sometimes a unit ("mAh/g"), sometimes
        # a provenance ("measured C/10"), sometimes both with a dash already
        # in it — so no single separator can join it to the value without
        # reading wrongly in one of those cases. It gets its own column.
        m += ["## The experiment", "", "| | | |", "|---|---|---|"]
        # A SEPARATOR. "| Cycles complete | 10 cycle 11 still running |"
        # and "| Rate | 0.1 C measured C/10 |" ran the value straight into
        # its note, so both read as one garbled value.
        m += [f"| {_md_cell(k)} | {_md_cell(v)} | {_md_cell(n2 or '')} |"
              for k, v, n2 in experiment]
        m += [""]
    if f.get("key_capacities"):
        _var = _rate_variable(f)
        m += ["## Discharge capacity", "",
              ("| Cycle | mAh/g |" if _var
               else "| Cycle | mAh/g | % of first |"),
              ("|---|---|" if _var else "|---|---|---|")]
        m += [(f"| {c} | {d:.0f} |" if _var else
               f"| {c} | {d:.0f} | "
               f"{('%.0f%%' % pct) if pct is not None else '—'} |")
              for c, d, pct in f["key_capacities"]]
        extra = [(k, v, n2) for k, v, n2 in stats if not k.startswith("Cycle ")]
        if extra:
            m += [""] + [f"**{k}** {v} {n2}".strip() for k, v, n2 in extra]
        _cav = _rate_caveat(f)
        if _cav:
            m += ["", _cav]
        m += [""]
    m += ["## What the run found", ""]
    MARK = {"ok": "", "warn": "⚠ ", "bad": "✘ "}
    for lvl, head, text in findings:
        m += [f"**{MARK.get(lvl, '')}{head}.** {text}", ""]
    if f["peaks"].get("Charge") or f["peaks"].get("Discharge"):
        _kinds = _kind_by_ref(f)
        m += ["## Redox features being tracked", "",
              "| Step | Reference peak / V | Fitted as |", "|---|---|---|"]
        for step in ("Charge", "Discharge"):
            for x in (f["peaks"].get(step) or []):
                m.append(f"| {step} | {x:.3f} | "
                         f"{_kinds.get((step, round(float(x), 3)), '—')} |")
        if not _kinds:
            m += ["", "Nothing here was fitted — these are the detected "
                      "reference features only."]
        m += [""]
    m += ["## Four figures, in this order", ""]
    for i, (title, blurb, path, rel) in enumerate(figs, 1):
        m += [f"### {i}. {title}", "", blurb, ""]
        # ANGLE BRACKETS ROUND THE DESTINATION. Every dataset name in this
        # project contains spaces and parentheses —
        # `JQ_LTO_2.5-1.2V_0.1C_A_08052026 (2)` — and CommonMark cannot parse
        # a bare destination containing either, so EVERY figure in EVERY
        # START_HERE.md rendered as literal text instead of an image. The
        # HTML page embeds its images as data URIs and was unaffected, which
        # is why this survived.
        m += [f"![{title}](<{rel}>)" if rel else "*(figure not found)*", ""]
    m += ["## Where everything else is", "", "| Folder | What is in it |",
          "|---|---|"]
    m += [f"| `{folder}` | {what} |" for folder, what in _FOLDERS]
    m += ["", f"---", "",
          "Generated by Ratatosk. Every number above comes from the CSVs in "
          "the folders listed."]
    return "\n".join(h), "\n".join(m), f


_FOLDERS = [
    ("1_cycling", "capacity, coulombic efficiency, retention, fade, "
     "average voltage, energy efficiency — and the cycle integrity table"),
    ("2_voltage_profiles", "voltage against capacity, all cycles and key cycles"),
    ("3_energy_power", "energy and power density on three bases, and the "
     "Ragone plot"),
    ("4_dqdv", "differential capacity: overlays, waterfalls, heatmaps, and "
     "the processed curves as CSV"),
    ("5_peak_fitting", "detected peaks, the fits themselves, every fitted "
     "parameter, and the coherence audit"),
    ("6_descriptors", "tracked peaks, polarisation, capacity attribution"),
    ("7_summary", "the per-cycle summary table"),
]


def write_report(dataset_dir, name, **kwargs):
    """
    Write both files at the top of the dataset folder.

    Returns `(html_path, md_path, facts)` — the facts go to
    `write_run_summary`, so the run-level page is assembled from exactly the
    numbers each per-cell page reported rather than recomputed.
    """
    html_doc, md, facts = build_report(name, dataset_dir=dataset_dir, **kwargs)
    os.makedirs(dataset_dir, exist_ok=True)
    ph = os.path.join(dataset_dir, "START_HERE.html")
    pm = os.path.join(dataset_dir, "START_HERE.md")
    with open(ph, "w", encoding="utf-8") as fh:
        fh.write(html_doc)
    with open(pm, "w", encoding="utf-8") as fh:
        fh.write(md)
    return ph, pm, facts


# =============================================================================
# The run summary — one page for the whole run
# =============================================================================
# `START_HERE` answers "what did this cell do". A run is usually a triplicate,
# and the question a triplicate exists to answer is different: do the cells
# agree, and if not, which one is lying?
#
# 1.8.x answered that with MAD-proximity scoring — pick the cell closest to the
# group median on every metric. That rewards a cell for being average, which is
# exactly wrong when the failure mode is all three cells being wrong in the
# same way, and it says nothing about whether the spread is small enough for
# the median to mean anything. So this reports the SPREAD first, names the
# outlier only when there is one, and declines to nominate a representative
# cell when the cells do not agree.

REPRESENTATIVE_MAX_SPREAD = 0.15    # relative spread above which "typical"
                                    # is not a thing this run has

# --- what a triplicate REPORTS ------------------------------------------
# Published reproducibility practice for replicate cells is mean +/- standard
# deviation across the replicates, with the individual cells shown. Nominating
# one cell as representative is a SELECTION STEP, and selection steps are
# where irreproducibility enters — it is the same move as choosing the
# prettiest diffraction pattern.
#
# So: the headline for a group of replicates is mean +/- SD with n stated, the
# representative cell is nominated for FIGURES ONLY and labelled as such, and
# the representative cell's number is never the reported number. A caption
# reads "cell B shown; n = 3, retention 98.6 +/- 0.4%", which is a stronger
# claim than any single cell can make and is what a reviewer would ask for.
#
# The SD is the sample standard deviation (ddof=1) — an estimate of the
# population spread from a sample, which is what three cells are. numpy's
# default ddof=0 is the population SD and understates a triplicate's spread
# by 18%.
# Mirrors `analyse.INTEGRAL_FIDELITY_FLOOR` and `INTEGRAL_FIDELITY_CEILING`.
# Restated rather than imported so `report` keeps its one-way dependency,
# exactly as DEAD_CELL_RUN is; if one moves, move both. The band is TWO-SIDED
# since 1.9.0.23: a curve carrying more charge than the cell delivered is as
# disqualifying as one carrying less, and only the deficit used to be caught.
INTEGRAL_FIDELITY_FLOOR = 0.80
INTEGRAL_FIDELITY_CEILING = 1.20


def _fidelity_outside_band(fid):
    """The half-cycles whose integral fidelity is outside the band, either
    way, as `{key: value}`."""
    return {k: v for k, v in (fid or {}).items()
            if np.isfinite(v) and not (INTEGRAL_FIDELITY_FLOOR <= v
                                       <= INTEGRAL_FIDELITY_CEILING)}

REPLICATE_SD_DDOF = 1
# READ, not decorative. This was defined and the guard below used a literal
# 2, so changing the documented knob did nothing.
REPLICATE_MIN_N = 2


def _spread(values):
    """
    Relative spread of a group: (max - min) / median. NaN if not usable.

    Coerced through pandas rather than filtered with `np.isfinite`, because
    these values come from report facts where a missing number is `None`, and
    `np.isfinite(None)` raises rather than returning False.
    """
    v = pd.to_numeric(pd.Series(list(values)), errors="coerce").dropna().values
    if v.size < 2:
        return np.nan
    med = float(np.median(v))
    if not np.isfinite(med) or med == 0:
        return np.nan
    return float((v.max() - v.min()) / abs(med))


# Two tracked peaks are the same redox feature if their reference voltages sit
# closer than this. Mirrors `analyse.TRACKING_TOLERANCE_MV`, which is the
# pipeline's existing answer to the same question, restated rather than
# imported to keep `report`'s one-way dependency.
FEATURE_GROUP_TOLERANCE_MV = 80.0


def _group_drift(points):
    """Group (reference_voltage, cell_name, drift) into redox features.

    Sorted by voltage and split wherever consecutive peaks are further apart
    than `FEATURE_GROUP_TOLERANCE_MV`. Returns one dict per feature with the
    mean voltage, and mean/sd/n of the drift where **n counts CELLS** — a
    cell contributing two peaks to one feature contributes their median, so
    it is never counted twice and never inflates the spread.
    """
    if not points:
        return []
    pts = sorted(points, key=lambda t: t[0])
    groups, cur = [], [pts[0]]
    for pt in pts[1:]:
        if (pt[0] - cur[-1][0]) * 1000.0 > FEATURE_GROUP_TOLERANCE_MV:
            groups.append(cur); cur = [pt]
        else:
            cur.append(pt)
    groups.append(cur)

    out = []
    for grp in groups:
        per_cell = {}
        for _pt in grp:
            per_cell.setdefault(_pt[1], []).append(_pt[2])
        vals = [float(np.median(v)) for v in per_cell.values()]
        mean, sd, n = _mean_sd(vals)
        # A point may carry the coherence verdict as a fourth element. A
        # feature backed only by "questionable" peaks is published — the
        # audit did not reject it — but marked, because "questionable" and
        # "coherent" are not the same evidence and a table that prints them
        # identically is asserting they are.
        _verds = {_pt[3] for _pt in grp if len(_pt) > 3 and _pt[3]}
        # NO VERDICT IS NOT A PASS. A peak with no row in the coherence table
        # — the audit never reached it, or peak fitting ran without the audit
        # — was let through the filter above and then printed in the same
        # typeface as an audited one, under a caption asserting that only
        # trend-worthy features are in the table. Unaudited is a third state
        # and gets its own mark.
        _unaudited = all(len(_pt) < 4 or not _pt[3] for _pt in grp)
        out.append(dict(voltage=float(np.mean([p[0] for p in grp])),
                        mean=mean, sd=sd, n=n, n_peaks=len(grp),
                        unaudited=bool(_unaudited),
                        questionable_only=bool(_verds)
                        and _verds <= {"questionable"}))
    return out


def _mean_sd(values, *, ddof=REPLICATE_SD_DDOF):
    """
    (mean, sd, n) across replicates. See REPLICATE_SD_DDOF.

    `sd` is NaN for a single cell — one measurement has no spread, and
    reporting 0 would claim a precision nothing established.
    """
    v = pd.to_numeric(pd.Series(list(values)), errors="coerce").dropna().values
    if v.size == 0:
        return (np.nan, np.nan, 0)
    if v.size < 2:
        return (float(v[0]), np.nan, 1)
    return (float(v.mean()), float(v.std(ddof=ddof)), int(v.size))


def _pm(mean, sd, n, unit="", dp=1, show_n=False):
    """`98.6 +/- 0.4% (n = 3)`, or the bare value when there is no spread.

    `show_n` keeps the count on a single measurement too. A table whose caption
    says "n is the number of CELLS in which that feature was tracked" must
    print n on every row, or a one-cell row is indistinguishable from a
    two-cell row whose replicates happened to agree exactly — which is the one
    thing that caption exists to prevent.
    """
    if not np.isfinite(mean):
        return "—"
    if not np.isfinite(sd) or n < 2:
        return (f"{mean:.{dp}f}{unit} (n = {n})" if show_n and n
                else f"{mean:.{dp}f}{unit}")
    return f"{mean:.{dp}f} ± {sd:.{dp}f}{unit} (n = {n})"


def _pct(x):
    """A relative spread as a percentage that never rounds away to nothing.

    `{:.0%}` turned a 0.34% spread into "0%", which reads as "the cells are
    identical" one line below a table quoting +/- 0.2. A spread small enough
    to round to zero is small, not absent, and the page should say so.
    """
    if not np.isfinite(x):
        return "—"
    if x <= 0:
        return "0%"
    if x < 0.005:
        return "<1%"
    if x < 0.095:
        return f"{100 * x:.1f}%"
    return f"{100 * x:.0f}%"


def _last_cell(last, last_cycle):
    """Last working capacity, with its cycle number when there is one."""
    if not np.isfinite(last):
        return "—"
    if not np.isfinite(last_cycle):
        return f"{last:.0f}"
    return f"{last:.0f} (c{int(last_cycle)})"


def build_run_summary(facts_by_name, *, run_id="", n_files=0):
    """
    One page for the whole run: the cells side by side, and whether they agree.

    `facts_by_name` is `{name: dict}` as `build_report` assembles — pass the
    `facts` each per-dataset report returned.
    """
    names = list(facts_by_name)
    rows = []
    for n in names:
        f = facts_by_name[n]
        p = f.get("params", {})
        dl = f.get("first_last_discharge") or (np.nan, np.nan)
        anom = f.get("anomalous")
        over = f.get("over_theoretical")
        rows.append(dict(
            name=n,
            composition=str(p.get("composition", n)),
            label=(f"{p.get('composition', n)}"
                   + (f" {p['cell_id']}" if p.get("cell_id") else "")),
            mass=p.get("active_material_mass_mg", np.nan),
            cycles=f.get("n_cycles", np.nan),
            first=dl[0], last=dl[1],
            last_cycle=f.get("last_good_cycle", np.nan),
            retention=f.get("retention", np.nan),
            unusable="; ".join(f.get("unusable") or []) or None,
            # The arithmetic that was barred from being called retention on a
            # variable-rate run. Kept under its own name so it is not LOST —
            # which it was, because nothing read it.
            retention_uncorrected=f.get("retention_uncorrected", np.nan),
            rate_is_variable=bool(f.get("rate_is_variable")),
            rate_label=(((f.get("params") or {}).get("rate_protocol")
                         or {}).get("label")),
            ce=f.get("median_ce", np.nan),
            died=f.get("fit_limit") or f.get("death_cycle"),
            n_anom=(len(anom) if anom is not None else 0),
            n_over=(len(over) if over is not None else 0),
            withheld=f.get("withheld", (0, 0))[0],
            unattributed=f.get("unattributed"),
            unattributed_withheld=f.get("unattributed_withheld", 0),
            no_baseline=f.get("no_baseline", False),
            mechanism_own=str((f.get("resolvability") or {}).get(
                "mechanism_own") or "") or None,
            # WHICH MODEL EACH CELL WAS FITTED WITH. The run page exists to
            # say whether the cells agree, and two replicates classified
            # differently — one `multi_transition`, one `mixed` — are not
            # being compared like with like. That is a difference this page
            # must show, not one a reader should have to open three folders
            # to find.
            mechanism=str((f.get("resolvability") or {}).get("mechanism")
                          or ""),
            # The WORST step fidelity for this cell, so the run summary can
            # say which ground the withholding rested on and can carry the
            # caveat itself. A reader who opens only this page must not be
            # left thinking the areas are capacities.
            # The WORST is the one FURTHEST FROM 1, not the smallest. The
            # band is two-sided (0.80-1.20); `min` recorded 0.95 for a cell
            # whose charge step carried 1.45, so the run page saw a healthy
            # number and the excess never appeared on it.
            fidelity=max([v for v in (f.get("integral_fidelity") or {}).values()
                          if np.isfinite(v)],
                         key=lambda v: abs(v - 1.0), default=np.nan),
            partial=bool(f.get("partial_final")),
        ))
    _COLS = ["name", "label", "composition", "mass", "cycles", "first",
             "last", "last_cycle", "retention", "ce", "died", "n_anom",
             "n_over", "withheld", "fidelity", "partial",
             # 1.9.0.56: the charge the model could not name, and whether
             # that alone withheld a share. Added here as well as to `rows`
             # for the reason the note below gives.
             "unattributed", "unattributed_withheld", "no_baseline",
             "mechanism_own", "rate_is_variable", "rate_label", "unusable",
             "retention_uncorrected",
             # ADDING A FIELD TO `rows` IS NOT ENOUGH. The list below is
             # explicit, so a key that is not in it is silently dropped and
             # every `r.<name>` after it raises AttributeError. That is what
             # took RUN_SUMMARY off the 1.9.0.47 NMC run: the per-cell pages
             # were written, the run page was not, and nothing said why.
             "mechanism"]
    # An explicit column list so a run where every dataset failed produces a
    # summary saying so, rather than a KeyError on the first coercion.
    T = pd.DataFrame(rows, columns=_COLS)

    # Every one of these is rendered with np.isfinite, and every one of them
    # can arrive as None or a string from a report that had nothing to put
    # there. Coerce once, at the boundary, rather than guarding each use:
    # the LTO run died on `int(r.cycles)` because one dataset carried None.
    for _c in ("mass", "cycles", "first", "last", "last_cycle",
               "retention", "ce", "died", "n_anom", "n_over", "withheld",
               "fidelity"):
        T[_c] = pd.to_numeric(T[_c], errors="coerce")
    for _c in ("n_anom", "n_over", "withheld", "unattributed_withheld"):
        T[_c] = pd.to_numeric(T[_c], errors="coerce").fillna(0)
    T["unattributed"] = pd.to_numeric(T["unattributed"], errors="coerce")
    T["no_baseline"] = T["no_baseline"].fillna(False).astype(bool)
    T["rate_is_variable"] = (T["rate_is_variable"].fillna(False)
                             .astype(bool))

    # THE TEXT COLUMNS NEED COERCING AT THE BOUNDARY TOO, and for a reason
    # the numeric ones do not have. pandas 3 infers a StringDtype whose
    # missing value is `float('nan')`, not None. nan is TRUTHY, so it passes
    # the `if v` guard on `mechanism_own` below and then raises on
    # `v.replace`, taking RUN_SUMMARY.md off the run with an AttributeError
    # after every per-cell page has already been written — the same silent
    # shape as the 1.9.0.47 missing-column failure two notes up.
    #
    # The column is only MIXED, and therefore only inferred as StringDtype,
    # when some cells went through reconciliation and others did not:
    # `reconcile_mechanisms` setdefaults `mechanism_own` on EVERY cell of a
    # group it changed, so a lone triplicate is all-string or all-None and
    # comes back as object either way. Two materials in one run, or one cell
    # left out of a reconciled group, is the mix that breaks it — which is a
    # normal thing to ask for, three datasets at a time out of a folder of
    # eight.
    #
    # On pandas 2 the column is object and None survives, which is why this
    # has never shown on Linux. Nik runs pandas 3.0.5.
    # `dtype=object` is not decoration: assigning a plain list back into a
    # StringDtype column keeps the dtype and turns the Nones straight back
    # into nan, which is the bug this loop exists to remove.
    for _c in ("mechanism_own", "mechanism", "composition",
               "rate_label", "unusable"):
        T[_c] = pd.Series(
            [None if (v is None
                      or (isinstance(v, float) and not np.isfinite(v)))
             else str(v) for v in T[_c]], index=T.index, dtype=object)

    # --- do they agree? ---------------------------------------------------
    verdicts = []
    # Group on the composition ITSELF. Reverse-engineering it from the
    # label by stripping the last word merged unrelated compositions whose
    # names differ only in the final token, and mangled any composition
    # recorded without a cell id.
    # A DATASET THAT IS NOT A MEASUREMENT DOES NOT GET A VOTE. Averaging a
    # two-cycle export at 41% coulombic efficiency into a triplicate mean does
    # not make the mean more robust; it makes it wrong, and hides the failure
    # inside a standard deviation. Excluded cells keep their own page and are
    # named on this one.
    _unusable_cells = [r.label for r in T.itertuples() if getattr(r, "unusable",
                                                                 None)]
    T_ok = T[[not bool(getattr(r, "unusable", None)) for r in T.itertuples()]]
    grouped = T_ok.groupby("composition")
    for comp, g in grouped:
        if len(g) < REPLICATE_MIN_N:
            continue
        s_ret = _spread(g["retention"])
        s_first = _spread(g["first"])
        entry = dict(composition=comp, n=len(g),
                     spread_retention=s_ret, spread_first=s_first)
        # THE HEADLINE. Computed for every group whatever the spread, because
        # a wide spread is a reason to report the SD loudly, not a reason to
        # stop reporting it.
        for _k, _col, _dp in (("retention", "retention", 1),
                              ("first", "first", 1),
                              ("last", "last", 1), ("ce", "ce", 2)):
            entry[f"{_k}_mean"], entry[f"{_k}_sd"], entry[f"{_k}_n"] = \
                _mean_sd(g[_col])
        # PEAK DRIFT, per step, across the replicates. On a dataset where the
        # areas are withheld this is the only quantity the run actually
        # measured, and it had appeared nowhere on this page.
        # ONE ROW PER REDOX FEATURE, not one row per step. Until 1.9.0.25
        # every primary peak from every cell went into a single list, so on
        # LTO — one charge peak per cell — n was the cell count and the
        # number read as a replicate statistic. On a layered oxide with
        # three charge features it silently became n = 9: the mean of three
        # DIFFERENT redox processes across three cells, whose spread is
        # dominated by the differences between the processes rather than
        # between the cells. The NNM run reported 0.80 +/- 2.31 mV/cycle
        # that way, which is not a quantity anyone can use.
        #
        # Peaks are grouped by reference voltage — the pipeline's own notion
        # of "the same feature" is TRACKING_TOLERANCE_MV, so the same number
        # separates features here — and each group reports mean +/- SD with
        # **n = the number of CELLS** contributing. A cell offering two peaks
        # to one group contributes their median, so no cell is counted twice.
        #
        # AND THE COHERENCE AUDIT DECIDES WHICH PEAKS GET IN. A drift rate is
        # only "the quantity that survives where the areas do not" if the peak
        # it belongs to moves like a redox feature. `coherence_audit` already
        # judges that, peak by peak; before 1.9.0.34 its verdict never reached
        # this page, so every primary peak was published here whatever the
        # audit had said about it. On the NMC111 pair that produced a table of
        # five drift rates on a run where the audit found ZERO coherent peaks
        # in either cell — including one rate read from four consecutive
        # pairs and marked "not assessable" two cells earlier.
        _n_excluded = 0
        for _step in ("Charge", "Discharge"):
            _pts = []
            for _n in g["name"]:
                _verd = {(c["step"], round(c["reference_voltage"], 3)):
                         c["verdict"]
                         for c in (facts_by_name[_n].get("coherence") or [])}
                for _t in (facts_by_name[_n].get("tracked") or []):
                    if not (_t.get("step") == _step
                            and not _t.get("is_shoulder")
                            and np.isfinite(_t.get(
                                "voltage_drift_mV_per_cycle", np.nan))
                            and np.isfinite(_t.get("reference_voltage",
                                                   np.nan))):
                        continue
                    _rv = float(_t["reference_voltage"])
                    # No audit at all (peak fitting off, or an older facts
                    # dict) leaves the old behaviour: publish it. A verdict
                    # that exists and is not trend-worthy excludes the point.
                    _v = _verd.get((_step, round(_rv, 3)))
                    if _v is not None and _v not in ("coherent",
                                                     "questionable"):
                        _n_excluded += 1
                        continue
                    _pts.append((_rv, _n,
                                 float(_t["voltage_drift_mV_per_cycle"]),
                                 _v))
            entry[f"drift_{_step.lower()}_features"] = _group_drift(_pts)
        entry["drift_excluded_incoherent"] = _n_excluded
        # The odd one out, if there is one: the cell furthest from the median
        # on retention, but only worth naming when the spread is wide enough
        # that "furthest" means something.
        # "Furthest from the median" needs at least three cells that HAVE a
        # retention figure. With two, the median sits between them and both
        # are equally far — on a run where one cell died and one had no
        # figure at all, that named the healthy cell as the outlier.
        # DO THE CELLS' NUMBERS COME FROM THE SAME CYCLE? "Last working" and
        # the retention beside it are read at each cell's own last good
        # cycle, so a group containing a cell that died early is being
        # averaged across DIFFERENT endpoints. On the NNM triplicate that
        # meant 46% at cycle 100, 58% at cycle 67 and 44% at cycle 100
        # pooled into "49.5 +/- 7.8 (n = 3)", which reads as retention after
        # 100 cycles for three cells and is not that — and the spread is
        # inflated by exactly the mismatch. The per-cell table above already
        # prints "58 (c67)"; the headline threw the qualifier away.
        _lc = pd.to_numeric(g["last_cycle"], errors="coerce").dropna()
        entry["last_cycle_min"] = float(_lc.min()) if len(_lc) else np.nan
        entry["last_cycle_max"] = float(_lc.max()) if len(_lc) else np.nan
        entry["last_cycle_mixed"] = bool(len(_lc) > 1
                                         and _lc.nunique() > 1)
        gf = g[g["retention"].notna()]
        entry["n_compared"] = len(gf)
        if np.isfinite(s_ret) and len(gf) >= 3:
            dist = (gf["retention"] - gf["retention"].median()).abs()
            if s_ret > REPRESENTATIVE_MAX_SPREAD:
                entry["outlier"] = gf.loc[dist.idxmax(), "label"]
                entry["representative"] = None
            else:
                entry["outlier"] = None
                entry["representative"] = gf.loc[dist.idxmin(), "label"]
        verdicts.append(entry)

    # --- markdown ---------------------------------------------------------
    m = ["# Run summary", ""]
    m += [f"*{run_id}* — {len(names)} dataset(s)"
          + (f" of {n_files} in the folder" if n_files else ""), ""]

    # THE MECHANISM ROW, above the table rather than in it: it is one word
    # per cell and it decides how everything below was measured, so it reads
    # as a premise rather than as another column.
    # KEYED ON THE DATASET, NOT ITS LABEL. `label` is composition + cell id
    # and the cell id is optional, so three replicates with no cell id shared
    # one label, `len(_mechs)` was 1, and the page told a three-row table
    # "There is one dataset in this run, so there was nothing to reconcile it
    # against."
    _label_of = {r.name: r.label for r in T.itertuples()}
    _mechs = {r.name: r.mechanism for r in T.itertuples() if r.mechanism}
    # What each cell's OWN reference cycle said, before reconciliation.
    _own = ({r.name: getattr(r, "mechanism_own", None)
             for r in T.itertuples() if r.mechanism}
            if "mechanism_own" in T else {})

    # WHICH CELLS COULD BE RECONCILED AT ALL, and if not, why not.
    # `quality.reconcile_mechanisms` groups by composition with the dataset's
    # own name as the fallback, and skips any group of fewer than two. So a
    # cell is left with its own reference cycle's call whenever its
    # composition is blank (the fallback name is unique) OR it is the only
    # cell of its composition in the run — and until 1.9.0.62 the run page
    # said neither.
    #
    # Measured, NNM 1.9.0.61: cell A's own call is `multi_transition`, whose
    # model permits no band at all. Reconciled with cells B and C it is
    # fitted `mixed` and gets 215 bands; in a group of one it gets NONE and
    # R2 falls 0.9930 -> 0.9314 over 160 half-cycles. That happened twice on
    # the same day — once from a blank composition box, once from re-running
    # the cell on its own to check the first — and both runs read as normal.
    _group_of = {}
    for r in T.itertuples():
        if not r.mechanism:
            continue
        _c = str(getattr(r, "composition", "") or "").strip()
        _group_of[r.name] = (_c or f"\x00{r.name}", bool(_c))
    _sizes = {}
    for _g, _ in _group_of.values():
        _sizes[_g] = _sizes.get(_g, 0) + 1
    _alone = {lab: named for lab, (g, named) in _group_of.items()
              if _sizes.get(g, 0) < 2}

    def _display(label, name):
        """A cell with no composition has a label of just its cell id."""
        return str(label).strip() or str(name)

    _CONSEQUENCE = (
        "The same cell analysed alongside its replicates can be reconciled "
        "to a more permissive model and fitted differently, so a result from "
        "a group of one is not interchangeable with one from a group run.")

    def _alone_sentence(*, why=True):
        """Why a cell kept its own call, named cell by cell."""
        if not _alone:
            return []
        _lead = ("A mechanism is chosen from ONE reference cycle of ONE cell, "
                 "and it is reconciled across a material's replicates because "
                 "a material either delivers charge across a composition "
                 "window or it does not. ")
        if not why:
            return [_lead + "There was no group to do that with here, so "
                            "this cell kept whatever its own reference cycle "
                            "said and was fitted with that model. "
                    + _CONSEQUENCE, ""]
        _bits = []
        for _nm0, named in _alone.items():
            _row = next((r for r in T.itertuples() if r.name == _nm0), None)
            _shown = _display(_label_of.get(_nm0, _nm0), _nm0)
            if not named:
                _bits.append(f"**{_shown}** has no composition recorded, so "
                             f"it could not be grouped with anything")
            else:
                _comp = str(getattr(_row, "composition", "") or "its material")
                _bits.append(f"**{_shown}** is the only {_comp} cell in this "
                             f"run")
        _tail = (" — so each kept whatever its own reference cycle said, and "
                 "was fitted with that model. "
                 if len(_bits) > 1 else
                 " — so it kept whatever its own reference cycle said, and "
                 "was fitted with that model. ")
        return [_lead + "; ".join(_bits) + _tail + _CONSEQUENCE, ""]

    if _mechs:
        _uniq = sorted(set(_mechs.values()))
        m += ["## How these cells were modelled", ""]
        if len(_mechs) == 1:
            # ONE CELL IS NOT AN AGREEMENT. "All 1 cells classified X, so the
            # same model was fitted to each and the numbers below are
            # comparable" reads as reassurance about a comparison that does
            # not exist, and hides that this cell was never reconciled.
            _nm, _mv = next(iter(_mechs.items()))
            m += [f"**{_display(_label_of.get(_nm, _nm), _nm)}** was classified "
                  f"**{_mv.replace('_', ' ')}** and fitted with that model. "
                  f"There is one dataset in this run, so there was nothing "
                  f"to reconcile it against.", ""]
            m += _alone_sentence(why=False)
        elif len(_uniq) == 1:
            m += [f"All {len(_mechs)} cells classified "
                  f"**{_uniq[0].replace('_', ' ')}**, so the same model was "
                  f"fitted to each and the numbers below are comparable.", ""]
            # ...but say if that agreement was ARRIVED AT rather than found.
            _rec = {k: v for k, v in (_own or {}).items()
                    if v and v != _mechs.get(k)}
            if _rec:
                m += ["Not every cell's own reference cycle said so. "
                      + "; ".join(
                          f"**{_display(_label_of.get(k, k), k)}** read "
                          f"{v.replace('_', ' ')}" for k, v in _rec.items())
                      + (" and were reconciled to " if len(_rec) > 1
                         else " and was reconciled to ")
                      + f"**{_uniq[0].replace('_', ' ')}**, the most "
                        f"permissive call in the group. A material either "
                        f"delivers charge across a composition window or it "
                        f"does not, so a replicate that did not resolve one "
                        f"is a detection miss rather than evidence of "
                        f"absence — and replicates fitted with different "
                        f"models cannot be pooled, which is what replicates "
                        f"are for.", ""]
            # ...and say which cells had no group to be reconciled with,
            # AFTER the reconciliation note, because it is the exception
            # to it.
            m += _alone_sentence()
        else:
            m += ["**The cells were not all fitted with the same model.**", "",
                  "| Cell | Mechanism |", "|---|---|"]
            m += [f"| {_display(_label_of.get(k, k), k)} | "
                  f"{v.replace('_', ' ')} |" for k, v in _mechs.items()]
            m += ["", "A mechanism is chosen per dataset from its own "
                      "reference cycle, so replicates can differ — but two "
                      "cells fitted with different models are not being "
                      "compared like with like, and areas in particular "
                      "should not be pooled across them.", ""]
            m += _alone_sentence()
    if _unusable_cells:
        m += ["## Not every dataset here is a measurement", ""]
        for r in T.itertuples():
            if getattr(r, "unusable", None):
                m += [f"- **{r.label}** — {r.unusable}"]
        m += ["", "These are excluded from the replicate mean and standard "
                  "deviation below, and from the comparison across cells. "
                  "They keep their own page, which is where to look for what "
                  "went wrong.", ""]
    _var = [r.label for r in T.itertuples() if getattr(r, "rate_is_variable",
                                                      False)]
    if _var:
        m += ["## These cells were cycled at more than one rate", "",
              "**Retention is withheld for "
              + ("every cell" if len(_var) == len(T) else ", ".join(_var))
              + ".** Capacity against cycle number in a rate-capability run "
                "is mostly the rate schedule: it falls because the current "
                "rose and returns when the current falls again, so a "
                "retention percentage across it reads as fading and "
                "recovery. Each cell's rate-capability summary compares "
                "every rate against the reference rate and reports the "
                "recovery to the starting rate, the fade measured during "
                "the ramp, and what that leaves for the ramp itself.", ""]
    m += ["## The cells side by side", "",
          "| Cell | Mass / mg | Cycles | 1st discharge | Last working | "
          "Retention | Median CE | Flags |",
          "|---|---|---|---|---|---|---|---|"]
    for r in T.itertuples():
        flags = []
        # `died` goes through the same DataFrame as everything else, so a
        # None arrives as NaN, which is truthy.
        if np.isfinite(r.died):
            flags.append(f"died c{int(r.died)}")
        if r.n_over:
            flags.append(f"{r.n_over} over Qtheo")
        if r.n_anom:
            flags.append(f"{r.n_anom} unaccounted")
        if r.withheld:
            flags.append(f"{r.withheld} withheld")
        if r.partial:
            flags.append("still cycling")
        fmt = lambda v, d=0: ("—" if not np.isfinite(v) else f"{v:.{d}f}")
        m.append(f"| {r.label} | {fmt(r.mass,2)} | "
                 f"{'—' if not np.isfinite(r.cycles) else int(r.cycles)} | "
                 f"{fmt(r.first)} | "
                 f"{_last_cell(r.last, r.last_cycle)} | "
                 f"{fmt(r.retention) + '%' if np.isfinite(r.retention) else '—'} | "
                 f"{fmt(r.ce,1) + '%' if np.isfinite(r.ce) else '—'} | "
                 f"{', '.join(flags) if flags else 'none'} |")
    m += [""]

    # The replicate result as a table anyone can read back — the headline
    # number for the group, machine-readable, beside the per-cell CSV.
    R = pd.DataFrame(verdicts) if verdicts else pd.DataFrame(
        columns=["composition", "n", "retention_mean", "retention_sd",
                 "retention_n"])

    if verdicts:
        # --- THE HEADLINE: mean +/- SD across replicates ------------------
        m += ["## The replicate result", "",
              "The number to quote for a group of cells is the mean and "
              "standard deviation across them, with n stated. A single cell "
              "nominated as representative is a selection step, and the "
              "figures below say which cell is *shown*, never which number "
              "is *reported*.", "",
              "| Material | n | 1st discharge / mAh g⁻¹ | Last working / "
              "mAh g⁻¹ | Retention / % | Median CE / % |",
              "|---|---|---|---|---|---|"]
        for v in verdicts:
            m.append(
                f"| {v['composition']} | {v['retention_n']} | "
                + _pm(v["first_mean"], v["first_sd"], v["first_n"]) + " | "
                + _pm(v["last_mean"], v["last_sd"], v["last_n"]) + " | "
                + _pm(v["retention_mean"], v["retention_sd"],
                      v["retention_n"]) + " | "
                + _pm(v["ce_mean"], v["ce_sd"], v["ce_n"], dp=2) + " |")
        m += ["", "*Standard deviation across cells, ddof = 1. A single cell "
              "shows its value with no ± : one measurement has no spread, and "
              "quoting 0 would claim a precision nothing established.*", ""]
        # Said plainly, next to the number it qualifies.
        for v in verdicts:
            if v.get("last_cycle_mixed"):
                m += [f"> **These cells did not all reach the same cycle.** "
                      f"For {v['composition']} the last working capacity and "
                      f"the retention beside it are read between cycle "
                      f"{v['last_cycle_min']:.0f} and cycle "
                      f"{v['last_cycle_max']:.0f} depending on the cell, so "
                      f"the mean above is **not** retention at a common "
                      f"cycle and its spread carries that mismatch as well "
                      f"as any real difference between the cells. The "
                      f"per-cell table shows each cell's own cycle. Quote "
                      f"these cells separately, or re-run capped at cycle "
                      f"{v['last_cycle_min']:.0f} so every cell is compared "
                      f"over the same life.", ""]

        # Peak drift, where there is any. This is the quantity that survives
        # a low integral fidelity, so on a two-phase material it is the
        # replicate result and belongs beside the capacities, not buried in a
        # per-cell page.
        _has_drift = [v for v in verdicts
                      if v.get("drift_charge_features")
                      or v.get("drift_discharge_features")]
        _no_drift = [v for v in verdicts
                     if v not in _has_drift
                     and v.get("drift_excluded_incoherent")]
        for v in _no_drift:
            m += ["### How fast the redox features move", "",
                  f"**Not reported for {v['composition']}.** Every primary "
                  f"peak that could have gone in this table "
                  f"({int(v['drift_excluded_incoherent'])} across "
                  f"{int(v['n'])} cell(s)) was excluded by the coherence "
                  f"audit: none of them moved like a redox feature. A drift "
                  f"rate read from a peak the fit could not hold on to is a "
                  f"property of the fit. The per-cell coherence tables say "
                  f"which peaks and why.", ""]
        if _has_drift:
            m += ["### How fast the redox features move", "",
                  "| Material | Step | Feature / V | Drift / mV cycle⁻¹ |",
                  "|---|---|---|---|"]
            for v in _has_drift:
                for _step in ("Charge", "Discharge"):
                    for _f in v.get(f"drift_{_step.lower()}_features", []):
                        if not np.isfinite(_f["mean"]):
                            continue
                        _mark = (" ‡" if _f.get("unaudited") else
                                 " †" if _f.get("questionable_only") else "")
                        m.append(
                            f"| {v['composition']} | {_step} | "
                            f"{_f['voltage']:.3f} | "
                            + _pm(_f["mean"], _f["sd"], _f["n"], dp=2,
                                  show_n=True)
                            + _mark + " |")
            m += ["", "*One row per redox feature, primary peaks only, "
                  "measured between medians of three cycles at each end from "
                  "the reference cycle onward. **n is the number of CELLS** "
                  "in which that feature was tracked — averaging different "
                  "features together would give a spread that measures the "
                  "difference between the processes rather than between the "
                  "cells. Peak positions do not depend on the dQ/dV integral, "
                  "so this is the quantity that survives where the areas do "
                  "not — but only for a peak the coherence audit found "
                  "trend-worthy; peaks it called NOT trend-worthy or not "
                  "assessable are left out.*", ""]
            if any(_f.get("questionable_only")
                   for v in _has_drift
                   for _step in ("charge", "discharge")
                   for _f in v.get(f"drift_{_step}_features", [])):
                m += ["*† every peak behind this row was graded "
                      "*questionable* by the coherence audit, not *coherent*: "
                      "its median drift cleared the implausible threshold but "
                      "not the clean one. The rate is reported; the evidence "
                      "for it is weaker than for an unmarked row.*", ""]
            if any(_f.get("unaudited")
                   for v in _has_drift
                   for _step in ("charge", "discharge")
                   for _f in v.get(f"drift_{_step}_features", [])):
                m += ["*‡ no peak behind this row has a coherence verdict at "
                      "all — the audit did not reach it, or peak fitting ran "
                      "without it. The rate is what the tracking measured; "
                      "nothing has tested whether the feature moves like a "
                      "redox process or like a fitting artefact.*", ""]
            _exc = sum(int(v.get("drift_excluded_incoherent") or 0)
                       for v in _has_drift)
            if _exc:
                m += [f"*{_exc} further primary peak(s) were excluded by the "
                      f"coherence audit and are not in the table above.*", ""]

        m += ["## Do the cells agree?", ""]
        for v in verdicts:
            s = v["spread_retention"]
            # THE VERDICT INHERITS THE CAVEAT. `spread_retention` is computed
            # from the same per-cell retention figures the note above warns
            # are read at different cycles, and this is the most
            # conclusion-shaped sentence in the report — so it is the one
            # place the qualifier most needs to reach.
            #
            # Measured on NNM cells B and C, same cells, same data, only the
            # cycle cap changed:
            #
            #   capped at 100   B 58% (c67), C 44% (c100)  spread 29%
            #                   -> "the 2 cells do NOT agree"
            #   capped at  80   B 58% (c67), C 53% (c80)   spread 9.0%
            #                   -> "2 cells agree"
            #
            # Opposite conclusions from where the operator stopped, and in
            # BOTH the two numbers come from different cycles. Neither
            # verdict is wrong about the arithmetic; both are silent about
            # what they compared.
            _mixed = bool(v.get("last_cycle_mixed"))
            _mixed_note = ""
            if _mixed:
                _mixed_note = (
                    f" **Read this with the endpoint warning above:** these "
                    f"retentions come from different cycles — between "
                    f"{v['last_cycle_min']:.0f} and "
                    f"{v['last_cycle_max']:.0f} — so this verdict is partly "
                    f"about where each cell stopped, not only about how they "
                    f"differ. Cap the run at cycle "
                    f"{v['last_cycle_min']:.0f} to settle it.")
            if not np.isfinite(s):
                m += [f"**{v['composition']}** — {v['n']} cell(s), of which "
                      f"{v.get('n_compared', 0)} have a retention figure; "
                      f"nothing to compare.", ""]
                continue
            if s <= REPRESENTATIVE_MAX_SPREAD:
                _sf = v["spread_first"]
                line = (f"**{v['composition']}** — {v['n']} cells agree. "
                        f"Retention spreads {_pct(s)} of the median"
                        + (f", and first discharge {_pct(_sf)}."
                           if np.isfinite(_sf) else "."))
                if v.get("representative"):
                    line += (f" For a FIGURE, show **{v['representative']}** "
                             f"— it sits closest to the group — and caption "
                             f"it as such: \"{v['representative']} shown; "
                             f"n = {v['retention_n']}, retention "
                             + _pm(v["retention_mean"], v["retention_sd"],
                                   v["retention_n"]).replace(
                                       f" (n = {v['retention_n']})", "")
                             + "%\". The reported number stays the mean "
                             "above; the representative cell is a choice of "
                             "illustration, not of result.")
                m += [line + _mixed_note, ""]
            else:
                line = (f"**{v['composition']}** — the {v['n']} cells do NOT "
                        f"agree. Retention spreads {_pct(s)} of the median")
                if v.get("outlier"):
                    line += f", with **{v['outlier']}** furthest out"
                elif v.get("n_compared", 0) < 3:
                    line += (f" across the {v['n_compared']} cell(s) that have "
                             f"one; too few to say which is the odd one")
                line += (". No representative cell is nominated: picking the "
                         "one nearest the median would be choosing a number, "
                         "not measuring one. Find out why they differ first.")
                m += [line + _mixed_note, ""]

    # --- what the run withheld or flagged ---------------------------------
    tot_over = int(T["n_over"].sum())
    tot_anom = int(T["n_anom"].sum())
    tot_with = int(T["withheld"].sum())
    # A cell whose curve carries the wrong amount of charge has something to
    # declare even when nothing was withheld. `capacity_attribution` withheld
    # only below the floor until 1.9.0.36, so a run with fidelity 1.45 had
    # `tot_with == 0` and this page printed "Nothing was withheld. Every
    # half-cycle's charge is accounted for" directly opposite a per-cell page
    # saying the areas are not capacities.
    _n_outband = int(((T["fidelity"] < INTEGRAL_FIDELITY_FLOOR)
                      | (T["fidelity"] > INTEGRAL_FIDELITY_CEILING)).sum())
    _un_col = T["unattributed"] if "unattributed" in T else None
    _un_vals = ([x for x in _un_col.tolist() if x is not None
                 and np.isfinite(x)] if _un_col is not None else [])
    _un_med = float(np.median(_un_vals)) if _un_vals else None
    _un_any = bool(int(T["unattributed_withheld"].sum())
                   if "unattributed_withheld" in T else 0)
    _no_baseline = bool(T["no_baseline"].all()
                        if "no_baseline" in T and len(T) else False)
    m += ["## What the run would not tell you", ""]
    if not (tot_over or tot_anom or tot_with or _n_outband):
        m += ["Nothing was withheld. Every half-cycle's charge is accounted "
              "for" + (", and every capacity share is a share of named "
                       "components rather than of a free background."
                       if _no_baseline else
                       ", and every capacity share survived a change of "
                       "baseline degree."), ""]
    else:
        if tot_over:
            m += [f"- **{tot_over} half-cycle(s) passed more charge than the "
                  f"material can hold.** Over theoretical is not a phase "
                  f"transition; it is charge going into something else.", ""]
        if tot_anom:
            m += [f"- **{tot_anom} half-cycle(s) carry charge no redox process "
                  f"accounts for** — at full current, away from the voltage "
                  f"limit, where |dQ/dV| is small.", ""]
        if tot_with:
            # WHICH GROUND. Closure (the peak/background split is undetermined)
            # and integral fidelity (the curve does not account for the cell's
            # charge) are different failures with different remedies, and this
            # page said "baseline degree" for both — sending a reader to look
            # at the fit when the limit is the instrument's voltage
            # resolution.
            # Two-sided, like `_fidelity_outside_band` above and like every
            # other reading of this number. Testing only the floor meant an
            # over-ceiling cell was never named here.
            _lowfid = T[(T["fidelity"] < INTEGRAL_FIDELITY_FLOOR)
                        | (T["fidelity"] > INTEGRAL_FIDELITY_CEILING)]
            if len(_lowfid):
                _lo = 100 * float(_lowfid["fidelity"].min())
                _hi = 100 * float(_lowfid["fidelity"].max())
                _rng = (f"{_lo:.0f}%" if abs(_hi - _lo) < 0.5
                        else f"{_lo:.0f}-{_hi:.0f}%")
                m += [f"- **{tot_with} half-cycle(s) had their capacity "
                      f"attribution withheld — and on "
                      f"{len(_lowfid)} of {len(T)} cell(s) the reason is not "
                      f"the fit.** The dQ/dV curve "
                      + ("accounts for only " if _hi <= 100.0 else
                         "carries " if _lo >= 100.0 else "accounts for ")
                      + f"{_rng} "
                      f"of the capacity those cells delivered — per cell and "
                      f"per step; where a cell's two steps differ, its own "
                      f"page gives both. On a flat "
                      f"two-phase plateau the voltage change between records "
                      f"falls below the instrument's resolution, so charge "
                      f"delivered there cannot appear in a dV integral and no "
                      f"model recovers it. **A fitted peak area is not a "
                      f"capacity on these cells**, and neither is its "
                      f"retention. Peak POSITIONS and their drift do not "
                      f"depend on the integral and are unaffected — they are "
                      f"the result to quote here. The fitted areas are still "
                      f"in the CSVs.", ""]
            elif _un_any:
                # THE THIRD GROUND. From 1.9.0.56 there is no free background,
                # so the peak/background split cannot be undetermined and this
                # page must not say it was. What can still fail is the model
                # not adding up to the cell.
                m += [f"- **{tot_with} cycle(s) had their capacity "
                      f"attribution withheld.** The named components do not "
                      f"add up to the charge the cell delivered"
                      # THE MEDIAN IS THE WRONG NUMBER TO PUT HERE. Quoting
                      # the run's typical unattributed fraction beside a
                      # withholding reads as though 3% had caused it, when the
                      # cycles that were withheld are the ones above the
                      # limit. Say what the limit is and let the CSV carry the
                      # per-cycle values.
                      + (f" on those cycles — more than the "
                         f"{UNATTRIBUTED_WITHHOLD_ABOVE:.0%} of the cell's "
                         f"charge above which a share stops being a "
                         f"measurement (the run's median is {_un_med:+.0%})"
                         if _un_med is not None else "")
                      + ". Every mAh in a dQ/dV passed through the cell, so a "
                        "share OF a total that is missing part of the "
                        "capacity is not a measurement. The fitted areas are "
                        "still in the CSVs.", ""]
            elif _no_baseline:
                # No free background: the split CANNOT be undetermined, so the
                # only remaining reason a share was withheld is that the
                # check fit did not converge on those half-cycles. Saying
                # "the baseline degree changed" here would send a reader to
                # look for a background this model does not have.
                m += [f"- **{tot_with} cycle(s) had their capacity "
                      f"attribution withheld.** This model has no free "
                      f"background, so the peak/background split cannot be "
                      f"undetermined; these cycles were withheld because the "
                      f"check fit did not converge on them and there is "
                      f"therefore no measured share to report. The fitted "
                      f"areas are still in the CSVs.", ""]
            else:
                m += [f"- **{tot_with} cycle(s) had their capacity "
                      f"attribution withheld.** The peak/background split "
                      f"moved more than the number would have reported when "
                      f"the baseline degree changed. The fitted areas are "
                      f"still in the CSVs.", ""]

    m += ["## Where to look", "",
          "Each cell has a `START_HERE.html` and `START_HERE.md` in its own "
          "folder, with four figures and what to look for in each. Start "
          "there; come back here to compare.", ""]
    return "\n".join(m), T, R


def write_run_summary(run_dir, facts_by_name, *, run_id="", n_files=0):
    """Write `RUN_SUMMARY.md` at the top of the run folder. Returns its path."""
    md, T, R = build_run_summary(facts_by_name, run_id=run_id,
                                 n_files=n_files)
    path = os.path.join(run_dir, "RUN_SUMMARY.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(md)
    T.to_csv(os.path.join(run_dir, "RUN_SUMMARY.csv"), index=False)
    if not R.empty:
        # The REPORTED number for each material — mean, SD and n across its
        # replicates. Separate from RUN_SUMMARY.csv, which is per cell, so
        # that nobody has to re-derive a group statistic from the cells and
        # get ddof wrong.
        R.to_csv(os.path.join(run_dir, "REPLICATE_SUMMARY.csv"), index=False)
    return path


## Loading, and what the cycler already knows

Ratatosk reads Neware `.xlsx` exports. Two sheets matter: `record`, which holds
the per-point voltage, current, capacity and dQ/dV, and `test`, which holds the
metadata the operator entered when the schedule was built — active mass,
voltage limits, C-rate, currents, planned cycle count.

**The `test` sheet is read first, and it matters.** Every number in it becomes
the default for a prompt in the next cell, marked `[from file]`. A mass typed
twice is a mass that can disagree with itself; a mass read from the schedule
cannot.

`read_neware` uses `calamine` when it is installed, which is 5–7× faster than
`openpyxl` on these files with a maximum numeric difference of exactly zero.

### On the electrode convention

For a **positive** electrode the cycler's step labels already match the
electrochemistry: charge oxidises the working electrode. For a **negative**
electrode they do not — the full cell discharges while the working electrode is
lithiated or sodiated.

So for a negative electrode Ratatosk swaps the step labels, the capacity and
energy columns, and the sign of dQ/dV, so that **"Discharge" is the useful
half-cycle for both polarities**. Every figure, table and ΔV downstream is then
directly comparable between a cathode and an anode.

This is applied in Cell 3b rather than at load, because which electrode is
under test is one of the things Cell 3b asks. Applying it twice is a no-op.


In [ ]:
# CELL 2 — WHAT TO ANALYSE            <-- this is the cell you edit
# =============================================================================
# ---------------------------- WHERE THE DATA IS, AND WHERE THE OUTPUT GOES
# The folder holding the Neware .xlsx exports.
#
# LEAVE IT BLANK AND IT IS SEARCHED FOR. Until 1.9.0.74 this line named one
# person's Google Drive letter, which meant the notebook ran on exactly one
# machine and raised NotADirectoryError on every other — including its own
# author's second computer. `find_data_dir` looks in the obvious places (cloud
# drives on every mounted letter, the home folder, the working directory) and
# prints which one it chose and why.
#
# Three ways to be explicit, in the order they win:
#
#   1. set the RATATOSK_DATA_DIR environment variable before starting
#      Jupyter — the one to use if you move between machines, run this on a
#      schedule, or hand it to somebody else;
#   2. put the path on the line below;
#   3. leave it blank and let the search find it.
#
# A folder that exists but holds no file matching PATTERN is not treated as
# the data folder, because otherwise the search would stop at Documents on
# almost every machine.
#
# BLANK, not `os.environ.get(...)`. The environment variable is read inside
# `find_data_dir`, which is the ONE place that decides where the data is —
# reading it here as well would be two places computing one quantity, which
# is the bug this build has now found four times.
DATA_DIR    = ""

# Where run folders are created. One per run, named <timestamp>_v<version>.
# Written relative to whoever is logged in, so the notebook does not carry one
# person's home directory into somebody else's run; replace it with an
# absolute path if you keep results somewhere specific.
OUTPUT_BASE = os.path.join(os.path.expanduser("~"), "Documents",
                           "electrochemical_analysis_output")

# Which files. Leave FILES empty and Cell 3a opens a file dialog on DATA_DIR
# — 1.8.7's Tkinter picker, so you can browse, sort by date and multi-select a
# triplicate. Where no dialog can be shown (a headless kernel, no display) it
# falls back to listing DATA_DIR grouped by chemistry and asking: answer with
# numbers, ranges like `1-3`, or blank for all. Set FILES explicitly to skip
# the question entirely, or FILE_DIALOG = False to always use the list.
FILES   = []
PATTERN = "*.xlsx"
FILE_DIALOG = True

# ------------------------------------------ WHERE THE PARAMETERS COME FROM
# How parameters are collected.
#   "ask"   Cell 3b prompts for each dataset, pre-filled from the cycler's own
#           'test' sheet. This is the 1.8.6 behaviour and the one to use: the
#           prompt shows "[from file: 4.72]" against a mass read out of the
#           workbook and "[default: 12.0]" against a convention, so you can see
#           which numbers you are actually responsible for.
#   "dict"  take them from PARAMETERS below, no prompting. For a scheduled run,
#           for a reproduction of an earlier run, or for a front end.
PARAMETER_MODE = "ask"

# Used only when PARAMETER_MODE == "dict". Composition, theoretical capacity
# and electrode type have no defensible default and must be given; everything
# else falls back to the cycler metadata and then to a documented default.
PARAMETERS = {
    # "JQ-P3-NaNiMnO2-35%-2-4.2V-0.1C-A-20062025": dict(
    #     composition="P3-NaNiMnO2", cell_id="A", electrode_type="Positive",
    #     theoretical_capacity_mAh_g=160.0,
    #     key_cycles=[1, 5, 10, 20, 50, 100],
    #     # dQ/dV preprocessing. Omit or leave "auto" to let the profile of
    #     # each dataset's own curve decide; set a value to override it.
    #     smoothing_window="auto", spike_removal="auto",
    #     rebin_width_mV="auto",
    # ),
}

# ---------------------------------- HOW MUCH OF THE CELL'S LIFE TO ANALYSE
# Cap the analysis while you are setting a run up; None = every cycle.
# Applied ONCE, to the dataset itself, in Cell 3b — so every later stage sees
# the same cycles and no stage can disagree with another about how many there
# were.
MAX_CYCLE = None

# Stop FITTING once the cell has stopped delivering charge. The plots still
# show every cycle — a cell going flat at 130 is exactly what the heatmap is
# for — but there is nothing to fit in a flat line, and on the P3 cells this
# is 40-70% of the fitting time.
STOP_FITTING_AT_DEATH = True

# --------------------------------------------------- HOW dQ/dV IS COMPUTED
# The single most consequential setting in the file.
#
#   "histogram"   charge binned by voltage — no derivative, no smoothing
#                 parameters. The curve integrates back to the delivered
#                 capacity by construction, so a peak area IS a capacity.
#   "derivative"  the cycler's own dQ/dV column, sorted, rebinned, despiked
#                 and smoothed. What every Ratatosk result before 1.9.0.14
#                 used.
#
# Measured on the LTO triplicate, same fits, only this changed:
#
#     integral fidelity   0.62 -> 0.995      area retention   141% -> 100%
#
# The derivative loses 40% of the charge on a two-phase plateau; the
# histogram does not. Keep "derivative" only to reproduce an earlier run —
# and if you do, note it, because the two are not comparable.
DQDV_COMPUTATION = "histogram"

# Voltage bin width for the histogram path, in mV — the ONLY parameter it has,
# and it means what it says: the voltage resolution of the curve.
#
# None lets the measured profile choose, as every other preprocessing setting
# already does: 1 mV for a sharp two-phase peak, 5 mV for a broad layered
# oxide whose features are 30-200 mV wide. Put a number to force one. It is
# widened automatically wherever the records are sparser than the bin, and
# the width actually used is recorded in dataset_info.txt.
DQDV_BIN_MV = None

# -------------------------------------------------- HOW MUCH FITTING TO DO
# Fit the dQ/dV peaks at all? The cycling analysis (Cells 5-7) does not need
# it, and on a rate test or a screening run there may be nothing worth fitting.
#
# In PARAMETER_MODE == "ask" this is only the DEFAULT: Cell 3b asks the same
# question at the end of the prompts and the answer there replaces this value.
# In "dict" mode nothing asks, and this is the setting.
RUN_PEAK_FITTING = True

# How many half-cycles the closure interval is measured on. Each one costs
# THREE fits (one per baseline degree), so this is the single biggest lever on
# how long Cell 10 takes: on a 200-cycle cell, None is 600 half-cycles and
# 1,800 fits — three times the cost of Cell 9's fitting.
# Evenly spaced, the default pins the median and the pass fraction well inside
# the width being measured; see quality.CLOSURE_SAMPLE_DEFAULT for the number
# and for why that is enough.
CLOSURE_SAMPLE = CLOSURE_SAMPLE_DEFAULT

# Fit a dataset whose reference cycle says its peaks are not resolvable?
#
# True keeps the old behaviour exactly: everything is fitted, and the
# resolvability verdict is reported for information. False skips the
# decomposition for those datasets and reports positions and the envelope
# instead — which is what a solid-solution layered oxide can actually
# support, and it turns an hour of fitting into a few seconds.
#
# Left ON by default, because the verdict is new and a reader should be able
# to disagree with it and see the fit for themselves.
FIT_UNRESOLVED = True

RUN_NOTE = ""      # free text, recorded in the manifest

# How a long stage reports progress. Nothing is printed until it has been
# running for PROGRESS_QUIET_S, so a fast dataset stays silent instead of
# emitting ten lines of "2 of 21 (0 s)"; after that, a line roughly every
# PROGRESS_EVERY_S with an estimate of what is left.
PROGRESS_QUIET_S = 3.0
PROGRESS_EVERY_S = 10.0

# FIGURE GEOMETRY, in inches, applied to every figure the run produces.
# 6.73 x 5.0 is an RSC double-column. Set here rather than in a figure cell:
# `set_figure_size` writes a MODULE GLOBAL that the dQ/dV figures (Cell 7)
# and the fit figures (Cell 12) inherit, so setting it inside Cell 6 made
# those two cells depend on a cell neither of them needs. It went unnoticed
# because the values were the module defaults already, making the call a
# no-op that looked like configuration.
FIGURE_WIDTH_IN = 6.73
FIGURE_HEIGHT_IN = 5.0
set_figure_size(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN)

# How much the cycling cells say. False quietens the per-dataset commentary
# in Cells 6a-6c and keeps the tables, figures and CSVs.
CYCLING_VERBOSE = True

# Which cycle the preprocessing QC figure (Cell 7) shows raw against
# processed for. Was a literal 2 in the call; a cycle inside formation is a
# poor advertisement for what the preprocessing does to a settled curve.
QC_CYCLE = 3

# ---------------------------------------------------------------- CHECKED
# The five headings above are SOURCE dividers, not prints. Printing them put
# six headings in the output with nothing underneath any of them — a settings
# cell that reported its own table of contents. Only a heading with something
# under it is worth a rule.
#
# Read back, and the run refused now if any of it is impossible. A mistyped
# PARAMETER_MODE used to fall through to the declarative branch and run on a
# NaN theoretical capacity; a mistyped DATA_DIR surfaced as an empty glob a
# cell later. See params.validate_run_settings for what each check is for.
print(rule("SETTINGS AS READ BACK"))
# Blank means "find it" — see the note against DATA_DIR above. Resolved here
# rather than on the assignment so the search prints inside this cell's own
# banner, with everything else that was read back.
if not DATA_DIR.strip():
    DATA_DIR = find_data_dir(pattern=PATTERN, verbose=True)
RUN_SETTINGS = validate_run_settings(
    data_dir=DATA_DIR, output_base=OUTPUT_BASE, files=FILES, pattern=PATTERN,
    parameter_mode=PARAMETER_MODE, parameters=PARAMETERS,
    max_cycle=MAX_CYCLE, dqdv_computation=DQDV_COMPUTATION,
    dqdv_bin_mV=DQDV_BIN_MV, closure_sample=CLOSURE_SAMPLE,
    run_peak_fitting=RUN_PEAK_FITTING)

# Written back NORMALISED, so the cells below compare against what was
# checked rather than against what was typed. `PARAMETER_MODE = "Ask"`
# satisfied the check and then failed a case-sensitive `== "ask"` two cells
# later, which is the same silent fall-through in a new place.
PARAMETER_MODE   = RUN_SETTINGS["parameter_mode"]
DQDV_COMPUTATION = RUN_SETTINGS["dqdv_computation"]
print(rule())


In [ ]:
# CELL 3a — CHOOSE THE FILES, AND READ WHAT THE CYCLER ALREADY KNOWS
# =============================================================================
# A folder holds more than one experiment. Eight files here are three
# chemistries, and loading all of them puts an LTO anode and an NMC cathode on
# the same axes. So the files are listed, grouped by chemistry, and you pick.
#
# Then the 'test' sheet of each chosen export is read: active mass, voltage
# limits, C-rate, currents, planned cycle count. Every one of those becomes the
# default for a prompt in the next cell, marked [from file].
#
# Each workbook is opened ONCE and both sheets read through the one handle.
# Opening it twice parsed it twice — and the second read, being by path, fell
# back to openpyxl while the first was using calamine. Measured on a 34 MB
# export: 3.02 s against 1.33 s, 36% of the whole load.

print(rule("CHOOSING FILES"))

def _natural(text):
    """Sort A2 before A10. `sorted()` alone puts A10 first, which makes a
    listing of a triplicate read as though cells were missing."""
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(text))]

_all = ([os.path.join(DATA_DIR, f) for f in FILES] if FILES
        else sorted(glob.glob(os.path.join(DATA_DIR, PATTERN)), key=_natural))
if not _all:
    # Raised whether or not a dialog was on offer. Handing an empty list to
    # the picker showed an empty dialog and then failed with a confusing
    # error about the selection, rather than saying the folder is empty.
    raise FileNotFoundError(
        f"No files matched {PATTERN!r} in:\n    {DATA_DIR}\n"
        f"  the folder holds "
        f"{len(os.listdir(DATA_DIR))} entries; check PATTERN in Cell 2.")

# FILES set explicitly means you have already chosen; otherwise, choose now.
_paths = _all if FILES else select_files(_all, initialdir=DATA_DIR,
                                         use_dialog=FILE_DIALOG)
if not _paths:
    raise FileNotFoundError("No files were selected — nothing to analyse.")

print(rule("READING"))
datasets, cycler_metadata = {}, {}
_t0 = time.time()
for _i, path in enumerate(_paths, 1):
    name = os.path.splitext(os.path.basename(path))[0]
    # Two exports with the same stem in different folders would silently
    # overwrite each other, and the run would report one file where two were
    # loaded. Named rather than dropped, so the second is still analysed.
    if name in datasets:
        _stem, _k = name, 2
        while name in datasets:
            name, _k = f"{_stem} ({_k})", _k + 1
        print(f"  two files share the stem {_stem!r}; this one is {name!r}")
    print(section(f"  [{_i}/{len(_paths)}] {name}"))
    _book, _engine = open_workbook(path)
    try:
        cycler_metadata[name] = extract_neware_metadata(_book)
        # Read WITHOUT applying the electrode convention: which electrode this
        # is is one of the things Cell 3b asks, and it cannot be applied before
        # it has been answered. This is the order 1.8.6 used, for the same
        # reason.
        datasets[name] = read_neware(path, name=name,
                                     electrode_type="Positive",
                                     meta=cycler_metadata[name],
                                     book=_book, engine=_engine, verbose=True)
    finally:
        _book.close()

# A record table missing any of these is not a Neware GCPL export, and every
# later stage assumes all of them. Said here, naming the file and the columns
# actually present, rather than as a KeyError inside a figure four cells
# later. The names are the RENAMED ones (COLUMNS), which is what the rest
# of the pipeline reads.
_needed = ("Cycle", "Step", "Voltage", "Charge_Capacity", "Discharge_Capacity")
for name, ds in datasets.items():
    _absent = [c for c in _needed if c not in ds.frame.columns]
    if _absent:
        raise KeyError(
            f"{name}: the 'record' sheet has no "
            f"{', '.join(_absent)} column{'s' if len(_absent) > 1 else ''}.\n"
            f"  columns found: {', '.join(map(str, ds.frame.columns))}")
    if ds.frame.empty:
        raise ValueError(f"{name}: the 'record' sheet is empty.")

# `electrochemical_data` was a second dict holding exactly `ds.frame` for every
# dataset, updated by hand in Cell 3b and read in three places. One of those
# updates was missed in 1.9.0.9 and the two disagreed for a whole release. It
# is now a VIEW built from the datasets, so it cannot fall behind them.
electrochemical_data = {n: ds.frame for n, ds in datasets.items()}

print(rule())
print(f"  {len(datasets)} file{'s' if len(datasets) != 1 else ''} read in "
      f"{time.time() - _t0:.1f} s")
print("\nRead from the cycler's own file:")
print(describe_defaults(cycler_metadata))


In [ ]:
# CELL 3b — PARAMETERS, AND THE ELECTRODE CONVENTION
# =============================================================================
# What the file cannot tell us: what the material is, what its theoretical
# capacity is, which electrode is under test, how the electrode was made, and
# how you want the figures to look.
#
# The electrode type is applied here rather than at load. For a NEGATIVE
# electrode the step labels and the sign of dQ/dV are swapped, so that the
# useful half-cycle reads as "Discharge" for both polarities and every figure
# and table downstream is comparable. It is applied ONCE, to a dataset loaded
# as Positive; `apply_electrode_convention` refuses to run over a dataset
# whose labels have already been swapped rather than quietly swapping them
# back.

# The run folder FIRST, so that a run which is then abandoned at the prompts
# leaves a folder saying what was attempted — and so the folder is not created
# only after five minutes of typing.
RUN_DIR, MANIFEST = start_run(OUTPUT_BASE, RATATOSK_VERSION,
                              input_paths=_paths, note=RUN_NOTE)
MANIFEST["environment"] = ENVIRONMENT
MANIFEST["run_settings"] = RUN_SETTINGS
write_manifest(RUN_DIR, MANIFEST)
print(rule("RUN FOLDER"))
print(f"  {RUN_DIR}")

print(rule("PARAMETERS"))
if PARAMETER_MODE == "ask":
    # The prompts ask about peak fitting too, and the answer given there is
    # the one that counts — Cell 2's RUN_PEAK_FITTING is only the default it
    # is offered against. Named on both sides so that is visible.
    user_parameters, RUN_PEAK_FITTING = collect_parameters(
        electrochemical_data, cycler_metadata)
else:
    # Not a fallback: PARAMETER_MODE is checked in Cell 2 and can only be
    # "ask" or "dict" by the time this runs, and dict mode with an empty
    # PARAMETERS is refused there. What used to be here was a synthetic
    # `theoretical_capacity_mAh_g=np.nan` for every dataset, which passed the
    # required-field check and then disabled the over-theoretical guard for
    # the whole run, because `x > 1.2 * nan` is False.
    user_parameters = parameters_from_dict(PARAMETERS, cycler_metadata)

print(rule("APPLYING THEM"))
for name, params in user_parameters.items():
    datasets[name] = apply_electrode_convention(
        datasets[name], params.get("electrode_type", "Positive"))
    # Cut the DATASET, which is what every later stage reads, and cut it once
    # for both caps: MAX_CYCLE from Cell 2 and the per-dataset `max_cycle`
    # answered in the prompts. Before 1.9.0.9 the prompt's answer was applied
    # to a copy that the next line overwrote, so it changed nothing while the
    # console, dataset_info.txt and the manifest all said it had — and
    # MAX_CYCLE reached only the dQ/dV half, so the two halves of one run
    # could cover different cycles.
    # `is not None`, not truthiness: `max_cycle = 0` is a mistake worth
    # reporting, not a cap to be silently ignored.
    _cap = [int(c) for c in (MAX_CYCLE, params.get("max_cycle"))
            if c is not None]
    if _cap:
        _n_before = int(datasets[name].frame["Cycle"].max())
        datasets[name] = truncate_cycles(datasets[name], min(_cap))
        _n_after = int(datasets[name].frame["Cycle"].max())
        params["max_cycle"] = min(_cap)
        if _n_after < _n_before:
            print(f"  {name}: cut at cycle {min(_cap)} "
                  f"(was 1-{_n_before}, now 1-{_n_after})")
    params["anode_labels_swapped"] = datasets[name].labels_swapped
    if datasets[name].labels_swapped:
        print(f"  {name}: negative electrode — labels and dQ/dV sign swapped")

# Rebuilt from the datasets, not patched alongside them. See Cell 3a.
electrochemical_data = {n: ds.frame for n, ds in datasets.items()}

print(rule("COMMITTED"))

def _or_dash(value, unit="", places=1):
    """A number and its unit, or an honest dash. `f"{None:.1f}"` is a
    TypeError and `f"{nan:.1f}"` prints "nan mAh/g", which reads as a
    measurement."""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return "not given"
    return f"{v:.{places}f} {unit}".strip() if np.isfinite(v) else "not given"

for name, params in user_parameters.items():
    print(section(f"  {name}"))
    print(entry("composition", params.get("composition", "—"),
                params.get("cell_id", "") and f"cell {params['cell_id']}"))
    print(entry("electrode", params.get("electrode_type", "Positive"),
                "labels and dQ/dV sign swapped"
                if params.get("anode_labels_swapped") else ""))
    print(entry("theoretical capacity",
                _or_dash(params.get("theoretical_capacity_mAh_g"), "mAh/g")))
    print(entry("active mass",
                _or_dash(params.get("active_material_mass_mg"), "mg", 2)))
    print(entry("cycles",
                f"1-{int(datasets[name].frame['Cycle'].max())}"))

MANIFEST["datasets"] = {
    n: dict(source=ds.source_path, sha256=ds.source_sha256,
            electrode_type=ds.electrode_type,
            labels_swapped=ds.labels_swapped,
            cycler_metadata=cycler_metadata.get(n, {}),
            parameters=user_parameters[n])
    for n, ds in datasets.items()}
MANIFEST["run_settings"]["run_peak_fitting"] = bool(RUN_PEAK_FITTING)
write_manifest(RUN_DIR, MANIFEST)
print(rule())
# The folder itself was printed when it was opened, at the top of this cell.
# Saying the whole path again here made the reader check whether it was a
# different one.
print("  manifest written")


## dQ/dV preprocessing

Differentiating a measurement amplifies its noise, and a raw dQ/dV from a
cycler is spiky. Four things are done about it, and the settings for all four
come from the **shape of each dataset's own curve** rather than from a single
house default.

| step | what it does | why it is not one setting for everything |
|---|---|---|
| **constant-voltage hold removal** | drops the trailing points where the current is tapering at the voltage limit | a CV hold puts a large spike at the limit that is protocol, not chemistry |
| **rebinning** | averages onto a regular voltage grid | a sharp profile logs at 0.3 mV with 22% duplicate voltages; a broad one at 9 mV. Rebinning helps the first and destroys the second |
| **spike removal** | drops points far from the local median | a genuine sharp peak *is* far from its local median. Disabled for sharp profiles |
| **Savitzky–Golay smoothing** | polynomial smoothing over a moving window | a window that flatters a broad feature erases a narrow one |

`classify_dqdv_profile` measures two things — the interquartile range of dQ/dV
as a fraction of its full range, and the fraction of the curve sitting on a
plateau — and returns **sharp**, **moderate** or **broad**. LTO and LFP come
out sharp; layered oxides come out broad. The classification is printed with
the numbers behind it so it can be argued with.

Anything you set in Cell 3b overrides the automatic choice for that dataset.
Leave a field on `"auto"` to let the profile decide.

### The voltage window

The window is computed **once per dataset** — global minimum and maximum,
trimmed by 50 mV at each end — and every half-cycle is clipped to it. Computing
it per half-cycle instead, which is the obvious-looking thing, trims 50 mV off
each half-cycle's own extremes and silently removes real data from all of them.

### One deliberate difference from 1.8.x

Records are sorted by voltage with a **stable** sort. 1.8.x used pandas'
default quicksort, which is not stable, so where several records share a
voltage the order among them was decided by the sort's internals rather than by
the data — and a first charge can carry 483 tied voltages in 1434 records,
because the cycler logs to 0.1 mV and the cell sits on a plateau. Despiking and
smoothing then ran over that arbitrary order, which means 1.8.x's smoothed
curve was not a well-defined function of its input: reorder the rows and it
changed. A stable sort breaks ties by acquisition order.


In [ ]:
# CELL 4 — Q, V, I, t  ->  dQ/dV
# =============================================================================
# The preprocessing is chosen from the SHAPE of each dataset's own dQ/dV, and
# the class it finds is printed — it is the only decision this cell makes, and
# it sets the smoothing, the bin width, and downstream the baseline degree and
# the sigma bounds.
#
# WHICH KNOBS APPLY DEPENDS ON THE PATH. The histogram reads two of them; the
# derivative reads the other seven. Anything set in Cell 3b for the path that
# is NOT running is named and ignored, rather than echoed back as though it
# had been applied — which is what happened for three releases after the
# histogram became the default.
#
#   histogram    histogram_bin_mV            voltage resolution of the curve
#                histogram_smooth_bins       binomial kernel half-width, in bins
#
#   derivative   smoothing_window            Savitzky-Golay window, in points
#                polyorder                   its polynomial order
#                spike_removal               median-deviation despiking on/off
#                spike_window_size           its window
#                spike_threshold_multiplier  how far from the local median counts
#                rebin_width_mV              regular voltage grid, in mV
#                second_smooth_window        an optional second pass
#
# Anything left on "auto" is decided here from the measured profile.
profiles, preprocess_params, windows, signals = {}, {}, {}, {}
# The cut-off approach removed per step, kept so Cell 14 can record it.
trims = {}
reconstruction = {}   # per dataset: APE and voltage RMSE per half-cycle

for name, ds in datasets.items():
    print(rule(name))
    prof = classify_dqdv_profile(ds.frame, verbose=True)
    profiles[name] = prof
    auto = auto_preprocess_params(prof)
    overrides = preprocess_overrides(user_parameters.get(name, {}))
    # An override for the path that is not running changes nothing. Say so
    # here, where it can still be acted on.
    _mine = set(KEYS_BY_DQDV_METHOD[DQDV_COMPUTATION])
    _theirs = {k: v for k, v in overrides.items() if k not in _mine}
    _used = {k: v for k, v in overrides.items() if k in _mine}
    if _used:
        print(entry("operator overrides",
                    ", ".join(f"{k}={v}" for k, v in sorted(_used.items()))))
    if _theirs:
        print(verdict("caution", f"{len(_theirs)} override(s) belong to the "
                                 f"{'derivative' if DQDV_COMPUTATION == 'histogram' else 'histogram'} "
                                 f"path and do nothing here"))
        print(bullet(", ".join(f"{k}={v}" for k, v in sorted(_theirs.items()))
                     + " — set DQDV_COMPUTATION in Cell 2 to use them, or "
                       "remove them."))
    preprocess_params[name] = {**auto, **overrides,
                               "dqdv_method": DQDV_COMPUTATION}
    if DQDV_BIN_MV is not None:           # None = let the profile choose
        preprocess_params[name]["histogram_bin_mV"] = DQDV_BIN_MV

    # THE ANALYSIS AXIS, not the protocol window. Cell 3b measured the window
    # the cell was CYCLED between (the median of the per-cycle limits, e.g.
    # 1.2-2.5 V) and put it in the methods record. This is the voltage axis
    # the curve is COMPUTED on: the full extent of the data, trimmed 50 mV, so
    # that every half-cycle of one dataset shares one axis. They are different
    # quantities and they do not have to agree — cell A's open-circuit start
    # at 2.684 V puts this 134 mV above the protocol top.
    windows[name] = voltage_window(ds)
    # No cycle filter here. The dataset itself was cut in Cell 3b, once, for
    # both caps — a second filter against MAX_CYCLE alone could only ever
    # disagree with it, by ignoring the per-dataset `max_cycle` the prompts
    # asked for.
    keys = list(ds.half_cycle_keys())
    signals[name] = {}
    # Progress only once the work is SLOW ENOUGH TO WAIT FOR. A fixed
    # every-nth-item ticker printed ten lines of "2 of 21 (0 s)" for a job
    # that finished in under a second, which is noise dressed as reassurance.
    # Nothing is printed for the first few seconds; after that, roughly every
    # PROGRESS_EVERY_S, with an estimate of what is left.
    _t0, _next = time.time(), PROGRESS_QUIET_S
    for _i, k in enumerate(keys, 1):
        signals[name][k] = half_cycle_report(ds, *k, preprocess_params[name],
                                             window=windows[name])
        _el = time.time() - _t0
        if _el >= _next and _i < len(keys):
            print(f"      {_i} of {len(keys)} half-cycles   {_el:.0f} s "
                  f"elapsed, ~{_el * (len(keys) - _i) / _i:.0f} s left",
                  flush=True)
            _next = _el + PROGRESS_EVERY_S
    n_curve = sum(1 for s in signals[name].values() if s.voltage.size)
    print(entry("curves", f"{n_curve} of {len(keys)}",
                f"half-cycles; axis {windows[name][0]:.4f}-"
                f"{windows[name][1]:.4f} V, {time.time() - _t0:.0f} s"))

    # DOES THE CURVE GIVE THE VOLTAGE CURVE BACK? A derivative is reversible,
    # so the cumulative integral of dQ/dV against V should reproduce the
    # (Q, V) curve it came from. That is the objective test of a dQ/dV
    # computation — many parameter choices look right, few reconstruct — and
    # it is how the histogram path earned the default.
    _rec, _bw, _empty, _bins = [], [], 0, 0
    for k, sg_ in signals[name].items():
        if not sg_.voltage.size:
            continue
        if np.isfinite(sg_.bin_width_mV):
            _bw.append(sg_.bin_width_mV)
        _empty += sg_.n_empty_bins
        _bins += sg_.voltage.size
        # The SAME frame the signal was built from. `ds.half_cycle` is now a
        # cached dict lookup rather than a full-frame scan, so asking twice is
        # no longer expensive — but asking once is still clearer.
        _hc = ds.half_cycle(*k)
        _qc = ("Charge_Capacity" if str(k[1]).lower().startswith("c")
               else "Discharge_Capacity")
        if _qc not in _hc.columns:
            continue
        _rec.append(voltage_reconstruction(
            sg_.voltage, sg_.dqdv,
            pd.to_numeric(_hc[_qc], errors="coerce").to_numpy(),
            pd.to_numeric(_hc["Voltage"], errors="coerce").to_numpy()))
    # --- TRIM THE CUT-OFF APPROACH ---------------------------------------
    # A half-cycle ends on a VOLTAGE, and as the cell reaches it dV/dQ
    # collapses so dQ/dV climbs: within 20 mV of the cut-off the NNM charge
    # curve stands at 5.2x its own median height. That rise is protocol, not
    # chemistry — the same argument this pipeline already makes about a
    # constant-voltage hold — and left in it is fitted as a peak, with the
    # cubic baseline straining to follow what the peaks cannot. That was the
    # whole cause of the fitting not converging.
    #
    # PER STEP, because a half-cycle has one cut-off and it is at a different
    # end for each: charge terminates high, discharge terminates low. The trim
    # is measured on every half-cycle and the step's MEDIAN applied to all of
    # them, so the voltage axis stays common within a step — which the
    # reference peak list depends on.
    trims[name] = {}
    if TERMINAL_TRIM and signals[name]:
        for _st in ("Charge", "Discharge"):
            _mine = [v for k, v in signals[name].items() if k[1] == _st]
            if not _mine:
                continue
            # A FIXED span at the terminating end when one is set, because
            # the measured rule is bimodal across a full run and its median
            # is not a physical quantity — see signal.TERMINAL_TRIM.
            _fx = TERMINAL_TRIM_FIXED_MV
            if _fx:
                _lo = (_fx / 1000.0) if _st == "Discharge" else 0.0
                _hi = (_fx / 1000.0) if _st == "Charge" else 0.0
            else:
                _m = [terminal_trim(x.voltage, x.dqdv) for x in _mine
                      if np.size(x.voltage) > 20]
                if not _m:
                    continue
                _lo = float(np.median([a for a, _ in _m]))
                _hi = float(np.median([b for _, b in _m]))
            trims[name][_st] = (_lo, _hi)
            if _lo <= 0 and _hi <= 0:
                continue
            for k in [k for k in signals[name] if k[1] == _st]:
                signals[name][k] = apply_terminal_trim(
                    signals[name][k], _lo, _hi)
        _said = {k: v for k, v in trims[name].items() if max(v) > 0}
        if _said:
            for _st, (_lo, _hi) in sorted(_said.items()):
                _end = "top" if _hi > _lo else "bottom"
                print(entry(f"{_st.lower()} cut-off trimmed",
                            f"{1000 * max(_lo, _hi):.0f} mV",
                            f"off the {_end} of the window — the approach to "
                            f"the cut-off, not a redox feature"))
            print(bullet("The charge that passed there is still the cell's "
                         "and is still counted; it is no longer in the curve, "
                         "so integral fidelity now reports the shortfall "
                         "instead of a baseline absorbing it."))
        else:
            print(entry("cut-off trim", "none needed",
                        "TERMINAL_TRIM is off — see §A11"))

    reconstruction[name] = _rec
    print(entry("dQ/dV method", DQDV_COMPUTATION,
                "charge binned by voltage, no derivative"
                if DQDV_COMPUTATION == "histogram"
                else "cycler column, sorted, rebinned, despiked, smoothed"))
    print(describe_reconstruction(
        _rec, method=DQDV_COMPUTATION, window=windows[name],
        bin_widths=_bw if DQDV_COMPUTATION == "histogram" else None,
        empty_fraction=(_empty / _bins) if _bins else None))

    # On a two-phase plateau the true dV per record drops below the cycler's
    # voltage resolution and the recorded voltage wobbles backwards, so the
    # instrument divides a positive dQ by a negative dV and reports the wrong
    # sign with a huge magnitude. Sorting by voltage then interleaves those
    # with the honest records and smoothing averages the two — which is the
    # up-and-down pattern on a first cycle, and at the LTO plateau an
    # inverted peak. A high count here is a FLAT PLATEAU, not a fault.
    #
    # DERIVATIVE PATH ONLY. The histogram never sorts and never divides, so
    # it has nothing to correct and reports 0 every time — this block was
    # printing nothing on the default path while explaining itself at length.
    if DQDV_COMPUTATION == "derivative":
        _nor = {k: s.n_oriented for k, s in signals[name].items()
                if s.n_oriented}
        if _nor:
            _tot = sum(_nor.values())
            _wor = max(_nor, key=_nor.get)
            print(entry("plateau sign corrected", f"{_tot} record(s)",
                        f"across {len(_nor)} half-cycle(s); most in cycle "
                        f"{_wor[0]} {_wor[1]} ({_nor[_wor]})"))
print(rule())


## Cycle integrity — where did the charge go?

During charge, dQ/dV should be positive: charge goes in and the potential
rises. Where it is negative the potential is moving *backwards* under applied
current. Three unrelated things do that, and 1.8.x conflated all three under a
single threshold that transferred neither between the cells of one triplicate
nor between chemistries.

Every contiguous backwards run is now inventoried — how much charge passed,
over what voltage band, at what current — and classified by **where on the
curve it sat**:

| verdict | test | what it is |
|---|---|---|
| `plateau` | at full current, and \|dQ/dV\| there is ≥ 5% of this half-cycle's own maximum | a two-phase equilibrium pinning the potential. Real, intended charge |
| `cv_hold` | current below the constant-current value, within 50 mV of the window edge | protocol, not chemistry |
| `parasitic` | at full current, \|dQ/dV\| small, away from the limit | charge passed with no redox process to account for it |

Only the third is a fault. Every term is measured against the half-cycle
**itself** — its own peak \|dQ/dV\|, its own current, its own capacity — which
is why it transfers, and it is exactly what a pooled threshold could not do.

The plateau test reads the curve, **not a peak list**. That is why integrity
runs before detection, and why the reference-cycle chooser in Cell 8 can be
handed its verdict.

**Measured on 1,384 half-cycles across six cells:** every half-cycle flagged
ANOMALOUS sits in a cycle whose coulombic efficiency is below 60%, and above
CE 60% nothing anywhere carries more than 0.4% of its capacity backwards.

### Bands

- **clean** — no charge passed backwards away from a redox process
- **suspect** — some did, but under 2% of this half-cycle's own capacity
- **ANOMALOUS** — at or above 2%
- **too few records** — the half-cycle is too short to read. *Not* the same as
  clean: it means we could not tell, and the reference-cycle chooser refuses to
  anchor a dataset to it.

### Where the cell died

This cell also finds the point at which the cell stopped delivering, judged
against its own working capacity (the 90th percentile, not the median — where
a cell dies a third of the way into a long record, the median is
0.04 mAh/g and useless).

**That limit stops the fitting, not the plots.** There is nothing to fit in a
flat line, and every such fit is discarded downstream — but the heatmap and the
waterfall should still show the whole record, because a cell going flat at
cycle 130 is exactly what the reader needs to see.


In [ ]:
# CELL 5 — CYCLE INTEGRITY
# =============================================================================
# Where did the charge go? Every contiguous run in which the potential moved
# against the applied current is classified by WHERE on the curve it sat:
#
#   plateau     |dQ/dV| there is a substantial share of this half-cycle's own
#               maximum — a two-phase equilibrium. Real, intended charge.
#   cv_hold     current below the constant-current value, at the voltage
#               limit. Protocol, not chemistry.
#   parasitic   full current, |dQ/dV| small, away from the limit. Charge with
#               no redox process to account for it.
#
# Only the third is a fault, and it is measured against the half-cycle's own
# capacity, never against a population.
#
# A SECOND, independent test needs no dQ/dV at all: did more charge pass than
# the material can hold? A half-cycle above 1.2x the theoretical capacity is
# condemned whatever the shape of its curve, because a curve cannot vouch for
# charge the material could not have held. On P3 cell A that is the difference
# between reporting 598 mAh/g of reassuring "two-phase plateau" and reporting
# 38 — the other 559 are on half-cycles running at up to 4.3x theoretical.
#
# A THIRD asks how each half-cycle ENDED. A galvanostatic half-cycle
# terminates on voltage, so one that stopped short either was caught by the
# export mid-flight (`in_progress`) or stopped on something else — a step
# timer, a safety limit, an aborted schedule (`early_stops`). Both are
# excluded from the FIT and kept in every figure.
integrity, integrity_band, fit_limit = {}, {}, {}
in_progress = {}      # (cycle, step) caught part-way through, per dataset
early_stops = {}      # half-cycles that stopped short WITHOUT being caught
not_measurements = {}  # the union of the two — what the fit skips
for name in datasets:
    print(rule(name))
    R, band = integrity_report(
        signals[name], window=windows[name], name=name, verbose=True,
        theoretical_mAh_g=user_parameters.get(name, {}).get(
            "theoretical_capacity_mAh_g"))
    integrity[name], integrity_band[name] = R, band
    # Flat, like every other output in this notebook; Cell 14's `organise_run`
    # files it into <cell>/1_cycling/ by its filename prefix.
    R.to_csv(os.path.join(RUN_DIR, f"{name}_cycle_integrity.csv"), index=False)

    # Where the cell stopped delivering. This limits the FIT, not the plots.
    _caps = {k: s.capacity for k, s in signals[name].items()}
    if not _caps:
        print(verdict("bad", "no half-cycles — nothing to judge"))
        fit_limit[name] = None
        in_progress[name], early_stops[name] = set(), pd.DataFrame()
        not_measurements[name] = set()
        continue
    _last, _why = last_useful_cycle(_caps)
    # `is not None`, not truthiness. `last_useful_cycle` returns the cycle
    # BEFORE the death and documents in its own body that returning 0 would
    # read as "never dies" to a caller testing truthiness — this is that
    # caller, and it should not be the reason that stays true.
    fit_limit[name] = _last if STOP_FITTING_AT_DEATH else None
    _n = max(c for c, _ in _caps)
    if fit_limit[name] is not None:
        print(verdict("caution", f"fitting stops at cycle {fit_limit[name]} "
                                 f"of {_n} "
                                 f"({(1 - fit_limit[name] / _n):.0%} of the "
                                 f"record skipped)"))
        print(bullet(_why))
    elif _last is not None:
        print(verdict("caution", "the cell died, and the whole record will "
                                 "be fitted anyway"))
        print(bullet(f"{_why}. STOP_FITTING_AT_DEATH is False in Cell 2."))
    else:
        print(verdict("ok", "the cell never stopped delivering"))
        print(bullet(f"{_why} — fitting the whole record"))

    # ONE termination check, two questions asked of it. Each used to run its
    # own: 10.6 s apiece on P3 cell A, for an identical frame.
    _T = termination_check(datasets[name])

    # A cycler export is usually written while the cell is still running,
    # so the file's last half-cycle is unfinished. The question is settled
    # on VOLTAGE, because that is what a galvanostatic half-cycle terminates
    # on: a finished one reaches the cut-off, an interrupted one stops
    # wherever it had got to. Omitted from the FIT at half-cycle granularity
    # — the completed half of the same cycle is kept — and still plotted.
    in_progress[name], _ip_why = half_cycles_in_progress(datasets[name],
                                                            checked=_T)
    print(verdict("caution" if in_progress[name] else "ok",
                  "the export caught a half-cycle mid-flight"
                  if in_progress[name] else "every half-cycle finished on "
                                            "its cut-off voltage",
                  label="termination"))
    print(bullet(_ip_why))

    # A half-cycle that stopped short of the cut-off but ran a NORMAL LENGTH
    # of time was not caught by the export — it terminated on something else
    # (a step timer, a safety limit, an aborted schedule). That is a finding
    # about the experiment, not a footnote about the file.
    #
    # It is now ACTED ON as well as printed. Until 1.9.0.16 this frame was
    # computed, described in one console line, and then never referred to
    # again: not excluded from the fit, not recorded in dataset_info.txt, not
    # in the manifest, not written to a CSV. A half-cycle that terminated on
    # a step timer is not a capacity measurement, and it was going into every
    # mean and every retention figure as though it were.
    early_stops[name] = early_terminations(datasets[name], checked=_T)
    if not early_stops[name].empty:
        early_stops[name].to_csv(
            os.path.join(RUN_DIR, f"{name}_early_terminations.csv"),
            index=False)
        print(verdict("bad", f"{len(early_stops[name])} half-cycle(s) ended "
                             f"short of the cut-off WITHOUT being caught "
                             f"mid-export"))
        for _, _e in early_stops[name].iterrows():
            _d = ("" if not np.isfinite(_e["duration_frac"])
                  else f", after {100 * _e['duration_frac']:.0f}% of a "
                       f"normal duration")
            print(bullet(f"cycle {int(_e['cycle'])} {_e['step']}: "
                         f"{_e['v_end']:.4f} V, {_e['short_mV']:.0f} mV short"
                         f"{_d} — {_e['termination_class']}"))
        print(bullet("Excluded from the fit and from the capacity tables, "
                     "kept in every figure, and listed in "
                     f"{name}_early_terminations.csv."))

    # What the fit skips, from both causes at once.
    not_measurements[name] = set(in_progress[name]) | {
        (int(r.cycle), str(r.step)) for r in early_stops[name].itertuples()}
print(rule())


## Cycling performance

Capacity, coulombic efficiency, retention, fade rate, energy and power density,
average discharge voltage, round-trip energy efficiency, and rate capability.

This is arithmetic on the cycler's own counters. Whether a given cycle's
numbers *mean* anything is the previous cell's question, and the two are kept
deliberately apart: a capacity is a capacity whether or not the charge that
produced it went where it was supposed to.

### Three cells, not one

This was one cell producing 23,500 characters of console and 26 figures, which
is more than Cells 4, 5, 8, 9, 10 and 11 put together and impossible to
navigate. It is now split by what you do with the output:

| | | output |
|---|---|---|
| **6a** | the per-cycle tables and headline metrics | 6.2k, 1 CSV |
| **6b** | the four figures a cycling paper is built from | 7.5k, 13 files |
| **6c** | retention, fade, energy, power, efficiency, rate | 9.8k, 22 files |

Run 6a first whatever else you skip: it returns `all_cycle_tables`, and every
figure in 6b and 6c reads it — both for the values and for which cycles Cell 5
found were not measurements. Without it each figure would rebuild the same
tables from the raw records and they would be free to disagree.

Capacity is computed as max − min per half-cycle rather than taken from the
cycler's running column, so the plotted values match the summary table.

**Energy and power densities here are on the ACTIVE-MASS basis**, computed
from the active mass, the blend fraction and the electrode area — all three
of which you supplied in Cell 3b. Whole-cell densities would additionally
need the separator, electrolyte and counter-electrode masses; Ratatosk
estimates only the counter electrode, from geometry and metal density, and
`counter_electrode_measured` records whether it was weighed or guessed.
Earlier versions of this cell claimed the other two were looked up and
recorded with `*_source` fields. They were not, and the machinery has been
removed rather than left reading as a feature.


In [ ]:
# CELL 6a — CYCLING PERFORMANCE: THE TABLES
# =============================================================================
# Capacity, coulombic efficiency, retention and fade, per cycle, per cell —
# and the headline metrics a paper quotes. Arithmetic on the cycler's own
# counters; whether a given cycle's numbers MEAN anything is Cell 5's
# question, deliberately kept separate.
#
# This cell also computes `all_cycle_tables`, which every figure in 6b and 6c
# reads. Run it first: without it each figure would rebuild the same tables
# from the raw records, and they would be free to disagree with these.
_CYCLING = dict(save_location=RUN_DIR, verbose=CYCLING_VERBOSE)

# WHAT RATE WAS THIS CELL ACTUALLY CYCLED AT? Answered from the recorded
# current, once, before any figure is drawn — because the cycler header
# records only its FIRST step, and a rate-capability run that ramps from
# C/20 to 5C had every caption in this notebook saying "0.05 C". The answer
# is stored on `user_parameters` and read by the captions, the retention
# figures and the report.
annotate_rate_protocol(electrochemical_data, user_parameters,
                          verbose=CYCLING_VERBOSE)

# `not_measurements` so the tables leave out exactly the half-cycles the fit
# does — one verdict, read in both places.
all_cycle_tables = cycling_summary(electrochemical_data, user_parameters,
                                      in_progress=not_measurements, **_CYCLING)


In [ ]:
# CELL 6b — CYCLING PERFORMANCE: THE FIGURES THAT GO IN THE PAPER
# =============================================================================
# The four a cycling paper is built from:
#
#   voltage_profiles        V against specific capacity, every cycle
#   voltage_profiles_key    the same, filtered to the key cycles
#   cycle_life              capacity and coulombic efficiency against cycle
#   comparative_capacity    every cell on one axis
#
# Each is saved in the format the operator asked for in Cell 3b, and each
# marks or omits the half-cycles Cell 5 found were not measurements.
voltage_profiles(electrochemical_data, user_parameters,
                    all_cycle_tables=all_cycle_tables, **_CYCLING)
voltage_profiles_key(electrochemical_data, user_parameters,
                        all_cycle_tables=all_cycle_tables, **_CYCLING)
cycle_life(electrochemical_data, user_parameters,
              all_cycle_tables=all_cycle_tables, **_CYCLING)
comparative_capacity(electrochemical_data, user_parameters,
                        all_cycle_tables=all_cycle_tables, **_CYCLING)


In [ ]:
# CELL 6c — CYCLING PERFORMANCE: THE SUPPORTING FIGURES
# =============================================================================
# Everything that supports the four above rather than appearing beside them:
#
#   capacity_retention          retention against a reference cycle
#   fade_rate                   loss per cycle, and cumulative
#   power_and_energy            three normalisations, plus the Ragone plot
#   average_discharge_voltage   mean discharge voltage against cycle
#   energy_efficiency           round-trip energy efficiency
#   rate_capability             capacity grouped by C-rate
#
# This is the largest of the three by output — `power_and_energy` alone
# writes eleven files — which is why it is last and on its own.
# `rate_capability` reports and skips on a single-rate run.
capacity_retention(electrochemical_data, user_parameters,
                      all_cycle_tables=all_cycle_tables, **_CYCLING)
fade_rate(electrochemical_data, user_parameters,
             all_cycle_tables=all_cycle_tables, **_CYCLING)
power_and_energy(electrochemical_data, user_parameters,
                    all_cycle_tables=all_cycle_tables, **_CYCLING)
average_discharge_voltage(electrochemical_data, user_parameters,
                             all_cycle_tables=all_cycle_tables, **_CYCLING)
energy_efficiency(electrochemical_data, user_parameters,
                     all_cycle_tables=all_cycle_tables, **_CYCLING)
rate_capability(electrochemical_data, user_parameters,
                   all_cycle_tables=all_cycle_tables, **_CYCLING)


## dQ/dV figures

Six views of the same data, because no single one shows everything.

1. **All-cycle overlay** — every cycle on one axis, coloured by cycle number.
   The figure every paper includes.
2. **Key-cycle overlay** — the same, filtered to the key cycles, for
   publication.
3. **Split key cycles** — charge and discharge on separate panels with
   sequential colourmaps. More readable when the peaks overlap.
4. **Waterfall** — stacked offset traces. Shows a peak splitting, shifting or
   dying in a way an overlay cannot. Every complete cycle is plotted: the
   figure height is bounded and the trace pitch compressed to fit, so a
   200-cycle run gives a dense ridgeline rather than a silently thinned subset.
5. **Voltage–cycle heatmap** — the whole degradation story in one figure.
6. **Preprocessing QC** — raw against processed for one cycle, to check the
   smoothing has not invented or erased a feature.

Figure widths default to 6.73 in, the RSC double-column width. Colours are
Okabe–Ito, chosen to survive the common forms of colour blindness and
greyscale printing.


In [ ]:
# CELL 7 — dQ/dV FIGURES
# =============================================================================
# Seven figures per dataset, each with a caption you can paste into a
# manuscript — which is why the captions are held to the same standard as the
# numbers. Two things they now get right that they did not before:
#
#   the voltage window   is the PROTOCOL window measured in Cell 3b, not the
#                        analysis axis from Cell 4. Those differ, and the
#                        caption used to quote the second: "cycled between
#                        1.25 and 2.63 V" for a cell cycled 1.2-2.5 V, with
#                        three different cut-offs across a triplicate cycled
#                        identically.
#   incomplete cycles    are named. They are KEPT in every figure on purpose —
#                        seeing where the run has got to is the point — but a
#                        caption has to say which ones are not measurements.
processed_dqdv = {}
for name, ds in datasets.items():
    # What Cell 5 found, handed to the captions. `_get_display_name` and the
    # caption builders read `user_parameters`, so that is where it goes.
    user_parameters.setdefault(name, {})["incomplete_cycles"] = sorted(
        {c for c, _ in not_measurements.get(name, ())})
    processed_dqdv.update(as_processed_dqdv(
        ds, signals[name], preprocess_params[name],
        window=windows[name], profile=profiles[name]))

print(rule("dQ/dV FIGURES"))
plot_dqdv_all_cycles(processed_dqdv, user_parameters, save_location=RUN_DIR)
plot_dqdv_key_cycles_combined(processed_dqdv, user_parameters,
                                 save_location=RUN_DIR)
plot_dqdv_key_cycles_split(processed_dqdv, user_parameters,
                              save_location=RUN_DIR)
for step in ("Charge", "Discharge"):
    plot_dqdv_waterfall(processed_dqdv, user_parameters, step=step,
                           save_location=RUN_DIR)
# The heat maps have their own cell below — see CELL 7b.

# THE QC FIGURE IS SAVED NOW. It was the only plot in the module with no
# `save_location` — it drew, called plt.show(), and vanished with the kernel,
# so a run folder had seven dQ/dV figures and no record of what the
# preprocessing had actually done to the curve.
# The cycle is a named setting rather than a literal 2 buried in the call.
plot_preprocessing_qc(processed_dqdv, user_parameters,
                         cycle=QC_CYCLE, save_location=RUN_DIR)

# ONE construction of the processed curve, not two. `as_processed_dqdv` above
# already built exactly these three columns per half-cycle; this used to
# rebuild them from `signals` a second time, so the CSV and the figures were
# free to diverge.
for name in datasets:
    rows = [df.assign(Cycle=cyc, Step=step)
            for (cyc, step), df in sorted(processed_dqdv[name]["data"].items())]
    if rows:
        pd.concat(rows).to_csv(
            os.path.join(RUN_DIR, f"{name}_processed_dqdv.csv"), index=False)
print(rule())


## 7b. The heat maps

One figure per step, and the one worth the most time. The waterfall above
shows every cycle drawn over every other; this shows the same data as a
surface, so a peak that walks, splits, or dies is a shape rather than a
thicket of overlapping lines.

Two things about how it is drawn, because both change what it means.

**The grid comes from the DATA, not from the protocol window.** A cell cycled
between 1.2 and 2.5 V may be analysed from 1.25 to 2.63 V, and binning onto
the protocol window throws away whatever lies outside it. It also used to fill
the region a half-cycle never reached with a repeat of its nearest measured
value, which renders as the same near-black as a measured zero — so a third of
some panels was colour that no measurement supported.

**A voltage a half-cycle never reached is now unmeasured, and drawn as such.**
Those cells are masked, one shade off the zero end of the ramp, and the blue
step line traces the edge of the range each cycle actually traversed. Where
that edge moves — LTO's delithiation stops 622 mV lower by cycle 10, and NNM
cell B's whole window collapses as it dies — the line is the finding, and the
per-cell page says the same thing in words.

Both versions are written: `..._dQdV_heatmap_<Step>.png` with the line, and
`..._dQdV_heatmap_<Step>_plain.png` without it, for a talk where the shape is
the whole point.


In [ ]:
# CELL 7b — dQ/dV HEAT MAPS
# =============================================================================
# Their own cell rather than two lines inside Cell 7. They are the figure most
# often read and the one most often exported, and they were being drawn in the
# middle of a loop that also produced six other things.
print(rule("dQ/dV HEAT MAPS"))
for step in ("Charge", "Discharge"):
    # WITH the envelope: the reading of the panel that cannot mislead.
    plot_dqdv_heatmap(processed_dqdv, user_parameters, step=step,
                         save_location=RUN_DIR, envelope=True)
    # ...and the version that does not raise the question. No line, no note,
    # and the unvisited cells take the colormap's own zero colour, so the
    # panel reads as one continuous surface — which is what it looked like
    # before any of this was fixed. It is a SECOND file and never a
    # replacement: two versions, one that hides the issue and one that
    # highlights it, and the run always writes both.
    plot_dqdv_heatmap(processed_dqdv, user_parameters, step=step,
                         save_location=RUN_DIR, envelope=False,
                         suffix="_plain")
print(rule())


## Peak detection, and choosing the reference cycle

Detection is two passes plus two filters. `find_peaks` with an adaptive
prominence finds clean maxima; a second-derivative pass finds shoulders on the
flanks that never become maxima; anything found twice within 20 mV is
deduplicated, keeping the more prominent; and a peak the voltage window has cut
in half is discarded rather than fitted, because its centre would be an
extrapolation.

### Why the reference cycle is chosen rather than assumed

One cycle's peak list is fitted and tracked in **every** cycle of the dataset.
A bad reference cycle is therefore not a bad cycle — it is a bad dataset, and
nothing downstream can recover from it.

Hard-coding it to an early cycle and checking only that the cycle exists is not
enough. On one
C, cycle 2 was a half-cycle that put in 580.9 mAh/g and got 115.8 back —
coulombic efficiency 19.9%, 1301 records against a median of 290. Its peak list
set the fit for all 220 cycles:

```
  charge reference peaks
    cell A   3.220*, 3.301, 3.643, 3.700*
    cell B           3.306, 3.648, 3.703*
    cell C           3.640, 3.879, 3.901, 4.193
```

Cell C misses the 3.30 V feature that both other cells track for a hundred
cycles, and carries a 4.193 V ghost instead. Nothing in the output said so.

**The rule now:** the lowest cycle at or above 2 whose Charge and Discharge
half-cycles both pass the integrity check. *Lowest*, not best — a fingerprint
should come from early in the life of the cell, and moving further into the run
means tracking against a peak list that has already aged. Pass
`reference_cycle=N` to choose it yourself; it is honoured, checked and warned
about, never overridden.


In [ ]:
# CELL 8 — PEAK DETECTION AND THE REFERENCE CYCLE
# =============================================================================
# The reference cycle's peak list is fitted and tracked in EVERY cycle, so a
# bad one is not a bad cycle — it is a bad dataset.
#
# It is chosen as the lowest cycle that is BOTH past formation AND passes Cell
# 5's integrity check. Those are two different questions and the second cannot
# answer the first: a cycle can be perfectly clean and still be the wrong
# fingerprint, because the cell was still forming when it was measured.
#
# There is no consensus number of formation cycles — the 2024 Energy &
# Environmental Science review says protocols run from one 1.7 h cycle to
# multi-day schedules and that no general comparison between them is possible.
# What the field agrees on is the CRITERION: coulombic efficiency settling.
# So it is measured per cell; see detect.formation_end, and set
# REFERENCE_CYCLE_FLOOR if you want a house rule (6 is a defensible
# conservative choice) applied on top.
#
# To choose it yourself, set `reference_cycle` in Cell 3b — an explicit answer
# is honoured, checked and warned about, never overridden.
#
# The minimum separation between two peaks is a statement about peak WIDTH, so
# it comes from the profile: 30 mV apart is one feature on a broad layered
# oxide and two on a sharp two-phase material.
detections = {}
for name in datasets:
    print(rule(_get_display_name(name, user_parameters.get(name, {}),
                                    user_parameters)))
    _ov = detection_overrides(user_parameters.get(name, {}))
    if _ov:
        print(entry("operator overrides",
                    ", ".join(f"{k}={v}" for k, v in sorted(_ov.items()))))
    _dspec = detect_spec_for_profile(
        profiles[name],
        min_distance_mV=_ov.get("peak_min_distance_mV"),
        prominence_fraction=_ov.get("peak_prominence_fraction"),
        min_width_mV=_ov.get("peak_min_width_mV"),
        edge_exclusion_mV=_ov.get("edge_exclusion_mV"),
        shoulder_detection=_ov.get("shoulder_detection"))
    print(entry("profile", profiles[name]["class"],
                f"peaks must be {_dspec.min_distance_mV:.0f} mV apart, "
                f"prominence >= {_dspec.prominence_fraction:.0%} of the "
                f"curve's own maximum"))

    # COULOMBIC EFFICIENCY PER CYCLE, from Cell 6a's own tables, with the
    # half-cycles Cell 5 judged were not measurements left out — an unfinished
    # final cycle reads as CE 52% and would drag the median it is compared
    # against. This is what tells the selector where formation ended.
    _ct = all_cycle_tables.get(name)
    _eff = {}
    if _ct is not None and not _ct.empty and "CE_%" in _ct.columns:
        _bad = {c for c, _ in not_measurements.get(name, ())}
        _eff = {int(c): float(v)
                for c, v, inc in zip(_ct["Cycle"], _ct["CE_%"],
                                     _ct.get("Incomplete", False)
                                     if "Incomplete" in _ct.columns
                                     else [False] * len(_ct))
                if int(c) not in _bad and not bool(inc)
                and v is not None and np.isfinite(v)}
    _an = analysis_overrides(user_parameters.get(name, {}))
    detections[name] = detect_all(signals[name], name=name, spec=_dspec,
                                     reference_cycle=_an.get("reference_cycle"),
                                     integrity=integrity_band[name],
                                     efficiency=_eff, verbose=True)
print(rule())

detected_peaks = {}
for d in detections.values():
    detected_peaks.update(as_detected_peaks(d))
plot_detected_peaks(processed_dqdv, detected_peaks, user_parameters,
                       save_location=RUN_DIR)


## Multi-peak fitting

Each half-cycle is fitted as a sum of pseudo-Voigt components on a polynomial
baseline, seeded from that cycle's own detected peaks.

**The model setup comes from the profile class, and it matters more than any
other fitting parameter:**

| profile | centre tolerance | sigma | baseline |
|---|---|---|---|
| sharp (LTO, LFP) | 30 mV | 3–30 mV | linear |
| moderate | 70 mV | 5–100 mV | quadratic |
| broad (layered oxides) | 70 mV | 5–200 mV | quadratic |

A sharp peak sits on a flat background, so a wide sigma bound would let a
component inflate until it *is* the background. A broad one sits on a curved
background that a straight line cannot follow — fit a broad dQ/dV with a
linear baseline and R² falls from 0.54 to 0.27 on the same data.

### Parallelism

`fit_many` measures before it commits: it times a few half-cycles serially,
estimates the total, and only spawns worker processes if the estimate clears
60 seconds. A process pool costs 5–8 s to start on Windows and a forty-second
job does not repay it. Every branch says which way it went and why.

The workers return plain dictionaries, never lmfit `ModelResult` objects — a
`ModelResult` cannot cross a process boundary through the standard library's
pickle, and that single constraint blocked parallel fitting in 1.8.x.

### Reproducibility, and what it depends on

**On one machine, a run repeats exactly.** Re-running the same build on the
same file returns bit-identical fitted parameters.

**Across machines it depends entirely on whether the fits converged**, and the
difference is stark. The same build was run on two computers — Windows and
Linux, different numpy/scipy/lmfit builds — on the same two files:

| dataset | fits converged | peak centres agree to | areas agree to |
|---|---|---|---|
| sharp, closure determined | all | 10⁻¹¹ V | 10⁻⁶ mAh g⁻¹ |
| broad, closure not determined | 13% | 2 mV typical, 40 mV worst | 2 mAh g⁻¹ typical |

On the broad dataset three half-cycles in thirty-six even ended with a
*different number of components*, one machine keeping a peak the other dropped
as degenerate.

This is not a numerical-hygiene problem, and tightening tolerances will not
touch it. A converged fit sits at a minimum, and both machines find the same
minimum to machine precision. A fit that stopped at the iteration ceiling sits
wherever the optimiser gave up, and where it gives up depends on the last bits
of the arithmetic — which differ between BLAS builds. **The non-convergence is
the reproducibility problem**; the same quantity that makes a fit slow makes it
machine-dependent.

Two consequences. Every coherence verdict was identical on both machines, so
the *verdicts* travel even where the numbers do not — which is the argument for
reporting verdicts rather than bare numbers. And `run_manifest.json` records
the platform, the Python version and the numpy, scipy and lmfit versions for
every run, because on an undetermined dataset those are part of the result.


In [ ]:
# CELL 9 — MULTI-PEAK FITTING
# =============================================================================
# Pseudo-Voigt components on a polynomial baseline, with the baseline degree
# and sigma bounds taken from each dataset's own profile class:
#
#   sharp     (LTO, LFP)          sigma 3-30 mV,  linear baseline
#   moderate                      sigma 5-100 mV, quadratic baseline
#   broad     (layered oxides)    sigma 5-200 mV, cubic baseline
#
# A sharp peak sits on a flat background, so a wide sigma bound would let a
# component inflate until it IS the background. A broad one sits on a curved
# background that a straight line cannot follow.
#
# `fit_many` TIMES a few half-cycles before deciding whether to spawn workers:
# a pool costs 5-8 s to start on Windows and a forty-second job does not repay
# it. It says which way it went and why.
#
# It also asks whether the machine can AFFORD the workers. Each one is a fresh
# interpreter that re-imports numpy, scipy and lmfit and holds that footprint
# — measured at 181 MB, and more on Windows — while the job it is sent is 5
# KiB. Sizing the pool at `cpu_count - 1` on a busy machine is what killed the
# kernel here twice: the fits were fine, the memory was not there. The pool is
# now held to whichever is smaller of the cores, the free memory and
# PARALLEL_MAX_WORKERS, and the console says which of the three decided.
# TWO STAGES, and the console says which is which. Cell 9 fits each half-cycle
# ONCE at its profile's baseline degree; Cell 10 refits a sample at three
# degrees to ask how much the answer depended on that choice. Both print
# "fitting N half-cycles", so both are now labelled.
fit_results, fit_jobs, shape, resolvability = {}, {}, {}, {}
band_sets = {}

# --- STAGE 0: is there anything here to decompose? -------------------------
# Asked on the REFERENCE half-cycle only, before the run commits to fitting
# every cycle three times over. Six fits, against about six hundred.
#
# The pipeline has always been able to answer this — Cell 10 does, after the
# work. On a solid-solution layered oxide it says NOT DETERMINED, withholds
# the capacity attribution on nearly every cycle, and finds no peak that moves
# like a redox feature. Correct, and an hour late. A curve with no resolved
# peaks does not become decomposable because the optimiser tries harder.
#
# The verdict does not stop the run. It sets expectations, and it is written
# into the reports, so a reader is told in the first minute which of three
# situations they are in rather than inferring it from a page of blanks.
if RUN_PEAK_FITTING:
    print(rule("IS THERE ANYTHING HERE TO DECOMPOSE?"))
    for name in datasets:
        det = detections[name]
        _ref_jobs, _n_sh, _n_pk = [], 0, 0
        for step in ("Charge", "Discharge"):
            _k = (det.reference_cycle, step)
            _s, _pk = signals[name].get(_k), det.peaks.get(_k)
            if _s is None or _s.voltage.size == 0 or _pk is None or _pk.empty:
                continue
            _n_pk += len(_pk)
            if "is_shoulder" in _pk:
                _n_sh += int(_pk["is_shoulder"].fillna(False).sum())
            _ref_jobs.append((_s.voltage, _s.dqdv,
                              spec_for_profile(_pk, profiles[name]),
                              _k))
        print("\n" + heading(_get_display_name(
            name, user_parameters.get(name, {}), user_parameters)))
        resolvability[name] = assess_resolvability(
            _ref_jobs,
            shoulder_fraction=(_n_sh / _n_pk) if _n_pk else None,
            n_primaries=(_n_pk - _n_sh) if _n_pk else None,
            n_components=(_n_pk // max(len(_ref_jobs), 1)) if _n_pk else None,
            verbose=True)
    # ONE MATERIAL, ONE MODEL. A mechanism is classified per dataset from one
    # reference cycle; replicates of the same material must not end up fitted
    # with different models, because pooling them is what replicates are for.
    # See `quality.reconcile_mechanisms` for why this takes the most
    # permissive call rather than a majority.
    resolvability = reconcile_mechanisms(
        resolvability,
        {n: str((user_parameters.get(n) or {}).get("composition") or n)
         for n in resolvability},
        verbose=True)
    _unres = [n for n, a in resolvability.items()
              if a.get("verdict") == RESOLVABILITY_UNRESOLVED]
    if _unres and not FIT_UNRESOLVED:
        print()
        print(verdict("caution", f"{len(_unres)} dataset(s) will NOT be "
                                 f"decomposed — FIT_UNRESOLVED is off"))
        print(bullet("Peak positions and the dQ/dV envelope are still "
                     "reported; the decomposition into per-peak areas is "
                     "not, because the reference cycle says the data does "
                     "not decide it. Set FIT_UNRESOLVED = True in Cell 2 to "
                     "fit anyway and see for yourself."))
    print("\n" + rule())

# ONE branch, not two. There were an `if not RUN_PEAK_FITTING:` block at the
# top AND a trailing `else:`, so switching fitting off printed both — one
# saying "everything from here to Cell 13", the other "skipping Cells 9 to
# 12". Two messages, two different ranges, both from the same condition.
if RUN_PEAK_FITTING:
    print(rule("Cell 9 — fitting"))
    # --- WHERE DO THE PEAKS FALL SHORT? One pass over every dataset before
    # any of them is fitted, so the regions can be pooled across the
    # replicates of a material. A sample can miss a region a longer look
    # would find, and a pair fitted with different band sets is the same
    # not-comparing-like-with-like the mechanism reconciliation exists to
    # stop. Costs nothing extra: the per-half-cycle discovery it replaces was
    # already a second fit per half-cycle.
    for name in datasets:
        det = detections[name]
        _jobs0, _limit0 = [], fit_limit.get(name)
        if (not FIT_UNRESOLVED
                and resolvability.get(name, {}).get("verdict")
                == RESOLVABILITY_UNRESOLVED):
            continue
        _mm0 = (resolvability.get(name, {}) or {}).get("model") or {}
        for k in sorted(signals[name]):
            s, pk = signals[name][k], det.peaks.get(k)
            if s.voltage.size == 0 or pk is None or pk.empty:
                continue
            if _limit0 is not None and k[0] > _limit0:
                continue
            _jobs0.append((s.voltage, s.dqdv, spec_for_profile(
                pk, profiles[name],
                band_ceiling_span_fraction=_mm0.get(
                    "band_ceiling_span_fraction", 0.0),
                primaries_only=bool(_mm0.get("primaries_only", False))), k))
        band_sets[name] = band_regions(_jobs0, verbose=False)
    band_sets = reconcile_band_sets(
        band_sets,
        {n: str((user_parameters.get(n) or {}).get("composition") or n)
         for n in band_sets},
        verbose=True)

    for name in datasets:
        det = detections[name]
        jobs = []
        limit = fit_limit.get(name)
        if (not FIT_UNRESOLVED
                and resolvability.get(name, {}).get("verdict")
                == RESOLVABILITY_UNRESOLVED):
            # Nothing is hidden: the verdict, its two numbers and its reason
            # were printed above, and the positions and envelope are already
            # in the run. What is skipped is a decomposition whose per-peak
            # areas Cell 11 would have withheld anyway.
            fit_results[name], fit_jobs[name] = [], []
            continue
        for k in sorted(signals[name]):
            s, pk = signals[name][k], det.peaks.get(k)
            if s.voltage.size == 0 or pk is None or pk.empty:
                continue
            if limit is not None and k[0] > limit:
                continue          # the cell is flat; there is nothing to fit
            if k in not_measurements.get(name, ()):
                continue          # caught mid-flight, or ended on something
                                  # other than its cut-off voltage

            # `spec_for_profile`, NOT a bare FitSpec: a broad layered-oxide
            # dQ/dV needs a cubic baseline and a sharp two-phase one needs a
            # tight sigma bound. Getting this wrong is what a visibly bad
            # fit looks like.
            # The DETECTED PEAKS FRAME, not just its centres: `is_shoulder`
            # is what lets `spec_for_profile` tie each shoulder's width to
            # its parent's. Pass the centres alone and every component is
            # refined free, which is how a shoulder inflates into its parent
            # and reports 136% area retention on a cell holding 99% of its
            # capacity.
            # THE MECHANISM CHOOSES THE MODEL, not the shape class. A
            # two-phase transition and a series of transitions both want
            # pseudo-Voigts; a solid solution wants bands; NMC-like data
            # wants both, because a transition sits on the envelope.
            _mm = (resolvability.get(name, {}) or {}).get("model") or {}
            jobs.append((s.voltage, s.dqdv,
                         spec_for_profile(
                             pk, profiles[name],
                             band_ceiling_span_fraction=_mm.get(
                                 "band_ceiling_span_fraction", 0.0),
                             primaries_only=bool(
                                 _mm.get("primaries_only", False))), k))
        # --- MEASURE THE PEAK SHAPE ONCE, THEN HOLD IT ------------------
        # The mixing fraction is a property of the MEASUREMENT — voltage
        # resolution, rebin width, smoothing — not of the redox process, and
        # none of those change from cycle to cycle. Refined per cycle it is
        # free to absorb area: on LTO cell A it wandered 0.65 -> 0.36 -> 0.52
        # across ten cycles and carried the fitted peak area from 61 to 85
        # mAh/g — 37% growth on a cell that had lost 1.7% of its capacity,
        # with R2 between 0.995 and 0.998 the whole way. Every fit was
        # excellent; the trend they formed was fiction. Same argument for the
        # shoulder width ratio.
        # THE NAME FIRST. The calibration below prints several lines, and
        # with the header after them they read as belonging to the PREVIOUS
        # dataset — which is exactly what made Cells 9 and 10 hard to tell
        # apart in the console.
        print("\n" + heading(_get_display_name(
            name, user_parameters.get(name, {}), user_parameters)))
        # --- ONE SET OF BANDS FOR THE DATASET ---------------------------
        # Discovered per half-cycle, a band comes and goes and the component
        # set changes identity from cycle to cycle, which is what put every
        # NMC111 feature at `not assessable` in the coherence audit. Establish
        # the regions once, from where the peaks fall short across a sample,
        # and fit them everywhere. Costs nothing extra: the per-half-cycle
        # discovery was already a second fit. Returns nothing at all where the
        # mechanism permits no band, so LTO is untouched.
        print(section("  Step 1 of 3 — where do the peaks fall short?"))
        # Regions were established for EVERY dataset before this loop, and
        # pooled across the replicates of each material, so both cells of a
        # pair are fitted with the same set. See `fitting.reconcile_band_sets`.
        for _r in _describe_band_set(band_sets.get(name) or {}):
            print(_r)
        _bs = band_sets.get(name) or {}
        if any(_bs.values()):
            jobs = [(v, y, sp.replace(band_seeds=[
                        (r["centre"], r["span_mV"])
                        for r in _bs.get(str(k[1]), [])]), k)
                    for (v, y, sp, k) in jobs]
        print(section("  Step 2 of 3 — calibrate the peak shape"))
        shape[name] = calibrate_shape(jobs, verbose=True)
        jobs = [(v, y, sp.replace(fraction=shape[name]["fraction"],
                                  sigma_ratio=shape[name]["sigma_ratio"]), k)
                for (v, y, sp, k) in jobs]
        print(section("  Step 3 of 3 — fit every half-cycle"))
        t0 = time.perf_counter()
        fit_results[name] = fit_many(jobs, verbose=True, label="fitting")
        fit_jobs[name] = jobs
        # "CONVERGED" HAS TO MEAN CONVERGED. Until 1.9.0.26 this counted
        # `r["success"]`, which `fit_half_cycle` set to True for every fit
        # that did not raise an exception — so the NNM triplicate printed
        # "converged 200 of 200" on a dataset where 57% of the fits ran to
        # the 20,000-evaluation ceiling and lmfit flagged them as failures.
        # The two are now counted separately: `fitted` is what returned a
        # result, `converged` is what the optimiser was satisfied with.
        _res = fit_results[name]
        ok = sum(1 for r in _res if r["success"])
        _conv = sum(1 for r in _res
                    if r["success"] and r.get("converged", True))
        _capped = [r for r in _res if r.get("hit_iteration_cap")]
        print(entry("fitted", f"{ok} of {len(jobs)}",
                    f"in {time.perf_counter() - t0:.0f} s"))
        print(entry("converged", f"{_conv} of {ok}",
                    "the optimiser was satisfied"
                    if _conv == ok else
                    f"{ok - _conv} stopped without converging"))
        if _capped:
            _nf = _capped[0].get("max_nfev", 0)
            print(verdict("caution", f"{len(_capped)} fit(s) ran to the "
                                     f"{_nf:,}-evaluation ceiling and "
                                     f"stopped"))
            print(bullet("Those parameters are where the optimiser ran out "
                         "of evaluations, not necessarily where it settled. "
                         "They are kept, and — unlike in 1.9.0.43 — they are "
                         "NOT demoted for it: measured on the NNM "
                         "triplicate, the fits that ran out of budget had "
                         "the HIGHER median R2, because running out while "
                         "still descending is not a failure. What decides "
                         "whether a component is a measurement is the "
                         "component: whether its area and centre are "
                         "determined, whether it was inferred, whether it "
                         "sits on a bound. This line is a fact about the "
                         "optimiser, and it is reported as one."))
    # THE POOL IS NOT RELEASED HERE, and 1.9.0.43 was wrong to do it.
    # Cell 10 fits again immediately, so releasing at the end of Cell 9 paid a
    # full teardown and a fresh set of Windows process spawns to hand back
    # memory that was wanted back a second later. Measured, the teardown and
    # respawn cost ~420 ms a pass on Linux and 5-8 s per worker on Windows,
    # for nothing — and shutting a loky executor down and immediately
    # rebuilding one is the pattern most likely to hang. Cell 10 releases it,
    # once, when nothing is going to fit again.
    print("\n" + rule())
else:
    print(rule("PEAK FITTING IS OFF"))
    print(verdict("caution", "Cells 9 to 12 will do nothing"))
    print(bullet("No fitted peaks, no tracking, no polarisation, no capacity "
                 "attribution, and no 5_peak_fitting or 6_descriptors "
                 "folders. The cycling analysis in Cells 6a-6c is unaffected "
                 "and complete."))
    print(bullet("To turn it on: answer 'yes' to 'Run full dQ/dV peak "
                 "fitting?' in Cell 3b, or set RUN_PEAK_FITTING = True in "
                 "Cell 2 — then re-run from Cell 9."))
    print(rule())


## Does the answer mean anything?

Two checks run in this cell, on two different things. The quickest way to see
what they are is to follow one half-cycle through.

### A worked example: LTO, cycle 5, charge

Three numbers describe the same half-cycle at three stages of the pipeline:

| stage | number |
|---|---|
| the cycler's own coulomb count | 129.1 mAh g⁻¹ |
| the area under the dQ/dV curve built from it, ∫(dQ/dV)dV | 128.4 mAh g⁻¹ |
| the one fitted peak on that curve (1.546 V), its area | 123.9 mAh g⁻¹ |

Each check is one ratio of a consecutive pair:

**Integral fidelity = 128.4 / 129.1 = 0.99.** Did anything get lost turning
the raw record into a curve? Here, no — the curve carries 99% of the charge
the cycler counted. Binning, smoothing and the ±50 mV edge exclusion together
cost 1%.

**Capacity closure = 123.9 / 128.4 = 0.96.** Of the curve that survived, how
much did the fitted peak actually claim? 96%; the other 4% went into the
baseline.

Closure is measured against the **curve**, not against the cycler, and that is
deliberate. Divide by the cycler's number instead and a preprocessing loss
would be charged to the fit — one bad number, and no way to tell which stage
produced it.

### Why closure is reported as an interval

The LTO peak above is about 9 mV wide on a 1.3 V window. Nothing a polynomial
baseline can do looks like that, so the choice of baseline barely matters:
refit the same half-cycle with a linear, a quadratic and a cubic baseline and
closure comes back 0.96, 0.96, 0.96. The split between "peak" and "background"
is a property of the data.

Now take a broad layered-oxide curve, where one component can be 100 mV wide
on a 2 V window. Over that span a wide pseudo-Voigt and a cubic are nearly the
same shape, and the fit has no basis on which to prefer either. The same three
refits can give 0.41, 0.45 and 0.48 — seven points of the curve's area moving
between "peak" and "background" on nothing but the baseline degree somebody
picked.

So the interval is reported, not a single value:

```
   interval width  =  max(closure) − min(closure)   over baseline degrees 1, 2, 3
```

Below **0.15** the split is determined by the data, and capacity shares are
reported. Above it the fitted areas still go to the CSVs — they remain the best
description of the curve the fit found — but the *share of capacity* is
withheld for that half-cycle, because at that width the share is a fact about
the baseline rather than about the material. The threshold is applied to the
width of the interval; no threshold is placed on the closure value itself.

### Reading the two together

They fail independently, which is the whole reason for keeping them apart:

| fidelity | closure | what that combination means |
|---|---|---|
| high | high | both stages sound — the fitted areas are capacities |
| high | low | the curve is right but the peak/background split is not; a broad, unresolved profile |
| low | high | the fit describes the curve well, but charge was lost upstream — check the bin width and the edge exclusion |
| low | low | do not read areas from this dataset |

A wide interval also carries physical meaning. A split that survives a change
of baseline describes a feature no polynomial can mimic — narrow, first-order,
two-phase. A split that collapses describes a solid-solution envelope, where
the peak was never a separate thing from its surroundings to begin with.


In [ ]:
# CELL 10 — DOES THE ANSWER MEAN ANYTHING?
# =============================================================================
# Cell 9 FITTED the curves. This cell asks whether the numbers that came out
# mean anything, and it asks two questions that are deliberately independent:
#
#   closure             sum(component areas) / integral(dQ/dV) dV
#                       "do the fitted peaks account for the CURVE?"
#   integral fidelity   integral(dQ/dV) dV / the counter's capacity
#                       "does the curve account for the CELL?"
#
# A dataset can pass one completely and fail the other. LTO scores 0.98-1.00
# on closure and 0.58-0.68 on fidelity: the fits are excellent descriptions of
# a curve that is missing 40% of the charge. Using the counter's capacity as
# closure's denominator would have hidden that inside a fitting number.
#
# Closure is reported as an INTERVAL across baseline degree, because a broad
# pseudo-Voigt and a cubic over a 1.4 V window are nearly the same function.
# Where the interval is wide the split is not decided by the data, and Cell 11
# withholds the capacity SHARE for that cycle.
#
# THIS CELL REFITS. Each sampled half-cycle is fitted once per baseline degree
# — three fits each — so it is the most expensive cell in the run for its
# size. CLOSURE_SAMPLE bounds how many are measured; None measures every
# half-cycle, which on a 200-cycle cell is 600 fits and takes longer than the
# fitting in Cell 9 did.
closure_intervals, fidelity, unattributed = {}, {}, {}
# One branch, not two. The same duplicated `if not X:` / `if X:` pair that
# printed two contradictory skip messages in Cell 9 was still here.
if RUN_PEAK_FITTING:
    print(rule("DOES THE ANSWER MEAN ANYTHING?"))
    for name in datasets:
        jobs = fit_jobs[name]
        keys = [j[3] for j in jobs]
        if CLOSURE_SAMPLE:
            sample = set(sample_half_cycles(keys, n=CLOSURE_SAMPLE))
            batch = [j for j in jobs if j[3] in sample]
        else:
            batch = jobs
        print("\n" + heading(_get_display_name(
            name, user_parameters.get(name, {}), user_parameters)))
        # How many fits this actually is depends on whether there is a
        # baseline to sweep. With none there is one fit per half-cycle and
        # saying "x 3 baseline degrees" would be a lie about the cost.
        _ndeg = len(degrees_for(batch[0][2])) if batch else 1
        print(entry("closure sample", f"{len(batch)} of {len(jobs)}",
                    (f"half-cycles × {_ndeg} baseline degrees = "
                     f"{len(batch) * _ndeg} fits") if _ndeg > 1 else
                    (f"half-cycles, one fit each — no free background to "
                     f"sweep, so the split has nothing to depend on")))
        results = assess_closure(batch, verbose=True)
        closure_intervals[name] = {r["key"]: (r["lo"], r["hi"])
                                   for r in results}
        # From 1.9.0.56 there is no free background, so the interval above is
        # zero by construction and cannot withhold anything. The charge the
        # model could not NAME is what carries that duty now — see
        # `quality.UNATTRIBUTED_QUALIFIED_ABOVE`.
        unattributed[name] = {r["key"]: r.get("unattributed", float("nan"))
                              for r in results}
        print(describe_partition(results))

        f = {}
        for (cyc, step), s in signals[name].items():
            if s.voltage.size:
                f[(cyc, step)] = integral_fidelity(s.voltage, s.dqdv,
                                                      s.capacity)
        fidelity[name] = f
        print(describe_fidelity(f))
    # THE ONE PLACE THE POOL IS RELEASED. This is the last cell that fits
    # anything, so from here the workers are pure cost: each holds its whole
    # import footprint — measured at ~181 MB on Linux, more on Windows — for
    # the life of the kernel, through every figure, every export, and however
    # long the notebook then stays open. Releasing here hands that back and
    # costs nothing, because nothing needs a worker again.
    release_workers()
    print("\n" + rule())
else:
    print(rule("QUALITY ASSESSMENT SKIPPED"))
    print(verdict("caution", "peak fitting is off, so there is no fit to "
                             "assess"))
    print(rule())


## Tracking, polarisation and capacity attribution

**Tracking** matches each cycle's fitted components to the reference peak list
by voltage proximity, one-to-one, by optimal assignment — primaries matched
before shoulders, so that when a primary and its shoulder merge into one
component it is the shoulder recorded as disappeared and the primary's centre
history stays continuous. Drift and area retention are measured from the
**reference cycle**, not cycle 1, so a peak that behaves differently during
formation is not given an artefactual retention.

A peak whose reference-cycle area is already near zero is a **ghost** — the
fitter kept a collapsed component — and retention is not reported for it at
all, rather than reported as a ratio of two meaningless numbers.

A peak that vanishes mid-life is a **discontinuity**. It counts as a *failure*
only if the peak was established first (reliably tracked for at least 15
cycles); a feature present only for the first few cycles and then gone was
never established, and its loss is reported as a transient.

**ΔV** pairs charge and discharge peaks by globally optimal assignment with an
expected-sign constraint that follows the electrode. Polarisation growth is
sigmoidal, not linear, so the trend is reported as a net change over the
leading coherent window rather than a fitted slope.

**Capacity attribution** gives each peak's area as a fraction of a fixed basis
— the summed fitted area at the reference cycle. A per-cycle denominator
shrinks as peaks drop out, which redistributes lost capacity onto the
survivors; a fixed one lets the stack fall below 100%, which is what actually
happened.

> **Attribution is withheld where the split is not determined.** Where the
> closure interval from the previous cell is wider than 0.15, the per-peak
> *fraction* is set to NaN and the reason recorded. The fitted areas are kept:
> withholding a share is not deleting a number. On NMC111 this suppresses the
> whole dataset, which is the correct answer; on real data it suppresses
> nothing.

**The coherence audit** asks whether each peak moves like a redox feature. At
C/10 a centre should drift a few mV per cycle and hold its area. The verdict is
driven by the median drift, not by the tail fractions — the tails are a noisy
statistic on a few dozen pairs and the median is not. Below eight consecutive
pairs it declines to judge and says so.


In [ ]:
# CELL 11 — TRACK, PAIR, ATTRIBUTE
# =============================================================================
parameters, tracking, deltav, attribution, coherence = {}, {}, {}, {}, {}
if RUN_PEAK_FITTING:
    print(rule("TRACK, PAIR, ATTRIBUTE"))
    for name, ds in datasets.items():
        print("\n" + heading(_get_display_name(
            name, user_parameters.get(name, {}), user_parameters)))
        det = detections[name]
        # A dataset skipped as UNRESOLVED has no fits, and everything below
        # this point describes fits. It is not an error and it is not a gap
        # in the output: the verdict, its two measurements and its reason
        # were printed in Cell 9, and the peak POSITIONS and the dQ/dV
        # envelope — the quantities such a curve can actually support — are
        # already in the run.
        if not fit_results.get(name):
            print(verdict("caution", "not decomposed — the reference cycle "
                                     "says the peak/background split is not "
                                     "decided by the data"))
            print(bullet("Positions and the dQ/dV envelope stand. There is "
                         "no per-peak area to track, pair or attribute, so "
                         "this section is empty by decision rather than by "
                         "failure."))
            parameters[name] = pd.DataFrame()
            continue
        # THE INTEGRITY VERDICT REACHES THE FIT. `cell_integrity` banded
        # these half-cycles two cells ago and nothing asked; see the note in
        # `parameters_frame`. A half-cycle that passed more charge than the
        # material can hold still gets fitted, tracked and drawn — it is data,
        # and the figure is where you see what happened — but its components
        # are marked and the trends leave them out.
        P = parameters_frame(fit_results[name], det, name=name,
                                integrity=integrity.get(name))
        parameters[name] = P
        # 80 mV is a judgement, not a constant: it varies with peak spacing,
        # and a crowded layered-oxide cathode needs it narrower.
        _an = analysis_overrides(user_parameters.get(name, {}))
        # verbose=False, then FLAG, then print: the summary carries the
        # fidelity and charge-balance warnings, and printing it before they
        # are set would show the numbers without the reasons to doubt them.
        tracking[name] = track_peaks(
            P, det, verbose=False,
            **({"tolerance_mV": _an["tracking_tolerance_mV"]}
               if "tracking_tolerance_mV" in _an else {}))
        # IS AN AREA A CAPACITY HERE AT ALL? A fitted peak area is a share
        # of the charge that appears in the dQ/dV CURVE, and integral
        # fidelity — measured in the previous cell — is exactly how much of
        # the cell's charge that curve accounts for. On a flat two-phase
        # plateau the true dV between records falls below the instrument's
        # voltage resolution and most of the charge is delivered in an
        # interval the derivative cannot resolve; on the LTO triplicate the
        # CYCLER'S OWN dQ/dV column recovers 43-68% of the CYCLER'S OWN
        # capacity. No model fixes that. Peak POSITIONS and drift are
        # unaffected and remain the honest headline.
        flag_low_fidelity(tracking[name], fidelity.get(name, {}))
        # THE CELL'S OWN CHARGE BALANCE, used as a bound on the fit. A peak
        # cannot grow while the cell it lives in shrinks; where it appears to,
        # the decomposition is wrong, and neither R2 nor the standard errors
        # will say so. Flagged, never deleted.
        # NEEDS CELL 6a. `capacity_retention_pct(None, ...)` returns NaN,
        # and a NaN bound silently stops bounding anything — the same
        # NaN-disables-a-guard hazard closed in Cell 3b. Said out loud
        # rather than left to be inferred from an absent warning.
        _ct6a = all_cycle_tables.get(name)
        if _ct6a is None or _ct6a.empty:
            print(verdict("caution", "no cycle table from Cell 6a — the "
                                     "area-growth guard is not bounded"))
            print(bullet("A fitted peak cannot grow while the cell it lives "
                         "in shrinks. Without Cell 6a's capacity retention "
                         "there is nothing to test that against. Run Cell 6a "
                         "first."))
        flag_area_growth(
            tracking[name],
            capacity_retention_pct(_ct6a, det.reference_cycle))
        print("\n" + tracking[name].summary_text())
        # The reference cycle IS the measured end of formation since
        # 1.9.0.18, so the polarisation trend now starts where the peak-drift
        # trend does instead of at cycle 2.
        deltav[name] = delta_v(tracking[name], is_anode=ds.labels_swapped,
                                  first_cycle=det.reference_cycle,
                                  verbose=True)
        attribution[name] = capacity_attribution(
            tracking[name], closure_intervals=closure_intervals.get(name),
            unattributed=unattributed.get(name), verbose=True)
        coherence[name] = coherence_audit(
            P, det, profile=profiles[name]["class"], verbose=True,
            rate_protocol=user_parameters.get(name, {}).get("rate_protocol"))
    print("\n" + rule())
else:
    # Was: "there is nothing to peak tracking, polarisation." A word missing
    # and the list unfinished.
    print(rule("TRACKING SKIPPED"))
    print(verdict("caution", "peak fitting is off, so there is nothing to "
                             "track"))
    print(bullet("No peak tracking, no polarisation, no capacity "
                 "attribution and no coherence audit. Cells 6a-6c are "
                 "unaffected."))
    print(rule())


## What the fitting found

Four figures, and the only ones in the run that show the fitted model rather
than the data. Each prints a suggested caption.

1. **Peak fits** — the model against the data for a handful of cycles, with
   the components filled and the residual beneath. This is where you decide
   whether to believe anything that follows.
2. **Peak trends** — each tracked peak's centre, area and width against cycle
   number, one figure per step. Filled markers passed the reliability test;
   open ones did not.
3. **Polarisation** — ΔV between the paired charge and discharge peaks.
   A widening |ΔV| is rising internal resistance.
4. **Capacity attribution** — how much of the reference cycle's capacity each
   peak still accounts for, as areas and as a stack.

Every band is ±1 standard error on the fitted parameter, straight from the
least-squares fit. Where a band is wider than the trend it draws, the fit is
telling you that number is not determined by your data — which is the most
useful thing a fit can say.

**Nothing here runs unless peak fitting is on.** If Cell 9 said SKIPPED, so
will this.


In [ ]:
# CELL 12 — FIT, TRACKING AND DESCRIPTOR FIGURES
# =============================================================================
# The model, not the data: fits, tracked trends, polarisation, attribution.
# Every one is written to RUN_DIR and prints a caption you can paste.
if RUN_PEAK_FITTING:
    print(rule("FIT, TRACKING AND DESCRIPTOR FIGURES"))
    _n_fig = 0
    for name in datasets:
        print("\n" + heading(_get_display_name(
            name, user_parameters.get(name, {}), user_parameters)))
        # A DATASET THAT WAS NOT DECOMPOSED IS NOT A REASON TO STOP.
        # Cell 11's skip path sets `parameters[name]` to an empty frame and
        # `continue`s WITHOUT setting `tracking`, `deltav`, `attribution` or
        # `coherence` — correctly, there is nothing to put in them. This loop
        # then did `tracking[name]` and raised KeyError, which is not caught
        # anywhere, so ONE undecomposed dataset destroyed every remaining
        # dataset's fit, trend, delta-V and attribution figures, and Cell 13's
        # export with them. The trigger needs no switch: any dataset whose
        # every half-cycle falls past the fit limit reaches it. A dataset with
        # nothing to draw is skipped; the others are drawn.
        if name not in tracking or not fit_results.get(name):
            print(verdict("caution", "no fitted model for this dataset, so "
                                     "there are no fit or descriptor figures "
                                     "to draw"))
            print(bullet("Cell 11 said why. The cycling figures, the dQ/dV "
                         "figures and the detected-peak positions for this "
                         "dataset are unaffected and are already in the run."))
            continue
        FR = as_fit_results(fit_results[name], signals[name],
                               parameters[name], name=name)
        TR = as_tracked(tracking[name])
        plot_fit_quality(processed_dqdv, detected_peaks, FR,
                            user_parameters, save_location=RUN_DIR)
        plot_tracked_trends(TR, user_parameters, save_location=RUN_DIR)
        if name in deltav:
            dv = deltav[name]
            # `pairs` and `delta_v_windows` are packed here and read by
            # nothing: `plot_delta_v` takes `delta_v_df` and
            # `delta_v_changes`. Passing a value that has no reader invites
            # the belief that the figure used it.
            plot_delta_v({name: dict(delta_v_df=dv.df,
                                        delta_v_changes=dv.changes)},
                            user_parameters, save_location=RUN_DIR)
        if name in attribution:
            plot_capacity_attribution({name: attribution[name]}, TR,
                                         user_parameters,
                                         save_location=RUN_DIR)
        _n_fig += 1
    print("\n" + rule())
    print(entry("figures drawn", f"{_n_fig} dataset(s)",
                "see 5_peak_fitting and 6_descriptors"))
else:
    print(rule("FIT FIGURES SKIPPED"))
    print(verdict("caution", "peak fitting is off, so there is no fitted "
                             "model to draw"))
    print(rule())


## Export, and closing the run

Every run writes to its own timestamped folder, and the manifest records what
produced it: the input file checksums, the package versions, the parameters for
each dataset, and every switch in force. With the version number deliberately
held across a series of behaviour changes, those switches are the only thing
distinguishing one run from another.

`finalise_run` sorts the outputs into numbered folders per dataset, checksums
everything written, and closes the manifest. A run that has not been finalised
has no record of what produced it.

```
   OUTPUT_BASE\runs\2026-08-31_143012_v1.9.0\
       run_manifest.json
       <composition>_cell<id>\
           1_cycling\   2_voltage_profiles\   3_energy_power\
           4_dqdv\      5_peak_fitting\       6_descriptors\   7_summary\
           dataset_info.txt
```


In [ ]:
# CELL 13 — EXPORT
# =============================================================================
# The fitted model, as CSVs, one set per dataset. The cycling tables and the
# dQ/dV curves were already written by Cells 6a and 7; this is everything that
# only exists because the peaks were fitted.
#
# `closure_interval_width` is NaN wherever the closure interval was NOT
# MEASURED — Cell 10 samples CLOSURE_SAMPLE half-cycles rather than all of
# them, so on a 200-cycle cell that is 40 rows measured and 400 blank. A blank
# there means "not sampled", not "the fit failed", and the column beside it
# now says which.
if RUN_PEAK_FITTING:
    print(rule("EXPORT"))
    _written = 0
    for name in datasets:
        # SAME GUARD AS THE FIGURE CELL, SAME REASON. An undecomposed dataset
        # has an empty `parameters` frame and no entry in `tracking`,
        # `deltav`, `attribution` or `coherence`, and this loop would raise
        # KeyError on the first of them — losing the exports of every dataset
        # after it in the list as well as its own.
        P = parameters.get(name)
        if P is None or P.empty or name not in tracking:
            print(bullet(f"{name}: nothing fitted, so no fitted-parameter, "
                         f"tracking or attribution table is written"))
            continue
        P = P.copy()
        iv = closure_intervals.get(name, {})
        P["closure_interval_width"] = [
            (iv[(c, s)][1] - iv[(c, s)][0]) if (c, s) in iv else np.nan
            for c, s in zip(P["cycle"], P["step"])]
        # Says WHY a width is blank. Without it a reader opening a P3 export
        # sees 400 empty cells in a quality column and has no way to tell an
        # unsampled half-cycle from a failed one.
        P["closure_interval_measured"] = [(c, s) in iv
                                          for c, s in zip(P["cycle"],
                                                          P["step"])]
        P["integral_fidelity"] = [fidelity[name].get((c, s), np.nan)
                                  for c, s in zip(P["cycle"], P["step"])]
        P.to_csv(os.path.join(RUN_DIR, f"{name}_fitted_parameters.csv"),
                 index=False)
        tracking[name].tracked_df.to_csv(
            os.path.join(RUN_DIR, f"{name}_tracked_peaks.csv"), index=False)
        _ts = pd.DataFrame(tracking[name].summary).T
        _ts.index.name = "peak"          # was an unnamed index column
        _ts.to_csv(os.path.join(RUN_DIR,
                                f"{name}_tracked_peaks_summary.csv"))
        if name in deltav and not deltav[name].df.empty:
            deltav[name].df.to_csv(
                os.path.join(RUN_DIR, f"{name}_delta_v.csv"), index=False)
        for step, A in (attribution.get(name) or {}).items():
            A.to_csv(os.path.join(
                RUN_DIR, f"{name}_capacity_attribution_{step}.csv"), index=False)
        if name in coherence:
            coherence[name].to_csv(
                os.path.join(RUN_DIR, f"{name}_fit_coherence.csv"), index=False)
        _written += 1
    _n_meas = sum(len(closure_intervals.get(n, {})) for n in datasets)
    _n_rows = sum(len(parameters[n]) for n in datasets)
    print(entry("exported", f"{_written} dataset(s)",
                "fitted parameters, tracking, polarisation, attribution "
                "and coherence"))
    if _n_rows:
        print(entry("closure measured on", f"{_n_meas} of {_n_rows}",
                    "half-cycle rows; the rest are marked not measured"))
    print(rule())
else:
    # Was outside the `if`, so a run with fitting off announced "Written to
    # <run folder>" having written nothing at all.
    print(rule("EXPORT SKIPPED"))
    print(verdict("caution", "peak fitting is off, so there is no fitted "
                             "model to export"))
    print(bullet("The cycling tables from Cell 6a and the dQ/dV curves from "
                 "Cell 7 are already written and are unaffected."))
    print(rule())


In [ ]:
# CELL 14 — CLOSE THE RUN
# =============================================================================
# Sorts the outputs into numbered folders, checksums everything written, and
# closes the manifest. Nothing downstream depends on this — but a run that has
# not been finalised has no record of what produced it.
# WHAT THE PIPELINE DECIDED, per dataset, written into each cell's
# dataset_info.txt beside what the operator asked for. Until 1.9.0.11 that
# file recorded only the request — the mass, the window, the theoretical
# capacity — and none of the measured choices that actually determined the
# result: which profile class was read off the curve, which reference cycle
# was chosen and why, what peak shape was calibrated, and how much of the
# cell's charge the dQ/dV curve accounted for. Same rule as the manifest:
# record what ran, never what was offered.
_decided = {}
for name in datasets:
    d = {}
    if name in profiles:
        d["dQ/dV profile class"] = profiles[name]["class"]
        d["Profile class chosen because"] = profiles[name]["description"]
    if name in preprocess_params:
        _pp = preprocess_params[name]
        d["dQ/dV computed by"] = _pp.get("dqdv_method")
        if _pp.get("dqdv_method") == "histogram":
            # The width ASKED FOR and the widths actually USED. They differ
            # on 354 of P3 cell A's 364 half-cycles, and recording only the
            # request described 3% of the data.
            d["Histogram bin requested (mV)"] = _pp.get("histogram_bin_mV")
            _w = [sg_.bin_width_mV for sg_ in signals.get(name, {}).values()
                  if sg_.voltage.size and np.isfinite(sg_.bin_width_mV)]
            if _w:
                d["Histogram bin used (mV)"] = (
                    f"{min(_w):g}" if max(_w) - min(_w) < 1e-9 else
                    f"{np.median(_w):.3g} median, {min(_w):g}-{max(_w):.3g} "
                    f"(widened on {sum(1 for x in _w if x > min(_w) + 1e-9)} "
                    f"of {len(_w)} half-cycles)")
            _e = sum(sg_.n_empty_bins for sg_ in signals.get(name, {}).values())
            _b = sum(sg_.voltage.size for sg_ in signals.get(name, {}).values())
            if _b:
                d["Bins with no charge (%)"] = round(100.0 * _e / _b, 1)
            # CHARGE THE ANALYSIS WINDOW LEFT OUT. Computed per half-cycle
            # since the histogram rewrite and never recorded anywhere. It is
            # the operator-actionable half of a low integral fidelity: charge
            # that fell outside the voltage range chosen for analysis, as
            # opposed to charge a dV integral cannot represent at all.
            # THE REACHABLE-VOLTAGE ENVELOPE. How far the cell actually got,
            # per half-cycle, per step. The dQ/dV heatmap draws it; nothing
            # said it. On LTO cell A the charge step's furthest voltage falls
            # 622 mV over ten cycles while capacity retention is 99% — a
            # window collapsing under an unchanged capacity, which is a
            # polarisation statement and not a capacity one, and it was
            # visible in no number anywhere in the run.
            # THE SAME HALF-CYCLES THE FIGURE BINS, or the sentence and the
            # line are two measurements of one quantity over two populations —
            # which is the fault this whole audit was about. `plot_dqdv_heatmap`
            # drops half-cycles whose voltage span is a small fraction of the
            # median (`_detect_short_halfcycles`), so the partial final
            # half-cycle is not in the figure. Measured over ALL half-cycles,
            # LTO cell B's delithiation envelope reads 914 mV — and every
            # millivolt of that comes from cycle 11, which the export caught
            # mid-flight and which the page elsewhere says is not a measurement
            # yet. The figure correctly drew no line; the prose announced a
            # collapse that had not happened.
            _short = {}
            _pd = (processed_dqdv.get(name) or {}).get("data") or {}
            for _st in ("Charge", "Discharge"):
                try:
                    _short[_st] = set(_detect_short_halfcycles(_pd, _st)[1])
                except Exception:
                    _short[_st] = set()
            _env = {}
            for _st in ("Charge", "Discharge"):
                # SORTED BY CYCLE. `hi_first`/`hi_last` are the ends of a
                # trend, so they cannot come out of a dict in insertion order.
                _ks = sorted((k for k in signals.get(name, {})
                              if k[1] == _st
                              and np.size(signals[name][k].voltage)
                              and k[0] not in _short.get(_st, ())
                              and k not in not_measurements.get(name, ())),
                             key=lambda k: k[0])
                _hi = [float(np.nanmax(signals[name][k].voltage)) for k in _ks]
                _lo = [float(np.nanmin(signals[name][k].voltage)) for k in _ks]
                if len(_hi) >= 3:
                    # ONE GATE, NOT TWO. The figure decides whether the
                    # envelope is worth drawing from the bin width and the
                    # panel span; the sentence had its own absolute
                    # millivolt threshold, and on LTO cell C's delithiation
                    # (66 mV, against a 62 mV panel gate and a 100 mV prose
                    # gate) the line was drawn and the page said nothing
                    # about it. Two thresholds for one quantity is how a page
                    # ends up contradicting its own figure. The decision is
                    # made HERE, once, with the plotting module's own
                    # constants, and `drawn` is what the report reads — so
                    # the sentence is literally a description of the line.
                    _spread_V = max(max(_hi) - min(_hi), max(_lo) - min(_lo))
                    _panel_V = max(_hi) - min(_lo)
                    _bws = [s.bin_width_mV for k, s in signals[name].items()
                            if k[1] == _st and np.isfinite(s.bin_width_mV)]
                    _bw_V = (float(np.median(_bws)) / 1000.0) if _bws else 0.0
                    _drawn = bool(
                        _spread_V > HEATMAP_ENVELOPE_MIN_BINS * _bw_V
                        and _panel_V > 0
                        and _spread_V / _panel_V
                        > HEATMAP_ENVELOPE_MIN_FRACTION)
                    _env[_st] = dict(
                        hi_spread_mV=1000.0 * (max(_hi) - min(_hi)),
                        lo_spread_mV=1000.0 * (max(_lo) - min(_lo)),
                        hi_first=_hi[0], hi_last=_hi[-1],
                        lo_first=_lo[0], lo_last=_lo[-1], n=len(_hi),
                        drawn=_drawn)
            if _env:
                user_parameters.setdefault(name, {})["voltage_envelope"] = _env
            _ow = [float(sg_.charge_outside_window)
                   for sg_ in signals.get(name, {}).values()
                   if np.isfinite(getattr(sg_, "charge_outside_window",
                                          np.nan))]
            if _ow:
                d["Charge outside the analysed window (mAh/g)"] = (
                    f"{np.median(_ow):.2f} median per half-cycle, "
                    f"worst {max(_ow):.2f}")
        else:
            d["Smoothing window (points)"] = _pp.get("smoothing_window")
            d["Rebin width (mV)"] = _pp.get("rebin_width_mV")
            d["Spike removal"] = _pp.get("spike_removal")
    if reconstruction.get(name):
        _r = reconstruction[name]
        # ROUNDED, like the RMSE beside it. The histogram path reconstructs
        # to floating-point exactness, and the raw value wrote itself into
        # the record as "1.063e-12" — which reads as a measurement of
        # something, precise to thirteen figures, rather than as zero.
        _ape = float(np.nanmedian([x["ape_pct"] for x in _r]))
        d["Reconstruction APE (%)"] = (
            "< 0.001" if _ape < 1e-3 else round(_ape, 3))
        d["Reconstruction RMSE (mV)"] = round(
            1000.0 * float(np.nanmedian([x["rmse_V"] for x in _r])), 2)
    if name in windows:
        d["Analysed window (V)"] = (f"{windows[name][0]:.4f} - "
                                    f"{windows[name][1]:.4f}")
    for _st, _tv in sorted((trims.get(name) or {}).items()):
        if max(_tv) > 0:
            d[f"Cut-off trim, {_st.lower()} (mV)"] = (
                f"{1000 * _tv[0]:.0f} off the bottom, "
                f"{1000 * _tv[1]:.0f} off the top")
    if in_progress.get(name):
        d["Caught mid-flight"] = ", ".join(f"cycle {c} {st}"
                                           for c, st in in_progress[name])
    if not early_stops.get(name, pd.DataFrame()).empty:
        d["Ended early, not on voltage"] = ", ".join(
            f"cycle {int(r.cycle)} {r.step} ({r.short_mV:.0f} mV short, "
            f"{r.termination_class})"
            for r in early_stops[name].itertuples())
    if fit_limit.get(name) is not None:
        d["Fitting stopped at cycle"] = fit_limit[name]
    if name in detections:
        d["Reference cycle"] = detections[name].reference_cycle
        d["Reference cycle chosen by"] = detections[name].reference_reason
    if RUN_PEAK_FITTING and name in shape:
        _sh = shape[name]
        d["Mixing fraction eta"] = (
            f"{_sh['fraction']:.3f} (held for every fit)"
            if _sh.get("fraction") is not None else
            f"refined per half-cycle; median {_sh['fraction_median']:.3f}, "
            f"10-90 spread {_sh['fraction_spread']:.3f} too wide to hold"
            if _sh.get("fraction_median") is not None else "not measurable")
        if _sh.get("sigma_ratio") is not None:
            d["Shoulder width ratio k"] = f"{_sh['sigma_ratio']:.3f} (held)"
    for _step, _f in sorted((tracking[name].integral_fidelity
                             if RUN_PEAK_FITTING and name in tracking
                             else {}).items()):
        d[f"Integral fidelity ({_step})"] = (
            f"{_f:.2f}" + ("  *** areas are not capacities ***"
                           if _f < INTEGRAL_FIDELITY_FLOOR else ""))
    _decided[name] = d

# `detection=` so the reference cycle each dataset was measured against
# reaches the manifest. It was read from globals() and was always empty.
finalise_run(RUN_DIR, MANIFEST, user_parameters=user_parameters,
                verbose=True, decided=_decided,
                detection=detections if "detections" in dir() else None)


## START HERE — the way in

A finished run is forty-six files across seven folders. That is the right way
to store the output and the wrong way to hand it to somebody.

This cell writes two more files at the top of each dataset's folder, above the
numbered directories:

| file | for |
|---|---|
| `START_HERE.html` | reading. Images are embedded, so it can be moved or emailed on its own |
| `START_HERE.md` | pasting into a report, a wiki or a supervision record |
| `RUN_SUMMARY.md` | all the cells side by side, and whether they agree |

Both say the same four things, in this order: **what the cell did**, **what
went wrong and on which cycles**, **what Ratatosk declined to tell you and
why**, and **four figures with what to look for in each**.

The third of those is the one that matters most for a newcomer. A capacity
share left blank because the closure interval was 0.46 wide is a *finding*.
Found as a gap in a spreadsheet three weeks later, it is a bug report.

`RUN_SUMMARY.md` is the triplicate view. It reports the **spread** first, names
an outlier only when there is one, and **declines to nominate a representative
cell when the cells disagree** — picking the one nearest the median would be
choosing a number rather than measuring one.

Nothing is moved or removed — this is a way in, not a replacement.


In [ ]:
# CELL 15 — START HERE, AND THE RUN SUMMARY
# =============================================================================
# Written after finalise_run has sorted the outputs, so the figures are in
# their folders and can be found and embedded.
#
#   <cell folder>/START_HERE.html   one cell, four figures, what to look for
#   <cell folder>/START_HERE.md     the same, to paste into a report
#   RUN_SUMMARY.md                  all the cells side by side, and whether
#                                   they agree
# Resolved for ALL datasets at once: two cells sharing a composition and a
# cell id get X and X_2 from `organise_run`, and asking one at a time gave
# both reports the name X, so the second overwrote the first.
print(rule("START HERE"))
_folders = dataset_folders(user_parameters, list(datasets))
_report_facts = {}
for name in datasets:
    _folder = os.path.join(RUN_DIR, _folders[name])
    if not os.path.isdir(_folder):
        _folder = RUN_DIR
    _ct = None
    if isinstance(all_cycle_tables, dict):
        _ct = all_cycle_tables.get(name)
    elif isinstance(all_cycle_tables, pd.DataFrame):
        _ct = all_cycle_tables
    _h, _m, _f = write_report(
        _folder, name,
        params=user_parameters.get(name, {}),
        integrity=integrity.get(name),
        detection=detections.get(name),
        tracking=tracking.get(name) if RUN_PEAK_FITTING else None,
        attribution=attribution.get(name) if RUN_PEAK_FITTING else None,
        coherence=coherence.get(name) if RUN_PEAK_FITTING else None,
        cycle_table=_ct,
        fit_limit=fit_limit.get(name),
        profile=profiles.get(name),
        # THE MECHANISM VERDICT REACHES THE PAGE. Cell 9 computes it, prints
        # it, and chooses the model from it; until now it went no further, so
        # START_HERE and RUN_SUMMARY described a fit without saying which
        # model had been fitted.
        resolvability=resolvability.get(name) if RUN_PEAK_FITTING else None,
        # The fit-quality numbers live on the parameter table, one row
        # per component; the page reads the per-half-cycle medians.
        parameters=parameters.get(name) if RUN_PEAK_FITTING else None,
        file_format=user_parameters.get(name, {}).get("file_format", "png"),
        in_progress=not_measurements.get(name),
        run_id=os.path.basename(RUN_DIR))
    _report_facts[name] = _f
    print(entry(_get_display_name(name, user_parameters.get(name, {}),
                                     user_parameters),
                os.path.relpath(_h, RUN_DIR),
                os.path.relpath(_m, RUN_DIR), value_width=26))

_summary = write_run_summary(RUN_DIR, _report_facts,
                             run_id=os.path.basename(RUN_DIR),
                             n_files=len(_all))
print(entry("run summary", os.path.relpath(_summary, RUN_DIR),
            "all the cells side by side", value_width=26))
# The summary is PRINTED as well as written. It is the one thing a reader
# should not have to open a file to see, and the file is there for when they
# want to send it.
print("\n" + rule("RUN SUMMARY"))
print(open(_summary, encoding="utf-8").read())


---

# Appendix — the science, and what this pipeline can and cannot tell you

*Everything below is reference material. It is at the END on purpose: Jupyter
cannot collapse all markdown at once, and in 1.8.x the explanations were
interleaved with the code and made the notebook unnavigable. Nothing here is
executed and nothing above depends on it.*

*Each section is self-contained and titled, so it can be lifted directly into
a help file or a tooltip.*

---

## A1. What a dQ/dV curve is

A galvanostatic cycle gives you charge **Q** and voltage **V**. Differentiating
one against the other, **dQ/dV**, converts a voltage plateau into a peak — and
plateaus are where the interesting electrochemistry is, because a plateau means
the potential is being *held* by something.

At constant current there is an identity worth keeping in mind, because most of
the artefacts in this pipeline come from it:

```
    dQ/dV  =  I / (dV/dt)          so at constant current,
    sign(dQ/dV) = sign(dV/dt)
```

The derivative is therefore only as good as the voltage record. Where the cell
sits on a flat plateau, dV between consecutive records is small; where it is
smaller than the instrument's voltage resolution, dQ/dV is not a measurement of
anything. This is the single largest limitation of the technique and §A4
returns to it.

**Three properties of a peak, and they are not equally trustworthy:**

| Property | What it means | How much to trust it |
|---|---|---|
| **Position** (V) | The potential of a redox process. Shifts as polarisation grows. | **Robust.** Does not involve the integral at all. |
| **Area** | Charge associated with that process — *if* the curve accounts for the charge. | **Conditional.** See §A4 (does the curve account for the cell?) and §A5 (do the peaks account for the curve?). |
| **Width** (FWHM) | Homogeneity and kinetics of the process; a broader peak is a less well-defined transition or a slower one. | Moderate. Correlated with amplitude in any fit. |

### How the curve is computed, and why that is not a detail

There is **no widely agreed method** for computing an IC curve — one review
found nineteen studies with nineteen different workflows. Two families exist:

**Numerical differentiation** (the conventional method; Ratatosk offers it,
but it is *not* the default). Finite differences on the voltage array, with
Savitzky–Golay smoothing before differentiation. Because ΔV sits in the
denominator, flat plateaus give very large IC values, so smoothing is
essential — and its parameters are usually chosen by trial and error until the
curve *looks* right. That makes curves hard to compare between studies, and
the smoothing itself introduces drifts, oscillations and boundary overshoots.
Ratatosk at least chooses the parameters from a measured profile class rather
than by eye, and records every one of them; that does not escape the
underlying fragility, which is why it is not the path taken.

**Voltage histograms** (derivative-free — **what Ratatosk does**,
`DQDV_COMPUTATION = "histogram"` in Cell 2). Bin the records by voltage and
**sum the charge that passed inside each bin**, then divide by the bin width.
That is the IC curve, and nothing is ever differentiated. One parameter, the
bin size, which means exactly what it says: the voltage resolution of the
result.

A note on that, because the textbook statement of the method is weaker than
the implementation. The usual formulation *counts* the records that land in
each bin, which is proportional to the charge only if capacity is sampled at
uniform intervals — a condition real cycler files do not always meet.
Ratatosk passes the charge itself as the histogram weight
(`np.histogram(..., weights=dq)`), so the bin value is the charge that passed
there whether or not the sampling was uniform, and the assumption is not
needed. §A4 measures what this buys.

**The objective test between them is voltage reconstruction.** A derivative is
reversible, so the cumulative integral of the IC curve, plotted against V,
should give back the original voltage curve. Two metrics quantify how well:
the absolute percentage error in maximum cumulative capacity (APE) and the
voltage RMSE over the overlapping interval. Many parameter choices give a
visually satisfying curve; few reconstruct the voltage. Ratatosk's
`integral_fidelity` is the scalar special case of the capacity half of this
test — see §A4, where it matters a great deal.

**Broad features, or none at all, are not a failure.** A material undergoing a
continuous solid solution reaction has no distinct phase transition to hold the
potential, so it gives broad humps. Overlapping reactions merge. High rates
flatten and broaden everything. Coarse voltage sampling hides fine structure. A
featureless dQ/dV curve is a result about the material, not a problem with the
measurement.

> **Tuning.** The bin width and smoothing half-width are the two knobs on this
> section, both set from the measured profile class. If your material is
> broader or sharper than the classifier decided, correct the class first — it
> moves several parameters coherently. §A11 has the table, and the entries for
> *"curve is sparse"*, *"fine structure smeared"* and *"noisy and spiky"*.


## A2. The central idea: coherence, not magnitude

This is the part worth slowing down for, and it is why the pipeline tracks
peaks at all rather than just reporting capacity.

There are two things a redox feature can do as a cell ages.

**Coherent evolution.** The peak drifts gradually, loses area slowly, broadens
steadily, and stays trackable throughout. This is ordinary ageing: the reaction
is still happening, with growing polarisation and slowly declining accessible
capacity. Large changes are not alarming on their own, provided they are
smooth.

**Discontinuity.** The peak jumps, swaps identity with a neighbour, collapses,
or disappears and does not return. This is instability or failure — the process
is not evolving, it is breaking.

**Capacity retention cannot tell these apart.** Two cells can fade to the same
capacity over the same number of cycles, one by coherent ageing and one by
failure, and their capacity curves will look alike. What separates them is
whether the descriptors stayed continuous.

That is the claim the whole pipeline exists to support, and it is the reason
`discontinuity_is_failure` distinguishes a peak that broke *after* being
established from one that was never established at all. Only the first is a
finding about the cell.

> **Tuning.** What counts as a discontinuity is `DISCONTINUITY_STEP_FLOOR_MV`
> (80 mV) and `DISCONTINUITY_STEP_K` (4× the peak's own step scatter),
> whichever is larger, and `DISCONTINUITY_ESTABLISH_MIN` (15 cycles) decides
> whether a peak was ever established. On runs much shorter than 15 cycles no
> peak can be "established", so every break is reported as transient — lower
> it deliberately for short runs rather than being puzzled by the silence.

---

## A3. Polarisation (ΔV) — the most reliable degradation descriptor here

A redox couple appears twice: once on charge at a higher voltage, once on
discharge at a lower one. The gap **ΔV** is the polarisation — the extra
voltage needed over the thermodynamic potential to actually drive the reaction.
It is set by kinetics: charge-transfer resistance at the interface, ohmic drop
through electrode and electrolyte, and how fast ions reach the reaction front.

A gradual, continuous rise in ΔV is the ordinary signature of ageing: internal
resistance building, the interface becoming sluggish, the diffusion path
degrading. Both the value and its **rate of change** are worth reporting — a
couple that starts more polarised and one that polarises faster are telling you
different things.

Pairing is done as a single optimal assignment across all peaks at once, and it
enforces one physical rule: **ΔV must be positive**, because a couple cannot
discharge above the voltage at which it charged. A negative ΔV is not a
measurement, it is a mis-pairing, and the assignment refuses it.

Because ΔV is a difference of two *positions*, it inherits none of the
integral's problems. On a material where the areas cannot be trusted at all,
ΔV and drift are still measurements.

> **Tuning.** `PAIR_MAX_SEPARATION_MV` (200 mV) is how far apart a charge and
> discharge peak may be and still be called the same couple. Raise it for a
> heavily polarised system — a high-rate or low-temperature cell can exceed
> 200 mV honestly — and lower it if unrelated peaks are being paired. Also
> `EXCLUDE_FORMATION_FROM_DELTAV` (True): the trend starts at the measured end
> of formation, because including formation absorbs a step change into what is
> reported as a rate.


## A4. When a peak area is NOT a capacity

This is the most important limitation in the pipeline and the least obvious.

A fitted peak's area is a share of the area under the **dQ/dV curve**. Whether
that is a share of the cell's **charge** depends on whether the curve accounts
for the charge, and that is a separate, measurable quantity:

```
    integral fidelity  =  ∫(dQ/dV) dV  /  the counter's delivered capacity
```

On a first-order two-phase material this can fail badly, for a reason that is
physics rather than software. A two-phase transition holds the potential
essentially invariant, so nearly all the charge is delivered within a few
millivolts. Where the true dV between consecutive records falls below the
instrument's voltage resolution, that charge is delivered in an interval the
derivative cannot resolve — and it cannot appear in ∫(dQ/dV)dV.

**Measured on a flat two-phase anode (Li₄Ti₅O₁₂):** the *cycler's own* dQ/dV
column, integrated over its *own* voltage axis, recovers only **43–68%** of the
*cycler's own* delivered capacity, and the shortfall swings by ten points from
cycle to cycle. No choice of fitting model recovers it. Fitting that curve
beautifully — R² 0.995–0.998, closure 0.98–1.00 — still gives peak areas that
are shares of the 60% that survived, which is how one cell reported **141% area
retention while holding 99% of its capacity**.

The pipeline therefore gates on fidelity, and the gate is **two-sided**: `INTEGRAL_FIDELITY_FLOOR` (0.80) and `INTEGRAL_FIDELITY_CEILING`
(1.20). Outside that band, area trends are marked untrustworthy and capacity
attribution is withheld. **Peak positions and drift are unaffected and remain
reportable.**

The ceiling closes a gap that a one-sided test leaves open.
`integral_fidelity`'s own docstring promises that "fabricated area shows up as
an excess, lost features as a deficit" — and only the deficit was ever tested.
The excess is the more damning of the two, because a deficit is charge the
curve never saw while an excess is charge the *cell never delivered*, and
nothing physical invents it. The 4.199 V constant-voltage artefact this
project spent time chasing is an excess, and a floor-only gate would pass it. The ceiling is set from measurement rather than symmetry:
differencing a voltage record quantised at 0.1–5 mV overshoots to at most
1.047 on the synthetic cases, so anything below about 1.10 would fire on
ordinary data.

### This limit belonged to the METHOD, not to the material — and it is fixed

That distinction matters, and it was settled by measurement. Recomputing the
same half-cycles as **voltage histograms** — the derivative-free path,
now the default (`DQDV_COMPUTATION`) — instead of from the cycler's dQ/dV
column:

| | derivative (cycler column, smoothed) | histogram |
|---|---|---|
| integral fidelity | 0.62 – 0.71 | **0.995 – 1.000** |
| capacity error (APE) | 28 – 38% | **0.001 – 0.05%** |
| voltage RMSE | 234 – 427 mV | **6 – 97 mV** |
| **tracked area retention** | 89% and **141%** | **98% and 100%** |
| peak positions | 1.5253 / 1.5855 V | 1.5230 / 1.5830 V |

The charge was never missing from the *data*. The derivative was losing it,
and a method that never differentiates recovers essentially all of it — at
every bin width from 0.5 to 5 mV. **A cell holding 99% of its capacity reports 100% area retention instead of
141%.** The physical bound check of
§A6 has nothing left to flag on this dataset, because the areas are finally
capacities.

### And on a broad layered oxide, where the problem looked different

A layered oxide never has the two-phase material's catastrophic fidelity — 0.91/0.97 on the
derivative path. It had a different symptom, and the same cure:

| tracked peak area retention, layered oxide, 10 cycles | derivative | histogram |
|---|---|---|
| charge | **152%**, 83%, **170%**, 70%, 54% | 61%, 95%, 98%, 102%, 71%, 50% |
| discharge | 85%, 95%, 105%, 100%, 100% | 43%, 65%, 106%, 89%, 24% |
| median fit R² | 0.971 | **0.991** |

**Two peaks over 150% become none over 106%.** A redox feature taking 70% more
charge at cycle 10 than at cycle 2, in a cell that lost a few per cent, was
never a measurement — it was the derivative's noise being partitioned between
components. The histogram removes it without any change to the fitting.

Note R² moved the *other way* here, 0.971 → 0.991. Between the two chemistries
R² goes up and down while the areas get right both times, which is as clean a
demonstration as this project has produced that **R² tracks how smooth the
curve is, not whether the answer is true.**

**Two pieces of honesty about that table.**

*Integral fidelity means something different on the histogram path.* The curve
is built by adding up the charge each record carried, so within the analysed
window it integrates back to that charge **by arithmetic**. A fidelity of
0.995 is not evidence that the method is right; it is the method's definition.

That does not make the number useless — it changes what it detects. Fidelity
below 1 on the histogram path means charge that is genuinely **not in the
curve**: a constant-voltage hold (charge passed at one voltage, which no dV
integral can represent), or charge outside the analysed voltage window. A
CCCV charge half-cycle sits at 0.95-0.96 for exactly that reason. So read it as
*"how much of the cell's charge this curve is about"*, and read the **voltage
RMSE** for the question fidelity used to stand in for: is the charge at the
right voltages?

*R² gets worse on the two-phase material, and that is fine.* The median fit R² falls from 0.996
to 0.934, because the histogram is a raw binned curve with real bin-to-bin
scatter where the old one had been heavily smoothed. The model now has genuine
noise to sit in. The **areas became correct while R² fell** — which is §A6's
argument, demonstrated by accident. (On a broad profile R² goes the other
way, 0.971 → 0.991, which is the same point from the other side: R² tracks how
smooth the curve is, not whether the answer is right.)

### The two parameters, and why they are not the old problem

The histogram path has a bin width and a smoothing half-width, both chosen
from the measured profile class exactly as the old smoothing parameters were:

| profile | bin | smoothing | why |
|---|---|---|---|
| sharp (two-phase) | 1 mV | 1 bin | FWHM ≈ 15 mV; the peak towers over the noise |
| moderate | 2 mV | 3 bins | |
| broad (layered oxides) | 5 mV | 6 bins | FWHM 60 mV+; features are shallow |

A bin narrower than the records themselves cannot be filled, so the width is
widened automatically where the data are sparse and the width **actually used**
is recorded in `dataset_info.txt`.

It is fair to ask whether this smuggles back the trial-and-error smoothing the
method exists to avoid. It does not, for a reason that is arithmetic rather
than rhetorical: **this smoothing is applied to the result, with a binomial
kernel that sums to one.** It cannot move a record's charge into a different
bin, and it conserves the total area exactly. The old smoothing was applied to
the *voltage* before differentiating, where it changed the derivative and the
integral together and could not be undone. Across a sweep of smoothing widths
from 1 to 8 bins on a layered oxide, integral fidelity moves 0.9403 to 0.9379 — a
quarter of a percent, and that is edge effect from the zero-padded
convolution, not lost charge.

What the smoothing *is* for: peak DETECTION runs on the curve, and a
histogram's bin-to-bin scatter is counting noise. Unsmoothed, detection found
13 peaks per half-cycle on a layered oxide where the derivative path found 5. At 6 bins it
finds 6. That is the whole reason the parameter exists, and it is set by
matching detection to the smoothed path rather than by eye.

### The related artefact: non-monotonic voltage, and what it actually is

On the same flat plateau the recorded voltage can move *backwards* between
records while the cell is still charging. There are **two causes and they are
easily conflated**:

1. **Instrument resolution.** The true voltage change is below the cycler's
   least significant bit, so the recorded value dithers.
2. **Real physics.** Local maxima and minima in the voltage–capacity curve are
   a genuine feature of first-order phase transitions, driven by
   lithium–vacancy interactions in the host. They are not noise.

Either way the consequence for IC is the same and it is severe: the instrument
divides a positive dQ by a negative dV and reports a wrong-sign,
large-magnitude value; sorting by voltage interleaves those with the honest
records; smoothing averages the two. That is the up-and-down pattern on a first
cycle, and on a flat plateau an *inverted* peak. Worse, a ΔV that oscillates
between negative, zero and positive makes the voltage reconstruction of §A1
undefined — which is precisely why fidelity collapses.

`signal.orient_dqdv` corrects the sign of exactly those records, before the
sort, and only where the current has not reversed and charge is still
accumulating. **Be clear about what that is:** it forces the curve to be
sign-definite so the integral is well posed. Where cause (1) dominates it
repairs an instrument artefact; where cause (2) dominates it *discards real
information* — deliberately, because IC assumes a one-to-one relation between
voltage and capacity and simply cannot represent a non-monotonic curve. The
literature's advice is the same in substance: approximate the non-monotonic
region with a smooth monotonic IC, and characterise it separately by
**differential voltage analysis (dV/dQ)**, which does not have this problem. A
high `n_oriented` count is a flat plateau, and a flag that dV/dQ would tell you
something this curve cannot.

---

## A5. Closure, and why attribution is sometimes withheld

Two orthogonal questions, deliberately never combined:

```
    closure            Σ(component areas) / ∫(dQ/dV) dV
                       "do the fitted peaks account for the CURVE?"

    integral fidelity  ∫(dQ/dV) dV / the counter's capacity
                       "does the curve account for the CELL?"
```

A dataset can pass one completely and fail the other, and a flat two-phase
material does exactly that. Using the counter's capacity as closure's denominator would hide a
preprocessing failure inside a fitting number.

**Closure is reported as an interval, not a value.** A pseudo-Voigt of σ ≈ 100
mV and a cubic across a 1.4 V window are nearly the same function. Where that
is true, the split between "peak" and "background" is not determined by the
data, and any single closure value is an artefact of the baseline degree
somebody chose. So the same data is refitted at several baseline degrees and
the **spread** is reported. Where the spread exceeds
`PARTITION_DETERMINED_BELOW`, the per-peak capacity *share* is set to NaN and
the reason recorded — the fitted areas are kept, because withholding a share is
not deleting a number.

> **Tuning.** `PARTITION_DETERMINED_BELOW` (0.15) and `CLOSURE_SAMPLE` (40
> half-cycles). Widening the first to make attribution appear is the one
> change in this appendix most likely to produce a number that cannot be
> defended: the spread *is* the evidence that the split is undetermined. If
> attribution is withheld often, the productive move is usually a lower
> baseline degree or a narrower analysed window, not a looser threshold.


## A6. Reading a fit: what good and bad look like

**Residuals** are the primary diagnostic, exactly as in Rietveld refinement.
Flat and featureless is good; structured residuals mean the model is missing
something real.

**R² is not a check on anything.** It cannot see the failure modes that
actually matter here. On a two-phase anode, refining the mixing fraction per
cycle
carried the fitted peak area from 61 to 85 mAh g⁻¹ — 37% growth on a cell that
had lost 1.5% of its capacity — with R² between 0.995 and 0.998 the whole way.
Every fit was excellent; the trend they formed was fiction. Adding free
parameters always lowers the residual. That is what free parameters do.

**What to look for instead:**

- A component that has inflated to span the whole voltage range — it has become
  the background.
- A component pinned at its σ bound (`sigma_max_fitted`).
- Area retention exceeding the cell's own capacity retention — a redox feature
  cannot take more charge than the cell delivers, so where this appears the
  decomposition is wrong. Flagged, never deleted.
- Two components collapsed onto one feature; the smaller is dropped and the
  half-cycle refitted once, recorded as `dropped_degenerate`.

**Constrain, don't fix, and don't free.** This is the crystallographic
instinct and it applies directly. Nobody refines an independent profile width
for every reflection in a Rietveld fit; widths are tied to a few physically
motivated parameters and released only when the data demand it. Here:

- A **shoulder's width is tied to its parent's** by a bounded ratio,
  σ_shoulder = k·σ_parent with 0.3 ≤ k ≤ 1, one k shared per half-cycle.
- The **mixing fraction η is a property of the measurement**, not of the redox
  process — it is set by voltage resolution, rebin width and smoothing, none of
  which change from cycle to cycle. It is measured once per dataset on a sample
  and then held.
- Both are **measured, not assumed**, and the calibration *declines* to hold
  them when the spread across the sample is too wide. A shape the data says is
  not constant is not asserted to be one.

> **Tuning.** `FIT_SIGMA_MAX_MV` (200 mV) is the width bound worth raising for
> a genuinely broad material; `SHOULDER_SIGMA_RATIO_MIN/MAX` (0.30–1.00) is
> the shoulder tie, and `SIGMA_RATIO_SPREAD_MAX` (0.30) is the spread at which
> the calibration gives up and refuses to hold a shape. If components keep
> pinning at the σ bound, raise the bound *or* reduce the baseline degree —
> a component spanning the window is trying to be the background.

---

## A7. How many peaks should there be? Ask the thermodynamics first

A worked example from this project, because the lesson generalises.

An early cycle of a two-phase anode appeared to show a **doublet** where one
peak was
expected. The evidence offered for it was that the pair survived a change of
bin width, and that R² improved from 0.937 to 0.999 when a second component was
added.

Both arguments are worthless.

- **Bin-width stability is necessary, not sufficient.** A reproducible
  processing artefact is reproducible.
- **An R² gain from 0.937 to 0.999 on one added component is what overfitting
  looks like**, not evidence of a second process.

The test that settled it was the **thermodynamics**. Li₄Ti₅O₁₂ → Li₇Ti₅O₁₂ is a
first-order two-phase transition. By the Gibbs phase rule a two-phase field in
a binary system at fixed T and P is invariant, so the potential is fixed while
both phases coexist — which means **one** flat plateau and therefore **one**
dQ/dV peak. There is no second process available to make a second peak.

The processing-free confirmation was **V(Q)**: Q is a monotonic integral and V
is a directly quantised measurement, so V(Q) shows a second plateau if there is
one. There was none. The "flattest point" sits at 1.546 ± 1 mV across cycles
3–10; the claimed pair appears in cycle 2 only, and not at all in a replicate
cell.

**The rule:** before adding a component, ask what process it would correspond
to. If the phase diagram does not offer one, the component is describing noise
or a baseline, and the fit will happily accommodate it.

### Checked against the literature, including the awkward paper

The claim above was tested against the work that most threatens it — a 2024
study reporting a **three-phase** α/β/γ transition in Li₄Ti₅O₁₂ — and it
survives, for a reason worth stating precisely rather than hedging:

- The third phase (γ, quasi-rock-salt) is reached only on discharge to
  **ultralow potential**. Its plateau sits at **ca. 0.53–0.63 V**, and the
  study's window is 2.0–0.3 V. A cell cycled **1.2–2.5 V never reaches it.**
- Within the ordinary window the α/β transition is exactly as described: a flat
  plateau at ca. 1.55 V on discharge and ca. 1.58 V on charge, with a GITT
  equilibrium potential of **E(α/β) = 1.565 V**.
- What the study *does* add for wide-window work is an asymmetry — discharge
  through α/γ, charge through α/β — plus the observation that the balance
  between the two discharge plateaus is tunable by particle density, current
  and temperature. None of that splits the 1.55 V plateau.

Departures from two-phase behaviour that **are** documented, and their
conditions: **nanosizing** (simultaneous 16c/8a site occupation near the
surface, giving a curved voltage profile and extra low-voltage capacity, much
like nano-LiFePO₄); **very low rate at room temperature**, which can reach an
equilibrium solid solution — note this is the opposite way round from the usual
intuition, since *fast* intercalation kinetically gives the two-phase reaction;
and **temperature**, which moves the system through a solid-solution →
nanodomain → true two-phase coexistence sequence. Sub-nanometre phase
coexistence can even *look* like a solid solution while remaining a first-order
transition.

None of these predicts a **resolved doublet on the main plateau** for
micron-scale powder at C/10 over 1.2–2.5 V, which is the case at hand.

### An independent check on the pipeline's peak positions

Worth recording, because it validates the quantity §A1 calls the robust one.
Across the triplicate, the tracked reference positions are:

```
    charge     1.548, 1.546, 1.550 V        mean 1.5480 V
    discharge  1.579, 1.582, 1.578 V        mean 1.5797 V
    midpoint (the polarisation-free estimate)     1.5638 V
```

The GITT equilibrium potential for the α/β transition is **1.565 V**. The
pipeline's midpoint agrees to **1.2 mV**, from ordinary C/10 galvanostatic
data, with no calibration against it.

*Where the reference cycle sits changes these numbers, and that is itself a
result.* Anchoring on cycle 2 — still inside formation on these cells — gives
1.533 / 1.590 V, a midpoint of 1.562 V and a half-gap of 28 mV. Choosing the
reference by **measured CE stabilisation** moves it out of formation, and the
polarisation drops to ΔV/2 ≈ **16 mV** while the midpoint moves *towards* the
equilibrium potential, from 3 mV out to 1.2 mV. A formation cycle is more
polarised than a settled one, so both movements are in the direction the
physics requires, and neither was tuned for.

16 mV sits well below the ~85 mV reported for a dense single-particle
microelectrode, which is what a porous composite electrode at C/10 should
give: less current per particle and a much shorter effective diffusion path.
**Positions are strong.** That is the whole argument of §A1's table, measured
on a material whose *areas* this same pipeline once refused to report.


## A8. Coulombic efficiency — and a caution the older literature does not give

CE is Q_discharge / Q_charge **for the same cycle**, and it is the most
precision-critical number in the field: distinguishing cells that will survive
500 cycles needs it read at the fourth decimal place, and ultra-high-precision
instruments quote 1 ppm.

Two consequences the pipeline enforces:

- **CE is NaN, never 0, when either half-cycle is absent.** A spurious 0%
  asserts that the cell passed charge and returned none, which is a different
  physical claim from "we could not measure it" — and in a series read at the
  fourth decimal place it is not a small error.
- **CE and capacity fade are not the same quantity.** The conventional
  inference — coulombic inefficiency accumulates into capacity loss — is a
  *systematic overestimation*. Recent work shows the measured coulombic
  inefficiency is far larger than the fractional capacity fade per cycle in
  good cells, so only part of the parasitic charge consumption contributes to
  fade. Report CE as CE. Do not convert it into a predicted lifetime without
  saying you have done so.

Voltage efficiency (VE) decomposes the energy loss alongside it: EE ≈ CE × VE /
100, with CE reflecting charge accounting and VE reflecting polarisation.

---

## A9. Replicates: report mean ± SD, not a representative cell

Published reproducibility practice for replicate cells is **mean ± standard
deviation across the replicates, with n stated**, and the individual cells
shown.

Nominating one cell as "representative" is a *selection step*, and selection
steps are where irreproducibility enters — it is the same move as choosing the
prettiest diffraction pattern.

So:

- The reported number for a group is the **mean ± SD** (sample SD, ddof = 1;
  three cells are a sample, and the population SD understates a triplicate's
  spread by 18%).
- A representative cell is nominated **for figures only**, and labelled as
  such: *"cell B shown; n = 3, retention 98.6 ± 0.4%"*. (The cell letter is the
only thing a caption should name.)
- **The representative cell's number is never the reported number.**
- A single cell shows its value with no ± at all. One measurement has no
  spread, and quoting 0 would claim a precision nothing established.

---

## A10. Known limits — read this before quoting anything

Measured, not speculated. Each was established against synthetic data with a
known answer (`synthetic.make_series`) or against the real triplicates.

**The ±80 mV tracking window is a PER-CYCLE step limit, not a lifetime
allowance.**

If every cycle were matched against the *reference cycle's* fitted positions, a peak drifting 2 mV/cycle exhausted the window in forty cycles
and was then matched to whichever component was nearest — on a crowded curve,
a different peak. The tolerance was therefore a ceiling on the drift the
tracker could measure, which is the one quantity ICA is most often read for.

Instead the expectation moves with the peak: each reference id carries a
running centre, updated every time it is matched, walking outward from the reference
cycle in both directions. A peak missing for some cycles keeps its last
matched position rather than reverting. Measured on synthetic data with a known trajectory, same fits, only the
matching rule changed:

| 2 mV/cycle | fixed reference | drifting reference |
|---|---|---|
| 100 cycles, true drift 192 mV | **−53 mV of a true +172 mV** | **192.21, 191.95, 131.74 mV** (err ≤ 0.3 mV) |
| 50 cycles, worst centre rms | 67.05 mV | **0.18 mV** |
| 50 cycles, worst drift error | −138.30 mV | **+0.21 mV** |
| tracking rate | 78–100% | **100%** |

The hazard of chaining is real — an error in one cycle is inherited by the
next — and it is bounded two ways: no single cycle may move further than the
window, and a single-cycle step that *does* exceed it is a discontinuity,
counted as one, and reported. `drift_exceeds_tolerance` changed meaning with
the rule: total drift is no longer a warning, it is the measurement, and the
flag now tests the per-cycle step. `TRACK_AGAINST_PREVIOUS_CYCLE = False`
restores the old behaviour.

A fixed window also COMPRESSES the drift it does report, by about 13% over 50
cycles, because it acts asymmetrically once a peak has crossed most of it. On
data whose drift stays inside the window the two rules agree exactly, which is
the test that matters: identical where a fixed reference is valid, correct
where it is not.

**Areas remain the weak quantity, and this is now the only open limit of the
three.** On synthetic data with a known answer, per-cycle fitted area is
recovered with a *bias* of 0.25–9.9% but an **rms of 10.7–59%**. It is
scatter, not bias — which is why `area_retention_pct` is computed from medians
over endpoint windows rather than single cycles, exactly as drift is, and why
a single cycle's area should not be compared with another single cycle's. The
suite's `TOL_AREA_PCT = 8` is an aspiration rather than a measured capability
and has deliberately **not** been relaxed to make the tests pass.

A caution about measuring this: two components 20 mV apart, one three times
the other's width, are not separable by any method, and scoring such a merged
doublet against either component alone reports a catastrophic error that is an
artefact of the scoring. Against their SUM the same peak's area bias is 0.25%.

**Integral fidelity below 0.80 is a property of the derivative path, not of
two-phase materials.** On the derivative-free histogram — the default — the
same data measures **0.99–1.00**. §A4 has the head-to-head.

---

## A11. Tuning: what to change, when, and how

Every default here was set on two chemistries — a sharp two-phase spinel (LTO)
and a broad layered oxide. **They are starting points, not constants of
nature.** This section is the one to read when the pipeline is applied to a
system it has not seen: it is organised by the *symptom you observe*, because
that is how the question actually arrives.

Three rules before any of the specifics:

1. **Change one thing, re-run, and look at the same figure.** Every parameter
   below is recorded in `run_manifest.json`, so two runs can be diffed. A
   change nobody can attribute is a change nobody can defend.
2. **Prefer the profile class to a hard-coded value.** Most of these are set
   per profile (`sharp` / `moderate` / `broad`) from the dataset's own curve.
   If the automatic classification is wrong for your material, fixing *that*
   fixes several parameters at once and stays honest about why.
3. **A parameter that makes the picture prettier has not made the answer more
   true.** R² in particular will improve for changes that make the result
   worse — see §A6. Judge a change on residual *structure*, on voltage
   reconstruction, and on whether the physical bounds stop complaining.

### Where they are set

Three places, and it matters which you use:

- **Cell 2** — run-level switches: `DQDV_COMPUTATION`, `DQDV_BIN_MV`,
  `MAX_CYCLE`, `RUN_PEAK_FITTING`, `CLOSURE_SAMPLE`.
- **Cell 3b, per dataset** — the operator overrides: `peak_min_width_mV`,
  `peak_min_distance_mV`, `peak_prominence_fraction`, `histogram_bin_mV`,
  `histogram_smooth_bins`. An override belonging to the dQ/dV path that is
  *not* running is reported as doing nothing rather than silently ignored.
- **Module constants** — everything else in the table below. These are
  deliberately not exposed per dataset: they are decisions about the method,
  not about a cell, and changing one should be a considered act recorded in
  the manifest.

### The parameters, by symptom

| Symptom | Parameter | Default | Which way, and why |
|---|---|---|---|
| **Curve is noisy and spiky; detection finds a peak per bump** | `histogram_smooth_bins` (per profile: sharp 1, moderate 3, broad 6) | by profile | **Up.** Binomial kernel, applied to the result, conserves area exactly — so this is safe in a way that pre-differentiation smoothing never was (§A4). Raise until detection count stabilises, not until the curve looks nice. |
| **Curve is sparse; many empty bins** | `DQDV_BIN_MV` / `histogram_bin_mV` (sharp 1, moderate 2, broad 5 mV) | by profile | **Up.** A bin narrower than the voltage spacing of the records cannot be filled. The width actually used is widened automatically and recorded in `dataset_info.txt` — if that says it was widened, the request was too fine. |
| **Fine structure is being smeared together** | same | | **Down**, but check `bins with no charge` afterwards: past ~90% empty the curve is mostly interpolation. |
| **Fits do not converge / the run is slow** | `PROFILE_SPECS[...]["baseline_degree"]` (sharp 1, moderate 2, broad 2) | | **Down.** This is the single biggest lever on a broad noisy curve. A cubic can carry an inflection, and an inflection is what lets a baseline impersonate a peak — the optimiser then crawls along that degenerate direction to the iteration ceiling. Measured on a layered oxide: cubic 40% of fits converging at 20,000 evaluations, quadratic 66-89% at a few hundred, for 0.004 of R². |
| **Still not converging on a broad curve** | `TERMINAL_TRIM` (off), `TERMINAL_TRIM_FIXED_MV` (75 mV) | | **On, deliberately.** Removes a fixed span at each half-cycle's TERMINATING end — high for charge, low for discharge — where the cell is approaching its cut-off and dQ/dV climbs for protocol reasons. Worth 66% → 87% convergence on a layered oxide with the quadratic. **Look at your curves before setting it**: it removes charge the cell really delivered, so integral fidelity falls by roughly the fraction removed, and automatic detection of the right span does NOT work (see below). |
| **A real peak is never detected** | `peak_min_width_mV` (5 mV), `peak_prominence_fraction` (0.05) | | **Down.** Both admit more peaks. Measured on LTO: dropping min width to 2.5 mV does find the discharge peak — and admits 2–3 spurious charge peaks per cycle. That trade was measured and rejected; make it deliberately, and check the spurious count. |
| **Too many peaks; components fighting over one feature** | `peak_prominence_fraction` (0.05), `peak_min_distance_mV` (30 mV) | | **Up.** `min_distance` is the blunter and safer of the two: it says how far apart two features must be to be two features, which is a statement about your material you can defend. |
| **Two peaks merge and swap identity** | `min_distance_mV`, or accept the merge | 30 mV | Components closer than roughly the wider one's σ are **not separable by any method** (§A10). Better to fit one feature and say so than to fit two and report their ratio. |
| **A tracked peak is lost partway through the run** | `TRACKING_TOLERANCE_MV` | 80 mV | **Up.** This is a *per-cycle* step limit, not a lifetime budget, so it no longer caps total drift. Raise it only if a feature genuinely moves >80 mV **between consecutive cycles**; if it does, suspect the fit before the cell. |
| **`drift_exceeds_tolerance` fires** | same | | Do **not** raise it to silence the flag. It now means a single-cycle jump, which is a discontinuity — look at the fit for those two cycles. |
| **Areas marked "not capacities"** | `INTEGRAL_FIDELITY_FLOOR` (0.80), `INTEGRAL_FIDELITY_CEILING` (1.20) | | **Usually neither.** First check `DQDV_COMPUTATION` is `"histogram"` — on the derivative path a flat plateau fails this routinely and correctly (§A4). On the histogram path a deficit means charge outside the analysed window, typically a CV hold, and the honest fix is to widen the window or accept that the areas exclude the hold. |
| **Capacity attribution withheld** | `PARTITION_DETERMINED_BELOW` | 0.15 | **Up only with an argument.** This is the spread in closure across baseline degrees; a wide spread means the peak/background split is genuinely undetermined, and raising the threshold asserts a split the data does not support. |
| **A peak "grows" past the cell's capacity** | `AREA_GROWTH_HEADROOM_PCT` | 15 pts | **Rarely.** A process can take a larger *share* as another fades, which is what the headroom allows. Consistent large excesses mean the decomposition is wrong, not that the bound is tight. |
| **Fit will not converge / σ pinned at the bound** | `FIT_SIGMA_MAX_MV` (200), `SHOULDER_SIGMA_RATIO_MIN/MAX` (0.30–1.00) | | **Up** for a genuinely broad material — a solid-solution cathode can exceed 200 mV honestly. Check `sigma_max_fitted` in the output: a component pinned at the bound is being asked to be the background. |
| **Shoulder ratio calibration declines to hold** | `SIGMA_RATIO_SPREAD_MAX` | 0.30 | Leave it. Declining is the correct behaviour when the shape is not constant across the sample; forcing it asserts a constancy the data denies. |
| **Reference cycle looks wrong (still in formation)** | `CE_SETTLED_TOLERANCE_PP` (0.1 pp), `CE_SETTLED_RUN` (3), `REFERENCE_CYCLE_FLOOR` (2), `REFERENCE_CYCLE_FALLBACK` (6) | | Formation end is **measured** from CE stabilisation, not assumed. A noisy CE series may never settle, in which case the fallback (cycle 6) is used and says so. Tighten the tolerance for high-precision data; loosen it for a noisy cycler. This choice moves the polarisation and drift numbers materially — see §A7. |
| **Retention quoted against the wrong baseline** | `RETENTION_REFERENCE_CYCLE` | 2 | Cycle 1 includes the first-cycle irreversible loss; cycle 2 does not. Both are reported in the headline metrics, so this only sets which one the *caption* names. |
| **Drift number is noisy** | `DRIFT_ENDPOINT_CYCLES` | 3 | **Up** for long runs. Two single cycles carry √2 times the per-cycle centre scatter; medians of 3 were enough to turn 90 mV of true drift from 79 into 90. |
| **Closure sampling too slow / too sparse** | `CLOSURE_SAMPLE` | 40 half-cycles | Cost is 3 fits per sampled half-cycle. Rows not sampled are marked `closure_interval_measured = False` rather than left ambiguously blank. |

### Trimming the cut-off approach, and why it is not automatic

A galvanostatic half-cycle ends on a voltage. As the cell reaches its cut-off
dV/dQ collapses and dQ/dV climbs — on a layered oxide to 1.5-2.1x the
interior of the curve, extending 200 mV in. That rise is protocol, not
chemistry, and left in it is fitted as a peak.

Removing it works: an extra 50-100 mV at the terminating end took one layered
oxide from 66%
of fits converging to 87-89%, and removed one spurious component per
half-cycle, for 0.003 of R².

**Measuring the right span automatically does not work, and the failure is
worth knowing about.** A rule that walks in from each end while the curve
stands above the interior gives a BIMODAL answer across a full run: half the
half-cycles want nothing, most of the rest saturate at the cap, and there is
no pattern in cycle number (median trim by 20-cycle band on charge: 180, 0, 0,
190, 0 mV). A 30-cycle sample can look clean and not be. So the span is an
operator setting with a default of 75 mV, not a measurement.

**The cost is real and is reported.** Trimming removes charge the cell
delivered, so integral fidelity falls by roughly the fraction removed — on a
layered oxide
about 7-9% on a layered oxide. That is the honest outcome rather than a penalty: the charge near
the cut-off *is* delivered and *is not* attributable to a resolvable feature,
and untrimmed it is silently absorbed into a fake peak and a straining
baseline. But it moves a headline number, so turn it on deliberately.

**It does nothing on a sharp two-phase profile**, which shows no terminal rise
at any threshold tested, on either step.

### When to change the *method* rather than a parameter

- **A CCCV protocol.** Charge passed during a constant-voltage hold happens at
  one voltage and cannot appear in any dV integral. Expect fidelity below 1 on
  that half-cycle; it is not an error, and the number to read is *how much of
  the cell's charge this curve is about*.
- **A strongly non-monotonic voltage curve.** If `n_oriented` is high, IC is
  being asked to represent something it structurally cannot (§A4). Use
  **differential voltage analysis (dV/dQ)** for that region instead — it has
  no such assumption.
- **A featureless material.** A continuous solid solution has no plateau to
  hold the potential and gives broad humps or nothing. That is a result about
  the material (§A1), not a parameter to tune your way out of.

---

## A12. Sources for the science in this appendix

The claims above that are not measurements from this project are drawn from:

- **E. Flores and S. Clark**, *Simple and Comparable Computation of Incremental
  Capacity Curves*, J. Electrochem. Soc. **173**, 120520 (2026), open access —
  §A1 (derivative vs histogram, voltage reconstruction, APE/RMSE) and §A4
  (sign-definite ΔV, non-monotonic voltage curves, IC as a one-to-one
  assumption).
- **Y. Ren, X. Lu, Y. Han, D. Li, Y. Chen**, *Three-Phase Transition of Spinel
  Li₄Ti₅O₁₂ with a Dense Single-Particle Microelectrode*, Energy Fuels **38**,
  16976 (2024) — §A7 (the γ phase below ca. 0.6 V, E(α/β) = 1.565 V by GITT,
  the discharge/charge asymmetry).
- **C. Zhang et al.**, *Li₄Ti₅O₁₂ spinel anode: fundamentals and advances in
  rechargeable batteries*, InfoMat **4**, e12228 (2022) — §A7 (conditions for
  solid-solution vs two-phase behaviour: nanosizing, rate, temperature;
  sub-nanometre coexistence).
- **Nature Communications 16 (2025)**, *Deciphering coulombic loss in
  lithium-ion batteries and beyond* — §A8 (coulombic inefficiency as a
  systematic overestimate of capacity fade).
- **Imperial College / J. Energy Storage**, peak-tracking to quantify
  degradation modes — §A2, §A10 (the coherence framing and what a tracking
  tolerance should be quoted as).

Everything else — the fidelity measurements, the doublet analysis, the
free-mixing-fraction demonstration, the synthetic recovery figures — was
measured in this project on the data in hand, and the numbers are reproducible
from the runs in the output folder.

---

## A13. Provenance

Every run writes `run_manifest.json` (every module constant that was in force,
input checksums, package versions, an inventory of every file written with
checksums) and a per-cell `dataset_info.txt` carrying two blocks: what the
operator **asked for**, and what the pipeline **measured and then used** —
profile class, analysed window, reference cycle and the reason for it,
calibrated peak shape, and integral fidelity per step.

The rule throughout: **the record describes what ran, never what was offered.**
A record field that disagrees with what actually happened is worse than an
absent one, because it reads as an assurance.
